# CAL01 공정 시계열 기반 이상 구간 탐지 및 공정 신호·정비 이력 관계 분석
**담당: 정수진 · Issue: #65**  
**분석 단위: LOT별 관측 구간 · 작업 WO · 실제 시각의 교집합**

2024년 **14 LOT 관측 구간**에서 운전 전환, 반복 CP 이탈, 같은 강종의 수준 차이와 정비 후 변화를 구분했다. 확인된 변화를 점검 우선순위와 후속 검증 항목으로 연결한다. 고장·품질 예측 모델은 만들지 않는다.

1. [Executive Summary — 핵심 결과](#overview)
2. [분석 목적 및 데이터 특성](#data-scope)
3. [전처리·탐지 방법과 판단 범위](#methods)
4. [전체 시계열 특성과 탐지 결과](#screening)
5. [주요 LOT 상세 분석과 계획수리 대조](#cases)
6. [공정별 시계열 특성 종합](#process-summary)
7. [종합 판단 및 현장 적용 방안](#conclusion)
8. [부록 — 14 LOT 전체 검토·재현 코드](#detail)

**읽는 연결:** 전체 관찰 → 탐지 근거 → 고정 구간 번호(예: 240024 (a)) → 같은 번호의 확대·동반 반응 → 정비와의 시간 관계 → 판단. 각 확대 상단에 전체 LOT의 같은 신호와 현재 위치를 다시 표시한다.

시계열 구간 표시색은 **빨강=CP 원본 이탈 / 주황=시간순 Z / 회색=운전 전환 참고 / 보라=정비 감지 대조**다. IQR 수염 밖 점은 청록이다. 색은 고장 등급이 아니며 모든 그림은 원본 CSV에서 코드로 생성한다.

**중심 연구 질문**

1. CP-Z4와 Z3~Z6의 TC·OP 변화가 사건마다 반복되는가?
2. 변화는 정비 감지·착수·완료의 어느 쪽에 위치하며 작업 대상과 연결되는가?
3. CP가 없는 반례와 운전 전환을 대조했을 때 점검할 근거가 남는가?

**코드 읽는 법:** 핵심 그림의 matplotlib 함수는 해당 절의 실행 셀에 있다. 공통 색·축·파일 저장·통계 전처리만 앞에서 재사용한다. 이미지를 읽어 붙이지 않고 CSV에서 다시 그린다.

관계 분석 히트맵의 빨강·파랑은 변화량 증가·감소를 뜻하며 해당 그림 범례를 따른다. [5.6 사건별 관계](#relation-events) → [5.7 CP 전후 파형](#relation-lags) → [5.7.1 TC-OP 관계](#relation-tc-op) → [5.7.2 출력 뒤 온도 복귀](#tc-op-response) → [5.7.3 품질 검증 근거](#tc-quality-evidence) → [5.8 CP 이탈 유무 비교](#relation-controls) → [5.9 채널·정비](#relation-maintenance) 순서로 연결한다.


In [ ]:
from pathlib import Path
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "notebooks").is_dir())
DATA_DIR = str(PROJECT_ROOT / "data")
if (PROJECT_ROOT / "data" / "raw" / "T-CR1-CAL01_온도.csv").is_file():
    DATA_DIR = str(PROJECT_ROOT / "data" / "raw")
FIG_DIR = str(PROJECT_ROOT / "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = [cand, "DejaVu Sans"]
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

# 한글은 설치된 한글 폰트, 지수·수학 기호는 DejaVu Sans가 보완한다.
# 빠진 글리프 경고를 숨기지 않고 실행 시 오류로 확인한다.
import warnings
warnings.filterwarnings("error", message=r"Glyph .* missing from font")
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110
"""Shared presentation rules for the five CAL01 time-series notebooks.

Only artists and display labels are changed here. Sensor values, thresholds,
aggregation windows and axis limits belong to the notebooks.
"""


import html
import re
from pathlib import Path

from matplotlib import rcParams
from matplotlib.lines import Line2D

EXPORT_DPI = 180
EXPORT_PADDING = 0.20
INK = '#1F2937'
MUTED = '#64748B'

# A color always identifies the same member of the same domain. Different
# domains use separate legends. LOT colors contrast most within the same grade.
PROCESS_COLORS = {'세정': '#2563EB', '건조': '#D97706', '가열·균열': '#DC4C3E', '과시효': '#059669', '냉각': '#7C3AED'}
GRADE_COLORS = {'SGCC': '#B9D6D9', 'SPFH590': '#B4634E', 'SPCE': '#3F909A', 'SPCEN': '#8676AC', 'SPHC': '#A48258', 'DP590': '#2B527A', 'SPCD': '#3E75A8', 'SPCC': '#9AA7B5'}
LOT_COLORS = {240001: '#334155', 240024: '#2563EB', 240108: '#F97316', 240037: '#1D4ED8', 240121: '#D946EF', 240133: '#10B981', 240038: '#0891B2', 240114: '#DC2626', 240044: '#0F766E', 240061: '#D97706', 240069: '#7C3AED', 240091: '#DB2777', 240058: '#92400E', 240117: '#475569'}
MAINTENANCE_COLORS = {'BM': '#C81E3A', 'CBM': '#6D28D9', 'TBM': '#0E7490'}
MAINTENANCE_LABELS = {"BM": "BM · 고장 후 정비", "CBM": "CBM · 상태 기준 정비", "TBM": "TBM · 주기 기준 정비"}
MAINTENANCE_HATCHES = {"BM": "///", "CBM": "...", "TBM": "xxx"}
ZONE_COLORS = {1: '#475569', 2: '#B45309', 3: '#7C3AED', 4: '#2563EB', 5: '#F97316', 6: '#059669', 7: '#DB2777', 8: '#0891B2', 9: '#A16207'}
SENSOR_COLORS = {'CLN-IN': '#0F766E', 'CP-Z4': '#E11D48', 'CP-Z4M': '#64748B', 'TC-OUT1': '#4F46E5', 'TC-OUT2': '#CA8A04'}

# LOT is encoded by color. Measured OP uses a dashed line; TC/CP use solid lines.
LOT_LINESTYLES = {240001: '-', 240024: '-', 240037: '-', 240038: '-', 240044: '-', 240058: '-', 240061: '-', 240069: '-', 240091: '-', 240108: '-', 240114: '-', 240117: '-', 240121: '-', 240133: '-'}

# Full original explanations remain available immediately below each figure.
# Short figure text describes the target, comparison rule and reading purpose.
_HEADINGS = {
    "연간 공정 흐름": ("2024년 CAL01 관측·정비 흐름", "2024년 관측창과 정비 시각을 대조해 비교 범위 확인"),
    "LOT별 자료 상태 · 분석에 쓸 수 있는 구간과 빠진 곳": ("CAL01 14 LOT 관측 품질 비교", "LOT별 기록량과 결측을 대조해 분석 가능 구간 확인"),
    "측정 기록 공백·센서 NaN 비교": ("CAL01 측정 공백·센서 결측 비교", "같은 LOT의 TC·OP를 대조해 결측 종류 구분"),
    "운전 상태별 급변 — 정지·승온을 따로 떼어야 하는 이유": ("CAL01 운전 상태별 급변 크기", "신호별 안정 운전 기준으로 정지·승온의 급변 비교"),
    "정비 일정 전후 정지·재가동 — 설명됨의 근거": ("CAL01 정비 전후 정지·재가동", "14 LOT의 온도·출력을 일정과 대조해 전환 구간 확인"),
    "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳": ("CAL01 14 LOT의 CP 기준 이탈", "CP 편차(무차원) ±0.05 이탈의 위치·횟수 비교"),
    "탐지된 구간이 어디로 갔나 — 근거가 있어 빼는 것과 03으로 넘기는 것": ("CAL01 탐지 구간의 검토 대상 선별", "탐지 구간을 설명 근거별로 나눠 상세 검토 대상 확인"),
    "안정 참고 구간 · 가열·균열 Z3 수준": ("CAL01 강종별 Z3 온도·출력 수준", "같은 강종의 안정 참고 중앙값으로 LOT 수준 비교"),
    "안정 참고 구간 · Z3 제어출력과 온도를 같이 보기": ("CAL01 LOT별 Z3 온도·출력 반응", "LOT별 안정 참고 구간에서 온도와 출력의 동행 확인"),
    "출력이 0%·100%에 붙을 때 Z3 온도는 어떻게 움직이나": ("CAL01 Z3 출력 한계·온도 반응", "출력 0%·100% 지속 구간의 온도 반응을 함께 확인"),
    "세정 입구 온도 (CLN-IN)": ("CAL01 LOT별 세정 입구 온도", "LOT 중앙값과 같은 강종의 흐름으로 세정 온도 비교"),
    "정비 전후 흐름 · LOT 사이 정비와 LOT별 상태 지표": ("CAL01 정비 시각·LOT 지표 비교", "관측 날짜 순 지표와 LOT 사이 정비의 시간 관계 확인"),
    "볼 것 고르기 — 평소 흔들림 대비 편차": ("CAL01 평소 변동 대비 검토 우선순위", "신호별 평소 변동으로 나눈 편차로 검토 대상 비교"),
    "CP의 평소 관계 — 14 LOT에서 먼저 확인": ("CAL01 14 LOT의 CP 동시 관계", "안정 참고 1초 변화 상관으로 CP와 센서의 동행 확인"),
    "어느 LOT이 CP의 평소 관계에서 벗어났나 — 14 LOT 비교": ("CAL01 LOT별 CP 변동·동조 비교", "안정 참고 구간의 이탈·변동·동조로 검토 LOT 비교"),
    "CP 관점에서 확인할 LOT · 같은 강종부터 비교": ("CAL01 강종별 CP 검토 LOT 비교", "같은 강종의 시간당 이탈 빈도와 크기로 후보 비교"),
    "가스 분석계 셀 교체와 CP 신호 상태": ("CAL01 분석계 셀 교체·CP 상태", "CP 계측 부품의 교체 시각과 시간별 신호 상태 대조"),
    "같은 강종끼리 대조한 LOT별 요약 · 과시효 Z7·Z8": ("CAL01 동일 강종의 Z7·Z8 비교", "같은 강종 다른 LOT 중앙값 대비 온도·출력 차이 확인"),
    "14 LOT × 공정 · 확인 이슈와 근거 한눈에 보기": ("CAL01 14 LOT 공정별 검토 근거", "공정별 비교 가능성과 근거로 추가 확인할 LOT 선별"),
    "과시효 Z7·Z8 · 같은 강종 LOT끼리 비교": ("CAL01 동일 강종의 과시효 온도 비교", "안정 참고 1분 중앙값 분포로 LOT 수준 차이 확인"),
}


def configure():
    rcParams.update({
        "font.size": 11.5, "axes.titlesize": 13, "axes.labelsize": 11.5,
        "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
        "savefig.dpi": EXPORT_DPI, "savefig.pad_inches": EXPORT_PADDING,
        "svg.fonttype": "none", "pdf.fonttype": 42,
    })


def sensor_color(sensor):
    if sensor.startswith(("TC-Z", "OP-Z")):
        return ZONE_COLORS[int(sensor.rsplit("Z", 1)[1])]
    return SENSOR_COLORS[sensor]


def lot_linestyle(lot):
    return '-'


def process_title_color(title, default=INK):
    return INK


def palette_preview(path):
    """Read the guide in order: signal type, zone color, then compared LOTs."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from IPython.display import display
    fig = plt.figure(figsize=(16, 10.2), facecolor="#F5F7FB")
    fig.text(.045, .955, "CAL01 시계열 · 색상과 선 표기", fontsize=25, weight="bold", color=INK, va="top")
    fig.text(.045, .909, "TC는 실선 · OP는 점선  |  같은 존은 같은 색  |  LOT 비교는 같은 강종 안에서 색으로 구분", fontsize=12, color=MUTED, va="top")

    def card(x, y, w, h, title, subtitle=""):
        fig.add_artist(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.008,rounding_size=0.012",
                       transform=fig.transFigure,fc="white",ec="#E2E8F0",lw=.8,zorder=0))
        ax=fig.add_axes([x+.014,y+.014,w-.028,h-.028],zorder=1)
        ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis("off")
        ax.text(0,.96,title,va="top",fontsize=14,weight="bold",color=INK)
        if subtitle:ax.text(0,.79,subtitle,va="top",fontsize=10.5,color=MUTED)
        return ax

    ax=card(.045,.661,.438,.207,"1  선 모양 = 측정 신호","TC·OP는 색을 바꾸지 않고 선 모양으로 구분")
    for y,s,label in [(.55,"TC-Z4","TC 온도 (℃) · 실선"),(.29,"OP-Z4","OP 제어출력 (%) · 점선")]:
        ax.plot([.015,.27],[y,y],color=ZONE_COLORS[4],lw=2.8,ls=signal_linestyle(s))
        ax.text(.32,y,label,va="center",fontsize=12.5,color=INK)
    ax.text(.015,.06,"CP는 실선 · 무차원",fontsize=10.5,color=MUTED)

    ax=card(.504,.661,.451,.207,"2  선 색 = 존(ZONE)","같은 ZONE의 TC와 OP는 아래 두 선처럼 같은 색")
    for i,(z,col) in enumerate(ZONE_COLORS.items()):
        row,column=divmod(i,3);x=column*.337;y=.56-row*.20
        ax.text(x,y,f"Z{z}",va="center",fontsize=11.5,color=INK)
        ax.plot([x+.085,x+.175],[y+.027,y+.027],color=col,lw=2.1)
        ax.plot([x+.085,x+.175],[y-.027,y-.027],color=col,lw=2.1,ls="--")

    for x,w,title,colors in [(.045,.291,"공정 · 축 옆 구획색",PROCESS_COLORS),
                              (.357,.291,"강종 · 그룹 표식",{k:GRADE_COLORS[k] for k in ["SPCC","SPCD","SPCE","SPCEN","SPHC","SPFH590","DP590","SGCC"]}),
                              (.669,.286,"정비 · 실제 작업 구간",MAINTENANCE_COLORS)]:
        ax=card(x,.476,w,.151,title)
        columns=3 if len(colors)>5 else 2 if len(colors)>3 else 3
        for i,(label,col) in enumerate(colors.items()):
            row,column=divmod(i,columns);xx=column/columns;yy=.61-row*.235
            ax.plot([xx+.01,xx+.048],[yy,yy],color=col,lw=6,solid_capstyle="round")
            ax.text(xx+.073,yy,label,va="center",fontsize=9.8,color=INK)
        if title.startswith("정비"):
            ax.text(.01,.18,"BM 고장후  ·  CBM 상태기준  ·  TBM 주기기준",fontsize=9,color=MUTED)

    fig.text(.045,.437,"3  LOT 비교 · 아래 같은 강종 안에서 선 색을 대조",fontsize=14,weight="bold",color=INK)
    groups=[("SGCC",[240001]),("SPFH590",[240024,240108]),("SPCE",[240037,240121,240133]),
            ("SPCEN",[240038,240114]),("SPHC",[240044,240061,240069,240091]),
            ("DP590",[240058]),("SPCD",[240117])]
    for i,(grade,ids) in enumerate(groups):
        row,column=divmod(i,4);x=.045+column*.232;y=.237-row*.196
        ax=card(x,y,.213,.171,grade)
        for j,k in enumerate(ids):
            yy=.69-j*.175
            ax.plot([.015,.29],[yy,yy],color=LOT_COLORS[k],lw=2.6,ls="-")
            ax.text(.35,yy,str(k),va="center",fontsize=11.5,color=INK)
    ax=card(.741,.041,.213,.171,"비교 그림 읽는 법")
    ax.text(.01,.67,"대상 LOT  굵게\n대조 LOT  얇게\nTC 실선 / OP 점선",va="top",fontsize=11.5,color=MUTED,linespacing=1.65)
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=EXPORT_DPI,facecolor=fig.get_facecolor(),bbox_inches="tight",pad_inches=.18)
    fig.set_dpi(72);display(fig);plt.close(fig)



def signal_linestyle(signal):
    return "--" if str(signal).startswith("OP-") else "-"


def _signal_kind(label, axis_text=""):
    label=str(label)
    if re.search(r"\bOP-Z\d",label) or label.startswith("OP "):return "OP"
    if re.search(r"\bTC-(?:Z\d|OUT)",label) or label=="CLN-IN" or label.startswith("TC "):return "TC"
    if label.startswith("CP-"):return "CP"
    if "출력" in label and "온도" not in label:return "OP"
    # A LOT label carries color identity; its panel supplies the measurement.
    if "제어출력" in axis_text or "출력[%]" in axis_text or "[%]" in axis_text:return "OP"
    if "온도" in axis_text or "[℃]" in axis_text:return "TC"
    if "무차원" in axis_text and "CP" in axis_text:return "CP"
    return None


def style_signal_lines(fig):
    """Color identifies a zone/LOT; a dashed measured curve identifies OP."""
    for ax in fig.axes:
        titles=" ".join([ax.title.get_text(),ax._left_title.get_text(),ax._right_title.get_text()])
        context=ax.get_ylabel()+" "+titles
        for line in ax.lines:
            if len(line.get_xdata())<3 or line.get_linestyle() in ["None","none","",None]:continue
            if line.get_transform()!=ax.transData:continue
            label=line.get_label();kind=_signal_kind(label,context)
            if kind:line.set_linestyle("--" if kind=="OP" else "-")
            if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",str(label)):
                line.set_color(sensor_color(label))
        # A restrained axis rail carries process context; titles stay dark.
        found=[p for p in PROCESS_COLORS if p in titles or (p=="가열·균열" and ("가열" in titles or "균열" in titles))]
        if len(found)==1 and ax.axison:
            ax.spines["left"].set_color(PROCESS_COLORS[found[0]])
            ax.spines["left"].set_alpha(.65)
            ax.spines["left"].set_linewidth(1.0)
        for title in [ax.title,ax._left_title,ax._right_title]:
            if title.get_text() and str(title.get_color()).lower() not in ["#c83745","#d62728","#e11d48"]:
                title.set_color(INK)
        legend=ax.get_legend()
        if legend is not None:_style_legend(legend,context)
    for legend in fig.legends:_style_legend(legend)


def _style_legend(legend, context=""):
    handles=getattr(legend,"legend_handles",getattr(legend,"legendHandles",[]))
    for handle,text in zip(handles,legend.get_texts()):
        if not isinstance(handle,Line2D) or handle.get_linestyle() in ["None","none","",None]:continue
        label=text.get_text()
        if any(word in label for word in ["기준","중앙값","경계","감지","정비","수리","범위"]):continue
        kind=_signal_kind(label,context)
        if kind:handle.set_linestyle("--" if kind=="OP" else "-")
        if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",label):
            handle.set_color(sensor_color(label))


def _heading(title, subtitle):
    if title in _HEADINGS:
        return _HEADINGS[title]
    patterns = [
        (r"CAL01 정비 이력 (\d+)건", lambda m: (f"CAL01 정비 {m[1]}건의 작업 흐름", "작업별 감지·착수·완료와 관측창을 대조해 해석 범위 확인")),
        (r"남은 짧은 변화 · (\d+)", lambda m: (f"LOT {m[1]} 탐지 구간과 반복 파형 대조", "같은 LOT의 1초 원본으로 탐지 구간의 동시 변화 확인")),
        (r"같은 강종 LOT 전체 비교 · (\w+) · (.+)", lambda m: (f"{m[1]} LOT별 공정 신호 비교", "같은 강종·같은 신호의 경과시간으로 LOT 간 차이 확인")),
        (r"같은 강종 LOT 비교 · (\w+) · 대상 (\d+) · Z4", lambda m: (f"LOT {m[2]} 동일 강종 Z4 비교", "안정 참고 시작을 맞춰 같은 강종의 온도·출력 비교")),
        (r"(\d+) 정비 전·중·후", lambda m: (f"LOT {m[1]} 정비 전후 신호 비교", "같은 LOT의 실제 작업 경계와 온도·출력 변화를 대조")),
        (r"(\d+) CP 변화·정비", lambda m: (f"LOT {m[1]} CP·정비 시각 대조", "같은 LOT의 CP 극값과 정비·교체 기록의 시간 관계 확인")),
        (r"(\d+) · 언제 CP", lambda m: (f"LOT {m[1]} CP 동조 변화 구간", "안정 참고의 시간별 변동과 1초 동조로 변화 시점 확인")),
        (r"CP 펄스 뒤 무엇이 달라졌나 · (.+)", lambda m: (f"CAL01 CP 펄스 전후 반응 {m[1]}", "같은 LOT의 펄스 없는 안정 참고 시각과 전후 반응 비교")),
    ]
    for pattern, make in patterns:
        match = re.search(pattern, title)
        if match:
            return make(match)
    return title, subtitle


def header(fig, title, subtitle="", handles=(), ncol=4):
    original_title, original_subtitle = title, subtitle
    title, subtitle = _heading(title, subtitle)
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=23, fontweight="bold", color=INK)
    sub = fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=12, color=MUTED)
    fig._report_subtitle = sub
    fig._report_original = {"title": original_title, "subtitle": original_subtitle}
    fig._report_details = []
    if handles:
        # Legend order is the caller's comparison order, never an alphabetic sort.
        labels = [h.get_label() for h in handles]
        fig.legend(handles=handles, labels=labels, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=11, handlelength=2.4,
                   columnspacing=1.5, labelspacing=.65)


def _panel_titles(fig, renderer):
    """Keep the identifying clause on the chart and retain the full caption."""
    for ax in fig.axes:
        title = ax._left_title if ax._left_title.get_text() else ax.title
        original = title.get_text()
        if title.get_color().lower() in [INK, "#17344d"]:
            title.set_color(process_title_color(original, title.get_color()))
        if not original or "\n" in original:
            continue
        if len(original) <= 66 and title.get_window_extent(renderer).width <= ax.bbox.width:
            continue
        # A long explanatory clause belongs below the figure. Do not cut a
        # label in the middle of a sensor name, quantity or comparison value.
        short = re.split(r" — | · (?=회색|위 |아래 |점 =|빨간|파란|평소|숫자|칸 =|세로)| \| ", original, maxsplit=1)[0]
        if short == original or len(short) > 66:
            continue
        fig._report_details.append(original)
        title.set_text(short)


def _legend_outside(fig, renderer):
    """Move an inside legend to a measured free title-row space when it fits."""
    moved = 0
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None or not leg.get_window_extent(renderer).overlaps(ax.bbox):
            continue
        handles, labels = ax.get_legend_handles_labels()
        if not labels:
            continue
        candidate = ax.legend(handles, labels, loc="lower right", bbox_to_anchor=(1, 1.025),
                              borderaxespad=0, ncol=min(4, len(labels)), frameon=False,
                              fontsize=10.5, handlelength=2, columnspacing=1.2)
        fig.canvas.draw()
        box = candidate.get_window_extent(renderer)
        obstacles = [a.bbox for a in fig.axes if a is not ax and a.bbox != ax.bbox]
        texts = list(fig.texts)
        for other in fig.axes:
            texts += [other._left_title, other._right_title, other.title, other.xaxis.label]
            texts += list(other.texts)
        obstacles += [t.get_window_extent(renderer) for t in texts if t.get_visible() and t.get_text()]
        if any(box.overlaps(b) for b in obstacles) or box.y1 > fig.bbox.y1:
            candidate.remove()
            ax.add_artist(leg)
            ax.legend_ = leg
        else:
            candidate._detail_legend = True
            moved += 1
    return moved


def prepare(fig):
    """Arrange the figure header in physical units, preserving data limits."""
    if getattr(fig, "_report_prepared", False):
        return
    fig._report_prepared = True
    style_signal_lines(fig)
    fig._report_details = getattr(fig, "_report_details", [])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    _panel_titles(fig, renderer)
    if hasattr(fig, "_report_subtitle"):
        height = fig.bbox.height
        gap = 9 * fig.dpi / 72
        title = fig._suptitle
        title.set_y(1 - .20 / fig.get_figheight())
        title.set_fontsize(min(25, title.get_fontsize()))
        fig.canvas.draw()
        sub = fig._report_subtitle
        sub.set_position((.035, (title.get_window_extent(renderer).y0 - gap) / height))
        sub.set_fontsize(12)
        sub.set_va("top")
        fig.canvas.draw()
        cursor = sub.get_window_extent(renderer).y0 - gap
        for legend in fig.legends:
            legend.set_bbox_to_anchor((.031, cursor / height), transform=fig.transFigure)
            fig.canvas.draw()
            cursor = legend.get_window_extent(renderer).y0 - gap
        # Existing charts already allocate a header. Grow it only if the
        # measured text requires additional room, including secondary axes.
        if fig.axes:
            positions = {ax: ax.get_position().frozen() for ax in fig.axes}
            low = min(p.y0 for p in positions.values())
            high = max(p.y1 for p in positions.values())
            visible_top = max((ax.get_tightbbox(renderer).y1 for ax in fig.axes if ax.get_visible()), default=high*height)
            extra = max(0, visible_top - high*height)
            target = (cursor - extra) / height
            target = min(target, high + .55 / fig.get_figheight())
            if abs(target-high) > .003:
                scale = (target-low) / (high-low)
                if scale < .55:
                    raise RuntimeError("그림 머리글에 비해 본문 높이가 부족합니다.")
                for ax, p in positions.items():
                    ax.set_position([p.x0, low+(p.y0-low)*scale, p.width, p.height*scale])
                for text in fig.texts:
                    if text in [title, sub]:
                        continue
                    x, y = text.get_position()
                    if low <= y <= high:
                        text.set_position((x, low+(y-low)*scale))
    fig.canvas.draw()
    fig._report_legends_moved = _legend_outside(fig, fig.canvas.get_renderer())


def details_html(fig):
    original = getattr(fig, "_report_original", {})
    details = [original.get("subtitle", ""), *getattr(fig, "_report_details", [])]
    details = list(dict.fromkeys(text for text in details if text))
    if not details:
        return ""
    body = "".join(f"<p>{html.escape(text)}</p>" for text in details)
    return '<details data-report-visuals="caption"><summary>그림 상세 기준·해석 펼치기</summary>' + body + "</details>"


# 각 노트북 안에서 완결되는 공통 그림 설정
from types import SimpleNamespace
report_viz = SimpleNamespace(
    EXPORT_DPI=EXPORT_DPI,
    EXPORT_PADDING=EXPORT_PADDING,
    PROCESS_COLORS=PROCESS_COLORS,
    GRADE_COLORS=GRADE_COLORS,
    LOT_COLORS=LOT_COLORS,
    MAINTENANCE_COLORS=MAINTENANCE_COLORS,
    MAINTENANCE_LABELS=MAINTENANCE_LABELS,
    MAINTENANCE_HATCHES=MAINTENANCE_HATCHES,
    ZONE_COLORS=ZONE_COLORS,
    SENSOR_COLORS=SENSOR_COLORS,
    LOT_LINESTYLES=LOT_LINESTYLES,
    header=header,
    prepare=prepare,
    details_html=details_html,
    sensor_color=sensor_color,
    lot_linestyle=lot_linestyle,
    process_title_color=process_title_color,
    palette_preview=palette_preview,
    signal_linestyle=signal_linestyle,
    style_signal_lines=style_signal_lines,
)
configure()

In [ ]:
NOTEBOOK_NO = "06"
from pathlib import Path
from IPython.display import display, Markdown
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg = [frames[k] for k in ["부품", "정비", "교체"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = report_viz.LOT_COLORS.copy()
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": report_viz.MAINTENANCE_COLORS["BM"],
       "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"], "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships)})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / "jinny" / "05_all_data_timeserise_jinny"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(event_text(text)))

def table(frame, columns=None, opened=False):
    """계산 결과표는 접어 둔다(눌러서 펼침). 본문에는 그림과 글로 쓴 결론·판단표만 펼쳐 둔다."""
    from IPython.display import HTML
    shown = frame if columns is None else frame[columns]
    if isinstance(shown, pd.Series):
        shown = shown.to_frame()
    shown = event_frame(shown)
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        body = shown.to_html()
    display(HTML(f"<details{' open' if opened else ''}><summary>표 펼쳐 보기 · {len(shown)}행</summary>{body}</details>"))

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":report_viz.MAINTENANCE_COLORS["BM"],"CBM":report_viz.MAINTENANCE_COLORS["CBM"],"TBM":report_viz.MAINTENANCE_COLORS["TBM"]})
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M [무차원]"
    if signal == "CP-Z4":
        return "CP-Z4 [무차원]"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    return report_viz.header(fig, title, subtitle, handles, ncol)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}", ls=report_viz.lot_linestyle(int(k))) for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.
layout_reviews = []

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in axes]
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in axes], "배치 변경으로 축 범위가 달라짐"
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.2인치) 둔다.
PAD = report_viz.EXPORT_PADDING
rcParams["savefig.pad_inches"] = PAD



# 실제 캔버스에서 글자·범례·폰트 지원 여부를 확인한다. 표본 그림만 검사하지 않는다.
render_audit_checks = {}
_render_font_chars = {}
_render_supported_chars = {}
def audit_rendered_figure(fig, name):
    from matplotlib.text import Text
    from matplotlib.transforms import Bbox
    from matplotlib import font_manager, ft2font
    import unicodedata
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    texts = list(fig.texts)
    legends = list(fig.legends)
    for ax in fig.axes:
        texts.extend(ax.texts)
        texts.extend([ax.title, ax._left_title, ax._right_title])
        if ax.get_legend() is not None:
            legends.append(ax.get_legend())
        if ax.axison:
            for axis in (ax.xaxis, ax.yaxis):
                if not axis.get_visible():
                    continue
                texts.extend([axis.label, axis.offsetText])
                low, high = sorted(axis.get_view_interval())
                for tick in axis.get_major_ticks() + axis.get_minor_ticks():
                    if low - 1e-7 <= tick.get_loc() <= high + 1e-7:
                        texts.extend([tick.label1, tick.label2])
    texts = list({id(t): t for t in texts if t.get_visible() and t.get_text()}.values())
    legends = list({id(l): l for l in legends if l.get_visible()}.values())
    # 범례 내부 글자는 범례 상자와 중복 비교하지 않되, 글리프는 함께 검사한다.
    glyph_texts = texts + [t for legend in legends for t in legend.get_texts()]
    for text in glyph_texts:
        prop = text.get_fontproperties()
        font_key = (tuple(prop.get_family()), prop.get_style(), str(prop.get_weight()),
                    str(prop.get_stretch()), prop.get_variant())
        if font_key not in _render_supported_chars:
            supported = set()
            for family in prop.get_family():
                single = prop.copy()
                single.set_family([family])
                path = font_manager.findfont(single)
                if path not in _render_font_chars:
                    _render_font_chars[path] = set(ft2font.FT2Font(path).get_charmap())
                supported.update(_render_font_chars[path])
            _render_supported_chars[font_key] = supported
        supported = _render_supported_chars[font_key]
        missing = sorted({ch for ch in text.get_text() if not ch.isspace()
                          and not unicodedata.category(ch).startswith('C') and ord(ch) not in supported})
        assert not missing, f'폰트 미지원: {name}: {missing}: {text.get_text()}'
        assert '\ufffd' not in text.get_text() and '\u25a1' not in text.get_text(), f'깨진 글자: {name}'
    artists = texts + legends
    boxes = [(a, a.get_window_extent(renderer)) for a in artists]
    for artist, box in boxes:
        assert box.x0 >= -.5 and box.y0 >= -.5 and box.x1 <= fig.bbox.x1 + .5 and box.y1 <= fig.bbox.y1 + .5, (
            f'글자/범례 잘림: {name}: {getattr(artist, "get_text", lambda: "범례")()}')
    min_gap = float('inf')
    for i, (artist, box) in enumerate(boxes):
        for other, ob in boxes[i+1:]:
            inter = Bbox.intersection(box, ob)
            assert inter is None or inter.width <= 1 or inter.height <= 1, (
                f'글자/범례 겹침: {name}: {getattr(artist, "get_text", lambda: "범례")()} / {getattr(other, "get_text", lambda: "범례")()}')
            if box.x0 < ob.x1 and ob.x0 < box.x1:
                min_gap = min(min_gap, max(box.y0 - ob.y1, ob.y0 - box.y1, 0))
    bodies = [ax for ax in fig.axes if ax.axison]
    for legend in legends:
        box = legend.get_window_extent(renderer)
        for ax in bodies:
            inter = Bbox.intersection(box, ax.bbox)
            assert inter is None or inter.width <= 1 or inter.height <= 1, f'범례가 파형을 가림: {name}'
    render_audit_checks[name] = dict(TEXTS=len(glyph_texts), LEGENDS=len(legends),
        GLYPHS='통과', TEXT_OVERLAP='통과', LEGEND_OVERLAP='통과', BOUNDS='통과',
        MIN_VERTICAL_GAP_PT=round(min_gap * 72 / fig.dpi, 2))


def finish(fig, name, svg=False, save=True):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다. save=False면 파일로 저장하지 않고 노트북에만 보인다."""
    fig.set_dpi(report_viz.EXPORT_DPI)
    report_viz.prepare(fig)
    layout = compact_explanations(fig)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for tag in ax.texts:
            if getattr(tag, "_review_label", False):
                assert not text_box(tag, renderer).overlaps(ax.get_window_extent(renderer)), "확인 표시가 파형을 가림"
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    # 여백을 잘라 저장할 때 그림 위쪽 범례만 따로 밀리지 않도록, 범례 위치를 그림 좌표에 다시 묶는다.
    for legend in fig.legends:
        anchor = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(anchor, transform=fig.transFigure)
    audit_rendered_figure(fig, name)
    path = OUT_DIR / name
    if save:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=report_viz.EXPORT_DPI, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg and save:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과", "SAVED": "저장" if save else "노트북만"})
    layout_reviews.append({"FILE": name, **layout})
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(180dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display_report_figure(fig)
    caption = report_viz.details_html(fig)
    if caption:
        from IPython.display import HTML
        display(HTML(caption))
    plt.close(fig)
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])
assert np.isclose(sum(b - a for a, b in work_ranges[240117]), 304)

NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]

relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "수리 전 하강 전환"), ("초기 승온", "수리 후 승온 형태")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())




# 강종 수준 대조는 수리 전의 완만한 냉각도 보수적으로 제외한다.
# 2℃는 사후 대조 구간을 자르는 기술 기준이며 설비 이상 임계값이 아니다.
comparison_states={k:st.copy() for k,st in states.items()}
for k in [240058,240114]:
    r=lot_meta.loc[k];p=profile[k]
    ref=raw[k].loc[raw[k].ELAPSED.between(r.MINUTES*.2,r.MINUTES*.6),HEAT[1:]].median()
    falling=pd.concat([(p[(s,'median')]-ref[s]).le(-2) for s in HEAT[1:]],axis=1).all(axis=1)
    hit=falling.rolling(3,min_periods=3).sum().eq(3)&pd.Series(p.index,index=p.index).gt(r.MINUTES*.5)
    if hit.any():comparison_states[k].loc[p.index>=max(0,int(hit.index[hit][0])-2)]='종료 전환'

level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(comparison_states[k].eq("안정 참고") & profile[k][(s,"count")].ge(54)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN



# 동종 LOT 수준 차이의 표시 범위. 시간순 Z 점수·고장 판정과 별개다.
LEVEL_TC=[f'TC-Z{i}' for i in range(1,9)]
LEVEL_OP=[f'OP-Z{i}' for i in range(1,9)]
LEVEL_SIGNALS=LEVEL_TC+LEVEL_OP
LEVEL_TC_LIMIT,LEVEL_OP_LIMIT=2.5,4.0
level_check=level_gap.loc[level_gap.SENSOR.isin(LEVEL_SIGNALS)].copy()
level_check['DISPLAY_LIMIT']=np.where(level_check.SENSOR.isin(LEVEL_TC),LEVEL_TC_LIMIT,LEVEL_OP_LIMIT)
level_check['DISPLAY_HIT']=level_check.N_PEERS.gt(0)&level_check.GAP.abs().ge(level_check.DISPLAY_LIMIT)
level_check['STATUS']=np.where(level_check.N_PEERS.eq(0),'동일 강종 대조 없음',np.where(level_check.DISPLAY_HIT,'수준 차이 표시 대상','표시 경계 미만'))
# 기존 우선 검토 5 LOT의 그림 범위는 이제 위 16신호 계산 결과에서 결정한다.
LEVEL_TARGETS={k:level_check.loc[level_check.LOT.eq(k)&level_check.DISPLAY_HIT,'SENSOR'].tolist()
               for k in [240038,240061,240091,240114,240121]}
level_inventory=level_gap.copy()
level_inventory['비교 용도']=level_inventory.SENSOR.map(lambda s:'TC·OP 수준 비교' if s in LEVEL_SIGNALS else 'LOT 내부 국소 변화' if s=='CLN-IN' else '주기·위상 별도 확인' if s in ['TC-Z9','TC-OUT1','TC-OUT2'] else 'CP 원본·계측 정의 별도 확인')
assert len(level_check)==len(lots)*16 and all(LEVEL_TARGETS.values())

from matplotlib.patches import Rectangle
SELECT_COLOR="#c83745"

def sensor_color(s):
    return report_viz.sensor_color(s)

def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"

def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]

def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")

# 품질 판단과 본문이 같은 온도 변화 점수 객체를 사용한다.
Z_WINDOW,Z_MIN,Z_LIMIT=30,20,3
zstats={};z_rows=[]
for k in lots.LOT_NO.astype(int):
    for s in NONCYCLIC+OP:
        x=profile[k][(s,'median')].where(profile[k][(s,'count')].ge(54))
        past=x.shift(1);center=past.rolling(Z_WINDOW,min_periods=Z_MIN).median()
        sd=past.rolling(Z_WINDOW,min_periods=Z_MIN).std().clip(lower=.5 if s in TC else 2.)
        z=(x-center)/sd;zstats[k,s]=pd.DataFrame({'X':x,'MEDIAN':center,'SD':sd,'Z':z})
    d=pd.DataFrame({s:zstats[k,s].Z for s in NONCYCLIC})
    for m,v in d.abs().max(axis=1).items():z_rows.append({'LOT':k,'MINUTE':m,'MAX_ABS_Z':v})
zscreen=pd.DataFrame(z_rows)


# IQR은 각 LOT·신호의 전환 제외 유효 1분 중앙값 분포로 계산한다.
# 강종을 섞거나 긴 LOT의 모든 기록을 합쳐 정상 기준을 만들지 않는다.
iqr_rows=[];iqr_intervals=[];iqr_data={};iqr_masks={}
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for sensor in LEVEL_SIGNALS:
        x=profile[k][(sensor,'median')].where(comparison_states[k].eq('안정 참고')&profile[k][(sensor,'count')].ge(54))
        usable=x.dropna();q1,q3=usable.quantile([.25,.75]);iq=q3-q1;lo,hi=q1-1.5*iq,q3+1.5*iq
        flag=x.lt(lo)|x.gt(hi) if iq>0 else pd.Series(False,index=x.index)
        iqr_data[k,sensor]=usable;iqr_masks[k,sensor]=flag
        groups=runs(flag)
        iqr_rows.append(dict(LOT=k,GRADE=r.GRADE,SENSOR=sensor,N=len(usable),MEAN=usable.mean(),MEDIAN=usable.median(),
                             Q1=q1,Q3=q3,IQR=iq,LOW=lo,HIGH=hi,OUT=int(flag.sum()),RUNS=len(groups),STATUS='탐색 가능' if iq>0 else 'IQR 0: 판정 보류'))
        for aa,bb in groups:
            y=x.loc[(x.index>=aa)&(x.index<bb)]
            iqr_intervals.append(dict(LOT=k,GRADE=r.GRADE,SENSOR=sensor,START=lot_meta.loc[k,'START']+pd.Timedelta(minutes=float(aa)),
                END=min(lot_meta.loc[k,'START']+pd.Timedelta(minutes=float(bb)),r.END),MINUTES=int(flag.loc[(flag.index>=aa)&(flag.index<bb)].sum()),
                LOW=lo,HIGH=hi,MIN=y.min(),MAX=y.max()))
iqr_summary=pd.DataFrame(iqr_rows);iqr_intervals=pd.DataFrame(iqr_intervals)
assert len(iqr_summary)==14*16


In [ ]:
# CP: 전체 LOT 중앙값을 기준으로 한 사후 탐색이다. 온라인 경보 기준으로 사용하지 않는다.
# 기존 탐색 설정이다. 공정 허용차·센서 사양에서 유도한 경계는 미확인이다.
CP_GROUP, CP_LIMIT = .02, .05
RED, AMBER, GREY = "#B11226", "#b8863b", "#8496a5"

# 음영·강조선은 구간 분류, 센서 파형은 기존 센서별 색을 사용한다.
EVENT_COLORS={'CP':'#B11226','Z':'#D17A00','전환':GREY,'감지':'#7540A8','참고':'#477B94'}
EVENT_LABELS={'CP':'CP 원본 이탈','Z':'국소 변화(Z)','전환':'승온·하강 전환','감지':'정비 감지 대조'}
EVENT_LINES={'CP':':','Z':'--','전환':'-.','감지':':','참고':':'}
assert len({EVENT_COLORS[k] for k in EVENT_LABELS})==4

cp_rows=[]
for k in lots.LOT_NO.astype(int):
    s=raw[k].set_index('MEAS_DT')['CP-Z4'].dropna()
    dev=s-s.median(); hit=dev.abs().gt(CP_GROUP)
    groups=(hit.ne(hit.shift()) | s.index.to_series().diff().dt.total_seconds().gt(5)).cumsum()
    # 최대 5초 간격의 기록은 한 사건으로 묶되 보간하지 않는다. 5초 초과 공백에서 분리한다.
    for _, block in dev[hit].groupby(groups[hit]):
        peak=block.abs().idxmax()
        if abs(block.loc[peak])>=CP_LIMIT:
            a,b,m=[(t-lot_meta.loc[k,'START']).total_seconds()/60 for t in (block.index[0],block.index[-1],peak)]
            cp_rows.append(dict(LOT=k,PEAK=peak,DEV=block.loc[peak],A=a,B=b,M=m,EXP=in_expected(k,m)))
cp_events=pd.DataFrame(cp_rows)
cp_real=cp_events.loc[~cp_events.EXP].copy()
cp_count=cp_real.groupby('LOT').size()
CP_REVIEW=[240037,240069,240117]  # 반복 2회 이상 또는 작업 완료 뒤의 변화 사례
LEVEL_REVIEW=list(LEVEL_TARGETS)
assert set(cp_count[cp_count.ge(2)].index)=={240037,240069}
assert set(CP_REVIEW)|set(LEVEL_REVIEW)=={240037,240038,240061,240069,240091,240114,240117,240121}
assert len(lots)==14 and len(cal_work)==44 and len(SIGNALS)==22

def show_table(frame):
    from IPython.display import HTML
    display(HTML(event_frame(frame).to_html(index=False,escape=True,float_format=lambda x:f'{x:.3g}')))


import textwrap
from matplotlib.transforms import Bbox

def report_page(main,sub,rows,width=20,info=None,event_key=False,cards=None):
    """제목·카드·설명·범례·파형을 독립 영역에 두어 실제 렌더링 경계를 검증한다."""
    main,sub=event_text(main),event_text(sub)
    info=[event_text(line) for line in info] if info else info
    cards=[{**c,'title':event_text(c['title']),'value':event_text(c['value']),
             'lines':[event_text(line) for line in c.get('lines',[])]} for c in cards] if cards else cards
    specs=[]
    for row in rows:
        if isinstance(row,dict):specs.append((row['panels'],row.get('height',1.4),row.get('captions')))
        else:specs.append((row,1.4,None))
    card_lines=[]
    if cards:
        for c in cards:
            lines=[]
            for line in c.get('lines',[]):lines.extend(textwrap.wrap(line,width=40) or [''])
            card_lines.append(lines)
        card_h=.85+.24*max(map(len,card_lines))
    else:card_h=0
    title_h=1.25+len(info or [])*.28+(.40 if event_key else 0)+(card_h+.18 if cards else 0)
    tick_h,gap=.55,.12
    height=title_h+sum((1.6 if captions else .58)+h+tick_h+gap for _,h,captions in specs)+.12
    fig=plt.figure(figsize=(width,height),facecolor='white')
    fig.suptitle(main,x=.07,y=1-.16/height,ha='left',va='top',fontsize=19,fontweight='bold',color='#17344d')
    fig.text(.07,1-.55/height,textwrap.fill(sub,width=125),ha='left',va='top',fontsize=10.5,color='#526d80',linespacing=1.5)
    offset=0
    if cards:
        from matplotlib.patches import FancyBboxPatch
        left,right,hgap=.075,.98,.022;w=(right-left-(len(cards)-1)*hgap)/len(cards)
        for j,(c,lines) in enumerate(zip(cards,card_lines)):
            x=left+j*(w+hgap);top=1.02;color=c.get('color','#477B94')
            fig.add_artist(FancyBboxPatch((x,1-(top+card_h)/height),w,card_h/height,
                boxstyle='round,pad=0.004,rounding_size=0.006',transform=fig.transFigure,
                facecolor=c.get('face','#f4f7fa'),edgecolor=color,linewidth=.7,zorder=0))
            for depth,text,size,bold,tint in [(top+.10,c['title'],11,True,color),(top+.37,c['value'],14,True,color)]+[
                (top+.77+q*.24,line,10.5,False,'#17344d') for q,line in enumerate(lines)]:
                fig.text(x+.012,1-depth/height,text,ha='left',va='top',fontsize=size,
                         fontweight='bold' if bold else 'normal',color=tint)
        offset=card_h+.18
    for i,line in enumerate(info or []):
        fig.text(.075,1-(1.12+offset+i*.28)/height,line,ha='left',va='top',fontsize=10.5,color='#17344d')
    if event_key:
        from matplotlib.patches import Patch
        handles=[Patch(facecolor=EVENT_COLORS[k],edgecolor=EVENT_COLORS[k],alpha=.28,label=label) for k,label in EVENT_LABELS.items()]
        fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.075,1-(1.12+offset+len(info or [])*.28)/height),
                   ncol=4,fontsize=10.5,frameon=False,handlelength=2.,columnspacing=2.,borderaxespad=0)
    cursor=title_h;cells=[];left,right,hgap=.075,.98,.065
    for panels,data_h,captions in specs:
        head_h=1.6 if captions else .58
        count=len(panels);w=(right-left-(count-1)*hgap)/count;rowcells=[]
        for col,(label,sensors) in enumerate(panels):
            x=left+col*(w+hgap)
            head=fig.add_axes([x,1-(cursor+head_h)/height,w,head_h/height]);head.axis('off');head._has_note_gutter=bool(captions)
            original_label=label
            label=event_text(label,include_lot=False)
            display_label_checks.append((original_label,label))
            head.text(0,.98,label,ha='left',va='top',fontsize=12,fontweight='bold',color='#17344d',transform=head.transAxes)
            if captions:
                head.text(0,.73,textwrap.fill(event_text(captions[col]),width=105 if count==1 else 64 if count==2 else 44),
                          ha='left',va='top',fontsize=9.5,color='#526d80',linespacing=1.35,transform=head.transAxes)
            ax=fig.add_axes([x,1-(cursor+head_h+data_h)/height,w,data_h/height])
            ax._note_gutter=.45 if captions else 0
            style(ax,'');ax.ticklabel_format(axis='y',style='plain',useOffset=False);ax.tick_params(labelsize=9,pad=3)
            ax.yaxis.set_major_locator(MaxNLocator(4));ax.xaxis.label.set_fontsize(10);ax.yaxis.label.set_fontsize(10)
            rowcells.append((ax,head,sensors))
        cells.append(rowcells);cursor+=head_h+data_h+tick_h+gap
    fig._report_cells=cells
    return fig,cells


def panel_legend(head,ax,extra=()):
    handles,labels=ax.get_legend_handles_labels()
    handles+=list(extra);labels+=[h.get_label() for h in extra]
    if handles:
        head.legend(handles,labels,loc='lower left',bbox_to_anchor=(0,.34 if getattr(head,'_has_note_gutter',False) else .1),ncol=min(4,len(handles)),
                    fontsize=9,frameon=False,handlelength=2.,columnspacing=1.6,borderaxespad=0)

def clock_ticks(ax,start=None,end=None):
    if start is not None and end is not None:ax.set_xlim(start,end if end>start else start+pd.Timedelta(seconds=1))
    locator = mdates.AutoDateLocator(minticks=3, maxticks=5)
    locator.intervald[mdates.HOURLY] = [1, 2, 3, 4, 6, 8, 12, 18, 24]
    locator.intervald[mdates.MINUTELY] = [1, 2, 3, 4, 5, 6, 10, 12, 15, 20, 30, 45, 60]
    locator.intervald[mdates.SECONDLY] = [1, 2, 3, 4, 5, 6, 10, 12, 15, 20, 30, 45, 60]
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d\n%H:%M'))
    ax.tick_params(axis='x',labelsize=9,pad=4)

def display_unit(s):
    return '온도[℃]' if s in TC else '출력[%]' if s in OP else 'CP-Z4M [무차원, 눈금×10⁻¹⁰]' if s=='CP-Z4M' else 'CP-Z4 [무차원]'

def report_signal(ax,k,s,lo=None,hi=None,fine=False,**kwargs):
    if fine or s.startswith('CP'):
        y=raw[k].set_index('MEAS_DT')[s].asfreq('s')
        if lo is not None:y=y.loc[lo:hi]
        xx=y.index
    else:
        y=profile[k][(s,'median')].where(profile[k][(s,'count')].ge(54));xx=at(k,y.index+.5)
    ax.plot(xx,y.to_numpy()*(1e10 if s=='CP-Z4M' else 1),color=sensor_color(s),lw=1.2,label=s,**kwargs)

def note_box(ann,renderer):
    return ann.get_bbox_patch().get_window_extent(renderer)


def label_hits_line(ax,box):
    # 실선·점선·수평 기준선의 실제 경로와 상자 경계를 대조한다. NaN 공백은 연결하지 않는다.
    for line in ax.lines:
        if not line.get_visible():continue
        path=line.get_path().transformed(line.get_transform())
        if path.intersects_bbox(box,filled=False):return True
    return False


def near_note(ax,x,y,label,color='#17344d'):
    """지점에서 가까운 빈 공간을 탐색한다. 글상자가 파형을 덮는 위치는 사용하지 않는다."""
    label=event_text(label,include_lot=False)
    if not np.isfinite(y):return
    fig=ax.figure
    if not hasattr(fig,'_near_notes'):fig._near_notes=[]
    ax.margins(y=.5)
    fig.canvas.draw();renderer=fig.canvas.get_renderer()
    existing=[note_box(a,renderer).expanded(1.04,1.08) for a in fig._near_notes if a.axes is ax]
    offsets=[(dx,dy) for radius in [4,6,10,16,24,34,46] for dx,dy in [(radius,radius),(-radius,radius),(radius,-radius),(-radius,-radius),(radius,0),(-radius,0),(0,radius),(0,-radius)]]
    for dx,dy in offsets:
        ann=ax.annotate(label,xy=(x,y),xytext=(dx,dy),textcoords='offset points',
            ha='right' if dx<0 else 'left' if dx>0 else 'center',va='top' if dy<0 else 'bottom',
            fontsize=9.5,color=color,bbox=dict(boxstyle='round,pad=.32',fc='white',ec=color,lw=.8),
            arrowprops=dict(arrowstyle='->',color=color,lw=.85,shrinkA=3,shrinkB=2),zorder=9)
        ann.update_positions(renderer);ann.update_bbox_position_size(renderer)
        box=note_box(ann,renderer).expanded(1.02,1.06)
        inside=ax.bbox.contains(box.x0,box.y0) and ax.bbox.contains(box.x1,box.y1)
        px,py=ax.transData.transform((mdates.date2num(x),y))
        distance=np.hypot(max(box.x0-px,px-box.x1,0),max(box.y0-py,py-box.y1,0))
        inside=inside and distance<=100*fig.dpi/150
        if inside and not label_hits_line(ax,box) and not any(box.overlaps(old) for old in existing):
            fig._near_notes.append(ann);return ann
        ann.remove()
    # 경계의 최고점은 같은 x 부근의 전용 상단 여백을 쓴다. 멀리 떨어진 모서리로 보내지 않는다.
    if getattr(ax,'_note_gutter',0):
        px,py=ax.transData.transform((mdates.date2num(x),y));dy=(ax.bbox.y1+5-py)*72/fig.dpi
        for ha in ['left','right','center']:
            ann=ax.annotate(label,xy=(x,y),xytext=(0,dy),textcoords='offset points',ha=ha,va='bottom',fontsize=9.5,
                color=color,bbox=dict(boxstyle='round,pad=.32',fc='white',ec=color,lw=.8),
                arrowprops=dict(arrowstyle='->',color=color,lw=.85,shrinkA=3,shrinkB=2),zorder=9)
            ann.update_positions(renderer);ann.update_bbox_position_size(renderer);box=note_box(ann,renderer)
            distance=np.hypot(max(box.x0-px,px-box.x1,0),max(box.y0-py,py-box.y1,0))
            if box.x0>=ax.bbox.x0 and box.x1<=ax.bbox.x1 and box.y1<=ax.bbox.y1+ax._note_gutter*fig.dpi and distance<=100*fig.dpi/150 and not label_hits_line(ax,box):
                fig._near_notes.append(ann);return ann
            ann.remove()
    raise AssertionError(f'가까운 빈 주석 위치 없음: {label}')


def validate_near_notes(fig):
    renderer=fig.canvas.get_renderer()
    for ann in getattr(fig,'_near_notes',[]):
        box=note_box(ann,renderer).expanded(1.01,1.02)
        assert not label_hits_line(ann.axes,box),f'주석이 파형을 가림: {ann.get_text()}'
        point=ann.axes.transData.transform((mdates.date2num(ann.xy[0]),ann.xy[1]))
        dx=max(box.x0-point[0],point[0]-box.x1,0);dy=max(box.y0-point[1],point[1]-box.y1,0)
        assert np.hypot(dx,dy)<=100, f'주석이 지점에서 너무 멂: {ann.get_text()}'


def save_lot_sheet(pixels,name,axes_count):
    # 저장 PNG를 읽지 않는다. 방금 코드로 그린 Figure의 캔버스를 수직으로 묶는다.
    from PIL import Image
    width=max(a.shape[1] for a in pixels);height=sum(a.shape[0] for a in pixels)
    sheet=Image.new('RGB',(width,height),'white');top=0
    for array in pixels:
        frame=Image.fromarray(array).convert('RGB');sheet.paste(frame,(0,top));top+=frame.height
    path=OUT_DIR/name;path.parent.mkdir(parents=True,exist_ok=True);sheet.save(path)
    figure_checks.append(dict(FILE=name,TITLE='모든 구간 동등 상세',N_AXES=axes_count,SAVED='저장',TEXT_OVERLAP='통과',LEGEND_BOUNDS='통과'))


def checked_emit(fig,name,save=True,record=True):
    """모든 제목/범례/눈금의 실제 렌더링 경계를 검사한다. 축 사각형만 검사하지 않는다."""
    fig.set_dpi(150);fig.canvas.draw();renderer=fig.canvas.get_renderer()
    artists=[]
    for t in fig.texts:
        if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
        leg=ax.get_legend()
        if leg is not None:artists.append((leg,leg.get_window_extent(renderer)))
        if ax.axison:
            for t in [ax.xaxis.label,ax.yaxis.label,ax.xaxis.offsetText,ax.yaxis.offsetText]:
                if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
            for axis in [ax.xaxis,ax.yaxis]:
                low,high=sorted(axis.get_view_interval())
                for tick in axis.get_major_ticks():
                    if low-1e-7<=tick.get_loc()<=high+1e-7:
                        for t in [tick.label1,tick.label2]:
                            if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
    for legend in fig.legends:
        artists.append((legend,legend.get_window_extent(renderer)))
        for body in [p[0] for row in getattr(fig,'_report_cells',[]) for p in row]:
            overlap=Bbox.intersection(legend.get_window_extent(renderer),body.bbox)
            assert overlap is None or overlap.height<=1 or overlap.width<=1, f'구간 범례가 파형을 가림: {name}'
    for a,bb in artists:
        assert bb.x0>=-.5 and bb.y0>=-.5 and bb.x1<=fig.bbox.x1+.5 and bb.y1<=fig.bbox.y1+.5, f'글자 잘림: {name} / {getattr(a,"get_text",lambda:"범례")()}'
    for i,(a,bb) in enumerate(artists):
        for other,ob in artists[i+1:]:
            inter=Bbox.intersection(bb,ob)
            if inter is not None and inter.width>1 and inter.height>1:
                raise AssertionError(f'글자/범례 겹침: {name} / {getattr(a,"get_text",lambda:"범례")()} / {getattr(other,"get_text",lambda:"범례")()}')
    # 범례가 데이터 파형 영역을 덮는지도 별도로 검사한다.
    for ax in fig.axes:
        leg=ax.get_legend()
        if leg is not None:
            for body in [p[0] for row in getattr(fig,'_report_cells',[]) for p in row]:
                overlap=Bbox.intersection(leg.get_window_extent(renderer),body.bbox)
                assert overlap is None or overlap.height<=1 or overlap.width<=1, f'범례가 파형을 가림: {name}'
    validate_near_notes(fig)
    audit_rendered_figure(fig, name)
    pixels=np.asarray(fig.canvas.buffer_rgba()).copy() if not save else None
    path=OUT_DIR/name
    if save:
        path.parent.mkdir(parents=True,exist_ok=True)
        fig.savefig(path,dpi=150,facecolor='white')
    if record:figure_checks.append({'FILE':name,'TITLE':fig._suptitle.get_text(),'N_AXES':len(fig.axes),'SAVED':'저장','TEXT_OVERLAP':'통과','LEGEND_BOUNDS':'통과'})
    fig.set_dpi(80)
    if not getattr(fig,'_silent',False):display_report_figure(fig)
    plt.close(fig)
    return pixels

emit=checked_emit


def at(k,m):return lot_meta.loc[k,'START']+pd.to_timedelta(m,unit='min')



def draw_boxes(ax,head,sensor,ids,highlight=None):
    values=[iqr_data[k,sensor].to_numpy() for k in ids]
    box=ax.boxplot(values,positions=np.arange(len(ids)),widths=.58,patch_artist=True,whis=1.5,
                   showmeans=True,meanprops=dict(marker='D',markersize=3.5,markerfacecolor='#17344d',markeredgecolor='#17344d'),
                   medianprops=dict(color='#17344d',linewidth=1.6),flierprops=dict(marker='o',markersize=2,markerfacecolor='#147f9c',markeredgecolor='none',alpha=.55))
    for patch,k in zip(box['boxes'],ids):patch.set_facecolor('#147f9c' if k==highlight else '#dce8ee');patch.set_alpha(.65)
    ax.set_xticks(range(len(ids)),[f'{k}\n{lot_meta.loc[k,"GRADE"]}' for k in ids],fontsize=8)
    ax.set_ylabel(display_unit(sensor));ax.set_xlim(-.7,len(ids)-.3)
    panel_legend(head,ax,[Line2D([0],[0],color='#17344d',label='중앙값'),Line2D([0],[0],marker='D',lw=0,color='#17344d',label='평균'),
                          Line2D([0],[0],marker='o',lw=0,color='#147f9c',label='1.5 IQR 밖 1분창')])


def display_report_figure(fig):
    label=getattr(fig,'_folded',None)
    if not label:display(fig);return
    from io import BytesIO
    from base64 import b64encode
    from html import escape
    from IPython.display import HTML
    data=BytesIO();fig.savefig(data,format='png',dpi=90,facecolor='white')
    display(HTML('<details><summary><b>'+escape(label)+'</b></summary><img style="max-width:100%;height:auto" src="data:image/png;base64,'+b64encode(data.getvalue()).decode()+'"></details>'))



def captured_details(label,captured):
    from IPython.display import HTML
    from html import escape
    import mistune
    pieces=[]
    for item in captured.outputs:
        data=item.data
        if 'text/html' in data:pieces.append(data['text/html'])
        elif 'image/png' in data:pieces.append('<img style="max-width:100%" src="data:image/png;base64,'+data['image/png']+'">')
        elif 'text/markdown' in data:pieces.append(mistune.html(data['text/markdown']))
        elif 'text/plain' in data:pieces.append('<pre>'+escape(data['text/plain'])+'</pre>')
    display(HTML('<details><summary><b>'+escape(label)+'</b></summary>'+''.join(pieces)+'</details>'))


In [ ]:
# 모든 LOT에 같은 검토 순서를 적용한다. 변화 점수를 고장 라벨로 바꾸지 않는다.
FLOW_ROWS=[('세정 CLN-IN',['CLN-IN'],None),('건조 Z1·Z2',['TC-Z1','TC-Z2'],['OP-Z1','OP-Z2']),
 ('가열 초입 Z3',['TC-Z3'],['OP-Z3']),('가열·균열 Z4–Z6',['TC-Z4','TC-Z5','TC-Z6'],['OP-Z4','OP-Z5','OP-Z6']),
 ('CP 기록',['CP-Z4'],['CP-Z4M']),('과시효 Z7·Z8',['TC-Z7','TC-Z8'],['OP-Z7','OP-Z8']),('Z9 / 냉각',['TC-Z9'],['TC-OUT1','TC-OUT2'])]
TYPE_COLOR=EVENT_COLORS

def review_events(k):
    duration=float(lot_meta.loc[k,'MINUTES']);out=[]
    ev=cp_real.loc[cp_real.LOT.eq(k)].sort_values('PEAK')
    for _,g in ev.groupby(ev.M.diff().gt(45).cumsum()):
        peak=g.loc[g.DEV.abs().idxmax()]
        out.append(dict(kind='CP',a=max(0,float(g.A.min())-8),b=min(duration,float(g.B.max())+10),point=peak.PEAK,
                        sensor='CP-Z4',deviation=float(peak.DEV),pulses=g.copy(),title=f'CP {len(g)}개 사건 · 최대 {peak.DEV:+.3f}'))
    for e in expected.loc[expected.LOT.eq(k)].itertuples():
        out.append(dict(kind='전환',a=max(0,float(e.START_MIN)-10),b=min(duration,float(e.END_MIN)+5),point=e.시작,
                        sensor='TC-Z3',title=e.종류))
    # 1분창 종료 시각 기준으로 표시한다. 원본의 순간 최고점이라는 뜻은 아니다.
    for signals in [NONCYCLIC,['CLN-IN']]:
        z=pd.DataFrame({s:zstats[k,s].Z for s in signals}).where(states[k].eq('안정 참고'),axis=0)
        ranked=z.abs().stack().sort_values(ascending=False)
        if ranked.empty or ranked.iloc[0]<Z_LIMIT:continue
        (m,sensor),strength=next(iter(ranked.items()))
        if any(e['a']<=m+1<=e['b'] for e in out):continue
        out.append(dict(kind='Z',a=max(0,float(m)-12),b=min(duration,float(m)+16),point=at(k,float(m)+1),
                        sensor=sensor,score=float(z.loc[m,sensor]),minute=int(m),title=f'{sensor} |Z|={strength:.2f} · 국소 변화'))
    for w in cal_work.loc[cal_work.DETC_DT.between(lot_meta.loc[k,'START'],lot_meta.loc[k,'END'])].itertuples():
        m=(w.DETC_DT-lot_meta.loc[k,'START']).total_seconds()/60
        if not any(e['a']<=m<=e['b'] for e in out):
            out.append(dict(kind='감지',a=max(0,m-20),b=min(duration,m+20),point=w.DETC_DT,sensor='CP-Z4',title=f'{w.MNT_TYPE} {w.WO_NO[-5:]} 감지 시각 대조'))
    out.sort(key=lambda e:e['a'])
    for i,e in enumerate(out):
        e['tag']=f'({chr(97+i)})';e['color']=TYPE_COLOR[e['kind']]
        e['transition_context']=any(e['a']<float(t.END_MIN) and e['b']>float(t.START_MIN) for t in expected.loc[expected.LOT.eq(k)].itertuples())
    return out

LOT_EVENTS={int(k):review_events(int(k)) for k in lots.LOT_NO}

def wo_distance(w,r):
    workgap=max((r.START-w.END_DT).total_seconds(),(w.STRT_DT-r.END).total_seconds(),0)
    detgap=max((r.START-w.DETC_DT).total_seconds(),(w.DETC_DT-r.END).total_seconds(),0)
    return min(workgap,detgap)

def lot_works(k):
    r=lot_meta.loc[k]
    direct=cal_work.loc[(cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START))|cal_work.DETC_DT.between(r.START,r.END)]
    if len(direct):return direct.sort_values('STRT_DT')
    distances=cal_work.apply(lambda w:wo_distance(w,r),axis=1)
    return cal_work.loc[[distances.idxmin()]]

def primary_work(k):
    ws=lot_works(k)
    # 240117은 두 실제 교집합 WO 중 CP 이전 마지막 완료 CBM을 기준으로 전후를 나눈다.
    ev=cp_real.loc[cp_real.LOT.eq(k)]
    if len(ev):
        peak=ev.loc[ev.DEV.abs().idxmax(),'PEAK']
        prev=ws.loc[ws.END_DT.le(peak)]
        if len(prev):return prev.sort_values('END_DT').iloc[-1]
    return ws.iloc[0]

def observation_phases(w,pad_hours=72):
    # 같은 LOT 안에서도 공백은 채우지 않는다. 전·후는 가장 가까운 실제 관측 1 LOT만 쓴다.
    before=temp.loc[temp.MEAS_DT.lt(w.STRT_DT)&temp.MEAS_DT.ge(w.STRT_DT-pd.Timedelta(hours=pad_hours))]
    after=temp.loc[temp.MEAS_DT.ge(w.END_DT)&temp.MEAS_DT.le(w.END_DT+pd.Timedelta(hours=pad_hours))]
    during=temp.loc[temp.MEAS_DT.ge(w.STRT_DT)&temp.MEAS_DT.lt(w.END_DT)]
    if len(before):
        last=before.MEAS_DT.max();key=int(before.loc[before.MEAS_DT.idxmax(),'LOT_NO'])
        before=before.loc[before.LOT_NO.eq(key)&before.MEAS_DT.ge(last-pd.Timedelta(minutes=45))]
    if len(after):
        first=after.MEAS_DT.min();key=int(after.loc[after.MEAS_DT.idxmin(),'LOT_NO'])
        after=after.loc[after.LOT_NO.eq(key)&after.MEAS_DT.le(first+pd.Timedelta(minutes=45))]
    return {'착수 전':before,'작업 중':during,'완료 후':after}

LOT_PHASES={k:observation_phases(primary_work(k)) for k in LOT_EVENTS}

def degree_zone(s):
    return int(s.split('-Z')[1]) if '-Z' in s and s.split('-Z')[1].isdigit() else None

def paint_event(ax,k,e,shade=True):
    if shade:ax.axvspan(at(k,e['a']),at(k,e['b']),color=e['color'],alpha=.18 if e['kind']!='전환' else .12,lw=0)
    ax.axvline(e['point'],color=e['color'],lw=1.15,ls=EVENT_LINES[e['kind']])

def clock_in_lot(ax,k):clock_ticks(ax,lot_meta.loc[k,'START'],lot_meta.loc[k,'END'])

def work_clock(ax,head,k,expanded=False):
    r=lot_meta.loc[k];ws=lot_works(k) if expanded else cal_work.loc[(cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START))|cal_work.DETC_DT.between(r.START,r.END)]
    ax.plot([r.START,r.END],[0,0],color='#147f9c',lw=6,solid_capstyle='butt',label='LOT 관측창')
    for i,w in enumerate(ws.itertuples(),1):
        color=COL[w.MNT_TYPE]
        ax.plot([w.DETC_DT,w.STRT_DT],[i,i],color=color,ls=':',lw=1.6)
        ax.plot([w.STRT_DT,w.END_DT],[i,i],color=color,lw=6,solid_capstyle='butt')
        for t,mark in [(w.DETC_DT,'D'),(w.STRT_DT,'|'),(w.END_DT,'s')]:ax.plot(t,i,marker=mark,color=EVENT_COLORS['감지'] if mark=='D' else color,ms=6)
    ax.set_yticks(list(range(len(ws)+1)),[f'LOT {k}']+[f'{w.MNT_TYPE} {w.WO_NO[-5:]}' for w in ws.itertuples()],fontsize=9)
    ax.set_ylim(-.6,len(ws)+.6)
    if expanded:
        times=[r.START,r.END]+[t for w in ws.itertuples() for t in [w.DETC_DT,w.STRT_DT,w.END_DT]]
        a,b=min(times),max(times);pad=max((b-a)*.035,pd.Timedelta(minutes=10));clock_ticks(ax,a-pad,b+pad)
    else:clock_in_lot(ax,k)
    panel_legend(head,ax,[Line2D([0],[0],marker='D',color=EVENT_COLORS['감지'],lw=0,label='감지 DETC'),Line2D([0],[0],marker='|',color=GREY,lw=0,label='착수 STRT'),Line2D([0],[0],marker='s',color=GREY,lw=0,label='완료 END')])


def add_work_remarks(ax,ws):
    # 각 작업 행의 막대 바로 위에 REMARK를 둔다. 범례·센서 파형 영역과 분리한다.
    lo,hi=ax.get_xlim();span=hi-lo
    for i,w in enumerate(ws.itertuples(),1):
        center=(mdates.date2num(w.STRT_DT)+mdates.date2num(w.END_DT))/2
        center=np.clip(center,lo+.20*span,hi-.20*span)
        ax.text(center,i+.26,remark_text(w),ha='center',va='bottom',fontsize=9,color=COL[w.MNT_TYPE],
                bbox=dict(boxstyle='round,pad=.18',fc='white',ec='none',alpha=.95))
    ax.set_ylim(-.6,len(ws)+1.)


In [ ]:
# 계산·링크용 ID와 보고서의 구간 표기를 분리한다.
def event_id(k,e):
    return f"{k}-{e['tag'][1]}"


def event_text(value,include_lot=True):
    """전체 구간 (a), 그 안의 개별 사건 (a-1). LOT 간 표에서는 LOT도 보인다."""
    if not isinstance(value,str):return value
    pattern=r'(?<![A-Za-z0-9_/#-])(\d{6})-\(?([a-z])(?:-?([1-9]\d*))?\)?(?![A-Za-z0-9_/-])'
    def replace(match):
        lot,letter,number=match.groups()
        tag='('+letter+('-'+number if number else '')+')'
        return (lot+' ' if include_lot else '')+tag
    value=re.sub(pattern,replace,value)
    # 해석·카드에 쓰던 a1 / b-1도 같은 표기로 맞춘다. 기존 괄호·앵커는 보존한다.
    return re.sub(r'(?<![A-Za-z0-9_(\-/])([abcABC])-?([1-9]\d*)(?![A-Za-z0-9_)])',
                  lambda m:'('+m[1]+'-'+m[2]+')',value)


def event_frame(frame):
    """원본 키를 바꾸지 않고 출력용 사본만 정리한다."""
    shown=frame.to_frame() if isinstance(frame,pd.Series) else frame.copy()
    return shown.map(event_text)


locator_checks=[]
overview_trigger_checks=[]
zoom_presentations=[]
locator_visual_checks=[]
zoom_alignment_checks=[]
display_label_checks=[]


In [ ]:
PLAN_CASES=[(0,240058,240061,'june'),(1,240114,240117,'november')]












# 시계열 특성: 확대창 폭과 실제 이탈 지속을 구분한다. 결측은 연결·보간하지 않는다.
cp_time_rows=[]
for k,events in LOT_EVENTS.items():
    y=raw[k].set_index('MEAS_DT')['CP-Z4'].asfreq('s');base=y.median();previous=None
    for event in events:
        if event['kind']!='CP':continue
        for pulse in event['pulses'].sort_values('PEAK').itertuples():
            start,end=at(k,pulse.A).round('s'),at(k,pulse.B).round('s')
            segment=y.loc[start:end];after=y.loc[end+pd.Timedelta(seconds=1):]
            # 복귀 확인은 사후 기술 규칙이다. 60개 연속 1초 실측이 ±0.02 안일 때의 첫 시각.
            inside=after.notna()&(after-base).abs().le(CP_GROUP)
            stable=inside.rolling(60,min_periods=60).sum().eq(60)
            confirmed=stable.index[stable][0] if stable.any() else pd.NaT
            back=confirmed-pd.Timedelta(seconds=59) if pd.notna(confirmed) else pd.NaT
            cp_time_rows.append(dict(LOT=k,EVENT_ID=event_id(k,event),START=start,END=end,PEAK=pulse.PEAK,DEV=pulse.DEV,
                SPAN_SEC=(end-start).total_seconds(),POINTS=int(segment.notna().sum()),
                PEAK_GAP_MIN=(pulse.PEAK-previous).total_seconds()/60 if previous is not None else np.nan,
                RETURN_START=back,RETURN_CONFIRMED=confirmed,RETURN_DELAY_SEC=(confirmed-pulse.PEAK).total_seconds() if pd.notna(confirmed) else np.nan))
            previous=pulse.PEAK
cp_time_features=pd.DataFrame(cp_time_rows).sort_values(['LOT','PEAK']).reset_index(drop=True)
interval_time_rows=[]
for k,events in LOT_EVENTS.items():
    for event in events:
        duration=np.nan;description='지속시간 정의 없음';recovery='별도 판단'
        if event['kind']=='CP':
            features=cp_time_features.loc[cp_time_features.EVENT_ID.eq(event_id(k,event))]
            duration=features.SPAN_SEC.max();description=f"CP {len(features)}사건 / 최장 원본 첫·끝 간격 {duration:g}초"
            recovery=f"60개 연속 실측 복귀 확인 {features.RETURN_CONFIRMED.notna().sum()}/{len(features)}사건"
        elif event['kind']=='Z':
            z=zstats[k,event['sensor']].Z;hit=z.abs().ge(Z_LIMIT)
            run=next((pair for pair in runs(hit) if pair[0]<=event['minute']<pair[1]),None)
            duration=float(hit.loc[(hit.index>=run[0])&(hit.index<run[1])].sum()) if run else np.nan
            description=f"선택 신호의 연속 |Z|≥3 {duration:g}개 1분창"
            good=z.notna()&z.abs().lt(Z_LIMIT);good.loc[good.index<=event['minute']]=False
            ok=good.rolling(3,min_periods=3).sum().eq(3)
            recovery=f"3개 연속 |Z|<3 확인 {at(k,int(ok.index[ok][0])+1):%m/%d %H:%M}" if ok.any() else '3개 연속 |Z|<3 미확인'
        elif event['kind']=='전환':
            transition=expected.loc[expected.LOT.eq(k)].iloc[0];duration=float(transition.END_MIN-transition.START_MIN)
            description=f'전환 참고 구간 {duration:g}분';recovery='운전 모드·SV 추가 확인'
        interval_time_rows.append(dict(LOT=k,ID=event_id(k,event),TYPE=event['kind'],SENSOR=event['sensor'],POINT=event['point'],
            WINDOW_START=at(k,event['a']),WINDOW_END=at(k,event['b']),WINDOW_MIN=event['b']-event['a'],DURATION_DESCRIPTION=description,RECOVERY=recovery))
interval_time_features=pd.DataFrame(interval_time_rows)
assert len(cp_time_features)==len(cp_real)==14
assert len(interval_time_features)==sum(len(events) for events in LOT_EVENTS.values())






report_case_checks=[]


# 240069: 피크별 반응과 반례를 직접 비교한다. 상관계수·고장 분류 모델은 만들지 않는다.



















# CP 피크별 Z4~Z6의 같은 시각 반응과 근접 최대편차를 함께 확인한다.

In [ ]:
# 모델 학습 없이 원자료·기존 Z 결과를 연결하는 LOT별 검토 이력
from html import escape as review_escape
from IPython.display import HTML as ReviewHTML
from io import BytesIO as ReviewBytesIO
import base64 as review_base64

REVIEW_MIN_RUN, REVIEW_MIN_REPEAT, REVIEW_REPEAT_WINDOW = 3, 3, 30
REVIEW_SIGNALS = NONCYCLIC + OP
REVIEW_COLORS = {'정상 범위 참고':'#148578', '국소 변화 후보':'#E4B13F',
                 '지속 이상 후보':'#D45D18', '판정 보류':'#ABB7C5', '단발 스파이크':'#B12261'}
review_exports, review_layout_checks = {}, []

def review_ts(k, minute):
    return lot_meta.loc[k,'START'] + pd.Timedelta(minutes=float(minute))

def review_clock(t):
    return pd.Timestamp(t).strftime('%m/%d %H:%M:%S') if pd.notna(t) else '미확인'

def review_recovery(mask):
    ok = mask.fillna(False).rolling(3,min_periods=3).sum().eq(3)
    return int(ok.index[ok][0]) if ok.any() else None

def review_work(k, start, end):
    # 시간 관계만 연결한다. 가장 가까운 WO를 해당 사건의 원인으로 붙이지 않는다.
    inside = cal_work.loc[cal_work.STRT_DT.le(end)&cal_work.END_DT.ge(start)]
    detected = cal_work.loc[cal_work.DETC_DT.between(start,end)]
    bits = [f'{w.WO_NO} 작업 교집합 · {w.MNT_TYPE} · {w.REMARK if pd.notna(w.REMARK) else '작업 내용 미기재'}' for w in inside.itertuples()]
    bits += [f'{w.WO_NO} 감지 교집합 · {w.REMARK if pd.notna(w.REMARK) else '작업 내용 미기재'}' for w in detected.itertuples()]
    distances = []
    for w in cal_work.itertuples():
        for name,t in [('감지',w.DETC_DT),('착수',w.STRT_DT),('완료',w.END_DT)]:
            distances.append((abs((start-t).total_seconds()),w.WO_NO,name,t,w.REMARK if pd.notna(w.REMARK) else '작업 내용 미기재'))
    _,wo,kind,t,remark = min(distances,key=lambda v:v[0])
    signed=(start-t).total_seconds()/60
    bits.append(f'가장 가까운 기록: {wo} {kind} {review_clock(t)} → 사건 시작 {signed:+.2f}분 · {remark}')
    plan = context.set_index('LOT').loc[k,'PLAN_RELATION']
    if plan: bits.append(plan+' (실제 모드 확인 필요)')
    return ' / '.join(bits)

def review_build():
    event_rows, minute_frames, raw_rows, summary_rows = [], [], [], []
    for lr in lots.itertuples():
        k=int(lr.LOT_NO); p=profile[k]
        zm=pd.DataFrame({s:zstats[k,s].Z for s in REVIEW_SIGNALS})
        minute_code=pd.DataFrame(0,index=p.index,columns=REVIEW_SIGNALS,dtype=int)
        minute_code[zm.isna()]=3
        persistent_mask=pd.DataFrame(False,index=p.index,columns=REVIEW_SIGNALS)
        for sensor in REVIEW_SIGNALS:
            d=zstats[k,sensor]
            h=d.X.shift(1).rolling(Z_WINDOW,min_periods=1).count().fillna(0)
            sd_raw=d.X.shift(1).rolling(Z_WINDOW,min_periods=Z_MIN).std()
            floor=.5 if sensor in TC else 2.
            starts=[]
            for a,b in runs(d.Z.abs().ge(Z_LIMIT)):
                starts.append(a)
                recent=[v for v in starts if a-REVIEW_REPEAT_WINDOW<=v<=a]
                n=b-a; persistent=n>=REVIEW_MIN_RUN or len(recent)>=REVIEW_MIN_REPEAT
                minute_code.loc[a:b-1,sensor]=2 if persistent else 1
                persistent_mask.loc[a:b-1,sensor]=persistent
                seg=d.loc[a:b-1]; peak=int(seg.Z.abs().idxmax()); row=d.loc[peak]
                signs=np.sign(seg.Z.to_numpy()); same=best=1
                for j in range(1,len(signs)):
                    same=same+1 if signs[j]==signs[j-1] else 1
                    best=max(best,same)
                # 같은 방향의 지속과 상·하방 왕복을 구분한다.
                pattern='동일 방향 지속' if best>=3 else '상·하방 반복' if n>=3 else '국소 변화 반복' if len(recent)>=3 else '짧은 국소 변화'
                future=d.loc[b:]
                moving=review_recovery(future.Z.notna()&future.Z.abs().lt(Z_LIMIT))
                fixed=review_recovery(future.X.notna()&(future.X-d.loc[a,'MEDIAN']).abs().lt(3*d.loc[a,'SD']))
                zreturn=review_ts(k,moving+1) if moving is not None else pd.NaT
                fixedreturn=review_ts(k,fixed+1) if fixed is not None else pd.NaT
                st=states[k].loc[a:b-1]; trans=bool(st.ne('안정 참고').any())
                start=review_ts(k,a); end=min(review_ts(k,b),lr.END+pd.Timedelta(seconds=1))
                rid=f'{k}-Z-{sensor}-{a:04d}'
                change=float(row.X-row.MEDIAN)
                why=f'연속 {n}개 1분창 / 같은 방향 최장 {best}개 / 직전 30분 시작 {len(recent)}사건'
                observation=(f'최대 |Z|={abs(row.Z):.2f}; {sensor} 중앙값 {row.X:.4g}, '
                             f'과거 중심 {row.MEDIAN:.4g}, 차이 {change:+.4g}; SD {row.SD:.4g} (하한 적용: {"예" if sd_raw.loc[peak]<floor else "아니오"}); '
                             f'피크창 실측 {int(p.loc[peak,(sensor,"count")])}/60개, 과거 {int(h.loc[peak])}/30창')
                recovery=(f'이동 Z 기준 3창 복귀 확인 {review_clock(zreturn)} / '
                          f'사건 시작 고정 기준 ±3SD 3창 복귀 확인 {review_clock(fixedreturn)}')
                if pd.notna(zreturn) and (pd.isna(fixedreturn) or fixedreturn>zreturn):
                    recovery+=' · Z 감소가 원수준 복귀보다 먼저여서 기준선 적응 가능'
                alternative='승온·종료 전환 형태와 겹침; 모드·SV 기록으로 의도된 전환인지 확인' if trans else '짧은 제어 응답·설정 변경·센서 문제 등 대안 원인은 미확정'
                action=('같은 센서·동일 조건의 후속 LOT에서 재발, 고정 기준 복귀, 대응 TC/OP·SV를 확인'
                        if persistent else '원본 전후 값과 다음 사건까지 간격을 보존하고 지속·반복 기준 도달 시 재검토')
                event_rows.append(dict(ID=rid,LOT=k,GRADE=lr.GRADE,SENSOR=sensor,KIND='Z',
                    START=start,END=end,PEAK=review_ts(k,peak+.5),A=a,B=b,N_BINS=n,SAME_BINS=best,
                    REPEATS=len(recent),PREV_GAP_MIN=(a-starts[-2]) if len(starts)>1 else np.nan,
                    MAX_ABS_Z=abs(float(row.Z)),VALUE=float(row.X),BASELINE=float(row.MEDIAN),DELTA=change,
                    SD=float(row.SD),FLOOR_USED=bool(sd_raw.loc[peak]<floor),N_CURRENT=int(p.loc[peak,(sensor,'count')]),
                    N_HISTORY=int(h.loc[peak]),PATTERN=pattern,OBS_CLASS='지속 이상 후보' if persistent else '국소 변화 후보',
                    PRIORITY=('높음' if n>=3 and len(recent)>=3 else '중간' if persistent else '기록 추적'),
                    TRANSITION=trans,Z_RETURN=zreturn,FIXED_RETURN=fixedreturn,POINTS=np.nan,SPAN_SEC=np.nan,
                    WHY=why,FACT=observation,RECOVERY=recovery,ALTERNATIVE=alternative,
                    WORK=review_work(k,start,end-pd.Timedelta(microseconds=1)),
                    ASSESSMENT='판정 보류',HOLD_REASON='SV·실제 모드·관련 점검 결과와 신호/부품 연결 미확인',
                    ACTION=action))
        # 원본 CP 전체 이탈: 전환 후보도 제외하지 않고 남긴다.
        cp=raw[k].set_index('MEAS_DT')['CP-Z4'].asfreq('s')
        cpbase=float(cp.median())
        selected=cp_events.loc[cp_events.LOT.eq(k)].sort_values('PEAK')
        previous=None
        for j,ev in enumerate(selected.itertuples(),1):
            start=review_ts(k,ev.A).round('s'); last=review_ts(k,ev.B).round('s')
            block=cp.loc[start:last]; npoint=int(block.notna().sum()); span=(last-start).total_seconds()
            prev_t=start-pd.Timedelta(seconds=1); next_t=last+pd.Timedelta(seconds=1)
            prev_v=cp.get(prev_t,np.nan); next_v=cp.get(next_t,np.nan)
            isolated=npoint==1 and span==0 and pd.notna(prev_v) and pd.notna(next_v) and abs(prev_v-cpbase)<=CP_GROUP and abs(next_v-cpbase)<=CP_GROUP
            stop=review_ts(k,selected.iloc[j].A).round('s') if j<len(selected) else lr.END+pd.Timedelta(seconds=1)
            after=cp.loc[(cp.index>last)&(cp.index<stop)]
            back=(after.notna()&(after-cpbase).abs().le(CP_GROUP)).rolling(60,min_periods=60).sum().eq(60)
            confirmed=back.index[back][0] if back.any() else pd.NaT
            pulse=pd.notna(prev_v) and abs(prev_v-cpbase)<=CP_GROUP and pd.notna(confirmed)
            recent=int(selected.PEAK.between(ev.PEAK-pd.Timedelta(minutes=30),ev.PEAK).sum())
            rid=f'{k}-CP-{j:02d}'; typ='단발 스파이크' if isolated else '판정 보류'
            shape='1초 단발 스파이크' if isolated else '복귀 펄스 후보' if pulse else '복귀 미확인 원본 이탈'
            why=f'실측 {npoint}점 / 첫·끝 간격 {span:g}초 / 직전 30분 피크 {recent}사건'
            fact=f'CP 원값 {cp.loc[ev.PEAK]:.6g}, LOT 기준 {cpbase:.6g}, 편차 {ev.DEV:+.6g} (탐색 경계 ±0.05)'
            recovery=f'직전 {review_clock(prev_t)}={prev_v:.6g} / 직후 {review_clock(next_t)}={next_v:.6g}; 이후 60개 연속 실측 ±0.02 복귀 확인 {review_clock(confirmed)}'
            event_rows.append(dict(ID=rid,LOT=k,GRADE=lr.GRADE,SENSOR='CP-Z4',KIND='CP',
                START=start,END=last+pd.Timedelta(seconds=1),PEAK=ev.PEAK,A=ev.A,B=ev.B,N_BINS=np.nan,SAME_BINS=np.nan,
                REPEATS=recent,PREV_GAP_MIN=(ev.PEAK-previous).total_seconds()/60 if previous is not None else np.nan,
                MAX_ABS_Z=np.nan,VALUE=float(cp.loc[ev.PEAK]),BASELINE=cpbase,DELTA=float(ev.DEV),
                SD=np.nan,FLOOR_USED=False,N_CURRENT=npoint,N_HISTORY=np.nan,
                PATTERN=shape,OBS_CLASS=typ,PRIORITY='높음' if recent>=3 else '중간',TRANSITION=bool(ev.EXP),
                Z_RETURN=pd.NaT,FIXED_RETURN=confirmed,POINTS=npoint,SPAN_SEC=span,
                WHY=why,FACT=fact,RECOVERY=recovery,
                ALTERNATIVE='원값이 복귀해도 계측 오류·정상 제어·공정 변화 중 무엇인지는 미확정',
                WORK=review_work(k,start,last),ASSESSMENT='판정 보류',
                HOLD_REASON='CP 물리량 정의·채널 연결 미확인 (단위는 무차원)'+(' / 관측 공백 있음' if block.isna().any() else '')+(' / 다점 펄스: 단발 폭 기준 미확정' if not isolated else ''),
                ACTION='사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조'))
            original=raw[k].loc[raw[k].MEAS_DT.between(prev_t,next_t)]
            for idx,rr in original.iterrows():
                raw_rows.append(dict(ID=rid,LOT=k,SENSOR='CP-Z4',SOURCE_ROW=int(idx),TIME=rr.MEAS_DT,
                    VALUE=rr['CP-Z4'],BASELINE=cpbase,DELTA=rr['CP-Z4']-cpbase,
                    ROLE='직전' if rr.MEAS_DT<start else '직후' if rr.MEAS_DT>last else '피크' if rr.MEAS_DT==ev.PEAK else '사건 원본'))
            previous=ev.PEAK
        zvalid=zm.notna().all(axis=1); anyhit=zm.abs().ge(3).any(axis=1)
        ref=zvalid&~anyhit; incomplete=~zvalid
        held_short=(zvalid&anyhit&~persistent_mask.any(axis=1))
        minute_frames.append(pd.DataFrame(dict(LOT=k,MINUTE=p.index,
            REFERENCE=ref.to_numpy(),PERSISTENT=persistent_mask.any(axis=1).to_numpy(),
            INCOMPLETE=incomplete.to_numpy(),SHORT_ONLY=held_short.to_numpy())))
        rows=[e for e in event_rows if e['LOT']==k]
        summary_rows.append(dict(LOT=k,GRADE=lr.GRADE,TOTAL_MIN=len(p),
            REF_MIN=int(ref.sum()),INCOMPLETE_MIN=int(incomplete.sum()),SHORT_MIN=int(held_short.sum()),
            PERSIST_MIN=int(persistent_mask.any(axis=1).sum()),PERSIST_EVENTS=sum(e['OBS_CLASS']=='지속 이상 후보' for e in rows),
            SINGLE_CP=sum(e['OBS_CLASS']=='단발 스파이크' for e in rows),
            PULSE_CP=sum(e['PATTERN']=='복귀 펄스 후보' for e in rows),
            CP_EVENTS=len(selected),ALL_EVENTS=len(rows),TRANSITION_EVENTS=sum(e['TRANSITION'] for e in rows)))
    return pd.DataFrame(event_rows),pd.concat(minute_frames,ignore_index=True),pd.DataFrame(raw_rows),pd.DataFrame(summary_rows)

review_events,review_minutes,review_raw_context,review_summary=review_build()
assert review_events.ID.is_unique
assert review_events.loc[review_events.KIND.eq('CP')].shape[0]==len(cp_events)
assert review_events.loc[review_events.KIND.eq('Z'),'N_BINS'].sum()==sum(d.Z.abs().ge(3).sum() for d in zstats.values())
assert set(review_summary.LOT)==set(lots.LOT_NO)
assert all(np.array_equal(review_raw_context.loc[review_raw_context.LOT.eq(k),'VALUE'].to_numpy(),
            temp.loc[review_raw_context.loc[review_raw_context.LOT.eq(k),'SOURCE_ROW'],'CP-Z4'].to_numpy(),equal_nan=True)
           for k in lots.LOT_NO.astype(int))

def review_pair_facts(k,sensor,a,b,kind,peak):
    if kind=='CP':
        g=raw[k];before=g.loc[g.MEAS_DT.between(peak-pd.Timedelta(minutes=10),peak-pd.Timedelta(minutes=5))]
        during=g.loc[g.MEAS_DT.between(peak-pd.Timedelta(seconds=30),peak+pd.Timedelta(seconds=30))]
        facts=[]
        for ch in ['TC-Z3','OP-Z3','CP-Z4M']:
            pre=before[ch].dropna();now=during[ch].dropna()
            if len(pre)>=190 and len(now)>=40:
                facts.append(f'{ch} 피크±30초 중앙값−전10~5분 중앙값 {now.median()-pre.median():+.4g} ({len(pre)}/{len(now)}점)')
            else:facts.append(f'{ch} 전후 실측 부족 ({len(pre)}/{len(now)}점)')
        return ' / '.join(facts)+'; 같은 시각 동반 관측이며 원인 확정 아님'
    zone=re.search(r'(TC|OP)-Z([1-8])$',sensor)
    if not zone:return '대응 OP 채널 정의 없음; 다른 신호의 동시 반응은 전체 공정 그림에서 확인'
    ch=('OP' if zone.group(1)=='TC' else 'TC')+'-Z'+zone.group(2)
    x=zstats[k,ch].X;before=x.loc[max(0,int(a)-10):int(a)-1].dropna();now=x.loc[int(a):int(b)-1].dropna()
    if len(before)<5 or not len(now):return f'{ch} 유효 전후 창 부족 ({len(before)}/{len(now)}창); 동반 반응 보류'
    return f'{ch} 사건창 중앙값−직전10분 중앙값 {now.median()-before.median():+.4g} ({len(before)}/{len(now)}유효창); 대응 존의 동시 관측이며 인과 미확정'

review_events['PAIRED_FACT']=[
    review_pair_facts(e.LOT,e.SENSOR,e.A,e.B,e.KIND,e.PEAK) for e in review_events.itertuples()
]

# 원본 이탈도 해당 1분 집계 결과와 연결한다. CP에 기존 Z를 새로 만들어 붙이지 않는다.
review_events['MINUTE']=[int((e.PEAK-lot_meta.loc[e.LOT,'START']).total_seconds()//60) for e in review_events.itertuples()]
review_minute_lookup=review_minutes.set_index(['LOT','MINUTE'])
review_events['MINUTE_REF17']=[bool(review_minute_lookup.loc[(e.LOT,e.MINUTE),'REFERENCE']) for e in review_events.itertuples()]
review_events['Z_AT_SAME_MINUTE']=[
    ' / '.join(f'{s} {zstats[e.LOT,s].loc[e.MINUTE,"Z"]:+.2f}' for s in REVIEW_SIGNALS
               if pd.notna(zstats[e.LOT,s].loc[e.MINUTE,'Z']) and abs(zstats[e.LOT,s].loc[e.MINUTE,'Z'])>=3) or '17신호에서 |Z|≥3 미검출 (평가 불가 채널은 별도)'
    for e in review_events.itertuples()
]




In [ ]:
# Z 채널 전체 구간: 같은 코드로 노트북에 표시하고 figures에 PNG 저장
def review_unit(sensor):
    if sensor in TC:
        return '온도 [℃]'
    if sensor in OP:
        return '제어출력 [%]'
    return sensor + ' [무차원]'

def review_save(fig, name):
    path = (OUT_DIR / name).resolve()
    assert path.is_relative_to(OUT_DIR.resolve()) and path.suffix == '.png'
    fig.set_dpi(150)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    legends = list(fig.legends) + [a.get_legend() for a in fig.axes if a.get_legend() is not None]
    texts = list(fig.texts)
    for ax in fig.axes:
        texts += list(ax.texts)
        if not ax.axison:
            continue
        texts += [ax.title, ax._left_title, ax._right_title]
        for axis in [ax.xaxis, ax.yaxis]:
            if axis.get_visible():
                texts += [axis.label, axis.offsetText]
                low, high = sorted(axis.get_view_interval())
                for tick in axis.get_major_ticks():
                    if low <= tick.get_loc() <= high:
                        texts += [tick.label1, tick.label2]
    texts = [t for t in texts if t.get_visible() and t.get_text()]
    for legend in legends:
        box = legend.get_window_extent(renderer)
        assert box.x0 >= 0 and box.y0 >= 0 and (box.x1 <= fig.bbox.x1) and (box.y1 <= fig.bbox.y1), f'범례 잘림: {name}'
        assert not any((box.overlaps(ax.bbox) for ax in fig.axes if ax.axison)), f'범례가 파형 가림: {name}'
        assert not any((box.overlaps(t.get_window_extent(renderer)) for t in texts)), f'범례가 글 가림: {name}'
    for i, legend in enumerate(legends):
        assert not any((legend.get_window_extent(renderer).overlaps(other.get_window_extent(renderer)) for other in legends[i + 1:])), name
    for text in texts:
        box = text.get_window_extent(renderer)
        assert box.x0 >= -0.5 and box.y0 >= -0.5 and (box.x1 <= fig.bbox.x1 + 0.5) and (box.y1 <= fig.bbox.y1 + 0.5), f'글자 잘림: {name}: {text.get_text()[:50]}'
    audit_rendered_figure(fig, name)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.18, facecolor='white')
    review_exports[name] = dict(FILE=str(path), DPI=150, LEGEND='파형·글 겹침 없음', BYTES=path.stat().st_size)
    review_layout_checks[:] = [row for row in review_layout_checks if row['NAME'] != name]
    review_layout_checks.append(dict(NAME=name, LEGENDS=len(legends), RESULT='통과'))
    data = ReviewBytesIO()
    fig.savefig(data, format='png', dpi=85, bbox_inches='tight', pad_inches=0.12)
    display(ReviewHTML('<img alt="' + review_escape(name) + '" style="max-width:100%;height:auto" src="data:image/png;base64,' + review_base64.b64encode(data.getvalue()).decode() + '">'))
    plt.close(fig)
    import gc
    gc.collect()

def review_axis(ax, xlabel='LOT 시작 후 경과시간 [분]'):
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.18)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.yaxis.set_major_locator(MaxNLocator(5))

REVIEW_Z_GROUPS = [('01_세정건조', '세정·건조', ['CLN-IN', 'TC-Z1', 'OP-Z1', 'TC-Z2', 'OP-Z2']), ('02_가열균열', '가열·균열', [s for z in range(3, 7) for s in (f'TC-Z{z}', f'OP-Z{z}')]), ('03_과시효', '과시효', ['TC-Z7', 'OP-Z7', 'TC-Z8', 'OP-Z8'])]

review_z_coverage = {}

def review_zscore_group_plot(k, group_key):
    if group_key == '02_가열균열':
        review_heating_cp_plot(k)
        return
    drawn_sensors = set()
    drawn_events = set()
    meta = lot_meta.loc[k]
    p = profile[k]
    xx = p.index.to_numpy() + 0.5
    for key, label, sensors in [group for group in REVIEW_Z_GROUPS if group[0] == group_key]:
        height = 2.5 + 2.2 * len(sensors)
        fig = plt.figure(figsize=(22, height), facecolor='white')
        fig.text(0.07, 1 - 0.18 / height, f'LOT {k} · {meta.GRADE} | {label} 전체 채널·전체 후보', va='top', fontsize=19, weight='bold')
        fig.text(0.07, 1 - 0.6 / height, '왼쪽: 1분 내 원본 최저~최고와 유효 중앙값. 오른쪽: 해당 채널의 Z. 대표 피크 순위로 채널을 생략하지 않음.', va='top', fontsize=10)
        handles = [Patch(color='#CBD2D9', alpha=0.65, label='1분 내 원본 최저~최고'), Line2D([], [], color='#176B94', label='TC 유효 1분 중앙값'), Line2D([], [], color='#176B94', ls='--', label='OP 유효 1분 중앙값'), Patch(color='#B8D9E5', alpha=0.5, label='이동 중심 ±3SD (Z 채널)'), Patch(color='#E4B13F', alpha=0.4, label='짧은 국소 후보'), Patch(color='#D45D18', alpha=0.35, label='지속·반복 후보'), Patch(color='#ABB7C5', alpha=0.3, label='관측/기준 부족'), Line2D([], [], color=REVIEW_WO_COLOR, ls='-.', label='WO 감지·작업 시각')]
        fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.065, 1 - 0.98 / height), frameon=False, ncol=4, fontsize=9, columnspacing=1.1)
        gs = fig.add_gridspec(len(sensors), 2, left=0.075, right=0.94, top=1 - 1.75 / height, bottom=0.85 / height, hspace=0.72, wspace=0.25, width_ratios=[1.45, 1])
        for row, sensor in enumerate(sensors):
            drawn_sensors.add(sensor)
            left = fig.add_subplot(gs[row, 0])
            right = fig.add_subplot(gs[row, 1])
            review_axis(left)
            review_axis(right)
            counts = p[sensor, 'count'].fillna(0)
            x = p[sensor, 'median'].where(counts.ge(54))
            left.fill_between(xx, p[sensor, 'min'].to_numpy(), p[sensor, 'max'].to_numpy(), color='#CBD2D9', alpha=0.8)
            left.plot(xx, x, color='#176B94', lw=1.0, ls='--' if sensor in OP else '-')
            ev = review_events.loc[review_events.LOT.eq(k) & review_events.SENSOR.eq(sensor)]
            drawn_events.update(ev.ID)
            left.set_title(f'{sensor} · {len(ev)}개 사건 · 전 구간 표시', loc='left', fontsize=10, pad=8)
            left.set_ylabel(review_unit(sensor), fontsize=9)
            left.set_xlim(0, len(p))
            d = zstats[k, sensor]
            left.fill_between(xx, (d.MEDIAN - 3 * d.SD).to_numpy(), (d.MEDIAN + 3 * d.SD).to_numpy(), color='#B8D9E5', alpha=0.4)
            right.plot(xx, d.Z, color='#176B94', lw=1)
            for v in [-3, 3]:
                right.axhline(v, color='#B24433', ls='--', lw=0.8)
            for a, b in runs(d.Z.isna()):
                left.axvspan(a, b, color='#ABB7C5', alpha=0.2)
                right.axvspan(a, b, color='#ABB7C5', alpha=0.2)
            for e in ev.itertuples():
                color = '#D45D18' if e.OBS_CLASS == '지속 이상 후보' else '#E4B13F'
                left.axvspan(e.A, e.B, color=color, alpha=0.28)
                right.axvspan(e.A, e.B, color=color, alpha=0.28)
            right.set_title(f'|Z|≥3 {int(d.Z.abs().ge(3).sum())}창 / 유효 {int(d.Z.notna().sum())}창 / 계산 보류 {int(d.Z.isna().sum())}창', loc='left', fontsize=10, pad=8)
            right.set_ylabel('Z · 설비 판정 아님', fontsize=9)
            right.set_xlim(0, len(p))
            review_mark_maintenance(left, k)
            review_mark_maintenance(right, k)
        fig.text(0.07, 0.18 / height, '각 센서의 전체 후보를 표시했다. 초과가 없는 채널도 설비 정상 확인을 뜻하지 않는다. 원본 최저~최고 띠는 통계적 정상 범위가 아니다.', fontsize=10, va='bottom')
        review_save(fig, f'06_EDA_LOT/{k}_{meta.GRADE}_{key}_전체채널.png')
    assert drawn_sensors == set(next((g[2] for g in REVIEW_Z_GROUPS if g[0] == group_key)))
    assert drawn_events == set(review_events.loc[review_events.LOT.eq(k) & review_events.KIND.eq('Z') & review_events.SENSOR.isin(drawn_sensors), 'ID'])
    review_z_coverage[k, group_key] = dict(SENSORS=drawn_sensors, EVENT_IDS=drawn_events)
    review_process_coverage[k, group_key] = set(drawn_sensors)

def zscore_md_table(headers,rows):
    def clean(value):
        return str(value).replace('|','\\|').replace('\n','; ')
    return '\n'.join(['| '+' | '.join(map(clean,headers))+' |',
                      '| '+' | '.join(['---']*len(headers))+' |']+
                     ['| '+' | '.join(map(clean,row))+' |' for row in rows])

def zscore_time(value):
    return pd.Timestamp(value).strftime('%m/%d %H:%M:%S') if pd.notna(value) else '미확인'

def zscore_reason(event):
    if event.TRANSITION:
        return '승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인'
    if pd.notna(event.Z_RETURN) and pd.isna(event.FIXED_RETURN):
        return '이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적'
    if pd.notna(event.Z_RETURN) and pd.notna(event.FIXED_RETURN) and event.FIXED_RETURN > event.Z_RETURN:
        gap=(event.FIXED_RETURN-event.Z_RETURN).total_seconds()/60
        return f'고정 범위 복귀가 이동 Z 복귀보다 {gap:g}분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음'
    if event.N_BINS < REVIEW_MIN_RUN and event.REPEATS < REVIEW_MIN_REPEAT and pd.notna(event.FIXED_RETURN):
        return f'{int(event.N_BINS)}개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인'
    if event.SAME_BINS >= REVIEW_MIN_RUN:
        return f'같은 방향의 초과가 최장 {int(event.SAME_BINS)}창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조'
    if event.N_BINS >= REVIEW_MIN_RUN:
        return f'상·하방 초과가 {int(event.N_BINS)}창 이어졌다. 한 방향 이탈과 반복 진동을 구분해 확인'
    if event.REPEATS >= REVIEW_MIN_REPEAT:
        return f'짧아도 최근 30분에 {int(event.REPEATS)}회 시작했다. 재발 간격과 동일 운전 조건을 추적'
    return '짧은 국소 변화지만 고정 범위 복귀는 미확인. 원본의 폭·전후 값을 보존하며 1초 스파이크로 단정하지 않음'


def reading_full_zscore_lot_intro(k):
    meta = lot_meta.loc[k]
    su = review_summary.set_index('LOT').loc[k]
    return f'#### Z-score로 확인한 구간 · LOT {k}\n\n- **관측 범위:** {zscore_time(meta.START)}–{zscore_time(meta.END)}.\n- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 {int(su.REF_MIN)}개. 설비 정상 판정은 아니다.\n- **지속·반복 후보:** {int(su.PERSIST_EVENTS)}개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.\n- **관측·기준 부족:** 17신호 중 Z가 없는 창 {int(su.INCOMPLETE_MIN)}개. 회색 구간은 정상으로 채우지 않았다.\n- **원본 스파이크 확인:** CP 원본 이탈 {int(su.CP_EVENTS)}건 중 확인된 1초 단발 {int(su.SINGLE_CP)}건, 복귀 펄스 후보 {int(su.PULSE_CP)}건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.\n\n**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.'

def _zscore_group_base(k, group_key):
    _, label, sensors = next((g for g in REVIEW_Z_GROUPS if g[0] == group_key))
    ev = review_events.loc[review_events.LOT.eq(k) & review_events.KIND.eq('Z') & review_events.SENSOR.isin(sensors)].sort_values(['START', 'SENSOR'])
    lines = [f'#### {label} · 구간별 판단 근거', '']
    nohit = [s for s in sensors if not zstats[k, s].Z.abs().ge(3).any()]
    if nohit:
        facts = []
        for s in nohit:
            z = zstats[k, s].Z
            facts.append(f'{s}: 유효 {int(z.notna().sum())}창, 최대 절대 Z={z.abs().max():.2f}')
        lines += ['**정상 범위 참고의 근거:** ' + ' / '.join(facts) + '. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.', '']
    if len(ev):
        rows = []
        for e in ev.itertuples():
            u = '℃' if e.SENSOR in TC else '%'
            du = '℃' if e.SENSOR in TC else '%p'
            observed = f'최대 절대 Z={e.MAX_ABS_Z:.2f} ({zscore_time(e.PEAK)}, 창 중심); 1분 중앙값 {e.VALUE:.3f}{u}, 과거 중심 {e.BASELINE:.3f}{u}, 차이 {e.DELTA:+.3f}{du}; SD={e.SD:.3f}' + (' (하한 적용)' if e.FLOOR_USED else '') + f'; 실측 {e.N_CURRENT}/60개·과거 {int(e.N_HISTORY)}/30창'
            classification = '지속·반복 후보' if e.OBS_CLASS == '지속 이상 후보' else '짧은 국소 후보·보류'
            why = f'{classification}: 연속 {int(e.N_BINS)}창, 같은 방향 최장 {int(e.SAME_BINS)}창, 최근 30분 내 시작 {int(e.REPEATS)}회(해당 사건 포함)'
            recovery = f'이동 Z 3창 복귀 {zscore_time(e.Z_RETURN)} / 시작 고정 기준 3창 복귀 {zscore_time(e.FIXED_RETURN)}'
            interval = f'{e.SENSOR} · {zscore_time(e.START)}–{zscore_time(e.END)} (끝 제외); {e.ID}'
            rows.append([interval, observed, why, recovery, zscore_reason(e) + '; ' + review_event_wo(e)])
        lines += [zscore_md_table(['구간·센서', '관측 근거', '분류한 이유', '복귀 확인', '해석·다음 확인'], rows), '']
        persistent = ev.loc[ev.OBS_CLASS.eq('지속 이상 후보')].sort_values(['N_BINS', 'REPEATS', 'MAX_ABS_Z'], ascending=False)
        picked = persistent.drop_duplicates('SENSOR').head(2)
        if len(picked):
            lines += ['**동반 신호로 다시 볼 근거**', '']
            for e in picked.itertuples():
                lines.append(f'- **{e.ID}:** {e.PAIRED_FACT}. 동시 관측의 근거이며 원인 확정은 아니다.')
            lines += ['']
    else:
        lines += ['이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.', '']
    p = profile[k]
    gaps = []
    matrix = pd.DataFrame({s: zstats[k, s].Z.isna() for s in sensors})
    for a, b in runs(matrix.any(axis=1)):
        affected = [s for s in sensors if matrix.loc[a:b - 1, s].any()]
        current_min = min((int(p.loc[a:b - 1, (s, 'count')].fillna(0).min()) for s in affected))
        history_min = min((int(zstats[k, s].X.shift(1).rolling(30, min_periods=1).count().fillna(0).loc[a:b - 1].min()) for s in affected))
        reason = []
        if current_min < 54:
            reason.append(f'현재창 실측 최소 {current_min}개 <54')
        if history_min < 20:
            reason.append(f'과거 유효 최소 {history_min}창 <20')
        gaps.append([f'{zscore_time(review_ts(k, a))}–{zscore_time(review_ts(k, b))} (끝 제외)', ', '.join(affected), ' / '.join(reason) or '기준 산출 불가'])
    if gaps:
        lines += ['**판정 보류의 근거 — 그림의 회색 구간**', '', zscore_md_table(['구간', '해당 채널', '점수를 계산하지 않은 이유'], gaps), '']
    lines += ['**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.']
    return '\n'.join(lines)

def _zscore_cp_base(k):
    d = review_events.loc[review_events.LOT.eq(k) & review_events.KIND.eq('CP')].sort_values('PEAK')
    if d.empty:
        return ''
    lines = ['#### CP 원본 이탈 · 단발 여부와 추적 근거', '', 'CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.', '']
    for e in d.itertuples():
        ctx = review_raw_context.loc[review_raw_context.ID.eq(e.ID)]
        lines += [f'**{e.ID} · 피크 {zscore_time(e.PEAK)} · {e.PATTERN}**', '', f'- **관측:** 원값 {e.VALUE:.6g}, LOT 중앙값 {e.BASELINE:.6g}, 편차 {e.DELTA:+.6g}. 탐색 경계 ±0.05 이상이다.', f'- **폭·반복:** 실측 {int(e.POINTS)}점, 첫·끝 실측 간격 {e.SPAN_SEC:g}초, 기대 1초 격자 내 유효 비율 {e.POINTS / (e.SPAN_SEC + 1):.1%}. 최근 30분 내 피크 {int(e.REPEATS)}회(해당 사건 포함).', f'- **전후·복귀:** {e.RECOVERY}.', '- **같은 1분의 결과:** ' + ('17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.' if e.MINUTE_REF17 else e.Z_AT_SAME_MINUTE + '.'), f'- **판단·추적:** {e.HOLD_REASON}. {e.ACTION}. 원본 DataFrame 행 번호 {int(ctx.SOURCE_ROW.min())}–{int(ctx.SOURCE_ROW.max())}에 연결했다(0부터 시작).', '']
    if k == 240117:
        task = cal_work.loc[cal_work.WO_NO.eq('WO-2024-01854')].iloc[0]
        peak = d.PEAK.min()
        delay = (peak - task.END_DT).total_seconds() / 60
        lines += [f'**시간 순서로 배제되는 설명:** CBM 완료 {zscore_time(task.END_DT)} 뒤 {delay:.2f}분에 CP 피크가 발생했다. 이 CP 사건을 해당 CBM의 사전 경보로 볼 수 없다. 정비와 원인이 같거나 수리 후 정상이라는 뜻은 아니다.', '']
    lines += ['**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.']
    return '\n'.join(lines)



REVIEW_EXTRA_SIGNALS=['CP-Z4','CP-Z4M','TC-Z9','TC-OUT1','TC-OUT2']
review_process_coverage={}
review_heating_checks={}
REVIEW_WO_HOURS=72
REVIEW_WO_COLOR='#7953A6'

def review_related_wo(k):
    meta=lot_meta.loc[k]
    lo=meta.START-pd.Timedelta(hours=REVIEW_WO_HOURS)
    hi=meta.END+pd.Timedelta(hours=REVIEW_WO_HOURS)
    near=cal_work[['DETC_DT','STRT_DT','END_DT']].apply(lambda col:col.between(lo,hi)).any(axis=1)
    overlap=cal_work.STRT_DT.le(meta.END)&cal_work.END_DT.ge(meta.START)
    return cal_work.loc[near|overlap].sort_values('STRT_DT')

def review_mark_maintenance(ax,k):
    meta=lot_meta.loc[k]
    for w in cal_work.itertuples():
        if w.STRT_DT<=meta.END and w.END_DT>=meta.START:
            a=(max(w.STRT_DT,meta.START)-meta.START).total_seconds()/60
            b=(min(w.END_DT,meta.END)-meta.START).total_seconds()/60
            ax.axvspan(a,b,color=REVIEW_WO_COLOR,alpha=.06,zorder=.4)
        for t,ls in [(w.DETC_DT,':'),(w.STRT_DT,'--'),(w.END_DT,'-.')]:
            if pd.notna(t) and meta.START<=t<=meta.END:
                x=(t-meta.START).total_seconds()/60
                ax.axvline(x,color=REVIEW_WO_COLOR,ls=ls,lw=.8,alpha=.65,zorder=1.5)

def review_event_wo(event):
    options=[]
    for w in review_related_wo(event.LOT).itertuples():
        for label,t in [('감지',w.DETC_DT),('착수',w.STRT_DT),('완료',w.END_DT)]:
            if pd.notna(t):
                delta=(event.PEAK-t).total_seconds()/60
                if abs(delta)<=REVIEW_WO_HOURS*60:options.append((abs(delta),w.WO_NO,label,delta))
    if not options:return '정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음'
    _,wo,label,delta=min(options,key=lambda item:item[0])
    return f'정비 대조: {wo} {label} 대비 {"Z 최대창 중심" if event.KIND=="Z" else "CP 원본 피크"} {delta:+.2f}분 (근접 시각이며 인과 미확정)'

def review_wo_boundary(k,w):
    g=raw[k].set_index('MEAS_DT').asfreq('s')
    out=[]
    for label,t in [('감지',w.DETC_DT),('착수',w.STRT_DT),('완료',w.END_DT)]:
        if pd.isna(t):continue
        pre=g.loc[(g.index>=t-pd.Timedelta(minutes=5))&(g.index<t)]
        post=g.loc[(g.index>=t)&(g.index<t+pd.Timedelta(minutes=5))]
        parts=[]
        for s in ['CP-Z4','CP-Z4M','TC-Z3','OP-Z3']:
            a=pre[s].dropna();b=post[s].dropna()
            if len(a)>=190 and len(b)>=190:
                unit='℃' if s in TC else '%p' if s in OP else '무차원'
                parts.append(f'{s} 후−전 중앙값 {b.median()-a.median():+.4g} {unit} ({len(a)}/{len(b)}점)')
        if parts:
            out.append(f'{label} {review_clock(t)}: '+' / '.join(parts))
        else:
            a=int(pre['CP-Z4'].count());b=int(post['CP-Z4'].count())
            out.append(f'{label} {review_clock(t)}: 전/후 5분 CP 실측 {a}/{b}점; 양쪽 190점 기준 미충족으로 효과 비교 보류')
    return out

def reading_full_review_maintenance_markdown(k):
    meta = lot_meta.loc[k]
    wo = review_related_wo(k)
    events = review_events.loc[review_events.LOT.eq(k)]
    cp = events.loc[events.KIND.eq('CP')].sort_values('PEAK')
    lines = ['#### 정비이력 대조 · 관측 사실과 검토할 가설', '', '같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.', '']
    if wo.empty:
        return '\n'.join(lines + ['**대조 가능한 근접 기록 없음:** 이 범위에 같은 설비 정비 기록이 없다. 기록 부재를 설비 정상이나 정비와 무관하다는 근거로 쓰지 않는다.'])
    rows = []
    for w in wo.itertuples():
        remark = str(w.REMARK).strip() if pd.notna(w.REMARK) else '내용 미기재'
        parts = cal_parts.loc[cal_parts.WO_NO.eq(w.WO_NO)]
        detail = ' / '.join((f'{r.PART_CD} {r.PART_NM} ×{r.QTY} (교체 사유: {r.CHG_RSN})' for r in parts.itertuples())) or '교체 상세 없음'
        overlap = w.STRT_DT <= meta.END and w.END_DT >= meta.START
        rows.append([f'{w.WO_NO} · {w.MNT_TYPE}', review_clock(w.DETC_DT), review_clock(w.STRT_DT), review_clock(w.END_DT), remark, detail, '작업과 LOT 관측 겹침' if overlap else '작업은 LOT 관측 밖'])
    lines += [zscore_md_table(['정비 기록', '감지', '착수', '완료', 'REMARK', '부품 기록', '관측 조건'], rows), '']
    for w in wo.itertuples():
        relevant = []
        for e in events.itertuples():
            stamps = [t for t in [w.DETC_DT, w.STRT_DT, w.END_DT] if pd.notna(t)]
            if any((abs((e.PEAK - t).total_seconds()) <= REVIEW_WO_HOURS * 3600 for t in stamps)):
                relevant.append(e)
        cc = [e for e in relevant if e.KIND == 'CP']
        zz = [e for e in relevant if e.KIND == 'Z']
        lines += [f'**{w.WO_NO} · 사실 → 시간 관계 → 가설 → 확인**', '', f'- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 {len(cc)}건, 센서별 Z 후보 {len(zz)}건. 중첩 센서·시간창을 합친 고장 건수가 아니다.']
        if cc:
            for e in cc:
                delta = (e.PEAK - w.END_DT).total_seconds() / 60
                det = (w.DETC_DT - e.PEAK).total_seconds() / 60
                lines.append(f'  - {e.ID} · {review_clock(e.PEAK)} · CP 편차 {e.DELTA:+.5g} · {int(e.POINTS)}점/{e.SPAN_SEC:g}초 · 완료 대비 피크 {delta:+.2f}분 · 감지는 피크 대비 {det:+.2f}분. {e.PATTERN}; {e.RECOVERY}.')
        else:
            base = float(raw[k]['CP-Z4'].median())
            dev = (raw[k]['CP-Z4'] - base).abs().max()
            lines.append(f'- **CP 대조:** LOT 전체의 최대 절대 편차 {dev:.5g} (무차원), 원본 ±0.05 사건 {len(cp)}건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.')
        if zz:
            facts = []
            for s, grp in pd.DataFrame([e._asdict() for e in zz]).groupby('SENSOR'):
                facts.append(f'{s} {len(grp)}건·최대 절대 Z {grp.MAX_ABS_Z.max():.2f}')
            lines.append('- **동반 양상:** ' + ' / '.join(facts) + '. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.')
        lines += ['- **정비 경계의 실제 전후 관측:**'] + ['  - ' + v for v in review_wo_boundary(k, w)]
        if cc and any((e.PEAK > w.END_DT for e in cc)):
            lines.append('- **시간 순서로 제한되는 해석:** 완료 뒤의 CP 사건은 해당 작업의 사전 경보가 아니다. 재가동·설정 복원 응답, 계측 경로 변동, 남아 있는 설비 변화를 경쟁 가설로 검토한다.')
        elif cc and all((e.PEAK < w.DETC_DT for e in cc)):
            lines.append('- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.')
        else:
            lines.append('- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.')
        lines += ['- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.', '']
    return '\n'.join(lines)

def reading_full_review_extra_markdown(k):
    g = raw[k]
    p = profile[k]
    rows = []
    for s in REVIEW_EXTRA_SIGNALS:
        x = g[s].dropna()
        base = float(x.median())
        dev = float((x - base).abs().max())
        ev = review_events.loc[review_events.LOT.eq(k) & review_events.SENSOR.eq(s) & review_events.KIND.eq('CP')]
        if s == 'CP-Z4':
            reason = f"원본 ±0.05 사건 {len(ev)}건 · 최대 절대 편차 {dev:.5g} · 1초 단발 {int(ev.OBS_CLASS.eq('단발 스파이크').sum())}건"
            status = '원본 이탈 추적·설비 판정 보류' if len(ev) else '원본 탐색 기준 내 참고·설비 판정 보류'
        else:
            reason = f'원본 중앙값 {base:.6g}, 최저 {x.min():.6g}, 최고 {x.max():.6g}; 별도 Z·이탈 경계 미적용'
            status = '관측 참고·판정 보류'
        rows.append([s, '무차원' if s.startswith('CP') else '℃', len(x), int(p[s, 'count'].fillna(0).lt(54).sum()), reason, status])
    return '#### CP·Z9·냉각 · 생략하지 않은 채널의 근거\n\n' + zscore_md_table(['채널', '단위', '실측 수', '54개 미만 1분창', '확인한 사실', '해석 범위'], rows) + '\n\nCP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.'

def review_cp_cooling_plot(k):
    meta=lot_meta.loc[k];p=profile[k];wo=review_related_wo(k)
    lane_h=.45+.27*(len(wo)+1)
    data_top=1.75+lane_h+.85
    height=data_top+11+.85
    fig=plt.figure(figsize=(22,height),facecolor='white')
    fig.text(.07,1-.18/height,f'LOT {k} · {meta.GRADE} | CP·Z9·냉각 원본과 정비 대조',va='top',fontsize=19,weight='bold')
    fig.text(.07,1-.6/height,'위: 실제 날짜의 LOT·정비 시각. 아래: 관측창 안의 원본·편차. CP는 무차원이며 별도의 Z를 적용하지 않음.',va='top',fontsize=10)
    handles=[Patch(color='#CBD2D9',alpha=.65,label='1분 내 원본 최저~최고'),
        Line2D([],[],color='#176B94',label='CP 1초 원본 / TC 유효 1분 중앙값'),
        Line2D([],[],color='#B24433',ls='--',label='CP-Z4 편차 ±0.05'),
        Patch(color='#B12261',alpha=.22,label='CP-Z4 원본 이탈'),
        Patch(color='#ABB7C5',alpha=.25,label='현재창 실측 부족'),
        Line2D([],[],color=REVIEW_WO_COLOR,ls='-.',label='WO 감지·작업 시각')]
    fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.065,1-.98/height),ncol=3,fontsize=9,frameon=False,columnspacing=1.4)
    timeline=fig.add_axes([.12,1-(1.75+lane_h)/height,.82,lane_h/height])
    timeline.set_title('실제 날짜 대조 · 정비 탐색 범위는 LOT 전후 72시간 · 관측 없는 기간은 센서 선으로 연결하지 않음',loc='left',fontsize=9,pad=7)
    timeline.plot([meta.START,meta.END],[0,0],color='#176B94',lw=5,solid_capstyle='butt')
    ticks=[0];labels=['LOT 관측']
    dates=[meta.START,meta.END]
    for j,w in enumerate(wo.itertuples(),1):
        timeline.plot([w.STRT_DT,w.END_DT],[j,j],color=REVIEW_WO_COLOR,lw=5,solid_capstyle='butt')
        timeline.plot(w.DETC_DT,j,marker='v',color=REVIEW_WO_COLOR,ms=5)
        ticks.append(j);labels.append(w.WO_NO.replace('WO-2024-','WO ')+f' ({w.MNT_TYPE})')
        dates.extend([w.DETC_DT,w.STRT_DT,w.END_DT])
    timeline.set_yticks(ticks,labels,fontsize=8)
    timeline.set_ylim(-.6,len(ticks)-.4)
    clock_ticks(timeline,min(dates)-pd.Timedelta(minutes=20),max(dates)+pd.Timedelta(minutes=20))
    timeline.tick_params(axis='x',labelsize=8);timeline.grid(axis='x',alpha=.15)
    timeline.spines[['top','right','left']].set_visible(False)
    gs=fig.add_gridspec(5,2,left=.075,right=.94,top=1-data_top/height,bottom=.85/height,hspace=.72,wspace=.25,width_ratios=[1.45,1])
    xx=p.index.to_numpy()+.5
    for row,s in enumerate(REVIEW_EXTRA_SIGNALS):
        left=fig.add_subplot(gs[row,0]);right=fig.add_subplot(gs[row,1])
        review_axis(left);review_axis(right)
        scale=1e10 if s=='CP-Z4M' else 1.
        base=float(raw[k][s].median())
        counts=p[s,'count'].fillna(0)
        left.fill_between(xx,p[s,'min'].to_numpy()*scale,p[s,'max'].to_numpy()*scale,color='#CBD2D9',alpha=.65)
        if s.startswith('CP'):
            series=raw[k].set_index('MEAS_DT')[s].asfreq('s')
            x=(series.index-meta.START).total_seconds()/60
            left.plot(x,series.to_numpy()*scale,color='#176B94',lw=.65)
            right.plot(x,(series.to_numpy()-base)*scale,color='#176B94',lw=.65)
            title='1초 원본' if s=='CP-Z4' else '1초 원본 · 원값×10¹⁰'
            left.set_ylabel(s+' [무차원]'+(' · 원값×10¹⁰' if scale!=1 else ''),fontsize=9)
            right.set_ylabel('편차 [무차원]'+(' · 원값×10¹⁰' if scale!=1 else ''),fontsize=9)
        else:
            v=p[s,'median'].where(counts.ge(54))
            left.plot(xx,v,color='#176B94',lw=1)
            right.plot(xx,v-base,color='#176B94',lw=1)
            title='유효 1분 중앙값 · Z 미적용'
            left.set_ylabel('온도 [℃]',fontsize=9);right.set_ylabel('편차 [℃]',fontsize=9)
        ev=review_events.loc[review_events.LOT.eq(k)&review_events.SENSOR.eq(s)&review_events.KIND.eq('CP')]
        if s=='CP-Z4':
            for v in [-.05,.05]:right.axhline(v,color='#B24433',ls='--',lw=.8)
            for v in [-.02,.02]:right.axhline(v,color='#8496A5',ls=':',lw=.7)
            for e in ev.itertuples():
                a=(e.START-meta.START).total_seconds()/60;b=(e.END-meta.START).total_seconds()/60
                for ax in [left,right]:ax.axvspan(a,b,color='#B12261',alpha=.2)
                px=(e.PEAK-meta.START).total_seconds()/60
                left.plot(px,e.VALUE,'o',ms=3,color='#B12261')
                right.plot(px,e.DELTA,'o',ms=3,color='#B12261')
            note=f'원본 이탈 {len(ev)}건 · LOT 중앙값 편차 · ±0.02 복귀 참고'
        else:note='LOT 중앙값 편차 · 별도 이탈 경계·Z 미적용'
        left.set_title(s+' · '+title,loc='left',fontsize=10,pad=8)
        right.set_title(note,loc='left',fontsize=10,pad=8)
        right.axhline(0,color='#8496A5',lw=.65)
        for a,b in runs(counts.lt(54)):
            for ax in [left,right]:ax.axvspan(a,b,color='#ABB7C5',alpha=.18)
        for ax in [left,right]:
            review_mark_maintenance(ax,k);ax.set_xlim(0,len(p))
    fig.text(.07,.18/height,'보라 시각은 같은 설비 WO의 실제 감지·착수·완료다. CP 피크와 정비의 선후 관계는 근거표에서 확인한다. 동시성·복귀만으로 원인·정상·정비 효과를 확정하지 않는다.',fontsize=10,va='bottom')
    review_save(fig,f'06_EDA_LOT/{k}_{meta.GRADE}_04_CP냉각_전체채널.png')
    review_process_coverage[k,'04_CP냉각']=set(REVIEW_EXTRA_SIGNALS)

def reading_full_zscore_cp_markdown(k):
    text = _zscore_cp_base(k)
    if not text:
        return ''
    cp = review_events.loc[review_events.LOT.eq(k) & review_events.KIND.eq('CP')]
    return text + '\n\n**CP 피크와 정비 시각 대조**\n\n' + zscore_md_table(['원본 사건', '정비와의 시간 관계', '동반 변화: TC ℃·OP %p·CP 무차원'], [[e.ID, review_event_wo(e), e.PAIRED_FACT] for e in cp.itertuples()])




def review_heating_cp_plot(k):
    key,label,zs=next(g for g in REVIEW_Z_GROUPS if g[0]=='02_가열균열')
    sensors=['CP-Z4','CP-Z4M']+list(zs)
    meta=lot_meta.loc[k];p=profile[k];xx=p.index.to_numpy()+.5
    cp=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP')]
    height=3.0+2.2*len(sensors)
    fig=plt.figure(figsize=(22,height),facecolor='white')
    fig.text(.07,1-.18/height,f'LOT {k} · {meta.GRADE} | 가열·균열 Z3~Z6 + CP 동시 비교',va='top',fontsize=19,weight='bold')
    fig.text(.07,1-.6/height,'위 2행: CP 1초 원본·편차. 이어지는 행: Z3~Z6의 TC·OP 중앙값·Z. 모든 패널은 같은 LOT 경과시간이며 분홍 세로선은 CP-Z4 피크 대조 시각.',va='top',fontsize=10)
    handles=[Patch(color='#CBD2D9',alpha=.65,label='1분 내 원본 최저~최고'),
       Line2D([],[],color='#176B94',label='TC 1분 중앙값 / CP 1초 원본'),
       Line2D([],[],color='#176B94',ls='--',label='OP 유효 1분 중앙값'),
       Patch(color='#B8D9E5',alpha=.5,label='이동 중심 ±3SD (TC·OP만)'),
       Patch(color='#E4B13F',alpha=.4,label='짧은 국소 후보 (Z)'),
       Patch(color='#D45D18',alpha=.35,label='지속·반복 후보 (Z)'),
       Patch(color='#ABB7C5',alpha=.3,label='관측/기준 부족'),
       Line2D([],[],color='#B12261',ls=':',label='CP-Z4 피크 시각 · 모든 행 대조'),
       Line2D([],[],color='#B24433',ls='--',label='CP-Z4 편차 ±0.05'),
       Line2D([],[],color=REVIEW_WO_COLOR,ls='-.',label='WO 감지·작업 시각')]
    fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.065,1-.98/height),frameon=False,ncol=4,fontsize=9,columnspacing=1.15)
    gs=fig.add_gridspec(len(sensors),2,left=.075,right=.94,top=1-2.2/height,bottom=.85/height,hspace=.72,wspace=.25,width_ratios=[1.45,1])
    zids=set()
    for row,s in enumerate(sensors):
        left=fig.add_subplot(gs[row,0]);right=fig.add_subplot(gs[row,1])
        for ax in [left,right]:review_axis(ax)
        counts=p[s,'count'].fillna(0)
        scale=1e10 if s=='CP-Z4M' else 1.
        left.fill_between(xx,p[s,'min'].to_numpy()*scale,p[s,'max'].to_numpy()*scale,color='#CBD2D9',alpha=.65)
        if s.startswith('CP'):
            raw_series=raw[k].set_index('MEAS_DT')[s].asfreq('s')
            x=(raw_series.index-meta.START).total_seconds()/60
            base=float(raw_series.median())
            left.plot(x,raw_series.to_numpy()*scale,color='#176B94',lw=.7)
            right.plot(x,(raw_series.to_numpy()-base)*scale,color='#176B94',lw=.7)
            unit='[무차원]'+(' · 원값×10¹⁰' if scale!=1 else '')
            left.set_ylabel(s+' '+unit,fontsize=9);right.set_ylabel('편차 '+unit,fontsize=9)
            left.set_title(s+' · 1초 원본'+(' · 원값×10¹⁰' if scale!=1 else ''),loc='left',fontsize=10,pad=8)
            right.set_title(f'LOT 중앙값 대비 편차 · 원본 이탈 {len(cp)}건' if s=='CP-Z4' else 'LOT 중앙값 대비 편차 · 별도 Z·경계 미적용',loc='left',fontsize=10,pad=8)
            right.axhline(0,color='#8496A5',lw=.65)
            if s=='CP-Z4':
                for v in [-.05,.05]:right.axhline(v,color='#B24433',ls='--',lw=.8)
                for v in [-.02,.02]:right.axhline(v,color='#8496A5',ls=':',lw=.7)
                for e in cp.itertuples():
                    px=(e.PEAK-meta.START).total_seconds()/60
                    left.plot(px,e.VALUE,'o',color='#B12261',ms=3)
                    right.plot(px,e.DELTA,'o',color='#B12261',ms=3)
            missing=counts.lt(54)
        else:
            d=zstats[k,s]
            left.plot(xx,d.X,color='#176B94',lw=1,ls='--' if s in OP else '-')
            left.fill_between(xx,(d.MEDIAN-3*d.SD).to_numpy(),(d.MEDIAN+3*d.SD).to_numpy(),color='#B8D9E5',alpha=.4)
            right.plot(xx,d.Z,color='#176B94',lw=1)
            for v in [-3,3]:right.axhline(v,color='#B24433',ls='--',lw=.8)
            ev=review_events.loc[review_events.LOT.eq(k)&review_events.SENSOR.eq(s)&review_events.KIND.eq('Z')]
            zids.update(ev.ID)
            for e in ev.itertuples():
                color='#D45D18' if e.OBS_CLASS=='지속 이상 후보' else '#E4B13F'
                for ax in [left,right]:ax.axvspan(e.A,e.B,color=color,alpha=.28)
            left.set_title(f'{s} · {len(ev)}개 Z 사건 · 전 구간 표시',loc='left',fontsize=10,pad=8)
            right.set_title(f'|Z|≥3 {int(d.Z.abs().ge(3).sum())}창 / 유효 {int(d.Z.notna().sum())}창 / 계산 보류 {int(d.Z.isna().sum())}창',loc='left',fontsize=10,pad=8)
            left.set_ylabel(review_unit(s),fontsize=9);right.set_ylabel('Z · 설비 판정 아님',fontsize=9)
            missing=d.Z.isna()
        for a,b in runs(missing):
            for ax in [left,right]:ax.axvspan(a,b,color='#ABB7C5',alpha=.18)
        for ax in [left,right]:
            for e in cp.itertuples():
                px=(e.PEAK-meta.START).total_seconds()/60
                ax.axvline(px,color='#B12261',ls=':',lw=.8,alpha=.6,zorder=1.4)
            review_mark_maintenance(ax,k)
            ax.set_xlim(0,len(p))
    fig.text(.07,.18/height,'분홍선은 CP 피크와 Z3~Z6 반응을 같은 시각에서 비교하는 표시다. TC·OP의 이상 판정을 뜻하지 않는다. CP는 무차원이며 기존 TC·OP Z와 별도로 읽는다. 정비 시각·가설·확인 자료는 바로 아래 근거표에 연결한다.',fontsize=10,va='bottom')
    review_save(fig,f'06_EDA_LOT/{k}_{meta.GRADE}_{key}_전체채널.png')
    assert len(sensors)==10 and set(zs)==set(sensors)-{'CP-Z4','CP-Z4M'}
    assert zids==set(review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('Z')&review_events.SENSOR.isin(zs),'ID'])
    review_z_coverage[k,key]=dict(SENSORS=set(zs),EVENT_IDS=zids)
    review_process_coverage[k,key]=set(sensors)
    if 'review_heating_checks' not in globals():globals()['review_heating_checks']={}
    review_heating_checks[k]=dict(SENSORS=set(sensors),CP_IDS=set(cp.ID),Z_IDS=zids,
                                 XLIM=(0,len(p)),ROWS=len(sensors))

def reading_full_zscore_group_markdown(k, group_key):
    text = _zscore_group_base(k, group_key)
    if group_key != '02_가열균열':
        return text
    cp = review_events.loc[review_events.LOT.eq(k) & review_events.KIND.eq('CP')]
    extra = f'**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 {len(cp)}건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.\n\n'
    head, body = text.split('\n\n', 1)
    return head + '\n\n' + extra + body



READING_MAX_CARDS=3

def reading_fold(title,body):
    return '<details>\n<summary><strong>'+review_escape(title)+'</strong></summary>\n\n'+body.strip()+'\n\n</details>'

def reading_duration(seconds):
    sec=int(round(abs(float(seconds))))
    if sec==0:return '동일 시각'
    d,sec=divmod(sec,86400);h,sec=divmod(sec,3600);m,sec=divmod(sec,60)
    pieces=[]
    if d:pieces.append(f'{d}일')
    if h:pieces.append(f'{h}시간')
    if m:pieces.append(f'{m}분')
    if sec and not d:pieces.append(f'{sec}초')
    return ' '.join(pieces[:3])

def reading_time(t,seconds=False):
    return pd.Timestamp(t).strftime('%H:%M:%S' if seconds else '%H:%M') if pd.notna(t) else '미확인'

def reading_interval(e):
    if e.KIND=='CP':return review_clock(e.PEAK)
    a=pd.Timestamp(e.START);b=pd.Timestamp(e.END)
    end=b.strftime('%H:%M') if a.date()==b.date() else b.strftime('%m/%d %H:%M')
    return a.strftime('%m/%d %H:%M')+'~'+end

def reading_events(k,sensors=None):
    d=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('Z')].copy()
    if sensors is not None:d=d.loc[d.SENSOR.isin(sensors)]
    return d

def reading_zone(sensor):
    match=re.search(r'-Z(\d+)',sensor)
    return match.group(1) if match else sensor


def reading_nearest_wo(k,event=None):
    wo=review_related_wo(k)
    if wo.empty:return None
    meta=lot_meta.loc[k]
    point=event.PEAK if event is not None else meta.START+(meta.END-meta.START)/2
    ranked=[]
    for w in wo.itertuples():
        anchors=[(name,t) for name,t in [('감지',w.DETC_DT),('착수',w.STRT_DT),('완료',w.END_DT)] if pd.notna(t)]
        label,t=min(anchors,key=lambda v:abs((point-v[1]).total_seconds()))
        overlap=w.STRT_DT<=meta.END and w.END_DT>=meta.START
        ranked.append((not overlap,abs((point-t).total_seconds()),w.WO_NO,w,label,t))
    _,_,_,w,label,t=min(ranked,key=lambda v:v[:3])
    return w,label,t

def reading_wo_relation(event,chosen=None):
    selected=chosen or reading_nearest_wo(event.LOT,event)
    if selected is None:return 'LOT 전후 72시간 안의 같은 설비 정비 기록 없음'
    w,label,t=selected
    sec=(event.PEAK-t).total_seconds()
    direction='후' if sec>0 else '전' if sec<0 else ''
    span=reading_duration(sec)
    point='CP 원본 피크' if event.KIND=='CP' else 'Z 최대창 중심'
    return f'{w.WO_NO} {label} {span} {direction}에 {point}'.replace('동일 시각 에','동일 시각에')


def reading_primary(k):
    cp=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP')]
    if len(cp):return next(cp.loc[cp.DELTA.abs().eq(cp.DELTA.abs().max())].sort_values('PEAK').itertuples())
    selected=reading_select(k)
    return selected[0][1] if selected else None

def reading_boundary(k,w,label,t):
    g=raw[k].set_index('MEAS_DT')
    before=g.loc[(g.index>=t-pd.Timedelta(minutes=5))&(g.index<t)]
    after=g.loc[(g.index>=t)&(g.index<t+pd.Timedelta(minutes=5))]
    n1=int(before['CP-Z4'].count());n2=int(after['CP-Z4'].count())
    if min(n1,n2)<190:return f'{label} 전후 5분 CP 실측 **{n1}/{n2}점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.'
    facts=[]
    for sensor,unit in [('CP-Z4','무차원'),('TC-Z3','℃'),('OP-Z3','%p')]:
        a=before[sensor].dropna();b=after[sensor].dropna()
        if min(len(a),len(b))>=190:
            v=b.median()-a.median()
            facts.append(f'{sensor} {v:+.4g}{unit}')
    return f'{label} 경계 전후 5분의 중앙값 차이(후−전): **'+' / '.join(facts)+'**. 피크 자체와 구분해서 읽는다.'




def review_maintenance_markdown(k):
    wo=review_related_wo(k);e=reading_primary(k)
    lines=['#### 정비 대조 · 사실과 가설을 나눠 보기','']
    if wo.empty:
        return '\n'.join(lines+['**대조할 기록 부족:** LOT 전후 72시간 안에 같은 설비 WO가 없다. 정비가 없었다거나 설비가 정상이라는 뜻은 아니다.'])
    chosen=reading_nearest_wo(k,e);w,label,t=chosen
    remark=str(w.REMARK).strip() if pd.notna(w.REMARK) else '내용 미기재'
    lines += [f'**먼저 볼 기록: {w.WO_NO} · {w.MNT_TYPE} / 메모: {remark}**','']
    lines += [f'**실제 시각:** 감지 {review_clock(w.DETC_DT)} / 작업 {review_clock(w.STRT_DT)}~{review_clock(w.END_DT)}. 감지와 수리 시점을 구분한다.','']
    if e is not None:
        lines += ['**시간 관계:** '+reading_wo_relation(e,chosen)+'.','',
                  '**실측으로 비교할 수 있는 범위:** '+reading_boundary(k,w,label,t),'']
        if e.KIND=='CP' and e.PEAK>w.END_DT:
            hypothesis='정비 완료 뒤의 CP 사건이므로 해당 작업의 사전 경보로 볼 수 없다. 재가동·제어 복원 응답, 계측 변동, 남아 있는 설비 변화를 구분해 검토한다.'
        elif e.PEAK<w.DETC_DT:
            hypothesis='변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.'
        else:
            hypothesis='작업 또는 제어 조건 변경과 동반됐을 가능성을 검토한다. 같은 시각이라는 이유만으로 정비가 원인이라고 판단하지 않는다.'
        lines+=['**검토할 가설:** '+hypothesis,'']
    lines += ['**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.','',
       f'같은 설비의 관련 WO는 총 {len(wo)}건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.','',
       reading_fold(f'정비 {len(wo)}건의 전체 시각·메모·부품·전후 관측·가설 펼치기',reading_full_review_maintenance_markdown(k))]
    return '\n'.join(lines)

def review_extra_markdown(k):
    cp=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP')]
    return ('#### CP·냉각 보조 대조 · 확인 범위\n\n'
       f'CP 원본 이탈 **{len(cp)}건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. '
       'CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.\n\n'
       +reading_fold('5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기',reading_full_review_extra_markdown(k)))


def reading_overlap_hint(e):
    zone=re.fullmatch(r'(TC|OP)-Z([1-8])',e.SENSOR)
    if not zone:return ''
    sensor=('OP' if zone.group(1)=='TC' else 'TC')+'-Z'+zone.group(2)
    other=review_events.loc[review_events.LOT.eq(e.LOT)&review_events.KIND.eq('Z')&
        review_events.SENSOR.eq(sensor)&review_events.START.lt(e.END)&review_events.END.gt(e.START)].copy()
    if other.empty:return ''
    other['READ_OVERLAP']=[(min(r.END,e.END)-max(r.START,e.START)).total_seconds() for r in other.itertuples()]
    peer=next(other.sort_values(['READ_OVERLAP','N_BINS','START'],ascending=[False,False,True]).itertuples())
    unit='℃' if sensor in TC else '%p'
    return f'겹치는 시간대의 **{sensor}도 {int(peer.N_BINS)}개 1분창 초과**({peer.DELTA:+.2f}{unit})했다.'


_evidence_peer_cache = {}
_evidence_pattern_cache = {}

# 후보 구간의 사후 EDA 대조: 학습·예측·정상 라벨 생성 없이 관측 사실을 비교한다.
def evidence_phase(e):
    st=states[e.LOT]
    if e.KIND=='CP':
        return str(st.loc[min(int(e.MINUTE),len(st)-1)])
    phases=st.loc[int(e.A):int(e.B)-1].unique()
    return str(phases[0]) if len(phases)==1 else '전환 혼합'

def evidence_peer_context(e):
    if e.ID in _evidence_peer_cache:return _evidence_peer_cache[e.ID]
    ids=lots.loc[lots.GRADE.eq(e.GRADE)&lots.LOT_NO.ne(e.LOT),'LOT_NO'].astype(int).tolist()
    phase=evidence_phase(e)
    peers=review_events.loc[review_events.LOT.isin(ids)&review_events.SENSOR.eq(e.SENSOR)&review_events.KIND.eq(e.KIND)].copy()
    if len(peers):
        peers=peers.loc[[evidence_phase(r)==phase and np.sign(r.DELTA)==np.sign(e.DELTA) for r in peers.itertuples()]]
    exposure={}
    for k in ids:
        mask=states[k].eq(phase)
        if e.KIND=='Z':exposure[k]=int((mask&zstats[k,e.SENSOR].Z.notna()).sum())
        else:
            g=raw[k]
            exposure[k]=int(g.loc[g.MINUTE.map(states[k]).eq(phase),e.SENSOR].count())
    # 분모 0인 LOT를 '후보 0회'로 비교하지 않는다.
    eligible=[k for k in ids if exposure[k]>0]
    peers=peers.loc[peers.LOT.isin(eligible)]
    result=ids,eligible,peers,exposure,phase
    _evidence_peer_cache[e.ID]=result
    return result

def evidence_peer_text(e):
    ids,eligible,peers,exposure,phase=evidence_peer_context(e)
    if not ids:return f'{e.GRADE}의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.'
    if not eligible:return f'같은 강종 LOT {", ".join(map(str,ids))}에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.'
    names=', '.join(map(str,eligible))
    if peers.empty:
        return f'같은 강종 LOT {names}의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.'
    unit='무차원' if e.KIND=='CP' else '℃' if e.SENSOR in TC else '%p'
    lo,hi=peers.DELTA.abs().min(),peers.DELTA.abs().max()
    if e.KIND=='Z':
        low,high=int(peers.N_BINS.min()),int(peers.N_BINS.max())
        span=f'연속 {low}~{high}개 1분창'
        outside=e.N_BINS>high or abs(e.DELTA)>hi
        target=f'이 구간의 {int(e.N_BINS)}창·{abs(e.DELTA):.3g}{unit}'
    else:
        low,high=peers.SPAN_SEC.min(),peers.SPAN_SEC.max()
        span=f'첫/끝 실측 간격 {low:g}~{high:g}초'
        outside=e.SPAN_SEC>high or abs(e.DELTA)>hi
        target=f'이 구간의 {e.SPAN_SEC:g}초 간격·{abs(e.DELTA):.3g}{unit}'
    observed=', '.join(map(str,sorted(peers.LOT.unique())))
    excess=[]
    if abs(e.DELTA)>hi:excess.append(f'절대 편차 {abs(e.DELTA):.3g}>{hi:.3g}{unit}')
    size=e.N_BINS if e.KIND=='Z' else e.SPAN_SEC
    if size>high:excess.append(f'길이 {size:g}>{high:g}'+('창' if e.KIND=='Z' else '초 간격'))
    result=('비교 관측의 상한을 넘는다('+', '.join(excess)+')' if excess else '비교한 변화 폭·길이의 상한 이내다')
    matching=''
    if e.KIND=='Z':
        pattern=evidence_pair_pattern(e)
        if pattern is not None:
            nmatch=sum(evidence_pair_pattern(r)==pattern for r in peers.itertuples())
            matching=f' 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 {nmatch}건이다.'
    return (f'동종 LOT {observed}에도 같은 방향 후보 {len(peers)}건({span}, 절대 편차 {lo:.3g}~{hi:.3g}{unit})이 있다. '
            f'{target}는 {result}.{matching} SV·속도·규격·모드가 없어 조건 일치는 미확인이다.')

def evidence_pair(e):
    m=re.fullmatch(r'(TC|OP)-Z([1-8])',e.SENSOR)
    if e.KIND!='Z' or not m:return None
    channel=('OP' if m.group(1)=='TC' else 'TC')+'-Z'+m.group(2)
    x=zstats[e.LOT,channel].X
    pre=x.loc[max(0,int(e.A)-10):int(e.A)-1].dropna()
    now=x.loc[int(e.A):int(e.B)-1].dropna()
    if len(pre)<5 or now.empty:return None
    return channel,float(now.median()-pre.median()),len(pre),len(now)

def evidence_domain(e):
    # 조건부 해석: TC/OP의 태그 대응·정역동작·자동모드·SV가 확인되어야 한다.
    if e.KIND=='CP':
        m=int(e.MINUTE)
        active=[s for s in [f'{kind}-Z{j}' for j in range(3,7) for kind in ['TC','OP']]
                if pd.notna(zstats[e.LOT,s].Z.loc[m]) and abs(zstats[e.LOT,s].Z.loc[m])>=3]
        absent=[s for s in [f'{kind}-Z{j}' for j in range(3,7) for kind in ['TC','OP']]
                if pd.isna(zstats[e.LOT,s].Z.loc[m])]
        concurrent=('동일 1분에 '+', '.join(active)+'도 기준을 넘었다' if active else '동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다')
        if absent:concurrent+=' (계산 보류: '+', '.join(absent)+')'
        return concurrent+'. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.'
    pair=evidence_pair(e)
    if pair is None:
        return '대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.'
    other,difference,pre,now=pair
    unit='℃' if other in TC else '%p'
    x=zstats[e.LOT,e.SENSOR].X
    ownpre=x.loc[max(0,int(e.A)-10):int(e.A)-1].dropna();ownnow=x.loc[int(e.A):int(e.B)-1].dropna()
    own=float(ownnow.median()-ownpre.median()) if len(ownpre)>=5 and len(ownnow) else np.nan
    ownunit='℃' if e.SENSOR in TC else '%p'
    fact=f'{other}의 사건창 중앙값−직전 10분 중앙값은 {difference:+.3g}{unit}({pre}/{now}유효창). '
    if pd.notna(own):fact+=f'같은 전후 기준의 {e.SENSOR} 변화는 {own:+.3g}{ownunit}. '
    if e.SENSOR.startswith(('TC-Z3','TC-Z4','TC-Z5','TC-Z6','OP-Z3','OP-Z4','OP-Z5','OP-Z6')):
        if pd.isna(own):
            hypothesis='주 채널의 유효 직전 관측이 부족해 TC·OP 방향 해석을 보류한다.'
        elif own*difference<0:
            hypothesis='OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.'
        else:
            hypothesis='가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.'
    else:
        hypothesis='제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.'
    if e.TRANSITION:hypothesis+=' 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.'
    return fact+hypothesis

def evidence_next(e):
    _,eligible,peers,_,_=evidence_peer_context(e)
    if e.KIND=='CP':
        back='60개 연속 실제 1초 값의 ±0.02 복귀 '+review_clock(e.FIXED_RETURN) if pd.notna(e.FIXED_RETURN) else '60개 연속 실제 1초 값의 ±0.02 복귀 미확인'
        return back+'. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. '+reading_wo_relation(e)+'.'
    fixed='시작 고정 대역 3창 복귀 '+reading_time(e.FIXED_RETURN) if pd.notna(e.FIXED_RETURN) else '시작 고정 대역 3창 복귀 미확인'
    moving=''
    if pd.notna(e.Z_RETURN) and pd.notna(e.FIXED_RETURN) and e.FIXED_RETURN>e.Z_RETURN:
        moving=f' (이동 Z 복귀보다 {reading_duration((e.FIXED_RETURN-e.Z_RETURN).total_seconds())} 늦음)'
    outside=len(peers)>0 and (e.N_BINS>peers.N_BINS.max() or abs(e.DELTA)>peers.DELTA.abs().max())
    status=('비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. ' if outside else '다른 LOT의 반복 응답 가설을 남기되, ' if len(peers) else '단독 변화의 원인을 열어 두고, ')
    return fixed+moving+'. '+status+'조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.'

def evidence_peer_table(e):
    ids,eligible,peers,exposure,phase=evidence_peer_context(e)
    if not ids:return '같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.'
    target=review_events.loc[review_events.LOT.eq(e.LOT)&review_events.SENSOR.eq(e.SENSOR)&review_events.KIND.eq(e.KIND)].copy()
    target=target.loc[[evidence_phase(r)==phase and np.sign(r.DELTA)==np.sign(e.DELTA) for r in target.itertuples()]]
    if e.KIND=='Z':own_exposure=int((states[e.LOT].eq(phase)&zstats[e.LOT,e.SENSOR].Z.notna()).sum())
    else:
        g=raw[e.LOT];own_exposure=int(g.loc[g.MINUTE.map(states[e.LOT]).eq(phase),e.SENSOR].count())
    rows=[]
    for k in [e.LOT]+ids:
        own=k==e.LOT
        part=target if own else peers.loc[peers.LOT.eq(k)]
        den=own_exposure if own else exposure[k]
        base=profile[k][e.SENSOR,'median'].where(states[k].eq(phase)&profile[k][e.SENSOR,'count'].ge(54)).median()
        name=f'{k} (현재 LOT)' if own else str(k)
        if den==0:
            rows.append([name,'비교 보류','해당 형태 유효 관측 없음','—','—']);continue
        amp='—' if part.empty else f'{part.DELTA.abs().min():.4g}~{part.DELTA.abs().max():.4g}'
        duration='—' if part.empty else (f'{int(part.N_BINS.min())}~{int(part.N_BINS.max())}창' if e.KIND=='Z' else f'{part.SPAN_SEC.min():g}~{part.SPAN_SEC.max():g}초 간격')
        rate=(f'{len(part)}건/{den}유효창 (100창당 {len(part)/den*100:.2f}건)' if e.KIND=='Z'
              else f'{len(part)}건/{den}실측점 (1만점당 {len(part)/den*10000:.2f}건)')
        rows.append([name,rate,amp,duration,f'{base:.5g}'])
    denominator=('Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.' if e.KIND=='Z'
        else 'CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.')
    route=('기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다.' if e.KIND=='Z'
        else 'CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다.')
    return (f'강종 {e.GRADE}·채널 {e.SENSOR}·동일 편차 방향·사후 관측 형태 {phase}로 제한했다. '
       '행은 같은 운전 조건이 확인된 정상 표본이 아니다. '+denominator+'\n\n'
       +zscore_md_table(['LOT','같은 방향 후보/비교 노출','절대 편차 범위','후보 길이','참고 중심 수준'],rows)
       +'\n\n'+route+' 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. '
       '과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.')

def evidence_counts(k,sensor):
    d=zstats[k,sensor];p=profile[k]
    n=p[sensor,'count'].fillna(0)
    history=d.X.shift(1).rolling(Z_WINDOW,min_periods=1).count()
    current_bad=n.lt(54);past_bad=n.ge(54)&history.lt(Z_MIN)
    other=d.Z.isna()&~current_bad&~past_bad
    assert int(current_bad.sum()+past_bad.sum()+other.sum())==int(d.Z.isna().sum())
    return [sensor,len(d),int(d.Z.notna().sum()),int(d.Z.abs().ge(3).sum()),int(d.Z.isna().sum()),int(current_bad.sum()),int(past_bad.sum()),int(other.sum())]

def reading_select(k,sensors=None,include_cp=False):
    z=reading_events(k,sensors);chosen=[];ids=set()
    def add(kind,e):
        if e.ID not in ids and len(chosen)<READING_MAX_CARDS:
            chosen.append((kind,e));ids.add(e.ID)
    cp=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP')]
    if include_cp and len(cp):
        e=next(cp.loc[[cp.DELTA.abs().idxmax()]].itertuples());add('분홍 · CP 원본 추적',e)
    persistent=z.loc[z.OBS_CLASS.eq('지속 이상 후보')].sort_values(['N_BINS','REPEATS','START'],ascending=[False,False,True])
    if len(persistent):add('주황 · 지속·반복 확인',next(persistent.itertuples()))
    used_zones={reading_zone(e.SENSOR) for _,e in chosen if e.KIND=='Z'}
    other=persistent.loc[~persistent.ID.isin(ids)&~persistent.SENSOR.map(reading_zone).isin(used_zones)]
    if len(other):add('주황 · 다른 존의 변화',next(other.itertuples()))
    recovery=[]
    for e in z.itertuples():
        if e.ID in ids or pd.isna(e.Z_RETURN):continue
        if pd.isna(e.FIXED_RETURN):recovery.append((True,float('inf'),e.START,e))
        elif e.FIXED_RETURN>e.Z_RETURN:recovery.append((False,(e.FIXED_RETURN-e.Z_RETURN).total_seconds(),e.START,e))
    if recovery:
        e=sorted(recovery,key=lambda x:(-int(x[0]),-x[1],x[2]))[0][3]
        add('복귀 추가 확인',e)
    if not chosen and len(z):add('노랑 · 짧은 국소 변화',next(z.sort_values(['REPEATS','N_BINS','START'],ascending=[False,False,True]).itertuples()))
    return chosen

def reading_cards(k,sensors=None,include_cp=False):
    lines=[]
    for i,(kind,e) in enumerate(reading_select(k,sensors,include_cp),1):
        unit='무차원' if e.KIND=='CP' else '℃' if e.SENSOR in TC else '%p'
        if e.KIND=='CP':
            fact=f'원본 편차 **{e.DELTA:+.4f}무차원**, {int(e.POINTS)}점·첫/끝 간격 {e.SPAN_SEC:g}초. 1초 단발 여부: '+('확인' if e.OBS_CLASS=='단발 스파이크' else '아님')+'.'
        else:
            rule=f'연속 **{int(e.N_BINS)}개 1분창**' if e.N_BINS>=3 else f'연속 {int(e.N_BINS)}창'
            fact=f'|Z|≥3 {rule}, 최근 30분 시작 {int(e.REPEATS)}회. 최대 |Z| **{e.MAX_ABS_Z:.2f}**, 과거 중심 대비 **{e.DELTA:+.3g}{unit}**.'
        lines += [f'**{i}. {kind} — {e.SENSOR} · {reading_interval(e)}**','',
          '**관측:** '+fact+'  ',
          '**다른 LOT:** '+evidence_peer_text(e)+'  ',
          '**왜 이렇게 보는가:** '+evidence_domain(e)+'  ',
          '**어떻게 볼 것인가:** '+evidence_next(e),'']
    return '\n'.join(lines)

def zscore_group_markdown(k,group_key):
    _,label,sensors=next(g for g in REVIEW_Z_GROUPS if g[0]==group_key)
    ev=reading_events(k,sensors);cp=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP')]
    persistent=int(ev.OBS_CLASS.eq('지속 이상 후보').sum())
    lines=[f'#### {label} · 표시한 변화의 해석','',
       f'**우선 볼 주황 후보 {persistent}건 · 짧은 노랑 후보 {len(ev)-persistent}건**. 센서별 관측 건수이며 고장 건수가 아니다.','']
    if group_key=='02_가열균열':
        lines+=['CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.','']
    lines += [reading_cards(k,sensors,include_cp=group_key=='02_가열균열') or '계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.','']
    if group_key=='02_가열균열' and k==240037:
        v=evidence_counts(k,'TC-Z3')
        lines += [f'**그림 숫자의 뜻(TC-Z3):** 전체 {v[1]}개 1분창 = 계산 가능 {v[2]}개 + 보류 {v[4]}개. '
          f'초과 {v[3]}개는 계산 가능한 {v[2]}개 중 일부이며, 연속창을 묶으면 TC-Z3 사건 {len(ev.loc[ev.SENSOR.eq("TC-Z3")])}건이다. '
          f'보류 {v[4]}개 = 현재 실측 54개 미만 {v[5]}개 + 현재창은 충족하지만 과거 유효 20창 미만 {v[6]}개. 고장 수가 아니다.','']
    selected=ev if group_key!='02_가열균열' else pd.concat([ev,cp],ignore_index=True)
    detail=[]
    for e in selected.sort_values(['START','SENSOR']).itertuples():
        detail += [f'**{e.ID} · {reading_interval(e)}**','',
           '**다른 LOT:** '+evidence_peer_text(e)+'  ',
           '**해석 가설:** '+evidence_domain(e)+'  ',
           '**추적:** '+evidence_next(e),'',evidence_peer_table(e),'']
    if detail:lines += [reading_fold(f'표시한 모든 후보 {len(selected)}건의 LOT 대조·해석·추적 근거 펼치기','\n'.join(detail)),'']
    counttext=('각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. '
       '초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. '
       '보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. '
       '54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.\n\n'
       +zscore_md_table(['센서','전체 1분창','계산 가능','그중 |Z|≥3','보류 합계','현재 <54점','현재 충족·과거 <20창','그 외'],[evidence_counts(k,s) for s in sensors]))
    lines += [reading_fold('창 수의 의미·계산 보류 이유 펼치기',counttext),'',
              reading_fold('원값·전체 통계 근거·회색 구간 상세 펼치기',reading_full_zscore_group_markdown(k,group_key))]
    return '\n'.join(lines)

def zscore_lot_intro(k):
    su=review_summary.set_index('LOT').loc[k]
    lines=[f'#### LOT {k} · 먼저 볼 변화','',
       f'**지속·반복 후보 {int(su.PERSIST_EVENTS)}건 / CP 원본 후보 {int(su.CP_EVENTS)}건**. 관측 결과와 설비 이상 판정은 구분한다.','',
       reading_cards(k,include_cp=True),
       '같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. '
       '비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.','',
       reading_fold('LOT 전체 집계·그림 읽는 법 펼치기',reading_full_zscore_lot_intro(k))]
    return '\n'.join(lines)

def zscore_cp_markdown(k):
    cp=review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP')].sort_values('PEAK')
    if cp.empty:return ''
    rows=[]
    for e in cp.itertuples():
        back='60개 연속 실제 1초 값 복귀 '+reading_time(e.FIXED_RETURN,True) if pd.notna(e.FIXED_RETURN) else '60개 연속 실제 1초 값 복귀 미확인'
        rows.append([e.ID+' · '+review_clock(e.PEAK),f'{e.DELTA:+.4f}무차원 · {int(e.POINTS)}점/{e.SPAN_SEC:g}초 간격',evidence_peer_text(e),back+'; '+reading_wo_relation(e)])
    return ('#### CP 원본 · 표시한 피크마다 남기는 근거\n\n'
        '**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.\n\n'
        +zscore_md_table(['피크·ID','원본 관측','동종 LOT 대조','복귀·정비 시간 관계'],rows)+'\n\n'
        +reading_fold('CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기',reading_full_zscore_cp_markdown(k)))


def evidence_cp_sensitivity():
    peaks=[]
    for k,g in raw.items():
        s=g.set_index('MEAS_DT')['CP-Z4'].dropna();dev=s-s.median()
        hit=dev.abs().gt(CP_GROUP)
        group=(hit.ne(hit.shift())|s.index.to_series().diff().dt.total_seconds().gt(5)).cumsum()
        peaks.extend([float(v.abs().max()) for _,v in dev[hit].groupby(group[hit])])
    return pd.DataFrame({'최대 절대 편차 경계':[.03,.05,.07],
       '선택 원본 사건 수':[sum(v>=limit for v in peaks) for limit in [.03,.05,.07]]})

def evidence_method_note():
    sensitivity=evidence_cp_sensitivity()
    rows=sensitivity.values.tolist()
    return (
       '\n\n### 표시한 후보에서 무엇을 확인하는가\n\n'
       '| 후보 | 확인할 변화 | 후속 대조 |\n|---|---|---|\n'
       '| 짧은 국소(노랑) | 1~2개 1분창의 변화 후 복귀하는가 | 원본 폭·전후 값·같은 채널의 재발; 1초 스파이크와 구분 |\n'
       '| 지속·반복(주황) | 한 방향의 수준 이탈, 상하 왕복, 짧은 변화의 재발 | 연속 ≥3창 또는 최근 30분 시작 ≥3회; 같은 방향 최장창·고정 대역 복귀·TC/OP 동반 확인 |\n'
       '| CP 원본(분홍) | 분 중앙값에 가려질 수 있는 순간·짧은 다점 이탈 | 원본 ID·전후 값·실측 점 수·첫/끝 간격·60개 연속 실제 1초 값 복귀를 보존 |\n\n'
       '표시한 구간에는 관측 → 동종 LOT 대조 → 도메인 가설 → 추적 방법을 적었다. '
       '우선 3건은 CP 원본, 최장 지속/반복, 다른 존의 지속 후보 순으로 고르고 남는 자리에 복귀 차이를 제시한다. '
       '모든 표시 후보의 해석과 원본 근거는 펼쳐 볼 수 있다. 회색 구간의 상세는 접었다.\n\n'
       '**CP 경계의 출처:** 기존 코드의 CP_GROUP=0.02, CP_LIMIT=0.05를 유지했다. '
       '공정 허용차·센서 정확도·정상 검증 표본에서 유도한 값이라는 근거는 저장소에서 확인되지 않았다. '
       '±0.02를 넘는 원본 묶음 중 최대 절대 편차가 0.05 이상인 사건을 골랐을 뿐이며 5% 상대 오차나 3σ가 아니다. '
       '기준은 전체 LOT 중앙값이므로 사후 EDA다. 공식 허용차/센서 분해능/물리량 정의 확보 후 조정 근거를 세워야 한다.\n\n'
       +zscore_md_table(['최대 절대 편차 경계(무차원)','선택 원본 사건 수'],[[f'{r[0]:.2f}',int(r[1])] for r in rows])
       +'\n\n위 표는 같은 ±0.02 묶음·공백 5초 분리 규칙을 고정하고 선택 경계만 바꾼 민감도 확인이다. '
       '전환 형태를 포함한 14 LOT 전체 사후 집계이며 최적 경계나 탐지 정확도를 뜻하지 않는다. 경계를 낮추면 더 많은 작은 변화를 검토하게 된다.\n\n'
       '**도메인 가설의 범위:** 가열 출력에 역동작 제어가 적용되어 있다면 TC 상승·OP 감소(또는 반대)는 보상 응답과 양립한다. '
       '열 공정에는 응답 지연·오버슈트가 있을 수 있어 온도와 출력의 선후·SV·자동/수동 모드를 확인해야 한다. '
       '이는 일반 온도 제어 원리를 이 자료에 적용해 검토하는 가설이며 이 설비의 제어 사양을 확인한 결과는 아니다. '
       '[Yokogawa 제어 방향 설명](https://www.yokogawa.com/us/library/resources/faqs/ns-faq-green-9017-term/), '
       '[Yokogawa 온도 제어 기술 자료](https://www.yokogawa.com/us/library/resources/yokogawa-technical-reports/ut100-series-of-temperature-controllers/).\n\n'
       '연속 소둔 비교에는 라인 속도·소재 규격·가열/냉각 조건도 필요하다. '
       '[EU 연속 소둔 공정 연구](https://op.europa.eu/en/publication-detail/-/publication/96d5a790-edb0-495b-9acf-eeaa7ddb2399). '
       '현재는 같은 강종·채널·편차 방향·사후 관측 형태로 제한한 기술 비교이고, 설정값·속도·규격·모드는 없다. '
       '다른 LOT에도 반복된다는 사실은 정상의 충분조건이 아니며 공통 제어 문제도 가능하다. '
       '비교보다 큰 폭/길이·미복귀는 우선 검토하고, 비교 이내이며 복귀가 확인되면 반복 공정/제어 응답 가설을 남긴다. 설비 판정은 보류한다.\n\n'
       'CP의 단위는 무차원이다. 물리량·CP-Z4M의 생성 관계·정비 부품 대응은 '
       '[저장소 데이터 계약](../../docs/02-data-contract.md)에서도 미확인 항목이므로 탄소 포텐셜·압력 등으로 임의 해석하지 않았다.\n'
    )


def evidence_pair_pattern(e):
    if e.ID in _evidence_pattern_cache:return _evidence_pattern_cache[e.ID]
    other=evidence_pair(e)
    if other is None:result=None
    else:
        x=zstats[e.LOT,e.SENSOR].X
        before=x.loc[max(0,int(e.A)-10):int(e.A)-1].dropna()
        now=x.loc[int(e.A):int(e.B)-1].dropna()
        if len(before)<5 or now.empty:result=None
        else:
            first=int(np.sign(now.median()-before.median()));second=int(np.sign(other[1]))
            result=(first,second) if e.SENSOR in TC else (second,first)
    _evidence_pattern_cache[e.ID]=result
    return result


<a id="overview"></a>
## 01 Executive Summary — 핵심 결과와 후속 확인

In [ ]:
note('### 보고서에서 먼저 볼 세 가지 결과')
show_table(pd.DataFrame([
 {'확인한 사실':'같은 SPHC의 OP-Z4 수준 차이','근거':'240061 중앙값 54.13%, 다른 3 LOT 41.82·42.12·44.37%','판단':'설정·부하 확인 대상. 온도 이상·고장 확정 아님'},
 {'확인한 사실':'240069 CP 원본 7사건','근거':'최대 +0.459, 이후 BM REMARK: 1호기 재발·절연 저하','판단':'시간상 선행. 원인·수리 효과는 미확인'},
 {'확인한 사실':'240117 CP 하강은 CBM 완료 뒤','근거':'REMARK 마모량 초과, 완료 12:44 → CP 최저 13:04:50','판단':'해당 CBM을 유발한 사전 경보로 해석 불가'}]))
rows=[{'panels':[('같은 SPHC | TC-Z4 분포',['TC-Z4']),('같은 SPHC | OP-Z4 분포',['OP-Z4'])],'height':2.1},
      {'panels':[('240069 | CP 반복 이탈 → 이후 BM',['CP-Z4']),('240117 | CBM 완료 뒤 CP 하강',['CP-Z4'])],'height':2.1}]
fig,cells=report_page('CAL01 · 보고서 핵심 근거 3가지',
 '위: 같은 강종의 온도·출력 분포 / 아래: CP 원본과 실제 정비 시각. 빨강은 CP 탐색 경계 이탈이며 고장 확정이 아니다.',rows,width=24)
ids=lots.loc[lots.GRADE.eq('SPHC'),'LOT_NO'].astype(int).tolist()
for ax,head,sigs in cells[0]:draw_boxes(ax,head,sigs[0],ids,highlight=240061)
for (ax,head,_),k in zip(cells[1],[240069,240117]):
    r=lot_meta.loc[k];report_signal(ax,k,'CP-Z4',fine=True)
    base=raw[k]['CP-Z4'].median()
    for value in [base-.05,base+.05]:ax.axhline(value,color=EVENT_COLORS['CP'],ls='--',lw=.8)
    events=cp_real.loc[cp_real.LOT.eq(k)];peak=events.loc[events.DEV.abs().idxmax()]
    clock_ticks(ax,r.START,r.END);ax.set_ylabel('CP-Z4 [무차원]');panel_legend(head,ax)
    near_note(ax,peak.PEAK,base+peak.DEV,f'{peak.PEAK:%H:%M:%S}\nCP 편차 {peak.DEV:+.3f}',EVENT_COLORS['CP'])
    if k==240117:
        done=cal_work.loc[cal_work.WO_NO.eq('WO-2024-01854'),'END_DT'].iloc[0]
        ax.axvline(done,color=COL['CBM'],ls='-.',lw=1)
emit(fig,'01_개요/00_보고서_핵심근거.png')
note('**읽은 결과:** 240061은 TC-Z4와 OP-Z4를 분리해 조건 차이를 확인해야 한다. 240069는 CP가 BM 전에 발생했지만 작업 후 관측이 없다. 240117은 CP가 CBM 후에 발생했다. 아래는 이 판단을 만든 일정·A/B 수리 대조와 구간별 근거다.')

**먼저 판단할 것:** 240069·240037의 CP는 반복성·지속·복귀를, 240117은 실제 CBM 완료와의 선후 관계를, 240061은 같은 강종의 온도·출력 수준을 확인한다. 계획수리 전후의 공통 하강·승온은 운전 전환 가설을 우선 검토한다. 핵심 사례와 남은 가설은 5절, 점검·데이터 확보 항목은 7절로 연결한다.

**추가 관계 분석의 핵심 결과**

- CP 14사건에서 Z3뿐 아니라 Z4~Z6도 모두 비교했다. CP 피크 시각의 TC-Z4~Z6 변화는 상대적으로 작지만, CP 이탈이 없던 같은 LOT의 다른 때보다 이탈 때 변화폭이 큰 경우가 TC-Z4 **10/12**, TC-Z5 **9/12**, TC-Z6 **8/12**였다. 작은 반복 반응을 함께 확인할 근거다.
- CP 하강 2사건에서는 TC-Z3 최고점이 CP 최저점보다 약 6분 앞에 있고 OP-Z3는 낮아졌다가 올라오는 양상이 반복됐다. CP 최저 부근의 TC-Z4·Z6는 앞 기준보다 낮고 TC-Z5는 높은 방향이다. 하강 사례 2개에서 본 관측이며 원인·정상 여부를 확정하지 않는다.
- 정상 제어·계측 변화·고장 원인을 구분하려면 CP 정의·SV·속도·부하·정비 진단과 조치 직전직후 자료가 필요하다. [사건별 관계](#relation-events)·[CP 전후 파형](#relation-lags)·[CP 이탈 유무 비교](#relation-controls)·[채널·정비](#relation-maintenance)에 그래프와 코드, 관찰·반례를 붙였다.
- **TC–OP 관계는 [5.7.1 독립 절](#relation-tc-op)에서 확인한다.** 같은 존의 온도·출력은 14 LOT의 짧은 동시 변화에서 주로 역방향이었다. 실제 파형과 전체 LOT 표를 함께 보며 선후·원인 방향은 분리하지 못했다.


TC–OP의 방향성 다음에는 [출력을 높인 뒤 온도 복귀](#tc-op-response)를 확인한다. ‘고출력인데 TC가 내려갔다’는 모양의 반복성과 240069 두 반응의 복귀 차이를 구분한다. SV 없는 복귀는 과거 관측 수준에 대한 묘사이며, [품질 검증에 필요한 근거](#tc-quality-evidence)는 별도로 정리했다.


<a id="data-scope"></a>
## 02 분석 목적 및 데이터 특성

**목적:** 어떤 시각·신호가 평소 또는 다른 LOT과 다르게 관측됐는지 찾고, 지속·반복·복귀와 정비의 실제 시각으로 변화의 성격을 좁힌다.

**관측 범위:** 연간 전체를 연속 측정한 자료가 아니다. LOT 사이 관측 공백은 온도 추세로 연결하거나 정상·정지로 채우지 않는다. LOT 관측창과 정비 감지·착수·완료의 교집합만 실제 기록으로 비교한다.

**분석팀이 확인한 미정의 항목:** CP-Z4·CP-Z4M의 물리량의 정의·측정/계산 방식과 계측점 대응이 현재 정의되지 않은 상태다. INS-006(가스 분석계 셀)과 각 CP 채널의 연결도 확인되지 않았다. CP의 통계는 제공된 관측값을 기준으로 읽는다.

In [ ]:
gaps=pd.concat([g.MEAS_DT.diff().dt.total_seconds().dropna() for g in raw.values()])
show_table(pd.DataFrame([
 {'항목':'온도 자료','범위':f'{len(temp):,}행 · {len(lots)} LOT · {lots.GRADE.nunique()}강종 · {len(SIGNALS)}신호'},
 {'항목':'관측 날짜 범위','범위':f'{temp.MEAS_DT.min():%Y/%m/%d}–{temp.MEAS_DT.max():%Y/%m/%d}의 분리된 LOT 관측창'},
 {'항목':'수집 간격','범위':f'LOT 내 양의 시각 차이 최빈값 {gaps.loc[gaps.gt(0)].mode().iloc[0]:g}초 · 누락 시각은 보간하지 않음'},
 {'항목':'CAL01 정비','범위':f'{len(cal_work)} WO · TBM 22 / BM 12 / CBM 10'},
 {'항목':'실제 작업과 관측 교집합','범위':'240117의 2 WO · 합집합 304분'},
 {'항목':'정기수리 중 온도','범위':'6월·11월 계획수리 날짜 안에는 제공 온도 0행'},
 {'항목':'추가로 필요한 자료','범위':'정상 기준·SV·라인 속도·소재 두께/부하·존 간 거리·품질 결과'}]))
table(lots[['LOT_NO','GRADE','START','END','MINUTES','N_ROWS']])

<a id="methods"></a>
## 03 전처리·탐지 방법과 판단 범위

| 경로 | 입력·기준 | 찾는 것 | 이 경로만으로 판단하지 않는 것 |
|---|---|---|---|
| LOT 추세 | 실제 기록 54개 이상인 1분창 중앙값 | 분 단위 중심 수준·함께 움직임 | 순간 최고·최저 |
| 시간순 Z | 이전 30분 중앙값·SD, 최소 20개 과거 유효창, 절대 Z≥3 | 이전 구간 대비 국소 변화 | 검증된 고장·정상 확률 |
| CP 원본 | LOT 원본 중앙값 편차 0.02 초과 묶음 중 최대 0.05 이상 | 짧은 펄스·반복 이탈 | 계측 오류·공정 원인 확정 |
| IQR | LOT·신호별 전환 제외 1분 중앙값, Q1−1.5IQR~Q3+1.5IQR | LOT 내부 분포의 드문 창·분포 폭 | 같은 강종의 정상 한계 |
| 같은 강종 수준 | LOT별 중앙값을 동일 가중으로 대조 | 분포 전체의 높낮이 차이 | 운전 조건 일치·에너지 증가·고장 |

**지속시간을 구분한다.** 확대창의 폭은 설명을 위한 문맥이며 이상 지속시간이 아니다. CP는 원본 이탈의 첫·끝 기록 간격과 실측 점 수를 함께 제시한다. 피크 간격은 반복의 기술량이며 주기 확정이 아니다.

**복귀도 관측 규칙으로만 기술한다.** CP는 이후 60개 연속 실제 1초 기록이 LOT 기준 ±0.02 안에 들어온 첫 확인 시각을 제시한다. Z는 3개 연속 유효 1분창이 절대 Z<3으로 들어오는지 확인한다. 두 규칙은 사후 비교를 위한 기술 기준이며 고장 수리 완료·정상 보증이 아니다. Z 기준선 자체의 이동도 복귀 판독에 영향을 준다.


### Z-score 구간 그림과 근거표를 읽는 기준

기존 1분 중앙값과 과거 30분 Z 계산은 유지했다. 현재창에 실측 54개 이상, 과거 30분에 유효창 20개 이상이 있을 때만 평가한다. 과거 중심은 중앙값, 분모는 표준편차이며 하한은 TC 0.5℃·OP 2%p다. **절대 Z≥3은 국소 변화 후보이지 고장 판정이나 고장 확률이 아니다.**

| 표시 | 이 노트북에서 확인하는 근거 | 처리 |
|---|---|---|
| 정상 범위 참고 | 해당 센서의 유효창에서 절대 Z<3. LOT 요약은 17신호 모두 평가 가능한 창만 집계 | 설비 정상 판정과 구분 |
| 단발 스파이크 | 원본 1초 격자의 단일 관측 이탈이며 직전·직후 실측이 복귀 | 원값·시각·행 번호를 보존. 여러 점의 펄스와 구분 |
| 지속 이상 후보 | 같은 센서에서 절대 Z≥3이 3개 이상 연속되거나, 해당 시작을 포함한 최근 30분에 3회 이상 시작 | 연속창·같은 방향 최장창·반복 횟수·복귀를 근거로 우선 검토 |
| 판정 보류 | 현재/과거 관측 부족, 짧은 국소 후보의 성격 미확정, SV·운전 모드·관련 점검 결과 부족 | 필요한 확인 항목을 구간별로 명시 |

노란 음영은 1~2개 창의 **짧은 국소 후보**다. 이것만으로 1초 스파이크나 정상으로 분류하지 않는다. 회색은 점수를 계산할 수 없는 구간이다. 주황·노랑은 통계적 관측 구분이며 실제 설비 판정은 별도다.

**복귀 확인:** 이동 Z가 3개 유효창 연속으로 기준 안에 든 때와, 사건 시작 시 중심±3SD를 고정한 기준에 3개 유효창 연속으로 든 때를 함께 적었다. 이동 기준의 적응으로 Z만 작아지는 경우를 구분한다. 연속 1분창 3개가 원자료의 180초 연속 이탈을 뜻하지는 않는다.

**그림과 근거의 대응:** 각 LOT에 세정·건조, 가열·균열, 과시효의 17개 Z 채널을 모두 표시하고, 음영에 해당하는 모든 센서별 사건을 바로 아래 마크다운 표에 남겼다. Z를 적용하지 않는 CP 두 채널과 Z9·냉각 3채널도 LOT별 원본·편차 그림에 표시했다. CP는 무차원이며 원본 이탈과 1분 Z 결과를 별도로 추적한다.


**정비 대조:** 같은 설비의 감지·착수·완료를 실제 시각으로 대조한다. LOT 전후 72시간은 탐색 범위다. 정비 경계 전후 5분에 양쪽 190개 이상 실측이 있을 때만 중앙값 차이를 제시하며, 인과·수리 효과는 별도 검토한다. 각 LOT의 정비 근거표에서 관측 사실과 경쟁 가설·확인 자료를 구분했다.


**CP와 가열·균열의 동시 비교:** 가열·균열 그림 위 2행에 CP-Z4·CP-Z4M 원본을, 이어지는 8행에 Z3~Z6의 TC·OP를 같은 LOT 경과시간으로 표시했다. 분홍 세로선은 CP-Z4 피크 대조 시각, 보라 선은 WO 실제 시각이다. CP 피크가 다른 신호의 이상이라는 뜻은 아니다.

**핵심부터 읽는 방법:** LOT 시작과 공정 그림 아래에는 원본 CP, 센서별 지속·반복, 다른 존의 지속 후보, 복귀 확인 차이/미확인 중 최대 3건을 먼저 제시한다. 지속 후보는 연속창 수·반복 횟수, 복귀 항목은 미확인 여부·확인 시각 차이를 기준으로 고른다. 최대 Z 순위만으로 나열하지 않는다. 이것은 읽는 순서이며 탐지 기준·설비 이상 등급을 새로 계산하는 규칙이 아니다. 건수는 센서별 관측 후보 수이며 겹치는 시간창을 합친 고장 건수가 아니다. 전체 사건·정확한 수치·보류 구간·WO 기록은 각 절의 '상세 근거 펼치기'에 보존했다.

겹치는 TC·OP 후보는 같은 시간대의 비교 근거로 덧붙였다. 서로 다른 센서의 창을 합쳐 지속·반복 판정을 만들지는 않았다.

### 표시한 후보에서 무엇을 확인하는가

| 후보 | 확인할 변화 | 후속 대조 |
|---|---|---|
| 짧은 국소(노랑) | 1~2개 1분창의 변화 후 복귀하는가 | 원본 폭·전후 값·같은 채널의 재발; 1초 스파이크와 구분 |
| 지속·반복(주황) | 한 방향의 수준 이탈, 상하 왕복, 짧은 변화의 재발 | 연속 ≥3창 또는 최근 30분 시작 ≥3회; 같은 방향 최장창·고정 대역 복귀·TC/OP 동반 확인 |
| CP 원본(분홍) | 분 중앙값에 가려질 수 있는 순간·짧은 다점 이탈 | 원본 ID·전후 값·실측 점 수·첫/끝 간격·60개 연속 실제 1초 값 복귀를 보존 |

표시한 구간에는 관측 → 동종 LOT 대조 → 도메인 가설 → 추적 방법을 적었다. 우선 3건은 CP 원본, 최장 지속/반복, 다른 존의 지속 후보 순으로 고르고 남는 자리에 복귀 차이를 제시한다. 모든 표시 후보의 해석과 원본 근거는 펼쳐 볼 수 있다. 회색 구간의 상세는 접었다.

**CP 경계의 출처:** 기존 코드의 CP_GROUP=0.02, CP_LIMIT=0.05를 유지했다. 공정 허용차·센서 정확도·정상 검증 표본에서 유도한 값이라는 근거는 저장소에서 확인되지 않았다. ±0.02를 넘는 원본 묶음 중 최대 절대 편차가 0.05 이상인 사건을 골랐을 뿐이며 5% 상대 오차나 3σ가 아니다. 기준은 전체 LOT 중앙값이므로 사후 EDA다. 공식 허용차/센서 분해능/물리량 정의 확보 후 조정 근거를 세워야 한다.

| 최대 절대 편차 경계(무차원) | 선택 원본 사건 수 |
| --- | --- |
| 0.03 | 24 |
| 0.05 | 15 |
| 0.07 | 11 |

위 표는 같은 ±0.02 묶음·공백 5초 분리 규칙을 고정하고 선택 경계만 바꾼 민감도 확인이다. 전환 형태를 포함한 14 LOT 전체 사후 집계이며 최적 경계나 탐지 정확도를 뜻하지 않는다. 경계를 낮추면 더 많은 작은 변화를 검토하게 된다.

**도메인 가설의 범위:** 가열 출력에 역동작 제어가 적용되어 있다면 TC 상승·OP 감소(또는 반대)는 보상 응답과 양립한다. 열 공정에는 응답 지연·오버슈트가 있을 수 있어 온도와 출력의 선후·SV·자동/수동 모드를 확인해야 한다. 이는 일반 온도 제어 원리를 이 자료에 적용해 검토하는 가설이며 이 설비의 제어 사양을 확인한 결과는 아니다. [Yokogawa 제어 방향 설명](https://www.yokogawa.com/us/library/resources/faqs/ns-faq-green-9017-term/), [Yokogawa 온도 제어 기술 자료](https://www.yokogawa.com/us/library/resources/yokogawa-technical-reports/ut100-series-of-temperature-controllers/).

연속 소둔 비교에는 라인 속도·소재 규격·가열/냉각 조건도 필요하다. [EU 연속 소둔 공정 연구](https://op.europa.eu/en/publication-detail/-/publication/96d5a790-edb0-495b-9acf-eeaa7ddb2399). 현재는 같은 강종·채널·편차 방향·사후 관측 형태로 제한한 기술 비교이고, 설정값·속도·규격·모드는 없다. 다른 LOT에도 반복된다는 사실은 정상의 충분조건이 아니며 공통 제어 문제도 가능하다. 비교보다 큰 폭/길이·미복귀는 우선 검토하고, 비교 이내이며 복귀가 확인되면 반복 공정/제어 응답 가설을 남긴다. 설비 판정은 보류한다.

CP의 단위는 무차원이다. 물리량·CP-Z4M의 생성 관계·정비 부품 대응은 [저장소 데이터 계약](../../docs/02-data-contract.md)에서도 미확인 항목이므로 탄소 포텐셜·압력 등으로 임의 해석하지 않았다.


In [ ]:
from IPython.utils.capture import capture_output
with capture_output() as captured:
    
    # 1) 본문과 동일한 9개 비주기 온도 채널의 1분 Z를 사용한다.
    # "계산 가능"은 최소 1개 채널에 점수가 있다는 뜻이며, 모든 채널의 정상 보장이 아니다.
    quality_effect_rows=[]
    for k in lots.LOT_NO.astype(int):
        z=pd.DataFrame({s:zstats[k,s].Z for s in NONCYCLIC})
        counts=profile[k].xs('count',axis=1,level=1)[NONCYCLIC]
        current_ok=counts.ge(54).any(axis=1)
        scored=z.notna().any(axis=1);flag=z.abs().ge(Z_LIMIT).any(axis=1)
        # 품질 조건을 생략한 경우도 같은 식으로 계산해 후보 위치 변화만 비교한다.
        unfiltered=[]
        for s in NONCYCLIC:
            y=profile[k][(s,'median')];past=y.shift(1)
            center=past.rolling(Z_WINDOW,min_periods=Z_MIN).median()
            sd=past.rolling(Z_WINDOW,min_periods=Z_MIN).std().clip(lower=.5)
            unfiltered.append(((y-center)/sd).abs().ge(Z_LIMIT))
        unfiltered_flag=pd.concat(unfiltered,axis=1).any(axis=1)
        quality_effect_rows.append({'LOT':k,'GRADE':lot_meta.loc[k,'GRADE'],'전체 창':len(z),
            '계산 후 3미만':int((scored&~flag).sum()),'3이상 후보':int(flag.sum()),
            '현재 기록 부족':int((~scored&~current_ok).sum()),'과거 기준 부족':int((~scored&current_ok).sum()),
            '조건 미적용 후보':int(unfiltered_flag.sum()),'조건 적용 후 제외':int((unfiltered_flag&~flag).sum()),
            '조건 적용 후 추가':int((flag&~unfiltered_flag).sum())})
    quality_effect=pd.DataFrame(quality_effect_rows)
    status_columns=['계산 후 3미만','3이상 후보','현재 기록 부족','과거 기준 부족']
    assert quality_effect[status_columns].sum(axis=1).eq(quality_effect['전체 창']).all()
    assert quality_effect['3이상 후보'].sum()==zscreen.MAX_ABS_Z.ge(Z_LIMIT).sum()
    qt=quality_effect.sum(numeric_only=True)
    
    note('**계산 가능 범위:** 기록 부족과 초기 기준 부족은 판단을 보류한다')
    fig,cells=report_page('온도 변화: 실제로 계산한 창과 판정을 보류한 창',
        '색별 길이=상태별 창 수 합계 · 실제 발생 시각은 8절 LOT에서 확인 · 9개 비주기 온도의 계산된 Z 사용 · 3미만도 정상 확정은 아님',
        [{'panels':[('14 LOT · 각 1분 집계창을 계산 결과로 분류',[])],'height':5.6}])
    ax,head,_=cells[0][0];yy=np.arange(len(quality_effect));left=np.zeros(len(yy))
    status_colors=['#4a92aa',EVENT_COLORS['Z'],'#b6c1cb','#e4d1ab']
    status_labels=['계산된 신호 3미만','3이상 검토 후보','기록 부족: 보류','과거 기준 부족: 보류']
    for col,color,label in zip(status_columns,status_colors,status_labels):
        values=quality_effect[col].to_numpy();ax.barh(yy,values,left=left,height=.66,color=color,label=label);left+=values
    for i,r in enumerate(quality_effect.to_dict('records')):
        held=r['현재 기록 부족']+r['과거 기준 부족']
        ax.text(r['전체 창']+12,i,f"후보 {r['3이상 후보']} / 보류 {held}",ha='left',va='center',fontsize=9,color='#34495e')
    ax.set_yticks(yy,[f"{r.LOT} · {r.GRADE}" for r in quality_effect.itertuples()],fontsize=9)
    ax.set_ylim(len(yy)-.4,-.6);ax.set_xlim(0,float(left.max())*1.32);ax.set_xlabel('1분 집계창[개] · 끝부분의 짧은 창 포함');panel_legend(head,ax)
    fig._folded='계산 가능한 관측창·보류 범위·Z 후보 수 확인 펼치기'
    emit(fig,'02_선별/01_관측품질.png')
    note(f"**이 그림의 결과:** 전체 {int(qt['전체 창']):,}개 집계창 중 {int(qt['계산 후 3미만']+qt['3이상 후보']):,}개에서 최소 한 온도 채널의 Z를 계산했다. {int(qt['현재 기록 부족'])}개는 기록 수가 부족하고, {int(qt['과거 기준 부족'])}개는 과거 기준이 부족해 판정을 보류했다. 계산된 창 중 **{int(qt['3이상 후보'])}개가 3 이상 검토 후보**다.")
    note('**그래서 무엇을 하는가:** 회색·베이지 구간을 정상으로 채우거나 그 구간에서 고장이 없었다고 결론 내리지 않는다. 특히 각 LOT 초기 20분은 온도 기록이 있어도 이전 기준이 없어 Z를 판정할 수 없다. 주황 구간은 뒤의 LOT 원본 확대와 정비 시각 대조 대상으로 넘긴다. 이 그림은 모든 9개 센서가 동시에 평가되었다는 뜻은 아니다.')
    note(f"**품질 조건이 실제로 바꾼 것:** 조건 미적용 시 {int(qt['조건 미적용 후보'])}개 후보였으나, 기록 조건을 반영하면 {int(qt['3이상 후보'])}개다. 기존 후보 {int(qt['조건 적용 후 제외'])}개가 빠지고 {int(qt['조건 적용 후 추가'])}개가 추가된다. 현재 값뿐 아니라 과거 중앙값·표준편차도 달라지기 때문이다. 이를 진짜·가짜 이상 개수로 해석하지 않고, 근거가 있는 점수와 계산 보류 범위를 구분하는 데 사용한다.")
    table(quality_effect)
    
    
    # 240069의 CP 후보에 필요한 기록 수만 검증한다. 별도 공백 그림은 만들지 않는다.
    gap_lot=240069
    cp_window_counts=profile[gap_lot][('CP-Z4','count')]
    candidates=cp_real.loc[cp_real.LOT.eq(gap_lot)]
    peak_counts=pd.Series({r.PEAK:int(cp_window_counts.loc[int(r.M)]) for r in candidates.itertuples()})
    assert len(peak_counts)==7 and peak_counts.ge(54).all()
    
    
    # 2) 실제 CP 사건으로 평균/중앙값의 의미와 짧은 피크를 별도 처리하는 이유를 보여 준다.
    # 최대값이 든 한 분만이 아니라 사건이 걸친 모든 유효한 분의 중앙값을 검사한다.
    cp_aggregation_rows=[]
    for ev in cp_real.itertuples():
        base=float(raw[int(ev.LOT)]['CP-Z4'].median())
        bins=profile[int(ev.LOT)].loc[int(ev.A):int(ev.B)]
        usable=bins[('CP-Z4','count')].ge(54)
        med=bins[('CP-Z4','median')].where(usable)
        cp_aggregation_rows.append({'LOT':int(ev.LOT),'PEAK':ev.PEAK,'RAW_DEV':float(ev.DEV),
            'USABLE_MINUTES':int(usable.sum()),'MEDIAN_ABS_MAX':float((med-base).abs().max()),
            'MEDIAN_EVENT_HIT':bool((med-base).abs().ge(CP_LIMIT).any())})
    cp_aggregation_review=pd.DataFrame(cp_aggregation_rows)
    assert cp_aggregation_review.USABLE_MINUTES.gt(0).all()
    missed=cp_aggregation_review.loc[~cp_aggregation_review.MEDIAN_EVENT_HIT]
    
    cp_quality_evidence=cp_aggregation_review.copy()
captured_details('계산 가능 범위·기록 조건과 후보 수 검증 펼치기',captured)

**계산에 반영한 조건:** 기록 부족 242개 창과 과거 기준 부족 280개 창은 온도 Z를 판정하지 않는다. CP는 모든 LOT의 원본에 같은 0.05 기준을 적용한다. 계산 정의는 마지막의 접힌 설명에서 확인한다.

### 3.1 왜 기본 검토선을 3으로 두는가?

일반적인 관리도는 **관리 상태의 평균을 중심으로 ±3σ**를 출발점으로 쓴다. 4를 기본값으로 삼을 검증 근거가 없어, 이 노트북의 **국소 변화 기본 검토 배수도 3**으로 바꾼다. 4·5는 민감도 비교로 유지한다. [NIST: 관리도의 중심선과 3σ 한계](https://www.itl.nist.gov/div898/handbook/pmc/section3/pmc32.htm)

**전통적인 3σ와 이 분석의 차이를 구분한다.** 이 분석은 요청한 시간순 중앙값 비교를 유지한다. 현재 분의 1분 중앙값을 \(x_t\), 현재 분을 제외한 **이전 30분의 중앙값**을 \(m_t\), 그 과거 1분 값들의 표준편차를 \(s_t\)로 둔다. 과거 유효 분이 20개 이상일 때만 계산한다.

\[
Z_t=\frac{x_t-m_t}{\max(s_t,s_{\min})},\qquad
L_t=m_t-3\max(s_t,s_{\min}),\quad U_t=m_t+3\max(s_t,s_{\min})
\]

따라서 그림의 범위는 **중앙값 중심 ±3SD 검토선**이다. 독립적으로 확보한 정상 평균과 σ로 만든 고정 관리한계가 아니며, 일반 평균 Z나 MAD 기반 robust Z도 아니다. 이를 근거로 정상 99.73% 보장·고장 확률·검증된 오경보율을 주장하지 않는다.

`shift(1)`로 현재 분을 과거 기준에서 제외한다. LOT 시작 20분은 기준이 부족해 비워 둔다. 작은 분산이 점수를 과도하게 키우지 않도록 표준편차 하한을 온도 0.5℃, 출력 2%p로 둔다. 이 하한도 탐색용 설정이며 센서 사양이나 공정 허용오차가 아니다. σ는 원본 1초 값이 아니라 과거의 **1분 중앙값들**에서 계산한다.

| 검토 기준 | 쓰는 목적 | 판단 범위 |
|---|---|---|
| 중앙값 중심 절대 Z ≥ 3 | LOT 내부 국소 변화의 기본 표시 | 이상 의심 후보, 고장 확정 아님 |
| 배수 4·5 | 3과 비교한 후보 수 변화 | 기본 후보를 숨기는 조건으로 쓰지 않음 |
| CP 원본과 LOT 중앙값 차이 ≥ 0.05 | 짧은 CP 펄스의 별도 검토 | 3σ가 아닌 무차원 값의 보조 기준, 전체 LOT를 쓴 사후 탐색 |
| CP 사건 묶음 | 0.02 초과 구간 중 0.05 도달, 5초 초과 공백에서 분리 | 보간하지 않음, 사건 길이는 기록 범위 |
| 같은 강종 수준 | 안정 참고의 LOT별 중앙값 비교 | 설정·속도·두께·부하가 일치한 대조는 아님 |
| 온도 2.5℃ / 출력 4%p 차이 | LOT 간 수준 차이의 보조 경계 | 3σ와 별개이며 공정 허용 한계 아님 |

3을 넘었다는 사실은 **확인할 위치를 찾은 것**이다. 이후 LOT 원본 확대에서 지속·반복·회복 양상과 TC–OP 반응을 보고, 실제 WO 감지·착수·완료 전후 관측으로 해석한다. 정지·승온과 강종 차이가 같은 분산을 따른다고 가정하지 않는다.

**안정 참고:** LOT 전체의 승온·하강 형태를 사후 분리하고 남은 구간이다. 검증된 정상 구간이나 실제 조업 모드가 아니며, 전환 분류에 미래 관측을 사용했다.

### 3.2 임계값 민감도와 사후 선별의 한계

3·4·5 배수와 CP 0.04·0.05·0.06의 후보 수 비교는 4.2절의 접힌 계산표에 제공한다.

**관찰:** 기본 검토선 |Z|≥3은 14 LOT의 356개 1분 집계창·155개 연속 구간을 표시한다. 비교 기준 |Z|≥4에서는 12 LOT의 93개 1분 집계창·52개 구간이다. CP 0.05는 전환 후보를 제외한 6 LOT의 14개 사건이다. 따라서 임계 초과 LOT 수와 결합 근거 우선 검토 8 LOT은 같은 집계가 아니다. 8 LOT은 반복 CP·작업 완료 뒤 CP·같은 강종 수준 차이를 함께 고려하여 사후에 지정한 검토 우선순위이며, 자동으로 정상·이상을 나눈 결과가 아니다. 모든 14 LOT은 뒤에서 같은 순서로 검토한다.

경계를 높이면 후보 수가 줄어드는 정도를 수치로 확인한다. ‘표시 1분창’은 계산된 비주기 온도 신호 중 하나라도 경계를 넘은 집계창 수이며, ‘사건 수’는 이 분들의 연속 구간 수다. CP는 원본 펄스 사건 수이므로 Z 사건 수와 직접 비교하지 않는다.

**가설 검토:** ‘3 또는 0.05가 이 자료에서 통계적으로 최적인 값이다’는 가설을 검증할 정답 데이터가 없다. 임계값 선택에 민감한 사례와 여러 기준에서 유지되는 사례를 나누어 원본 검토 우선순위에 쓴다. 반복 CP(2회 이상), 작업 완료 뒤 CP, 큰 강종 수준 차이를 함께 고려하여 검토 우선순위를 정했다. 240108도 같은 온도 기준으로 검사하고 부록 상세 검토에 포함한다. 본문 사례는 변화의 성격과 정비 시각을 설명하는 역할로 골랐다.

**판단 범위:** 이 자료를 본 뒤 정한 수치와 선별 결과다. 민감도는 오경보율 검증이나 독립 검증이 아니다. 정규성·자기상관을 무시한 ‘3σ 확률’로 해석하지 않는다.

<a id="screening"></a>
## 04 전체 시계열 특성과 탐지 결과

전체 관측 위치와 정비 기록을 먼저 확인한 뒤 14 LOT 검토표로 상세 사례를 선별한다. 우선 검토 사례는 반복 CP, 작업 완료 뒤의 CP, 같은 강종 수준 차이, 계획수리 전후 전환이라는 서로 다른 질문을 대표한다. 다른 LOT을 정상으로 판정해 제외한 것은 아니다.

### 4.1 연간 관측·정비와 공정별 검토

In [ ]:
focus = set(context.loc[context.CAL_WORK.gt(0)|context.CAL_DETECT.gt(0)|context.PLAN_RELATION.ne(""),"LOT"])
# 행과 주석 사이에 여백을 확보하되, 연간 날짜축의 가로 폭은 유지한다.
fig, ax = plt.subplots(figsize=(30,19.5))
fig.subplots_adjust(left=.14,right=.977,top=.83,bottom=.12)
rows = {"LOT":9,"GRADE":8,"BM":7,"CBM":6,"TBM":5,"DETECT":4,"INS-006":3,"SEL-006":2,"SEL-009":1,"PLAN":0}
for label,y in rows.items():
    color = COL.get(label,COL["PART"])
    ax.axhspan(y-.09,y+.09,color=color,alpha=.055,lw=0)
for p in plans.itertuples():
    ax.axvspan(p.STRT_DT,p.END_EXCL,color=COL["PLAN"],alpha=.65,lw=0)
    ax.plot([p.STRT_DT,p.END_EXCL],[0,0],color=COL["LOT"],lw=5,solid_capstyle="butt")
for r in grade_evt.itertuples():
    ax.plot(r.EVT_DT,8,marker="|",color=COL["GRADE"],ms=8,mew=.9)
for t in cal_work.itertuples():
    y = rows[t.MNT_TYPE]
    ax.plot([t.STRT_DT,t.END_DT],[y,y],color=COL[t.MNT_TYPE],lw=2)
    ax.plot(t.STRT_DT,y,"x",color=COL[t.MNT_TYPE],ms=6)
    ax.plot(t.DETC_DT,4,"v",color=COL["DETECT"],ms=5,alpha=.75)
# 동일 WO의 정비와 부품교체 연결은 LOT 시간 겹침과 별도로 유지한다.
# 한 WO에 부품이 둘이어도 세로 점선 하나를 쓰고 해당 부품 행에 마름모를 찍는다.
annual_part_links = []
for wo, group in cal_parts.groupby("WO_NO",sort=False):
    shown = group.loc[group.PART_CD.isin(rows)]
    if shown.empty:
        continue
    assert shown.MNT_TYPE.nunique()==1 and shown.END_DT.nunique()==1
    maintenance = shown.MNT_TYPE.iloc[0]
    completed = shown.END_DT.iloc[0]
    bottom = min(rows[code] for code in shown.PART_CD)
    ax.plot([completed,completed],[rows[maintenance],bottom],
            color=COL["PART"],lw=1.8,ls="--",alpha=.85,zorder=1.8,
            label=f"_wo_part_link_{wo}")
    for p in shown.itertuples():
        ax.plot(p.END_DT,rows[p.PART_CD],"D",color=COL["PART"],ms=7,zorder=5)
    annual_part_links.append({
        "WO":wo,"정비 종류":maintenance,"WO 완료 시각":completed,
        "연결 부품":" · ".join(f"{p.PART_NM} [{p.PART_CD}] × {p.QTY}" for p in shown.itertuples()),
        "교체 기록[행]":len(shown),"REMARK":shown.REMARK.fillna("미기재").iloc[0]})
annual_part_summary = pd.DataFrame(annual_part_links)
assert len(annual_part_links)==cal_parts.WO_NO.nunique()
work_by_wo = cal_work.set_index("WO_NO")
labels = []
for i,r in enumerate(lots.itertuples()):
    color = COL["FOCUS"] if r.LOT_NO in focus else COL["LOT"]

    # 온도 관측 범위만 배경으로 표시한다. 실제 정비 겹침은 빈 원과 교집합 표로 확인한다.
    ax.fill_betweenx(
        [rows["PLAN"], rows["LOT"]], r.START, r.END,
        color=COL["LOT"], alpha=0.10, zorder=0.5,
    )
    ax.vlines(
        [r.START, r.END], ymin=rows["PLAN"], ymax=rows["LOT"],
        color=COL["LOT"], linestyle="--", linewidth=0.9,
        alpha=0.35, zorder=1.0,
    )

    ax.plot([r.START,r.END],[9,9],color=color,lw=3)
    ax.plot(r.START,9,"o",color=color,ms=6)
    y = 10.4 if i%2==0 else 12.3
    label_text = f"{r.LOT_NO}\n{r.GRADE}\n{r.START:%m/%d}"
    label = ax.text(r.START,y,label_text,ha="center",va="center",fontsize=14.5,color=color,
                    bbox=dict(boxstyle="round,pad=.30",fc="white",ec=color,lw=.65))
    labels.append((label,r.START,y,color))

# 한 LOT당 세로 안내선 하나를 사용한다.
# 빈 원만 LOT 관측창과 실제로 겹친 기록을 뜻하며, 지나가는 선만으로는 겹침을 뜻하지 않는다.
# 원은 실제 사건/교집합 대표시각에 찍고, 정확한 시간·구간은 아래 표에 보존한다.
annual_links, annual_overlap_rows, annual_guides = [], [], []

def clock_union_minutes(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return sum((end-start).total_seconds()/60 for start,end in merged)

def mark_lot_record(lot, category, at_time, start, end, key=""):
    color = COL[category]
    # 원의 위치는 실제 시각 그대로 두어, 서로 다른 사건을 같은 순간으로 옮기지 않는다.
    ax.plot(at_time, rows[category], "o", mfc="white", mec=color, ms=9, mew=2, zorder=5)
    assert lot_meta.loc[lot,"START"] <= at_time <= lot_meta.loc[lot,"END"]
    annual_links.append({"LOT":lot, "연결 기록":category, "키":key,
                         "겹침 시작":start, "겹침 끝":end, "빈 원 표시 시각":at_time,
                         "구간 교집합[분]":(end-start).total_seconds()/60})

for r in lots.itertuples():
    k = int(r.LOT_NO)
    ge = grade_evt.loc[grade_evt.EVT_DT.between(r.START,r.END)]
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START,r.END)]
    for event_row in ge.itertuples():
        at_time = event_row.EVT_DT
        mark_lot_record(k, "GRADE", at_time, at_time, at_time,
                        key=f"{event_row.BEF_VAL}→{event_row.AFT_VAL}")
    work_intervals = []
    for task in tasks.itertuples():
        start,end = max(task.STRT_DT,r.START),min(task.END_DT,r.END)
        at_time = start + (end-start)/2
        mark_lot_record(k, task.MNT_TYPE, at_time, start, end, key=task.WO_NO)
        work_intervals.append((start,end))
    for task in detected.itertuples():
        at_time = task.DETC_DT
        mark_lot_record(k, "DETECT", at_time, at_time, at_time, key=task.WO_NO)
    annual_overlap_rows.append({
        "LOT":k, "강종":r.GRADE, "온도 관측 시작":r.START, "온도 관측 끝":r.END,
        "강종변경 동시 기록[건]":len(ge),
        "실제 CAL01 작업[WO]":len(tasks),
        "실제 작업 교집합 합집합[분]":clock_union_minutes(work_intervals),
        "관측 중 감지 기록[건]":len(detected)})
annual_overlap_summary = pd.DataFrame(annual_overlap_rows)
annual_interval_links = pd.DataFrame(annual_links)

# 연간 날짜축에서는 한 LOT의 짧은 관측창을 같은 세로 위치에서 읽도록 묶는다.
# 아래쪽 끝은 마지막으로 겹친 기록 행이며, 겹치지 않은 행에는 빈 원을 그리지 않는다.
GUIDE_COLOR = "#607080"
for r in lots.itertuples():
    k = int(r.LOT_NO)
    linked = annual_interval_links.loc[annual_interval_links.LOT.eq(k)]
    if linked.empty:
        continue
    bottom_y = min(rows[category] for category in linked["연결 기록"])
    guide, = ax.plot([r.START,r.START], [rows["LOT"],bottom_y],
                     color=GUIDE_COLOR,lw=2.2,alpha=.75,zorder=2,
                     label=f"_lot_guide_{k}",solid_capstyle="butt")
    ax.plot(r.START,rows["LOT"],"o",mfc="none",mec=GUIDE_COLOR,ms=10,mew=1.5,zorder=5)
    annual_guides.append({"LOT":k,"안내선 기준":r.START,"끝 행":bottom_y,
                          "겹친 기록 종류":",".join(linked["연결 기록"].drop_duplicates())})

assert annual_overlap_summary.LOT.nunique() == len(lots)
assert annual_overlap_summary["강종변경 동시 기록[건]"].sum() == 14
assert annual_overlap_summary.loc[annual_overlap_summary["실제 CAL01 작업[WO]"].gt(0),"LOT"].tolist() == [240117]
assert np.isclose(annual_overlap_summary.set_index("LOT").loc[240117,"실제 작업 교집합 합집합[분]"],304)
assert annual_overlap_summary.loc[annual_overlap_summary["관측 중 감지 기록[건]"].gt(0),"LOT"].tolist() == [240117,240133]

# 글상자 폭을 측정하여 같은 층에서 겹치는 라벨만 조금 이동한다.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for y in [10.4,12.3]:
    previous = None
    for label,x,level,color in labels:
        if level != y:
            continue
        box = text_box(label,renderer)
        if previous is not None and box.x0 < previous+7:
            delta = previous+7-box.x0
            screen = ax.transData.transform((mdates.date2num(x),y))
            label.set_x(ax.transData.inverted().transform((screen[0]+delta,screen[1]))[0])
            fig.canvas.draw()
            box = text_box(label,renderer)
        previous = box.x1
for label,x,y,color in labels:
    ax.plot([x,label.get_position()[0]],[9,y-.45],color=color,alpha=.45,lw=.7)
counts = cal_work.MNT_TYPE.value_counts()
axis_labels = ["CR1 계획수리 · 2회","O링 세트 · SEL-009","오일씰 · SEL-006","가스 분석계 셀 · INS-006",
               "정비 감지 [DETC_DT]",f"TBM 시간기반 · {counts['TBM']} WO",
               f"CBM 상태기반 · {counts['CBM']} WO",f"BM 사후 · {counts['BM']} WO",
               f"강종변경 · {len(grade_evt)}건",f"온도 관측 · {len(lots)} LOT"]
ax.set_yticks(range(10),axis_labels)
axis_colors = [COL["LOT"],COL["PART"],COL["PART"],COL["PART"],COL["DETECT"],
               COL["TBM"],COL["CBM"],COL["BM"],COL["GRADE"],COL["LOT"]]
for label,color in zip(ax.get_yticklabels(),axis_colors):
    label.set_color(color)
    label.set_fontweight("bold")
    label.set_fontsize(14.5)
ax.set_ylim(-.65,13.8)
ax.set_xlim(YEAR_START-pd.Timedelta(days=2),YEAR_END+pd.Timedelta(days=3))
ax.set_xticks(pd.date_range("2024-01-01","2024-12-01",freq="MS"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
style(ax,"2024년 기록 시각[월]")
ax.set_ylabel("")
handles = [Line2D([0],[0],color=COL["FOCUS"],marker="o",label="작업 겹침·감지 포함·수리 전후 LOT"),
           Line2D([0],[0],color=COL["LOT"],marker="o",label="기타 온도 관측 LOT"),
           Line2D([0],[0],color=COL["GRADE"],marker="|",label="CR1 강종변경"),
           Line2D([0],[0],color=GUIDE_COLOR,lw=2.2,label="LOT별 세로 안내선"),
           Line2D([0],[0],color="#34495e",marker="o",mfc="white",ls="None",ms=8,label="빈 원 · LOT 관측창과 겹친 기록"),
           *[Line2D([0],[0],color=COL[k],marker="x",label=k+" 작업") for k in ["BM","CBM","TBM"]],
           Line2D([0],[0],color=COL["DETECT"],marker="v",label="DETC_DT 감지 기록 · 방법 미제공"),
           Line2D([0],[0],color=COL["PART"],marker="D",ls="--",lw=1.8,label="동일 WO 정비→부품교체 · 완료 기준"),
           Patch(facecolor=COL["PLAN"],label="계획수리 날짜"),
           Patch(facecolor=COL["LOT"],alpha=.15,label="온도 관측 범위, 경계 점선과 음영")]
# 연간 축에는 주요 REMARK만 표시한다. 전체 WO 내용은 월별 상세 그림에서 확인한다.
for wo,x,y,align,prefix in [
    ("WO-2024-00960","2024-06-17",5.5,"right","6월 수리 TBM 6건 · 대표 REMARK"),
    ("WO-2024-01094","2024-07-10",7.5,"right","07/08 BM"),
    ("WO-2024-01859","2024-11-07",5.5,"right","수리 TBM 5건 이후 · 11/13 TBM"),
    ("WO-2024-01854","2024-11-09",6.5,"right","11/13 CBM"),
    ("WO-2024-02105","2024-12-27",6.5,"right","12/28 CBM · 셀 3개 교체 연결")]:
    t=work_by_wo.loc[wo]
    ax.annotate(f"{prefix}\n{str(t.REMARK).strip()}",xy=(t.STRT_DT,rows[t.MNT_TYPE]),
                xytext=(pd.Timestamp(x),y),ha=align,va="center",fontsize=14,color=COL[t.MNT_TYPE],
                bbox=dict(boxstyle="round,pad=.42",fc="white",ec=COL[t.MNT_TYPE],lw=.8),
                arrowprops=dict(arrowstyle="->",color=COL[t.MNT_TYPE],lw=.9),zorder=8)
for p in plans.itertuples():
    scope = p.SCOPE
    ax.annotate(f"{p.STRT_DT:%m/%d}–{p.END_DT:%m/%d}\n{scope}",xy=(p.STRT_DT,0),
                xytext=(p.END_EXCL+pd.Timedelta(days=3),.5),ha="left",va="center",fontsize=14,color=COL["LOT"],
                bbox=dict(boxstyle="round,pad=.42",fc="white",ec=COL["LOT"],lw=.8),
                arrowprops=dict(arrowstyle="->",color=COL["LOT"],lw=.9),zorder=8)
for y,text in [
    (.081,"빈 원: 강종변경 14 LOT · 실제 작업 240117(TBM·CBM) · 감지 240117·240133"),
    (.053,"240117: 실제 작업 합집합 304분(TBM 115+CBM 230-동시 41)"),
    (.025,"선 구분: 빈 원이 있는 행만 LOT 관측창과 겹침 · 보라 점선은 같은 WO의 정비→부품교체 | 7월 BM·12월 작업은 관측 종료 뒤")]:
    fig.text(.14,y,text,ha="left",va="center",fontsize=14.2,color="#34495e")
header(fig,"2024년 CAL01 관측·정비 흐름","TIME AXIS · 실제 측정 날짜·시각 | 회색 직선·빈 원=LOT 관측창 겹침 · 보라 점선·◆=동일 WO 정비→부품교체",handles,ncol=4)
ax.tick_params(axis="x",labelsize=14)
ax.xaxis.label.set_fontsize(14.5)
fig._suptitle.set_fontsize(24)
fig.texts[-1].set_fontsize(14)
# 제목·설명·범례는 위쪽에 모으고, 본문과 범례 사이에 별도 여백을 둔다.
fig.texts[-1].set_position((.035,.94))
fig.legends[0].set_bbox_to_anchor((.031,.907))
for text in fig.legends[0].get_texts(): text.set_fontsize(13)
# LOT당 선 하나이며 모두 직선이고, 화면 안에 표시됐는지 확인한다.
fig.canvas.draw()
guide_lines=[line for line in ax.lines if line.get_label().startswith("_lot_guide_")]
assert len(guide_lines)==len(annual_guides)==len(lots)
for guide in guide_lines:
    points=guide.get_transform().transform_path(guide.get_path()).vertices
    assert points.shape==(2,2) and np.isclose(points[0,0],points[1,0]), "LOT 안내선이 직선이 아님"
    assert np.all(points[:,0]>=ax.bbox.x0-1) and np.all(points[:,0]<=ax.bbox.x1+1), "안내선이 날짜축 밖으로 나감"
    assert np.all(points[:,1]>=ax.bbox.y0-1) and np.all(points[:,1]<=ax.bbox.y1+1), "안내선이 그림 밖으로 나감"

part_lines=[line for line in ax.lines if line.get_label().startswith("_wo_part_link_")]
assert len(part_lines)==len(annual_part_links)==cal_parts.WO_NO.nunique()
for link in part_lines:
    points=link.get_transform().transform_path(link.get_path()).vertices
    assert points.shape==(2,2) and np.isclose(points[0,0],points[1,0]), "WO 교체 연결선이 직선이 아님"
    assert np.all(points[:,0]>=ax.bbox.x0-1) and np.all(points[:,0]<=ax.bbox.x1+1), "WO 연결선이 날짜축 밖으로 나감"

finish(fig,"01_개요/01_연간_관측정비.png")
table(annual_overlap_summary)
table(annual_interval_links)
table(annual_part_summary)

**관찰:** 134건의 CR1 강종변경 중 온도 자료는 14개 LOT 관측창으로 제공된다. CAL01 44 WO 가운데 실제 작업이 관측창과 겹치는 것은 240117의 TBM·CBM 2건이다. 교집합 합계는 동시 작업을 한 번만 세어 **304분(115 + 230 - 41)**이다.

**가설 검토:** ‘정비일 부근 온도 변화는 정비 작업 중 변화다’라는 해석은 성립하지 않는다. 7월 BM은 240069 관측 뒤, 12월 CBM 작업·교체는 240133 관측 뒤에 있다.

**판단 범위:** 빈 원이 있는 행에서 실제 겹침을 확인한다. 관측 사이의 흰 구간은 온도 자료가 없는 기간이다. 부품 ◆ 위치는 동일 WO의 완료 시각을 대신 사용했으며 실제 교체 순간은 알 수 없다.

### 4.1.1 14 LOT × 공정 검토표

**읽을 질문:** 반복 펄스·LOT 전체 수준 차이·운전 전환 후보를 공정별로 어디에서 구분할 수 있는가?

In [ ]:
from matplotlib.patches import Rectangle
PROC=[('정지·승온','일정과 형태 대조'),('세정\nCLN-IN','LOT 내 시간 변화'),('건조\nZ1·Z2','같은 강종 대조'),
      ('가열·균열\nZ3–Z6','TC–OP · 같은 강종'),('CP\nCP-Z4','1초 기록의 이탈'),('과시효\nZ7·Z8','같은 강종 대조'),('Z9·냉각','주기 파형 구분')]
STATUS={'CP 이탈':('#fbe4e7',EVENT_COLORS['CP']),'국소 변화(Z)':('#fff1d6',EVENT_COLORS['Z']),'전환 후보':('#e9edf0',EVENT_COLORS['전환']),'조건 대조':('#e6f0f8','#086788'),'대조 없음':('#f4f6f7','#8c9aa4')}
grid={};ids=lots.LOT_NO.astype(int).tolist()
for k in ids:
    if (lots.GRADE==lot_meta.loc[k,'GRADE']).sum()==1:
        for j in [2,3,5]:grid[k,j]=('대조 없음','같은 강종 LOT 없음')
for r in expected.itertuples():grid[r.LOT,0]=('전환 후보',f'{r.종류}\n{r.시작:%m/%d %H:%M}')
for k,n in cp_count.items():
    d=cp_real.loc[cp_real.LOT.eq(k),'DEV'];peak=d.loc[d.abs().idxmax()]
    grid[k,4]=('CP 이탈',f'{n}개 사건 · 최대 {peak:+.3f}')
for k in cp_events.loc[cp_events.EXP,'LOT'].unique():grid[k,4]=('전환 후보','하강 전환에 포함\n별도 검토')

for k in LEVEL_REVIEW:
    for column,zones in [(2,{1,2}),(3,{3,4,5,6}),(5,{7,8})]:
        d=level_check.loc[level_check.LOT.eq(k)&level_check.DISPLAY_HIT&level_check.SENSOR.map(lambda s:int(s.split('-Z')[1]) in zones)]
        if d.empty:continue
        items=[f'{r.SENSOR} {r.GAP:+.1f}'+('℃' if r.SENSOR in LEVEL_TC else '%p') for r in d.itertuples()]
        lines=[' / '.join(items[i:i+2]) for i in range(0,len(items),2)]
        grid[k,column]=('조건 대조','\n'.join(lines)+f'\n대조 {int(d.N_PEERS.min())} LOT')

grid[240108,1]=('국소 변화(Z)','짧은 국소 변화\n부록 240108 확대 확인')
grid[240117,4]=('CP 이탈','CBM 완료 뒤 CP -0.083\n원인·효과 미확정')
H=15;fig,ax=plt.subplots(figsize=(28,H));fig.subplots_adjust(left=.11,right=.99,top=.82,bottom=.035)
for i,k in enumerate(ids):
    for j in range(7):
        st,text=grid.get((k,j),('', '—'));face,ink=STATUS.get(st,('white','#aebbc5'))
        ax.add_patch(Rectangle((j+.025,i+.05),.95,.9,facecolor=face,edgecolor='#d3dce3',lw=.8,hatch='///' if st=='대조 없음' else None))
        ax.text(j+.5,i+.5,text,ha='center',va='center',fontsize=10.5,color=ink,linespacing=1.3)
    hot=k in CP_REVIEW
    ax.text(-.07,i+.5,f"{k} · {lot_meta.loc[k,'GRADE']}\n{lot_meta.loc[k,'START']:%m/%d}",ha='right',va='center',fontsize=12,color=RED if hot else '#17344d',fontweight='bold' if hot else 'normal')
for j,(name,rule) in enumerate(PROC):
    ax.text(j+.5,-.65,name,ha='center',va='center',fontsize=14,color='#17344d',fontweight='bold')
    ax.text(j+.5,-.13,rule,ha='center',va='bottom',fontsize=10,color='#526d80')
ax.set(xlim=(0,7),ylim=(14,-1.25));ax.axis('off')
header(fig,'CAL01 14 LOT 공정별 검토 근거','빨강 = CP · 주황 = 국소 변화(Z) · 회색 = 전환 · 파랑 = 강종 조건 대조 · 빈 칸 = 요약 미선정',
       [Patch(facecolor=v[0],label=k) for k,v in STATUS.items()]+[Patch(facecolor='white',edgecolor='#d3dce3',label='— 요약 미선정')],ncol=6)
compact_header(fig,.94,.90,title_size=25,sub_size=13,legend_size=13)
finish(fig,'01_개요/02_14LOT_공정별검토.png')

**관찰:** 빨간 칸은 CP 원본 이탈이 있는 6 LOT이다. 그중 반복·작업 완료 뒤 사례 3 LOT을 우선 검토한다. 파란 칸은 기존 수준 검토 5 LOT에서 TC-Z1–TC-Z8·OP-Z1–OP-Z8의 표시 경계 이상 차이를 요약한 **운전 조건 대조**다. OP-Z4 외 OP-Z8 등 다른 차이도 포함하며, 4.2절의 접힌 표에서 전체 22신호 수치를 확인한다. 수리 전후 승온·하강을 제외한 부분에서 계산하며, 그 차이도 고장 판정은 아니다. SPCEN은 2 LOT뿐이므로 서로 다른 수준이라는 사실과 어느 쪽이 정상인지는 분리해야 한다. 표의 출력 차이는 **대상 LOT 중앙값 - 같은 강종의 다른 LOT 중앙값들의 중앙값**이다.

**가설 검토:** ‘LOT 안에서 평탄하면 특이점이 없다’는 가설은 강종 대조에서 반례를 보인다. 반대로 정비 전후의 큰 하강·승온은 운전 전환이라는 대안 설명을 먼저 검토한다.

**판단 범위:** 표는 검토 경로 안내다. 모든 신호·임계 초과를 빠짐없이 채운 고장 판정표는 아니다. 임계 초과·두 LOT 차이·사후 전환 분류를 고장 정답으로 해석하지 않는다. Z3의 반복 응답과 Z9·냉각의 주기 파형은 한 번의 큰 변화만으로 선정하지 않는다.

### 4.2 전체 14 LOT의 선별 근거와 후속 경로

같은 온도·CP 기준을 모든 LOT에 적용한 결과다. 반복성·정비의 실제 시각·같은 강종 수준 차이를 함께 읽는다.

In [ ]:
from IPython.utils.capture import capture_output
with capture_output() as captured:
    sensitivity=[]
    for limit in [3,4,5]:
        flags=zscreen.MAX_ABS_Z.ge(limit)
        nrun=sum(len(runs(g.set_index('MINUTE').MAX_ABS_Z.ge(limit))) for _,g in zscreen.groupby('LOT'))
        sensitivity.append({'기준':f'온도 |Z| ≥ {limit}','선정 LOT':zscreen.loc[flags,'LOT'].nunique(),'표시 1분창[개]':int(flags.sum()),'사건 수':nrun})
    # CP 사건은 묶음 기준 0.02를 고정하고 최고 편차 경계만 바꾼다.
    for limit in [.04,.05,.06]:
        found=[]
        for k in lots.LOT_NO.astype(int):
            s=raw[k].set_index('MEAS_DT')['CP-Z4'].dropna();dev=s-s.median();hit=dev.abs().gt(.02)
            grp=(hit.ne(hit.shift())|s.index.to_series().diff().dt.total_seconds().gt(5)).cumsum()
            for _,d in dev[hit].groupby(grp[hit]):
                peak=d.abs().idxmax();m=(peak-lot_meta.loc[k,'START']).total_seconds()/60
                if abs(d.loc[peak])>=limit and not in_expected(k,m):found.append(k)
        sensitivity.append({'기준':f'CP 최대 편차 ≥ {limit:.2f}','선정 LOT':len(set(found)),'표시 1분창[개]':np.nan,'사건 수':len(found)})
    sensitivity=pd.DataFrame(sensitivity)
    table(sensitivity)
    
    # 아래 8 LOT은 결합 근거에 따른 검토 우선순위다. 14 LOT의 계산·상세 검토를 제한하지 않는다.
    selected_rows=[]
    for k in CP_REVIEW:
        d=cp_real.loc[cp_real.LOT.eq(k)];r=d.loc[d.DEV.abs().idxmax()]
        selected_rows.append({'LOT':k,'강종':lot_meta.loc[k,'GRADE'],'유형':'CP','근거':f'{len(d)}개 사건, 최대 {r.DEV:+.3f}','시간 범위':r.PEAK.strftime('%m/%d %H:%M:%S')})
    for k in LEVEL_REVIEW:
        sensors=LEVEL_TARGETS[k]
        d=level_gap.loc[level_gap.LOT.eq(k)&level_gap.SENSOR.isin(sensors)]
        selected_rows.append({'LOT':k,'강종':lot_meta.loc[k,'GRADE'],'유형':'강종 대조','근거':' / '.join(f'{r.SENSOR} {r.GAP:+.1f}'+('℃' if r.SENSOR in TC else '%p') for r in d.itertuples()),'시간 범위':'전환 제외 구간의 LOT 중앙값'})
    selected=pd.DataFrame(selected_rows).sort_values('LOT')
    selected_by_lot=selected.set_index('LOT')
    screening_rows=[]
    for r in lots.itertuples():
        k=int(r.LOT_NO);c=context.set_index('LOT').loc[k]
        nz=int((zscreen.LOT.eq(k)&zscreen.MAX_ABS_Z.ge(Z_LIMIT)).sum())
        nc=int(cp_count.get(k,0));parts=[]
        if c.PLAN_RELATION:parts.append(c.PLAN_RELATION)
        if c.CAL_WORK:parts.append(f'실제 작업 {int(c.CAL_WORK)} WO 겹침')
        if c.CAL_DETECT:parts.append(f'감지 시각 {int(c.CAL_DETECT)}건 포함')
        if k in selected_by_lot.index:
            d=selected_by_lot.loc[k]
            reason=('반복 CP 원본·동반 반응' if k in [240037,240069] else 'CBM 완료 뒤 CP 하강' if k==240117 else '동일 강종의 조건·수준 대조')+' · '+d['근거']
            priority='결합 근거 우선 검토'
        else:
            reason=('수리 전 하강의 전환 설명 확인' if k==240058 else f'CP 원본 이탈 {nc}건 원본·TC–OP 대조' if nc else '국소 변화의 원본·TC–OP 대조')
            if k==240133:reason+=' · 감지 후 착수는 관측 종료 뒤'
            priority='해당 구간 확인'
        screening_rows.append({'LOT·강종':f'{k} · {r.GRADE}','온도 |Z|≥3[1분창]':nz,'CP[사건]':nc,'정비·전환 문맥':' / '.join(parts) or '관측 중 작업·감지 교집합 없음','검토 경로':priority,'확인할 이유':reason})
    screening_result=pd.DataFrame(screening_rows)
    note('### 4.2 전체 14 LOT의 검토 근거와 후속 경로')
    note('온도 Z 창 수에는 수리 전후 전환도 포함한다. CP 사건 수는 전환 후보를 제외한 값이다. 3σ 초과만으로는 14 LOT 모두가 표시되므로, 검토 우선순위는 반복성·정비 시각·강종 수준 차이와 함께 읽는다.')
    table(screening_result)
    assert len(screening_result)==14 and screening_result['CP[사건]'].sum()==len(cp_real)
    assert screening_result['온도 |Z|≥3[1분창]'].sum()==zscreen.MAX_ABS_Z.ge(Z_LIMIT).sum()
    
    note('**같은 강종 비교의 범위:** 같은 강종 대조 LOT이 있으면 22신호 모두의 중앙값 차이를 계산하고, 단독 강종은 비교를 보류한다. 수준 차이 표시는 TC-Z1–TC-Z8·OP-Z1–OP-Z8의 16신호를 사용한다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 따로 확인한다. 그림은 기존 우선 검토 5 LOT의 상세 사례이며, 모든 수준 차이를 고장으로 선별한 결과가 아니다.')
    table(level_inventory)
captured_details('3SD·CP 민감도와 신호별 상세 수치 펼치기',captured)
show_table(screening_result)

<a id="grade-iqr"></a>
### 4.3 강종·공정별 분포 — 수준 차이와 순간 이탈을 나누어 본다

각 LOT의 전환 제외 유효 1분 중앙값으로 **평균 ◆·중앙값·Q1~Q3 상자·1.5 IQR 수염과 밖의 점**을 계산했다. **평균 ◆는 이 유효 1분 중앙값들의 산술평균**이다. 공정별 그림을 펼치면 16신호 전체를 강종별로 묶어 비교할 수 있다. 상자 밖 전체를 이상으로 세지 않는다. 후보 경계는 `Q1−1.5×IQR`, `Q3+1.5×IQR`이다. [정의: NIST](https://www.itl.nist.gov/div898/handbook/eda/section3/boxplot.htm)

**읽을 것:** 상자 전체가 이동하면 LOT 간 수준 차이, 상자가 넓으면 변동 폭 차이, 수염 밖 점은 LOT 내부의 드문 1분창이다. 강종 대조는 SPHC 4 LOT·SPCE 3 LOT·그 외 1~2 LOT로 정상 한계를 확정하기 어렵다. 아래 IQR은 LOT별 탐색 기준이며, 같은 강종 정상 기준이나 3σ와 같은 확률 경계가 아니다.

In [ ]:
grade_order=lots.GRADE.drop_duplicates().tolist()
box_ids=[int(k) for grade in grade_order for k in lots.loc[lots.GRADE.eq(grade),'LOT_NO']]
for label,zones,number in [('건조',[1,2],2),('가열균열',[3,4,5,6],3),('과시효',[7,8],4)]:
    rows=[{'panels':[(f'TC-Z{z} · 강종별 LOT 분포',[f'TC-Z{z}']),(f'OP-Z{z} · 강종별 LOT 분포',[f'OP-Z{z}'])],'height':2.0} for z in zones]
    fig,cells=report_page(f'강종별 {label} · LOT 내부 분포와 LOT 간 수준',
        '평균 ◆ · 중앙값 실선 · 상자=가운데 50% · 수염=1.5 IQR 안 실제 끝값 · 점=수염 밖 유효 1분 중앙값',rows,width=26)
    for row in cells:
        for ax,head,sigs in row:draw_boxes(ax,head,sigs[0],box_ids)
    fig._folded=f'{label}: 전체 LOT·각 TC/OP의 평균·중앙값·IQR 비교 펼치기'
    emit(fig,f'02_선별/{number:02d}_강종분포_{label}.png')
summary=iqr_summary.groupby(['LOT','GRADE'],sort=False).agg(유효창_최소=('N','min'),수염밖_신호창=('OUT','sum'),연속구간_신호별합=('RUNS','sum')).reset_index()
table(summary)
table(iqr_summary)
example=iqr_summary.loc[iqr_summary.SENSOR.eq('OP-Z4')&iqr_summary.GRADE.eq('SPHC')]
note('**실제 비교 결과:** SPHC의 OP-Z4 LOT 중앙값은 '+ ' / '.join(f'{r.LOT} {r.MEDIAN:.2f}%' for r in example.itertuples())+'. 240061은 분포 중심이 다른 SPHC LOT보다 높다. 자기 LOT 내부 수염 밖 점만 세면 이 수준 차이를 설명할 수 없다. TC-Z4와 OP-Z4를 함께 비교해 온도 수준과 출력 수준의 차이를 구분한다.')
note('**실제 시각 연결:** 각 LOT의 상세 검토에는 수염 밖 값의 신호·시작/끝·지속 창 수를 접힌 표로 제공한다. 이 구간이 기존 CP·Z 확대와 겹치는지 확인한다. IQR이 넘었다는 이유만으로 새 고장 사건을 추가하지 않는다.')

In [ ]:
note('**시계열 특성 전체 검토:** CP 원본 14사건의 시작·종료·피크 간격·복귀 확인을 같은 정의로 계산했다. 아래 표를 펼치면 순간 이탈과 확대창 폭을 구분할 수 있다.')
table(cp_time_features)
table(interval_time_features)

<a id="cases"></a>
## 05 주요 LOT 상세 시계열 분석

각 사례는 **분석 질문 → 선정 근거 → 같은 번호의 원본·동반 신호 → 지속·반복·복귀와 정비 시각 → 잠정 판단** 순서다. 그래프 상단의 작은 전체 신호가 현재 확대 위치를 반복해서 알려 준다. 표시된 동시 반응은 동일 소재 위치의 반응으로 정렬한 결과가 아니다.
각 절은 **질문 → 선정 근거 → 생성 코드·그래프 → 관찰 결과 → 반례·해석** 순서로 읽는다. 함수 정의는 최초 사용 셀에 두고, 같은 레이아웃을 쓰는 다음 LOT은 그 함수를 재사용한다.

| 질문 | 직접 볼 그림·생성 코드 | 판독 범위 |
|---|---|---|
| CP 사건에서 어느 존이 움직였는가? | 5.1.1의 피크별 원본, 5.6의 전체 사건 TC·OP 변화량 | 반복 방향·크기·다른 사건과의 차이 |
| CP 하강 전후 주변 존은 어떤 모양인가? | 5.7의 CP·TC-Z3~Z6·OP-Z3~Z6 동시 파형 | CP 하강 2사건의 선후·반복 양상, 상승 사건 대조 |
| 같은 존의 TC와 OP는 어떤 관계인가? | [5.7.1 TC–OP](#relation-tc-op)의 실제 동시 파형·14 LOT 변화 방향 표 | 온도·출력의 반대 움직임, 선후 방향 미분리 |
| CP가 없을 때도 같은 반응인가? | 5.1.2의 원본 반례, 5.8의 같은 LOT·서로 다른 두 시간대 파형 | CP 사건의 구별 가능성 |
| 정비 전후 언제 발생했는가? | 5.3·5.5의 실제 작업과 A/B 원본, 5.9의 감지·작업·REMARK 표 | 선후 관계·비교 가능 범위 |
| LOT 간 수준 차이는 무엇인가? | 5.4의 SPHC TC·OP 분포와 부록 동일 강종 원본 | 설정·부하 차이 가설, 동일 운전 조건 미확인 |

| 실제 생성 함수 | 정의·최초 실행 위치 | 재사용 위치 |
|---|---|---|
| `draw_report_case`, `draw_cp69_overview` | 5.1 | 5.2·5.3·5.4 |
| `draw_cp69_peaks`, `draw_cp69_zone_peaks` | 5.1.1 | CP 피크별 Z3~Z6 대조 |
| `draw_cp69_b`, `draw_cp69_c` | 각각 5.1.2·5.1.3 | 해당 그림에서 정의 후 실행 |
| `draw_plan_case` | 5.5의 6월 코드 | 11월 코드 |
| 관계 분석 6개 그림 | 5.6·5.7·5.7.1·5.7.2·5.8·5.9의 직접 작성 코드 | 각 절에서 계산·그림·해석 연결 |
| `draw_lot_overview`, `draw_lot_zoom`, `event_evidence`, `draw_lot_maintenance` | 8.1의 각 그림 직전 | 이후 13 LOT |
| `draw_lot_level` | 8.4 동일 강종 대조 | 동일 강종 수준 차이 LOT |


<a id="case-240069"></a>
### 5.1 LOT 240069 — 반복 CP 이탈

**분석 질문:** 반복 CP가 어떤 지속·복귀 양상을 보이며 이후 BM 기록보다 앞서는가?

**선정 근거:** 전환 후보를 제외한 CP 원본 7사건, 최대 편차 +0.459. (a)는 CP 6사건, (b)는 TC-Z3의 국소 Z 변화, (c)는 가장 큰 CP 사건이다. (b)에 CP가 없다는 이유로 (b)를 생략하지 않는다.

**정비 대조:** 가까운 BM REMARK는 “1호기 재발, 절연 저하”. 감지 07/08 08:49, 작업 11:11–19:04이며 LOT 관측은 그 전에 끝난다.

**읽는 순서:** 먼저 (a)(b)(c)의 전체 위치와 CP·TC-Z3·OP-Z3를 같은 행에서 보고, 아래에서 (a)의 피크 6개, (b)의 첫 반응과 약 34분 뒤 별도 반응, (c) 이후 변동성과 30일 내 WO를 각각 확인한다.

**잠정 판단:** 반복 CP와 동반 반응은 관찰되지만, TC-Z3 상승 하나로 모든 CP를 설명하기 어렵다. 07/25 가스 분석계 셀 손상 교체는 계측 경로 확인의 단서이며 CP와의 부품 연결·인과는 미확정이다.

**관계 분석의 현재 범위:** CP 두 채널의 정의가 미확인이므로 관측값의 이탈·반복, 다른 신호의 동반 변화와 시간 순서를 확인한다. 이를 가스 성분·온도·전기 상태의 변화 또는 특정 부품의 고장으로 번역할 근거는 부족하다. 후속 WO와 부품 교체는 같은 설비의 작업 기록이며, CP 원인으로 선정한 결과가 아니다. 채널 정의가 확보되면 물리적으로 대응하는 TC·OP와 계측·제어 경로를 먼저 정하고 다시 비교해야 한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
CP69=240069
cp69_raw=raw[CP69].set_index('MEAS_DT').asfreq('s')
cp69_base=cp69_raw['CP-Z4'].median()
cp69_pulses=cp_real.loc[cp_real.LOT.eq(CP69)].sort_values('PEAK').reset_index(drop=True).copy()
cp69_pulses['ID']=[f'240069-a{i}' if i<=6 else '240069-c1' for i in range(1,8)]
cp69_pulses['ONSET']=[at(CP69,v).round('s') for v in cp69_pulses.A]
cp69_rows=[]
for pulse in cp69_pulses.itertuples():
    for sensor in [s for s in SIGNALS if s!='CP-Z4']:
        y=cp69_raw[sensor]
        ref=y.loc[pulse.ONSET-pd.Timedelta(minutes=5):pulse.ONSET-pd.Timedelta(minutes=1)]
        q=y.loc[pulse.ONSET-pd.Timedelta(minutes=2):pulse.ONSET+pd.Timedelta(minutes=2)]
        dev=(q-ref.median()).dropna()
        t=dev.abs().idxmax() if len(dev) else pd.NaT
        cp69_rows.append(dict(ID=pulse.ID,SENSOR=sensor,CP_START=pulse.ONSET,CP_PEAK=pulse.PEAK,CP_DEV=pulse.DEV,
            BASE=ref.median(),MAX_DEV=dev.loc[t] if pd.notna(t) else np.nan,MAX_AT=t,
            OFFSET_SEC=(t-pulse.ONSET).total_seconds() if pd.notna(t) else np.nan,N_BASE=ref.count(),N_WINDOW=q.count()))
cp69_signalstats=pd.DataFrame(cp69_rows)
cp69_by_peak=[]
for pulse in cp69_pulses.itertuples():
    d=cp69_signalstats.loc[cp69_signalstats.ID.eq(pulse.ID)].set_index('SENSOR')
    row={'피크':pulse.ID,'CP 시작':pulse.ONSET.strftime('%m/%d %H:%M:%S'),'CP 피크':pulse.PEAK.strftime('%H:%M:%S'),'CP 편차':pulse.DEV}
    for sensor in ['TC-Z3','OP-Z3','TC-Z4','OP-Z4']:
        item=d.loc[sensor];unit='℃' if sensor in TC else '%p'
        row[sensor]=f'{item.MAX_DEV:+.2f}{unit} / {item.OFFSET_SEC:+.0f}초'
    cp69_by_peak.append(row)
cp69_by_peak=pd.DataFrame(cp69_by_peak)
cp69_b=next(e for e in LOT_EVENTS[CP69] if e['tag']=='(b)')
cp69_c=cp69_pulses.iloc[-1]
cp69_later_point=cp69_raw['TC-Z3'].loc[cp69_b['point']:cp69_b['point']+pd.Timedelta(hours=1)].idxmax()
cp69_b_compare=[]
for label,t,m in [('b-1 · 최대 Z',cp69_b['point'],cp69_b['minute']),
                  ('b-2 · 뒤 1시간 내 원본 최고',cp69_later_point,int((cp69_later_point-lot_meta.loc[CP69,'START']).total_seconds()//60))]:
    d=zstats[CP69,'TC-Z3'].loc[m]
    cp=cp69_raw['CP-Z4'].loc[t-pd.Timedelta(minutes=2):t+pd.Timedelta(minutes=2)]
    cp69_b_compare.append(dict(LABEL=label,POINT=t,MINUTE=m,X=d.X,REF=d.MEDIAN,SD=d.SD,Z=d.Z,
                              CP_DEV=(cp-cp69_base).abs().max()))
cp69_b_compare=pd.DataFrame(cp69_b_compare)
cp69_var=[]
for span in [25,50]:
    lo,hi=(5,30) if span==25 else (10,60)
    for phase,a,b in [('전',-hi,-lo),('후',lo,hi)]:
        q=cp69_raw.loc[cp69_c.PEAK+pd.Timedelta(minutes=a):cp69_c.PEAK+pd.Timedelta(minutes=b)]
        cp69_var.append(dict(WINDOW=f'전후 {span}분',PHASE=phase,START=q.index.min(),END=q.index.max(),
            TC3_SD=q['TC-Z3'].std(),TC3_MIN=q['TC-Z3'].min(),TC3_MAX=q['TC-Z3'].max(),
            OP3_LOW_PCT=100*q['OP-Z3'].lt(10).sum()/q['OP-Z3'].notna().sum(),N=q['TC-Z3'].count()))
cp69_variability=pd.DataFrame(cp69_var)
cp69_sd=cp69_raw['TC-Z3'].rolling(600,min_periods=540).std()
cp69_works=cal_work.loc[cal_work.DETC_DT.between(cp69_c.PEAK,cp69_c.PEAK+pd.Timedelta(days=30))].sort_values('DETC_DT').copy()
cp69_after_rows=[]
for w in cp69_works.itertuples():
    cp69_after_rows.append({'WO':w.WO_NO,'종류':w.MNT_TYPE,'감지':w.DETC_DT,'착수':w.STRT_DT,'완료':w.END_DT,
        'REMARK':remark_text(w),'부품·교체 사유':' / '.join(parts_for(w.WO_NO)) or '교체 기록 없음',
        'CP 최대→감지[시간]':(w.DETC_DT-cp69_c.PEAK).total_seconds()/3600})
cp69_after_work=pd.DataFrame(cp69_after_rows)
cp69_peak_checks=[]
assert len(cp69_signalstats)==7*(len(SIGNALS)-1)==147
assert len(cp69_works)==3

def event_reason(k,e):
    if e['kind']=='CP':return f"CP-Z4 · 원본 {len(e['pulses'])}사건 · 최대 편차 {e['deviation']:+.4f} (±0.05 초과)"
    if e['kind']=='Z':return f"{e['sensor']} · 1분 중앙값 Z={e['score']:+.2f} (|Z|≥3)"
    if e['kind']=='전환':return 'TC·OP 동반 하강/승온 · 수리 일정과 전환 형태 대조'
    return '정비 DETC 기록 · 센서 자동 경보라는 뜻은 아님'

def draw_event_locator(ax,head,k,e):
    # 위와 아래 모두 같은 1초 원본·센서색이다. 음영과 테두리만 구간 유형색이다.
    sensor=e['sensor'] if e['kind']=='Z' else 'TC-Z3' if e['kind']=='전환' else 'CP-Z4'
    y=raw[k].set_index('MEAS_DT')[sensor].asfreq('s');xx=y.index
    factor=1e10 if sensor=='CP-Z4M' else 1
    ax.plot(xx,y.to_numpy()*factor,color='#aebac3',lw=.75,label=f'{sensor} · 전체 LOT')
    for other in LOT_EVENTS[k]:
        chosen=other['tag']==e['tag']
        ax.axvspan(at(k,other['a']),at(k,other['b']),color=e['color'] if chosen else '#9aabb8',
                   alpha=.17 if chosen else .035,ec=e['color'] if chosen else 'none',lw=1.3 if chosen else 0)
    a,b=at(k,e['a']).round('s'),at(k,e['b']).round('s');mask=(xx>=a)&(xx<=b)
    selected,=ax.plot(xx[mask],y.to_numpy()[mask]*factor,color=sensor_color(sensor),lw=1.7,label=f'{sensor} · 아래 확대 원본')
    clock_in_lot(ax,k);ax.set_ylabel(display_unit(sensor),fontsize=9);panel_legend(head,ax)
    assert selected.get_color()==sensor_color(sensor)
    locator_visual_checks.append(dict(LOT=k,TAG=e['tag'],SENSOR=sensor,LINE=selected.get_color(),START=a,END=b))
    locator_checks.append((k,e['tag']))


def raw_window(k,s,a,b):
    return raw[k].set_index('MEAS_DT')[s].asfreq('s').loc[a:b]

def zoom_clock(ax,a,b):
    clock_ticks(ax,a,b)
    minutes=(b-a).total_seconds()/60
    if minutes<=80:
        step=next(step for step in [1,2,3,5,10,15,20,30] if minutes/step<=5)
        ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0,60,step)))


def event_focus_ranges(k,e):
    if e['kind']=='Z':return [(e['point']-pd.Timedelta(minutes=1),e['point'])]
    if e['kind']=='CP':return [(at(k,p.A).round('s'),at(k,p.B).round('s')) for p in e['pulses'].itertuples()]
    if e['kind']=='감지':return [(e['point']-pd.Timedelta(seconds=30),e['point']+pd.Timedelta(seconds=30))]
    return [(row.시작,row.끝) for row in expected.loc[expected.LOT.eq(k)].itertuples()]


def shade_event_focus(ax,k,e):
    for a,b in event_focus_ranges(k,e):ax.axvspan(a,b,color=e['color'],alpha=.16 if e['kind']!='전환' else .12,lw=0)
    if e['kind']=='감지':ax.axvline(e['point'],color=e['color'],ls=EVENT_LINES['감지'],lw=1.1)


def z_window_marker(ax,k,e,annotate=False):
    # 1분 요약값은 실제 창 중앙에 별도 마커로 놓는다. 종료 시각의 원본 값처럼 표시하지 않는다.
    start,end=event_focus_ranges(k,e)[0];middle=start+(end-start)/2
    value=float(zstats[k,e['sensor']].loc[e['minute'],'X'])
    ax.plot([start,end],[value,value],color=e['color'],lw=1.8,zorder=7)
    ax.plot(middle,value,marker='D',ms=4,mfc='white',mec=e['color'],ls='',zorder=8,label='선정 1분창 중앙값')
    if annotate:
        near_note(ax,middle,value,f'{start:%H:%M}–{end:%H:%M} 중앙값 {value:.2f}\nZ={e["score"]:+.2f}',e['color'])


def plot_event_trace(ax,head,k,e,a,b,large=False):
    s=e['sensor'] if e['kind']=='Z' else 'CP-Z4' if e['kind'] in ['CP','감지'] else 'TC-Z3'
    report_signal(ax,k,s,a,b,fine=True);t=e['point'];y=raw_window(k,s,a,b)
    shade_event_focus(ax,k,e)
    if e['kind']=='CP':
        base=raw[k]['CP-Z4'].median()
        for v in [base-.05,base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.9)
        ax.axhline(base,color=GREY,ls=':',lw=.75)
        for pulse in e['pulses'].itertuples():
            q=y.loc[at(k,pulse.A).round('s'):at(k,pulse.B).round('s')]
            ax.plot(q.index,q,color=e['color'],lw=2.2,zorder=5)
            if a<=pulse.PEAK<=b:
                ax.scatter(pulse.PEAK,y.get(pulse.PEAK,np.nan),s=25,facecolors='white',edgecolors=e['color'],lw=1.5,zorder=6)
                ax.vlines(pulse.PEAK,base,y.get(pulse.PEAK,np.nan),color=e['color'],ls=':',lw=1.)
    elif e['kind']=='Z':
        d=zstats[k,s];xx=at(k,d.index+1);mask=(xx>=a)&(xx<=b)
        for values in [d.MEDIAN-3*d.SD,d.MEDIAN+3*d.SD]:ax.step(xx[mask],values[mask],where='pre',color=e['color'],ls='--',lw=.9)
        z_window_marker(ax,k,e,annotate=False)
    ax.set_ylabel(display_unit(s));zoom_clock(ax,a,b);panel_legend(head,ax)
    if large and e['kind']=='CP':
        value=y.get(t,np.nan)
        near_note(ax,t,value,f'{t:%H:%M:%S} · {s} {value:.4f}\n편차 {e["deviation"]:+.4f}',e['color'])
    elif large and e['kind']=='Z':
        start,end=event_focus_ranges(k,e)[0];middle=start+(end-start)/2
        value=float(zstats[k,s].loc[e['minute'],'X'])
        near_note(ax,middle,value,f'{start:%H:%M}–{end:%H:%M} 중앙값 {value:.2f}\nZ={e["score"]:+.2f}',e['color'])


def case_interval_table(k):
    frame=interval_time_features.loc[interval_time_features.LOT.eq(k)].copy()
    out=pd.DataFrame({'구간 번호':frame.ID,'선정 신호':frame.SENSOR,'기준 시각':frame.POINT.dt.strftime('%m/%d %H:%M:%S'),
        '지속·반복':frame.DURATION_DESCRIPTION,'복귀 기록':frame.RECOVERY})
    show_table(out)

def draw_report_case(k):
    r=lot_meta.loc[k];events=LOT_EVENTS[k]
    if k==240061:
        rows=[{'panels':[(f'TC-Z{zone} · 같은 SPHC의 온도',[f'TC-Z{zone}']),(f'OP-Z{zone} · 같은 SPHC의 출력',[f'OP-Z{zone}'])],'height':2.1} for zone in [4,8]]
        fig,cells=report_page('Case 04 · 240061 | 같은 강종의 운전 수준 차이',
            '승온·종료 전환 제외 후 분포. TC와 OP를 별도로 비교한다. 평균 ◆ · 중앙값 · IQR · 수염 밖 1분창.',rows,width=20,
            info=['OP-Z4뿐 아니라 OP-Z8도 함께 확인한다. 분포 이동과 LOT 내부의 드문 1분창은 서로 다른 근거다.'])
        peers=lots.loc[lots.GRADE.eq('SPHC'),'LOT_NO'].astype(int).tolist()
        for row in cells:
            for ax,head,sigs in row:draw_boxes(ax,head,sigs[0],peers,highlight=k)
    elif k==240069:
        fig=draw_cp69_overview()
    else:
        rows=[{'panels':[(event_id(k,e)+' · 전체 LOT 위치',[]) for e in events],'height':.8},
              {'panels':[(event_id(k,e)+' · '+e['sensor'],[]) for e in events],'height':2.0,
               'captions':[event_reason(k,e) for e in events]},
              {'panels':[(e['tag']+' 동시 온도 · TC-Z3–TC-Z6',HEAT) for e in events],'height':1.25},
              {'panels':[(e['tag']+' 동시 출력 · OP-Z3–OP-Z6',[f'OP-Z{i}' for i in range(3,7)]) for e in events],'height':1.25}]
        number={240069:1,240037:2,240117:3}[k]
        fig,cells=report_page(f'Case {number:02d} · {k} {r.GRADE} | 선정 지점과 동반 반응',
            '각 열은 동일한 구간 번호다. 전체 위치 → 선정 신호 확대 → 같은 시각의 TC·OP를 읽는다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.',rows,width=24,
            info=[f'구간 {len(events)}개 · 정확한 번호·시각·지속·복귀는 아래 표, 22신호 상세는 부록의 같은 번호로 연결된다.',
                  {240069:'BM 01094 · 1호기 재발, 절연 저하 | 감지 07/08 08:49 · 작업 11:11–19:04 · LOT 관측 뒤의 기록',
                   240037:'LOT 관측 중 작업·감지 교집합 없음. 가까운 WO와 CP 펄스의 원인 관계는 확인되지 않음',
                   240117:'CBM 01854 · 마모량 초과 | 완료 12:44 → CP 최저 13:04:50 (+20.83분) · TBM 01859 계획 점검 07:40–09:35'}[k]],event_key=True)
        for (ax,head,_),event in zip(cells[0],events):draw_event_locator(ax,head,k,event)
        for (ax,head,_),event in zip(cells[1],events):plot_event_trace(ax,head,k,event,at(k,event['a']),at(k,event['b']),large=True)
        for row in cells[2:]:
            for (ax,head,sensors),event in zip(row,events):
                a,b=at(k,event['a']),at(k,event['b'])
                for sensor in sensors:report_signal(ax,k,sensor,a,b,fine=True)
                ax.axvspan(event['point']-pd.Timedelta(seconds=30),event['point']+pd.Timedelta(seconds=30),color=event['color'],alpha=.13,lw=0)
                ax.set_ylabel(display_unit(sensors[0]));zoom_clock(ax,a,b);panel_legend(head,ax)
    emit(fig,f'01_개요/사례_{k}_보고서.png')
    if k==240061:
        show_table(level_check.loc[level_check.LOT.eq(k)&level_check.SENSOR.isin(['TC-Z4','OP-Z4','TC-Z8','OP-Z8']),
            ['SENSOR','LOT_MEDIAN','PEER_MEDIAN','GAP','N_PEERS']].rename(columns={'SENSOR':'신호','LOT_MEDIAN':'240061 중앙값','PEER_MEDIAN':'다른 SPHC LOT 중앙값의 중앙값','GAP':'차이','N_PEERS':'대조 LOT 수'}))
    else:case_interval_table(k)
    if k in [240069,240037,240117]:
        features=cp_time_features.loc[cp_time_features.LOT.eq(k)]
        show_table(features[['EVENT_ID','START','END','SPAN_SEC','POINTS','DEV','PEAK_GAP_MIN','RETURN_CONFIRMED']].rename(columns={
            'EVENT_ID':'구간 번호','START':'CP 이탈 시작','END':'끝','SPAN_SEC':'첫·끝 간격[초]','POINTS':'실측 점 수','DEV':'최대 편차','PEAK_GAP_MIN':'이전 피크 간격[분]','RETURN_CONFIRMED':'60개 연속 복귀 확인'}))
    report_case_checks.append(k)

def draw_cp69_overview():
    events=LOT_EVENTS[CP69]
    rows=[{'panels':[(event_id(CP69,e)+' · 전체 LOT 위치',[]) for e in events],'height':.8},
          {'panels':[(e['tag']+' CP-Z4 · 원본 이탈' if e['kind']=='CP' else e['tag']+' CP-Z4 · TC 변화 때의 대조',[]) for e in events],
           'height':1.6,'captions':[event_reason(CP69,e) if e['kind']=='CP' else '선정 이유는 TC-Z3의 최대 Z. 이 시간의 CP 원본도 직접 확인한다.' for e in events]},
          {'panels':[(e['tag']+' TC-Z3 · 가열 초입 온도',[]) for e in events],'height':1.5},
          {'panels':[(e['tag']+' OP-Z3 · 대응 출력',['OP-Z3']) for e in events],'height':1.4},
          {'panels':[(e['tag']+' TC-Z4·TC-Z5·TC-Z6',['TC-Z4','TC-Z5','TC-Z6']) for e in events],'height':1.2},
          {'panels':[(e['tag']+' OP-Z4·OP-Z5·OP-Z6',['OP-Z4','OP-Z5','OP-Z6']) for e in events],'height':1.2}]
    fig,cells=report_page('Case 01 · 240069 SPHC | CP와 TC-Z3·OP-Z3의 시간 관계',
        '모든 열의 행 순서는 같다. CP → TC-Z3 → OP-Z3 → 다른 가열·균열 신호. 같은 TC-Z3를 위아래에 중복해 그리지 않는다.',rows,width=24,
        info=['(a) 6개 CP 피크는 아래에서 각각 비교한다. (b) 최대 Z와 뒤의 더 큰 온도 변화를 함께 대조한다. (c) 전후 변동 폭·이후 WO를 확인한다.',
              'BM 01094 · 1호기 재발, 절연 저하 | 감지 07/08 08:49 · 작업 11:11–19:04 · 실제 작업은 LOT 관측 뒤'],event_key=True)
    for (ax,head,_),e in zip(cells[0],events):draw_event_locator(ax,head,CP69,e)
    for (ax,head,_),e in zip(cells[1],events):
        ax.set_ylim(cp69_base-.15,cp69_base+cp69_pulses.DEV.max()+.14)
        a,b=at(CP69,e['a']),at(CP69,e['b'])
        if e['kind']=='CP':plot_event_trace(ax,head,CP69,e,a,b,large=True)
        else:
            report_signal(ax,CP69,'CP-Z4',a,b,fine=True);ax.set_ylabel('CP-Z4 [무차원]');zoom_clock(ax,a,b);panel_legend(head,ax)
    for (ax,head,_),e in zip(cells[2],events):
        ax.set_ylim(848,873)
        a,b=at(CP69,e['a']),at(CP69,e['b'])
        if e['kind']=='Z':plot_event_trace(ax,head,CP69,e,a,b,large=True)
        else:
            report_signal(ax,CP69,'TC-Z3',a,b,fine=True);ax.set_ylabel('온도 [℃]');zoom_clock(ax,a,b);panel_legend(head,ax)
    for row in cells[3:]:
        for (ax,head,sensors),e in zip(row,events):
            a,b=at(CP69,e['a']),at(CP69,e['b'])
            for sensor in sensors:report_signal(ax,CP69,sensor,a,b,fine=True)
            ax.axvspan(e['point']-pd.Timedelta(seconds=30),e['point']+pd.Timedelta(seconds=30),color=e['color'],alpha=.12,lw=0)
            ax.set_ylabel(display_unit(sensors[0]));zoom_clock(ax,a,b);panel_legend(head,ax)
            if sensors==['OP-Z3']:ax.set_ylim(-3,103)
    return fig

draw_report_case(240069)

#### 5.1.1 (a) 피크별로 CP와 Z3·Z4~Z6의 TC·OP를 비교한다

**비교 범위:** 그래프는 CP 이탈 첫 기록 전 3분~후 4분, 표의 최대편차는 시작 전후 2분이다. 근접 반응을 대조한 결과이며, 더 긴 지연 관계까지 배제하는 분석은 아니다. 첫 6개 CP는 밀집해 있어 긴 반응창이 서로 겹치므로 각 피크의 영향을 독립적으로 분리하기 어렵다.

**Z3만 원인 후보로 고른 것은 아니다.** 기존 최대 시간순 Z를 설명하는 예시로 Z3를 먼저 보였다. 아래에서 Z3와 Z4~Z6를 같은 CP 피크·비교 기준으로 각각 확인한다. Z4~Z6 그림은 채널별 평소 높낮이를 빼고 변화량으로 겹쳐 보여, 다른 기본 출력 수준에 변화가 가려지지 않게 했다.

**번호 표기:** (a)는 전체 비교 구간, **(a-1)~(a-6)**은 그 안의 개별 CP 피크다. 같은 번호의 CP·TC·OP를 세로로 읽는다. (b-1)·(b-2)는 서로 다른 시각의 두 온도 반응, (c-1)은 (c)의 CP 사건이다. 여러 LOT이 섞인 표에서는 `240069 (a-1)`처럼 LOT을 함께 표시한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
CP69_ZONE_SENSORS=[f'{prefix}-Z{zone}' for prefix in ['TC','OP'] for zone in [4,5,6]]
cp69_zone_rows=[]
for pulse in cp69_pulses.itertuples():
    for sensor in CP69_ZONE_SENSORS:
        metric=cp69_signalstats.loc[cp69_signalstats.ID.eq(pulse.ID)&cp69_signalstats.SENSOR.eq(sensor)].iloc[0]
        q=cp69_raw[sensor].loc[pulse.PEAK-pd.Timedelta(seconds=15):pulse.PEAK+pd.Timedelta(seconds=15)]
        cp69_zone_rows.append(dict(ID=pulse.ID,SENSOR=sensor,CP_DEV=pulse.DEV,AT_PEAK=q.median()-metric.BASE,
            N_PEAK=q.count(),MAX_DEV=metric.MAX_DEV,OFFSET_SEC=metric.OFFSET_SEC,BASE=metric.BASE))
cp69_zone_metrics=pd.DataFrame(cp69_zone_rows)
cp69_zone_peak_checks=[]
cp69_zone_limits={}
for prefix in ['TC','OP']:
    values=[]
    for pulse in cp69_pulses.iloc[:6].itertuples():
        for sensor in [f'{prefix}-Z{i}' for i in [4,5,6]]:
            base=cp69_zone_metrics.loc[cp69_zone_metrics.ID.eq(pulse.ID)&cp69_zone_metrics.SENSOR.eq(sensor),'BASE'].iloc[0]
            values.append(cp69_raw[sensor].loc[pulse.ONSET-pd.Timedelta(minutes=3):pulse.ONSET+pd.Timedelta(minutes=4)]-base)
    amplitude=pd.concat(values).abs().max()
    cp69_zone_limits[prefix]=(-amplitude*1.15,amplitude*1.15)

def cp69_event_panel(ax,head,pulse,sensor,annotate=False):
    t=pulse.ONSET;a=t-pd.Timedelta(minutes=3);b=t+pd.Timedelta(minutes=4)
    report_signal(ax,CP69,sensor,a,b,fine=True)
    ax.axvspan(t,at(CP69,pulse.B).round('s'),color=EVENT_COLORS['CP'],alpha=.12,lw=0)
    ax.set_xlim(a,b);ax.set_xticks([t+pd.Timedelta(minutes=m) for m in [-2,0,2,4]],['-2','0','+2','+4'])
    ax.set_xlabel('CP 이탈 시작 대비 [분]');ax.set_ylabel(display_unit(sensor));panel_legend(head,ax)
    if sensor=='CP-Z4':
        ax.set_ylim(cp69_base-.12,cp69_base+cp69_pulses.iloc[:6].DEV.max()+.10)
        for level in [cp69_base-.05,cp69_base+.05]:ax.axhline(level,color=EVENT_COLORS['CP'],ls='--',lw=.7)
        y=cp69_raw.loc[pulse.PEAK,sensor];ax.scatter(pulse.PEAK,y,s=25,facecolors='white',edgecolors=EVENT_COLORS['CP'],zorder=7)
        if annotate:near_note(ax,pulse.PEAK,y,f'{pulse.PEAK:%H:%M:%S}  +{pulse.DEV:.3f}',EVENT_COLORS['CP'])
    else:
        # 6개 피크에 같은 세로축을 사용해 진폭을 비교한다.
        windows=[cp69_raw[sensor].loc[p.ONSET-pd.Timedelta(minutes=3):p.ONSET+pd.Timedelta(minutes=4)] for p in cp69_pulses.iloc[:6].itertuples()]
        values=pd.concat(windows).dropna();lo,hi=values.min(),values.max();pad=max((hi-lo)*.12,.1)
        ax.set_ylim(lo-pad,hi+pad)

def draw_cp69_peaks():
    rows=[]
    for start in [0,3]:
        group=list(cp69_pulses.iloc[start:start+3].itertuples())
        for sensor in ['CP-Z4','TC-Z3','OP-Z3']:
            panels=[(f'{event_text(p.ID,include_lot=False)} {sensor}',[sensor]) for p in group]
            captions=[]
            for p in group:
                if sensor=='CP-Z4':captions.append(f'시작 {p.ONSET:%H:%M:%S} · 피크 {p.PEAK:%H:%M:%S} · CP 편차 +{p.DEV:.3f}')
                else:
                    d=cp69_signalstats.loc[cp69_signalstats.ID.eq(p.ID)&cp69_signalstats.SENSOR.eq(sensor)].iloc[0]
                    captions.append(f'기준 대비 최대편차 {d.MAX_DEV:+.2f}'+('℃' if sensor in TC else '%p')+f' · CP 시작 {d.OFFSET_SEC:+.0f}초 위치 (반응 시작 시각 아님)')
            rows.append({'panels':panels,'height':1.5,'captions':captions if sensor=='CP-Z4' else None})
    fig,cells=report_page('240069-a | 6개 CP 피크를 하나씩 같은 시간 폭으로 대조',
        '각 열에서 CP → TC-Z3 → OP-Z3. 0=CP 이탈 첫 기록이며 모든 패널은 전 3분~후 4분이다. 7번째 큰 사건은 c에서 별도 확인한다.',rows,width=24,
        info=['6개 피크에 신호별 같은 세로축을 쓴다. 음영은 원본 CP 편차 0.02 초과 묶음의 첫·끝 기록 범위이며 공백은 채우지 않는다.',
              '전체 21개 다른 신호의 원본 편차·최대편차 시각은 아래 표에 남겼다. 특정 채널을 생략했다고 정상으로 판정하지 않는다.'])
    for row_index,row in enumerate(cells):
        start=0 if row_index<3 else 3;sensor=['CP-Z4','TC-Z3','OP-Z3'][row_index%3]
        for (ax,head,_),pulse in zip(row,cp69_pulses.iloc[start:start+3].itertuples()):
            cp69_event_panel(ax,head,pulse,sensor,annotate=sensor=='CP-Z4')
            if sensor=='CP-Z4':cp69_peak_checks.append(pulse.ID)
    emit(fig,'01_개요/사례_240069_a_피크별대조.png')
    show_table(cp69_by_peak)
    note('**표의 읽기:** 기준은 CP 시작 전 5~1분 중앙값, 비교는 시작 전후 2분이다. 부호는 최대 절대편차의 방향, 초는 그 값의 위치다. 반응 시작·인과 시차 추정이 아니다. 가까운 CP 사건은 비교창이 서로 겹치므로 독립 실험 7회로 해석하지 않는다.')
    table(cp69_signalstats.rename(columns={'MAX_DEV':'기준 대비 최대편차','OFFSET_SEC':'CP 시작 대비 최대편차 위치[초]'}))
    draw_cp69_zone_peaks()

def cp69_b_observations():
    # 서로 다른 두 반응을 한 연속 창에서 확인한다. 기존 (b)의 추가 배율이 아니다.
    a=cp69_b['point'].floor('min')-pd.Timedelta(minutes=14)
    b=cp69_later_point.ceil('min')+pd.Timedelta(minutes=17)
    q=cp69_raw.loc[a:b]
    cp=q['CP-Z4'].dropna();dev=cp-cp69_base
    points=[]
    for number,d in enumerate(cp69_b_compare.itertuples(),1):
        end=at(CP69,d.MINUTE+1)
        minute=cp69_raw['TC-Z3'].loc[end-pd.Timedelta(minutes=1):end-pd.Timedelta(seconds=1)].dropna()
        tc_at=minute.idxmax()
        op=cp69_raw['OP-Z3'].loc[d.POINT-pd.Timedelta(minutes=5):d.POINT+pd.Timedelta(minutes=5)].dropna()
        points.append(dict(ID=f'b-{number}',END=end,TC_AT=tc_at,TC=float(minute.loc[tc_at]),
                           OP_AT=op.idxmin(),OP=float(op.min())))
    return dict(START=a,END=b,DATA=q,POINTS=points,CP_MAX_AT=dev.abs().idxmax(),
                CP_MAX_DEV=float(dev.abs().max()),CP_HITS=int(dev.abs().gt(.05).sum()),
                CP_VALID=len(cp),SECONDS=len(q))

def cp69_results(section):
    if section=='a':
        d=cp69_signalstats.loc[cp69_signalstats.ID.ne('240069-c1')]
        tc=d.loc[d.SENSOR.eq('TC-Z3')];op=d.loc[d.SENSOR.eq('OP-Z3')]
        return f'''**이 그림으로 확인한 것**

- (a)는 CP 피크 6개를 각각 확인한다. 최대편차는 피크마다 다르고 TC-Z3의 최대 절대편차 방향은 증가 {(tc.MAX_DEV>0).sum()}개·감소 {(tc.MAX_DEV<0).sum()}개다. OP-Z3도 감소 {(op.MAX_DEV<0).sum()}개·증가 {(op.MAX_DEV>0).sum()}개로 한 방향으로 반복되지 않는다.
- CP 시작 앞뒤에서 TC·OP가 움직이는 시각도 다르다. 따라서 **TC-Z3 상승 하나로 6개 CP 상승을 모두 설명하기 어렵다.** 크기·방향·최대편차 위치는 반복 반응을 찾는 단서이며, 실제 반응 시작 또는 인과를 뜻하지 않는다.
- **왜 Z3부터 보았는가:** 기존 온도 선별의 최대 시간순 Z가 TC-Z3에서 관측되어 설명 예시로 먼저 확대했다. CP와 가장 강하게 연결됐다는 검정이나 물리적 대응을 근거로 고른 것은 아니다. 바로 위 Z4~Z6 그림과 같은 기준으로 함께 판단한다.
- TC-Z1–Z9·전체 OP·세정·냉각·CP-Z4M까지 CP 외 21신호를 모두 같은 기준으로 계산했다. 접힌 표의 147행은 개별 신호를 다시 확대할 위치를 찾는 목록이다. 단위가 다른 편차를 한 크기로 순위 매기지 않는다.
- CP 계측 정의·원시 변환식·센서 보정·SV·제어/라인 조건이 없어 **CP가 튄 물리적 원인은 아직 미확정**이다. CP-Z4M도 같은 단위의 독립 검증 센서라고 가정할 수 없다.'''
    if section=='b':
        left,right=list(cp69_b_compare.itertuples());evidence=cp69_b_observations()
        gap=(right.POINT-left.POINT).total_seconds()/60
        return f'''**결과: TC-Z3 온도 상승과 OP-Z3 저하가 보여도 CP가 반드시 튀지는 않는다.**

- **b-2는 b-1을 더 확대한 것이 아니다.** b-1은 {left.POINT:%m/%d %H:%M}의 Z 선별 지점이고, b-2는 약 {gap:.0f}분 뒤 {right.POINT:%H:%M:%S}에 나타난 별도의 반응이다. 뒤의 더 높은 온도를 생략하지 않기 위해 함께 확인했다. 그래프는 {evidence['START']:%H:%M}–{evidence['END']:%H:%M}를 끊지 않고 보여준다.
- **그림에서 직접 읽는 것:** 두 반응 모두 TC-Z3가 상승한 뒤 하강하고, OP-Z3는 낮아졌다가 복귀한다. 그동안 기록된 CP의 LOT 중앙값 대비 최대 절대편차는 **{evidence['CP_MAX_DEV']:.4f}로 0.05보다 작다.** 비교창의 CP 경계 초과 기록은 {evidence['CP_HITS']}초다. 관측된 {evidence['CP_VALID']:,}초를 기준으로 하며 미관측 {evidence['SECONDS']-evidence['CP_VALID']:,}초를 정상으로 채우지 않았다.
- **이 대조가 필요한 이유:** (a)·(c)에서는 CP 피크 주변의 TC·OP를 보았다. 여기서는 반대로 **TC·OP가 크게 움직인 때에도 CP 피크가 나타나는가**를 확인한다. 두 사례에서 CP 피크가 없으므로, TC-Z3 상승 또는 OP-Z3 저하 하나만으로 CP 피크를 설명하거나 선별하기 어렵다. 같은 시각의 관측 범위에 대한 결과이며 장시간 지연이나 다른 조건에서의 관계를 배제한 것은 아니다.
- **왜 앞의 작은 온도를 먼저 골랐는가:** b-1은 최고 온도가 아니라 과거 대비 Z가 가장 컸다. 1분 중앙값 {left.X:.2f}℃·과거 SD {left.SD:.2f}℃에서 Z={left.Z:+.2f}였다. 뒤의 b-2는 원본 최고 {cp69_raw.loc[cp69_later_point,'TC-Z3']:.2f}℃·1분 중앙값 {right.X:.2f}℃로 더 높지만, 과거 SD도 {right.SD:.2f}℃로 커져 Z={right.Z:+.2f}였다. **온도의 크기와 이전 대비 이례성은 다르다.** 상세 계산은 접힌 표에서 확인한다.
- **여기서 결론 내리지 않는 것:** CP가 튄 물리적 원인, 두 온도 반응의 정상 여부, OP 저하와 TC 상승 사이의 인과다. 이 그림은 CP 원인 검토에 쓰는 대조 근거이며 설비 정상 판정의 근거는 아니다.'''
    before=cp69_variability.loc[cp69_variability.PHASE.eq('전')].set_index('WINDOW')
    after=cp69_variability.loc[cp69_variability.PHASE.eq('후')].set_index('WINDOW')
    return f'''**CP 이후 TC-Z3가 흔들리는 것은 어떤 관계인가?**

- CP 최고 {cp69_c.PEAK:%m/%d %H:%M:%S} 뒤 TC-Z3·OP-Z3의 큰 반응이 관측된다. 주석은 후 0~15분·15~30분 각각의 TC 최고·OP 최저까지의 시간 차이를 표시한다. 뒤의 반복 반응도 함께 확인한다. 이는 선후 관계이며 전달 지연이나 원인을 추정한 값은 아니다.
- 피크 전 30~5분과 후 5~30분의 TC-Z3 SD는 {before.loc['전후 25분','TC3_SD']:.2f}→{after.loc['전후 25분','TC3_SD']:.2f}℃다. 전 60~10분과 후 10~60분으로 넓히면 {before.loc['전후 50분','TC3_SD']:.2f}→{after.loc['전후 50분','TC3_SD']:.2f}℃다. **비교창에 따라 증가 폭이 달라진다.** OP-Z3<10% 비율은 0 부근의 출력을 요약하는 기술량이며 공정 허용 한계가 아니다.
- 전체 LOT의 이동 SD를 보면 CP 전에 이미 큰 흔들림이 있다. 특히 07/07 23시대의 원본 TC-Z3 SD는 {cp69_raw['TC-Z3'].loc['2024-07-07 23:00':'2024-07-07 23:59:59'].std():.2f}℃다. CP가 변동성을 처음 만들었다고 볼 근거가 부족하다. 독립된 동시 변화, 공유된 운전 조건, 계측·제어 경로라는 설명을 구분해 확인해야 한다.

**뒤의 정비가 CP 원인을 설명하는가?**

- CP 최대 이후 30일 안의 CAL01 감지 기록은 3 WO다. 07/08 BM은 “1호기 재발, 절연 저하”와 O링 3개 예방 교체, 07/10 TBM은 REMARK 미기재, 07/25 BM은 가스 분석계 셀 1개 손상 교체다. 정비 건수가 많다는 인상만으로 CP와 연결하지 않는다.
- 특히 **가스 분석계 셀 손상 교체는 계측 경로를 확인할 단서**지만, CP 최대에서 감지까지 약 {(cp69_works.iloc[-1].DETC_DT-cp69_c.PEAK).total_seconds()/86400:.1f}일 떨어져 있다. CP 채널과 INS-006의 연결, 손상 진단·보정 기록이 없어 원인으로 확정할 수 없다. 07/08 절연 저하도 CP와 동일 원인인지 확인되지 않았다.
- 모든 작업은 240069 관측 뒤다. 다음 온도 관측은 09/08의 240091이며 그동안 운전 조건도 달라질 수 있어 수리로 개선됐는지 검증할 근접 관측이 없다.

**현재 가장 필요한 검증:** CP-Z4·CP-Z4M의 측정/계산 정의와 INS-006 연결 → 각 CP 시작 전후 제어·가스/전기·라인 로그 → 07/08·07/25 실제 작업 진단과 보정 결과 → 조치 직전·직후 같은 조건의 관측 순서로 확인한다. 단일 상관계수로 원인을 단정하지 않는다.'''

def draw_cp69_zone_peaks():
    rows=[]
    for start in [0,3]:
        group=list(cp69_pulses.iloc[start:start+3].itertuples())
        rows.append({'panels':[(event_text(p.ID,include_lot=False)+' CP-Z4',['CP-Z4']) for p in group],'height':1.45,
            'captions':[f'시작 {p.ONSET:%H:%M:%S} · CP 피크 {p.PEAK:%H:%M:%S} · 편차 +{p.DEV:.3f}' for p in group]})
        for prefix in ['TC','OP']:
            sensors=[f'{prefix}-Z{z}' for z in [4,5,6]]
            rows.append({'panels':[(event_text(p.ID,include_lot=False)+' '+prefix+'-Z4·Z5·Z6',sensors) for p in group],'height':1.6})
    fig,cells=report_page('240069-a | CP 피크별 Z4·Z5·Z6의 온도·출력 반응',
        '각 열은 같은 CP 피크다. CP 원본 → TC-Z4~Z6 변화 → OP-Z4~Z6 변화. Z3 비교와 같은 전 3분~후 4분을 사용한다.',rows,width=24,
        info=['TC·OP는 CP 시작 전 5~1분 중앙값을 뺀 변화량이다. 0은 해당 구간의 비교 기준이며 정상 한계가 아니다.',
              '각 신호군은 6개 피크에 같은 세로축이다. 음영=CP 이탈 기록 범위 · 세로 점선=CP 피크 · 색 점=그 시각의 실제 관측값 변화.'])
    for index,row in enumerate(cells):
        start=0 if index<3 else 3;kind=['CP','TC','OP'][index%3]
        for (ax,head,sensors),pulse in zip(row,cp69_pulses.iloc[start:start+3].itertuples()):
            if kind=='CP':cp69_event_panel(ax,head,pulse,'CP-Z4',annotate=True);continue
            a=pulse.ONSET-pd.Timedelta(minutes=3);b=pulse.ONSET+pd.Timedelta(minutes=4)
            for sensor in sensors:
                base=cp69_zone_metrics.loc[cp69_zone_metrics.ID.eq(pulse.ID)&cp69_zone_metrics.SENSOR.eq(sensor),'BASE'].iloc[0]
                y=cp69_raw[sensor].loc[a:b]-base
                ax.plot(y.index,y,color=sensor_color(sensor),lw=1.15,label=sensor)
                value=cp69_raw.loc[pulse.PEAK,sensor]-base
                ax.scatter(pulse.PEAK,value,s=20,color=sensor_color(sensor),edgecolors='white',linewidths=.4,zorder=6)
                cp69_zone_peak_checks.append((pulse.ID,sensor))
            ax.axhline(0,color=GREY,lw=.7,ls='--');ax.axvline(pulse.PEAK,color=EVENT_COLORS['CP'],ls=':',lw=.8)
            ax.axvspan(pulse.ONSET,at(CP69,pulse.B).round('s'),color=EVENT_COLORS['CP'],alpha=.12,lw=0)
            ax.set_xlim(a,b);ax.set_ylim(*cp69_zone_limits[kind])
            ax.set_xticks([pulse.ONSET+pd.Timedelta(minutes=m) for m in [-2,0,2,4]],['-2','0','+2','+4'])
            ax.set_xlabel('CP 이탈 시작 대비 [분]');ax.set_ylabel('기준 대비 변화 [℃]' if kind=='TC' else '기준 대비 변화 [%p]');panel_legend(head,ax)
    emit(fig,'01_개요/사례_240069_a_Z4-Z6_피크별대조.png')
    for prefix in ['TC','OP']:
        d=cp69_zone_metrics.loc[cp69_zone_metrics.SENSOR.str.startswith(prefix)]
        frame=d.pivot(index='ID',columns='SENSOR',values='AT_PEAK').reindex(cp69_pulses.ID).reset_index()
        frame.insert(1,'CP 편차',cp69_pulses.DEV.to_numpy())
        note('**CP 피크 전후 15초 중앙값의 기준 대비 변화 · '+('TC [℃]' if prefix=='TC' else 'OP [%p]')+'**')
        show_table(frame.rename(columns={'ID':'피크'}))
    note('**두 표의 기준:** CP 시작 전 5~1분 중앙값을 빼서 계산했다. 같은 CP 피크 전후 15초를 대조한 값이다. 그래프의 색 점은 피크 시각의 1초 실제 값이므로 31초 중앙값과 구분한다. c1은 아래 c 상세의 큰 CP 사건이다.')
    table(cp69_zone_metrics[['ID','SENSOR','MAX_DEV','OFFSET_SEC','N_PEAK']].rename(columns={
        'ID':'피크','SENSOR':'신호','MAX_DEV':'시작 전후 2분 최대절대편차의 부호 있는 값','OFFSET_SEC':'그 값의 위치[시작 대비 초]','N_PEAK':'피크 전후 15초 실측 수'}))
    note(cp69_zone_result())
    assert len(cp69_zone_metrics)==42 and cp69_zone_metrics.N_PEAK.ge(24).all()
    assert len(cp69_zone_peak_checks)==36 and set(cp69_zone_peak_checks)=={(p.ID,s) for p in cp69_pulses.iloc[:6].itertuples() for s in CP69_ZONE_SENSORS}

def cp69_zone_result():
    d=cp69_zone_metrics.pivot(index='ID',columns='SENSOR',values='AT_PEAK')
    r1,r2,r7=d.loc['240069-a1'],d.loc['240069-a2'],d.loc['240069-c1']
    near=cp69_zone_metrics.loc[cp69_zone_metrics.ID.ne('240069-c1')]
    mt=near.loc[near.SENSOR.str.startswith('TC'),'MAX_DEV'].abs().max()
    mo=near.loc[near.SENSOR.str.startswith('OP'),'MAX_DEV'].abs().max()
    return f'''**Z4~Z6에서 확인한 관계와 남는 질문**

- a1의 CP 편차는 +{cp69_pulses.iloc[0].DEV:.3f}다. 같은 피크 부근의 기준 대비 온도는 TC-Z4 {r1['TC-Z4']:+.3f}℃, TC-Z5 {r1['TC-Z5']:+.3f}℃, TC-Z6 {r1['TC-Z6']:+.3f}℃다. Z4와 Z5·Z6가 같은 방향으로 움직인 사례가 아니다.
- a2에서도 CP는 +{cp69_pulses.iloc[1].DEV:.3f} 상승했지만, TC-Z4 {r2['TC-Z4']:+.3f}℃·TC-Z5 {r2['TC-Z5']:+.3f}℃·TC-Z6 {r2['TC-Z6']:+.3f}℃로 모두 기준보다 낮다. a5·a6의 반응도 표와 곡선으로 함께 확인한다. **양의 CP 피크마다 같은 온도 상승이 반복되지는 않는다.**
- 첫 6개 사건의 시작 전후 2분에서 Z4~Z6 최대 절대편차는 온도 {mt:.3f}℃, 출력 {mo:.3f}%p다. 이는 탐색창의 관측 크기이며 정상·이상의 공정 한계가 아니다. OP 변화의 방향과 최대편차 시각도 피크마다 다르다.
- 가장 큰 c1의 CP 편차는 +{cp69_c.DEV:.3f}인데, 피크 부근 TC-Z4 {r7['TC-Z4']:+.3f}℃·TC-Z5 {r7['TC-Z5']:+.3f}℃·TC-Z6 {r7['TC-Z6']:+.3f}℃다. 이 사건도 Z4~Z6 전체의 동시 상승 하나로 설명할 근거가 부족하다. Z3 외 신호를 생략해서 도출한 결론이 아니다.
- 현재 확인한 것은 **정해진 근접 시간창의 동반 반응이 일정하지 않다**는 사실이다. 6개 사건이 밀집해 있어 긴 지연 응답이 겹친다. 이 결과로 더 긴 지연 관계나 공통 운전 조건의 영향을 배제하지 않는다.
- CP-Z4라는 이름만으로 TC-Z4와 같은 계측점이거나 물리적으로 직접 연결됐다고 가정하지 않는다. 채널 정의·계측점·SV·제어 로그가 확인되면 대응 채널과 지연 시간부터 정해 다시 검증해야 한다.'''

draw_cp69_peaks()
note(cp69_results("a"))

#### 5.1.2 (b) 두 차례 TC-Z3·OP-Z3 반응에서 CP를 대조한다

**(b-1)과 (b-2)는 약 34분 떨어진 별도의 반응이다.** 앞 구간을 더 확대한 것이 아니다. 두 반응 사이를 포함한 연속 시간축에서 TC-Z3·OP-Z3와 CP를 대조하고, 아래 해석에서 이 결과가 CP 원인 검토에 어떤 의미가 있는지 설명한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def draw_cp69_b():
    evidence=cp69_b_observations();a,b=evidence['START'],evidence['END']
    first,second=list(cp69_b_compare.itertuples())
    assert evidence['CP_HITS']==0 and evidence['CP_MAX_DEV']<.05
    assert evidence['POINTS'][1]['TC_AT']==cp69_later_point
    cards=[dict(title='b-1 · 먼저 선별한 반응',value=f'{first.POINT:%H:%M} · 과거 대비 Z 최대',
                lines=[f'1분 중앙값 {first.X:.2f}℃ · Z={first.Z:+.2f}',
                       '직전보다 얼마나 이례적인지로 선별'],color=EVENT_COLORS['Z']),
           dict(title='b-2 · 약 34분 뒤 별도 반응',value=f'{cp69_later_point:%H:%M:%S} · {cp69_raw.loc[cp69_later_point,"TC-Z3"]:.2f}℃',
                lines=['b-1 뒤 1시간 안의 원본 TC-Z3 최고',
                       'b-1을 더 확대한 그림이 아님'],color=EVENT_COLORS['Z']),
           dict(title='두 반응에서 확인한 CP 결과',value='CP 탐색 경계 초과 0초',
                lines=[f'전체 비교창 최대 절대편차 {evidence["CP_MAX_DEV"]:.4f} < 0.05',
                       f'CP 실측 {evidence["CP_VALID"]:,}/{evidence["SECONDS"]:,}초 · 미관측은 판정 제외'],color='#477B94')]
    rows=[dict(panels=[(f'전체 LOT에서 아래 비교 범위의 위치 | 07/07 {a:%H:%M}–{b:%H:%M}',['TC-Z3'])],height=.65),
          dict(panels=[('TC-Z3 | 두 차례의 온도 상승과 이후 하강',['TC-Z3'])],height=2.05),
          dict(panels=[('OP-Z3 | 같은 시각의 출력 저하와 복귀',['OP-Z3'])],height=1.55),
          dict(panels=[('CP-Z4 | 두 반응과 그 사이 구간을 같은 경계로 대조',['CP-Z4'])],height=1.45)]
    fig,cells=report_page('240069-(b) | 두 차례 TC-Z3 반응에도 CP 탐색 경계 초과는 없었다',
        'b-1과 b-2는 서로 다른 시각의 반응이다. 아래 세 그래프는 하나의 연속 시간축이며, TC·OP 변화만으로 CP 피크를 설명할 수 있는지 대조한다.',
        rows,width=20,cards=cards,
        info=['주황 음영=각 선정 시점의 1분창 · 지점 주석=원본 값 · CP 점선=LOT 중앙값 ±0.05(탐색 기준, 공정 허용 한계 아님).'])
    ax,head,_=cells[0][0]
    whole=cp69_raw['TC-Z3']
    ax.plot(whole.index,whole,color='#aebac3',lw=.65,label='TC-Z3 · 전체 LOT')
    selected=whole.loc[a:b]
    ax.axvspan(a,b,color=EVENT_COLORS['Z'],alpha=.14,lw=0)
    ax.plot(selected.index,selected,color=EVENT_COLORS['Z'],lw=1.15,label='아래 연속 비교창')
    clock_in_lot(ax,CP69);ax.set_ylabel('TC-Z3 [℃]');panel_legend(head,ax)
    for row,sensor in zip(cells[1:],['TC-Z3','OP-Z3','CP-Z4']):
        ax,head,_=row[0]
        report_signal(ax,CP69,sensor,a,b,fine=True)
        zoom_clock(ax,a,b);ax.set_ylabel(display_unit(sensor));ax.set_xlabel('07/07 실제 측정 시각 · 같은 가로축')
        for point in evidence['POINTS']:
            ax.axvspan(point['END']-pd.Timedelta(minutes=1),point['END'],color=EVENT_COLORS['Z'],alpha=.16,lw=0)
        if sensor=='TC-Z3':
            ax.set_ylim(846,876)
            for point in evidence['POINTS']:
                ax.plot(point['TC_AT'],point['TC'],'o',ms=4,mfc='white',mec=EVENT_COLORS['Z'])
                near_note(ax,point['TC_AT'],point['TC'],
                          f'{point["ID"]} · {point["TC_AT"]:%H:%M:%S}\n원본 {point["TC"]:.2f}℃',EVENT_COLORS['Z'])
        elif sensor=='OP-Z3':
            ax.set_ylim(-18,119)
            for point in evidence['POINTS']:
                ax.plot(point['OP_AT'],point['OP'],'o',ms=4,mfc='white',mec=EVENT_COLORS['Z'])
                near_note(ax,point['OP_AT'],point['OP'],
                          f'{point["ID"]} 부근 · {point["OP_AT"]:%H:%M:%S}\nOP-Z3 최저 {point["OP"]:.1f}%',EVENT_COLORS['Z'])
        else:
            for v in [cp69_base-.05,cp69_base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.85)
            ax.set_ylim(cp69_base-.085,cp69_base+.085)
            t=evidence['CP_MAX_AT'];value=cp69_raw.loc[t,'CP-Z4']
            near_note(ax,t,value,f'최대 절대편차 {evidence["CP_MAX_DEV"]:.4f} < 0.05',color='#477B94')
        panel_legend(head,ax,extra=[Line2D([0],[0],color=EVENT_COLORS['CP'],ls='--',lw=.85,label='LOT 중앙값 ±0.05')] if sensor=='CP-Z4' else ())
    emit(fig,'01_개요/사례_240069_b_Z와후속온도.png')
    comparison=cp69_b_compare.copy()
    comparison['Z 계산 1분창 끝']=[at(CP69,m+1) for m in comparison.MINUTE]
    table(comparison.rename(columns={'LABEL':'서로 다른 두 반응','POINT':'선정 기준 시각','X':'1분 중앙값[℃]','REF':'과거 중앙값[℃]',
        'SD':'과거 SD[℃]','Z':'시간순 Z','CP_DEV':'선정 시각 ±2분 CP 최대절대편차'}).drop(columns=['MINUTE']))

draw_cp69_b()
note(cp69_results("b"))

#### 5.1.3 (c) 전후 변동성과 이후 30일의 정비를 대조한다

**정비 이력의 해석:** 가스 분석계 셀 손상 교체와 CP 변화의 연결은 현재 보류한다. 부품명과 CP라는 채널명만으로 관련 계측 경로라고 가정하지 않는다. 정의와 설비 대응이 확인된 뒤 작업 진단·보정 결과 및 조치 전후 자료로 검증할 항목이다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def draw_cp69_c():
    e=next(e for e in LOT_EVENTS[CP69] if e['tag']=='(c)');t=e['point']
    a=t-pd.Timedelta(minutes=20);b=t+pd.Timedelta(minutes=30)
    rows=[{'panels':[('전체 LOT | TC-Z3의 10분 이동 표준편차',[])],'height':1.4},
          {'panels':[('(c-1) CP-Z4 원본',['CP-Z4']),('CP-Z4M · 별도 정의의 계측값',['CP-Z4M'])],'height':2.0},
          {'panels':[('TC-Z3 · CP 전후의 온도',['TC-Z3']),('OP-Z3 · CP 전후의 대응 출력',['OP-Z3'])],'height':1.9},
          {'panels':[('TC-Z4·TC-Z5·TC-Z6',['TC-Z4','TC-Z5','TC-Z6']),('OP-Z4·OP-Z5·OP-Z6',['OP-Z4','OP-Z5','OP-Z6'])],'height':1.3},
          {'panels':[('별도 날짜축 | CP 최대 이후 30일의 CAL01 실제 WO·REMARK',[])],'height':2.3}]
    fig,cells=report_page('240069-c | CP 뒤의 TC-Z3 변화와 정비를 어떻게 해석하는가?',
        '가운데 3행은 같은 실제 시각이다. 맨 위는 LOT 전체 변동성, 맨 아래는 이후 30일의 별도 날짜축이다. 선후와 원인을 구분한다.',rows,width=24,
        info=['온도 변동은 CP 이전에도 있다. 전후 비교창을 바꾸면 SD 차이도 달라지므로 CP가 변동성 증가를 일으켰다고 결론 내리지 않는다.',
              '10분 SD는 원본 600초 중 실측 540개 이상에서만 표시한다. OP·CP의 크기와 직접 비교하는 점수가 아니다.'])
    ax,head,_=cells[0][0];ax.plot(cp69_sd.index,cp69_sd,color=sensor_color('TC-Z3'),lw=1.1,label='TC-Z3 · 10분 SD')
    ax.axvspan(a,b,color=EVENT_COLORS['CP'],alpha=.12,lw=0)
    for pulse in cp69_pulses.itertuples():ax.scatter(pulse.PEAK,cp69_sd.get(pulse.PEAK,np.nan),color=EVENT_COLORS['CP'],s=18,zorder=5)
    peak=cp69_sd.idxmax();near_note(ax,peak,cp69_sd.loc[peak],f'{peak:%H:%M} · SD {cp69_sd.loc[peak]:.2f}℃')
    clock_in_lot(ax,CP69);ax.set_ylabel('TC-Z3 SD [℃]');panel_legend(head,ax,[Line2D([0],[0],marker='o',color=EVENT_COLORS['CP'],lw=0,label='7개 CP 피크 시각')])
    for row in cells[1:4]:
        for ax,head,sensors in row:
            if sensors==['CP-Z4']:plot_event_trace(ax,head,CP69,e,a,b,large=True)
            else:
                for sensor in sensors:report_signal(ax,CP69,sensor,a,b,fine=True)
                ax.axvspan(t-pd.Timedelta(seconds=30),t+pd.Timedelta(seconds=30),color=EVENT_COLORS['CP'],alpha=.15,lw=0)
                zoom_clock(ax,a,b);ax.set_ylabel(display_unit(sensors[0]));panel_legend(head,ax)
                if sensors in [['TC-Z3'],['OP-Z3']]:
                    sensor=sensors[0]
                    for lo,hi in [(0,15),(15,30)]:
                        q=cp69_raw[sensor].loc[t+pd.Timedelta(minutes=lo):t+pd.Timedelta(minutes=hi)].dropna()
                        pt=q.idxmax() if sensor=='TC-Z3' else q.idxmin();value=q.loc[pt]
                        near_note(ax,pt,value,f'{pt:%H:%M:%S} · {value:.2f}'+('℃' if sensor in TC else '%')+chr(10)+f'CP 피크 뒤 {(pt-t).total_seconds()/60:.2f}분')
    ax,head,_=cells[-1][0];r=lot_meta.loc[CP69]
    ax.plot([r.START,r.END],[0,0],color='#147f9c',lw=6,label='온도 관측창')
    for i,w in enumerate(cp69_works.itertuples(),1):
        color=COL[w.MNT_TYPE];ax.plot([w.DETC_DT,w.STRT_DT],[i,i],color=color,ls=':',lw=1.3)
        ax.plot([w.STRT_DT,w.END_DT],[i,i],color=color,lw=6);ax.scatter(w.DETC_DT,i,color=EVENT_COLORS['감지'],marker='D',s=25,zorder=6)
        parts=' / '.join(parts_for(w.WO_NO));label=f'{w.STRT_DT:%m/%d %H:%M} · {remark_text(w)}'+(' · '+parts if parts else '')
        ax.text(w.END_DT+pd.Timedelta(hours=8),i,label,ha='left',va='center',fontsize=9.5,color=color)
    ax.set_yticks(range(4),['LOT 240069']+[f'{w.MNT_TYPE} {w.WO_NO[-5:]}' for w in cp69_works.itertuples()],fontsize=9)
    ax.set_ylim(3.6,-.6);clock_ticks(ax,r.START-pd.Timedelta(hours=6),t+pd.Timedelta(days=30));ax.set_xlabel('실제 기록 날짜 [월/일] · 위의 분 단위 확대와 다른 축')
    panel_legend(head,ax,[Line2D([0],[0],marker='D',color=EVENT_COLORS['감지'],lw=0,label='정비 감지'),Line2D([0],[0],color=COL['BM'],lw=4,label='실제 작업')])
    emit(fig,'01_개요/사례_240069_c_선후와정비.png')
    cp69_peak_checks.append('240069-c1')
    show_table(cp69_variability.rename(columns={'WINDOW':'비교창','PHASE':'전후','TC3_SD':'TC-Z3 SD[℃]','TC3_MIN':'최저[℃]','TC3_MAX':'최고[℃]',
        'OP3_LOW_PCT':'OP-Z3<10% 기록 비율[%]','N':'실측 수'}))
    show_table(cp69_after_work)

draw_cp69_c()
note(cp69_results("c"))

<a id="case-240037"></a>
### 5.2 LOT 240037 — 짧은 CP 펄스와 국소 변화

**분석 질문:** 짧은 CP가 반복되는 동안 TC·OP도 같은 폭으로 이탈하는가?

**선정 근거:** CP 3사건. (a)의 2사건과 (c)의 1사건을 모두 원본으로 확인하고, (b)의 TC-Z3 국소 변화는 별도 근거로 대조한다. 1분 중앙값만으로 짧은 CP를 검사하면 일부 사건이 작게 요약된다.

**정비 대조:** 관측 중 실제 작업·감지 교집합은 없다. 가까운 WO라는 이유로 펄스 원인과 연결하지 않는다.

**잠정 판단:** CP 원본·동시 TC/OP·계측 정의를 함께 확인할 대상이다. 반복 자체가 고장 확정은 아니다.

In [ ]:
# draw_report_case 생성 함수 정의: 5.1. 같은 레이아웃을 재사용한다.
draw_report_case(240037)

<a id="case-240117"></a>
### 5.3 LOT 240117 — CBM 완료 뒤 CP 하강

**분석 질문:** 이 CP 하강이 해당 CBM을 유발한 사전 경보인가?

**선정 근거와 시간 순서:** (a) 수리 후 승온, (b) CP 하강, (c) CLN-IN 국소 변화를 구분한다. CBM 01854 REMARK는 “마모량 초과”, 완료 12:44다. (b)의 CP 최저 13:04:50은 완료 후 20.83분이다. TBM 01859 “계획 점검 수행”도 07:40–09:35에 진행됐다.

**잠정 판단:** 이 CBM의 사전 경보라는 설명과 시간 순서가 맞지 않는다. 수리 완료 뒤 발생했다는 이유만으로 수리 실패를 확정하지도 않는다. 실제 작업 대상·완료 점검·재가동 조건과 CP 계측 로그가 필요하다.

In [ ]:
# draw_report_case 생성 함수 정의: 5.1. 같은 레이아웃을 재사용한다.
draw_report_case(240117)

<a id="case-240061"></a>
### 5.4 LOT 240061 — 같은 SPHC에서 출력 수준 차이

**분석 질문:** 비슷한 온도 중심 수준에서 출력 분포가 다른가, 일부 점만 튀는가?

**선정 근거:** 승온을 제외한 OP-Z4·OP-Z8이 다른 SPHC LOT 대비 표시 경계를 넘는다. TC-Z4·TC-Z8도 함께 보여 온도 자체의 이상으로 오해하지 않게 한다. 같은 강종 4 LOT의 분포 위치·평균·중앙값·폭을 확인한다.

**정비·조건 대조:** 6월 계획수리 후 첫 관측이다. 승온은 제외했지만 설정·속도·두께·부하·보정이 같은지는 확인되지 않았다. 전후 강종도 다르므로 수준 차이를 수리 효과로 비교하지 않는다.

**잠정 판단:** 운전 조건·설정 확인 대상이다. 출력 차이를 고장·에너지 증가로 확정하지 않는다. 이 LOT의 (a)(b)(c) 원본 상세는 부록의 같은 번호로 보존했다.

In [ ]:
# draw_report_case 생성 함수 정의: 5.1. 같은 레이아웃을 재사용한다.
draw_report_case(240061)

### 5.5 계획수리 전(A)·후(B) — 전환과 남는 국소 변화

수리 기간 안에는 온도 관측이 없다. 캘린더·실제 WO·REMARK를 먼저 확인하고, A의 하강과 B의 승온을 원본으로 대조한다. (B-1)은 전환 뒤 남는 TC-Z3 반응 또는 CP 이탈을 확인하기 위한 창이다. 전후 강종이 달라 온도·출력 높낮이를 개선 효과로 해석하지 않는다.

<a id="repair-june"></a>
#### 6월 정기수리 · 일정 → 전후 원본 → 같은 시각 확대

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def plan_case_data(idx,pre,post):
    plan=plans.iloc[idx];pr,po=lot_meta.loc[pre],lot_meta.loc[post]
    ws=cal_work.loc[cal_work.STRT_DT.lt(po.END)&cal_work.END_DT.gt(pr.START)].sort_values('STRT_DT')
    before_a=max(pr.START,pr.END-pd.Timedelta(minutes=100));before_b=pr.END
    after_a,after_b=po.START,po.END
    # 승온 뒤 남은 단발 응답을 본다. 11월은 CBM 뒤 CP 사건과 TC·OP를 같은 시각에 대조한다.
    if post==240117:
        hit=cp_real.loc[cp_real.LOT.eq(post)].iloc[0];point=hit.PEAK
        zoom_a,zoom_b=point-pd.Timedelta(minutes=18),point+pd.Timedelta(minutes=12)
    else:
        d=zstats[post,'TC-Z3'].Z.where(states[post].eq('안정 참고'))
        m=int(d.abs().idxmax());point=at(post,m+1)
        zoom_a,zoom_b=point-pd.Timedelta(minutes=10),point+pd.Timedelta(minutes=16)
    return dict(plan=plan,pre=pr,post=po,ws=ws,pre_a=before_a,pre_b=before_b,post_a=after_a,post_b=after_b,point=point,zoom_a=zoom_a,zoom_b=zoom_b)

def draw_plan_case(idx,pre,post,slug):
    d=plan_case_data(idx,pre,post);plan,pr,po,ws=d['plan'],d['pre'],d['post'],d['ws']
    month='6월' if idx==0 else '11월'
    focus_kind='CP' if post==240117 else 'Z'
    focus_color=EVENT_COLORS[focus_kind]
    rows=[{'panels':[('① 정기수리 일정 · 강종 · 관측창 · 실제 WO 작업',[])],'height':2.3+len(ws)*.34},
          {'panels':[(f'수리 전(A) {pre} · {pr.GRADE} | 온도',HEAT),(f'수리 후(B) {post} · {po.GRADE} | 온도',HEAT)],'height':2.0},
          {'panels':[(f'수리 전(A) {pre} | 출력',[f'OP-Z{i}' for i in range(3,7)]),(f'수리 후(B) {post} | 출력',[f'OP-Z{i}' for i in range(3,7)])],'height':2.0},
          {'panels':[('(B-1) 수리 후 확인 구간 · 온도',HEAT),('(B-1) 같은 시각 · 출력',[f'OP-Z{i}' for i in range(3,7)]),('(B-1) 같은 시각 · CP',['CP-Z4'])],'height':1.8}]
    info=[f'계획: {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d} (종료일 포함) · {plan.SCOPE} | 이 기간 온도 기록 0행',
          f'전 {pre} {pr.GRADE} {pr.START:%m/%d %H:%M}–{pr.END:%H:%M} / 후 {post} {po.GRADE} {po.START:%m/%d %H:%M}–{po.END:%H:%M}',
          f'전체 시각 범위에 실제 작업 {len(ws)} WO · ▼=감지 / 실선=착수~완료 / ◆=교체 WO 완료(교체 시각 대용) · 음영 안을 보간하지 않음']
    fig,cells=report_page(f'{month} 정기수리 | 일정 → 전후 원본 → 같은 구간 확대',
        '먼저 일정과 다신호의 함께 움직임을 확인하고, 승온·하강 이후 남는 변화를 따로 본다. 전후 강종이 달라 개선 효과로 해석하지 않는다.',rows,width=26,info=info,event_key=True)
    ax,head,_=cells[0][0]
    lo,hi=pr.START-pd.Timedelta(hours=4),po.END+pd.Timedelta(hours=4)
    ax.axvspan(plan.STRT_DT,plan.END_EXCL,color=COL['PLAN'],alpha=.48,lw=0)
    changes=grade_evt.loc[grade_evt.EVT_DT.gt(lo)&grade_evt.EVT_DT.lt(hi)]
    earlier=grade_evt.loc[grade_evt.EVT_DT.le(lo)].tail(1)
    points=[lo]+changes.EVT_DT.tolist()+[hi]
    grades=([earlier.AFT_VAL.iloc[0]] if len(earlier) else ['미기록'])+changes.AFT_VAL.tolist()
    grade_colors={'DP590':'#3f6385','SPCC':'#8e9da9','SPHC':'#ac8c62','SPCEN':'#8e7bb7','SPCE':'#4c929b','SPFH590':'#b66d55','SPCD':'#4b7fb3'}
    for a,b,g in zip(points[:-1],points[1:],grades):
        ax.plot([a,b],[0,0],lw=15,color=grade_colors.get(g,GREY),solid_capstyle='butt')
        if (b-a)/(hi-lo)>.055:ax.text(a+(b-a)/2,0,g,ha='center',va='center',fontsize=10,color='white',fontweight='bold')
    for key in [pre,post]:
        r=lot_meta.loc[key];ax.plot([r.START,r.END],[1,1],color=LOT_COLOR[key],lw=7,solid_capstyle='butt')
        # 짧은 관측 막대의 날짜·시각은 상단 정보에 모아 표시한다.
    for tag,key,align in [('A',pre,'left'),('B',post,'right')]:
        rr=lot_meta.loc[key];xx=rr.START if tag=='A' else rr.END
        ax.text(xx,1.33,f'{tag} · '+('수리 전' if tag=='A' else '수리 후'),ha=align,va='center',fontsize=11,fontweight='bold',color='#17344d')
    for i,w in enumerate(ws.itertuples(),2):
        color=COL[w.MNT_TYPE]
        ax.plot([w.DETC_DT,w.STRT_DT],[i,i],color=color,ls=':',lw=1.2)
        ax.plot([w.STRT_DT,w.END_DT],[i,i],color=color,lw=6,solid_capstyle='butt')
        if lo<=w.DETC_DT<=hi:ax.scatter(w.DETC_DT,i,marker='v',s=32,color=EVENT_COLORS['감지'],zorder=5)
        if parts_for(w.WO_NO):ax.scatter(w.END_DT,i,marker='D',s=32,color=COL['PART'],zorder=5)
        label=f'{remark_text(w)} · {(w.END_DT-w.STRT_DT).total_seconds()/60:.0f}분'
        after=w.END_DT+pd.Timedelta(minutes=35)
        if (hi-after)/(hi-lo)>.25:where,align=after,'left'
        else:where,align=w.DETC_DT-pd.Timedelta(minutes=35),'right'
        ax.text(where,i,label,fontsize=9.5,color=color,ha=align,va='center')
    ax.set_yticks(range(len(ws)+2),['강종 구간','관측 A / B']+[f'{w.MNT_TYPE} {w.WO_NO[-5:]}' for w in ws.itertuples()],fontsize=9)
    ax.set_ylim(len(ws)+1.6,-.8);clock_ticks(ax,lo,hi)
    ax.xaxis.set_major_locator(mdates.DayLocator());ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    panel_legend(head,ax,[Patch(facecolor=COL['PLAN'],label='계획수리 날짜'),Line2D([0],[0],color=COL['LOT'],lw=5,label='온도 관측창'),Line2D([0],[0],color=COL['TBM'],lw=5,label='TBM 작업')]+([Line2D([0],[0],color=COL['CBM'],lw=5,label='CBM 작업')] if ws.MNT_TYPE.eq('CBM').any() else []))
    for rownum in [1,2]:
        for j,(ax,head,sigs) in enumerate(cells[rownum]):
            key=pre if j==0 else post;a,b=(d['pre_a'],d['pre_b']) if j==0 else (d['post_a'],d['post_b'])
            for sensor in sigs:report_signal(ax,key,sensor,a,b,fine=True)
            for tr in expected.loc[expected.LOT.eq(key)].itertuples():ax.axvspan(tr.시작,tr.끝,color=GREY,alpha=.15,lw=0)
            if j==1:
                ax.axvspan(d['zoom_a'],d['zoom_b'],color=focus_color,alpha=.15,lw=0)
                for w in ws.loc[ws.STRT_DT.lt(po.END)&ws.END_DT.gt(po.START)].itertuples():
                    for when,ls in [(w.STRT_DT,'--'),(w.END_DT,'-.')]:ax.axvline(when,color=COL[w.MNT_TYPE],ls=ls,lw=.9)
            ax.set_ylabel('온도[℃]' if rownum==1 else '출력[%]');clock_ticks(ax,a,b);panel_legend(head,ax)
        low=min(c[0].get_ylim()[0] for c in cells[rownum]);high=max(c[0].get_ylim()[1] for c in cells[rownum])
        for c in cells[rownum]:c[0].set_ylim(low,high)
    for ax,head,sigs in cells[3]:
        for sensor in sigs:report_signal(ax,post,sensor,d['zoom_a'],d['zoom_b'],fine=True)
        ax.axvline(d['point'],color=focus_color,ls=EVENT_LINES[focus_kind],lw=1.5)
        ax.axvspan(d['point']-pd.Timedelta(minutes=1),d['point']+pd.Timedelta(minutes=1),color=focus_color,alpha=.18,lw=0)
        if 'CP-Z4' in sigs:
            base=raw[post]['CP-Z4'].median()
            for y in [base-.05,base+.05]:ax.axhline(y,color=EVENT_COLORS['CP'],ls='--',lw=.9)
        ax.set_ylabel(display_unit(sigs[0]));clock_ticks(ax,d['zoom_a'],d['zoom_b']);panel_legend(head,ax)
    for ax,head,sigs in cells[3]:
        sensor='CP-Z4' if 'CP-Z4' in sigs else 'TC-Z3' if 'TC-Z3' in sigs else 'OP-Z3'
        y=raw_window(post,sensor,d['zoom_a'],d['zoom_b']).dropna()
        point=(y-raw[post][sensor].median()).abs().idxmax() if sensor=='CP-Z4' else y.idxmax() if sensor=='TC-Z3' else y.idxmin()
        unit='℃' if sensor in TC else '%' if sensor in OP else ''
        near_note(ax,point,y.loc[point],f'{point:%H:%M:%S}\n{sensor} {y.loc[point]:.3f}{unit}',focus_color)
    emit(fig,f'03_상세/04_정비대조/정기수리_{month}_일정_전후_확대.png')
    show_table(pd.DataFrame([{'WO':w.WO_NO,'유형':w.MNT_TYPE,'감지':w.DETC_DT,'착수':w.STRT_DT,'완료':w.END_DT,'작업분':(w.END_DT-w.STRT_DT).total_seconds()/60,'내용':remark_text(w),'교체 기록':' / '.join(parts_for(w.WO_NO)) or '-'} for w in ws.itertuples()]))
    return {'계획수리 시작':plan.STRT_DT,'계획수리 끝(날짜)':plan.END_DT,'전 LOT':pre,'후 LOT':post,'수리 중 온도 행':int(temp.MEAS_DT.between(plan.STRT_DT,plan.END_EXCL,inclusive='left').sum())}

plan_compare=[]
plan_compare.append(draw_plan_case(0,240058,240061,'june'))

**이 그림에서 확인한 사실**

1. **수리 기간과 관측을 분리했다.** 계획수리 안에는 온도 0행이다. 위 파란 날짜 범위에서는 작업 WO와 강종 이벤트를 확인하며, 온도는 수리 전(A) 240058(DP590)·수리 후(B) 240061(SPHC)에서만 관측된다. 전체 범위 실제 작업은 6 WO다. 기간 안의 온도·정지 상태를 추정해서 그리지 않았다. ◆는 교체 WO 완료 시각을 대용하며 실제 교체 순간은 아니다.
2. **일정과 맞는 전환은 먼저 설명한다.** 전 LOT 하강 06/09 07:37–09:02, 후 LOT 승온 06/15 01:04–02:40를 회색으로 표시했다. 여러 TC·OP의 동시 변화와 수리 전후라는 위치는 운전 전환 가설을 뒷받침한다. 사후 형태 분류여서 실제 모드·SV·정지 명령 확인은 남는다.
3. **수리 후(B)의 남는 응답을 (B-1)로 확대했다.** (B-1) 06/15 03:05–03:31의 TC·OP·CP는 같은 시간축이다. 확인 시각 전 10~5분과 후 5~10분 TC-Z3 중앙값은 858.63℃ → 858.47℃다. 원본의 복귀·지속과 함께 읽는다. 이 창의 CP 최대 절대 편차는 0.0209다. 국소 응답 근거이며 정상 범위의 검증 자료는 아니다.

**어떻게 이상이 아닐 가능성을 검토하는가?** 일정과 다신호 동시 하강·승온 → 기준 수준으로 접근·복귀 → 같은 TC–OP의 반복 제어 응답 → 별도 CP 이탈 잔존 여부 순서로 본다. 근거가 있는 전환은 **고장 후보와 분리한 설명 가능 구간**으로 둔다. 구간 전체의 정상 보증으로 바꾸지 않는다. 전환 뒤의 단발 응답·CP 사건도 별도 검토한다.

**수리 효과로 결론 내릴 수 없는 이유:** 강종 DP590→SPHC, 관측 시점과 운전 조건이 다르고 수리 중 온도도 없다. 출력·온도 높낮이를 수리의 개선·악화 효과로 해석하지 않는다.

**왜 (B-1)을 골랐는가?** 승온 참고 구간 이후 TC-Z3의 절대 Z가 가장 큰 창을 코드로 선택했다. 모든 온도 또는 전체 수리 효과가 이상이라는 뜻은 아니다.

**수리를 했는데 왜 다시 변화가 있는가? — 현재 자료로 나눠 볼 가설**

- **재가동 전환:** 공통 승온은 재가동 형태로 설명할 수 있어 수준 비교에서 제외했다. (B-1)은 그 이후의 반응을 따로 확인한다.
- **국소 제어 응답:** TC-Z3의 단발 상승·하강과 OP-Z3의 저하·복귀를 같은 시간에 대조한다. 이 동반 반응은 제어 응답 가설과 부합할 수 있지만 설정값·제어 로그가 없어 원인을 확정하지 않는다.
- **정비 대상·원인 불일치:** 캘린더의 정기수리와 WO REMARK는 작업 기록이다. 그 작업이 Z3 제어 또는 CP 변화의 원인을 제거했다는 검증 기록은 없다. 수리 후 관측됐다는 이유만으로 수리 실패·재고장으로 부르지 않는다.
- **추가 검증:** 같은 강종·설정·속도·두께에서 재현되는지, SV 대비 편차와 OP 응답, 정비 대상 부품·계측 보정·완료 점검 결과를 확인해야 원인을 좁힐 수 있다.

**6월 확대의 핵심:** 승온 이후 Z3의 단발 응답이 있어도 Z4–Z6·CP가 같은 폭으로 지속 이탈하는지 따로 본다. 온도·출력이 돌아오는 모습은 일시 제어 반응의 근거이며, 정상 제어임을 확정하려면 SV·조업 조건이 필요하다.

<a id="repair-november"></a>
#### 11월 정기수리 · 일정 → 전후 원본 → 같은 시각 확대

In [ ]:
# draw_plan_case 생성 함수 정의: 5.5의 6월 코드. 같은 레이아웃을 재사용한다.
plan_compare.append(draw_plan_case(1,240114,240117,'november'))
plan_compare=pd.DataFrame(plan_compare)

**이 그림에서 확인한 사실**

1. **수리 기간과 관측을 분리했다.** 계획수리 안에는 온도 0행이다. 위 파란 날짜 범위에서는 작업 WO와 강종 이벤트를 확인하며, 온도는 수리 전(A) 240114(SPCEN)·수리 후(B) 240117(SPCD)에서만 관측된다. 전체 범위 실제 작업은 7 WO다. 기간 안의 온도·정지 상태를 추정해서 그리지 않았다. ◆는 교체 WO 완료 시각을 대용하며 실제 교체 순간은 아니다.
2. **일정과 맞는 전환은 먼저 설명한다.** 전 LOT 하강 11/04 20:22–20:59, 후 LOT 승온 11/13 07:08–08:37를 회색으로 표시했다. 여러 TC·OP의 동시 변화와 수리 전후라는 위치는 운전 전환 가설을 뒷받침한다. 사후 형태 분류여서 실제 모드·SV·정지 명령 확인은 남는다.
3. **수리 후(B)의 남는 응답을 (B-1)로 확대했다.** (B-1) 11/13 12:46–13:16의 TC·OP·CP는 같은 시간축이다. 확인 시각 전 10~5분과 후 5~10분 TC-Z3 중앙값은 869.33℃ → 855.14℃다. 원본의 복귀·지속과 함께 읽는다. 이 창의 CP 최대 절대 편차는 0.0829다. 국소 응답 근거이며 정상 범위의 검증 자료는 아니다.

**어떻게 이상이 아닐 가능성을 검토하는가?** 일정과 다신호 동시 하강·승온 → 기준 수준으로 접근·복귀 → 같은 TC–OP의 반복 제어 응답 → 별도 CP 이탈 잔존 여부 순서로 본다. 근거가 있는 전환은 **고장 후보와 분리한 설명 가능 구간**으로 둔다. 구간 전체의 정상 보증으로 바꾸지 않는다. 전환 뒤의 단발 응답·CP 사건도 별도 검토한다.

**수리 효과로 결론 내릴 수 없는 이유:** 강종 SPCEN→SPCD, 관측 시점과 운전 조건이 다르고 수리 중 온도도 없다. 출력·온도 높낮이를 수리의 개선·악화 효과로 해석하지 않는다.

**왜 (B-1)을 골랐는가?** CBM 완료 뒤 CP 원본 이탈이 남는 창을 선택했다. 온도 최대점만 보고 정한 창이 아니다.

**수리를 했는데 왜 다시 변화가 있는가? — 현재 자료로 나눠 볼 가설**

- **재가동 전환:** 공통 승온은 재가동 형태로 설명할 수 있어 수준 비교에서 제외했다. (B-1)은 그 이후의 반응을 따로 확인한다.
- **국소 제어 응답:** TC-Z3의 단발 상승·하강과 OP-Z3의 저하·복귀를 같은 시간에 대조한다. 이 동반 반응은 제어 응답 가설과 부합할 수 있지만 설정값·제어 로그가 없어 원인을 확정하지 않는다.
- **정비 대상·원인 불일치:** 캘린더의 정기수리와 WO REMARK는 작업 기록이다. 그 작업이 Z3 제어 또는 CP 변화의 원인을 제거했다는 검증 기록은 없다. 수리 후 관측됐다는 이유만으로 수리 실패·재고장으로 부르지 않는다.
- **추가 검증:** 같은 강종·설정·속도·두께에서 재현되는지, SV 대비 편차와 OP 응답, 정비 대상 부품·계측 보정·완료 점검 결과를 확인해야 원인을 좁힐 수 있다.

**11월 확대의 핵심:** CP 하강은 CBM 12:44 완료 후에 발생한다. 승온이라는 설명만으로 이 CP 사건까지 정상 처리하지 않으며, 해당 CBM 전 경보라는 해석도 시간 순서와 맞지 않는다.

<a id="relation-events"></a>
### 5.6 Q1. CP 사건마다 Z3~Z6의 변화 방향이 반복되는가?

**질문:** 240069에서 본 관계가 다른 CP 사건과 존에서도 반복되는가?

**선정 근거:** 14 LOT 전체를 같은 CP 기준으로 검사한 뒤, 운전 전환 후보를 제외한 **6 LOT·14사건을 모두** 비교한다. CP 양·음 방향을 그대로 둔다. CP 피크 전후 15초의 신호 중앙값에서 **해당 사건 시작 전 5~1분 중앙값**을 뺀다. TC는 ℃, OP는 %p로 따로 표시한다. 최고 편차를 골라 색칠하는 방식과 달리 같은 CP 시각에 얼마나 변했는지 보는 그림이다.

**확인하려는 것:** 어느 존이 같은 방향으로 반복되는지, CP 크기·부호가 다를 때 반응도 달라지는지다. 앞 기준 구간은 신호 190초 이상, 피크 주변은 31초 중 24초 이상 실제 기록이 있을 때만 표시한다. 0은 변화량의 기준이며 정상 한계가 아니다.

**구간 연결:** `240069 (a-1)`은 기존 상세 그림 (a)의 첫 CP 펄스, `240069 (c-1)`은 (c)의 CP 사건이다. 다른 LOT도 같은 상세 구간 문자와 CP 순번을 사용한다.


In [ ]:
# CP 사건 목록 → 사건별 기준·같은 CP 시각 변화량 → 직접 그리는 히트맵
REL_SENSORS=[f'{kind}-Z{z}' for kind in ['TC','OP'] for z in range(3,7)]
relation_raw={k:g.set_index('MEAS_DT').asfreq('s') for k,g in raw.items()}
relation_events=cp_real.sort_values(['LOT','PEAK']).reset_index(drop=True).copy()
relation_events['EVENT_NO']=relation_events.groupby('LOT').cumcount()+1
# 원본 상세 그림의 a/b/c 구간 ID에 CP 시간순 번호를 붙여 같은 사건을 추적한다.
relation_event_source=cp_time_features[['LOT','PEAK','EVENT_ID']].copy()
relation_events=relation_events.merge(relation_event_source,on=['LOT','PEAK'],how='left',validate='one_to_one')
relation_events['ID']=relation_events.EVENT_ID+(relation_events.groupby('EVENT_ID').cumcount()+1).astype(str)
assert relation_events.ID.is_unique and relation_events.EVENT_ID.notna().all()
relation_events['ONSET']=[at(int(r.LOT),r.A).round('s') for r in relation_events.itertuples()]
relation_rows=[]
for ev in relation_events.itertuples():
    frame=relation_raw[int(ev.LOT)]
    for sensor in REL_SENSORS+['CP-Z4M']:
        pre=frame[sensor].loc[(frame.index>=ev.ONSET-pd.Timedelta(minutes=5))&(frame.index<ev.ONSET-pd.Timedelta(minutes=1))]
        center=frame[sensor].loc[ev.PEAK-pd.Timedelta(seconds=15):ev.PEAK+pd.Timedelta(seconds=15)]
        valid=pre.count()>=190 and center.count()>=24
        relation_rows.append(dict(ID=ev.ID,LOT=int(ev.LOT),SENSOR=sensor,CP_DEV=ev.DEV,PEAK=ev.PEAK,
            BASE=pre.median(),DELTA=center.median()-pre.median() if valid else np.nan,
            N_BASE=int(pre.count()),N_PEAK=int(center.count()),VALID=valid))
relation_event_delta=pd.DataFrame(relation_rows)
relation_repetition=[]
for sensor,g in relation_event_delta.loc[relation_event_delta.SENSOR.isin(REL_SENSORS)].groupby('SENSOR',sort=False):
    v=g.DELTA.dropna()
    relation_repetition.append({'신호':sensor,'유효 사건':len(v),'증가':int(v.gt(0).sum()),'감소':int(v.lt(0).sum()),
        '0':int(v.eq(0).sum()),'사건 변화량 중앙값':v.median(),
        'LOT 동일 가중 변화량 중앙값':g.groupby('LOT').DELTA.median().median()})
relation_repetition=pd.DataFrame(relation_repetition)
from matplotlib.colors import TwoSlopeNorm
fig,cells=report_page('CP 14사건 | 같은 CP 피크 시각의 Z3~Z6 변화량',
    '행=CP 사건 · 열=존. 앞 기준 대비 증가(빨강)·감소(파랑)와 실제 변화량을 함께 본다. CP 크기만으로 TC·OP 반응을 대신 판단하지 않는다.',
    [dict(panels=[('TC-Z3~Z6 | CP 피크 부근 온도 변화',[f'TC-Z{i}' for i in range(3,7)]),
                  ('OP-Z3~Z6 | CP 피크 부근 출력 변화',[f'OP-Z{i}' for i in range(3,7)])],height=5.6)],width=24,
    info=['기준=사건 시작 전 5~1분 중앙값 · 표시=CP 피크 ±15초 중앙값 - 기준 · 같은 LOT의 가까운 사건은 독립 표본이 아님.'])
for (ax,head,_),kind in zip(cells[0],['TC','OP']):
    sensors=[f'{kind}-Z{i}' for i in range(3,7)]
    matrix=relation_event_delta.pivot(index='ID',columns='SENSOR',values='DELTA').reindex(index=relation_events.ID,columns=sensors)
    limit=max(float(matrix.abs().max().max()),.001)
    cmap=plt.get_cmap('RdBu_r').copy();cmap.set_bad('#e5e9ed')
    ax.imshow(np.ma.masked_invalid(matrix.to_numpy()),aspect='auto',cmap=cmap,norm=TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit))
    ax.set_xticks(range(4),sensors);ax.set_yticks(range(len(matrix)),
        [f'{event_text(ev.ID)}\nCP {ev.DEV:+.3f}' for ev in relation_events.itertuples()],fontsize=9)
    ax.set_xlabel('신호 · 색 범위 ±'+f'{limit:.2f}'+('℃' if kind=='TC' else '%p'))
    ax.grid(False)
    for j,row in enumerate(matrix.to_numpy()):
        for i,value in enumerate(row):
            ax.text(i,j,(f'{value:+.3f}' if kind=='TC' else f'{value:+.2f}') if np.isfinite(value) else '보류',ha='center',va='center',fontsize=9.5,
                color='white' if np.isfinite(value) and abs(value)>limit*.6 else '#17344d')
    panel_legend(head,ax,extra=[Patch(facecolor='#b2182b',label='증가'),Patch(facecolor='#2166ac',label='감소'),Patch(facecolor='#e5e9ed',label='기록 부족')])
emit(fig,'05_관계검토/01_CP사건별_Z3-Z6_변화량.png')
show_table(relation_repetition)
table(relation_event_delta)

parts=[]
for sensor in REL_SENSORS:
 r=relation_repetition.loc[relation_repetition['신호'].eq(sensor)].iloc[0]
 parts.append(f"{sensor}: 증가 {r['증가']}/감소 {r['감소']}(유효 {r['유효 사건']})")
note('**실제 계산 결과:** '+ ' · '.join(parts)+'. 이는 CP 피크 시각의 앞 기준 대비 변화 방향이다. 신호별 방향이 섞이는지 표·원본과 함께 읽으며, 증가·감소 횟수를 고장 확률로 바꾸지 않는다.')

small=relation_event_delta.loc[relation_event_delta.SENSOR.isin(['TC-Z4','TC-Z5','TC-Z6']),'DELTA'].abs().max()
z3=relation_event_delta.loc[relation_event_delta.SENSOR.eq('TC-Z3'),'DELTA'].abs().max()
note(f'**존 간 크기 차이:** 같은 CP 피크 시각의 앞 기준 대비 최대 절대 온도 변화는 TC-Z3 {z3:.2f}℃, TC-Z4~Z6 {small:.2f}℃다. Z4~Z6를 생략할 근거가 아니라 작은 동반 반응도 별도 축·시차·대조로 확인할 이유다.')


**관찰 결과·반례·해석**

- 색의 방향이 사건마다 같은지 먼저 보고, 그다음 숫자로 진폭을 확인한다. 동일한 CP 상승이라도 온도·출력이 매번 같은 방향·크기로 움직여야 단순한 공통 반응 가설을 지지할 수 있다.
- 이 그림의 값은 CP 피크 부근의 변화다. 앞뒤에 더 큰 변화가 있었는지와 어느 신호가 먼저 움직였는지는 5.7의 같은 시간축 원본·앞뒤 파형으로 따로 확인한다. CP 상승·하강을 섞지 않고 방향별로 본다. 작은 수치나 0에 가까운 변화만으로 긴 지연 관계까지 배제하지 않는다.
- 같은 LOT에서 가까이 발생한 펄스는 앞 기준 창이 겹친다. 14사건을 독립 실험 14회로 취급하지 않고 사건 수와 LOT 수를 나눠 표시한다. 운전 설정과 CP 정의가 없으므로 반복 방향만으로 고장 원인을 선정하지 않는다.


<a id="relation-lags"></a>
### 5.7 Q2. CP가 떨어질 때, 주변 존의 온도·출력은 어떻게 움직이는가?

**질문:** CP 하강을 기준으로 앞뒤에 TC-Z3~Z6·OP-Z3~Z6가 어떤 모양을 보이는가? CP 최저점 전에 이미 변화가 있었는가?

**선정 근거:** 전환 후보를 제외한 14 CP 사건 중 하강은 **240024 (a-1)·240117 (b-1), 2사건**이다. 여기서 하강은 LOT 중앙값보다 0.05 이상 낮아진 이탈을 뜻한다. 양의 CP 피크가 내려와 복귀하는 과정은 상승 사건의 원본에서 따로 확인한다. 상승 12사건과 섞어 평균내지 않고 두 하강 사건을 먼저 나란히 본다. 상승 사건도 아래에서 모두 같은 형식으로 펼쳐볼 수 있다.

**그림 읽기:** 각 열은 한 CP 사건이다. 위에서 아래로 **CP → TC-Z3 실제 온도 → OP-Z3 실제 출력 → TC-Z4~Z6 변화 → OP-Z4~Z6 변화**를 읽는다. 모든 행은 같은 실제 시각으로 맞추며, 가로축 **0은 그 사건의 CP 최저점**, 앞뒤 15분이다. 붉은 음영은 해당 CP 이탈 기록 범위이고 세로 점선은 CP 최저점이다. CP 가로 점선은 LOT 중앙값 ±0.05의 탐색 기준이며 물리적 정상 한계가 아니다. 선을 결측 구간에 이어 붙이지 않는다.

Z4~Z6는 작은 움직임도 보이도록 **각 신호의 CP 이탈 시작 전 5~1분 중앙값을 뺀 변화량**을 그린다. 0은 비교 기준이다. CP 최저점 ±15초의 중앙값 변화는 패널 위 숫자, 그 시각의 1초 실제 값은 점으로 구분한다. Z3는 원본 값으로 그려 온도 상승·하강, 출력 저하·복귀가 직접 보이게 했다. 기준창 자체가 정상 운전이라는 뜻은 아니다.

**확인할 것:** CP 최저점의 위치에서 각 선을 위아래로 읽고, 앞선 TC 최고·OP 최저점과 비교한다. 서로 다른 신호의 극값 시각은 반응 시작 시각·소재 전달 시간·물리적 원인으로 해석하지 않는다. 기존 시차 상관 표는 접힌 계산 참고에 두었다.


In [ ]:
# CP 하강과 상승을 분리하고, 같은 사건의 CP·온도·출력을 실제 파형으로 비교한다.
# 이 셀의 matplotlib 코드가 바로 아래 본문 그림과 접힌 사건별 그림을 만든다.
REL_WAVE_MIN=15
relation_wave_checks=[]
relation_wave_rows=[]
for ev in relation_events.itertuples():
    frame=relation_raw[int(ev.LOT)]
    for sensor in REL_SENSORS:
        source=relation_event_delta.loc[relation_event_delta.ID.eq(ev.ID)&relation_event_delta.SENSOR.eq(sensor)].iloc[0]
        q=frame[sensor].loc[ev.PEAK-pd.Timedelta(minutes=REL_WAVE_MIN):ev.PEAK+pd.Timedelta(minutes=REL_WAVE_MIN)].dropna()
        extreme=q.idxmax() if sensor.startswith('TC') else q.idxmin()
        before=frame[sensor].loc[ev.PEAK-pd.Timedelta(minutes=10):ev.PEAK-pd.Timedelta(minutes=5)]
        after=frame[sensor].loc[ev.PEAK+pd.Timedelta(minutes=5):ev.PEAK+pd.Timedelta(minutes=10)]
        relation_wave_rows.append(dict(ID=ev.ID,LOT=int(ev.LOT),DIRECTION='하강' if ev.DEV<0 else '상승',SENSOR=sensor,
            BASE=source.BASE,AT_CP=source.BASE+source.DELTA if source.VALID else np.nan,DELTA=source.DELTA,
            N_BASE=int(source.N_BASE),N_PEAK=int(source.N_PEAK),
            BEFORE=before.median(),AFTER=after.median(),N_BEFORE=int(before.count()),N_AFTER=int(after.count()),
            EXTREME_AT=extreme,EXTREME=float(q.loc[extreme]),OFFSET_MIN=(extreme-ev.PEAK).total_seconds()/60))
relation_wave_metrics=pd.DataFrame(relation_wave_rows)
relation_down=relation_events.loc[relation_events.DEV.lt(0)].copy()
relation_up=relation_events.loc[relation_events.DEV.gt(0)].copy()
assert len(relation_down)==2 and len(relation_up)==12


def cp_wave_caption(ev,kind):
    d=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)].set_index('SENSOR')
    if kind=='CP':
        value=relation_raw[int(ev.LOT)].loc[ev.PEAK,'CP-Z4']
        return f'{ev.PEAK:%m/%d %H:%M:%S} · 원본 {value:.4f} · LOT 중앙값 대비 {ev.DEV:+.4f}'
    if kind in ['TC3','OP3']:
        sensor='TC-Z3' if kind=='TC3' else 'OP-Z3';r=d.loc[sensor]
        when=f'CP보다 {abs(r.OFFSET_MIN):.2f}분 '+('앞' if r.OFFSET_MIN<0 else '뒤')
        label='온도 최고점' if kind=='TC3' else '출력 최저점'
        return f'{r.EXTREME_AT:%H:%M:%S} {label} {r.EXTREME:.2f}'+('℃' if kind=='TC3' else '%')+f' · {when} (반응 시작 시각 아님)'
    prefix='TC' if kind=='TC456' else 'OP';unit='℃' if prefix=='TC' else '%p'
    words=[f'{prefix}-Z{z} {d.loc[f"{prefix}-Z{z}","DELTA"]:+.2f}' for z in [4,5,6]]
    return ('CP 최저 부근의 앞 기준 대비 변화: ' if ev.DEV<0 else 'CP 최고 부근의 앞 기준 대비 변화: ')+' / '.join(words)+' '+unit


def draw_cp_zone_patterns(group,heading):
    # 2열=서로 다른 CP 사건, 5행=CP / TC-Z3 / OP-Z3 / TC-Z4~6 / OP-Z4~6.
    # 모든 존의 같은 실제 시각을 맞춘다. 숫자 시차 r 대신 원본의 높낮이·모양을 읽는다.
    import textwrap
    from matplotlib.ticker import MaxNLocator
    events=list(group.itertuples());count=len(events);width=24 if count==2 else 14
    specs=[('CP',['CP-Z4'],1.45),('TC3',['TC-Z3'],1.3),('OP3',['OP-Z3'],1.2),
           ('TC456',[f'TC-Z{i}' for i in [4,5,6]],1.3),('OP456',[f'OP-Z{i}' for i in [4,5,6]],1.3)]
    head_h,tick_h,gap,top_h=.88,.43,.16,1.60
    height=top_h+sum(head_h+h+tick_h+gap for _,_,h in specs)+.15
    fig=plt.figure(figsize=(width,height),dpi=150,facecolor='white')
    fig.suptitle(heading,x=.07,y=1-.14/height,ha='left',va='top',fontsize=19,fontweight='bold',color='#17344d')
    fig.text(.07,1-.55/height,'각 열=한 CP 사건 · 위에서 아래로 같은 시각의 CP → TC-Z3 → OP-Z3 → TC-Z4~Z6 → OP-Z4~Z6.',
             ha='left',va='top',fontsize=11,color='#17344d')
    fig.text(.07,1-.89/height,'0=선택 CP의 최저점/최고점 · 전후 15분 · 붉은 음영=해당 CP 이탈 기록 · 회색 음영=다른 CP 사건/LOT 관측 밖.',
             ha='left',va='top',fontsize=10,color='#526d80')
    fig.text(.07,1-1.18/height,'Z3는 실제 온도·출력, Z4~Z6는 CP 이탈 시작 전 5~1분 중앙값을 뺀 변화량이다. 비교 기준은 정상 한계가 아니다.',
             ha='left',va='top',fontsize=10,color='#526d80')
    cells=[];cursor=top_h;left,right,hgap=.075,.98,.075;panel_w=(right-left-(count-1)*hgap)/count
    for kind,sensors,data_h in specs:
        row=[]
        for j,ev in enumerate(events):
            tag=event_text(ev.ID,include_lot=False);x=left+j*(panel_w+hgap)
            head=fig.add_axes([x,1-(cursor+head_h)/height,panel_w,head_h/height]);head.axis('off');head._has_note_gutter=False
            titles={'CP':event_text(ev.ID)+' | CP-Z4 '+('하강' if ev.DEV<0 else '상승'),
                    'TC3':tag+' TC-Z3 | 가열 초입의 실제 온도',
                    'OP3':tag+' OP-Z3 | 같은 시각의 실제 출력',
                    'TC456':tag+' TC-Z4·TC-Z5·TC-Z6 | 기준에서 움직인 온도',
                    'OP456':tag+' OP-Z4·OP-Z5·OP-Z6 | 기준에서 움직인 출력'}
            head.text(0,.98,titles[kind],ha='left',va='top',fontsize=12,fontweight='bold',color='#17344d')
            head.text(0,.66,textwrap.fill(cp_wave_caption(ev,kind),width=100 if count==1 else 87),
                      ha='left',va='top',fontsize=9.5,color='#526d80',linespacing=1.2)
            ax=fig.add_axes([x,1-(cursor+head_h+data_h)/height,panel_w,data_h/height]);ax._note_gutter=0
            style(ax,'');ax.ticklabel_format(axis='y',style='plain',useOffset=False);ax.tick_params(labelsize=9,pad=3)
            ax.yaxis.set_major_locator(MaxNLocator(4));row.append((ax,head,sensors))
        cells.append(row);cursor+=head_h+data_h+tick_h+gap
    fig._report_cells=cells
    for i,((kind,sensors,_),row) in enumerate(zip(specs,cells)):
        pooled=[]
        for (ax,head,_),ev in zip(row,events):
            k=int(ev.LOT);t=ev.PEAK;a,b=t-pd.Timedelta(minutes=15),t+pd.Timedelta(minutes=15)
            frame=relation_raw[k];metrics=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)].set_index('SENSOR')
            for sensor in sensors:
                y=frame[sensor].loc[a:b]
                if kind in ['TC456','OP456']:
                    y=y-metrics.loc[sensor,'BASE'] if metrics.loc[sensor,'N_BASE']>=190 else y*np.nan
                ax.plot(y.index,y,color=sensor_color(sensor),lw=.95 if sensor.startswith('OP') else 1.15,label=sensor+' · '+('변화량' if kind in ['TC456','OP456'] else '1초 원본'))
                pooled.append(y.dropna())
            # 다른 CP 사건이 같은 창에 있음을 표시한다. 서로 독립된 반응이라고 가정하지 않는다.
            for other in relation_events.loc[relation_events.LOT.eq(k)&relation_events.ID.ne(ev.ID)].itertuples():
                lo,hi=at(k,other.A).round('s'),at(k,other.B).round('s')
                if lo<=b and hi>=a:ax.axvspan(max(lo,a),min(hi,b),color='#8496a5',alpha=.10,lw=0)
            start,end=at(k,ev.A).round('s'),at(k,ev.B).round('s')
            ax.axvspan(start,end,color=EVENT_COLORS['CP'],alpha=.13,lw=0)
            ax.axvline(t,color=EVENT_COLORS['CP'],ls=':',lw=1.2)
            if kind=='CP':
                base=frame['CP-Z4'].median()
                for bound,y in enumerate([base-.05,base+.05]):ax.axhline(y,color=EVENT_COLORS['CP'],ls='--',lw=.8,label='LOT 중앙값 ±0.05 · 탐색선' if bound==0 else '_nolegend_')
                selected=frame['CP-Z4'].loc[start:end]
                ax.plot(selected.index,selected,color=EVENT_COLORS['CP'],lw=1.8,zorder=5)
            if kind in ['TC456','OP456']:ax.axhline(0,color='#8496a5',ls='--',lw=.7)
            # LOT 관측 밖으로 선을 연장하지 않고, 실제 관측 범위 밖만 옅은 회색으로 표시한다.
            lo,hi=lot_meta.loc[k,'START'],lot_meta.loc[k,'END']
            if a<lo:ax.axvspan(a,min(lo,b),color='#8496a5',alpha=.15,lw=0)
            if b>hi:ax.axvspan(max(hi,a),b,color='#8496a5',alpha=.15,lw=0)
            ticks=[t+pd.Timedelta(minutes=m) for m in [-15,-10,-5,0,5,10,15]]
            ax.set_xticks(ticks,['-15','-10','-5','0','+5','+10','+15']);ax.set_xlim(a,b)
            if i==4:ax.set_xlabel('CP '+('최저점' if ev.DEV<0 else '최고점')+f' 대비 [분] · 0 = {t:%H:%M:%S}',fontsize=10)
            else:ax.set_xlabel('CP '+('최저점' if ev.DEV<0 else '최고점')+' 대비 [분]',fontsize=9)
            ax.set_ylabel({'CP':'CP-Z4 [무차원]','TC3':'온도 [℃]','OP3':'출력 [%]',
                           'TC456':'앞 기준 대비 온도 변화 [℃]','OP456':'앞 기준 대비 출력 변화 [%p]'}[kind],fontsize=10)
            panel_legend(head,ax)
        values=pd.concat(pooled).dropna();lo,hi=values.min(),values.max();pad=max((hi-lo)*.32,.01 if kind=='CP' else .2)
        limits=(-35,115) if kind=='OP3' else (lo-pad,hi+pad)
        for (ax,head,_),ev in zip(row,events):
            ax.set_ylim(*limits)
            if kind=='OP3':ax.set_yticks([0,25,50,75,100])
            if kind in ['CP','TC3','OP3']:
                if kind=='CP':
                    point=ev.PEAK;value=relation_raw[int(ev.LOT)].loc[point,'CP-Z4'];label=('CP 최저 ' if ev.DEV<0 else 'CP 최고 ')+f'{value:.4f}'
                else:
                    sensor=sensors[0];r=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)&relation_wave_metrics.SENSOR.eq(sensor)].iloc[0]
                    point,value=r.EXTREME_AT,r.EXTREME
                    label=('온도 최고 ' if kind=='TC3' else '출력 최저 ')+f'{value:.2f}'+('℃' if kind=='TC3' else '%')
                ax.plot(point,value,'o',ms=4,mfc='white',mec=EVENT_COLORS['CP'],zorder=6)
                if ev.DEV<0:near_note(ax,point,value,label,EVENT_COLORS['CP'])
            if kind in ['TC456','OP456']:
                for sensor in sensors:
                    v=relation_raw[int(ev.LOT)].loc[ev.PEAK,sensor]-relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)&relation_wave_metrics.SENSOR.eq(sensor),'BASE'].iloc[0]
                    ax.plot(ev.PEAK,v,'o',ms=3,color=sensor_color(sensor),zorder=6)
    for col,ev in enumerate(events):
        axes=[row[col][0] for row in cells]
        assert all(np.array_equal(ax.get_xticks(),axes[0].get_xticks()) for ax in axes)
        assert all(np.allclose(ax.get_xlim(),axes[0].get_xlim(),rtol=0,atol=1e-10) for ax in axes)
        shown={sensor for _,_,sensors in [row[col] for row in cells] for sensor in sensors}
        assert shown==set(REL_SENSORS+['CP-Z4'])
        relation_wave_checks.append(dict(ID=ev.ID,LOT=int(ev.LOT),DIRECTION='하강' if ev.DEV<0 else '상승',N_PANELS=5,SENSORS=shown))
    return fig


fig=draw_cp_zone_patterns(relation_down,'CP 하강 2사건 | CP가 낮아진 전후, 주변 존에서는 무엇이 움직였는가?')
emit(fig,'05_관계검토/02_CP하강_주변존_동시변화.png')
# 해석은 그림의 실제 시각·동일 기준 수치로 작성한다.
for ev in relation_down.itertuples():
    d=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)].set_index('SENSOR');tc=d.loc['TC-Z3'];op=d.loc['OP-Z3']
    values=' / '.join(f'TC-Z{z} {d.loc[f"TC-Z{z}","DELTA"]:+.2f}℃, OP-Z{z} {d.loc[f"OP-Z{z}","DELTA"]:+.2f}%p' for z in [4,5,6])
    note(f'**{event_text(ev.ID)}에서 읽는 순서:** CP 최저 {ev.PEAK:%H:%M:%S}보다 TC-Z3 원본 최고점은 {abs(tc.OFFSET_MIN):.2f}분 '+('앞' if tc.OFFSET_MIN<0 else '뒤')+
         f', OP-Z3 원본 최저점은 {abs(op.OFFSET_MIN):.2f}분 '+('앞' if op.OFFSET_MIN<0 else '뒤')+
         f'이다. CP 최저 부근의 TC-Z3는 앞 기준 대비 {tc.DELTA:+.2f}℃, OP-Z3는 {op.DELTA:+.2f}%p다. 다른 존의 같은 시각 변화는 **{values}**다. 극값의 순서이며 반응 시작·원인을 뜻하지 않는다.')
show_table(event_frame(relation_wave_metrics.loc[relation_wave_metrics.DIRECTION.eq('하강'),['ID','SENSOR','BASE','AT_CP','DELTA']]).rename(columns={
    'ID':'CP 사건','SENSOR':'신호','BASE':'CP 이탈 앞 기준','AT_CP':'CP 최저 ±15초 중앙값','DELTA':'기준 대비 변화'}))

# 상승 12사건도 같은 5행·같은 30분 폭으로 전부 남긴다. 파일을 따로 늘리지 않는다.
from IPython.utils.capture import capture_output
for first in range(0,len(relation_up),2):
    group=relation_up.iloc[first:first+2]
    with capture_output() as captured:
        fig=draw_cp_zone_patterns(group,'CP 상승 사건 | 같은 CP 시각의 주변 존 원본·변화량')
        checked_emit(fig,'CP상승_사건별_파형',save=False,record=False)
    captured_details('CP 상승 사건 원본 대조: '+' / '.join(event_text(v) for v in group.ID),captured)
table(relation_wave_metrics.rename(columns={'ID':'CP 사건','SENSOR':'신호','BEFORE':'CP 전 10~5분 중앙값','AFTER':'CP 후 5~10분 중앙값',
    'EXTREME_AT':'창 안의 TC 최고/OP 최저 시각','OFFSET_MIN':'그 극값의 CP 대비 위치[분]','N_BEFORE':'앞 창 기록수','N_AFTER':'뒤 창 기록수'}))
assert len(relation_wave_checks)==14 and {r['ID'] for r in relation_wave_checks}==set(relation_events.ID)
assert len(relation_wave_metrics)==14*8
legacy=OUT_DIR/'05_관계검토/02_사건별_시차와_OP_TC.png'
if legacy.is_file():
    assert legacy.resolve().parent==(OUT_DIR/'05_관계검토').resolve()
    assert (OUT_DIR/'05_관계검토/02_CP하강_주변존_동시변화.png').is_file()
    legacy.unlink()


# 5초 유효 중앙값의 차분을 만들고 시차를 바꾸어 직접 사건별 r을 계산한다.
from scipy.stats import spearmanr
REL_LAGS=np.arange(-120,121,10)
relation_five={};relation_diff={};relation_phase={}
for k,frame in relation_raw.items():
    p=frame[REL_SENSORS+['CP-Z4','CP-Z4M']].resample('5s',origin=lot_meta.loc[k,'START']).agg(['median','count'])
    v=p.xs('median',axis=1,level=1).where(p.xs('count',axis=1,level=1).ge(4))
    minute=np.floor((v.index-lot_meta.loc[k,'START']).total_seconds()/60).astype(int)
    stable=pd.Series(comparison_states[k].reindex(minute).eq('안정 참고').to_numpy(),index=v.index)
    relation_five[k]=v;relation_diff[k]=v.diff();relation_phase[k]=stable&stable.shift(1,fill_value=False)

def observed_lag(x,y,min_pairs):
    rows=[]
    for lag in REL_LAGS:
        # 양의 lag: x(t)와 나중의 y(t+lag)를 같은 행에 맞춘다.
        q=pd.concat([x.rename('x'),y.shift(-int(lag//5)).rename('y')],axis=1).dropna()
        valid=len(q)>=min_pairs and q.x.nunique()>1 and q.y.nunique()>1
        r=float(spearmanr(q.x,q.y).statistic) if valid else np.nan
        rows.append(dict(LAG_SEC=int(lag),R=r,N_PAIRS=len(q)))
    return pd.DataFrame(rows)

relation_lag_rows=[]
for ev in relation_events.itertuples():
    k=int(ev.LOT);d=relation_diff[k].where(relation_phase[k],axis=0)
    window=d.loc[ev.PEAK-pd.Timedelta(minutes=5):ev.PEAK+pd.Timedelta(minutes=5)]
    for sensor in REL_SENSORS+['CP-Z4M']:
        curve=observed_lag(window['CP-Z4'],window[sensor],40)
        for r in curve.to_dict('records'):relation_lag_rows.append(dict(PAIR='CP',ID=ev.ID,LOT=k,SENSOR=sensor,**r))
for k,d in relation_diff.items():
    d=d.where(relation_phase[k],axis=0)
    for zone in range(3,7):
        curve=observed_lag(d[f'OP-Z{zone}'],d[f'TC-Z{zone}'],200)
        for r in curve.to_dict('records'):relation_lag_rows.append(dict(PAIR='OP_TC',ID=str(k),LOT=k,SENSOR=f'TC-Z{zone}',**r))
relation_lags=pd.DataFrame(relation_lag_rows)

def lag_ensemble(frame):
    per_lot=frame.groupby(['LOT','LAG_SEC']).R.median().unstack('LOT').reindex(REL_LAGS)
    line=per_lot.median(axis=1)
    if per_lot.shape[1]>1:
        omitted=pd.concat([per_lot.drop(columns=[k]).median(axis=1) for k in per_lot],axis=1)
        low,high=omitted.min(axis=1),omitted.max(axis=1)
    else:low=high=line*np.nan
    return line,low,high,per_lot.notna().sum(axis=1)

relation_lag_summary=[]
for (pair,sensor),g in relation_lags.groupby(['PAIR','SENSOR'],sort=False):
    med,low,high,count=lag_ensemble(g);zero=g.loc[g.LAG_SEC.eq(0)].dropna(subset=['R'])
    peak=med.abs().idxmax() if med.notna().any() else np.nan
    relation_lag_summary.append({'비교':pair,'신호':sensor,'0초 유효 사건/LOT':len(zero),'0초 r 중앙값(LOT 동일 가중)':med.loc[0],
        '0초 양의 r':int(zero.R.gt(0).sum()),'0초 음의 r':int(zero.R.lt(0).sum()),
        '중앙 곡선 최대절대 r 시차[초]':peak,'그 시차 r':med.loc[peak] if pd.notna(peak) else np.nan,
        '그 시차 유효 LOT':int(count.loc[peak]) if pd.notna(peak) else 0})
relation_lag_summary=pd.DataFrame(relation_lag_summary)


# 보조 통계: CP-Z4M 대조 절에서도 쓰는 차분·시차 계산을 보존한다.
# 아래 계산은 위 파형의 변화 순서를 확정하거나 물리적 인과로 판정하지 않는다.
with capture_output() as lag_reference:
    note('**계산 참고:** CP와 각 TC·OP의 짧은 변동을 비교한 통계다. 동일 존 OP–TC는 예를 들어 OP-Z4와 TC-Z4의 비교이며 CP 하강 사건 자체의 파형이 아니다. 본문은 위 실제 사건 파형을 먼저 읽는다.')
    shown=relation_lag_summary.copy();shown['비교']=shown['비교'].replace({'CP':'CP와 TC·OP','OP_TC':'동일 존 OP와 TC'})
    show_table(shown)
    note('실제 시각의 5초 중앙값(5개 중 4개 이상)의 차분, 보간 없음. CP 피크 전후 5분은 40쌍 이상, LOT 전환 제외 구간은 200쌍 이상일 때 Spearman r을 계산한다. -120~+120초를 10초 간격으로 검사한다. 양의 시차는 CP 또는 OP 기록이 먼저인 방향이다. 사후 선정·자기상관·시차 탐색이 있어 원인 검정으로 쓰지 않는다.')
    table(relation_lags)
captured_details('보조 통계 표·계산 정의 펼치기 — 원본 파형과 별도로 확인',lag_reference)


**이 그림으로 확인한 결과**

1. **두 CP 하강 모두 Z3의 변화가 먼저 보인다.** TC-Z3는 CP 최저점보다 약 6분 앞에서 높은 온도를 보인 뒤 내려온다. OP-Z3도 먼저 낮은 출력을 보인 뒤 CP 최저점 부근에서는 올라온다. 따라서 “CP가 떨어진 뒤 처음 Z3가 움직였다”는 순서와는 맞지 않는다. 극값 위치를 비교한 관찰이며 실제 제어 동작 시작을 추정한 것은 아니다.
2. **Z4·Z5·Z6가 같은 방향으로 한꺼번에 변하지 않는다.** 두 사건의 CP 최저점 부근에서 TC-Z4·TC-Z6는 앞 기준보다 낮고 TC-Z5는 높다. 출력은 OP-Z4·OP-Z6가 높고 OP-Z5가 낮은 방향이다. Z3와 같은 축에 묻히지 않도록 다른 존도 별도 크기로 확인했다. 정확한 크기는 각 패널 위와 표에 표시한다.
3. **반복되는 양상은 원인 후보를 좁히는 단서다.** 두 LOT의 강종과 관측 날짜가 다르고 하강 사례도 2개뿐이다. 반복 모양만으로 Z3 또는 특정 존이 CP를 떨어뜨렸다고 확정할 수 없다. CP 없는 Z3 반응 및 CP 상승 사건과도 대조해야 한다.
4. **정비 후 CP 하강과 제어 반응을 구분한다.** 240117의 CP 최저는 CBM 완료 뒤에 있지만, 이 사실만으로 정비가 CP 하강을 만들었거나 수리 실패라고 판단하지 않는다. 동일 시각의 CP 계측·SV·제어 로그, 라인 조건과 실제 작업 대상을 확인할 근거다.

CP 두 채널의 정의가 미확인이라 “왜 CP가 떨어졌는가”의 물리적 원인은 아직 정할 수 없다. 이 그림은 어떤 신호·시간대를 함께 확인할지 구체적으로 정하는 관측 근거다.


<a id="relation-tc-op"></a>
### 5.7.1 TC-OP 관계 — 같은 존의 온도와 출력은 어떻게 함께 움직였는가?

**분석 질문:** `TC-Z3 ↔ OP-Z3`, `TC-Z4 ↔ OP-Z4`, `TC-Z5 ↔ OP-Z5`, `TC-Z6 ↔ OP-Z6`를 각각 비교한다. 온도가 오를 때 출력도 오르는가, 반대로 움직이는가? CP와의 관계와 구분해 읽는다.

**본문 파형:** 5.1.2에서 확인한 **240069의 두 차례 Z3 반응(07/07 22:45~23:50)**을 같은 시간 범위로 다시 사용한다. 각 열이 한 존이고, 위는 TC(℃), 아래는 대응 OP(%)다. 다른 존도 같은 시각으로 맞춰 함께 본다. 특정 존·시차의 상관이 가장 큰 창을 새로 골라낸 것은 아니다.

회색은 1초 원본, 색 선은 **5개 중 4개 이상 실제 값이 있는 5초 중앙값**이다. 원 표시는 원본 값이다. 세로 점선 ①·②는 Z3 온도 최고점의 실제 시각이며, 다른 존에서는 같은 시각의 참고선이다. 존마다 세로축 범위가 다르므로 값·단위를 함께 읽는다.

**전체 LOT의 반복성:** 예시 파형 한 장으로 일반화하지 않는다. 아래 표는 **14 LOT 각각의 전환 제외 구간**에서 TC·OP의 5초 중앙값이 직전 5초보다 어느 방향으로 바뀌었는지 비교한 Spearman r이다. 0초는 같은 시각끼리 비교했다는 뜻이다. 온도 수준과 출력 수준의 상관계수와 구분한다. 결측은 채우지 않고 200쌍 이상 있을 때만 계산한다.


In [ ]:
# 같은 존의 TC–OP 파형을 직접 그리고, 기존 전체 LOT 차분 통계를 별도 표로 연결한다.
TCOP_LOT=240069
case=relation_raw[TCOP_LOT]
z_event=next(e for e in LOT_EVENTS[TCOP_LOT] if e['kind']=='Z' and e['sensor']=='TC-Z3')
first_end=z_event['point']
first_peak=case['TC-Z3'].loc[first_end-pd.Timedelta(minutes=1):first_end].idxmax()
second_peak=case['TC-Z3'].loc[first_end:first_end+pd.Timedelta(hours=1)].idxmax()
tcop_a=first_end.floor('min')-pd.Timedelta(minutes=14)
tcop_b=second_peak.ceil('min')+pd.Timedelta(minutes=17)
tcop_points=[first_peak,second_peak]
tcop_zero=relation_lags.loc[relation_lags.PAIR.eq('OP_TC')&relation_lags.LAG_SEC.eq(0)].copy()
tcop_zero['ZONE']=tcop_zero.SENSOR.str.extract(r'Z(\d+)',expand=False).astype(int)
tcop_summary=[]
for zone in range(3,7):
    values=tcop_zero.loc[tcop_zero.ZONE.eq(zone)].dropna(subset=['R'])
    tcop_summary.append({'같은 존의 신호':f'TC-Z{zone} / OP-Z{zone}','유효 LOT':values.LOT.nunique(),
        '동시 변화가 역방향인 LOT':int(values.R.lt(0).sum()),'0초 변화 r 중앙값':values.R.median(),
        '그 LOT들의 최소 유효 비교 수':int(values.N_PAIRS.min())})
tcop_summary=pd.DataFrame(tcop_summary)


def draw_tc_op_relations():
    rows=[dict(panels=[(f'TC-Z{z} | '+('가열 초입 온도' if z==3 else '가열·균열 온도'),[f'TC-Z{z}']) for z in range(3,7)],height=2.1),
          dict(panels=[(f'OP-Z{z} | 같은 존의 출력',[f'OP-Z{z}']) for z in range(3,7)],height=2.1)]
    fig,cells=report_page('TC–OP 관계 | 같은 존의 온도와 출력을 위아래로 읽는다',
        f'대표 파형: LOT {TCOP_LOT} · {lot_meta.loc[TCOP_LOT,"GRADE"]} · {tcop_a:%Y/%m/%d %H:%M}–{tcop_b:%H:%M}. 각 열은 하나의 존이고 모든 패널의 실제 시간축이 같다.',
        rows,width=26,info=['회색=1초 원본 · 색=5초 중앙값(4개 이상 실측) · 원 표시=1초 원본 값. 각 존의 세로축 범위·단위는 별도로 읽는다.',
        f'세로선 ① {first_peak:%H:%M:%S} / ② {second_peak:%H:%M:%S} = Z3 원본 온도 최고점. 다른 존에는 같은 시각을 대조하는 참고선이다.',
        '아래 14 LOT 표는 짧은 변화 방향의 반복성을 요약한다. 온도·출력 중 어느 것이 원인인지 정한 그림은 아니다.'])
    ticks=list(pd.date_range(tcop_a,tcop_b,freq='20min'))
    if (tcop_b-ticks[-1]).total_seconds()<10*60:ticks=ticks[:-1]
    if ticks[-1]!=tcop_b:ticks.append(tcop_b)
    tcop_axes=[]
    for row,kind in zip(cells,['TC','OP']):
        for (ax,head,_),zone in zip(row,range(3,7)):
            sensor=f'{kind}-Z{zone}'
            raw_y=case[sensor].loc[tcop_a:tcop_b]
            short=relation_five[TCOP_LOT][sensor].loc[tcop_a:tcop_b]
            ax.plot(raw_y.index,raw_y,color='#aebac3',alpha=.45,lw=.55,label='1초 원본')
            ax.plot(short.index,short,color=sensor_color(sensor),lw=1.4,label='5초 중앙값')
            for point in tcop_points:ax.axvline(point,color='#64748b',ls=':',lw=.9,alpha=.75)
            values=raw_y.dropna();lo,hi=values.min(),values.max();pad=max((hi-lo)*.35,.15 if kind=='TC' else .5)
            ax.set_ylim(lo-pad,hi+pad)
            if kind=='OP' and zone==3:ax.set_ylim(-35,115);ax.set_yticks([0,25,50,75,100])
            ax.set_xlim(tcop_a,tcop_b);ax.set_xticks(ticks);ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.set_ylabel('온도 [℃]' if kind=='TC' else '출력 [%]',fontsize=10)
            ax.set_xlabel('07/07 실제 측정 시각' if kind=='OP' else '',fontsize=10)
            panel_legend(head,ax)
            if zone==3:
                for number,point in enumerate(tcop_points,1):
                    value=float(case.loc[point,sensor]);ax.plot(point,value,'o',ms=4,mfc='white',mec=sensor_color(sensor),zorder=6)
                    near_note(ax,point,value,f'{number} · {value:.2f}'+('℃' if kind=='TC' else '%'),sensor_color(sensor))
            tcop_axes.append(ax)
    assert len(tcop_axes)==8
    assert all(np.array_equal(ax.get_xticks(),tcop_axes[0].get_xticks()) for ax in tcop_axes)
    assert all(np.allclose(ax.get_xlim(),tcop_axes[0].get_xlim(),atol=1e-10,rtol=0) for ax in tcop_axes)
    return fig


fig=draw_tc_op_relations()
emit(fig,'05_관계검토/05_TC_OP_동시파형.png')
show_table(tcop_summary)
tcop_point_results=[]
for number,point in enumerate(tcop_points,1):
    tcop_point_results.append({'확인 지점':number,'실제 시각':point,'TC-Z3 원본[℃]':case.loc[point,'TC-Z3'],'같은 시각 OP-Z3 원본[%]':case.loc[point,'OP-Z3']})
show_table(pd.DataFrame(tcop_point_results))
negative=tcop_zero.R.lt(0).sum();ranges=tcop_summary['0초 변화 r 중앙값']
note(f'**읽은 결과:** Z3 온도가 높은 두 지점의 OP-Z3 원본은 각각 {case.loc[first_peak,"OP-Z3"]:.2f}%·{case.loc[second_peak,"OP-Z3"]:.2f}%다. 파형에서는 온도 상승·하강과 출력의 반대 움직임을 같은 시각에 확인한다. 전체 데이터에서도 유효 {len(tcop_zero)}개 LOT·존 조합 중 {negative}개가 동시 5초 변화의 음의 r이며, 존별 중앙값은 {ranges.min():.3f}~{ranges.max():.3f}다. 예시 한 창과 전체 LOT의 결과를 구분해서 읽는다.')
all_lots=tcop_zero[['LOT','ZONE','R','N_PAIRS']].copy();all_lots['신호']=all_lots.ZONE.map(lambda z:f'TC-Z{z} / OP-Z{z}')
table(all_lots[['LOT','신호','R','N_PAIRS']].rename(columns={'R':'같은 시각 5초 변화 r','N_PAIRS':'유효 비교 수'}))
assert len(tcop_zero)==14*4 and set(tcop_zero.LOT)==set(raw)
assert tcop_zero.N_PAIRS.ge(200).all() and tcop_zero.R.between(-1,1).all()


**관찰 결과와 해석**

- **같은 존의 TC와 OP는 짧은 시간에서 주로 반대로 움직였다.** Z3의 두 반응에서는 온도가 올라간 동안 출력이 낮아지고, 이후 출력이 올라오는 동안 온도가 내려오는 모습이 보인다. Z4~Z6도 대응하는 TC·OP를 각각 읽으며 작은 변화는 각 세로축의 실제 수치로 확인한다.
- **14 LOT에서도 같은 방향성이 반복됐다.** 전환 제외 구간의 동시 5초 변화 r은 Z3~Z6의 56개 LOT·존 조합 모두 음수였고, 존별 중앙값은 약 −0.61~−0.76이다. 이는 온도 변화와 출력 변화가 반대로 기록되는 경향이다. 모든 시간·운전 조건에서 같은 강도로 반복된다는 뜻은 아니다.
- **선후·원인 방향은 별도 문제다.** 기존 시차 탐색에서 집계 최대 절대 r은 0초에 있었으므로, 이 해상도·시차 간격에서 OP가 먼저인지 TC가 먼저인지 분리하지 못했다. 온도에 맞춰 출력을 조절했을 수 있고, 출력과 온도가 서로 반응하거나 공통 설정을 따를 수도 있다. SV·제어 모드·로그 없이 정상 제어 또는 특정 신호의 원인으로 확정하지 않는다.
- **CP 관계와 합쳐서 읽는 방법:** TC–OP의 반대 움직임 자체는 CP 이탈이 없을 때에도 관측된다. 이 절은 온도·출력의 관계이고, [다음 절](#relation-controls)은 그 변화가 CP 이탈 때에만 더 커지는지 확인한다. TC–OP 관계가 관측됐다고 CP 원인이 입증된 것은 아니다.


<a id="tc-op-response"></a>
### 5.7.2 출력을 높인 뒤에도 온도가 내려가는 지점과 이전 수준 복귀

**질문:** OP가 높아졌는데 TC가 계속 내려가는가? 내려가는 동안 다른 존도 같은 모양인가? 온도가 이전 수준에 다시 머물기까지 얼마나 걸렸는가?

**같은 존의 TC–OP는 서로 연결된 제어 루프일 수 있다.** 가열 출력이라면 온도 오차에 따라 출력을 낮추거나 높이는 피드백 설명이 가능하지만, 이 자료만으로 CAL01의 제어 방식·작동 방향을 확인한 것은 아니다. 일반적인 제어에서도 TC·목표값(SV)·실제 구동량을 함께 보아야 한다. OP 명령이 높다는 사실과 실제 열이 투입됐다는 사실도 구분한다. [Eurotherm 제어 점검 설명](https://www.eurotherm.com/en/temperature-control-en/trouble-shooting-temperature-control-equipment/)

**계산 기준 — 고장 기준이나 목표 온도가 아니다.**

- 14 LOT의 Z3~Z6를 모두 검사한다. 기존 승온·하강 전환을 제외하고 **5초 중앙값 13개가 연속 유효**하면서, OP가 모두 90% 이상이고 TC가 60초 동안 2℃ 이상 내려간 창을 찾는다. 90%는 실제 출력 상한·포화값으로 확인된 값이 아니다. 겹치는 창의 시간 범위는 합쳐서 같은 변화의 중복 집계를 줄인다. 빈 구간을 메우지 않는다.
- 본문은 앞 절에서 이미 본 **240069 (b-1), (b-2)**를 이어서 설명한다. 각각 TC 최고점 뒤 12분 안에 처음 나타난 **30초 연속 OP-Z3 ≥90%** 구간을 고른다. t₀는 그 구간 첫 5초창의 끝 시각이다. OP가 최초로 조절된 시각이나 제어 명령 시각과 다르다.
- 온도 기준은 **각 TC 최고점 전 10~5분의 TC 중앙값**을 고정한다. 이 기준은 SV나 정상 판정값이 아니다. t₀ 뒤 10분 내 TC 최저점부터 관측 종료(t₀+15분)까지, 기준 ±1℃ 안의 **12개 연속 유효 5초창(60초)**이 처음 확인된 시각을 찾는다. 5초창은 1초 실측 4개 이상일 때만 유효하다. 도중 결측창이 있으면 60초 확인을 다시 시작한다.
- ±0.5/1/2℃로 복귀 기준을 바꾼 결과도 함께 남긴다. 기존 **시간순 |Z|≥3 탐지와 별개인 복귀 묘사**다. 이전 수준에 잠깐 닿은 것과 60초 머문 것은 구분하며, 15분 내 미확인은 고장·영구 미복귀 판정이 아니다.


In [ ]:
from IPython.utils.capture import capture_output
# 모든 LOT·존의 같은 모양을 찾고, 기존 두 반응의 고정 기준 복귀를 계산한다.
# relation_five는 결측을 채우지 않은 5초 중앙값이다. 이 절에서는 창의 끝 시각으로 표시한다.
response_five={k:v.copy() for k,v in relation_five.items()}
response_phase={k:v.copy() for k,v in relation_phase.items()}
for k in response_five:
    response_five[k].index+=pd.Timedelta(seconds=5)
    response_phase[k].index+=pd.Timedelta(seconds=5)


def output_falling_windows(output_floor=90.,fall_c=2.):
    found=[];coverage=[]
    for k,v in response_five.items():
        for zone in range(3,7):
            tc,op=v[f'TC-Z{zone}'],v[f'OP-Z{zone}']
            eligible=(tc.notna()&op.notna()&response_phase[k]).rolling(13,min_periods=13).sum().eq(13)
            hit=eligible&op.ge(output_floor).rolling(13,min_periods=13).sum().eq(13)&tc.diff(12).le(-fall_c)
            coverage.append(dict(LOT=k,ZONE=zone,VALID_END_POINTS=int(eligible.sum())))
            blocks=[]
            for end in hit.index[hit]:
                start=end-pd.Timedelta(seconds=60)
                if blocks and start<=blocks[-1][1]:blocks[-1][1]=end
                else:blocks.append([start,end])
            for start,end in blocks:
                q=v.loc[start:end];ends=hit.loc[start:end];ends=ends.index[ends]
                found.append(dict(LOT=k,ZONE=zone,START=start,END=end,
                    MIN_OP=float(q[f'OP-Z{zone}'].min()),
                    WORST_60S_C=float(tc.diff(12).loc[ends].min()),
                    HIT_WINDOWS=len(ends),OUTPUT_FLOOR=output_floor))
    return pd.DataFrame(found),pd.DataFrame(coverage)


output_fall_intervals,output_fall_coverage=output_falling_windows()
output_fall_summary=[]
for zone in range(3,7):
    subset=output_fall_intervals.loc[output_fall_intervals.ZONE.eq(zone)]
    output_fall_summary.append({'존':f'TC-Z{zone} / OP-Z{zone}',
        '검사 LOT':output_fall_coverage.loc[output_fall_coverage.ZONE.eq(zone),'LOT'].nunique(),
        '이 모양이 있는 LOT':subset.LOT.nunique(),'겹치는 창을 합친 구간 수':len(subset)})
output_fall_summary=pd.DataFrame(output_fall_summary)

response_returns=[];response_cases=[]
for j,peak in enumerate(tcop_points,1):
    k=TCOP_LOT;v=response_five[k];tc,op=v['TC-Z3'],v['OP-Z3']
    pre=(v.index>peak-pd.Timedelta(minutes=10))&(v.index<=peak-pd.Timedelta(minutes=5))
    assert tc.loc[pre].count()>=48  # 5분 기준 창 60개 중 48개 이상 실측 요약
    base=float(tc.loc[pre].median())
    high=op.loc[peak:peak+pd.Timedelta(minutes=12)].ge(90)
    confirms=high.rolling(6,min_periods=6).sum().eq(6)
    if not confirms.any():raise AssertionError('이 확인 구간에는 30초 연속 높은 출력이 없음')
    high_confirm=confirms.index[confirms][0]
    t0=high_confirm-pd.Timedelta(seconds=25)
    end=t0+pd.Timedelta(minutes=15)
    search=tc.loc[t0:t0+pd.Timedelta(minutes=10)]
    assert search.count()>=108 and tc.loc[t0:end].count()>=162
    assert response_phase[k].loc[peak-pd.Timedelta(minutes=10):end].all()
    trough=search.idxmin()
    rec=dict(TAG=f'(b-{j})',LOT=k,PEAK=peak,BASE=base,T0=t0,END=end,
        HIGH_CONFIRMED=high_confirm,TROUGH=trough,LOW=float(tc.loc[trough]),
        FALL_C=float(tc.loc[trough]-tc.loc[t0]),
        FALL_MIN=(trough-t0).total_seconds()/60,
        OTHER_BASE={s:float(v.loc[pre,s].median()) for s in REL_SENSORS})
    for band in [.5,1.,2.]:
        after=tc.loc[trough:end];inside=after.sub(base).abs().le(band)
        ok=inside.rolling(12,min_periods=12).sum().eq(12)
        returned=ok.index[ok][0] if ok.any() else pd.NaT
        entry=inside.index[inside][0] if inside.any() else pd.NaT
        response_returns.append(dict(TAG=rec['TAG'],LOT=k,BASE=base,BAND_C=band,T0=t0,
            TROUGH=trough,FIRST_ENTRY=entry,CONFIRMED=returned,
            DELAY_MIN=(returned-t0).total_seconds()/60 if pd.notna(returned) else np.nan,
            OBSERVATION_END=end))
        if band==1.:
            rec['CONFIRMED']=returned
            rec['RETURN_START']=returned-pd.Timedelta(seconds=60) if pd.notna(returned) else pd.NaT
    response_cases.append(rec)
response_returns=pd.DataFrame(response_returns)

with capture_output() as response_tables:
    show_table(output_fall_summary)
    note('90% 이상·60초 하강이라는 모양의 집계다. 구간 수는 고장 수가 아니다. Z4~Z6에서 이 조건이 없다고 정상으로 판정하지 않는다. 존마다 평소 출력 수준과 한계가 다르다.')
    table(output_fall_intervals.rename(columns={'LOT':'LOT','ZONE':'존','START':'관측 창 시작','END':'관측 창 끝','MIN_OP':'최저 OP [%]','WORST_60S_C':'최대 60초 하강 [℃]'}))
    show_table(response_returns.rename(columns={'TAG':'구간','BASE':'이전 TC 기준 [℃]','BAND_C':'복귀 폭 ±℃','T0':'고출력 기준 시각','TROUGH':'TC 최저 창 끝','FIRST_ENTRY':'최저 뒤 첫 진입','CONFIRMED':'60초 복귀 확인','DELAY_MIN':'고출력 기준부터 확인까지 [분]','OBSERVATION_END':'관측 종료'}))
captured_details('14 LOT·모든 존 검사와 복귀 기준 민감도 표',response_tables)
assert len(response_returns)==6 and len(output_fall_coverage)==14*4
for r in response_cases:
    assert r['T0']<r['TROUGH']<r['END']
    v=response_five[r['LOT']]
    assert v.loc[r['T0']:r['HIGH_CONFIRMED'],'OP-Z3'].ge(90).all()
    if pd.notna(r['CONFIRMED']):
        inside=v['TC-Z3'].loc[(v.index>r['RETURN_START'])&(v.index<=r['CONFIRMED'])]
        assert len(inside)==12 and inside.sub(r['BASE']).abs().le(1.).all()


**아래 그림에서 바로 볼 점:** 두 열은 서로 다른 반응이다. 위 두 행에서 **고출력 시작 → TC 추가 하강 → 이전 수준 복귀 여부**를 읽고, 아래 두 행에서 같은 실제 시각의 Z4~Z6를 대조한다. 한 열의 네 패널은 시간 범위·눈금이 같다. 주황 음영은 t₀부터 TC 최저점까지, 회색 띠는 TC의 이전 기준 ±1℃다. 파란 세로선은 60초 복귀 확인이며, 목표 온도 도달을 뜻하지 않는다.


In [ ]:
# 실제 시각과 같은 존의 TC·OP를 맞추고, 다른 존의 반응을 아래에서 대조한다.
def draw_tc_op_return():
    rows=[dict(panels=[(r['TAG']+' TC-Z3 | 고출력 이후 추가 하강과 복귀',['TC-Z3']) for r in response_cases],height=2.65),
          dict(panels=[(r['TAG']+' OP-Z3 | 같은 존의 실제 출력',['OP-Z3']) for r in response_cases],height=1.6),
          dict(panels=[(r['TAG']+' TC-Z4·Z5·Z6 | 이전 기준 대비 변화',[f'TC-Z{z}' for z in range(4,7)]) for r in response_cases],height=1.35),
          dict(panels=[(r['TAG']+' OP-Z4·Z5·Z6 | 이전 기준 대비 변화',[f'OP-Z{z}' for z in range(4,7)]) for r in response_cases],height=1.35)]
    first,second=response_cases
    one=response_returns.loc[response_returns.TAG.eq(first['TAG'])&response_returns.BAND_C.eq(1.)].iloc[0]
    fig,cells=report_page('240069 | OP를 높인 뒤 TC는 언제 이전 수준으로 돌아왔는가?',
        f'07/07 · SPHC · 왼쪽 {first["TAG"]} 고출력 기준 {first["T0"]:%H:%M:%S} / 오른쪽 {second["TAG"]} {second["T0"]:%H:%M:%S}. 각 열의 모든 행은 같은 실제 시각이다.',
        rows,width=24,info=[
        f'{first["TAG"]} 추가 하강 {first["FALL_MIN"]:.1f}분, ±1℃ 60초 복귀 확인까지 {one.DELAY_MIN:.1f}분 / {second["TAG"]} 추가 하강 {second["FALL_MIN"]:.1f}분, 15분 내 같은 복귀 미확인.',
        '회색 띠=TC 최고점 전 10~5분 중앙값 ±1℃ · 주황 음영=고출력 시작~TC 최저 · 파란 선=60초 복귀 확인. SV·고장 한계가 아니다.',
        '색 선=유효 5초 중앙값(창 끝 시각) · 회색 가는 선=1초 원본. OP≥90%는 비교용 기준이며 실제 포화·전력 투입은 미확인.'])
    response_axis_checks=[]
    context_limits={}
    for kind in ['TC','OP']:
        peak=0.
        for r in response_cases:
            v=response_five[r['LOT']].loc[r['T0']-pd.Timedelta(minutes=12):r['END']]
            for z in range(4,7):
                sensor=f'{kind}-Z{z}'
                peak=max(peak,float((v[sensor]-r['OTHER_BASE'][sensor]).abs().max()))
        context_limits[kind]=max(2. if kind=='TC' else 8.,peak*1.25)
    for col,rec in enumerate(response_cases):
        v=response_five[rec['LOT']];a=rec['T0']-pd.Timedelta(minutes=12);b=rec['END']
        ticks=[rec['T0']+pd.Timedelta(minutes=m) for m in [-10,0,5,10,15]]
        for row,(ax,head,sensors) in enumerate([cells[q][col] for q in range(4)]):
            for sensor in sensors:
                y=v[sensor].loc[a:b]
                offset=rec['OTHER_BASE'][sensor] if row>=2 else 0
                if row<2:
                    raw_y=relation_raw[rec['LOT']][sensor].loc[a:b]
                    ax.plot(raw_y.index,raw_y,color='#acb8c2',alpha=.4,lw=.55)
                ax.plot(y.index,y-offset,lw=1.35,color=sensor_color(sensor),label=sensor)
            ax.axvspan(rec['T0'],rec['TROUGH'],color=EVENT_COLORS['Z'],alpha=.13,zorder=0)
            ax.axvline(rec['T0'],color=EVENT_COLORS['Z'],ls='--',lw=1.)
            if pd.notna(rec['CONFIRMED']):ax.axvline(rec['CONFIRMED'],color='#166b8a',ls='-.',lw=1.)
            ax.set_xlim(a,b);ax.set_xticks(ticks);ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            ax.set_xlabel('07/07 실제 측정 시각' if row==3 else '')
            ax.set_ylabel(['TC-Z3 [℃]','OP-Z3 [%]','TC 변화 [℃]','OP 변화 [%p]'][row])
            if row==0:
                ax.axhspan(rec['BASE']-1,rec['BASE']+1,color='#8ca2b2',alpha=.15,zorder=0)
                ax.axhline(rec['BASE'],color='#64748b',ls=':',lw=.8)
                ax.set_ylim(846,877)
                ax.plot(rec['TROUGH'],rec['LOW'],'o',mfc='white',mec=sensor_color('TC-Z3'),ms=4,zorder=6)
                near_note(ax,rec['TROUGH'],rec['LOW'],f'{rec["LOW"]:.2f}℃\n추가 하강 {rec["FALL_MIN"]:.1f}분',sensor_color('TC-Z3'))
                if pd.notna(rec['CONFIRMED']):
                    val=float(v.loc[rec['CONFIRMED'],'TC-Z3']);ax.plot(rec['CONFIRMED'],val,'o',mfc='white',mec='#166b8a',ms=4,zorder=6)
                    near_note(ax,rec['CONFIRMED'],val,'60초 복귀 확인','#166b8a')
                else:
                    val=float(v.loc[b,'TC-Z3']);ax.plot(b,val,'s',mfc='white',mec='#166b8a',ms=4,zorder=6)
                    near_note(ax,b,val,'15분 내 미확인','#166b8a')
                extra=[Patch(facecolor='#8ca2b2',alpha=.3,label=f'이전 {rec["BASE"]:.2f}℃ ±1℃')]
            elif row==1:
                ax.axhline(90,color='#526d80',ls=':',lw=.9,label='비교 기준 90%');ax.set_ylim(-8,112)
                ax.set_yticks([0,50,90]);extra=[]
            else:
                ax.axhline(0,color='#64748b',ls=':',lw=.8);limit=context_limits['TC' if row==2 else 'OP'];ax.set_ylim(-limit,limit);extra=[]
            panel_legend(head,ax,extra=extra)
            for line in ax.lines:
                yy=np.asarray(line.get_ydata(),dtype=float)
                if len(yy)>2 and np.isfinite(yy).any():
                    lo,hi=ax.get_ylim();assert np.nanmin(yy)>=lo and np.nanmax(yy)<=hi,'파형이 세로축 밖으로 잘림'
            response_axis_checks.append((col,ax.get_xlim(),ax.get_xticks().copy()))
    for col in range(2):
        axes=[x for x in response_axis_checks if x[0]==col]
        assert len(axes)==4 and all(np.allclose(v[1],axes[0][1],atol=1e-10,rtol=0) and np.array_equal(v[2],axes[0][2]) for v in axes)
    return fig


fig=draw_tc_op_return()
emit(fig,'05_관계검토/06_TC_OP_고출력후_하강과복귀.png')
return_summary=[]
for rec in response_cases:
    rr=response_returns.loc[response_returns.TAG.eq(rec['TAG'])&response_returns.BAND_C.eq(1.)].iloc[0]
    return_summary.append({'확인 구간':'240069 '+rec['TAG'],'고출력 기준 시각':rec['T0'],
        '고출력 뒤 추가 하강 [℃]':rec['FALL_C'],'TC 최저 창 끝':rec['TROUGH'],
        '추가 하강 [분]':rec['FALL_MIN'],'이전 수준 ±1℃ 60초 확인':rr.CONFIRMED if pd.notna(rr.CONFIRMED) else '15분 내 미확인',
        '고출력부터 확인까지 [분]':round(rr.DELAY_MIN,2) if pd.notna(rr.DELAY_MIN) else '미확인'})
show_table(pd.DataFrame(return_summary))
pattern_lots=output_fall_intervals.LOT.nunique()
note(f'**이 모양만으로 이상 판정할 수 없는 이유:** 높은 출력과 온도 하강의 동시 기록은 14 LOT 중 {pattern_lots}개에서 관측됐다. 두 반응 모두 TC-Z3의 추가 하강과 OP-Z3의 높은 값이 함께 남지만, 복귀 양상은 다르다. 다른 존의 온도·출력은 아래 같은 시간축에서 개별 크기를 비교한다. 고출력 이전의 출력 감소, 열적 지연, 제어 설정·부하 변화 또는 구동·센서 문제는 확인할 가설이며 현재 자료로 구분하지 못한다.')


**관찰 결과 — 무엇을 확인했고 어디를 다시 볼 것인가?**

1. **TC에 따라 OP가 움직였는가?** 동시 변화는 주로 역방향이며, 일반적인 피드백 조절과 양립한다. 그러나 SV·자동/수동 모드가 없으므로 TC가 OP 변화를 유발했다고 확정하지 않는다. 관측된 0초 차분 상관을 ‘OP의 효과가 즉시 나타났다’고 해석하지 않는다.
2. **OP 뒤 몇 분 만에 돌아왔는가?** 이 절의 시간은 고출력 구간 기준 시각부터 **과거 관측 수준에 60초 머문 확인 시각**까지다. 목표 온도 복귀 시간·설비의 고유 지연이 아니다. 첫 반응은 복귀가 관측됐고 두 번째는 15분 창에서 확인되지 않았다. ±0.5/1/2℃의 표를 함께 읽어 임의의 복귀 폭에 결과가 얼마나 달라지는지 확인한다.
3. **우선 확인할 지점:** 240069 **07/07 23:36경~23:51경의 TC-Z3 / OP-Z3**, 특히 OP가 높아진 뒤에도 TC가 추가 하강하는 23:36경~23:40경이다. 같은 시각의 TC-Z4~Z6·OP-Z4~Z6를 함께 보아 Z3의 큰 반응이 공정 전체의 동일한 변화인지 구분한다. 앞선 반응과의 복귀 차이는 점검 질문을 구체화하지만 정상/고장의 라벨은 주지 않는다.
4. **‘출력을 줬는데 제어가 안 됐다’는 판단에 필요한 조건:** SV가 일정하고, OP가 무엇을 구동하며 어느 방향으로 작동하는지 확인되고, 수동·인터록·정지·부하 전환이 없어야 한다. 그 조건에서 허용 오차 밖 TC가 평소 응답시간보다 오래 지속되는지, 실제 전류·전력·유량·밸브 피드백도 명령을 따랐는지를 확인한다. OP 명령만 크면 실제 열 전달 불량·명령 미실행·센서 문제를 구분할 수 없다. 이때에야 제어 성능 저하 후보를 정의할 수 있다. [Eurotherm 점검 기준](https://www.eurotherm.com/en/temperature-control-en/trouble-shooting-temperature-control-equipment/)

**정비와 연결할 때:** 이 23시대 반응은 CP의 큰 피크(07/08 02:31)보다 앞서며, BM 01094의 감지(07/08 08:49)와도 떨어져 있다. 시간상 앞선다는 사실은 고장 전조의 증거가 아니다. BM의 ‘1호기 재발, 절연 저하’가 Z3 구동 계통인지 확인되지 않아 원인으로 연결하지 않는다. 관련 설비 태그·제어 로그를 확보한 뒤 이 시각을 대조해야 한다.


<a id="tc-quality-evidence"></a>
### 5.7.3 존 온도에서 제품 품질의 근거로 연결하려면

**현재는 존 온도만으로 제품 품질을 결정할 수 없다.** 연속 소둔의 열 이력과 재질 사이에는 근거가 있지만, 이 자료에서 그 관계를 검증한 것은 아니다. JFE의 기술 보고는 강종·성분에 따른 열처리 경로와 냉각·과시효 조건이 기계적 성질에 영향을 주는 과정을 설명한다. 논문의 조건을 CAL01의 정상 온도나 불량 임계값으로 옮겨 쓰지 않는다. [연속 소둔 기술과 제품 개발](https://www.jfe-steel.co.jp/archives/en/ksc_giho/no.22/tobira048.html), [열 사이클·탄소량과 기계적 성질](https://www.jfe-steel.co.jp/archives/ksc_giho/16-4/tobira264.htm)

| 확인할 품질 가설 | 연결할 공정 정보 | 필요한 검증 근거 |
| --- | --- | --- |
| 가열·균열 조건이 강도·연신율 차이와 연결되는가? | 소재의 가열 최고 온도·승온 속도·유지시간, 강종·성분·두께 | 소재 온도 측정 위치·정의, 해당 소재의 인장강도·항복강도·연신율 및 판정 규격 |
| 냉각 조건이 재질 차이와 연결되는가? | 소재의 실제 냉각 속도·경로·도달 온도 | 소재 이동 경로·라인 속도, 냉각 설비 상태, 같은 위치의 재질 시험 결과 |
| 과시효 조건이 품질 차이와 연결되는가? | 강종별 실제 유지 온도·시간 | 강종별 승인 레시피, 관련 시험 항목·결과와 소재 위치 연결 |
| Z3의 큰 변동이 제품까지 전달됐는가? | 같은 소재가 Z3에서 이후 존을 지나간 열 이력 | 존 간 거리·체류시간·속도 변화와 코일/소재 위치. 같은 시계 시각의 존끼리는 같은 소재를 뜻하지 않음 |

**지금 만들 수 있는 것은 검증용 요약이다.** 기존 LOT·존별 중앙값·평균·IQR·변화 크기·지속시간을 보존하고, 소재가 추적되면 강종별 허용 범위 밖 유지시간과 냉각 속도를 추가한다. 이후 동일 소재 위치의 품질 결과와 연결해 조건별 차이 및 반례를 확인한다. 현재 LOT 기준 평균만으로 짧은 구간의 품질 영향을 평가하지 않는다.

현재 CAL01 CSV에는 TC·OP·CP와 LOT·시각이 있으며 **SV, CAL01 라인 속도, 소재 온도 정의, 두께·성분, 품질 시험값·규격이 없다.** 다른 `WLD01_용접.csv`의 `SPD`는 WLD01 신호로, CAL01 속도로 사용할 연결 근거가 없다. 따라서 이번 결과는 **TC–OP 관계 및 점검할 구간을 제시한 탐색 분석**이며, 제어 불능이나 품질 불량을 확정한 결과가 아니다.


<a id="relation-controls"></a>
### 5.8 Q3. CP가 튄 때에만 주변 존도 더 크게 움직였는가?

**질문:** CP 이탈이 있던 때의 TC·OP 변화가, 같은 LOT에서 CP 이탈이 없던 다른 때보다 큰가? 다른 때에도 비슷하거나 더 큰 움직임이 있는가?

**먼저 구분할 것:** 아래의 차이는 **두 시간대에서 관측한 변화폭의 차이**다. CP가 주변 존에 준 영향의 크기가 아니다. CP가 존을 바꾸었는지, 존의 변화가 CP와 연결되는지, 공통 운전 조건이 둘을 바꾸었는지는 이 비교만으로 결정할 수 없다.

**이전 절과의 연결:** 5.7은 한 CP 구간 안에서 신호가 어떤 순서로 움직였는지 본다. 여기서는 비슷한 움직임이 CP 이탈이 없는 때에도 나오는지 확인한다.

**바꾼 읽기 방식:** 연결점 대신 두 시간대의 **CP 원본 → TC-Z3/OP-Z3 → TC-Z4/OP-Z4 → TC-Z5/OP-Z5 → TC-Z6/OP-Z6** 파형을 겹쳐 본다. 빨강은 CP 이탈이 있던 때, 회색 점선은 이탈이 없던 다른 때다. **두 선은 서로 다른 실제 시각**이며 각 시작을 0분으로 정렬했다. 원래 날짜·시각은 맨 위에 표시한다. 회색도 CP가 실제 측정된 구간이다.

**변화폭의 뜻:** 각 시간대의 시작 전 5~1분 중앙값을 기준으로, 시작 뒤 0~4분 중 **위아래로 가장 멀어진 값의 절댓값**이다. 온도는 ℃, 출력은 %p로 따로 읽는다. 예를 들어 앞 기준 860℃에서 뒤 4분에 858℃까지 내려가면 최대폭은 2℃다. 선을 보면 방향도 알 수 있고, 점은 그 최대폭이 기록된 위치를 뜻한다. 존마다 세로축 범위가 다르므로 최대폭 숫자를 함께 읽는다. 정상·고장 임계값이나 지속시간은 아니다.

**시간대 선정:** 같은 LOT에서 CP가 LOT 중앙값 ±0.02 안에 기록된 다른 시간대를 찾는다. 5분 간격 후보 중 CP 이탈 시작~끝 ±10분과 겹치는 창을 제외하고, 전환 참고 구간도 제외했다. CP 실측 90% 이상, TC·OP의 앞·뒤 기록 각각 216/240초 이상을 요구한다. 앞 기준 TC-Z4~Z6가 각각 2.5℃ 이내인 후보 중 온도 차이가 작은 시간을 선택하고 시간 거리도 약하게 반영한다. 다른 시간대는 겹치거나 중복되지 않는다. 세부 선정 수치는 아래 접힌 표에 둔다.

**대표 선정·누락 방지:** 14 CP 구간 중 이런 비교가 가능한 것은 12개·5 LOT이다. 가장 큰 CP 피크인 **240069 (c-1)**을 본문에서 먼저 보이고, 나머지 11개도 모두 같은 파형 형식으로 펼쳐볼 수 있다. 같은 강종·LOT이지만 SV·속도·두께·부하까지 같은 운전 조건은 아니다. 12개 중 7개가 240069에 속하므로 독립된 실험 12회로 일반화하지 않는다.


In [ ]:
# CP 사건과 같은 LOT의 CP 없는 대조 창을 짝짓고 8개 신호를 같은 정의로 계산한다.
def relation_window(k,t,need_quiet=False):
    frame=relation_raw[k];a=t-pd.Timedelta(minutes=5);b=t+pd.Timedelta(minutes=4)
    if a<frame.index.min() or b>frame.index.max()+pd.Timedelta(seconds=1):return None,'관측 범위 부족'
    mins=np.arange(int((a-lot_meta.loc[k,'START']).total_seconds()//60),int(np.ceil((b-lot_meta.loc[k,'START']).total_seconds()/60)))
    if not comparison_states[k].reindex(mins).eq('안정 참고').all():return None,'전환 구간 포함'
    pre=frame.loc[(frame.index>=a)&(frame.index<t-pd.Timedelta(minutes=1)),REL_SENSORS]
    post=frame.loc[(frame.index>=t)&(frame.index<b),REL_SENSORS]
    if pre.count().min()<216 or post.count().min()<216:return None,'TC·OP 기록 부족'
    cp=frame['CP-Z4'].loc[(frame.index>=a)&(frame.index<b)]
    if cp.count()<.9*len(cp):return None,'CP 기록 부족'
    cp_dev=(cp-frame['CP-Z4'].median()).abs().max()
    if need_quiet and cp_dev>CP_GROUP:return None,'CP 0.02 범위 초과'
    base=pre.median();amp=(post-base).abs().max()
    return dict(BASE=base,AMP=amp,CP_MAX=float(cp_dev),TIME=t),''

relation_control_pool={}
for k in relation_events.LOT.astype(int).unique():
    frame=relation_raw[k];pool=[]
    first=frame.index.min()+pd.Timedelta(minutes=5);last=frame.index.max()-pd.Timedelta(minutes=4)
    for t in pd.date_range(first.ceil('5min'),last,freq='5min'):
        occupied=False
        for ev in relation_events.loc[relation_events.LOT.eq(k)].itertuples():
            end=at(k,ev.B).round('s')
            if t+pd.Timedelta(minutes=4)>=ev.ONSET-pd.Timedelta(minutes=10) and t-pd.Timedelta(minutes=5)<=end+pd.Timedelta(minutes=10):occupied=True;break
        if occupied:continue
        features,reason=relation_window(k,t,need_quiet=True)
        if features is not None:pool.append(features)
    relation_control_pool[k]=pool
relation_matches=[];relation_match_coverage=[];used={k:[] for k in relation_control_pool}
for ev in relation_events.itertuples():
    k=int(ev.LOT);features,reason=relation_window(k,ev.ONSET)
    if features is None:
        relation_match_coverage.append(dict(ID=ev.ID,LOT=k,STATUS=reason));continue
    candidates=[]
    for control in relation_control_pool[k]:
        if any(abs((control['TIME']-t).total_seconds())<9*60 for t in used[k]):continue
        diff=(control['BASE'][HEAT[1:]]-features['BASE'][HEAT[1:]]).abs()
        if diff.max()>2.5:continue
        cost=float(diff.sum()/.5)+.001*abs((control['TIME']-ev.ONSET).total_seconds()/60)
        candidates.append((cost,control,float(diff.max())))
    if not candidates:
        relation_match_coverage.append(dict(ID=ev.ID,LOT=k,STATUS='조건을 만족하는 별도 대조 창 없음'));continue
    _,control,gap=min(candidates,key=lambda x:x[0]);used[k].append(control['TIME'])
    relation_match_coverage.append(dict(ID=ev.ID,LOT=k,STATUS='대조 가능'))
    for sensor in REL_SENSORS:
        relation_matches.append(dict(ID=ev.ID,LOT=k,SENSOR=sensor,EVENT_START=ev.ONSET,CONTROL_START=control['TIME'],
            EVENT_AMP=features['AMP'][sensor],CONTROL_AMP=control['AMP'][sensor],
            BASE_TC456_MAX_GAP=gap,CONTROL_CP_MAX=control['CP_MAX']))
relation_matches=pd.DataFrame(relation_matches);relation_match_coverage=pd.DataFrame(relation_match_coverage)
relation_match_summary=[]
for sensor in REL_SENSORS:
    g=relation_matches.loc[relation_matches.SENSOR.eq(sensor)]
    difference=g.EVENT_AMP-g.CONTROL_AMP
    relation_match_summary.append({'신호':sensor,'비교 쌍':len(g),'사건>대조':int(difference.gt(0).sum()),
        '사건 최대변화 중앙값':g.EVENT_AMP.median(),'대조 최대변화 중앙값':g.CONTROL_AMP.median(),
        '짝 차이 중앙값':difference.median(),'해당 LOT 수':g.LOT.nunique()})
relation_match_summary=pd.DataFrame(relation_match_summary)
matched=relation_match_coverage.STATUS.eq('대조 가능')
# 점 두 개 대신 서로 다른 두 시간대의 실제 파형을 같은 경과시간에 겹친다.
# 선색은 신호 종류가 아니라 두 시간대를 뜻하며, 원본 값·기준·최대폭을 확인한다.
PAIR_RED='#B11226';PAIR_GREY='#688092'
relation_pair_wave_checks=[];relation_pair_amplitude_checks=[]
matched_events=relation_events.loc[relation_events.ID.isin(relation_matches.ID.unique())].copy()
representative_id=matched_events.set_index('ID').DEV.abs().idxmax()  # 비교 가능한 CP 중 절대 피크 최대


def draw_cp_present_absent(ident):
    from matplotlib.patches import Patch
    from matplotlib.ticker import MaxNLocator
    ev=relation_events.loc[relation_events.ID.eq(ident)].iloc[0];k=int(ev.LOT)
    records=relation_matches.loc[relation_matches.ID.eq(ident)].set_index('SENSOR')
    quiet=records.CONTROL_START.iloc[0];frame=relation_raw[k]
    first,_=relation_window(k,ev.ONSET);second,_=relation_window(k,quiet,need_quiet=True)
    assert first is not None and second is not None
    times=[ev.ONSET,quiet];colors=[PAIR_RED,PAIR_GREY]
    periods=['CP 이탈이 있던 때','CP 이탈이 없던 다른 때']
    tag=event_text(ident);width=20;head_h,tick_h,gap,top=.65,.40,.14,2.1
    layout=[(['CP-Z4'],None,1.45)]+[([f'TC-Z{z}'],[f'OP-Z{z}'],1.25) for z in range(3,7)]
    height=top+sum(head_h+h+tick_h+gap for _,_,h in layout)+.15
    fig=plt.figure(figsize=(width,height),dpi=150,facecolor='white')
    fig.suptitle(tag+' | CP가 튄 때, 주변 존도 항상 더 크게 움직였는가?',x=.07,y=1-.14/height,
                 ha='left',va='top',fontsize=18,fontweight='bold',color='#17344d')
    fig.text(.07,1-.54/height,f'빨강: {ev.ONSET:%Y/%m/%d %H:%M:%S} 시작 · 회색: {quiet:%Y/%m/%d %H:%M:%S} 시작 · 같은 LOT {k} / {lot_meta.loc[k,"GRADE"]}',
             ha='left',va='top',fontsize=10.5,color='#17344d')
    fig.text(.07,1-.86/height,'서로 다른 실제 시각의 두 파형이다. 가로축은 각 시작에서의 경과 분이며, 같은 가로 위치가 실제 동시 관측이라는 뜻은 아니다.',
             ha='left',va='top',fontsize=10,color='#526d80')
    fig.text(.07,1-1.16/height,'TC·OP는 각 시간대의 앞 5~1분 중앙값을 뺀 값 · 점=뒤 4분 중 기준에서 가장 멀어진 위치 · 수치=그 폭의 절댓값.',
             ha='left',va='top',fontsize=10,color='#526d80')
    fig.text(.07,1-1.46/height,'크기뿐 아니라 선의 위아래 방향·변화 시점도 읽는다. 같은 LOT·비슷한 Z4~Z6 시작 온도이며, 설정·속도·부하까지 같다는 근거는 없다.',
             ha='left',va='top',fontsize=9.5,color='#526d80')
    handles=[Line2D([0],[0],color=PAIR_RED,lw=1.8,label=periods[0]),Line2D([0],[0],color=PAIR_GREY,lw=1.6,ls='--',label=periods[1]),
             Line2D([0],[0],marker='o',color='#475569',ls='',label='비교한 4분의 최대폭 위치'),Patch(facecolor='#e9f1f5',label='시작 뒤 4분')]
    fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.075,1-1.74/height),ncol=4,fontsize=9.5,frameon=False,borderaxespad=0)
    cursor=top;cells=[]
    for left,right,data_h in layout:
        row=[];signals=[left] if right is None else [left,right]
        ncols=len(signals);hgap=.075;w=(.98-.075-(ncols-1)*hgap)/ncols
        for col,sensors in enumerate(signals):
            sensor=sensors[0];x=.075+col*(w+hgap)
            head=fig.add_axes([x,1-(cursor+head_h)/height,w,head_h/height]);head.axis('off')
            ax=fig.add_axes([x,1-(cursor+head_h+data_h)/height,w,data_h/height]);style(ax,'')
            ax.yaxis.set_major_locator(MaxNLocator(4));ax.tick_params(labelsize=9,pad=3)
            title='CP-Z4 | 실제 원본값' if sensor=='CP-Z4' else sensor+' | '+('온도 변화' if sensor.startswith('TC') else '출력 변화')
            head.text(0,.98,title,ha='left',va='top',fontsize=12,fontweight='bold',color='#17344d')
            if sensor=='CP-Z4':
                caption=f'선택 CP 피크의 LOT 중앙값 편차 {ev.DEV:+.4f} · 다른 때 9분 전체의 최대 절대편차 {second["CP_MAX"]:.4f} · 가로 점선=중앙값 ±0.05 탐색선'
            else:
                r=records.loc[sensor];unit='℃' if sensor.startswith('TC') else '%p'
                caption=f'기준에서 가장 멀어진 폭: CP 이탈 때 {r.EVENT_AMP:.2f}{unit} / 이탈 없던 다른 때 {r.CONTROL_AMP:.2f}{unit}'
            head.text(0,.49,caption,ha='left',va='top',fontsize=9.5,color='#526d80')
            values=[]
            for number,(start,color,label,features) in enumerate(zip(times,colors,periods,[first,second])):
                a,b=start-pd.Timedelta(minutes=5),start+pd.Timedelta(minutes=4)
                y=frame[sensor].loc[(frame.index>=a)&(frame.index<b)]
                if sensor!='CP-Z4':y=y-features['BASE'][sensor]
                elapsed=(y.index-start).total_seconds()/60
                ax.plot(elapsed,y,color=color,lw=1.25 if number==0 else 1.15,ls='-' if number==0 else '--',label=label)
                values.append(y.dropna())
                post=y.loc[(y.index>=start)&(y.index<b)]
                if sensor=='CP-Z4':
                    if number==0:point=ev.PEAK
                    else:point=(post-frame['CP-Z4'].median()).abs().idxmax()
                else:
                    point=post.abs().idxmax();amp=float(abs(post.loc[point]))
                    expected=float(records.loc[sensor,'EVENT_AMP' if number==0 else 'CONTROL_AMP'])
                    assert np.isclose(amp,expected,rtol=0,atol=1e-9),(ident,sensor,number,amp,expected)
                    relation_pair_amplitude_checks.append((ident,sensor,number,amp))
                ax.plot((point-start).total_seconds()/60,y.loc[point],'o',ms=4,mfc='white',mec=color,mew=1.2,zorder=6)
            ax.axvspan(0,4,color='#e9f1f5',alpha=.55,lw=0,zorder=0)
            ax.axvline(0,color='#8496a5',ls=':',lw=.9)
            if sensor=='CP-Z4':
                base=frame[sensor].median()
                for bound in [base-.05,base+.05]:ax.axhline(bound,color=PAIR_RED,ls=':',lw=.8,alpha=.6)
            else:ax.axhline(0,color='#8496a5',lw=.7,ls=':')
            pooled=pd.concat(values);lo,hi=pooled.min(),pooled.max();pad=max((hi-lo)*.25,.01 if sensor=='CP-Z4' else .1)
            ax.set_ylim(lo-pad,hi+pad);ax.set_xlim(-5,4);ax.set_xticks([-5,-3,-1,0,2,4])
            ax.set_ylabel('CP [무차원]' if sensor=='CP-Z4' else '앞 기준 대비 [℃]' if sensor.startswith('TC') else '앞 기준 대비 [%p]',fontsize=10)
            if cursor==top or sensor in ['TC-Z6','OP-Z6']:ax.set_xlabel('각 시간대의 시작 대비 [분] · 0은 서로 다른 실제 시각',fontsize=9.5)
            row.append((ax,head,sensors))
        cells.append(row);cursor+=head_h+data_h+tick_h+gap
    fig._report_cells=cells
    axes=[ax for row in cells for ax,_,_ in row]
    assert len(axes)==9 and all(np.array_equal(ax.get_xticks(),axes[0].get_xticks()) for ax in axes)
    assert all(ax.get_xlim()==(-5,4) for ax in axes)
    relation_pair_wave_checks.append(dict(ID=ident,LOT=k,CP_TIME=ev.ONSET,OTHER_TIME=quiet,N_PANELS=9))
    return fig


fig=draw_cp_present_absent(representative_id)
emit(fig,'05_관계검토/03_CP이탈_유무_실제파형.png')
primary=relation_matches.loc[relation_matches.ID.eq(representative_id)].set_index('SENSOR')
note(f'**이 사례에서 읽는 결과:** 비교 가능한 CP 중 절대 피크가 가장 큰 {event_text(representative_id)}를 먼저 보였다. TC-Z3의 4분 최대폭은 CP 이탈 때 **{primary.loc["TC-Z3","EVENT_AMP"]:.2f}℃**, CP 이탈이 없던 다른 때 **{primary.loc["TC-Z3","CONTROL_AMP"]:.2f}℃**다. OP-Z3도 각각 {primary.loc["OP-Z3","EVENT_AMP"]:.2f}%p·{primary.loc["OP-Z3","CONTROL_AMP"]:.2f}%p로 다른 때가 더 크다. 반면 Z4~Z6는 신호별 방향·크기가 다르므로 아래 각 패널을 함께 읽는다.')
summary_rows=[]
for r in relation_match_summary.to_dict('records'):
    nlarger=int(r['사건>대조']);total=int(r['비교 쌍'])
    summary_rows.append({'신호':r['신호'],'CP 이탈 때 더 크게 움직임':f'{total}번 중 {nlarger}번','다른 때가 크거나 같음':f'{total}번 중 {total-nlarger}번',
                         '두 시간대 최대폭 차이의 중앙값':r['짝 차이 중앙값'],'단위':'℃' if r['신호'].startswith('TC') else '%p'})
show_table(pd.DataFrame(summary_rows))
note('**이 표의 답:** CP 이탈이 있는 때에 더 큰 변화가 반복되는지 살펴본 결과다. 영향의 크기·고장 확률·정비 효과를 계산한 표가 아니다. 변화량이 더 큰 경우에도 CP→존, 존→CP, 공통 운전 조건의 영향을 나누어 검증해야 한다.')
# 모든 비교를 남기되 대표 파형만 본문에 둔다. 추가 PNG 파일은 만들지 않는다.
from IPython.utils.capture import capture_output
ordered=matched_events.assign(ABS=matched_events.DEV.abs()).sort_values('ABS',ascending=False)
for ident in ordered.ID:
    if ident==representative_id:continue
    with capture_output() as captured:
        fig=draw_cp_present_absent(ident)
        checked_emit(fig,'CP이탈유무_시간대별_파형',save=False,record=False)
    captured_details('다른 CP 구간도 같은 방식으로 보기: '+event_text(ident),captured)
coverage=relation_match_coverage.rename(columns={'ID':'CP 구간','LOT':'LOT','STATUS':'비교할 시간대 확보 결과'}).copy()
coverage['비교할 시간대 확보 결과']=coverage['비교할 시간대 확보 결과'].replace({'대조 가능':'서로 다른 두 시간대 비교 가능','조건을 만족하는 별도 대조 창 없음':'조건을 맞춘 다른 시간대 없음'})
table(coverage)
table(relation_matches.rename(columns={'ID':'CP 구간','LOT':'LOT','SENSOR':'신호','EVENT_START':'CP 이탈 때 시작','CONTROL_START':'이탈 없던 다른 때 시작',
    'EVENT_AMP':'CP 이탈 때 최대폭','CONTROL_AMP':'이탈 없던 다른 때 최대폭','BASE_TC456_MAX_GAP':'앞 기준 TC-Z4~Z6 최대 차이','CONTROL_CP_MAX':'다른 때 CP 최대 절대편차'}))
assert len(relation_pair_wave_checks)==int(matched.sum())==12
assert len(relation_pair_amplitude_checks)==12*8*2
assert {v['ID'] for v in relation_pair_wave_checks}==set(relation_matches.ID)
legacy=OUT_DIR/'05_관계검토/03_CP사건과_비사건_짝대조.png'
if legacy.is_file():
    assert legacy.resolve().parent==(OUT_DIR/'05_관계검토').resolve()
    assert (OUT_DIR/'05_관계검토/03_CP이탈_유무_실제파형.png').is_file()
    legacy.unlink()


**이 비교로 알 수 있는 것**

- **Z3의 큰 움직임은 CP 이탈 때에만 나타나지 않았다.** 가장 큰 CP 피크인 240069 (c-1)에서 TC-Z3 최대폭은 CP 이탈 때 2.70℃, 이탈이 없던 다른 때 7.19℃다. OP-Z3도 42.95%p 대 61.59%p로 다른 때가 더 크다. 따라서 “CP가 크게 튀면 Z3도 항상 더 크게 움직인다”는 설명과는 맞지 않는다.
- **Z4~Z6에는 함께 더 커지는 경향을 확인할 단서가 있다.** CP 이탈 때 온도 최대폭이 더 큰 경우는 TC-Z4 12번 중 10번, TC-Z5 9번, TC-Z6 8번이었다. 다만 두 시간대 최대폭 차이의 중앙값은 각각 약 0.200℃·0.243℃·0.200℃다. 빈도와 실제 크기를 함께 읽는다. 이를 CP의 인과적 영향량으로 부르지 않는다.
- **이 비교가 하지 못하는 판단:** CP가 원인이라는 확정, 반대로 존이 CP를 바꾸었다는 확정, 고장·정상 분류, 수리 효과 평가다. 설정·속도·부하·제어 경로가 다를 수 있고 각 최대값이 같은 순간에 나타난 것도 아니다. 파형에서 시각·방향을 확인한 뒤 공통 조건과 계측·제어 로그를 검토할 근거로 쓴다.
- **다른 시간대를 찾지 못한 CP 구간은 판단 보류다.** 기록 부족·전환 또는 범위 부족을 정상이나 관계없음으로 처리하지 않는다. 시작 후 4분 밖의 느린 반응도 이 최대폭으로 배제하지 않는다.


<a id="relation-maintenance"></a>
### 5.9 Q4. CP 두 채널과 정비 기록은 어디까지 연결되는가?

**질문:** CP-Z4와 CP-Z4M은 같은 사건에서 같이 변하는가? CP 사건은 정비 감지·착수·완료의 어느 쪽에 위치하는가?

**선정 근거:** CP-Z4M도 5.6과 같은 앞 기준·피크 창으로 계산한다. 왼쪽은 14사건 각각의 같은 시각 변화량, 오른쪽은 5.7의 접힌 보조 통계와 같은 차분 시차다. 표시를 위해 CP-Z4M 관측값에 10¹⁰을 곱하며 물리적 단위를 새로 정의하지 않는다.

정비 표는 각 CP 피크에서 감지·착수·완료 중 하나라도 ±72시간 안에 있는 WO를 모두 보여준다. 72시간은 대조 표의 범위이며 고장 전조 임계값이 아니다. 작업이 있으면 REMARK·교체 기록도 함께 보되, 가까운 WO를 원인으로 자동 연결하지 않는다.


In [ ]:
# 두 CP 기록의 관측 관계를 직접 그리고, 각 사건과 작업 시각·REMARK를 대조한다.
g=relation_event_delta.loc[relation_event_delta.SENSOR.eq('CP-Z4M')].copy()
curve=relation_lags.loc[relation_lags.PAIR.eq('CP')&relation_lags.SENSOR.eq('CP-Z4M')]
fig,cells=report_page('CP 두 기록의 관계 | 동반 변화와 계측 정의 확인을 나누어 본다',
    'CP-Z4·CP-Z4M의 물리량의 정의·측정/계산 경로는 미정의 상태다. 관측상의 동반 변화가 같은 센서·물리적 원인임을 뜻하지 않는다.',
    [dict(panels=[('CP 사건별 | 피크 부근의 CP-Z4M 변화',['CP-Z4M']),('CP-Z4 → CP-Z4M | 차분 시차',['CP-Z4M'])],height=3.0)],width=24,
    info=['왼쪽 각 점=CP 사건 · 가로=CP-Z4 피크 편차 · 세로=CP-Z4M 피크 ±15초 중앙값 - 사건 시작 전 5~1분 중앙값(×10^10).',
          '오른쪽=사건별 회색 곡선, LOT 동일 가중 중앙값, LOT 하나씩 제외한 범위. 정의 미확인 상태에서 독립 센서 검증·단위 환산으로 쓰지 않음.'])
ax,head,_=cells[0][0]
ax.scatter(g.CP_DEV,g.DELTA*1e10,s=38,color=sensor_color('CP-Z4M'),alpha=.8,label='유효 CP 사건')
ax.axhline(0,color='#aebac3',lw=.7);ax.axvline(0,color='#aebac3',lw=.7)
ax.set_xlabel('CP-Z4 · LOT 중앙값 대비 피크 편차 [무차원]');ax.set_ylabel('CP-Z4M · 앞 기준 대비 변화량 [무차원, 원값×10¹⁰]')
ax.margins(x=.2,y=.45);panel_legend(head,ax)
selected=g.loc[g.LOT.eq(240069)].sort_values('CP_DEV').iloc[-1]
ax.annotate(event_text(selected.ID)+' · 최대 CP 사건',xy=(selected.CP_DEV,selected.DELTA*1e10),xytext=(-6,8),textcoords='offset points',ha='right',va='bottom',fontsize=9.5,color='#477B94',bbox=dict(boxstyle='round,pad=.3',fc='white',ec='#477B94',lw=.8),arrowprops=dict(arrowstyle='->',lw=.8,color='#477B94'))
ax,head,_=cells[0][1]
for _,v in curve.groupby('ID'):ax.plot(v.LAG_SEC,v.R,color='#b8c2cc',alpha=.5,lw=.7)
med,lo,hi,count=lag_ensemble(curve)
ax.fill_between(REL_LAGS,lo,hi,color='#64748b',alpha=.15,lw=0)
ax.plot(REL_LAGS,med,color='#64748b',lw=1.8,label='LOT 동일 가중 중앙값')
ax.axhline(0,color='#aebac3',lw=.7);ax.axvline(0,color='#aebac3',ls=':',lw=.7)
ax.set(xlim=(-120,120),ylim=(-1.05,1.05),xlabel='시차 [초] · +는 CP-Z4 먼저',ylabel='차분 Spearman r');panel_legend(head,ax)
emit(fig,'05_관계검토/04_CP두채널_관측관계.png')
relation_work_rows=[]
for ev in relation_events.itertuples():
    near=cal_work.loc[cal_work[['DETC_DT','STRT_DT','END_DT']].sub(ev.PEAK).abs().min(axis=1).le(pd.Timedelta(hours=72))]
    if near.empty:
        relation_work_rows.append({'사건':ev.ID,'CP 피크':ev.PEAK,'WO':'±72h 연결 작업 없음','REMARK':'-','시간 관계':'해당 범위 비교 보류'})
    for w in near.itertuples():
        phase='착수 전' if ev.PEAK<w.STRT_DT else '작업 중' if ev.PEAK<=w.END_DT else '완료 뒤'
        relation_work_rows.append({'사건':ev.ID,'CP 피크':ev.PEAK,'WO':w.WO_NO,'유형':w.MNT_TYPE,
            '감지':w.DETC_DT,'착수':w.STRT_DT,'완료':w.END_DT,'CP→감지[h]':(w.DETC_DT-ev.PEAK).total_seconds()/3600,
            '완료→CP[분]':(ev.PEAK-w.END_DT).total_seconds()/60,'시간 관계':phase,
            'REMARK':remark_text(w),'교체 기록':' / '.join(parts_for(w.WO_NO)) or '-(미기재)'})
relation_work_links=pd.DataFrame(relation_work_rows)
show_table(relation_work_links.loc[relation_work_links.사건.str.startswith(('240069','240117')),
    ['사건','WO','유형','시간 관계','CP→감지[h]','완료→CP[분]','REMARK']])
table(relation_work_links)
# 7월 25일 교체는 ±72h 밖의 별도 후속 단서로 기존 30일 표를 유지한다.
show_table(pd.DataFrame([
 {'질문':'CP 두 채널은 같은 물리량인가?','현재 근거':'그림의 기록 변화·시차만 계산','보류·추가 정보':'측정/계산 정의·배선·변환식·보정·INS-006 대응'},
 {'질문':'240117의 CP가 해당 CBM을 미리 알렸는가?','현재 근거':'CBM 01854 완료 12:44 → CP 최저 13:04:50 (+20.83분)','보류·추가 정보':'해당 작업의 사전 경보로 해석 불가; 작업 대상·완료 점검·후속 조건 확인'},
 {'질문':'240069의 이후 교체가 CP 원인인가?','현재 근거':'07/08 절연 저하 BM, 07/25 가스 분석계 셀 손상 교체','보류·추가 정보':'동일 설비 후속 기록; CP 채널 연결·진단·보정 미확인, 조치 직전직후 같은 조건 관측 부족'},
 {'질문':'정기수리로 개선됐는가?','현재 근거':'전후 강종 다름, 수리 중 온도 0행','보류·추가 정보':'같은 강종·SV·속도·두께·부하에서 횟수·크기·지속시간을 재비교'},
 {'질문':'온도 변화가 품질에 영향을 주었는가?','현재 근거':'품질 결과·소재 이동 정렬 조건 없음','보류·추가 정보':'라인 속도·존 간 거리·소재 추적·품질 데이터로 다음 단계 검증'}]))

v=g.DELTA.dropna()
note(f'**CP-Z4M의 실제 관측:** 유효 {len(v)}사건에서 앞 기준 대비 증가 {int(v.gt(0).sum())}건·감소 {int(v.lt(0).sum())}건이다. 이는 관측값 변화의 집계이며 CP-Z4와 같은 물리량인지 확인한 결과가 아니다. 정비의 원인·효과 판단은 위 표의 시간 순서와 채널 정의 확인이 먼저다.')


**관찰 결과·반례·해석**

- 두 CP 기록의 같은 시각 변화량과 차분 시차는 각각 다른 질문이다. 일부 점이나 일부 시차에서 함께 움직여도 공통 계산식·같은 센서·고장 원인의 증거로 바로 바꾸지 않는다. 부호·시차가 사건마다 다른 경우에도 미정의 채널의 전체 관계가 없다고 단정하지 않는다.
- 정비 표의 시간 부호를 먼저 읽는다. `CP→감지[h]`가 양수면 CP 뒤에 감지가 기록됐고, `완료→CP[분]`가 양수면 완료 뒤 CP가 기록됐다. 감지 시각은 실제 고장 시작 시각과 같다고 가정하지 않는다.
- 240069의 7월 25일 가스 분석계 셀 손상 교체는 CP 최고에서 약 16.9일 뒤의 단서다. CP–INS-006 대응이 없어 인과를 제시할 수 없다. 그 뒤 다른 LOT 관측이 있어도 조치 직전·직후의 같은 조건 비교가 아니므로 수리 효과가 검증된 것은 아니다.
- 정상 제어 반응인지 점검 신호인지는 SV·제어 모드·운전 조건, 실제 작업 진단·보정, 같은 조건에서의 재현과 조치 후 변화로 구분한다. 현재 자료에서는 **관측 관계와 원인 가설의 근거·반례**까지 제시한다.


<a id="process-summary"></a>
## 06 공정별 시계열 특성 종합

동일한 임계값 하나로 모든 공정을 판정하지 않는다. 온도 추세·제어 출력·CP 펄스·주기 신호를 서로 다른 질문으로 정리한다.

In [ ]:
def stable_z_count(signals):
    return sum(int((zstats[k,sensor].Z.abs().ge(3)&states[k].eq('안정 참고')).sum()) for k in LOT_EVENTS for sensor in signals)
def gap_value(k,sensor):return float(level_gap.loc[level_gap.LOT.eq(k)&level_gap.SENSOR.eq(sensor),'GAP'].iloc[0])
show_table(pd.DataFrame([
 {'공정':'세정','주요 신호':'CLN-IN','관측 결과':f'전환 제외 |Z|≥3 {stable_z_count(["CLN-IN"])}개 신호·1분창','해석·확인':'LOT 내부 수준 변화와 반복 제어 형태를 대조. 강종 정상 한계로 사용하지 않음'},
 {'공정':'건조','주요 신호':'TC-Z1·TC-Z2 / OP-Z1·OP-Z2','관측 결과':f'전환 제외 온도 |Z|≥3 {stable_z_count(["TC-Z1","TC-Z2"])}개 신호·1분창','해석·확인':'온도와 대응 출력의 시각·복귀를 구분. 초기·종료 전환 문맥 확인'},
 {'공정':'가열 초입','주요 신호':'TC-Z3 / OP-Z3','관측 결과':'단발 온도 반응과 큰 출력 변동이 함께 관측됨','해석·확인':'반복 제어 응답 가설. SV·제어 로그 없이 정상 제어·센서 고장 확정 불가'},
 {'공정':'가열·균열','주요 신호':'TC-Z4–TC-Z6 / OP-Z4–OP-Z6','관측 결과':f'240061 OP-Z4 차이 {gap_value(240061,"OP-Z4"):+.2f}%p, TC-Z4 {gap_value(240061,"TC-Z4"):+.3f}℃','해석·확인':'전환 제외 후 분포 중심 차이. 설정·부하·속도·두께 확인'},
 {'공정':'CP 계측','주요 신호':'CP-Z4 / CP-Z4M','관측 결과':f'원본 CP {len(cp_real)}사건 / {cp_real.LOT.nunique()} LOT','해석·확인':'1초 펄스의 폭·반복·복귀를 확인. 두 CP 기록의 측정 정의를 같게 가정하지 않음'},
 {'공정':'과시효','주요 신호':'TC-Z7·TC-Z8 / OP-Z7·OP-Z8','관측 결과':f'240061 OP-Z8 차이 {gap_value(240061,"OP-Z8"):+.2f}%p; 다른 LOT의 온도 수준 차이도 관측','해석·확인':'동일 강종 분포로 확인. 어느 LOT이 정상인지 정할 독립 기준은 없음'},
 {'공정':'Z9·냉각','주요 신호':'TC-Z9 / TC-OUT1·TC-OUT2','관측 결과':'반복 파형 관측; 비주기 온도 Z 선별에서는 분리','해석·확인':'주기·위상·운전 조건을 따로 확인. 전 신호에 자기상관·시차 분석을 억지로 적용하지 않음'}]))
note('**표의 창 수는 고장 수가 아니다.** 여러 신호·시간창이 중첩될 수 있다. 지속·반복·복귀와 정비의 실제 시각으로 재검토한다.')

**공정 간 시간 관계의 한계:** 같은 시각에 기록된 가열존·과시효·냉각 값이 반드시 같은 소재 위치를 뜻하지 않는다. 품질과 연결하려면 라인 속도·존 간 거리로 소재 이동 시차를 정렬하고 품질 결과와 맞춰야 한다. 여기서는 동시 기록의 반응을 기술했으며 인과나 품질 영향을 검증하지 않았다.

<a id="conclusion"></a>
## 07 종합 판단 및 현장 적용 방안

| 분석 대상 | 확인된 사실 | 현재 판단 | 후속 점검·확보 기록 |
|---|---|---|---|
| 240069 | CP 7사건·시작 전후 2분 TC/OP 최대편차 방향 불일치; 큰 TC 변화에 CP 이탈이 없는 대조 사례 | 단일 TC 원인 설명 부족; 정비와의 인과 미확정 | CP-Z4·CP-Z4M 정의, INS-006 채널 연결, 07/08·07/25 진단·보정 및 제어 로그 |
| 240037 | CP 반복 3사건 | 짧은 원본 펄스와 동반 반응 확인 필요 | 동시 TC·OP, CP 계측 정의·수집 로그 |
| 240117 | CBM 완료 후 20.83분에 CP 최저 | 해당 CBM의 사전 경보 아님 | 작업 대상·완료 점검·후속 운전 조건·CP 로그 |
| 240061 | SPHC의 OP-Z4·OP-Z8 분포 중심 차이 | 운전 조건 차이 가능, 고장 미확정 | SV·라인 속도·부하·두께·보정·같은 조건의 반복 LOT |
| 계획수리 전후 | 공통 하강·승온과 남는 국소 반응 | 운전 전환 가설 우선, 수리 효과 미확정 | 운전 모드·재가동 기록·정비 대상 부품·완료 확인 |

**현장 적용 순서:** 먼저 기록·계측 정의와 시각 정합성을 확인하고, 사례별 계측·제어·운전 조건 점검으로 범위를 좁힌다. 추가 기록이 확보되면 같은 강종·같은 운전 조건에서 재현 여부와 조치 후 변화를 검증한다.

**현재 판단 불가:** 정상 기준 방향, 고장 정답, 수리의 인과 효과, 품질 영향·예측 성능. 제공 자료에 품질 결과와 소재 이동 정렬 조건이 없어 품질 영향은 후속 검증 가설이다.

**종합 메시지:** 14 LOT의 운전 전환, 반복적 순간 이탈, 강종별 운전 수준 차이와 정비 후 변화를 구분해 추가 점검 신호와 필요한 검증 항목을 도출했다.

**원인 해석을 제한하는 우선 정보:** CP-Z4·CP-Z4M의 정의와 INS-006 연결은 분석팀에서도 미확인이다. 현재 결론은 관측 관계까지이며 물리적 원인·고장 부품 판정은 보류한다. 정의가 확보되더라도 인과 판단에는 운전 조건·제어/계측 로그와 조치 전후의 비교가 추가로 필요하다.
**새 관계 분석을 반영한 판단:** CP 이탈이 없던 시간대에도 큰 Z3 변화가 관측된다. Z4~Z6는 작은 진폭이지만 이탈 때 변화폭이 더 큰 경우가 있어 후속 점검에서 제외하지 않는다. CP 하강 2사건에서 Z3 온도·출력 변화가 CP 최저점보다 앞서 보이고 Z4~Z6의 동반 방향도 나뉜다는 점을 근거로, CP 계측·계산 경로와 당시 제어 응답을 함께 확인한다. 이는 관측 관계의 근거이며 특정 존·부품이 CP 원인이라는 결론은 아니다.


In [ ]:
table(plan_compare)

# 새 관계 분석의 계산값을 종합 판단에 연결한다.
matched=relation_match_coverage.STATUS.eq('대조 가능')
show_table(relation_match_summary[['신호','비교 쌍','사건>대조','짝 차이 중앙값','해당 LOT 수']].rename(columns={'비교 쌍': '비교한 시간대 수', '사건>대조': 'CP 이탈 때 최대폭이 더 큰 횟수', '짝 차이 중앙값': '두 시간대 최대폭 차이 중앙값'}))
note(f'**관계 검토의 범위:** CP {len(relation_events)}사건/{relation_events.LOT.nunique()} LOT의 같은 시각 변화량, 사건별 CP 시차와 14 LOT OP–TC 시차를 계산했다. CP 이탈이 없던 같은 LOT의 다른 시간대는 {int(matched.sum())}번 확보했다. 사건 선별·겹치는 창·미측정 운전 조건이 있어 이 관측 비교를 인과·고장 확률로 해석하지 않는다.')


<a id="detail"></a>
## 08 부록 — 14 LOT 전체 검토 및 재현 코드

본문의 핵심 사례로 결론을 읽은 뒤 원본 근거를 확인하는 부분이다. 모든 LOT의 전체 22신호·모든 (a)(b)(c)·강종 대조·정비 시각 검토를 보존했다. **고정 구간 번호·시각·선정 이유가 전체 그림과 확대 그림에 반복되며, 확대 상단의 전체 신호가 현재 위치를 다시 보여 준다.**

**표시 기준:** CP 0.05 사건은 전부 표시하고, 45분 안의 사건을 한 확대 묶음으로 합친다. 온도 |Z|≥3 중 전환 후보 밖의 가장 큰 1구간을 추가한다. CLN-IN의 이탈은 별도로 표시한다. Z9·냉각 주기 파형을 온도 Z 선별에 넣지 않는다. 수리 전후 전환 후보와 관측 중 DETC_DT도 별도 표시한다. 같은 강종 수준 차이는 (L)로 표시하고 시간 펄스와 구분한다.

**정비 대조 기준:** 실제 작업·감지 교집합 WO를 우선한다. 교집합이 없으면 감지 또는 작업 구간이 가장 가까운 1 WO를 고른다. 전·후 관측은 착수 전 마지막 45분, 완료 후 첫 45분 중 **72시간 이내에 실제 기록된 범위**만 보여 준다. 작업 중은 실제 교집합을 쓴다. 관측이 없으면 해당 칸을 비워 이유를 표시한다. 시간상 가까움은 원인·효과의 증거가 아니다.
**확대 그림의 색·시간:** 전체 위치와 확대의 선은 같은 센서색이다. 음영·테두리는 구간 유형색이고, 위 음영은 확대 범위, 아래 음영은 실제 CP 기록 범위·Z 1분창·감지 시각 대조를 뜻한다. 확대 패널끼리 실제 시각·범위·눈금을 고정한다.


**LOT 바로가기:** [240001](#lot-240001) · [240024](#lot-240024) · [240037](#lot-240037) · [240038](#lot-240038) · [240044](#lot-240044) · [240058](#lot-240058) · [240061](#lot-240061) · [240069](#lot-240069) · [240091](#lot-240091) · [240108](#lot-240108) · [240114](#lot-240114) · [240117](#lot-240117) · [240121](#lot-240121) · [240133](#lot-240133)

In [ ]:
lot_execution=[]

<a id="lot-240001"></a>
### 8.1 LOT 240001 · SGCC · 01/02

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def focus_event(k):
    """표시 순서만 정한다. CP 내 최대 절대편차, Z 내 최대 절대점수이며 고장 순위가 아니다."""
    events=LOT_EVENTS[k]
    cp=[e for e in events if e['kind']=='CP']
    if cp:return max(cp,key=lambda e:abs(e['deviation']))
    zs=[e for e in events if e['kind']=='Z' and not e.get('transition_context')]
    if zs:return max(zs,key=lambda e:abs(e['score']))
    return next((e for e in events if e['kind']=='감지'),events[0])

def lot_cards(k):
    r=lot_meta.loc[k];e=focus_event(k);ws=lot_works(k);w=primary_work(k)
    evs=LOT_EVENTS[k];cp=sum(len(t['pulses']) for t in evs if t['kind']=='CP')
    if e['kind']=='CP':
        base=raw[k]['CP-Z4'].median();value=f"CP-Z4 {e['deviation']:+.3f} · {e['tag']}"
        facts=[f"{e['point']:%m/%d %H:%M:%S} · 원본 {base+e['deviation']:.4f}",f'LOT 중앙값 {base:.4f} 대비 · 전체 {cp}사건',
               '확인: 탐색 기준 ±0.05 이탈','고장·원인은 추가 확인 필요']
    elif e['kind']=='Z':
        value=f"{e['sensor']} Z={e['score']:+.2f} · {e['tag']}"
        facts=[f"{e['point']:%m/%d %H:%M:%S} 종료 1분창",'확인: 이전 30분 기준에서 |Z| ≥ 3','지속 변화·일시 제어 반응을 아래에서 대조','Z 초과는 고장 판정이 아님']
    else:
        value=e['title'];facts=[f"{e['point']:%m/%d %H:%M:%S}",'시각 기록과 다신호 원본 대조','감지 기록은 온도 자동 경보 정의가 아님']
    direct=len(linked_work(k));det=int(cal_work.DETC_DT.between(r.START,r.END).sum())
    worklines=[f'{w.MNT_TYPE} {w.WO_NO[-5:]} 감지 {w.DETC_DT:%m/%d %H:%M}',f'작업 {w.STRT_DT:%m/%d %H:%M}–{w.END_DT:%m/%d %H:%M}']
    for other in ws.loc[ws.WO_NO.ne(w.WO_NO)].itertuples():
        worklines.extend([f'{other.MNT_TYPE} {other.WO_NO[-5:]} · {remark_text(other)}',f'감지 {other.DETC_DT:%m/%d %H:%M} · 작업 {other.STRT_DT:%H:%M}–{other.END_DT:%H:%M}'])
    if k==240117:worklines.append(f"CP 최저: CBM 완료 후 {(e['point']-w.END_DT).total_seconds()/60:.2f}분")
    elif parts_for(w.WO_NO):worklines.append('교체: '+' / '.join(parts_for(w.WO_NO)))
    if not direct:worklines.append('이 LOT 관측 중 실제 작업 겹침 없음')
    c=context.set_index('LOT').loc[k]
    obs=[f'{r.START:%Y/%m/%d %H:%M}–{r.END:%m/%d %H:%M}',f'{r.MINUTES:.1f}분 · 22신호 · {len(raw[k]):,}행',
         f'작업 교집합 {direct} WO · 감지 {det}건',str(c.PLAN_RELATION) if c.PLAN_RELATION else '정기수리 경계 LOT 아님']
    return [dict(title='먼저 볼 근거 · '+EVENT_LABELS[e['kind']],value=value,lines=facts,color=e['color']),
            dict(title='정비 REMARK · '+('관측과 작업 겹침' if direct else '가까운 정비 기록'),
                 value=f'{w.MNT_TYPE} · {remark_text(w)}',lines=worklines,color='#7540A8',face='#f7f3fb'),
            dict(title='LOT 관측 정보 · 실제 시각',value=f'{k} · {r.GRADE}',lines=obs,color='#477B94')]

def event_line(k,e):return f"{event_text(event_id(k,e))} | {e['point']:%m/%d %H:%M:%S} | {event_reason(k,e)}"

def is_trigger(e,sensor):
    return sensor==e['sensor'] if e['kind'] in ['CP','Z'] else sensor in HEAT+[f'OP-Z{i}' for i in range(3,7)] if e['kind']=='전환' else False

def show_event_index(k):
    from IPython.display import HTML
    from html import escape
    rows=''.join('<tr><td><a href="#event-'+event_id(k,e)+'"><b>'+event_text(event_id(k,e))+'</b></a></td><td>'+escape(event_reason(k,e))+'</td><td>'+e['point'].strftime('%m/%d %H:%M:%S')+'</td></tr>' for e in LOT_EVENTS[k])
    display(HTML('<p><b>이 LOT에서 확인할 지점 — 같은 번호가 전체·확대·해석에 반복됩니다.</b></p><table><thead><tr><th>고정 번호</th><th>왜 보는가</th><th>기준 시각</th></tr></thead><tbody>'+rows+'</tbody></table>'))

def draw_lot_overview(k):
    evs=LOT_EVENTS[k];r=lot_meta.loc[k]
    labels=' / '.join(e['tag']+' '+('전환' if e['kind']=='전환' else '국소 변화(Z)' if e['kind']=='Z' else e['kind']) for e in evs)
    rows=[{'panels':[('관측·정비 시각과 확대 위치 | '+labels,[])],'height':.85}]
    for name,left,right in FLOW_ROWS:
        panels=[(name+' · 온도',left)] if right is None else [(name+' · '+('무차원' if name.startswith('CP') else '온도'),left),(name+' · '+('무차원' if name.startswith('CP') else '온도' if name.startswith('Z9') else '출력'),right)]
        rows.append({'panels':panels,'height':1.3})
    fig,cells=report_page(f'{k} · {r.GRADE} | ① LOT 전체 22신호',
        '진한 원·굵은 선·테두리=이 구간을 선택한 신호. 다른 공정의 옅은 음영은 같은 시각의 대조이며 이상 판정이 아니다.',rows,cards=lot_cards(k),info=[event_line(k,e) for e in evs],event_key=True)
    show_event_index(k)
    ax,head,_=cells[0][0];work_clock(ax,head,k)
    for e in evs:
        ax.axvspan(at(k,e['a']),at(k,e['b']),color=e['color'],alpha=.22 if e['kind']!='전환' else .12,lw=0)
        ax.text(at(k,(e['a']+e['b'])/2),-.4,e['tag'],ha='center',va='center',fontsize=10,fontweight='bold',color=e['color'])
    for row in cells[1:]:
        for ax,head,sensors in row:
            for s in sensors:report_signal(ax,k,s)
            selected=[]
            for e in evs:
                primary=any(is_trigger(e,sensor) for sensor in sensors)
                ax.axvspan(at(k,e['a']),at(k,e['b']),color=e['color'],alpha=.18 if primary else .035,lw=0)
                ax.axvline(e['point'],color=e['color'],lw=1.1 if primary else .55,ls=EVENT_LINES[e['kind']],alpha=1 if primary else .35)
                if primary:
                    selected.append(e)
                    for sensor in sensors:
                        if not is_trigger(e,sensor):continue
                        if e['kind']=='CP':
                            y=raw[k].set_index('MEAS_DT')[sensor]
                            for pulse in e['pulses'].itertuples():
                                q=y.loc[at(k,pulse.A).round('s'):at(k,pulse.B).round('s')]
                                ax.plot(q.index,q,color=e['color'],lw=2.4,zorder=6)
                            value=y.loc[e['point']];point=e['point']
                        elif e['kind']=='Z':value=zstats[k,sensor].loc[e['minute'],'X'];point=at(k,e['minute']+.5)
                        else:continue
                        ax.scatter(point,value,s=55,facecolors='white',edgecolors=e['color'],lw=2.1,zorder=8)
                        overview_trigger_checks.append((k,e['tag'],sensor))
            if selected:
                color=selected[0]['color']
                head.text(1,.98,'확인 '+', '.join(e['tag']+' '+('동반 전환' if e['kind']=='전환' else e['sensor']) for e in selected),ha='right',va='top',fontsize=11,fontweight='bold',color=color,transform=head.transAxes)
                for spine in ax.spines.values():spine.set_edgecolor(color);spine.set_linewidth(1.2)

            for w in linked_work(k).itertuples():
                for t,ls in [(w.STRT_DT,'--'),(w.END_DT,'-.')]:ax.axvline(t,color=COL[w.MNT_TYPE],lw=.8,ls=ls,alpha=.7)
            if 'CP-Z4' in sensors:
                base=raw[k]['CP-Z4'].median()
                for v in [base-.05,base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.75)
            ax.set_ylabel(display_unit(sensors[0]));clock_in_lot(ax,k);panel_legend(head,ax)
    fig._folded=f'{k} {r.GRADE}: 전체 22신호·LOT 정보·정비 REMARK 펼치기'
    emit(fig,f'03_상세/01_LOT전체/{k}_{r.GRADE}.png')
    note('**확대 위치:** '+' / '.join(f"{e['tag']} {at(k,e['a']):%m/%d %H:%M}–{at(k,e['b']):%H:%M}: {e['title']}" for e in evs))

draw_lot_overview(240001)

#### LOT 240001 · 먼저 볼 변화

**지속·반복 후보 10건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — TC-Z3 · 01/02 13:25~13:30**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **10.14**, 과거 중심 대비 **+6.79℃**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -72.9%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.37℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:48 (이동 Z 복귀보다 15분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z1 · 01/02 14:43~14:48**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.81**, 과거 중심 대비 **-11.1%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.46℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -10.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 14:54 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z3 · 01/02 06:48~06:49**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.83**, 과거 중심 대비 **-19.5%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.01℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -19.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 07:13 (이동 Z 복귀보다 21분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240001

- **관측 범위:** 01/02 03:36:00–01/02 15:04:30.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 595개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 10개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 25개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240001, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 2건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 01/02 14:43~14:48**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.81**, 과거 중심 대비 **-11.1%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.46℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -10.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 14:54 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 01/02 14:46~14:51**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.45**, 과거 중심 대비 **-8.89%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.09℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.78%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 14:57 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z1 · 01/02 11:04~11:07**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.32**, 과거 중심 대비 **-6.65%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.25℃(10/3유효창). 같은 전후 기준의 OP-Z1 변화는 -8.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 11:13 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 6건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240001-Z-OP-Z1-0047 · 01/02 04:23~04:26**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.36℃(10/3유효창). 같은 전후 기준의 OP-Z1 변화는 -9.68%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 04:31 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z2-0049 · 01/02 04:25~04:27**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.62℃(10/2유효창). 같은 전후 기준의 OP-Z2 변화는 -5.46%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 04:32 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z1-0448 · 01/02 11:04~11:07**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.25℃(10/3유효창). 같은 전후 기준의 OP-Z1 변화는 -8.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 11:13 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z1-0667 · 01/02 14:43~14:48**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.46℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -10.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 14:54 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z2-0670 · 01/02 14:46~14:51**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.09℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.78%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 14:57 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z1-0670 · 01/02 14:46~14:48**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -11.7%p(10/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.45℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 14:51. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| TC-Z1 | 689 | 664 | 2 | 25 | 5 | 20 | 0 |
| OP-Z1 | 689 | 664 | 11 | 25 | 5 | 20 | 0 |
| TC-Z2 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| OP-Z2 | 689 | 664 | 7 | 25 | 5 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 664창, 최대 절대 Z=2.74 / TC-Z2: 유효 664창, 최대 절대 Z=2.75. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 01/02 04:23:00–01/02 04:26:00 (끝 제외); 240001-Z-OP-Z1-0047 | 최대 절대 Z=4.47 (01/02 04:24:30, 창 중심); 1분 중앙값 62.540%, 과거 중심 72.082%, 차이 -9.542%p; SD=2.133; 실측 57/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 04:29:00 / 시작 고정 기준 3창 복귀 01/02 04:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z2 · 01/02 04:25:00–01/02 04:27:00 (끝 제외); 240001-Z-OP-Z2-0049 | 최대 절대 Z=3.39 (01/02 04:26:30, 창 중심); 1분 중앙값 12.648%, 과거 중심 19.424%, 차이 -6.777%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 04:30:00 / 시작 고정 기준 3창 복귀 01/02 04:32:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z1 · 01/02 11:04:00–01/02 11:07:00 (끝 제외); 240001-Z-OP-Z1-0448 | 최대 절대 Z=3.32 (01/02 11:04:30, 창 중심); 1분 중앙값 65.334%, 과거 중심 71.983%, 차이 -6.649%p; SD=2.000; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 11:10:00 / 시작 고정 기준 3창 복귀 01/02 11:13:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z1 · 01/02 14:43:00–01/02 14:48:00 (끝 제외); 240001-Z-OP-Z1-0667 | 최대 절대 Z=4.81 (01/02 14:45:30, 창 중심); 1분 중앙값 61.115%, 과거 중심 72.254%, 차이 -11.139%p; SD=2.315; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:51:00 / 시작 고정 기준 3창 복귀 01/02 14:54:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z2 · 01/02 14:46:00–01/02 14:51:00 (끝 제외); 240001-Z-OP-Z2-0670 | 최대 절대 Z=4.45 (01/02 14:47:30, 창 중심); 1분 중앙값 13.248%, 과거 중심 22.141%, 차이 -8.893%p; SD=2.000 (하한 적용); 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:54:00 / 시작 고정 기준 3창 복귀 01/02 14:57:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z1 · 01/02 14:46:00–01/02 14:48:00 (끝 제외); 240001-Z-TC-Z1-0670 | 최대 절대 Z=3.20 (01/02 14:47:30, 창 중심); 1분 중앙값 101.642℃, 과거 중심 100.041℃, 차이 +1.601℃; SD=0.500 (하한 적용); 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:51:00 / 시작 고정 기준 3창 복귀 01/02 14:51:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |

**동반 신호로 다시 볼 근거**

- **240001-Z-OP-Z1-0667:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.456 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240001-Z-OP-Z2-0670:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.09 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 01/02 03:36:00–01/02 03:56:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 01/02 06:36:00–01/02 06:37:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 01/02 06:54:00–01/02 06:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 01/02 07:09:00–01/02 07:10:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 01/02 14:25:00–01/02 14:26:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 01/02 15:04:00–01/02 15:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 28개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240001, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 6건 · 짧은 노랑 후보 31건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 주황 · 지속·반복 확인 — TC-Z3 · 01/02 13:25~13:30**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **10.14**, 과거 중심 대비 **+6.79℃**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -72.9%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.37℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:48 (이동 Z 복귀보다 15분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z4 · 01/02 07:23~07:26**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.06**, 과거 중심 대비 **-8.42%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.01℃(10/3유효창). 같은 전후 기준의 OP-Z4 변화는 -7.18%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 07:29. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z3 · 01/02 06:48~06:49**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.83**, 과거 중심 대비 **-19.5%p**.  
**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.01℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -19.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 07:13 (이동 Z 복귀보다 21분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 37건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240001-Z-OP-Z3-0034 · 01/02 04:10~04:11**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.19℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -22.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:26 (이동 Z 복귀보다 12분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0040 · 01/02 04:16~04:18**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +25.8%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.31℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:21. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0045 · 01/02 04:21~04:22**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.31℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -27.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:25. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0106 · 01/02 05:22~05:23**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.36℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -35.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:27 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0110 · 01/02 05:26~05:27**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +24.6%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.99℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:31 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0192 · 01/02 06:48~06:49**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.01℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -19.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:13 (이동 Z 복귀보다 21분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0195 · 01/02 06:51~06:53**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +22%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.05℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:58. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0204 · 01/02 07:00~07:01**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.81℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -29%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:04. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0207 · 01/02 07:03~07:06**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +34.6%p(9/3유효창). 같은 전후 기준의 TC-Z3 변화는 -6.49℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:13 (이동 Z 복귀보다 4분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0223 · 01/02 07:19~07:21**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.72℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -74.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:24. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0224 · 01/02 07:20~07:21**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -73.4%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +6.85℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:24. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z4-0227 · 01/02 07:23~07:26**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.01℃(10/3유효창). 같은 전후 기준의 OP-Z4 변화는 -7.18%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:29. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z4-0229 · 01/02 07:25~07:26**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -6.45%p(10/1유효창). 같은 전후 기준의 TC-Z4 변화는 +1.06℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:29. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0285 · 01/02 08:21~08:23**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.08℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -34.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:26. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0334 · 01/02 09:10~09:12**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.45℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -33.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:15. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0338 · 01/02 09:14~09:16**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +26.5%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -4.94℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:21 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0352 · 01/02 09:28~09:30**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.15℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -59.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:33. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0370 · 01/02 09:46~09:47**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.76℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -69%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:50. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z4-0375 · 01/02 09:51~09:52**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.12℃(10/1유효창). 같은 전후 기준의 OP-Z4 변화는 -6.32%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:59. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z4-0378 · 01/02 09:54~09:56**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +0.601℃(10/2유효창). 같은 전후 기준의 OP-Z4 변화는 -4.54%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:59. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z6-0383 · 01/02 09:59~10:00**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.01℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +5.93%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:03. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0414 · 01/02 10:30~10:32**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.38℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -61.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:36 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0415 · 01/02 10:31~10:33**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -55.7%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.94℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:36. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0458 · 01/02 11:14~11:16**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.74℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -35.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:24 (이동 Z 복귀보다 5분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0458 · 01/02 11:14~11:16**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -35.3%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +1.74℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:26 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0462 · 01/02 11:18~11:21**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +25.7%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 -4.58℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:25 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0537 · 01/02 12:33~12:34**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.96℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -52.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:38 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0538 · 01/02 12:34~12:35**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -47.9%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.13℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:45 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0541 · 01/02 12:37~12:39**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +27.9%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -5.28℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:44 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0589 · 01/02 13:25~13:29**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.91℃(10/4유효창). 같은 전후 기준의 OP-Z3 변화는 -72.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:44 (이동 Z 복귀보다 12분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0589 · 01/02 13:25~13:30**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -72.9%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.37℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:48 (이동 Z 복귀보다 15분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z4-0594 · 01/02 13:30~13:32**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.76℃(10/2유효창). 같은 전후 기준의 OP-Z4 변화는 -9.72%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:43 (이동 Z 복귀보다 8분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z4-0595 · 01/02 13:31~13:33**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -8.98%p(10/2유효창). 같은 전후 기준의 TC-Z4 변화는 +1.83℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:36. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0638 · 01/02 14:14~14:16**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.01℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -76.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:19. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-TC-Z3-0638 · 01/02 14:14~14:16**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -76.8%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +5.01℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:20 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z3-0674 · 01/02 14:50~14:51**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.19℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -34%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:54. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240001-Z-OP-Z4-0679 · 01/02 14:55~14:56**

**다른 LOT:** SGCC의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +0.558℃(10/1유효창). 같은 전후 기준의 OP-Z4 변화는 -4.29%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:59. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 689 | 664 | 28 | 25 | 5 | 20 | 0 |
| OP-Z3 | 689 | 664 | 26 | 25 | 5 | 20 | 0 |
| TC-Z4 | 689 | 664 | 3 | 25 | 5 | 20 | 0 |
| OP-Z4 | 689 | 664 | 9 | 25 | 5 | 20 | 0 |
| TC-Z5 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| OP-Z5 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| TC-Z6 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| OP-Z6 | 689 | 664 | 1 | 25 | 5 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z5: 유효 664창, 최대 절대 Z=1.64 / OP-Z5: 유효 664창, 최대 절대 Z=2.08 / TC-Z6: 유효 664창, 최대 절대 Z=2.72. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 01/02 04:10:00–01/02 04:11:00 (끝 제외); 240001-Z-OP-Z3-0034 | 최대 절대 Z=4.79 (01/02 04:10:30, 창 중심); 1분 중앙값 58.042%, 과거 중심 81.040%, 차이 -22.999%p; SD=4.797; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 04:14:00 / 시작 고정 기준 3창 복귀 01/02 04:26:00 | 고정 범위 복귀가 이동 Z 복귀보다 12분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 04:16:00–01/02 04:18:00 (끝 제외); 240001-Z-TC-Z3-0040 | 최대 절대 Z=3.93 (01/02 04:17:30, 창 중심); 1분 중앙값 858.050℃, 과거 중심 860.335℃, 차이 -2.285℃; SD=0.581; 실측 57/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 04:21:00 / 시작 고정 기준 3창 복귀 01/02 04:21:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 04:21:00–01/02 04:22:00 (끝 제외); 240001-Z-OP-Z3-0045 | 최대 절대 Z=3.27 (01/02 04:21:30, 창 중심); 1분 중앙값 45.174%, 과거 중심 78.918%, 차이 -33.743%p; SD=10.319; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 04:25:00 / 시작 고정 기준 3창 복귀 01/02 04:25:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 05:22:00–01/02 05:23:00 (끝 제외); 240001-Z-OP-Z3-0106 | 최대 절대 Z=4.21 (01/02 05:22:30, 창 중심); 1분 중앙값 44.667%, 과거 중심 81.038%, 차이 -36.371%p; SD=8.646; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 05:26:00 / 시작 고정 기준 3창 복귀 01/02 05:27:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 05:26:00–01/02 05:27:00 (끝 제외); 240001-Z-TC-Z3-0110 | 최대 절대 Z=3.57 (01/02 05:26:30, 창 중심); 1분 중앙값 857.785℃, 과거 중심 860.427℃, 차이 -2.642℃; SD=0.741; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 05:30:00 / 시작 고정 기준 3창 복귀 01/02 05:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 06:48:00–01/02 06:49:00 (끝 제외); 240001-Z-OP-Z3-0192 | 최대 절대 Z=3.83 (01/02 06:48:30, 창 중심); 1분 중앙값 62.923%, 과거 중심 82.444%, 차이 -19.521%p; SD=5.098; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 06:52:00 / 시작 고정 기준 3창 복귀 01/02 07:13:00 | 고정 범위 복귀가 이동 Z 복귀보다 21분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 06:51:00–01/02 06:53:00 (끝 제외); 240001-Z-TC-Z3-0195 | 최대 절대 Z=3.91 (01/02 06:52:30, 창 중심); 1분 중앙값 858.174℃, 과거 중심 860.165℃, 차이 -1.991℃; SD=0.509; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 06:58:00 / 시작 고정 기준 3창 복귀 01/02 06:58:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 07:00:00–01/02 07:01:00 (끝 제외); 240001-Z-OP-Z3-0204 | 최대 절대 Z=3.90 (01/02 07:00:30, 창 중심); 1분 중앙값 37.538%, 과거 중심 81.203%, 차이 -43.665%p; SD=11.186; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 07:04:00 / 시작 고정 기준 3창 복귀 01/02 07:04:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 07:03:00–01/02 07:06:00 (끝 제외); 240001-Z-TC-Z3-0207 | 최대 절대 Z=4.63 (01/02 07:03:30, 창 중심); 1분 중앙값 855.892℃, 과거 중심 860.444℃, 차이 -4.552℃; SD=0.983; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 07:09:00 / 시작 고정 기준 3창 복귀 01/02 07:13:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 07:19:00–01/02 07:21:00 (끝 제외); 240001-Z-OP-Z3-0223 | 최대 절대 Z=4.38 (01/02 07:19:30, 창 중심); 1분 중앙값 3.554%, 과거 중심 81.694%, 차이 -78.140%p; SD=17.830; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 07:24:00 / 시작 고정 기준 3창 복귀 01/02 07:24:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 07:20:00–01/02 07:21:00 (끝 제외); 240001-Z-TC-Z3-0224 | 최대 절대 Z=3.02 (01/02 07:20:30, 창 중심); 1분 중앙값 866.579℃, 과거 중심 859.551℃, 차이 +7.028℃; SD=2.329; 실측 55/60개·과거 28/30창 | 지속·반복 후보: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 07:24:00 / 시작 고정 기준 3창 복귀 01/02 07:24:00 | 짧아도 최근 30분에 3회 시작했다. 재발 간격과 동일 운전 조건을 추적; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z4 · 01/02 07:23:00–01/02 07:26:00 (끝 제외); 240001-Z-OP-Z4-0227 | 최대 절대 Z=4.06 (01/02 07:24:30, 창 중심); 1분 중앙값 41.221%, 과거 중심 49.640%, 차이 -8.418%p; SD=2.072; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 07:29:00 / 시작 고정 기준 3창 복귀 01/02 07:29:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z4 · 01/02 07:25:00–01/02 07:26:00 (끝 제외); 240001-Z-TC-Z4-0229 | 최대 절대 Z=3.17 (01/02 07:25:30, 창 중심); 1분 중앙값 861.832℃, 과거 중심 860.233℃, 차이 +1.599℃; SD=0.504; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 07:29:00 / 시작 고정 기준 3창 복귀 01/02 07:29:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 08:21:00–01/02 08:23:00 (끝 제외); 240001-Z-OP-Z3-0285 | 최대 절대 Z=3.18 (01/02 08:21:30, 창 중심); 1분 중앙값 49.603%, 과거 중심 82.379%, 차이 -32.776%p; SD=10.295; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 08:26:00 / 시작 고정 기준 3창 복귀 01/02 08:26:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 09:10:00–01/02 09:12:00 (끝 제외); 240001-Z-OP-Z3-0334 | 최대 절대 Z=3.54 (01/02 09:11:30, 창 중심); 1분 중앙값 37.401%, 과거 중심 77.449%, 차이 -40.047%p; SD=11.315; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 09:15:00 / 시작 고정 기준 3창 복귀 01/02 09:15:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 09:14:00–01/02 09:16:00 (끝 제외); 240001-Z-TC-Z3-0338 | 최대 절대 Z=5.02 (01/02 09:14:30, 창 중심); 1분 중앙값 856.034℃, 과거 중심 860.559℃, 차이 -4.525℃; SD=0.901; 실측 56/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 09:19:00 / 시작 고정 기준 3창 복귀 01/02 09:21:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 09:28:00–01/02 09:30:00 (끝 제외); 240001-Z-OP-Z3-0352 | 최대 절대 Z=3.54 (01/02 09:29:30, 창 중심); 1분 중앙값 8.735%, 과거 중심 78.232%, 차이 -69.497%p; SD=19.616; 실측 57/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 09:33:00 / 시작 고정 기준 3창 복귀 01/02 09:33:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 09:46:00–01/02 09:47:00 (끝 제외); 240001-Z-OP-Z3-0370 | 최대 절대 Z=3.24 (01/02 09:46:30, 창 중심); 1분 중앙값 9.872%, 과거 중심 81.235%, 차이 -71.362%p; SD=22.031; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 09:50:00 / 시작 고정 기준 3창 복귀 01/02 09:50:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z4 · 01/02 09:51:00–01/02 09:52:00 (끝 제외); 240001-Z-OP-Z4-0375 | 최대 절대 Z=3.15 (01/02 09:51:30, 창 중심); 1분 중앙값 42.406%, 과거 중심 48.696%, 차이 -6.290%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 09:59:00 / 시작 고정 기준 3창 복귀 01/02 09:59:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z4 · 01/02 09:54:00–01/02 09:56:00 (끝 제외); 240001-Z-OP-Z4-0378 | 최대 절대 Z=3.53 (01/02 09:55:30, 창 중심); 1분 중앙값 40.952%, 과거 중심 48.539%, 차이 -7.587%p; SD=2.148; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 09:59:00 / 시작 고정 기준 3창 복귀 01/02 09:59:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z6 · 01/02 09:59:00–01/02 10:00:00 (끝 제외); 240001-Z-OP-Z6-0383 | 최대 절대 Z=3.49 (01/02 09:59:30, 창 중심); 1분 중앙값 75.639%, 과거 중심 68.651%, 차이 +6.988%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 10:03:00 / 시작 고정 기준 3창 복귀 01/02 10:03:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 10:30:00–01/02 10:32:00 (끝 제외); 240001-Z-OP-Z3-0414 | 최대 절대 Z=5.49 (01/02 10:31:30, 창 중심); 1분 중앙값 4.040%, 과거 중심 78.613%, 차이 -74.573%p; SD=13.586; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 10:35:00 / 시작 고정 기준 3창 복귀 01/02 10:36:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 10:31:00–01/02 10:33:00 (끝 제외); 240001-Z-TC-Z3-0415 | 최대 절대 Z=4.67 (01/02 10:31:30, 창 중심); 1분 중앙값 864.817℃, 과거 중심 859.465℃, 차이 +5.352℃; SD=1.146; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 10:36:00 / 시작 고정 기준 3창 복귀 01/02 10:36:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 11:14:00–01/02 11:16:00 (끝 제외); 240001-Z-OP-Z3-0458 | 최대 절대 Z=5.37 (01/02 11:14:30, 창 중심); 1분 중앙값 43.792%, 과거 중심 78.842%, 차이 -35.050%p; SD=6.530; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 11:19:00 / 시작 고정 기준 3창 복귀 01/02 11:24:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 11:14:00–01/02 11:16:00 (끝 제외); 240001-Z-TC-Z3-0458 | 최대 절대 Z=4.70 (01/02 11:15:30, 창 중심); 1분 중앙값 862.743℃, 과거 중심 860.309℃, 차이 +2.434℃; SD=0.518; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 11:24:00 / 시작 고정 기준 3창 복귀 01/02 11:26:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 11:18:00–01/02 11:21:00 (끝 제외); 240001-Z-TC-Z3-0462 | 최대 절대 Z=5.10 (01/02 11:18:30, 창 중심); 1분 중앙값 856.404℃, 과거 중심 860.363℃, 차이 -3.959℃; SD=0.776; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 11:24:00 / 시작 고정 기준 3창 복귀 01/02 11:25:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 12:33:00–01/02 12:34:00 (끝 제외); 240001-Z-OP-Z3-0537 | 최대 절대 Z=3.87 (01/02 12:33:30, 창 중심); 1분 중앙값 25.167%, 과거 중심 76.400%, 차이 -51.232%p; SD=13.230; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 12:37:00 / 시작 고정 기준 3창 복귀 01/02 12:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 12:34:00–01/02 12:35:00 (끝 제외); 240001-Z-TC-Z3-0538 | 최대 절대 Z=3.09 (01/02 12:34:30, 창 중심); 1분 중앙값 863.643℃, 과거 중심 860.326℃, 차이 +3.317℃; SD=1.073; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 12:42:00 / 시작 고정 기준 3창 복귀 01/02 12:45:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 12:37:00–01/02 12:39:00 (끝 제외); 240001-Z-TC-Z3-0541 | 최대 절대 Z=3.46 (01/02 12:38:30, 창 중심); 1분 중앙값 855.210℃, 과거 중심 860.326℃, 차이 -5.115℃; SD=1.478; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 12:42:00 / 시작 고정 기준 3창 복귀 01/02 12:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 13:25:00–01/02 13:29:00 (끝 제외); 240001-Z-OP-Z3-0589 | 최대 절대 Z=12.84 (01/02 13:25:30, 창 중심); 1분 중앙값 13.108%, 과거 중심 79.818%, 차이 -66.710%p; SD=5.196; 실측 57/60개·과거 30/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 13:32:00 / 시작 고정 기준 3창 복귀 01/02 13:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 12분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 13:25:00–01/02 13:30:00 (끝 제외); 240001-Z-TC-Z3-0589 | 최대 절대 Z=10.14 (01/02 13:26:30, 창 중심); 1분 중앙값 866.922℃, 과거 중심 860.133℃, 차이 +6.789℃; SD=0.669; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 13:33:00 / 시작 고정 기준 3창 복귀 01/02 13:48:00 | 고정 범위 복귀가 이동 Z 복귀보다 15분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z4 · 01/02 13:30:00–01/02 13:32:00 (끝 제외); 240001-Z-OP-Z4-0594 | 최대 절대 Z=5.09 (01/02 13:31:30, 창 중심); 1분 중앙값 39.132%, 과거 중심 49.316%, 차이 -10.183%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 13:35:00 / 시작 고정 기준 3창 복귀 01/02 13:43:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z4 · 01/02 13:31:00–01/02 13:33:00 (끝 제외); 240001-Z-TC-Z4-0595 | 최대 절대 Z=3.84 (01/02 13:31:30, 창 중심); 1분 중앙값 862.130℃, 과거 중심 860.212℃, 차이 +1.918℃; SD=0.500 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 13:36:00 / 시작 고정 기준 3창 복귀 01/02 13:36:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 14:14:00–01/02 14:16:00 (끝 제외); 240001-Z-OP-Z3-0638 | 최대 절대 Z=4.28 (01/02 14:14:30, 창 중심); 1분 중앙값 3.847%, 과거 중심 78.646%, 차이 -74.800%p; SD=17.483; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:19:00 / 시작 고정 기준 3창 복귀 01/02 14:19:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| TC-Z3 · 01/02 14:14:00–01/02 14:16:00 (끝 제외); 240001-Z-TC-Z3-0638 | 최대 절대 Z=4.51 (01/02 14:15:30, 창 중심); 1분 중앙값 865.582℃, 과거 중심 859.419℃, 차이 +6.163℃; SD=1.367; 실측 54/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:19:00 / 시작 고정 기준 3창 복귀 01/02 14:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z3 · 01/02 14:50:00–01/02 14:51:00 (끝 제외); 240001-Z-OP-Z3-0674 | 최대 절대 Z=3.09 (01/02 14:50:30, 창 중심); 1분 중앙값 45.648%, 과거 중심 81.683%, 차이 -36.036%p; SD=11.648; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:54:00 / 시작 고정 기준 3창 복귀 01/02 14:54:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |
| OP-Z4 · 01/02 14:55:00–01/02 14:56:00 (끝 제외); 240001-Z-OP-Z4-0679 | 최대 절대 Z=3.06 (01/02 14:55:30, 창 중심); 1분 중앙값 43.591%, 과거 중심 49.720%, 차이 -6.129%p; SD=2.000 (하한 적용); 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 01/02 14:59:00 / 시작 고정 기준 3창 복귀 01/02 14:59:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: 해당 피크±72시간 내 같은 설비 기록 시각 없음 |

**동반 신호로 다시 볼 근거**

- **240001-Z-TC-Z3-0589:** OP-Z3 사건창 중앙값−직전10분 중앙값 -72.85 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240001-Z-OP-Z3-0589:** TC-Z3 사건창 중앙값−직전10분 중앙값 +7.909 (10/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 01/02 03:36:00–01/02 03:56:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 과거 유효 최소 0창 <20 |
| 01/02 06:36:00–01/02 06:37:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 01/02 06:54:00–01/02 06:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 01/02 07:09:00–01/02 07:10:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 01/02 14:25:00–01/02 14:26:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 01/02 15:04:00–01/02 15:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 28개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240001, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| OP-Z7 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| TC-Z8 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |
| OP-Z8 | 689 | 664 | 0 | 25 | 5 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 664창, 최대 절대 Z=0.53 / OP-Z7: 유효 664창, 최대 절대 Z=2.40 / TC-Z8: 유효 664창, 최대 절대 Z=0.43 / OP-Z8: 유효 664창, 최대 절대 Z=0.94. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 01/02 03:36:00–01/02 03:56:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 01/02 06:36:00–01/02 06:37:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 01/02 06:54:00–01/02 06:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 01/02 07:09:00–01/02 07:10:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 01/02 14:25:00–01/02 14:26:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 01/02 15:04:00–01/02 15:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 28개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240001)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 40402 | 5 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.017606 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 40402 | 5 | 원본 중앙값 1.14791e-10, 최저 1.13316e-10, 최고 1.15941e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 40402 | 5 | 원본 중앙값 330.296, 최저 318.118, 최고 341.161; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 40402 | 5 | 원본 중앙값 281.33, 최저 266.754, 최고 294.609; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 40402 | 5 | 원본 중앙값 277.311, 최저 266.768, 최고 287.985; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**대조할 기록 부족:** LOT 전후 72시간 안에 같은 설비 WO가 없다. 정비가 없었다거나 설비가 정상이라는 뜻은 아니다.

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def event_relation(k,e):
    r=lot_meta.loc[k];w=primary_work(k);t=e['point']
    dt=lambda x:f'{x:%m/%d %H:%M}'
    if w.STRT_DT<=t<=w.END_DT:
        return f'{w.MNT_TYPE} {w.WO_NO[-5:]} 작업 중 관측 · REMARK: {remark_text(w)}'
    if t>w.END_DT:
        return f'{w.MNT_TYPE} {w.WO_NO[-5:]} 완료 뒤 {(t-w.END_DT).total_seconds()/60:.2f}분 · REMARK: {remark_text(w)}'
    return f'{w.MNT_TYPE} {w.WO_NO[-5:]} 감지 {dt(w.DETC_DT)} / 착수 {dt(w.STRT_DT)} · REMARK: {remark_text(w)} · '+('작업 겹침 있음' if len(linked_work(k)) else '작업 겹침 없음')

def response_stats(k,e,s):
    """동시 움직임을 기술한다. 정상 범위나 다른 공정의 이상 판정에 사용하지 않는다."""
    t=e['point'];g=raw[k].set_index('MEAS_DT')[s]
    before=g.loc[t-pd.Timedelta(minutes=10):t-pd.Timedelta(minutes=5)].dropna()
    during=g.loc[t-pd.Timedelta(minutes=1):t+pd.Timedelta(minutes=1)].dropna()
    after=g.loc[t+pd.Timedelta(minutes=5):t+pd.Timedelta(minutes=10)].dropna()
    center=before.median();dev=during-center
    peak=dev.loc[dev.abs().idxmax()] if len(dev) and pd.notna(center) else np.nan
    return dict(sensor=s,before=center,after=after.median(),change=after.median()-center,
                peak=peak,n_before=len(before),n_focus=len(during),n_after=len(after))

def response_caption(k,e,sensors):
    metrics=[response_stats(k,e,s) for s in sensors]
    unit='℃' if sensors[0] in TC else '%p' if sensors[0] in OP else '무차원'
    factor=1e10 if sensors==['CP-Z4M'] else 1
    if sensors==['CP-Z4M']:unit='무차원 · 눈금×10⁻¹⁰'
    show=lambda v:f'{v*factor:+.3f}' if pd.notna(v) else '자료 없음'
    return '표시 시각 ±1분 최대편차: '+' / '.join(f"{d['sensor']} {show(d['peak'])}" for d in metrics)+f' {unit}'

def interval_main(e):
    if e['kind']=='전환':return 3,[('TC-Z3–TC-Z6',HEAT),('OP-Z3–OP-Z6',[f'OP-Z{i}' for i in range(3,7)])]
    if e['kind'] in ['CP','감지']:return 4,[('CP-Z4',['CP-Z4']),('CP-Z4M',['CP-Z4M'])]
    zone=degree_zone(e['sensor'])
    if not zone:return 0,[(e['sensor'],[e['sensor']])]
    index=1 if zone<=2 else 2 if zone==3 else 3 if zone<=6 else 5
    name,left,right=FLOW_ROWS[index]
    return index,[(name+' · 온도',left),(name+' · 출력',right)]


def detail_clock(ax,a,b):
    # 모든 확대 패널에 같은 실제 시각·범위·눈금을 적용한다. 양 끝도 눈금으로 표시한다.
    span=(b-a).total_seconds()
    seconds=next((step for step in [1,2,5,10,15,20,30,60,120,300,600,900,1200,1800,3600,7200] if span/step<=4),7200)
    inner=pd.date_range(a.ceil(f'{seconds}s'),b.floor(f'{seconds}s'),freq=f'{seconds}s')
    ticks=[a]+[t for t in inner if (t-a).total_seconds()>=span*.10 and (b-t).total_seconds()>=span*.10]+[b]
    ticks=list(dict.fromkeys(ticks));values=mdates.date2num(ticks)
    seconds_visible=any(t.second or t.microsecond or t.nanosecond for t in ticks)
    fmt=('%H:%M:%S' if seconds_visible else '%H:%M')
    if a.date()!=b.date():fmt='%m/%d\n'+fmt
    ax.set_xticks(values);ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
    ax.set_xlim(a,b);ax.tick_params(axis='x',labelsize=10,pad=5)
    return values


def draw_lot_zoom(k):
    r=lot_meta.loc[k];pictures=[];naxes=0;interpretations=[]
    for e in LOT_EVENTS[k]:
        a,b=at(k,e['a']).round('s'),at(k,e['b']).round('s');main_index,main=interval_main(e)
        flow=[i for i in range(6) if i!=main_index and (i in [2,3,4,5] or e['kind']=='전환')]
        labels=[]
        for label,sigs in main:
            role='감지 시각 대조' if e['kind']=='감지' else '보조 채널 대조' if sigs==['CP-Z4M'] else '선정 근거'
            labels.append((e['tag']+' '+role+' | '+label,sigs))
        captions=[e['title']+(' · 점선=LOT 중앙값 ±0.05' if sigs==['CP-Z4'] else '') for _,sigs in main]
        if e['kind']=='Z':
            start,end=event_focus_ranges(k,e)[0]
            captions=[text+f' · 음영={start:%H:%M}–{end:%H:%M} 1분창' for text in captions]
        rows=[{'panels':[(f'전체 LOT → 아래 확대창 | {event_id(k,e)} · {a:%H:%M:%S}–{b:%H:%M:%S}',[])],'height':.9},
              {'panels':labels,'height':2.1,'captions':captions}]
        for i in flow:
            name,left,right=FLOW_ROWS[i];panels=[(name+' | '+', '.join(left),left)]
            if right:panels.append((name+' | '+', '.join(right),right))
            rows.append({'panels':panels,'height':1.3,'captions':[response_caption(k,e,sigs) for _,sigs in panels]})
        meaning={'Z':'아래 주황 음영=선정된 실제 1분창 · 마름모=그 창의 중앙값(원본 순간값과 구분).',
                 'CP':'아래 빨강 음영=CP 원본 이탈 기록 범위 · 원=해당 사건의 실제 피크.',
                 '감지':'아래 보라 음영=감지 시각 ±30초 대조 · 세로선=DETC_DT. CP 이상을 탐지했다는 뜻이 아님.',
                 '전환':'아래 회색 음영=전환 참고 구간. 실제 운전 모드·정상 여부는 별도 확인.'}[e['kind']]
        fig,cells=report_page(f'{event_id(k,e)} · {r.GRADE} | {EVENT_LABELS[e["kind"]]} · 구간 상세',
            f'{a:%Y/%m/%d %H:%M:%S}–{b:%m/%d %H:%M:%S} · 위 음영은 아래 그래프의 확대 범위. 아래 모든 패널의 x축은 같은 실제 시각이다.',
            rows,width=20,info=[event_line(k,e),'정비 대조: '+event_relation(k,e),
                              '선=센서색(위·아래 동일) · 음영·테두리=구간 유형색. '+meaning],event_key=True)
        ax,head,_=cells[0][0];draw_event_locator(ax,head,k,e)
        bodies=[]
        for index,row in enumerate(cells[1:],1):
            for ax,head,sensors in row:
                bodies.append(ax)
                can_primary=(e['kind'] in ['CP','Z'] and e['sensor'] in sensors)
                if can_primary and len(sensors)==1:plot_event_trace(ax,head,k,e,a,b,large=True)
                else:
                    for sensor in sensors:report_signal(ax,k,sensor,a,b,fine=True)
                    shade_event_focus(ax,k,e)
                    if 'CP-Z4' in sensors:
                        base=raw[k]['CP-Z4'].median()
                        for v in [base-.05,base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.8)
                    ax.set_ylabel(display_unit(sensors[0]));zoom_clock(ax,a,b)
                    if can_primary:z_window_marker(ax,k,e,annotate=True)
                    panel_legend(head,ax)
                if index==1:
                    head.texts[0].set_color(e['color'])
                    for spine in ax.spines.values():spine.set_edgecolor(e['color']);spine.set_linewidth(1.1)
                detail_clock(ax,a,b)
                if index==len(cells)-1:
                    ax.set_xlabel(f'{a:%Y/%m/%d} 실제 측정 시각' if a.date()==b.date() else '실제 측정 날짜·시각')
                if sensors==['TC-Z3'] and e['kind']=='CP':
                    y=raw_window(k,'TC-Z3',a,b).dropna();point=y.idxmax()
                    near_note(ax,point,y.loc[point],f'{point:%H:%M:%S} TC-Z3 최고 {y.loc[point]:.2f}℃')
                if sensors==['OP-Z3'] and e['kind']=='CP':
                    y=raw_window(k,'OP-Z3',a,b).dropna();point=y.idxmin()
                    near_note(ax,point,y.loc[point],f'{point:%H:%M:%S} OP-Z3 최저 {y.loc[point]:.2f}%')
        expected_limits=mdates.date2num([a,b]);ticks=bodies[0].get_xticks()
        for ax in bodies:
            assert np.allclose(ax.get_xlim(),expected_limits,atol=1e-10,rtol=0),event_id(k,e)
            assert np.array_equal(ax.get_xticks(),ticks),event_id(k,e)
        focus=event_focus_ranges(k,e)
        if e['kind']=='Z':assert focus[0][1]==e['point'] and focus[0][1]-focus[0][0]==pd.Timedelta(minutes=1)
        zoom_alignment_checks.append(dict(LOT=k,TAG=e['tag'],START=a,END=b,N_PANELS=len(bodies),N_TICKS=len(ticks),COLOR=e['color'],FOCUS=focus))
        naxes+=len(fig.axes);fig._silent=True
        pictures.append(checked_emit(fig,f'{k}_{e["tag"]}',save=False,record=False))
        interpretations.append(event_result_markdown(k,e))
        zoom_presentations.append(dict(LOT=k,TAG=e['tag'],KIND=e['kind'],PRIMARY_SENSOR=e['sensor']))
    save_lot_sheet(pictures,f'03_상세/02_구간확대/{k}_전체구간_공정순서.png',naxes)
    from io import BytesIO
    from base64 import b64encode
    from html import escape
    from PIL import Image
    from IPython.display import HTML
    import mistune
    pieces=[]
    for e,array,text in zip(LOT_EVENTS[k],pictures,interpretations):
        data=BytesIO();Image.fromarray(array).save(data,format='PNG')
        pieces.append('<section id="event-'+event_id(k,e)+'"><div style="position:sticky;top:0;background:white;border-left:5px solid '+e['color']+';padding:8px;z-index:2"><b>'+escape(event_line(k,e))+'</b></div><img style="width:100%;height:auto" src="data:image/png;base64,'+b64encode(data.getvalue()).decode()+'">'+mistune.html(text)+'</section>')
    display(HTML('<details><summary><b>'+str(k)+' 전체 '+str(len(pictures))+'개 구간: (a)(b)(c) 각각의 동일 크기 상세·해석 펼치기</b></summary>'+''.join(pieces)+'</details>'))
    table(iqr_intervals.loc[iqr_intervals.LOT.eq(k)])


def event_result_markdown(k,e):
    a,b=at(k,e['a']),at(k,e['b']);t=e['point'];lines=[f'**{event_text(event_id(k,e))}에서 무엇을 확인했는가?**','']
    if e['kind']=='CP':
        base=raw[k]['CP-Z4'].median();after=raw_window(k,'CP-Z4',t+pd.Timedelta(minutes=5),t+pd.Timedelta(minutes=10)).median()-base
        facts=' / '.join(f'{r.PEAK:%H:%M:%S} {r.DEV:+.4f}' for r in e['pulses'].itertuples())
        lines += [f'- **선정 근거:** CP 원본 {len(e["pulses"])}사건. 시각·LOT 중앙값 편차: {facts}. 탐색 경계 ±0.05를 넘은 기록이다.',
                  f'- **복귀·지속:** 가장 큰 사건 후 5~10분 CP 중앙값 편차는 {after:+.4f}. '+('이 창의 중앙값은 경계 안으로 복귀했다.' if abs(after)<.05 else '이 창의 중앙값은 경계 밖에 남았다.')+' 모든 순간값의 정상 또는 고장 부재를 뜻하지 않는다.']
    elif e['kind']=='Z':
        d=zstats[k,e['sensor']].loc[e['minute']]
        lines += ['- **시각 표시:** 음영은 종료 시각 직전의 실제 1분창이며, 마름모는 그 창 중앙에 놓은 1분 중앙값이다.', f'- **선정 근거:** {e["sensor"]}의 {t:%H:%M} 종료 1분 중앙값 {d.X:.3f}, 과거 중앙값 {d.MEDIAN:.3f}, SD {d.SD:.3f} → Z={d.Z:+.2f}. 과거 기준 ±3 SD를 벗어난 창이다.',
                  '- **확인할 가설:** 해당 TC의 변화와 대응 OP의 저하·증가·복귀를 같은 시각에 대조한다. 반복 제어 응답인지 지속 수준 변화인지는 파형과 아래 전후 수치를 함께 본다.']
    elif e['kind']=='전환':lines += ['- **선정 근거:** 수리 전후의 여러 TC·OP가 함께 하강 또는 승온하는 구간이다. 한 센서만 튀는지와 구분하기 위해 온도·출력·CP를 모두 대조했다.', '- **판단:** 일정과 다신호의 함께 움직임은 운전 전환 가설과 부합한다. 실제 모드·SV·정지 명령을 확인하지 못해 정상 정지·재가동을 확정한 것은 아니다.']
    else:lines += [f'- **선정 근거:** {t:%H:%M}는 정비 DETC_DT 기록이다. 자동 온도 이상 경보의 정의는 없다. 그 시각에 실제 TC·OP·CP 변화가 있는지 비교한다.']
    details=[]
    for sensor in ['TC-Z3','OP-Z3','TC-Z4','TC-Z5','TC-Z6','TC-Z7','TC-Z8']:
        d=response_stats(k,e,sensor);unit='℃' if sensor in TC else '%p'
        if pd.notna(d['peak']):details.append(f"{sensor} {d['peak']:+.2f}{unit}")
    lines += ['- **다른 공정의 동반 반응:** 표시 시각 ±1분의 기준 대비 최대 편차는 '+' / '.join(details)+'. 기준은 전 10~5분 중앙값이며 정상 한계가 아니다.',
              '- **정비 REMARK·시각:** '+event_relation(k,e)+'.']
    if k==240117 and e['kind']=='CP':lines.append('- **가설 판정:** 마모량 초과 CBM 완료 후 20.83분에 CP 최저점이 발생했다. 이 CBM의 사전 경보라는 설명과 시간 순서가 맞지 않는다. 정비 대상과 CP 이탈 원인이 같은지는 미확인이다.')
    if k==240069 and e['kind']=='CP':lines.append('- **가설 판정:** CP 사건은 이후 BM 감지보다 먼저지만 작업 뒤 관측이 없다. REMARK “1호기 재발, 절연 저하”의 원인 신호이거나 수리로 개선됐다고 입증할 수 없다.')
    overlap=iqr_intervals.loc[iqr_intervals.LOT.eq(k)&iqr_intervals.START.lt(b)&iqr_intervals.END.gt(a)]
    lines.append(f'- **IQR 대조:** 이 시간 범위와 겹치는 신호별 IQR 후보 구간은 {len(overlap)}개다. 서로 겹치는 신호·창을 합친 고장 사건 수가 아니다. 전환 제외·기록 수·LOT별 분포를 적용한 시간표는 아래에 제공한다.')
    return '\n'.join(lines)

draw_lot_zoom(240001)

**240001 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def event_evidence(k):
    from IPython.display import HTML
    from html import escape
    data=raw[k].set_index('MEAS_DT');pieces=[]
    for e in LOT_EVENTS[k]:
        t=e['point'];pieces.append('<h4>'+escape(e['tag']+' '+e['title'])+'</h4>')
        if e['kind']=='CP':
            base=data['CP-Z4'].median()
            frame=pd.DataFrame([{'피크':f"({e['tag'][1]}-{number})",'확인 시각':p.PEAK,'CP 원본':base+p.DEV,'LOT 중앙값':base,'편차':p.DEV,'사건 폭[초]':(p.B-p.A)*60} for number,p in enumerate(e['pulses'].itertuples(),1)])
            pieces.append(frame.to_html(index=False,float_format=lambda v:f'{v:.4f}'))
        elif e['kind']=='Z':
            d=zstats[k,e['sensor']].loc[e['minute']]
            frame=pd.DataFrame([{'신호':e['sensor'],'1분창 종료':t,'현재 중앙값':d.X,'이전 중앙값':d.MEDIAN,'SD(하한 적용)':d.SD,'Z':d.Z}])
            pieces.append(frame.to_html(index=False,float_format=lambda v:f'{v:.4f}'))
        zone=degree_zone(e['sensor'])
        sensors=SIGNALS  # 대조 그림에서 생략한 신호·나머지 구간도 22신호를 모두 수치로 확인한다.
        windows=[('표시 전 10~5분',t-pd.Timedelta(minutes=10),t-pd.Timedelta(minutes=5)),('표시 전후 5분',t-pd.Timedelta(minutes=5),t+pd.Timedelta(minutes=5)),('표시 후 5~10분',t+pd.Timedelta(minutes=5),t+pd.Timedelta(minutes=10))]
        evidence=[]
        for s in sensors:
            rec={'신호':s,'단위':display_unit(s)}
            for label,a,b in windows:
                y=data.loc[(data.index>=a)&(data.index<b),s].dropna()
                rec[label+' 중앙값']=y.median()*(1e10 if s=='CP-Z4M' else 1);rec[label+' 기록수']=len(y)
            evidence.append(rec)
        pieces.append(pd.DataFrame(evidence).to_html(index=False,float_format=lambda v:f'{v:.4f}'))
    display(HTML('<details><summary><b>구간별 선정 수치와 전·주변·후 원본 중앙값 펼치기</b></summary><p>각 창의 유효 기록수도 표시한다. 값이 없는 창은 판정 보류이며, 이 표는 정비 효과 검증표가 아니다.</p>'+''.join(pieces)+'</details>'))

event_evidence(240001)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def phase_sensors(k):
    if k in [240091,240114,240121]:return [['TC-Z7','TC-Z8'],['OP-Z7','OP-Z8'],['CP-Z4']]
    evs=LOT_EVENTS[k];z=next((degree_zone(e['sensor']) for e in evs if e['kind']=='Z'),3) or 3
    return [[f'TC-Z{z}'],[f'OP-Z{z}'],['CP-Z4']]

def draw_lot_maintenance(k):
    w=primary_work(k);phases=LOT_PHASES[k].copy();ws=lot_works(k);r=lot_meta.loc[k];sensors=phase_sensors(k)
    if k not in [240069,240117,240133]:
        if k in [240058,240061,240114]:
            anchor='repair-june' if k in [240058,240061] else 'repair-november'
            note(f'**수리 일정 대조:** [공통 정기수리 일정·전후 원본](#{anchor})에서 이 LOT을 확인한다. 이 LOT 관측 중 실제 작업·감지 교집합은 없다. 전환 형태를 정비 발생 또는 고장으로 표시하지 않는다.')
        else:
            note(f'**정비와의 관계:** 관측 중 실제 작업·감지 교집합이 없다. 가장 가까운 {w.MNT_TYPE} {w.WO_NO} 감지 {w.DETC_DT:%m/%d %H:%M}, 작업 {w.STRT_DT:%m/%d %H:%M}–{w.END_DT:%m/%d %H:%M}. 근접했다는 이유만으로 연결하지 않는다. 비교 가능한 전후가 없어 빈 그림은 생성하지 않는다.')
        show_table(pd.DataFrame([{'WO':t.WO_NO,'유형':t.MNT_TYPE,'감지':t.DETC_DT,'착수':t.STRT_DT,'완료':t.END_DT,'내용':remark_text(t)} for t in ws.itertuples()]))
        return
    if k in [240069,240133]:phases['착수 전']=raw[k].copy()
    plotted={key:g for key,g in phases.items() if len(g)}
    labels={}
    for phase,g in phases.items():
        if len(g):
            keys=g.LOT_NO.unique();labels[phase]=' / '.join(f'{int(key)} {lot_meta.loc[int(key),"GRADE"]}' for key in keys)
        else:labels[phase]='근접 관측 없음'
    rows=[{'panels':[('LOT 관측창·실제 WO·REMARK',[])],'height':1.5}]
    for sigs in sensors:
        rows.append({'panels':[(f'{phase} · {labels[phase]}',sigs) for phase in plotted],'height':1.4})
    fig,cells=report_page(f'{k} | ④ 정비 감지·착수·완료 전후 · {w.MNT_TYPE} {w.WO_NO[-5:]}',
        '실제 관측이 있는 단계만 비교한다. 작업 행 위에는 REMARK, 파형 부근에는 실제 특이점과 시각을 표시한다.',rows,width=22,event_key=True,info=[f'기준 WO {w.WO_NO} · REMARK: {remark_text(w)} · 실제 작업 {w.STRT_DT:%m/%d %H:%M}–{w.END_DT:%m/%d %H:%M}'])
    ax,head,_=cells[0][0];work_clock(ax,head,k,expanded=True);add_work_remarks(ax,ws)
    for i,sigs in enumerate(sensors,1):
        populated=[]
        for (phase,g),(ax,head,_) in zip(plotted.items(),cells[i]):
            if not len(g):
                ax.axis('off');ax.text(.5,.5,'실제 관측 없음\n작업 경계 ±72시간 안에\n비교할 센서 기록이 없음',transform=ax.transAxes,ha='center',va='center',fontsize=11,color=GREY,linespacing=1.7)
                continue
            for kk,gg in g.groupby('LOT_NO'):
                gg=gg.set_index('MEAS_DT').sort_index()
                for s in sigs:
                    y=gg[s].asfreq('s')
                    ax.plot(y.index,y.to_numpy()*(1e10 if s=='CP-Z4M' else 1),color=sensor_color(s),lw=1.1,label=s)
            for e in LOT_EVENTS[k]:
                if g.MEAS_DT.min()<=e['point']<=g.MEAS_DT.max():ax.axvline(e['point'],color=e['color'],ls=EVENT_LINES[e['kind']],lw=1.15)
            clock_ticks(ax,g.MEAS_DT.min(),g.MEAS_DT.max());ax.set_ylabel(display_unit(sigs[0]));panel_legend(head,ax);populated.append(ax)
        if populated:
            low=min(ax.get_ylim()[0] for ax in populated);high=max(ax.get_ylim()[1] for ax in populated)
            for ax in populated:ax.set_ylim(low-(high-low)*.5,high+(high-low)*.5)
    for row,sigs in zip(cells[1:],sensors):
        for (phase,g),(ax,head,_) in zip(plotted.items(),row):
            if not len(g):continue
            events=cp_real.loc[cp_real.LOT.eq(k)&cp_real.PEAK.between(g.MEAS_DT.min(),g.MEAS_DT.max())]
            if 'CP-Z4' in sigs and len(events):
                ev=events.loc[events.DEV.abs().idxmax()];value=raw[k]['CP-Z4'].median()+ev.DEV
                near_note(ax,ev.PEAK,value,f'{ev.PEAK:%H:%M:%S} CP {ev.DEV:+.3f}',EVENT_COLORS['CP'])
            elif k==240133 and 'CP-Z4' in sigs and g.MEAS_DT.min()<=w.DETC_DT<=g.MEAS_DT.max():
                value=raw_window(k,'CP-Z4',w.DETC_DT,w.DETC_DT).iloc[0]
                near_note(ax,w.DETC_DT,value,f'{w.DETC_DT:%H:%M} 감지\n{remark_text(w)}',EVENT_COLORS['감지'])
    if k!=240117:fig._folded=f'{k}: REMARK·실제 작업 시각과 관측 가능한 전후 확인 펼치기'
    emit(fig,f'03_상세/04_정비대조/{k}_WO전중후.png')
    show_table(pd.DataFrame([{'WO':t.WO_NO,'유형':t.MNT_TYPE,'감지 DETC':t.DETC_DT,'착수 STRT':t.STRT_DT,'완료 END':t.END_DT,'정지[분]':t.DOWN_MIN,'REMARK':remark_text(t),'교체 기록':' / '.join(parts_for(t.WO_NO)) or '없음'} for t in ws.itertuples()]))
    records=[];tc,op=sensors[0][0],sensors[1][0]
    for phase,g in phases.items():
        records.append({'구분':phase,'관측 LOT·강종':labels[phase],'관측 시작':g.MEAS_DT.min(),'관측 끝':g.MEAS_DT.max(),'원본 행':len(g),
                        f'{tc} 중앙값[℃]':g[tc].median(),f'{op} 중앙값[%]':g[op].median(),'CP 중앙값':g['CP-Z4'].median(),'CP 최저':g['CP-Z4'].min(),'CP 최고':g['CP-Z4'].max()})
    show_table(pd.DataFrame(records).set_index('구분').T.reset_index().rename(columns={'index':'비교 항목'}))
    direct=cal_work.loc[cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START)]
    det=cal_work.loc[cal_work.DETC_DT.between(r.START,r.END)]
    if len(direct):
        overlap=sum(b-a for a,b in work_ranges[k]);fact=f'실제 작업 {len(direct)} WO가 이 LOT과 {overlap:.1f}분(합집합) 겹친다.'
    elif len(det):fact=f'감지 {len(det)}건은 관측 중이지만 실제 작업 교집합은 없다. 착수는 관측 종료 뒤다.'
    else:fact=f'이 LOT에는 실제 작업·감지의 교집합이 없다. 선택 WO와 가장 가까운 기록 간격은 {wo_distance(w,r)/3600:.1f}시간이다.'
    available=[phase for phase,g in phases.items() if len(g)]
    if len(phases['착수 전']) and len(phases['완료 후']):
        before,after=phases['착수 전'],phases['완료 후'];dtc=after[tc].median()-before[tc].median();dop=after[op].median()-before[op].median()
        grades={lot_meta.loc[int(key),'GRADE'] for key in pd.concat([before,after]).LOT_NO.unique()}
        difference=f'완료 후-착수 전 중앙값은 {tc} {dtc:+.2f}℃, {op} {dop:+.2f}%p다. '+('전후 강종이 달라 수리 효과로 비교할 수 없다.' if len(grades)>1 else '같은 강종이어도 설정·운전 상태가 같다는 증거는 없어 수리 효과를 확정하지 않는다.')
    else:difference=('관측 가능한 단계는 '+', '.join(available)+'뿐이다. ' if available else '작업 경계 72시간 안의 전·중·후 관측이 모두 없다. ')+'자료가 없는 단계를 가정해 회복·개선을 판정할 수 없다.'
    note('**정비 전후 확인:** '+fact+' **관측된 양상:** '+difference)
    if k==240117:
        note('**주의:** CBM 착수 전 45분에도 TBM 01859가 진행 중이다. 정비가 없는 정상 기준과의 비교가 아니다.')
        cbm=ws.loc[ws.WO_NO.eq('WO-2024-01854')].iloc[0];e=cp_real.loc[cp_real.LOT.eq(k)].iloc[0]
        note(f'**이 LOT의 핵심:** CP 최저점 {e.PEAK:%H:%M:%S}은 CBM 완료 {cbm.END_DT:%H:%M} 후 {(e.PEAK-cbm.END_DT).total_seconds()/60:.2f}분이다. 완료 후 패널의 하강을 확인할 수 있으나, 해당 CBM을 유발한 사전 경보는 아니다.')
    if k==240069:note('**이 LOT의 핵심:** 반복 CP는 이후 BM 감지보다 앞서지만 BM 작업 뒤 관측이 없다. REMARK의 절연 저하와 CP의 원인 연결, 조치 효과는 확인되지 않는다.')
    if k==240133:note('**이 LOT의 핵심:** DETC 15:54는 관측 중, 실제 작업 16:54부터는 관측 종료 16:44 뒤다. 셀 교체 후 CP·온도는 제공되지 않아 효과 비교가 불가능하다.')

def lot_conclusion(k):
    evs=LOT_EVENTS[k];cp=sum(len(e['pulses']) for e in evs if e['kind']=='CP');z=[e for e in evs if e['kind']=='Z'];tr=[e for e in evs if e['kind']=='전환']
    lines=[]
    if cp:lines.append(f'CP {cp}개 사건의 원본 이탈 확인')
    if z:lines.append('국소 기준을 벗어난 '+', '.join(e['sensor'] for e in z)+' 반응 확인')
    if tr:lines.append('수리 전후 하강·승온은 운전 전환 가설을 우선 검토')
    if k in LEVEL_TARGETS:lines.append('같은 강종 수준 차이 확인, 정상 기준 방향·운전 조건 추가 확인')
    if not lines:lines.append('현재 CP·온도 Z 기준으로 추가 고장 근거를 선정하지 않음')
    note('**이 LOT의 판정:** '+' / '.join(lines)+'. 고장 확정이나 정상 보증으로 쓰지 않는다. 정비 관련 판단은 바로 위 WO 교집합·전후 관측 유무를 따른다.')

draw_lot_maintenance(240001)
lot_conclusion(240001)
lot_execution.append({'LOT':240001,'확대 구간':len(LOT_EVENTS[240001]),'강종 대조':240001 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240024"></a>
### 8.2 LOT 240024 · SPFH590 · 03/07

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240024)

#### LOT 240024 · 먼저 볼 변화

**지속·반복 후보 6건 / CP 원본 후보 1건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 03/08 04:59:35**

**관측:** 원본 편차 **-0.0664무차원**, 126점·첫/끝 간격 135초. 1초 단발 여부: 아님.  
**다른 LOT:** 같은 강종 LOT 240108의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 03/08 05:01:33. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00378 감지 2일 10시간 10분 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — OP-Z1 · 03/08 06:09~06:14**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.59**, 과거 중심 대비 **-10.4%p**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 4건(연속 1~3개 1분창, 절대 편차 7.57~14%p)이 있다. 이 구간의 5창·10.4%p는 비교 관측의 상한을 넘는다(길이 5>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.58℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -12.4%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:27 (이동 Z 복귀보다 10분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z2 · 03/08 06:11~06:16**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.46**, 과거 중심 대비 **-9.16%p**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 3건(연속 2~3개 1분창, 절대 편차 8.59~11%p)이 있다. 이 구간의 5창·9.16%p는 비교 관측의 상한을 넘는다(길이 5>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.16℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:27 (이동 Z 복귀보다 8분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240024

- **관측 범위:** 03/07 21:22:00–03/08 06:29:48.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 467개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 6개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 48개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 1건 중 확인된 1초 단발 0건, 복귀 펄스 후보 1건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240024, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 2건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 03/08 06:09~06:14**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.59**, 과거 중심 대비 **-10.4%p**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 4건(연속 1~3개 1분창, 절대 편차 7.57~14%p)이 있다. 이 구간의 5창·10.4%p는 비교 관측의 상한을 넘는다(길이 5>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.58℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -12.4%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:27 (이동 Z 복귀보다 10분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 03/08 06:11~06:16**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.46**, 과거 중심 대비 **-9.16%p**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 3건(연속 2~3개 1분창, 절대 편차 8.59~11%p)이 있다. 이 구간의 5창·9.16%p는 비교 관측의 상한을 넘는다(길이 5>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.16℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:27 (이동 Z 복귀보다 8분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z1 · 03/08 06:11~06:13**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **3.28**, 과거 중심 대비 **+1.64℃**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 1.6~1.6℃)이 있다. 이 구간의 2창·1.64℃는 비교 관측의 상한을 넘는다(절대 편차 1.64>1.6℃, 길이 2>1창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -11.9%p(10/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.5℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:17 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 3건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240024-Z-OP-Z1-0527 · 03/08 06:09~06:14**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 4건(연속 1~3개 1분창, 절대 편차 7.57~14%p)이 있다. 이 구간의 5창·10.4%p는 비교 관측의 상한을 넘는다(길이 5>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.58℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -12.4%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 06:27 (이동 Z 복귀보다 10분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 1건/500유효창 (100창당 0.20건) | 10.41~10.41 | 5~5창 | 72.476 |
| 240108 | 4건/421유효창 (100창당 0.95건) | 7.573~14.02 | 1~3창 | 71.137 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z2-0529 · 03/08 06:11~06:16**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 3건(연속 2~3개 1분창, 절대 편차 8.59~11%p)이 있다. 이 구간의 5창·9.16%p는 비교 관측의 상한을 넘는다(길이 5>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.16℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 06:27 (이동 Z 복귀보다 8분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 1건/500유효창 (100창당 0.20건) | 9.162~9.162 | 5~5창 | 22.248 |
| 240108 | 3건/421유효창 (100창당 0.71건) | 8.589~11.01 | 2~3창 | 21.025 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z1-0529 · 03/08 06:11~06:13**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 1.6~1.6℃)이 있다. 이 구간의 2창·1.64℃는 비교 관측의 상한을 넘는다(절대 편차 1.64>1.6℃, 길이 2>1창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -11.9%p(10/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.5℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 06:17 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 1건/500유효창 (100창당 0.20건) | 1.638~1.638 | 2~2창 | 100.02 |
| 240108 | 1건/421유효창 (100창당 0.24건) | 1.601~1.601 | 1~1창 | 99.999 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| TC-Z1 | 548 | 500 | 2 | 48 | 28 | 20 | 0 |
| OP-Z1 | 548 | 500 | 5 | 48 | 28 | 20 | 0 |
| TC-Z2 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| OP-Z2 | 548 | 500 | 5 | 48 | 28 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 500창, 최대 절대 Z=2.69 / TC-Z2: 유효 500창, 최대 절대 Z=2.65. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 03/08 06:09:00–03/08 06:14:00 (끝 제외); 240024-Z-OP-Z1-0527 | 최대 절대 Z=4.59 (03/08 06:10:30, 창 중심); 1분 중앙값 62.001%, 과거 중심 72.407%, 차이 -10.406%p; SD=2.265; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 06:17:00 / 시작 고정 기준 3창 복귀 03/08 06:27:00 | 고정 범위 복귀가 이동 Z 복귀보다 10분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3419.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 03/08 06:11:00–03/08 06:16:00 (끝 제외); 240024-Z-OP-Z2-0529 | 최대 절대 Z=4.46 (03/08 06:12:30, 창 중심); 1분 중앙값 13.664%, 과거 중심 22.826%, 차이 -9.162%p; SD=2.056; 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 06:19:00 / 시작 고정 기준 3창 복귀 03/08 06:27:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3417.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 03/08 06:11:00–03/08 06:13:00 (끝 제외); 240024-Z-TC-Z1-0529 | 최대 절대 Z=3.28 (03/08 06:12:30, 창 중심); 1분 중앙값 101.699℃, 과거 중심 100.061℃, 차이 +1.638℃; SD=0.500 (하한 적용); 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 06:16:00 / 시작 고정 기준 3창 복귀 03/08 06:17:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3417.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240024-Z-OP-Z1-0527:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.582 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240024-Z-OP-Z2-0529:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.159 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 03/07 21:22:00–03/07 21:42:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 03/07 21:46:00–03/07 21:47:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 03/07 21:55:00–03/07 21:56:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/07 22:35:00–03/07 22:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 03/07 22:45:00–03/07 22:46:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/07 22:59:00–03/07 23:00:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 03/07 23:08:00–03/07 23:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 03/07 23:26:00–03/07 23:27:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/07 23:31:00–03/07 23:32:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/07 23:55:00–03/07 23:56:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 03/08 00:04:00–03/08 00:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 00:35:00–03/08 00:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 00:44:00–03/08 00:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 03/08 01:15:00–03/08 01:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 03/08 01:24:00–03/08 01:25:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 03/08 01:44:00–03/08 01:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 02:23:00–03/08 02:24:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 03/08 03:15:00–03/08 03:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 03:24:00–03/08 03:25:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 03:30:00–03/08 03:31:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 03/08 03:49:00–03/08 03:50:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 03/08 04:45:00–03/08 04:46:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 05:28:00–03/08 05:29:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 05:32:00–03/08 05:33:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 03/08 05:54:00–03/08 05:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 03/08 06:22:00–03/08 06:24:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 03/08 06:28:00–03/08 06:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240024, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 13건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 03/08 04:59:35**

**관측:** 원본 편차 **-0.0664무차원**, 126점·첫/끝 간격 135초. 1초 단발 여부: 아님.  
**다른 LOT:** 같은 강종 LOT 240108의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 03/08 05:01:33. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00378 감지 2일 10시간 10분 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 03/08 04:51~04:55**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.76**, 과거 중심 대비 **+6.21℃**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 1.85~9.85℃)이 있다. 이 구간의 4창·6.21℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -66.6%p(9/4유효창). 같은 전후 기준의 TC-Z3 변화는 +9.18℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 05:02 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z6 · 03/08 05:03~05:06**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.80**, 과거 중심 대비 **+7.6%p**.  
**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 4건(연속 1~2개 1분창, 절대 편차 6.75~8.83%p)이 있다. 이 구간의 3창·7.6%p는 비교 관측의 상한을 넘는다(길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.24℃(10/3유효창). 같은 전후 기준의 OP-Z6 변화는 +6.76%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 05:11 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 18건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240024-Z-OP-Z3-0068 · 03/07 22:30~22:31**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 1창·24.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.995℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -25.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 22:34. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0101 · 03/07 23:03~23:04**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 1창·23.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.18℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -21.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:07. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0105 · 03/07 23:07~23:08**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 2.11~5.55℃)이 있다. 이 구간의 1창·1.66℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +14.6%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -1.7℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:12. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |
| 240108 | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0178 · 03/08 00:20~00:21**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 2.11~5.55℃)이 있다. 이 구간의 1창·1.53℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +16.6%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -1.59℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:24. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |
| 240108 | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0200 · 03/08 00:42~00:44**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 2창·52.4%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.88℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -42.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:48. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0201 · 03/08 00:43~00:44**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 1.85~9.85℃)이 있다. 이 구간의 1창·2.47℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -53%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.37℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:48. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |
| 240108 | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0209 · 03/08 00:51~00:53**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 2.11~5.55℃)이 있다. 이 구간의 2창·3.38℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +23%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 -3.8℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:57 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |
| 240108 | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0213 · 03/08 00:55~00:56**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 1창·45.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.12℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -56.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:59. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0220 · 03/08 01:02~01:04**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 2창·78.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.44℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -79%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 01:07. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0303 · 03/08 02:25~02:26**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 2.11~5.55℃)이 있다. 이 구간의 1창·2.22℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +18.8%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.48℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:29. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |
| 240108 | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0328 · 03/08 02:50~02:52**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 2창·35.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.71℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -35.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:55. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0333 · 03/08 02:55~02:56**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 2.11~5.55℃)이 있다. 이 구간의 1창·3.38℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +16.4%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -3.71℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:59. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |
| 240108 | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0407 · 03/08 04:09~04:10**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 1창·29.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.1℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -29.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:14 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0408 · 03/08 04:10~04:11**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 1.85~9.85℃)이 있다. 이 구간의 1창·1.93℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -27.6%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +1.97℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:14. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |
| 240108 | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z3-0448 · 03/08 04:50~04:53**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 10건(연속 1~4개 1분창, 절대 편차 15.7~82.1%p)이 있다. 이 구간의 3창·61.4%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.24℃(9/3유효창). 같은 전후 기준의 OP-Z3 변화는 -66.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:02 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |
| 240108 | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-TC-Z3-0449 · 03/08 04:51~04:55**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 1.85~9.85℃)이 있다. 이 구간의 4창·6.21℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -66.6%p(9/4유효창). 같은 전후 기준의 TC-Z3 변화는 +9.18℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:02 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |
| 240108 | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-CP-01 · 03/08 04:59:35**

**다른 LOT:** 같은 강종 LOT 240108의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 03/08 05:01:33. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00378 감지 2일 10시간 10분 전에 CP 원본 피크.

강종 SPFH590·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 1건/31729실측점 (1만점당 0.32건) | 0.06636~0.06636 | 135~135초 간격 | 0.44997 |
| 240108 | 0건/26982실측점 (1만점당 0.00건) | — | — | 0.45006 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240024-Z-OP-Z6-0461 · 03/08 05:03~05:06**

**다른 LOT:** 동종 LOT 240108에도 같은 방향 후보 4건(연속 1~2개 1분창, 절대 편차 6.75~8.83%p)이 있다. 이 구간의 3창·7.6%p는 비교 관측의 상한을 넘는다(길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.24℃(10/3유효창). 같은 전후 기준의 OP-Z6 변화는 +6.76%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:11 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 1건/500유효창 (100창당 0.20건) | 7.596~7.596 | 3~3창 | 73.831 |
| 240108 | 4건/421유효창 (100창당 0.95건) | 6.755~8.832 | 1~2창 | 69.353 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 548 | 500 | 12 | 48 | 28 | 20 | 0 |
| OP-Z3 | 548 | 500 | 13 | 48 | 28 | 20 | 0 |
| TC-Z4 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| OP-Z4 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| TC-Z5 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| OP-Z5 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| TC-Z6 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| OP-Z6 | 548 | 500 | 3 | 48 | 28 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 1건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 500창, 최대 절대 Z=1.60 / OP-Z4: 유효 500창, 최대 절대 Z=2.22 / TC-Z5: 유효 500창, 최대 절대 Z=1.43 / OP-Z5: 유효 500창, 최대 절대 Z=2.32 / TC-Z6: 유효 500창, 최대 절대 Z=2.85. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 03/07 22:30:00–03/07 22:31:00 (끝 제외); 240024-Z-OP-Z3-0068 | 최대 절대 Z=3.30 (03/07 22:30:30, 창 중심); 1분 중앙값 64.007%, 과거 중심 88.867%, 차이 -24.860%p; SD=7.524; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/07 22:34:00 / 시작 고정 기준 3창 복귀 03/07 22:34:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3879.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/07 23:03:00–03/07 23:04:00 (끝 제외); 240024-Z-OP-Z3-0101 | 최대 절대 Z=3.28 (03/07 23:03:30, 창 중심); 1분 중앙값 64.116%, 과거 중심 87.578%, 차이 -23.462%p; SD=7.163; 실측 57/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/07 23:07:00 / 시작 고정 기준 3창 복귀 03/07 23:07:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3846.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/07 23:07:00–03/07 23:08:00 (끝 제외); 240024-Z-TC-Z3-0105 | 최대 절대 Z=3.25 (03/07 23:07:30, 창 중심); 1분 중앙값 859.160℃, 과거 중심 860.823℃, 차이 -1.663℃; SD=0.512; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/07 23:12:00 / 시작 고정 기준 3창 복귀 03/07 23:12:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3842.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 00:20:00–03/08 00:21:00 (끝 제외); 240024-Z-TC-Z3-0178 | 최대 절대 Z=3.06 (03/08 00:20:30, 창 중심); 1분 중앙값 859.358℃, 과거 중심 860.889℃, 차이 -1.532℃; SD=0.500 (하한 적용); 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 00:24:00 / 시작 고정 기준 3창 복귀 03/08 00:24:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3769.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/08 00:42:00–03/08 00:44:00 (끝 제외); 240024-Z-OP-Z3-0200 | 최대 절대 Z=6.05 (03/08 00:43:30, 창 중심); 1분 중앙값 32.706%, 과거 중심 85.063%, 차이 -52.357%p; SD=8.661; 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 00:48:00 / 시작 고정 기준 3창 복귀 03/08 00:48:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3746.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 00:43:00–03/08 00:44:00 (끝 제외); 240024-Z-TC-Z3-0201 | 최대 절대 Z=4.46 (03/08 00:43:30, 창 중심); 1분 중앙값 863.174℃, 과거 중심 860.705℃, 차이 +2.469℃; SD=0.554; 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 00:48:00 / 시작 고정 기준 3창 복귀 03/08 00:48:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3746.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 00:51:00–03/08 00:53:00 (끝 제외); 240024-Z-TC-Z3-0209 | 최대 절대 Z=4.27 (03/08 00:51:30, 창 중심); 1분 중앙값 857.396℃, 과거 중심 860.781℃, 차이 -3.385℃; SD=0.792; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 00:56:00 / 시작 고정 기준 3창 복귀 03/08 00:57:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3738.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/08 00:55:00–03/08 00:56:00 (끝 제외); 240024-Z-OP-Z3-0213 | 최대 절대 Z=3.09 (03/08 00:55:30, 창 중심); 1분 중앙값 41.560%, 과거 중심 87.116%, 차이 -45.556%p; SD=14.757; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 00:59:00 / 시작 고정 기준 3창 복귀 03/08 00:59:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3734.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/08 01:02:00–03/08 01:04:00 (끝 제외); 240024-Z-OP-Z3-0220 | 최대 절대 Z=3.77 (03/08 01:03:30, 창 중심); 1분 중앙값 8.845%, 과거 중심 87.608%, 차이 -78.763%p; SD=20.871; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 01:07:00 / 시작 고정 기준 3창 복귀 03/08 01:07:00 | 짧아도 최근 30분에 3회 시작했다. 재발 간격과 동일 운전 조건을 추적; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3726.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 02:25:00–03/08 02:26:00 (끝 제외); 240024-Z-TC-Z3-0303 | 최대 절대 Z=3.22 (03/08 02:25:30, 창 중심); 1분 중앙값 858.483℃, 과거 중심 860.705℃, 차이 -2.222℃; SD=0.691; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 02:29:00 / 시작 고정 기준 3창 복귀 03/08 02:29:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3644.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/08 02:50:00–03/08 02:52:00 (끝 제외); 240024-Z-OP-Z3-0328 | 최대 절대 Z=4.20 (03/08 02:50:30, 창 중심); 1분 중앙값 51.153%, 과거 중심 86.988%, 차이 -35.835%p; SD=8.533; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 02:55:00 / 시작 고정 기준 3창 복귀 03/08 02:55:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3619.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 02:55:00–03/08 02:56:00 (끝 제외); 240024-Z-TC-Z3-0333 | 최대 절대 Z=3.20 (03/08 02:55:30, 창 중심); 1분 중앙값 857.244℃, 과거 중심 860.620℃, 차이 -3.376℃; SD=1.055; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 02:59:00 / 시작 고정 기준 3창 복귀 03/08 02:59:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3614.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/08 04:09:00–03/08 04:10:00 (끝 제외); 240024-Z-OP-Z3-0407 | 최대 절대 Z=3.27 (03/08 04:09:30, 창 중심); 1분 중앙값 58.939%, 과거 중심 88.076%, 차이 -29.137%p; SD=8.905; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 04:13:00 / 시작 고정 기준 3창 복귀 03/08 04:14:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3540.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 04:10:00–03/08 04:11:00 (끝 제외); 240024-Z-TC-Z3-0408 | 최대 절대 Z=3.16 (03/08 04:10:30, 창 중심); 1분 중앙값 862.542℃, 과거 중심 860.611℃, 차이 +1.931℃; SD=0.610; 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 04:14:00 / 시작 고정 기준 3창 복귀 03/08 04:14:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3539.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 03/08 04:50:00–03/08 04:53:00 (끝 제외); 240024-Z-OP-Z3-0448 | 최대 절대 Z=5.08 (03/08 04:50:30, 창 중심); 1분 중앙값 22.387%, 과거 중심 83.771%, 차이 -61.384%p; SD=12.084; 실측 56/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 04:56:00 / 시작 고정 기준 3창 복귀 03/08 05:02:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3499.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 03/08 04:51:00–03/08 04:55:00 (끝 제외); 240024-Z-TC-Z3-0449 | 최대 절대 Z=5.76 (03/08 04:51:30, 창 중심); 1분 중앙값 866.958℃, 과거 중심 860.745℃, 차이 +6.213℃; SD=1.080; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 04:58:00 / 시작 고정 기준 3창 복귀 03/08 05:02:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3498.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 03/08 05:03:00–03/08 05:06:00 (끝 제외); 240024-Z-OP-Z6-0461 | 최대 절대 Z=3.80 (03/08 05:03:30, 창 중심); 1분 중앙값 82.847%, 과거 중심 75.251%, 차이 +7.596%p; SD=2.000 (하한 적용); 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 05:09:00 / 시작 고정 기준 3창 복귀 03/08 05:11:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3486.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240024-Z-TC-Z3-0449:** OP-Z3 사건창 중앙값−직전10분 중앙값 -66.64 (9/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240024-Z-OP-Z3-0448:** TC-Z3 사건창 중앙값−직전10분 중앙값 +6.243 (9/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 03/07 21:22:00–03/07 21:42:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 과거 유효 최소 0창 <20 |
| 03/07 21:46:00–03/07 21:47:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 03/07 21:55:00–03/07 21:56:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/07 22:35:00–03/07 22:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 03/07 22:45:00–03/07 22:46:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/07 22:59:00–03/07 23:00:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 03/07 23:08:00–03/07 23:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 03/07 23:26:00–03/07 23:27:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/07 23:31:00–03/07 23:32:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/07 23:55:00–03/07 23:56:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 03/08 00:04:00–03/08 00:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 00:35:00–03/08 00:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 00:44:00–03/08 00:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 03/08 01:15:00–03/08 01:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 03/08 01:24:00–03/08 01:25:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 03/08 01:44:00–03/08 01:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 02:23:00–03/08 02:24:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 03/08 03:15:00–03/08 03:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 03:24:00–03/08 03:25:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 03:30:00–03/08 03:31:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 03/08 03:49:00–03/08 03:50:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 03/08 04:45:00–03/08 04:46:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 05:28:00–03/08 05:29:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 05:32:00–03/08 05:33:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 03/08 05:54:00–03/08 05:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 03/08 06:22:00–03/08 06:24:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 03/08 06:28:00–03/08 06:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240024-CP-01 · 03/08 04:59:35 | -0.0664무차원 · 126점/135초 간격 | 같은 강종 LOT 240108의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다. | 60개 연속 실제 1초 값 복귀 05:01:33; WO-2024-00378 감지 2일 10시간 10분 전에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240024-CP-01 · 피크 03/08 04:59:35 · 복귀 펄스 후보**

- **관측:** 원값 0.38364, LOT 중앙값 0.450002, 편차 -0.066362. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 126점, 첫·끝 실측 간격 135초, 기대 1초 격자 내 유효 비율 92.6%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 03/08 04:58:17=0.430195 / 직후 03/08 05:00:34=0.430854; 이후 60개 연속 실측 ±0.02 복귀 확인 03/08 05:01:33.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 66851–66978에 연결했다(0부터 시작).

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240024-CP-01 | 정비 대조: WO-2024-00378 감지 대비 CP 원본 피크 -3490.42분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -5.626 (293/57점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +47.38 (293/57점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +1.75e-12 (293/57점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240024, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 복귀 추가 확인 — OP-Z7 · 03/08 05:13~05:14**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.08**, 과거 중심 대비 **+6.15%p**.  
**다른 LOT:** 같은 강종 LOT 240108의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -0.267℃(10/1유효창). 같은 전후 기준의 OP-Z7 변화는 +4.86%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 05:18 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 1건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240024-Z-OP-Z7-0471 · 03/08 05:13~05:14**

**다른 LOT:** 같은 강종 LOT 240108의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -0.267℃(10/1유효창). 같은 전후 기준의 OP-Z7 변화는 +4.86%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 05:18 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z7·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240024 (현재 LOT) | 1건/500유효창 (100창당 0.20건) | 6.154~6.154 | 1~1창 | 63.663 |
| 240108 | 0건/421유효창 (100창당 0.00건) | — | — | 62.538 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| OP-Z7 | 548 | 500 | 1 | 48 | 28 | 20 | 0 |
| TC-Z8 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |
| OP-Z8 | 548 | 500 | 0 | 48 | 28 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 500창, 최대 절대 Z=0.66 / TC-Z8: 유효 500창, 최대 절대 Z=0.69 / OP-Z8: 유효 500창, 최대 절대 Z=1.66. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z7 · 03/08 05:13:00–03/08 05:14:00 (끝 제외); 240024-Z-OP-Z7-0471 | 최대 절대 Z=3.08 (03/08 05:13:30, 창 중심); 1분 중앙값 69.404%, 과거 중심 63.250%, 차이 +6.154%p; SD=2.000 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 03/08 05:17:00 / 시작 고정 기준 3창 복귀 03/08 05:18:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00378 감지 대비 Z 최대창 중심 -3476.50분 (근접 시각이며 인과 미확정) |

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 03/07 21:22:00–03/07 21:42:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 03/07 21:46:00–03/07 21:47:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 03/07 21:55:00–03/07 21:56:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/07 22:35:00–03/07 22:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 03/07 22:45:00–03/07 22:46:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/07 22:59:00–03/07 23:00:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 03/07 23:08:00–03/07 23:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 03/07 23:26:00–03/07 23:27:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/07 23:31:00–03/07 23:32:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/07 23:55:00–03/07 23:56:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 03/08 00:04:00–03/08 00:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 00:35:00–03/08 00:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 00:44:00–03/08 00:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 03/08 01:15:00–03/08 01:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 03/08 01:24:00–03/08 01:25:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 03/08 01:44:00–03/08 01:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 02:23:00–03/08 02:24:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 03/08 03:15:00–03/08 03:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 03:24:00–03/08 03:25:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 03:30:00–03/08 03:31:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 03/08 03:49:00–03/08 03:50:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 03/08 04:45:00–03/08 04:46:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 05:28:00–03/08 05:29:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 05:32:00–03/08 05:33:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 03/08 05:54:00–03/08 05:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 03/08 06:22:00–03/08 06:24:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 03/08 06:28:00–03/08 06:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240024)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **1건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 31729 | 28 | 원본 ±0.05 사건 1건 · 최대 절대 편차 0.066362 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 31727 | 28 | 원본 중앙값 1.15447e-10, 최저 1.14241e-10, 최고 1.16761e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 31727 | 28 | 원본 중앙값 328.553, 최저 317.074, 최고 340.158; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 31729 | 28 | 원본 중앙값 278.872, 최저 263.274, 최고 292.226; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 31727 | 28 | 원본 중앙값 274.936, 최저 264.561, 최고 285.971; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-00378 · TBM / 메모: 1호기 재발, 정기점검 - 이상없음**

**실제 시각:** 감지 03/10 15:10:00 / 작업 03/10 15:46:00~03/10 18:36:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-00378 감지 2일 10시간 10분 전에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00378 · TBM | 03/10 15:10:00 | 03/10 15:46:00 | 03/10 18:36:00 | 1호기 재발, 정기점검 - 이상없음 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-00378 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 21건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240024-CP-01 · 03/08 04:59:35 · CP 편차 -0.066362 · 126점/135초 · 완료 대비 피크 -3696.42분 · 감지는 피크 대비 +3490.42분. 복귀 펄스 후보; 직전 03/08 04:58:17=0.430195 / 직후 03/08 05:00:34=0.430854; 이후 60개 연속 실측 ±0.02 복귀 확인 03/08 05:01:33.
- **동반 양상:** OP-Z1 1건·최대 절대 Z 4.59 / OP-Z2 1건·최대 절대 Z 4.46 / OP-Z3 8건·최대 절대 Z 6.05 / OP-Z6 1건·최대 절대 Z 3.80 / OP-Z7 1건·최대 절대 Z 3.08 / TC-Z1 1건·최대 절대 Z 3.28 / TC-Z3 8건·최대 절대 Z 5.76. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 03/10 15:10:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 03/10 15:46:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 03/10 18:36:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240024)

**240024 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240024)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240024)
lot_conclusion(240024)
lot_execution.append({'LOT':240024,'확대 구간':len(LOT_EVENTS[240024]),'강종 대조':240024 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240037"></a>
### 8.3 LOT 240037 · SPCE · 04/13

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240037)

#### LOT 240037 · 먼저 볼 변화

**지속·반복 후보 11건 / CP 원본 후보 3건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 04/13 23:57:59**

**관측:** 원본 편차 **+0.1567무차원**, 149점·첫/끝 간격 155초. 1초 단발 여부: 아님.  
**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 04/14 00:04:11. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 15시간 6분 1초 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 04/14 07:50~07:55**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.68**, 과거 중심 대비 **+6.61℃**.  
**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 5창·6.61℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -79.3%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.69℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:11 (이동 Z 복귀보다 13분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z1 · 04/14 09:09~09:14**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.74**, 과거 중심 대비 **-10.2%p**.  
**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~11.2%p)이 있다. 이 구간의 5창·10.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.81℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 09:20 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240037

- **관측 범위:** 04/13 23:16:00–04/14 09:27:19.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 519개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 11개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 45개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 3건 중 확인된 1초 단발 0건, 복귀 펄스 후보 3건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240037, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 2건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 04/14 09:09~09:14**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.74**, 과거 중심 대비 **-10.2%p**.  
**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~11.2%p)이 있다. 이 구간의 5창·10.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.81℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 09:20 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 04/14 09:11~09:16**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.49**, 과거 중심 대비 **-9.64%p**.  
**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 3건(연속 2~7개 1분창, 절대 편차 9.36~10.7%p)이 있다. 이 구간의 5창·9.64%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.24℃(9/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.82%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 09:23 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z1 · 04/14 09:11~09:13**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **3.30**, 과거 중심 대비 **+1.74℃**.  
**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 2건(연속 2~3개 1분창, 절대 편차 1.74~1.92℃)이 있다. 이 구간의 2창·1.74℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -13.9%p(9/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.87℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 09:17 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 3건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240037-Z-OP-Z1-0593 · 04/14 09:09~09:14**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~11.2%p)이 있다. 이 구간의 5창·10.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.81℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 09:20 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 1건/568유효창 (100창당 0.18건) | 10.22~10.22 | 5~5창 | 73.753 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 11.22~11.22 | 5~5창 | 74.374 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 10.16~10.16 | 5~5창 | 73.551 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z2-0595 · 04/14 09:11~09:16**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 3건(연속 2~7개 1분창, 절대 편차 9.36~10.7%p)이 있다. 이 구간의 5창·9.64%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.24℃(9/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.82%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 09:23 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 1건/568유효창 (100창당 0.18건) | 9.645~9.645 | 5~5창 | 22.003 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 9.364~9.364 | 7~7창 | 23.41 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 9.377~10.7 | 2~3창 | 23.012 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z1-0595 · 04/14 09:11~09:13**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 2건(연속 2~3개 1분창, 절대 편차 1.74~1.92℃)이 있다. 이 구간의 2창·1.74℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -13.9%p(9/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.87℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 09:17 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 1건/568유효창 (100창당 0.18건) | 1.743~1.743 | 2~2창 | 100 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 1.743~1.743 | 2~2창 | 100.01 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 1.918~1.918 | 3~3창 | 99.941 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| TC-Z1 | 612 | 568 | 2 | 44 | 24 | 20 | 0 |
| OP-Z1 | 612 | 568 | 5 | 44 | 24 | 20 | 0 |
| TC-Z2 | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| OP-Z2 | 612 | 568 | 5 | 44 | 24 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 568창, 최대 절대 Z=2.54 / TC-Z2: 유효 568창, 최대 절대 Z=2.60. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 04/14 09:09:00–04/14 09:14:00 (끝 제외); 240037-Z-OP-Z1-0593 | 최대 절대 Z=4.74 (04/14 09:10:30, 창 중심); 1분 중앙값 63.021%, 과거 중심 73.243%, 차이 -10.222%p; SD=2.159; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 09:17:00 / 시작 고정 기준 3창 복귀 04/14 09:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -353.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 04/14 09:11:00–04/14 09:16:00 (끝 제외); 240037-Z-OP-Z2-0595 | 최대 절대 Z=4.49 (04/14 09:13:30, 창 중심); 1분 중앙값 13.310%, 과거 중심 22.954%, 차이 -9.645%p; SD=2.150; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 09:19:00 / 시작 고정 기준 3창 복귀 04/14 09:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -350.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 04/14 09:11:00–04/14 09:13:00 (끝 제외); 240037-Z-TC-Z1-0595 | 최대 절대 Z=3.30 (04/14 09:12:30, 창 중심); 1분 중앙값 101.911℃, 과거 중심 100.168℃, 차이 +1.743℃; SD=0.527; 실측 56/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 09:16:00 / 시작 고정 기준 3창 복귀 04/14 09:17:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -351.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240037-Z-OP-Z1-0593:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.807 (9/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240037-Z-OP-Z2-0595:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.242 (9/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 04/13 23:16:00–04/13 23:37:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 04/13 23:51:00–04/13 23:52:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 04/14 01:23:00–04/14 01:24:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 02:13:00–04/14 02:14:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 04/14 02:29:00–04/14 02:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 04/14 02:37:00–04/14 02:38:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 03:20:00–04/14 03:21:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 03:25:00–04/14 03:26:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 04/14 03:29:00–04/14 03:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 04/14 03:37:00–04/14 03:38:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 04/14 04:19:00–04/14 04:20:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 04/14 04:51:00–04/14 04:52:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 05:29:00–04/14 05:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 05:33:00–04/14 05:34:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 05:41:00–04/14 05:42:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 05:46:00–04/14 05:47:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 05:54:00–04/14 05:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 06:02:00–04/14 06:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 04/14 06:58:00–04/14 06:59:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 07:06:00–04/14 07:07:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 07:10:00–04/14 07:11:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 04/14 08:50:00–04/14 08:51:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 04/14 09:04:00–04/14 09:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 04/14 09:27:00–04/14 09:28:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 17개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240037, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 9건 · 짧은 노랑 후보 17건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 04/13 23:57:59**

**관측:** 원본 편차 **+0.1567무차원**, 149점·첫/끝 간격 155초. 1초 단발 여부: 아님.  
**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 04/14 00:04:11. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 15시간 6분 1초 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 04/14 07:50~07:55**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.68**, 과거 중심 대비 **+6.61℃**.  
**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 5창·6.61℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -79.3%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.69℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:11 (이동 Z 복귀보다 13분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z6 · 04/14 08:02~08:06**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.62**, 과거 중심 대비 **+9.38%p**.  
**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 3건(연속 1~3개 1분창, 절대 편차 6.86~10.6%p)이 있다. 이 구간의 4창·9.38%p는 비교 관측의 상한을 넘는다(길이 4>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.51℃(10/4유효창). 같은 전후 기준의 OP-Z6 변화는 +9.45%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:12 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


**그림 숫자의 뜻(TC-Z3):** 전체 612개 1분창 = 계산 가능 568개 + 보류 44개. 초과 28개는 계산 가능한 568개 중 일부이며, 연속창을 묶으면 TC-Z3 사건 13건이다. 보류 44개 = 현재 실측 54개 미만 24개 + 현재창은 충족하지만 과거 유효 20창 미만 20개. 고장 수가 아니다.

<details>
<summary><strong>표시한 모든 후보 29건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240037-CP-01 · 04/13 23:20:37**

**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다 (계산 보류: TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6). CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 04/13 23:22:37. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 15시간 43분 23초 전에 CP 원본 피크.

강종 SPCE·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 3건/35348실측점 (1만점당 0.85건) | 0.06203~0.1567 | 46~155초 간격 | 0.45006 |
| 240121 | 0건/38237실측점 (1만점당 0.00건) | — | — | 0.44999 |
| 240133 | 0건/6218실측점 (1만점당 0.00건) | — | — | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-CP-02 · 04/13 23:57:59**

**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 04/14 00:04:11. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 15시간 6분 1초 전에 CP 원본 피크.

강종 SPCE·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 3건/35348실측점 (1만점당 0.85건) | 0.06203~0.1567 | 46~155초 간격 | 0.45006 |
| 240121 | 0건/38237실측점 (1만점당 0.00건) | — | — | 0.44999 |
| 240133 | 0건/6218실측점 (1만점당 0.00건) | — | — | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0070 · 04/14 00:26~00:28**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 2창·76.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.36℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -61.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:32 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0071 · 04/14 00:27~00:29**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 2창·4.08℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -61.8%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.54℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:38 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0075 · 04/14 00:31~00:33**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 11건(연속 1~3개 1분창, 절대 편차 1.51~5.51℃)이 있다. 이 구간의 2창·4.79℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +9.35%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -5.26℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:37 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0128 · 04/14 01:24~01:27**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 3창·57.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.16℃(9/3유효창). 같은 전후 기준의 OP-Z3 변화는 -72.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 01:31 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0128 · 04/14 01:24~01:28**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 4창·4.31℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -64.2%p(9/4유효창). 같은 전후 기준의 TC-Z3 변화는 +4.41℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 01:49 (이동 Z 복귀보다 13분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0134 · 04/14 01:30~01:33**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 11건(연속 1~3개 1분창, 절대 편차 1.51~5.51℃)이 있다. 이 구간의 3창·6.8℃는 비교 관측의 상한을 넘는다(절대 편차 6.8>5.51℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +16.2%p(9/3유효창). 같은 전후 기준의 TC-Z3 변화는 -8.66℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 01:38 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0172 · 04/14 02:08~02:09**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 1창·26.3%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.55℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -28.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:12. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0197 · 04/14 02:33~02:35**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 2창·67.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.28℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -67.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:41. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0197 · 04/14 02:33~02:35**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 2창·4.45℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -67.9%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.28℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:53 (이동 Z 복귀보다 11분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0202 · 04/14 02:38~02:39**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 11건(연속 1~3개 1분창, 절대 편차 1.51~5.51℃)이 있다. 이 구간의 1창·3.98℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +14.9%p(8/1유효창). 같은 전후 기준의 TC-Z3 변화는 -4.46℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:42. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0205 · 04/14 02:41~02:43**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 2창·69%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.66℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -71.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:46. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-CP-03 · 04/14 03:11:03**

**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 04/14 03:13:37. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 11시간 52분 57초 전에 CP 원본 피크.

강종 SPCE·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 3건/35348실측점 (1만점당 0.85건) | 0.06203~0.1567 | 46~155초 간격 | 0.45006 |
| 240121 | 0건/38237실측점 (1만점당 0.00건) | — | — | 0.44999 |
| 240133 | 0건/6218실측점 (1만점당 0.00건) | — | — | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0296 · 04/14 04:12~04:15**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 11건(연속 1~3개 1분창, 절대 편차 1.51~5.51℃)이 있다. 이 구간의 3창·3.1℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +22.3%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 -3.58℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:19 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0344 · 04/14 05:00~05:02**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 2창·22.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.908℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -21.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:05. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0350 · 04/14 05:06~05:07**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 11건(연속 1~3개 1분창, 절대 편차 1.51~5.51℃)이 있다. 이 구간의 1창·1.8℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +12.8%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -1.74℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:11 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0363 · 04/14 05:19~05:20**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 1창·33.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.752℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -31.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:23. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0371 · 04/14 05:27~05:28**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 1창·56.4%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.42℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -57%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:33. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0371 · 04/14 05:27~05:28**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 1창·3.07℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -57%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.42℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:37 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0416 · 04/14 06:12~06:14**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 2창·40.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.9℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -39.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:17. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0417 · 04/14 06:13~06:14**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 1창·2.58℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -41.4%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.5℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:23 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0420 · 04/14 06:16~06:18**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 11건(연속 1~3개 1분창, 절대 편차 1.51~5.51℃)이 있다. 이 구간의 2창·3.28℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +19.7%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -4.08℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:22 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0514 · 04/14 07:50~07:53**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 3창·66.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.9℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -79.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:02 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0514 · 04/14 07:50~07:55**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 5창·6.61℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -79.3%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.69℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:11 (이동 Z 복귀보다 13분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z6-0526 · 04/14 08:02~08:06**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 3건(연속 1~3개 1분창, 절대 편차 6.86~10.6%p)이 있다. 이 구간의 4창·9.38%p는 비교 관측의 상한을 넘는다(길이 4>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.51℃(10/4유효창). 같은 전후 기준의 OP-Z6 변화는 +9.45%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:12 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 1건/568유효창 (100창당 0.18건) | 9.377~9.377 | 4~4창 | 74.298 |
| 240121 | 2건/611유효창 (100창당 0.33건) | 6.858~10.64 | 1~3창 | 72.826 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 8.591~8.591 | 1~1창 | 71.427 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z6-0527 · 04/14 08:03~08:06**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 1.77~1.77℃)이 있다. 이 구간의 3창·1.8℃는 비교 관측의 상한을 넘는다(절대 편차 1.8>1.77℃, 길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 +8.72%p(10/3유효창). 같은 전후 기준의 TC-Z6 변화는 -1.47℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:09. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 1건/567유효창 (100창당 0.18건) | 1.795~1.795 | 3~3창 | 860.44 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 1.769~1.769 | 2~2창 | 860.41 |
| 240133 | 0건/81유효창 (100창당 0.00건) | — | — | 860.27 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-OP-Z3-0592 · 04/14 09:08~09:10**

**다른 LOT:** 동종 LOT 240121, 240133에도 같은 방향 후보 15건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 2창·58.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.94℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -56.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:13. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240037-Z-TC-Z3-0593 · 04/14 09:09~09:10**

**다른 LOT:** 동종 LOT 240121에도 같은 방향 후보 11건(연속 1~5개 1분창, 절대 편차 2.16~6.9℃)이 있다. 이 구간의 1창·4.26℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -63.5%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.25℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:17 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240121 | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 612 | 568 | 28 | 44 | 24 | 20 | 0 |
| OP-Z3 | 612 | 568 | 21 | 44 | 24 | 20 | 0 |
| TC-Z4 | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| OP-Z4 | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| TC-Z5 | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| OP-Z5 | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| TC-Z6 | 612 | 567 | 3 | 45 | 25 | 20 | 0 |
| OP-Z6 | 612 | 568 | 4 | 44 | 24 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 3건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 568창, 최대 절대 Z=2.34 / OP-Z4: 유효 568창, 최대 절대 Z=2.86 / TC-Z5: 유효 568창, 최대 절대 Z=1.29 / OP-Z5: 유효 568창, 최대 절대 Z=1.75. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 04/14 00:26:00–04/14 00:28:00 (끝 제외); 240037-Z-OP-Z3-0070 | 최대 절대 Z=5.39 (04/14 00:27:30, 창 중심); 1분 중앙값 10.608%, 과거 중심 87.196%, 차이 -76.588%p; SD=14.211; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 00:31:00 / 시작 고정 기준 3창 복귀 04/14 00:32:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -876.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 00:27:00–04/14 00:29:00 (끝 제외); 240037-Z-TC-Z3-0071 | 최대 절대 Z=4.95 (04/14 00:27:30, 창 중심); 1분 중앙값 865.338℃, 과거 중심 861.260℃, 차이 +4.077℃; SD=0.823; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 00:36:00 / 시작 고정 기준 3창 복귀 04/14 00:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -876.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 00:31:00–04/14 00:33:00 (끝 제외); 240037-Z-TC-Z3-0075 | 최대 절대 Z=3.31 (04/14 00:31:30, 창 중심); 1분 중앙값 856.471℃, 과거 중심 861.260℃, 차이 -4.789℃; SD=1.445; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 00:36:00 / 시작 고정 기준 3창 복귀 04/14 00:37:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -872.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 01:24:00–04/14 01:27:00 (끝 제외); 240037-Z-OP-Z3-0128 | 최대 절대 Z=10.45 (04/14 01:24:30, 창 중심); 1분 중앙값 31.710%, 과거 중심 89.390%, 차이 -57.680%p; SD=5.518; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 01:30:00 / 시작 고정 기준 3창 복귀 04/14 01:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -819.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 01:24:00–04/14 01:28:00 (끝 제외); 240037-Z-TC-Z3-0128 | 최대 절대 Z=7.15 (04/14 01:25:30, 창 중심); 1분 중앙값 865.447℃, 과거 중심 861.134℃, 차이 +4.313℃; SD=0.603; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 01:36:00 / 시작 고정 기준 3창 복귀 04/14 01:49:00 | 고정 범위 복귀가 이동 Z 복귀보다 13분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -818.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 01:30:00–04/14 01:33:00 (끝 제외); 240037-Z-TC-Z3-0134 | 최대 절대 Z=3.77 (04/14 01:30:30, 창 중심); 1분 중앙값 854.338℃, 과거 중심 861.134℃, 차이 -6.796℃; SD=1.801; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 01:36:00 / 시작 고정 기준 3창 복귀 04/14 01:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -813.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 02:08:00–04/14 02:09:00 (끝 제외); 240037-Z-OP-Z3-0172 | 최대 절대 Z=3.18 (04/14 02:08:30, 창 중심); 1분 중앙값 66.365%, 과거 중심 92.653%, 차이 -26.288%p; SD=8.267; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 02:12:00 / 시작 고정 기준 3창 복귀 04/14 02:12:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -775.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 02:33:00–04/14 02:35:00 (끝 제외); 240037-Z-OP-Z3-0197 | 최대 절대 Z=6.32 (04/14 02:33:30, 창 중심); 1분 중앙값 18.931%, 과거 중심 86.541%, 차이 -67.611%p; SD=10.706; 실측 54/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 02:41:00 / 시작 고정 기준 3창 복귀 04/14 02:41:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -750.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 02:33:00–04/14 02:35:00 (끝 제외); 240037-Z-TC-Z3-0197 | 최대 절대 Z=4.92 (04/14 02:34:30, 창 중심); 1분 중앙값 865.627℃, 과거 중심 861.182℃, 차이 +4.445℃; SD=0.903; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 02:42:00 / 시작 고정 기준 3창 복귀 04/14 02:53:00 | 고정 범위 복귀가 이동 Z 복귀보다 11분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -749.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 02:38:00–04/14 02:39:00 (끝 제외); 240037-Z-TC-Z3-0202 | 최대 절대 Z=3.13 (04/14 02:38:30, 창 중심); 1분 중앙값 857.280℃, 과거 중심 861.255℃, 차이 -3.975℃; SD=1.271; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 02:42:00 / 시작 고정 기준 3창 복귀 04/14 02:42:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -745.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 02:41:00–04/14 02:43:00 (끝 제외); 240037-Z-OP-Z3-0205 | 최대 절대 Z=3.42 (04/14 02:41:30, 창 중심); 1분 중앙값 17.489%, 과거 중심 86.463%, 차이 -68.974%p; SD=20.142; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 02:46:00 / 시작 고정 기준 3창 복귀 04/14 02:46:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -742.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 04:12:00–04/14 04:15:00 (끝 제외); 240037-Z-TC-Z3-0296 | 최대 절대 Z=4.31 (04/14 04:13:30, 창 중심); 1분 중앙값 858.084℃, 과거 중심 861.183℃, 차이 -3.099℃; SD=0.720; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 04:18:00 / 시작 고정 기준 3창 복귀 04/14 04:19:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -650.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 05:00:00–04/14 05:02:00 (끝 제외); 240037-Z-OP-Z3-0344 | 최대 절대 Z=4.04 (04/14 05:00:30, 창 중심); 1분 중앙값 64.434%, 과거 중심 86.963%, 차이 -22.529%p; SD=5.576; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 05:05:00 / 시작 고정 기준 3창 복귀 04/14 05:05:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -603.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 05:06:00–04/14 05:07:00 (끝 제외); 240037-Z-TC-Z3-0350 | 최대 절대 Z=3.54 (04/14 05:06:30, 창 중심); 1분 중앙값 859.370℃, 과거 중심 861.172℃, 차이 -1.803℃; SD=0.509; 실측 54/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 05:10:00 / 시작 고정 기준 3창 복귀 04/14 05:11:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -597.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 05:19:00–04/14 05:20:00 (끝 제외); 240037-Z-OP-Z3-0363 | 최대 절대 Z=3.20 (04/14 05:19:30, 창 중심); 1분 중앙값 52.000%, 과거 중심 85.712%, 차이 -33.712%p; SD=10.531; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 05:23:00 / 시작 고정 기준 3창 복귀 04/14 05:23:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -584.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 05:27:00–04/14 05:28:00 (끝 제외); 240037-Z-OP-Z3-0371 | 최대 절대 Z=3.77 (04/14 05:27:30, 창 중심); 1분 중앙값 32.584%, 과거 중심 88.990%, 차이 -56.406%p; SD=14.965; 실측 59/60개·과거 30/30창 | 지속·반복 후보: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 05:33:00 / 시작 고정 기준 3창 복귀 04/14 05:33:00 | 짧아도 최근 30분에 3회 시작했다. 재발 간격과 동일 운전 조건을 추적; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -576.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 05:27:00–04/14 05:28:00 (끝 제외); 240037-Z-TC-Z3-0371 | 최대 절대 Z=3.06 (04/14 05:27:30, 창 중심); 1분 중앙값 864.062℃, 과거 중심 860.995℃, 차이 +3.067℃; SD=1.001; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 05:33:00 / 시작 고정 기준 3창 복귀 04/14 05:37:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -576.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 06:12:00–04/14 06:14:00 (끝 제외); 240037-Z-OP-Z3-0416 | 최대 절대 Z=5.24 (04/14 06:12:30, 창 중심); 1분 중앙값 44.836%, 과거 중심 85.649%, 차이 -40.812%p; SD=7.791; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 06:17:00 / 시작 고정 기준 3창 복귀 04/14 06:17:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -531.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 06:13:00–04/14 06:14:00 (끝 제외); 240037-Z-TC-Z3-0417 | 최대 절대 Z=4.18 (04/14 06:13:30, 창 중심); 1분 중앙값 863.852℃, 과거 중심 861.275℃, 차이 +2.577℃; SD=0.617; 실측 56/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 06:21:00 / 시작 고정 기준 3창 복귀 04/14 06:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -530.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 06:16:00–04/14 06:18:00 (끝 제외); 240037-Z-TC-Z3-0420 | 최대 절대 Z=3.87 (04/14 06:16:30, 창 중심); 1분 중앙값 858.058℃, 과거 중심 861.341℃, 차이 -3.283℃; SD=0.848; 실측 58/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 06:21:00 / 시작 고정 기준 3창 복귀 04/14 06:22:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -527.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 07:50:00–04/14 07:53:00 (끝 제외); 240037-Z-OP-Z3-0514 | 최대 절대 Z=5.26 (04/14 07:50:30, 창 중심); 1분 중앙값 20.412%, 과거 중심 87.108%, 차이 -66.695%p; SD=12.680; 실측 57/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 07:56:00 / 시작 고정 기준 3창 복귀 04/14 08:02:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -433.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 07:50:00–04/14 07:55:00 (끝 제외); 240037-Z-TC-Z3-0514 | 최대 절대 Z=5.68 (04/14 07:51:30, 창 중심); 1분 중앙값 867.583℃, 과거 중심 860.974℃, 차이 +6.609℃; SD=1.163; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 07:58:00 / 시작 고정 기준 3창 복귀 04/14 08:11:00 | 고정 범위 복귀가 이동 Z 복귀보다 13분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -432.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 04/14 08:02:00–04/14 08:06:00 (끝 제외); 240037-Z-OP-Z6-0526 | 최대 절대 Z=4.62 (04/14 08:03:30, 창 중심); 1분 중앙값 82.735%, 과거 중심 73.358%, 차이 +9.377%p; SD=2.031; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 08:09:00 / 시작 고정 기준 3창 복귀 04/14 08:12:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -420.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 04/14 08:03:00–04/14 08:06:00 (끝 제외); 240037-Z-TC-Z6-0527 | 최대 절대 Z=3.59 (04/14 08:04:30, 창 중심); 1분 중앙값 858.636℃, 과거 중심 860.431℃, 차이 -1.795℃; SD=0.500 (하한 적용); 실측 54/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 08:09:00 / 시작 고정 기준 3창 복귀 04/14 08:09:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -419.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/14 09:08:00–04/14 09:10:00 (끝 제외); 240037-Z-OP-Z3-0592 | 최대 절대 Z=4.64 (04/14 09:08:30, 창 중심); 1분 중앙값 26.993%, 과거 중심 85.709%, 차이 -58.716%p; SD=12.658; 실측 55/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 09:13:00 / 시작 고정 기준 3창 복귀 04/14 09:13:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -355.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/14 09:09:00–04/14 09:10:00 (끝 제외); 240037-Z-TC-Z3-0593 | 최대 절대 Z=3.66 (04/14 09:09:30, 창 중심); 1분 중앙값 865.068℃, 과거 중심 860.807℃, 차이 +4.261℃; SD=1.164; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 09:13:00 / 시작 고정 기준 3창 복귀 04/14 09:17:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -354.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240037-Z-TC-Z3-0514:** OP-Z3 사건창 중앙값−직전10분 중앙값 -79.28 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240037-Z-OP-Z6-0526:** TC-Z6 사건창 중앙값−직전10분 중앙값 -1.514 (10/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 04/13 23:16:00–04/13 23:38:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 04/13 23:51:00–04/13 23:52:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 04/14 01:23:00–04/14 01:24:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 02:13:00–04/14 02:14:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 04/14 02:29:00–04/14 02:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 04/14 02:37:00–04/14 02:38:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 03:20:00–04/14 03:21:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 03:25:00–04/14 03:26:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 04/14 03:29:00–04/14 03:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 04/14 03:37:00–04/14 03:38:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 04/14 04:19:00–04/14 04:20:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 04/14 04:51:00–04/14 04:52:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 05:29:00–04/14 05:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 05:33:00–04/14 05:34:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 05:41:00–04/14 05:42:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 05:46:00–04/14 05:47:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 05:54:00–04/14 05:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 06:02:00–04/14 06:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 04/14 06:58:00–04/14 06:59:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 07:06:00–04/14 07:07:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 07:10:00–04/14 07:11:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 04/14 08:50:00–04/14 08:51:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 04/14 09:04:00–04/14 09:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 04/14 09:27:00–04/14 09:28:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 17개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240037-CP-01 · 04/13 23:20:37 | +0.0620무차원 · 77점/78초 간격 | 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다. | 60개 연속 실제 1초 값 복귀 23:22:37; WO-2024-00627 감지 15시간 43분 23초 전에 CP 원본 피크 |
| 240037-CP-02 · 04/13 23:57:59 | +0.1567무차원 · 149점/155초 간격 | 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다. | 60개 연속 실제 1초 값 복귀 00:04:11; WO-2024-00627 감지 15시간 6분 1초 전에 CP 원본 피크 |
| 240037-CP-03 · 04/14 03:11:03 | +0.0720무차원 · 46점/46초 간격 | 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다. | 60개 연속 실제 1초 값 복귀 03:13:37; WO-2024-00627 감지 11시간 52분 57초 전에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240037-CP-01 · 피크 04/13 23:20:37 · 복귀 펄스 후보**

- **관측:** 원값 0.512126, LOT 중앙값 0.450097, 편차 +0.062029. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 77점, 첫·끝 실측 간격 78초, 기대 1초 격자 내 유효 비율 97.5%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 04/13 23:20:18=0.466319 / 직후 04/13 23:21:38=0.470089; 이후 60개 연속 실측 ±0.02 복귀 확인 04/13 23:22:37.
- **같은 1분의 결과:** 17신호에서 |Z|≥3 미검출 (평가 불가 채널은 별도).
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 72380–72458에 연결했다(0부터 시작).

**240037-CP-02 · 피크 04/13 23:57:59 · 복귀 펄스 후보**

- **관측:** 원값 0.60676, LOT 중앙값 0.450097, 편차 +0.156663. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 149점, 첫·끝 실측 간격 155초, 기대 1초 격자 내 유효 비율 95.5%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 04/13 23:57:29=0.462489 / 직후 04/14 00:00:06=0.4693; 이후 60개 연속 실측 ±0.02 복귀 확인 04/14 00:04:11.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 74533–74683에 연결했다(0부터 시작).

**240037-CP-03 · 피크 04/14 03:11:03 · 복귀 펄스 후보**

- **관측:** 원값 0.522137, LOT 중앙값 0.450097, 편차 +0.07204. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 46점, 첫·끝 실측 간격 46초, 기대 1초 격자 내 유효 비율 97.9%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 04/14 03:10:49=0.466616 / 직후 04/14 03:11:37=0.470036; 이후 60개 연속 실측 ±0.02 복귀 확인 04/14 03:13:37.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 85735–85782에 연결했다(0부터 시작).

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240037-CP-01 | 정비 대조: WO-2024-00627 감지 대비 CP 원본 피크 -943.38분 (근접 시각이며 인과 미확정) | TC-Z3 전후 실측 부족 (0/60점) / OP-Z3 전후 실측 부족 (0/60점) / CP-Z4M 전후 실측 부족 (0/60점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240037-CP-02 | 정비 대조: WO-2024-00627 감지 대비 CP 원본 피크 -906.02분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +0.086 (289/56점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -4.286 (289/56점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +4.04e-13 (289/56점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240037-CP-03 | 정비 대조: WO-2024-00627 감지 대비 CP 원본 피크 -712.95분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -2.11 (292/60점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +7.712 (292/60점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +1.16e-13 (292/60점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240037, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 노랑 · 짧은 국소 변화 — OP-Z7 · 04/14 00:41~00:42**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.16**, 과거 중심 대비 **+6.53%p**.  
**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -0.498℃(10/1유효창). 같은 전후 기준의 OP-Z7 변화는 +9.05%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 00:45. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 1건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240037-Z-OP-Z7-0085 · 04/14 00:41~00:42**

**다른 LOT:** 같은 강종 LOT 240121, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -0.498℃(10/1유효창). 같은 전후 기준의 OP-Z7 변화는 +9.05%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 00:45. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z7·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240037 (현재 LOT) | 1건/567유효창 (100창당 0.18건) | 6.533~6.533 | 1~1창 | 66.183 |
| 240121 | 0건/611유효창 (100창당 0.00건) | — | — | 66.953 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 70.311 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 612 | 567 | 0 | 45 | 25 | 20 | 0 |
| OP-Z7 | 612 | 567 | 1 | 45 | 25 | 20 | 0 |
| TC-Z8 | 612 | 568 | 0 | 44 | 24 | 20 | 0 |
| OP-Z8 | 612 | 567 | 0 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 567창, 최대 절대 Z=0.74 / TC-Z8: 유효 568창, 최대 절대 Z=0.48 / OP-Z8: 유효 567창, 최대 절대 Z=1.07. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z7 · 04/14 00:41:00–04/14 00:42:00 (끝 제외); 240037-Z-OP-Z7-0085 | 최대 절대 Z=3.16 (04/14 00:41:30, 창 중심); 1분 중앙값 72.662%, 과거 중심 66.129%, 차이 +6.533%p; SD=2.064; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/14 00:45:00 / 시작 고정 기준 3창 복귀 04/14 00:45:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 -862.50분 (근접 시각이며 인과 미확정) |

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 04/13 23:16:00–04/13 23:38:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 04/13 23:51:00–04/13 23:52:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 04/14 01:23:00–04/14 01:24:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 02:13:00–04/14 02:14:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 04/14 02:29:00–04/14 02:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 04/14 02:37:00–04/14 02:38:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 03:20:00–04/14 03:21:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 03:25:00–04/14 03:26:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 04/14 03:29:00–04/14 03:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 04/14 03:37:00–04/14 03:38:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 04/14 04:19:00–04/14 04:20:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 04/14 04:51:00–04/14 04:52:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 05:29:00–04/14 05:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 05:33:00–04/14 05:34:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 05:41:00–04/14 05:42:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 05:46:00–04/14 05:47:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 05:54:00–04/14 05:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 06:02:00–04/14 06:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 04/14 06:58:00–04/14 06:59:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 07:06:00–04/14 07:07:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 07:10:00–04/14 07:11:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 04/14 08:50:00–04/14 08:51:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 04/14 09:04:00–04/14 09:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 04/14 09:27:00–04/14 09:28:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 17개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240037)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **3건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 35348 | 24 | 원본 ±0.05 사건 3건 · 최대 절대 편차 0.15666 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 35348 | 24 | 원본 중앙값 1.15168e-10, 최저 1.1382e-10, 최고 1.16365e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 35348 | 24 | 원본 중앙값 330.419, 최저 319.248, 최고 342.25; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 35347 | 24 | 원본 중앙값 285.392, 최저 271.071, 최고 298.412; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 35346 | 24 | 원본 중앙값 281.756, 최저 271.229, 최고 291.994; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-00627 · CBM / 메모: 확인 중**

**실제 시각:** 감지 04/14 15:04:00 / 작업 04/23 15:27:00~04/23 18:29:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-00627 감지 15시간 6분 1초 전에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00627 · CBM | 04/14 15:04:00 | 04/23 15:27:00 | 04/23 18:29:00 | 확인 중 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-00627 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 3건, 센서별 Z 후보 30건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240037-CP-01 · 04/13 23:20:37 · CP 편차 +0.062029 · 77점/78초 · 완료 대비 피크 -14108.38분 · 감지는 피크 대비 +943.38분. 복귀 펄스 후보; 직전 04/13 23:20:18=0.466319 / 직후 04/13 23:21:38=0.470089; 이후 60개 연속 실측 ±0.02 복귀 확인 04/13 23:22:37.
  - 240037-CP-02 · 04/13 23:57:59 · CP 편차 +0.15666 · 149점/155초 · 완료 대비 피크 -14071.02분 · 감지는 피크 대비 +906.02분. 복귀 펄스 후보; 직전 04/13 23:57:29=0.462489 / 직후 04/14 00:00:06=0.4693; 이후 60개 연속 실측 ±0.02 복귀 확인 04/14 00:04:11.
  - 240037-CP-03 · 04/14 03:11:03 · CP 편차 +0.07204 · 46점/46초 · 완료 대비 피크 -13877.95분 · 감지는 피크 대비 +712.95분. 복귀 펄스 후보; 직전 04/14 03:10:49=0.466616 / 직후 04/14 03:11:37=0.470036; 이후 60개 연속 실측 ±0.02 복귀 확인 04/14 03:13:37.
- **동반 양상:** OP-Z1 1건·최대 절대 Z 4.74 / OP-Z2 1건·최대 절대 Z 4.49 / OP-Z3 11건·최대 절대 Z 10.45 / OP-Z6 1건·최대 절대 Z 4.62 / OP-Z7 1건·최대 절대 Z 3.16 / TC-Z1 1건·최대 절대 Z 3.30 / TC-Z3 13건·최대 절대 Z 7.15 / TC-Z6 1건·최대 절대 Z 3.59. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 04/14 15:04:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 04/23 15:27:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 04/23 18:29:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240037)

**240037 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240037)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240037)
lot_conclusion(240037)
lot_execution.append({'LOT':240037,'확대 구간':len(LOT_EVENTS[240037]),'강종 대조':240037 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240038"></a>
### 8.4 LOT 240038 · SPCEN · 04/16

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240038)

#### LOT 240038 · 먼저 볼 변화

**지속·반복 후보 4건 / CP 원본 후보 1건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 04/16 13:38:02**

**관측:** 원본 편차 **+0.0556무차원**, 85점·첫/끝 간격 84초. 1초 단발 여부: 아님.  
**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 OP-Z3도 기준을 넘었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 미확인. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 1일 22시간 34분 후에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 04/16 12:04~12:09**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.32**, 과거 중심 대비 **+7.81℃**.  
**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 2건(연속 2~5개 1분창, 절대 편차 2.7~6.25℃)이 있다. 이 구간의 5창·7.81℃는 비교 관측의 상한을 넘는다(절대 편차 7.81>6.25℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -81%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +10.7℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 12:28 (이동 Z 복귀보다 16분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z1 · 04/16 13:23~13:28**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.94**, 과거 중심 대비 **-9.88%p**.  
**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.52℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:33 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240038

- **관측 범위:** 04/16 11:35:00–04/16 13:39:56.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 83개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 4개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 22개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 1건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240038, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 2건 · 짧은 노랑 후보 2건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 04/16 13:23~13:28**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.94**, 과거 중심 대비 **-9.88%p**.  
**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.52℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:33 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 04/16 13:25~13:29**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.15**, 과거 중심 대비 **-8.31%p**.  
**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.935℃(10/4유효창). 같은 전후 기준의 OP-Z2 변화는 -7.87%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:36 (이동 Z 복귀보다 4분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 04/16 12:38~12:40**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **3.22**, 과거 중심 대비 **-1.61℃**.  
**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 1건(연속 4~4개 1분창, 절대 편차 1.78~1.78℃)이 있다. 이 구간의 2창·1.61℃는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:31 (이동 Z 복귀보다 48분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 4건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240038-Z-CLN-IN-0063 · 04/16 12:38~12:40**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 1건(연속 4~4개 1분창, 절대 편차 1.78~1.78℃)이 있다. 이 구간의 2창·1.61℃는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 13:31 (이동 Z 복귀보다 48분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 1.612~1.612 | 2~2창 | 69.389 |
| 240114 | 1건/254유효창 (100창당 0.39건) | 1.784~1.784 | 4~4창 | 66.486 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-OP-Z1-0108 · 04/16 13:23~13:28**

**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.52℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 13:33 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 9.88~9.88 | 5~5창 | 72.668 |
| 240114 | 0건/254유효창 (100창당 0.00건) | — | — | 73.785 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-OP-Z2-0110 · 04/16 13:25~13:29**

**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.935℃(10/4유효창). 같은 전후 기준의 OP-Z2 변화는 -7.87%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 13:36 (이동 Z 복귀보다 4분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 8.309~8.309 | 4~4창 | 20.892 |
| 240114 | 0건/254유효창 (100창당 0.00건) | — | — | 21.487 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-TC-Z1-0111 · 04/16 13:26~13:27**

**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -11.6%p(10/1유효창). 같은 전후 기준의 TC-Z1 변화는 +1.55℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 13:31 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 1.58~1.58 | 1~1창 | 99.975 |
| 240114 | 0건/254유효창 (100창당 0.00건) | — | — | 99.972 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 125 | 103 | 2 | 22 | 2 | 20 | 0 |
| TC-Z1 | 125 | 103 | 1 | 22 | 2 | 20 | 0 |
| OP-Z1 | 125 | 103 | 5 | 22 | 2 | 20 | 0 |
| TC-Z2 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| OP-Z2 | 125 | 103 | 4 | 22 | 2 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z2: 유효 103창, 최대 절대 Z=2.76. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| CLN-IN · 04/16 12:38:00–04/16 12:40:00 (끝 제외); 240038-Z-CLN-IN-0063 | 최대 절대 Z=3.22 (04/16 12:38:30, 창 중심); 1분 중앙값 69.279℃, 과거 중심 70.891℃, 차이 -1.612℃; SD=0.500 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 12:43:00 / 시작 고정 기준 3창 복귀 04/16 13:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 48분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2734.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 04/16 13:23:00–04/16 13:28:00 (끝 제외); 240038-Z-OP-Z1-0108 | 최대 절대 Z=4.94 (04/16 13:24:30, 창 중심); 1분 중앙값 62.100%, 과거 중심 71.981%, 차이 -9.880%p; SD=2.000 (하한 적용); 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 13:31:00 / 시작 고정 기준 3창 복귀 04/16 13:33:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2780.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 04/16 13:25:00–04/16 13:29:00 (끝 제외); 240038-Z-OP-Z2-0110 | 최대 절대 Z=4.15 (04/16 13:26:30, 창 중심); 1분 중앙값 12.638%, 과거 중심 20.946%, 차이 -8.309%p; SD=2.000 (하한 적용); 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 13:32:00 / 시작 고정 기준 3창 복귀 04/16 13:36:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2782.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 04/16 13:26:00–04/16 13:27:00 (끝 제외); 240038-Z-TC-Z1-0111 | 최대 절대 Z=3.16 (04/16 13:26:30, 창 중심); 1분 중앙값 101.680℃, 과거 중심 100.100℃, 차이 +1.580℃; SD=0.500 (하한 적용); 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 13:30:00 / 시작 고정 기준 3창 복귀 04/16 13:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2782.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240038-Z-OP-Z1-0108:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.516 (9/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240038-Z-OP-Z2-0110:** TC-Z2 사건창 중앙값−직전10분 중앙값 +0.935 (10/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 04/16 11:35:00–04/16 11:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 04/16 13:08:00–04/16 13:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 04/16 13:14:00–04/16 13:15:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240038, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 2건 · 짧은 노랑 후보 4건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 04/16 13:38:02**

**관측:** 원본 편차 **+0.0556무차원**, 85점·첫/끝 간격 84초. 1초 단발 여부: 아님.  
**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 OP-Z3도 기준을 넘었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 미확인. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 1일 22시간 34분 후에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 04/16 12:04~12:09**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.32**, 과거 중심 대비 **+7.81℃**.  
**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 2건(연속 2~5개 1분창, 절대 편차 2.7~6.25℃)이 있다. 이 구간의 5창·7.81℃는 비교 관측의 상한을 넘는다(절대 편차 7.81>6.25℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -81%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +10.7℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 12:28 (이동 Z 복귀보다 16분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z3 · 04/16 12:03~12:07**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **6.92**, 과거 중심 대비 **-77.7%p**.  
**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 12.2~56.4%p)이 있다. 이 구간의 4창·77.7%p는 비교 관측의 상한을 넘는다(절대 편차 77.7>56.4%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.79℃(10/4유효창). 같은 전후 기준의 OP-Z3 변화는 -81%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 12:15 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 7건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240038-Z-TC-Z3-0021 · 04/16 11:56~11:57**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 1건(연속 4~4개 1분창, 절대 편차 4.18~4.18℃)이 있다. 이 구간의 1창·2.51℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +12.8%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.53℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:01 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 2.508~2.508 | 1~1창 | 861.59 |
| 240114 | 1건/254유효창 (100창당 0.39건) | 4.179~4.179 | 4~4창 | 860.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-OP-Z3-0028 · 04/16 12:03~12:07**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 12.2~56.4%p)이 있다. 이 구간의 4창·77.7%p는 비교 관측의 상한을 넘는다(절대 편차 77.7>56.4%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.79℃(10/4유효창). 같은 전후 기준의 OP-Z3 변화는 -81%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:15 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |
| 240114 | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-TC-Z3-0029 · 04/16 12:04~12:09**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 2건(연속 2~5개 1분창, 절대 편차 2.7~6.25℃)이 있다. 이 구간의 5창·7.81℃는 비교 관측의 상한을 넘는다(절대 편차 7.81>6.25℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -81%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +10.7℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:28 (이동 Z 복귀보다 16분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 7.806~7.806 | 5~5창 | 861.59 |
| 240114 | 2건/254유효창 (100창당 0.79건) | 2.698~6.252 | 2~5창 | 860.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-OP-Z6-0042 · 04/16 12:17~12:19**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 8.89~8.89%p)이 있다. 이 구간의 2창·7.57%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.57℃(10/2유효창). 같은 전후 기준의 OP-Z6 변화는 +7.55%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:23 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/103유효창 (100창당 0.97건) | 7.567~7.567 | 2~2창 | 73.205 |
| 240114 | 1건/254유효창 (100창당 0.39건) | 8.888~8.888 | 3~3창 | 73.502 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-OP-Z3-0072 · 04/16 12:47~12:49**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 12.2~56.4%p)이 있다. 이 구간의 2창·43.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.49℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -41.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:52. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |
| 240114 | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-CP-01 · 04/16 13:38:02**

**다른 LOT:** 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 동일 1분에 OP-Z3도 기준을 넘었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 미확인. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00627 감지 1일 22시간 34분 후에 CP 원본 피크.

강종 SPCEN·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 1건/7342실측점 (1만점당 1.36건) | 0.05561~0.05561 | 84~84초 간격 | 0.45005 |
| 240114 | 0건/16592실측점 (1만점당 0.00건) | — | — | 0.45014 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240038-Z-OP-Z3-0123 · 04/16 13:38~13:39**

**다른 LOT:** 동종 LOT 240114에도 같은 방향 후보 6건(연속 1~4개 1분창, 절대 편차 12.2~56.4%p)이 있다. 이 구간의 1창·36.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 6건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.64℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -37.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240038 (현재 LOT) | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |
| 240114 | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 125 | 103 | 6 | 22 | 2 | 20 | 0 |
| OP-Z3 | 125 | 103 | 7 | 22 | 2 | 20 | 0 |
| TC-Z4 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| OP-Z4 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| TC-Z5 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| OP-Z5 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| TC-Z6 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| OP-Z6 | 125 | 103 | 2 | 22 | 2 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 1건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 103창, 최대 절대 Z=2.49 / OP-Z4: 유효 103창, 최대 절대 Z=2.40 / TC-Z5: 유효 103창, 최대 절대 Z=1.65 / OP-Z5: 유효 103창, 최대 절대 Z=1.66 / TC-Z6: 유효 103창, 최대 절대 Z=2.88. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| TC-Z3 · 04/16 11:56:00–04/16 11:57:00 (끝 제외); 240038-Z-TC-Z3-0021 | 최대 절대 Z=3.50 (04/16 11:56:30, 창 중심); 1분 중앙값 859.642℃, 과거 중심 862.150℃, 차이 -2.508℃; SD=0.716; 실측 57/60개·과거 21/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 12:00:00 / 시작 고정 기준 3창 복귀 04/16 12:01:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2692.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/16 12:03:00–04/16 12:07:00 (끝 제외); 240038-Z-OP-Z3-0028 | 최대 절대 Z=6.92 (04/16 12:04:30, 창 중심); 1분 중앙값 10.824%, 과거 중심 88.481%, 차이 -77.657%p; SD=11.219; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 12:10:00 / 시작 고정 기준 3창 복귀 04/16 12:15:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2700.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 04/16 12:04:00–04/16 12:09:00 (끝 제외); 240038-Z-TC-Z3-0029 | 최대 절대 Z=6.32 (04/16 12:05:30, 창 중심); 1분 중앙값 869.717℃, 과거 중심 861.911℃, 차이 +7.806℃; SD=1.235; 실측 57/60개·과거 30/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 12:12:00 / 시작 고정 기준 3창 복귀 04/16 12:28:00 | 고정 범위 복귀가 이동 Z 복귀보다 16분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2701.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 04/16 12:17:00–04/16 12:19:00 (끝 제외); 240038-Z-OP-Z6-0042 | 최대 절대 Z=3.78 (04/16 12:17:30, 창 중심); 1분 중앙값 81.221%, 과거 중심 73.653%, 차이 +7.567%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 12:22:00 / 시작 고정 기준 3창 복귀 04/16 12:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2713.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/16 12:47:00–04/16 12:49:00 (끝 제외); 240038-Z-OP-Z3-0072 | 최대 절대 Z=6.87 (04/16 12:47:30, 창 중심); 1분 중앙값 50.030%, 과거 중심 93.635%, 차이 -43.605%p; SD=6.351; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 04/16 12:52:00 / 시작 고정 기준 3창 복귀 04/16 12:52:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2743.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 04/16 13:38:00–04/16 13:39:00 (끝 제외); 240038-Z-OP-Z3-0123 | 최대 절대 Z=4.60 (04/16 13:38:30, 창 중심); 1분 중앙값 49.809%, 과거 중심 85.956%, 차이 -36.147%p; SD=7.863; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 짧은 국소 변화지만 고정 범위 복귀는 미확인. 원본의 폭·전후 값을 보존하며 1초 스파이크로 단정하지 않음; 정비 대조: WO-2024-00627 감지 대비 Z 최대창 중심 +2794.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240038-Z-TC-Z3-0029:** OP-Z3 사건창 중앙값−직전10분 중앙값 -80.97 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240038-Z-OP-Z3-0028:** TC-Z3 사건창 중앙값−직전10분 중앙값 +6.791 (10/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 04/16 11:35:00–04/16 11:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 과거 유효 최소 0창 <20 |
| 04/16 13:08:00–04/16 13:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 04/16 13:14:00–04/16 13:15:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240038-CP-01 · 04/16 13:38:02 | +0.0556무차원 · 85점/84초 간격 | 같은 강종 LOT 240114의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다. | 60개 연속 실제 1초 값 복귀 미확인; WO-2024-00627 감지 1일 22시간 34분 후에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240038-CP-01 · 피크 04/16 13:38:02 · 복귀 미확인 원본 이탈**

- **관측:** 원값 0.505652, LOT 중앙값 0.450046, 편차 +0.055606. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 85점, 첫·끝 실측 간격 84초, 기대 1초 격자 내 유효 비율 100.0%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 04/16 13:37:42=0.468133 / 직후 04/16 13:39:08=0.469612; 이후 60개 연속 실측 ±0.02 복귀 확인 미확인.
- **같은 1분의 결과:** OP-Z3 -4.60.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 114686–114772에 연결했다(0부터 시작).

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240038-CP-01 | 정비 대조: WO-2024-00627 감지 대비 CP 원본 피크 +2794.03분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +1.12 (296/61점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -26.79 (296/61점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +3.54e-13 (296/61점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240038, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| OP-Z7 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| TC-Z8 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |
| OP-Z8 | 125 | 103 | 0 | 22 | 2 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 103창, 최대 절대 Z=0.63 / OP-Z7: 유효 103창, 최대 절대 Z=2.90 / TC-Z8: 유효 103창, 최대 절대 Z=0.54 / OP-Z8: 유효 103창, 최대 절대 Z=1.33. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 04/16 11:35:00–04/16 11:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 04/16 13:08:00–04/16 13:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 04/16 13:14:00–04/16 13:15:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240038)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **1건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 7342 | 2 | 원본 ±0.05 사건 1건 · 최대 절대 편차 0.055606 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 7341 | 2 | 원본 중앙값 1.15048e-10, 최저 1.14002e-10, 최고 1.16618e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 7342 | 2 | 원본 중앙값 334.264, 최저 321.845, 최고 342.294; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 7341 | 2 | 원본 중앙값 285.402, 최저 271.378, 최고 298.572; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 7341 | 2 | 원본 중앙값 283.054, 최저 272.66, 최고 290.839; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-00627 · CBM / 메모: 확인 중**

**실제 시각:** 감지 04/14 15:04:00 / 작업 04/23 15:27:00~04/23 18:29:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-00627 감지 1일 22시간 34분 후에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 작업 또는 제어 조건 변경과 동반됐을 가능성을 검토한다. 같은 시각이라는 이유만으로 정비가 원인이라고 판단하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00627 · CBM | 04/14 15:04:00 | 04/23 15:27:00 | 04/23 18:29:00 | 확인 중 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-00627 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 10건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240038-CP-01 · 04/16 13:38:02 · CP 편차 +0.055606 · 85점/84초 · 완료 대비 피크 -10370.97분 · 감지는 피크 대비 -2794.03분. 복귀 미확인 원본 이탈; 직전 04/16 13:37:42=0.468133 / 직후 04/16 13:39:08=0.469612; 이후 60개 연속 실측 ±0.02 복귀 확인 미확인.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.22 / OP-Z1 1건·최대 절대 Z 4.94 / OP-Z2 1건·최대 절대 Z 4.15 / OP-Z3 3건·최대 절대 Z 6.92 / OP-Z6 1건·최대 절대 Z 3.78 / TC-Z1 1건·최대 절대 Z 3.16 / TC-Z3 2건·최대 절대 Z 6.32. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 04/14 15:04:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 04/23 15:27:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 04/23 18:29:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240038)

**240038 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240038)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def draw_lot_level(k):
    if k not in LEVEL_TARGETS:return
    grade=lot_meta.loc[k,'GRADE'];zones=sorted({degree_zone(s) for s in LEVEL_TARGETS[k]})
    ids=lots.loc[lots.GRADE.eq(grade)].sort_values('START').LOT_NO.astype(int).tolist()
    rows=[]
    for z in zones:
        rows.append({'panels':[(f'TC-Z{z} · 평균·중앙값·IQR',[f'TC-Z{z}']),(f'OP-Z{z} · 평균·중앙값·IQR',[f'OP-Z{z}'])],'height':1.8})
        rows.append({'panels':[(f'TC-Z{z} · 전환 제외 실제 추세',[f'TC-Z{z}']),(f'OP-Z{z} · 전환 제외 실제 추세',[f'OP-Z{z}'])],'height':1.5})
    fig,cells=report_page(f'{k} · {grade} | ③ 같은 강종 분포 → 실제 추세',
        '신호별 상자와 실제 추세를 함께 본다. 파랑=이 LOT · 회색=다른 LOT. 수리 전환은 온도값 대신 축 아래 회색 표식으로 표시한다.',rows,
        info=['IQR 수염 밖 점은 해당 LOT 내부 분포 기준이다. LOT 간 수준 차이는 같은 강종의 상자 위치·중앙값으로 따로 확인한다.',
              f'상세 신호: 기존 16신호 차이에서 |TC| ≥ {LEVEL_TC_LIMIT:g}℃ 또는 |OP| ≥ {LEVEL_OP_LIMIT:g}%p인 신호와 대응 TC·OP. 고장 한계가 아님.'])
    for i,z in enumerate(zones):
        for ax,head,sensors in cells[2*i]:draw_boxes(ax,head,sensors[0],ids,highlight=k)
        for ax,head,sensors in cells[2*i+1]:
            sensor=sensors[0]
            for kk in ids:
                y=profile[kk][(sensor,'median')].where(comparison_states[kk].eq('안정 참고')&profile[kk][(sensor,'count')].ge(54))
                ax.plot(y.index,y,color='#086788' if kk==k else '#8496a5',lw=1.5 if kk==k else .9,label=str(kk))
                excluded=profile[kk].index[comparison_states[kk].ne('안정 참고')]
                ax.plot(excluded,np.full(len(excluded),.025),transform=ax.get_xaxis_transform(),color='#bec7ce',marker='|',ms=3,lw=0)
            limits=iqr_summary.loc[iqr_summary.LOT.eq(k)&iqr_summary.SENSOR.eq(sensor)].iloc[0]
            y=profile[k][(sensor,'median')];mask=iqr_masks[k,sensor]
            ax.scatter(y.index[mask],y[mask],s=10,color='#086788',zorder=5)
            ax.set_ylabel(display_unit(sensor));ax.set_xlabel('각 LOT 관측 시작 후 경과시간[분]');ax.xaxis.set_major_locator(MaxNLocator(5));panel_legend(head,ax)
    fig._folded=f'{k} {grade}: 동일 강종 분포·IQR·실제 추세와 전환 제외 범위 펼치기'
    emit(fig,f'03_상세/03_강종비교/{k}_{grade}_수준대조.png')
    table(iqr_summary.loc[iqr_summary.LOT.isin(ids)])
    note('**이 그림에서 확인할 것:** 다른 LOT보다 상자 전체가 이동했는지, 상자 폭이 커졌는지, 일부 1분창만 수염 밖인지 구분한다. 같은 강종이어도 설정·부하·보정이 같다는 증거는 없어 차이를 고장으로 확정하지 않는다.')

draw_lot_level(240038)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240038(SPCEN)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 330.187 - 대조 333.233 = **-3.046℃**, 다른 1 LOT과 대조.
- TC-Z8: 이 LOT 330.200 - 대조 333.607 = **-3.407℃**, 다른 1 LOT과 대조.
- OP-Z4: 이 LOT 70.460 - 대조 53.331 = **+17.129%p**, 다른 1 LOT과 대조.
- OP-Z5: 이 LOT 58.724 - 대조 54.505 = **+4.219%p**, 다른 1 LOT과 대조.
- OP-Z7: 이 LOT 71.180 - 대조 66.910 = **+4.270%p**, 다른 1 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 -0.058℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- TC-Z5: 차이 -0.148℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 +3.022%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240038: 전환 제외 없음.
- 240114: 11/04 20:08–20:59.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.
같은 강종이 두 LOT뿐이므로 차이의 방향은 대칭적이다. 어느 LOT이 정상인지 정할 수 없다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240038)
lot_conclusion(240038)
lot_execution.append({'LOT':240038,'확대 구간':len(LOT_EVENTS[240038]),'강종 대조':240038 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240044"></a>
### 8.5 LOT 240044 · SPHC · 05/03

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240044)

#### LOT 240044 · 먼저 볼 변화

**지속·반복 후보 4건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — OP-Z2 · 05/03 19:45~19:51**

**관측:** |Z|≥3 연속 **6개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.40**, 과거 중심 대비 **-10.7%p**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 5건(연속 1~6개 1분창, 절대 편차 6.68~9.56%p)이 있다. 이 구간의 6창·10.7%p는 비교 관측의 상한을 넘는다(절대 편차 10.7>9.56%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.28℃(10/6유효창). 같은 전후 기준의 OP-Z2 변화는 -10.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 19:59 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z1 · 05/03 19:44~19:49**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.28**, 과거 중심 대비 **-10.6%p**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 5건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·10.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.72℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 19:55. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z3 · 05/03 19:44~19:46**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **3.45**, 과거 중심 대비 **-52.6%p**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 32건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·52.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 32건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.04℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -43.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 19:55 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240044

- **관측 범위:** 05/03 18:50:00–05/03 20:04:38.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 39개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 4개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 24개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240044, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 3건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z2 · 05/03 19:45~19:51**

**관측:** |Z|≥3 연속 **6개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.40**, 과거 중심 대비 **-10.7%p**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 5건(연속 1~6개 1분창, 절대 편차 6.68~9.56%p)이 있다. 이 구간의 6창·10.7%p는 비교 관측의 상한을 넘는다(절대 편차 10.7>9.56%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.28℃(10/6유효창). 같은 전후 기준의 OP-Z2 변화는 -10.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 19:59 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z1 · 05/03 19:44~19:49**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.28**, 과거 중심 대비 **-10.6%p**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 5건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·10.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.72℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 19:55. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 3건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240044-Z-OP-Z1-0054 · 05/03 19:44~19:49**

**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 5건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·10.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.72℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 19:55. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/53유효창 (100창당 1.89건) | 10.55~10.55 | 5~5창 | 74.266 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.656~12.58 | 2~3창 | 72.702 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 10.54~11.55 | 5~5창 | 72.861 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 9.204~9.204 | 5~5창 | 73.431 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240044-Z-OP-Z2-0055 · 05/03 19:45~19:51**

**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 5건(연속 1~6개 1분창, 절대 편차 6.68~9.56%p)이 있다. 이 구간의 6창·10.7%p는 비교 관측의 상한을 넘는다(절대 편차 10.7>9.56%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.28℃(10/6유효창). 같은 전후 기준의 OP-Z2 변화는 -10.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 19:59 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/53유효창 (100창당 1.89건) | 10.68~10.68 | 6~6창 | 25.212 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.682~9.561 | 1~3창 | 19.189 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 8.473~8.846 | 4~5창 | 21.399 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 8.744~8.744 | 6~6창 | 22.73 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240044-Z-TC-Z1-0056 · 05/03 19:46~19:49**

**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 2건(연속 2~3개 1분창, 절대 편차 1.72~1.8℃)이 있다. 이 구간의 3창·1.91℃는 비교 관측의 상한을 넘는다(절대 편차 1.91>1.8℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -13.8%p(10/3유효창). 같은 전후 기준의 TC-Z1 변화는 +1.77℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 19:55. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/52유효창 (100창당 1.92건) | 1.907~1.907 | 3~3창 | 99.946 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 99.97 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 1.717~1.799 | 2~3창 | 99.998 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 100 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 75 | 52 | 0 | 23 | 3 | 20 | 0 |
| TC-Z1 | 75 | 52 | 3 | 23 | 3 | 20 | 0 |
| OP-Z1 | 75 | 53 | 5 | 22 | 2 | 20 | 0 |
| TC-Z2 | 75 | 52 | 0 | 23 | 3 | 20 | 0 |
| OP-Z2 | 75 | 53 | 6 | 22 | 2 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 52창, 최대 절대 Z=1.37 / TC-Z2: 유효 52창, 최대 절대 Z=2.73. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 05/03 19:44:00–05/03 19:49:00 (끝 제외); 240044-Z-OP-Z1-0054 | 최대 절대 Z=5.28 (05/03 19:45:30, 창 중심); 1분 중앙값 63.884%, 과거 중심 74.436%, 차이 -10.552%p; SD=2.000 (하한 적용); 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 19:55:00 / 시작 고정 기준 3창 복귀 05/03 19:55:00 | 같은 방향의 초과가 최장 5창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -240.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 05/03 19:45:00–05/03 19:51:00 (끝 제외); 240044-Z-OP-Z2-0055 | 최대 절대 Z=4.40 (05/03 19:47:30, 창 중심); 1분 중앙값 13.892%, 과거 중심 24.572%, 차이 -10.680%p; SD=2.427; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 6창, 같은 방향 최장 6창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 19:55:00 / 시작 고정 기준 3창 복귀 05/03 19:59:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -238.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 05/03 19:46:00–05/03 19:49:00 (끝 제외); 240044-Z-TC-Z1-0056 | 최대 절대 Z=3.81 (05/03 19:47:30, 창 중심); 1분 중앙값 101.931℃, 과거 중심 100.023℃, 차이 +1.907℃; SD=0.500 (하한 적용); 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 19:55:00 / 시작 고정 기준 3창 복귀 05/03 19:55:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -238.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240044-Z-OP-Z2-0055:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.281 (10/6유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240044-Z-OP-Z1-0054:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.716 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 05/03 18:50:00–05/03 19:11:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 / 과거 유효 최소 0창 <20 |
| 05/03 19:51:00–05/03 19:52:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 05/03 20:04:00–05/03 20:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 39개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240044, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 1건 · 짧은 노랑 후보 4건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 주황 · 지속·반복 확인 — TC-Z3 · 05/03 19:52~19:55**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.30**, 과거 중심 대비 **-9.5℃**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 15건(연속 1~3개 1분창, 절대 편차 2.28~5.86℃)이 있다. 이 구간의 3창·9.5℃는 비교 관측의 상한을 넘는다(절대 편차 9.5>5.86℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +64.9%p(9/3유효창). 같은 전후 기준의 TC-Z3 변화는 -11.9℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 20:01 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 복귀 추가 확인 — OP-Z3 · 05/03 19:44~19:46**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **3.45**, 과거 중심 대비 **-52.6%p**.  
**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 32건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·52.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 32건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.04℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -43.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 19:55 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 5건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240044-Z-OP-Z3-0054 · 05/03 19:44~19:46**

**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 32건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·52.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 32건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.04℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -43.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:55 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240044-Z-TC-Z3-0055 · 05/03 19:45~19:46**

**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 16건(연속 1~5개 1분창, 절대 편차 2.45~11.1℃)이 있다. 이 구간의 1창·3.17℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 16건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -41.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.99℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:51 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240044-Z-TC-Z3-0062 · 05/03 19:52~19:55**

**다른 LOT:** 동종 LOT 240061, 240069, 240091에도 같은 방향 후보 15건(연속 1~3개 1분창, 절대 편차 2.28~5.86℃)이 있다. 이 구간의 3창·9.5℃는 비교 관측의 상한을 넘는다(절대 편차 9.5>5.86℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +64.9%p(9/3유효창). 같은 전후 기준의 TC-Z3 변화는 -11.9℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 20:01 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240044-Z-OP-Z4-0066 · 05/03 19:56~19:58**

**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 6.95~6.95%p)이 있다. 이 구간의 2창·7.68%p는 비교 관측의 상한을 넘는다(절대 편차 7.68>6.95%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 -1.69℃(9/2유효창). 같은 전후 기준의 OP-Z4 변화는 +7.94%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 20:03 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/53유효창 (100창당 1.89건) | 7.678~7.678 | 2~2창 | 41.819 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 54.132 |
| 240069 | 1건/737유효창 (100창당 0.14건) | 6.946~6.946 | 3~3창 | 42.121 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 44.37 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240044-Z-TC-Z4-0067 · 05/03 19:57~19:58**

**다른 LOT:** 같은 강종 LOT 240061, 240069, 240091의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 +8.14%p(9/1유효창). 같은 전후 기준의 TC-Z4 변화는 -1.81℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 20:01. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240044 (현재 LOT) | 1건/52유효창 (100창당 1.92건) | 1.574~1.574 | 1~1창 | 859.65 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 859.8 |
| 240069 | 0건/737유효창 (100창당 0.00건) | — | — | 859.66 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 859.69 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 75 | 53 | 4 | 22 | 2 | 20 | 0 |
| OP-Z3 | 75 | 51 | 2 | 24 | 4 | 20 | 0 |
| TC-Z4 | 75 | 52 | 1 | 23 | 3 | 20 | 0 |
| OP-Z4 | 75 | 53 | 2 | 22 | 2 | 20 | 0 |
| TC-Z5 | 75 | 53 | 0 | 22 | 2 | 20 | 0 |
| OP-Z5 | 75 | 53 | 0 | 22 | 2 | 20 | 0 |
| TC-Z6 | 75 | 52 | 0 | 23 | 3 | 20 | 0 |
| OP-Z6 | 75 | 53 | 0 | 22 | 2 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z5: 유효 53창, 최대 절대 Z=1.19 / OP-Z5: 유효 53창, 최대 절대 Z=1.63 / TC-Z6: 유효 52창, 최대 절대 Z=2.42 / OP-Z6: 유효 53창, 최대 절대 Z=2.77. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 05/03 19:44:00–05/03 19:46:00 (끝 제외); 240044-Z-OP-Z3-0054 | 최대 절대 Z=3.45 (05/03 19:45:30, 창 중심); 1분 중앙값 24.269%, 과거 중심 76.866%, 차이 -52.598%p; SD=15.236; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 19:49:00 / 시작 고정 기준 3창 복귀 05/03 19:55:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -240.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 05/03 19:45:00–05/03 19:46:00 (끝 제외); 240044-Z-TC-Z3-0055 | 최대 절대 Z=3.21 (05/03 19:45:30, 창 중심); 1분 중앙값 864.240℃, 과거 중심 861.073℃, 차이 +3.167℃; SD=0.986; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 19:49:00 / 시작 고정 기준 3창 복귀 05/03 19:51:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -240.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 05/03 19:52:00–05/03 19:55:00 (끝 제외); 240044-Z-TC-Z3-0062 | 최대 절대 Z=6.30 (05/03 19:52:30, 창 중심); 1분 중앙값 851.731℃, 과거 중심 861.232℃, 차이 -9.501℃; SD=1.509; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 19:58:00 / 시작 고정 기준 3창 복귀 05/03 20:01:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -233.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 05/03 19:56:00–05/03 19:58:00 (끝 제외); 240044-Z-OP-Z4-0066 | 최대 절대 Z=3.84 (05/03 19:57:30, 창 중심); 1분 중앙값 49.497%, 과거 중심 41.819%, 차이 +7.678%p; SD=2.000 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 20:01:00 / 시작 고정 기준 3창 복귀 05/03 20:03:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -228.50분 (근접 시각이며 인과 미확정) |
| TC-Z4 · 05/03 19:57:00–05/03 19:58:00 (끝 제외); 240044-Z-TC-Z4-0067 | 최대 절대 Z=3.15 (05/03 19:57:30, 창 중심); 1분 중앙값 858.114℃, 과거 중심 859.688℃, 차이 -1.574℃; SD=0.500 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 05/03 20:01:00 / 시작 고정 기준 3창 복귀 05/03 20:01:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00717 감지 대비 Z 최대창 중심 -228.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240044-Z-TC-Z3-0062:** OP-Z3 사건창 중앙값−직전10분 중앙값 +64.91 (9/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 05/03 18:50:00–05/03 19:12:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 0개 <54 / 과거 유효 최소 0창 <20 |
| 05/03 19:51:00–05/03 19:52:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 05/03 20:04:00–05/03 20:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 39개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240044, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 75 | 52 | 0 | 23 | 3 | 20 | 0 |
| OP-Z7 | 75 | 52 | 0 | 23 | 3 | 20 | 0 |
| TC-Z8 | 75 | 52 | 0 | 23 | 3 | 20 | 0 |
| OP-Z8 | 75 | 52 | 0 | 23 | 3 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 52창, 최대 절대 Z=0.45 / OP-Z7: 유효 52창, 최대 절대 Z=1.97 / TC-Z8: 유효 52창, 최대 절대 Z=0.30 / OP-Z8: 유효 52창, 최대 절대 Z=0.67. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 05/03 18:50:00–05/03 19:11:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 46개 <54 / 과거 유효 최소 0창 <20 |
| 05/03 19:51:00–05/03 19:52:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 05/03 20:04:00–05/03 20:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 39개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240044)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 4334 | 2 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.03117 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 4334 | 2 | 원본 중앙값 1.14066e-10, 최저 1.13073e-10, 최고 1.15165e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 4334 | 2 | 원본 중앙값 337.715, 최저 326.105, 최고 342.219; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 4334 | 2 | 원본 중앙값 282.244, 최저 270.529, 최고 297.096; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 4333 | 3 | 원본 중앙값 282.363, 최저 271.251, 최고 289.909; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-00717 · BM / 메모: 동작 불량**

**실제 시각:** 감지 05/03 23:46:00 / 작업 05/04 04:50:00~05/04 08:15:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-00717 감지 3시간 58분 30초 전에 Z 최대창 중심.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00717 · BM | 05/03 23:46:00 | 05/04 04:50:00 | 05/04 08:15:00 | 동작 불량 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-00717 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 8건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.03117 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** OP-Z1 1건·최대 절대 Z 5.28 / OP-Z2 1건·최대 절대 Z 4.40 / OP-Z3 1건·최대 절대 Z 3.45 / OP-Z4 1건·최대 절대 Z 3.84 / TC-Z1 1건·최대 절대 Z 3.81 / TC-Z3 2건·최대 절대 Z 6.30 / TC-Z4 1건·최대 절대 Z 3.15. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 05/03 23:46:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 05/04 04:50:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 05/04 08:15:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240044)

**240044 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240044)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240044)
lot_conclusion(240044)
lot_execution.append({'LOT':240044,'확대 구간':len(LOT_EVENTS[240044]),'강종 대조':240044 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240058"></a>
### 8.6 LOT 240058 · DP590 · 06/09

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240058)

#### LOT 240058 · 먼저 볼 변화

**지속·반복 후보 21건 / CP 원본 후보 1건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 06/09 09:02:35**

**관측:** 원본 편차 **-0.3511무차원**, 5238점·첫/끝 간격 5416초. 1초 단발 여부: 아님.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다 (계산 보류: TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6). CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 미확인. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00916 감지 21시간 20분 25초 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 06/09 08:10~08:23**

**관측:** |Z|≥3 연속 **13개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.49**, 과거 중심 대비 **-4.06℃**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.391%p(10/13유효창). 같은 전후 기준의 TC-Z3 변화는 -3.99℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z6 · 06/09 08:53~09:02**

**관측:** |Z|≥3 연속 **9개 1분창**, 최근 30분 시작 2회. 최대 |Z| **10.22**, 과거 중심 대비 **-20.4%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -10.4℃(8/9유효창). 같은 전후 기준의 OP-Z6 변화는 -44.4%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240058

- **관측 범위:** 06/09 05:26:00–06/09 09:02:53.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 130개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 21개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 31개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 1건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240058, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 3건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 06/09 08:53~09:00**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 3회. 최대 |Z| **4.14**, 과거 중심 대비 **-37.1%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 -2.71℃(8/7유효창). 같은 전후 기준의 OP-Z1 변화는 -29.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 06/09 08:44~08:51**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.63**, 과거 중심 대비 **-9.78%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.416℃(9/7유효창). 같은 전후 기준의 OP-Z2 변화는 -11.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z1 · 06/09 08:42~08:43**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.29**, 과거 중심 대비 **-6.58%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.527℃(10/1유효창). 같은 전후 기준의 OP-Z1 변화는 -6.79%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 7건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240058-Z-OP-Z1-0196 · 06/09 08:42~08:43**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.527℃(10/1유효창). 같은 전후 기준의 OP-Z1 변화는 -6.79%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z1-0198 · 06/09 08:44~08:48**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.11℃(9/4유효창). 같은 전후 기준의 OP-Z1 변화는 -15.4%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z2-0198 · 06/09 08:44~08:51**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.416℃(9/7유효창). 같은 전후 기준의 OP-Z2 변화는 -11.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z1-0205 · 06/09 08:51~08:52**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -2.2%p(9/1유효창). 같은 전후 기준의 TC-Z1 변화는 -1.71℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z1-0207 · 06/09 08:53~09:00**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 -2.71℃(8/7유효창). 같은 전후 기준의 OP-Z1 변화는 -29.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z1-0207 · 06/09 08:53~08:55**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -14.6%p(8/2유효창). 같은 전후 기준의 TC-Z1 변화는 -2.71℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z2-0208 · 06/09 08:54~09:00**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z2의 사건창 중앙값−직전 10분 중앙값은 -4.88%p(9/6유효창). 같은 전후 기준의 TC-Z2 변화는 -2.22℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 217 | 186 | 0 | 31 | 11 | 20 | 0 |
| TC-Z1 | 217 | 186 | 3 | 31 | 11 | 20 | 0 |
| OP-Z1 | 217 | 186 | 12 | 31 | 11 | 20 | 0 |
| TC-Z2 | 217 | 186 | 6 | 31 | 11 | 20 | 0 |
| OP-Z2 | 217 | 186 | 7 | 31 | 11 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 186창, 최대 절대 Z=2.16. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 06/09 08:42:00–06/09 08:43:00 (끝 제외); 240058-Z-OP-Z1-0196 | 최대 절대 Z=3.29 (06/09 08:42:30, 창 중심); 1분 중앙값 67.955%, 과거 중심 74.533%, 차이 -6.578%p; SD=2.000 (하한 적용); 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:51:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1300.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 06/09 08:44:00–06/09 08:48:00 (끝 제외); 240058-Z-OP-Z1-0198 | 최대 절대 Z=6.26 (06/09 08:44:30, 창 중심); 1분 중앙값 62.032%, 과거 중심 74.546%, 차이 -12.514%p; SD=2.000 (하한 적용); 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:51:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1298.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 06/09 08:44:00–06/09 08:51:00 (끝 제외); 240058-Z-OP-Z2-0198 | 최대 절대 Z=4.63 (06/09 08:46:30, 창 중심); 1분 중앙값 14.103%, 과거 중심 23.881%, 차이 -9.778%p; SD=2.114; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:56:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1296.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 06/09 08:51:00–06/09 08:52:00 (끝 제외); 240058-Z-TC-Z1-0205 | 최대 절대 Z=3.37 (06/09 08:51:30, 창 중심); 1분 중앙값 97.139℃, 과거 중심 98.847℃, 차이 -1.709℃; SD=0.507; 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:58:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1291.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 06/09 08:53:00–06/09 09:00:00 (끝 제외); 240058-Z-OP-Z1-0207 | 최대 절대 Z=4.14 (06/09 08:55:30, 창 중심); 1분 중앙값 35.919%, 과거 중심 72.974%, 차이 -37.056%p; SD=8.940; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1287.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 06/09 08:53:00–06/09 08:55:00 (끝 제외); 240058-Z-TC-Z1-0207 | 최대 절대 Z=4.09 (06/09 08:53:30, 창 중심); 1분 중앙값 96.401℃, 과거 중심 98.782℃, 차이 -2.381℃; SD=0.583; 실측 58/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:58:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |
| TC-Z2 · 06/09 08:54:00–06/09 09:00:00 (끝 제외); 240058-Z-TC-Z2-0208 | 최대 절대 Z=3.69 (06/09 08:55:30, 창 중심); 1분 중앙값 96.733℃, 과거 중심 98.578℃, 차이 -1.846℃; SD=0.500 (하한 적용); 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 6창, 같은 방향 최장 6창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1287.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240058-Z-OP-Z1-0207:** TC-Z1 사건창 중앙값−직전10분 중앙값 -2.712 (8/7유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240058-Z-OP-Z2-0198:** TC-Z2 사건창 중앙값−직전10분 중앙값 +0.416 (9/7유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 06/09 05:26:00–06/09 05:48:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 / 과거 유효 최소 0창 <20 |
| 06/09 05:52:00–06/09 05:53:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/09 06:21:00–06/09 06:22:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 06/09 06:54:00–06/09 06:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/09 07:31:00–06/09 07:33:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 06/09 07:57:00–06/09 07:58:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/09 08:43:00–06/09 08:44:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/09 08:52:00–06/09 08:53:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/09 09:02:00–06/09 09:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240058, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 13건 · 짧은 노랑 후보 9건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 06/09 09:02:35**

**관측:** 원본 편차 **-0.3511무차원**, 5238점·첫/끝 간격 5416초. 1초 단발 여부: 아님.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다 (계산 보류: TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6). CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 미확인. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00916 감지 21시간 20분 25초 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 06/09 08:10~08:23**

**관측:** |Z|≥3 연속 **13개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.49**, 과거 중심 대비 **-4.06℃**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.391%p(10/13유효창). 같은 전후 기준의 TC-Z3 변화는 -3.99℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z6 · 06/09 08:53~09:02**

**관측:** |Z|≥3 연속 **9개 1분창**, 최근 30분 시작 2회. 최대 |Z| **10.22**, 과거 중심 대비 **-20.4%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -10.4℃(8/9유효창). 같은 전후 기준의 OP-Z6 변화는 -44.4%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 23건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240058-Z-TC-Z3-0024 · 06/09 05:50~05:52**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +16.3%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.32℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:56. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z3-0074 · 06/09 06:40~06:42**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.76℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -59.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:46 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z3-0075 · 06/09 06:41~06:43**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -59.1%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.19℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:54 (이동 Z 복귀보다 5분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z3-0079 · 06/09 06:45~06:46**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +21.9%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -5.36℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:51 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z3-0118 · 06/09 07:24~07:28**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.16℃(10/4유효창). 같은 전후 기준의 OP-Z3 변화는 -79.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:38 (이동 Z 복귀보다 7분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z3-0118 · 06/09 07:24~07:30**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -79.6%p(10/6유효창). 같은 전후 기준의 TC-Z3 변화는 +9.81℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:38 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-CP-01 · 06/09 09:02:35**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다 (계산 보류: TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6). CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 미확인. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-00916 감지 21시간 20분 25초 전에 CP 원본 피크.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z6-0128 · 06/09 07:34~07:35**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.856℃(8/1유효창). 같은 전후 기준의 OP-Z6 변화는 +5.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:47 (이동 Z 복귀보다 5분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z6-0130 · 06/09 07:36~07:39**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.13℃(8/3유효창). 같은 전후 기준의 OP-Z6 변화는 +8.13%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 07:47 (이동 Z 복귀보다 5분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z6-0132 · 06/09 07:38~07:39**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 +5.03%p(8/1유효창). 같은 전후 기준의 TC-Z6 변화는 -0.712℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 07:42. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z4-0145 · 06/09 07:51~07:55**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.82℃(10/4유효창). 같은 전후 기준의 OP-Z4 변화는 -8.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z4-0145 · 06/09 07:51~07:54**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -8.68%p(10/3유효창). 같은 전후 기준의 TC-Z4 변화는 +1.78℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 08:02 (이동 Z 복귀보다 5분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z5-0161 · 06/09 08:07~08:15**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z5의 사건창 중앙값−직전 10분 중앙값은 +3.44%p(9/8유효창). 같은 전후 기준의 TC-Z5 변화는 -2.34℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z3-0164 · 06/09 08:10~08:23**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.391%p(10/13유효창). 같은 전후 기준의 TC-Z3 변화는 -3.99℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z6-0166 · 06/09 08:12~08:19**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.644%p(10/7유효창). 같은 전후 기준의 TC-Z6 변화는 -3.1℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z6-0173 · 06/09 08:19~08:20**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.47℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 -10%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 08:23. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z6-0176 · 06/09 08:22~08:25**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.406%p(10/3유효창). 같은 전후 기준의 TC-Z6 변화는 -3.95℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z3-0205 · 06/09 08:51~08:52**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -5.29℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -7.09%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z6-0205 · 06/09 08:51~08:52**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -5.26℃(9/1유효창). 같은 전후 기준의 OP-Z6 변화는 -6.97%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z3-0207 · 06/09 08:53~09:02**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -10.4℃(8/9유효창). 같은 전후 기준의 OP-Z3 변화는 -36%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z4-0207 · 06/09 08:53~09:02**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 -10.3℃(8/9유효창). 같은 전후 기준의 OP-Z4 변화는 -30.9%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z5-0207 · 06/09 08:53~09:02**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z5의 사건창 중앙값−직전 10분 중앙값은 -10.1℃(8/9유효창). 같은 전후 기준의 OP-Z5 변화는 -30.4%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z6-0207 · 06/09 08:53~09:02**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -10.4℃(8/9유효창). 같은 전후 기준의 OP-Z6 변화는 -44.4%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 217 | 186 | 24 | 31 | 11 | 20 | 0 |
| OP-Z3 | 217 | 186 | 16 | 31 | 11 | 20 | 0 |
| TC-Z4 | 217 | 186 | 3 | 31 | 11 | 20 | 0 |
| OP-Z4 | 217 | 186 | 13 | 31 | 11 | 20 | 0 |
| TC-Z5 | 217 | 186 | 8 | 31 | 11 | 20 | 0 |
| OP-Z5 | 217 | 186 | 9 | 31 | 11 | 20 | 0 |
| TC-Z6 | 217 | 186 | 11 | 31 | 11 | 20 | 0 |
| OP-Z6 | 217 | 186 | 15 | 31 | 11 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 1건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| TC-Z3 · 06/09 05:50:00–06/09 05:52:00 (끝 제외); 240058-Z-TC-Z3-0024 | 최대 절대 Z=3.54 (06/09 05:51:30, 창 중심); 1분 중앙값 858.031℃, 과거 중심 860.515℃, 차이 -2.484℃; SD=0.701; 실측 58/60개·과거 23/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 05:56:00 / 시작 고정 기준 3창 복귀 06/09 05:56:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1471.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/09 06:40:00–06/09 06:42:00 (끝 제외); 240058-Z-OP-Z3-0074 | 최대 절대 Z=6.58 (06/09 06:40:30, 창 중심); 1분 중앙값 36.132%, 과거 중심 85.803%, 차이 -49.671%p; SD=7.544; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 06:45:00 / 시작 고정 기준 3창 복귀 06/09 06:46:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1422.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/09 06:41:00–06/09 06:43:00 (끝 제외); 240058-Z-TC-Z3-0075 | 최대 절대 Z=7.43 (06/09 06:41:30, 창 중심); 1분 중앙값 864.920℃, 과거 중심 860.486℃, 차이 +4.434℃; SD=0.597; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 06:49:00 / 시작 고정 기준 3창 복귀 06/09 06:54:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1421.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/09 06:45:00–06/09 06:46:00 (끝 제외); 240058-Z-TC-Z3-0079 | 최대 절대 Z=3.13 (06/09 06:45:30, 창 중심); 1분 중앙값 856.125℃, 과거 중심 860.509℃, 차이 -4.384℃; SD=1.401; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 06:49:00 / 시작 고정 기준 3창 복귀 06/09 06:51:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1417.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/09 07:24:00–06/09 07:28:00 (끝 제외); 240058-Z-OP-Z3-0118 | 최대 절대 Z=6.74 (06/09 07:25:30, 창 중심); 1분 중앙값 7.979%, 과거 중심 84.680%, 차이 -76.700%p; SD=11.385; 실측 59/60개·과거 30/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 07:31:00 / 시작 고정 기준 3창 복귀 06/09 07:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 7분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1377.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/09 07:24:00–06/09 07:30:00 (끝 제외); 240058-Z-TC-Z3-0118 | 최대 절대 Z=8.66 (06/09 07:25:30, 창 중심); 1분 중앙값 865.904℃, 과거 중심 860.444℃, 차이 +5.460℃; SD=0.630; 실측 59/60개·과거 30/30창 | 지속·반복 후보: 연속 6창, 같은 방향 최장 6창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 07:36:00 / 시작 고정 기준 3창 복귀 06/09 07:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1377.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 06/09 07:34:00–06/09 07:35:00 (끝 제외); 240058-Z-OP-Z6-0128 | 최대 절대 Z=3.06 (06/09 07:34:30, 창 중심); 1분 중앙값 77.338%, 과거 중심 71.210%, 차이 +6.128%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 07:42:00 / 시작 고정 기준 3창 복귀 06/09 07:47:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1368.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 06/09 07:36:00–06/09 07:39:00 (끝 제외); 240058-Z-OP-Z6-0130 | 최대 절대 Z=3.81 (06/09 07:37:30, 창 중심); 1분 중앙값 79.974%, 과거 중심 71.405%, 차이 +8.568%p; SD=2.250; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 07:42:00 / 시작 고정 기준 3창 복귀 06/09 07:47:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1365.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 06/09 07:38:00–06/09 07:39:00 (끝 제외); 240058-Z-TC-Z6-0132 | 최대 절대 Z=3.00 (06/09 07:38:30, 창 중심); 1분 중앙값 859.041℃, 과거 중심 860.542℃, 차이 -1.501℃; SD=0.500 (하한 적용); 실측 58/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 07:42:00 / 시작 고정 기준 3창 복귀 06/09 07:42:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1364.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 06/09 07:51:00–06/09 07:55:00 (끝 제외); 240058-Z-OP-Z4-0145 | 최대 절대 Z=3.39 (06/09 07:52:30, 창 중심); 1분 중앙값 49.211%, 과거 중심 56.824%, 차이 -7.613%p; SD=2.244; 실측 57/60개·과거 28/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:01:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1350.50분 (근접 시각이며 인과 미확정) |
| TC-Z4 · 06/09 07:51:00–06/09 07:54:00 (끝 제외); 240058-Z-TC-Z4-0145 | 최대 절대 Z=3.54 (06/09 07:52:30, 창 중심); 1분 중앙값 861.970℃, 과거 중심 860.200℃, 차이 +1.770℃; SD=0.500 (하한 적용); 실측 57/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 07:57:00 / 시작 고정 기준 3창 복귀 06/09 08:02:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1350.50분 (근접 시각이며 인과 미확정) |
| TC-Z5 · 06/09 08:07:00–06/09 08:15:00 (끝 제외); 240058-Z-TC-Z5-0161 | 최대 절대 Z=3.37 (06/09 08:09:30, 창 중심); 1분 중앙값 858.432℃, 과거 중심 860.787℃, 차이 -2.355℃; SD=0.699; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 8창, 같은 방향 최장 8창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:18:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1333.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/09 08:10:00–06/09 08:23:00 (끝 제외); 240058-Z-TC-Z3-0164 | 최대 절대 Z=3.49 (06/09 08:16:30, 창 중심); 1분 중앙값 856.231℃, 과거 중심 860.293℃, 차이 -4.062℃; SD=1.164; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 13창, 같은 방향 최장 13창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:26:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1326.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 06/09 08:12:00–06/09 08:19:00 (끝 제외); 240058-Z-TC-Z6-0166 | 최대 절대 Z=3.29 (06/09 08:14:30, 창 중심); 1분 중앙값 857.328℃, 과거 중심 860.655℃, 차이 -3.327℃; SD=1.012; 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:22:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1328.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 06/09 08:19:00–06/09 08:20:00 (끝 제외); 240058-Z-OP-Z6-0173 | 최대 절대 Z=5.02 (06/09 08:19:30, 창 중심); 1분 중앙값 63.557%, 과거 중심 73.598%, 차이 -10.041%p; SD=2.000 (하한 적용); 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:23:00 / 시작 고정 기준 3창 복귀 06/09 08:23:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1323.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 06/09 08:22:00–06/09 08:25:00 (끝 제외); 240058-Z-TC-Z6-0176 | 최대 절대 Z=3.19 (06/09 08:23:30, 창 중심); 1분 중앙값 852.279℃, 과거 중심 859.639℃, 차이 -7.360℃; SD=2.306; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:28:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1319.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/09 08:51:00–06/09 08:52:00 (끝 제외); 240058-Z-OP-Z3-0205 | 최대 절대 Z=3.22 (06/09 08:51:30, 창 중심); 1분 중앙값 51.701%, 과거 중심 58.134%, 차이 -6.432%p; SD=2.000 (하한 적용); 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1291.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 06/09 08:51:00–06/09 08:52:00 (끝 제외); 240058-Z-OP-Z6-0205 | 최대 절대 Z=3.48 (06/09 08:51:30, 창 중심); 1분 중앙값 65.057%, 과거 중심 72.012%, 차이 -6.955%p; SD=2.000 (하한 적용); 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1291.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/09 08:53:00–06/09 09:02:00 (끝 제외); 240058-Z-OP-Z3-0207 | 최대 절대 Z=8.72 (06/09 08:53:30, 창 중심); 1분 중앙값 40.664%, 과거 중심 58.100%, 차이 -17.435%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 06/09 08:53:00–06/09 09:02:00 (끝 제외); 240058-Z-OP-Z4-0207 | 최대 절대 Z=6.92 (06/09 08:53:30, 창 중심); 1분 중앙값 35.908%, 과거 중심 49.756%, 차이 -13.848%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |
| OP-Z5 · 06/09 08:53:00–06/09 09:02:00 (끝 제외); 240058-Z-OP-Z5-0207 | 최대 절대 Z=7.29 (06/09 08:53:30, 창 중심); 1분 중앙값 33.248%, 과거 중심 47.826%, 차이 -14.578%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 06/09 08:53:00–06/09 09:02:00 (끝 제외); 240058-Z-OP-Z6-0207 | 최대 절대 Z=10.22 (06/09 08:53:30, 창 중심); 1분 중앙값 51.564%, 과거 중심 72.012%, 차이 -20.447%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240058-Z-TC-Z3-0164:** OP-Z3 사건창 중앙값−직전10분 중앙값 -0.3912 (10/13유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240058-Z-OP-Z6-0207:** TC-Z6 사건창 중앙값−직전10분 중앙값 -10.43 (8/9유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 06/09 05:26:00–06/09 05:48:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 / 과거 유효 최소 0창 <20 |
| 06/09 05:52:00–06/09 05:53:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/09 06:21:00–06/09 06:22:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 06/09 06:54:00–06/09 06:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/09 07:31:00–06/09 07:33:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 06/09 07:57:00–06/09 07:58:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/09 08:43:00–06/09 08:44:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/09 08:52:00–06/09 08:53:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/09 09:02:00–06/09 09:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240058-CP-01 · 06/09 09:02:35 | -0.3511무차원 · 5238점/5416초 간격 | DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다. | 60개 연속 실제 1초 값 복귀 미확인; WO-2024-00916 감지 21시간 20분 25초 전에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240058-CP-01 · 피크 06/09 09:02:35 · 복귀 미확인 원본 이탈**

- **관측:** 원값 0.0969111, LOT 중앙값 0.447995, 편차 -0.351084. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 5238점, 첫·끝 실측 간격 5416초, 기대 1초 격자 내 유효 비율 96.7%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 06/09 07:32:36=0.428035 / 직후 06/09 09:02:54=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 미확인.
- **같은 1분의 결과:** 17신호에서 |Z|≥3 미검출 (평가 불가 채널은 별도).
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 126475–131713에 연결했다(0부터 시작).

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240058-CP-01 | 정비 대조: WO-2024-00916 감지 대비 CP 원본 피크 -1280.42분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -8.584 (292/48점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -31.6 (292/48점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 -1.45e-11 (292/48점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240058, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z7 · 06/09 08:53~09:02**

**관측:** |Z|≥3 연속 **9개 1분창**, 최근 30분 시작 1회. 최대 |Z| **11.49**, 과거 중심 대비 **-23%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -3.73℃(8/9유효창). 같은 전후 기준의 OP-Z7 변화는 -48.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z8 · 06/09 08:53~09:02**

**관측:** |Z|≥3 연속 **9개 1분창**, 최근 30분 시작 1회. 최대 |Z| **8.65**, 과거 중심 대비 **-17.3%p**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z8의 사건창 중앙값−직전 10분 중앙값은 -3.69℃(8/9유효창). 같은 전후 기준의 OP-Z8 변화는 -39.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z7 · 06/09 08:15~08:20**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.47**, 과거 중심 대비 **-2.02℃**.  
**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z7의 사건창 중앙값−직전 10분 중앙값은 +1.94%p(10/5유효창). 같은 전후 기준의 TC-Z7 변화는 -1.35℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 4건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240058-Z-TC-Z7-0169 · 06/09 08:15~08:20**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z7의 사건창 중앙값−직전 10분 중앙값은 +1.94%p(10/5유효창). 같은 전후 기준의 TC-Z7 변화는 -1.35℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-TC-Z8-0169 · 06/09 08:15~08:21**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z8의 사건창 중앙값−직전 10분 중앙값은 +1.48%p(10/6유효창). 같은 전후 기준의 TC-Z8 변화는 -1.38℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z7-0207 · 06/09 08:53~09:02**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -3.73℃(8/9유효창). 같은 전후 기준의 OP-Z7 변화는 -48.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240058-Z-OP-Z8-0207 · 06/09 08:53~09:02**

**다른 LOT:** DP590의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z8의 사건창 중앙값−직전 10분 중앙값은 -3.69℃(8/9유효창). 같은 전후 기준의 OP-Z8 변화는 -39.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 217 | 186 | 5 | 31 | 11 | 20 | 0 |
| OP-Z7 | 217 | 186 | 9 | 31 | 11 | 20 | 0 |
| TC-Z8 | 217 | 186 | 6 | 31 | 11 | 20 | 0 |
| OP-Z8 | 217 | 186 | 9 | 31 | 11 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| TC-Z7 · 06/09 08:15:00–06/09 08:20:00 (끝 제외); 240058-Z-TC-Z7-0169 | 최대 절대 Z=3.47 (06/09 08:17:30, 창 중심); 1분 중앙값 323.821℃, 과거 중심 325.837℃, 차이 -2.016℃; SD=0.581; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:23:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1325.50분 (근접 시각이며 인과 미확정) |
| TC-Z8 · 06/09 08:15:00–06/09 08:21:00 (끝 제외); 240058-Z-TC-Z8-0169 | 최대 절대 Z=3.30 (06/09 08:16:30, 창 중심); 1분 중앙값 323.848℃, 과거 중심 325.679℃, 차이 -1.831℃; SD=0.555; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 6창, 같은 방향 최장 6창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/09 08:24:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1326.50분 (근접 시각이며 인과 미확정) |
| OP-Z7 · 06/09 08:53:00–06/09 09:02:00 (끝 제외); 240058-Z-OP-Z7-0207 | 최대 절대 Z=11.49 (06/09 08:53:30, 창 중심); 1분 중앙값 53.987%, 과거 중심 76.977%, 차이 -22.989%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |
| OP-Z8 · 06/09 08:53:00–06/09 09:02:00 (끝 제외); 240058-Z-OP-Z8-0207 | 최대 절대 Z=8.65 (06/09 08:53:30, 창 중심); 1분 중앙값 44.922%, 과거 중심 62.219%, 차이 -17.298%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00916 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240058-Z-OP-Z7-0207:** TC-Z7 사건창 중앙값−직전10분 중앙값 -3.731 (8/9유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240058-Z-OP-Z8-0207:** TC-Z8 사건창 중앙값−직전10분 중앙값 -3.685 (8/9유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 06/09 05:26:00–06/09 05:48:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 / 과거 유효 최소 0창 <20 |
| 06/09 05:52:00–06/09 05:53:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/09 06:21:00–06/09 06:22:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 06/09 06:54:00–06/09 06:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/09 07:31:00–06/09 07:33:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 06/09 07:57:00–06/09 07:58:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/09 08:43:00–06/09 08:44:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/09 08:52:00–06/09 08:53:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/09 09:02:00–06/09 09:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240058)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **1건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 12559 | 11 | 원본 ±0.05 사건 1건 · 최대 절대 편차 0.35108 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 12558 | 11 | 원본 중앙값 1.1513e-10, 최저 3.7766e-11, 최고 1.17209e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 12557 | 11 | 원본 중앙값 326.384, 최저 309.914, 최고 339.901; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 12558 | 11 | 원본 중앙값 276.891, 최저 259.152, 최고 291.126; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 12557 | 11 | 원본 중앙값 273.224, 최저 257.558, 최고 283.873; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-00916 · TBM / 메모: 내용 미기재**

**실제 시각:** 감지 06/10 06:23:00 / 작업 06/10 08:45:00~06/10 10:48:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-00916 감지 21시간 20분 25초 전에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 3건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 3건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00916 · TBM | 06/10 06:23:00 | 06/10 08:45:00 | 06/10 10:48:00 | 내용 미기재 | SEL-006 오일씰 150x190x18 ×2 (교체 사유: 수명도래) / SEL-009 O링 세트 ×1 (교체 사유: 손상) | 작업은 LOT 관측 밖 |
| WO-2024-00931 · TBM | 06/11 14:51:00 | 06/11 18:40:00 | 06/11 23:37:00 | 마모량 초과 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-00940 · TBM | 06/12 06:09:00 | 06/12 09:51:00 | 06/12 15:23:00 | 수리기간 중 점검 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-00916 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 33건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240058-CP-01 · 06/09 09:02:35 · CP 편차 -0.35108 · 5238점/5416초 · 완료 대비 피크 -1545.42분 · 감지는 피크 대비 +1280.42분. 복귀 미확인 원본 이탈; 직전 06/09 07:32:36=0.428035 / 직후 06/09 09:02:54=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 미확인.
- **동반 양상:** OP-Z1 3건·최대 절대 Z 6.26 / OP-Z2 1건·최대 절대 Z 4.63 / OP-Z3 4건·최대 절대 Z 8.72 / OP-Z4 2건·최대 절대 Z 6.92 / OP-Z5 1건·최대 절대 Z 7.29 / OP-Z6 5건·최대 절대 Z 10.22 / OP-Z7 1건·최대 절대 Z 11.49 / OP-Z8 1건·최대 절대 Z 8.65 / TC-Z1 2건·최대 절대 Z 4.09 / TC-Z2 1건·최대 절대 Z 3.69 / TC-Z3 5건·최대 절대 Z 8.66 / TC-Z4 1건·최대 절대 Z 3.54 / TC-Z5 1건·최대 절대 Z 3.37 / TC-Z6 3건·최대 절대 Z 3.29 / TC-Z7 1건·최대 절대 Z 3.47 / TC-Z8 1건·최대 절대 Z 3.30. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/10 06:23:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/10 08:45:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/10 10:48:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-00931 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 33건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240058-CP-01 · 06/09 09:02:35 · CP 편차 -0.35108 · 5238점/5416초 · 완료 대비 피크 -3754.42분 · 감지는 피크 대비 +3228.42분. 복귀 미확인 원본 이탈; 직전 06/09 07:32:36=0.428035 / 직후 06/09 09:02:54=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 미확인.
- **동반 양상:** OP-Z1 3건·최대 절대 Z 6.26 / OP-Z2 1건·최대 절대 Z 4.63 / OP-Z3 4건·최대 절대 Z 8.72 / OP-Z4 2건·최대 절대 Z 6.92 / OP-Z5 1건·최대 절대 Z 7.29 / OP-Z6 5건·최대 절대 Z 10.22 / OP-Z7 1건·최대 절대 Z 11.49 / OP-Z8 1건·최대 절대 Z 8.65 / TC-Z1 2건·최대 절대 Z 4.09 / TC-Z2 1건·최대 절대 Z 3.69 / TC-Z3 5건·최대 절대 Z 8.66 / TC-Z4 1건·최대 절대 Z 3.54 / TC-Z5 1건·최대 절대 Z 3.37 / TC-Z6 3건·최대 절대 Z 3.29 / TC-Z7 1건·최대 절대 Z 3.47 / TC-Z8 1건·최대 절대 Z 3.30. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/11 14:51:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/11 18:40:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/11 23:37:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-00940 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 32건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240058-CP-01 · 06/09 09:02:35 · CP 편차 -0.35108 · 5238점/5416초 · 완료 대비 피크 -4700.42분 · 감지는 피크 대비 +4146.42분. 복귀 미확인 원본 이탈; 직전 06/09 07:32:36=0.428035 / 직후 06/09 09:02:54=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 미확인.
- **동반 양상:** OP-Z1 3건·최대 절대 Z 6.26 / OP-Z2 1건·최대 절대 Z 4.63 / OP-Z3 4건·최대 절대 Z 8.72 / OP-Z4 2건·최대 절대 Z 6.92 / OP-Z5 1건·최대 절대 Z 7.29 / OP-Z6 5건·최대 절대 Z 10.22 / OP-Z7 1건·최대 절대 Z 11.49 / OP-Z8 1건·최대 절대 Z 8.65 / TC-Z1 2건·최대 절대 Z 4.09 / TC-Z2 1건·최대 절대 Z 3.69 / TC-Z3 4건·최대 절대 Z 8.66 / TC-Z4 1건·최대 절대 Z 3.54 / TC-Z5 1건·최대 절대 Z 3.37 / TC-Z6 3건·최대 절대 Z 3.29 / TC-Z7 1건·최대 절대 Z 3.47 / TC-Z8 1건·최대 절대 Z 3.30. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/12 06:09:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/12 09:51:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/12 15:23:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240058)

**240058 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240058)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240058)
lot_conclusion(240058)
lot_execution.append({'LOT':240058,'확대 구간':len(LOT_EVENTS[240058]),'강종 대조':240058 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240061"></a>
### 8.7 LOT 240061 · SPHC · 06/15

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240061)

#### LOT 240061 · 먼저 볼 변화

**지속·반복 후보 5건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — CLN-IN · 06/15 03:45~03:49**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.55**, 과거 중심 대비 **-1.85℃**.  
**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 1.6~1.6℃)이 있다. 이 구간의 4창·1.85℃는 비교 관측의 상한을 넘는다(절대 편차 1.85>1.6℃, 길이 4>1창). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 04:40 (이동 Z 복귀보다 48분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 06/15 04:35~04:38**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 2회. 최대 |Z| **4.66**, 과거 중심 대비 **-9.56%p**.  
**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 4~6개 1분창, 절대 편차 8.47~10.7%p)이 있다. 이 구간의 3창·9.56%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.04℃(9/3유효창). 같은 전후 기준의 OP-Z2 변화는 -8.67%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 04:44 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z3 · 06/15 03:12~03:14**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 2회. 최대 |Z| **3.67**, 과거 중심 대비 **-72.7%p**.  
**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 29건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·72.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 29건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.07℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -78.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 03:23 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240061

- **관측 범위:** 06/15 01:04:00–06/15 04:49:23.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 169개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 5개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 27개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240061, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 3건 · 짧은 노랑 후보 2건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — CLN-IN · 06/15 03:45~03:49**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.55**, 과거 중심 대비 **-1.85℃**.  
**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 1.6~1.6℃)이 있다. 이 구간의 4창·1.85℃는 비교 관측의 상한을 넘는다(절대 편차 1.85>1.6℃, 길이 4>1창). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 04:40 (이동 Z 복귀보다 48분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 06/15 04:35~04:38**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 2회. 최대 |Z| **4.66**, 과거 중심 대비 **-9.56%p**.  
**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 4~6개 1분창, 절대 편차 8.47~10.7%p)이 있다. 이 구간의 3창·9.56%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.04℃(9/3유효창). 같은 전후 기준의 OP-Z2 변화는 -8.67%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 04:44 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z1 · 06/15 04:31~04:34**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.63**, 과거 중심 대비 **-12.6%p**.  
**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 5~5개 1분창, 절대 편차 9.2~11.6%p)이 있다. 이 구간의 3창·12.6%p는 비교 관측의 상한을 넘는다(절대 편차 12.6>11.6%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.34℃(10/3유효창). 같은 전후 기준의 OP-Z1 변화는 -10.6%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 04:41 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 5건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240061-Z-OP-Z1-0154 · 06/15 03:38~03:40**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 5~5개 1분창, 절대 편차 9.2~11.6%p)이 있다. 이 구간의 2창·6.66%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.05℃(10/2유효창). 같은 전후 기준의 OP-Z1 변화는 -7.49%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 03:43. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 2건/127유효창 (100창당 1.57건) | 6.656~12.58 | 2~3창 | 72.702 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.55~10.55 | 5~5창 | 74.266 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 10.54~11.55 | 5~5창 | 72.861 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 9.204~9.204 | 5~5창 | 73.431 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-CLN-IN-0161 · 06/15 03:45~03:49**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 1.6~1.6℃)이 있다. 이 구간의 4창·1.85℃는 비교 관측의 상한을 넘는다(절대 편차 1.85>1.6℃, 길이 4>1창). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 04:40 (이동 Z 복귀보다 48분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/127유효창 (100창당 0.79건) | 1.848~1.848 | 4~4창 | 69.526 |
| 240044 | 0건/52유효창 (100창당 0.00건) | — | — | 64.529 |
| 240069 | 0건/737유효창 (100창당 0.00건) | — | — | 66.613 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 1.605~1.605 | 1~1창 | 66.662 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z1-0207 · 06/15 04:31~04:34**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 5~5개 1분창, 절대 편차 9.2~11.6%p)이 있다. 이 구간의 3창·12.6%p는 비교 관측의 상한을 넘는다(절대 편차 12.6>11.6%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.34℃(10/3유효창). 같은 전후 기준의 OP-Z1 변화는 -10.6%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 04:41 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 2건/127유효창 (100창당 1.57건) | 6.656~12.58 | 2~3창 | 72.702 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.55~10.55 | 5~5창 | 74.266 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 10.54~11.55 | 5~5창 | 72.861 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 9.204~9.204 | 5~5창 | 73.431 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z2-0209 · 06/15 04:33~04:34**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 4~6개 1분창, 절대 편차 8.47~10.7%p)이 있다. 이 구간의 1창·6.68%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.978℃(10/1유효창). 같은 전후 기준의 OP-Z2 변화는 -6.76%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 04:44 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 2건/127유효창 (100창당 1.57건) | 6.682~9.561 | 1~3창 | 19.189 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.68~10.68 | 6~6창 | 25.212 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 8.473~8.846 | 4~5창 | 21.399 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 8.744~8.744 | 6~6창 | 22.73 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z2-0211 · 06/15 04:35~04:38**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 4건(연속 4~6개 1분창, 절대 편차 8.47~10.7%p)이 있다. 이 구간의 3창·9.56%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.04℃(9/3유효창). 같은 전후 기준의 OP-Z2 변화는 -8.67%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 04:44 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 2건/127유효창 (100창당 1.57건) | 6.682~9.561 | 1~3창 | 19.189 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.68~10.68 | 6~6창 | 25.212 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 8.473~8.846 | 4~5창 | 21.399 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 8.744~8.744 | 6~6창 | 22.73 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 226 | 202 | 4 | 24 | 4 | 20 | 0 |
| TC-Z1 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| OP-Z1 | 226 | 202 | 5 | 24 | 4 | 20 | 0 |
| TC-Z2 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| OP-Z2 | 226 | 202 | 4 | 24 | 4 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z1: 유효 202창, 최대 절대 Z=2.74 / TC-Z2: 유효 202창, 최대 절대 Z=2.37. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 06/15 03:38:00–06/15 03:40:00 (끝 제외); 240061-Z-OP-Z1-0154 | 최대 절대 Z=3.33 (06/15 03:39:30, 창 중심); 1분 중앙값 66.109%, 과거 중심 72.765%, 차이 -6.656%p; SD=2.001; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:43:00 / 시작 고정 기준 3창 복귀 06/15 03:43:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +576.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 06/15 03:45:00–06/15 03:49:00 (끝 제외); 240061-Z-CLN-IN-0161 | 최대 절대 Z=3.55 (06/15 03:46:30, 창 중심); 1분 중앙값 69.114℃, 과거 중심 70.962℃, 차이 -1.848℃; SD=0.520; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:52:00 / 시작 고정 기준 3창 복귀 06/15 04:40:00 | 고정 범위 복귀가 이동 Z 복귀보다 48분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +583.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 06/15 04:31:00–06/15 04:34:00 (끝 제외); 240061-Z-OP-Z1-0207 | 최대 절대 Z=3.63 (06/15 04:33:30, 창 중심); 1분 중앙값 58.493%, 과거 중심 71.074%, 차이 -12.581%p; SD=3.464; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 04:38:00 / 시작 고정 기준 3창 복귀 06/15 04:41:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +630.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 06/15 04:33:00–06/15 04:34:00 (끝 제외); 240061-Z-OP-Z2-0209 | 최대 절대 Z=3.34 (06/15 04:33:30, 창 중심); 1분 중앙값 11.177%, 과거 중심 17.859%, 차이 -6.682%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 04:41:00 / 시작 고정 기준 3창 복귀 06/15 04:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +630.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 06/15 04:35:00–06/15 04:38:00 (끝 제외); 240061-Z-OP-Z2-0211 | 최대 절대 Z=4.66 (06/15 04:35:30, 창 중심); 1분 중앙값 8.151%, 과거 중심 17.712%, 차이 -9.561%p; SD=2.050; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 04:41:00 / 시작 고정 기준 3창 복귀 06/15 04:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +632.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240061-Z-CLN-IN-0161:** 대응 OP 채널 정의 없음; 다른 신호의 동시 반응은 전체 공정 그림에서 확인. 동시 관측의 근거이며 원인 확정은 아니다.
- **240061-Z-OP-Z2-0211:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.041 (9/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 06/15 01:04:00–06/15 01:24:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 06/15 01:47:00–06/15 01:48:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 06/15 02:40:00–06/15 02:41:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/15 04:34:00–06/15 04:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 06/15 04:49:00–06/15 04:50:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 24개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240061, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 2건 · 짧은 노랑 후보 10건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 주황 · 지속·반복 확인 — OP-Z3 · 06/15 02:19~02:22**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.18**, 과거 중심 대비 **-73%p**.  
**다른 LOT:** 같은 강종 LOT 240044, 240069, 240091에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +8.24℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -72.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 02:26 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 복귀 추가 확인 — OP-Z3 · 06/15 03:12~03:14**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 2회. 최대 |Z| **3.67**, 과거 중심 대비 **-72.7%p**.  
**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 29건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·72.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 29건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.07℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -78.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 03:23 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 12건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240061-Z-OP-Z3-0075 · 06/15 02:19~02:22**

**다른 LOT:** 같은 강종 LOT 240044, 240069, 240091에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +8.24℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -72.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 02:26 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 초기 승온로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/72유효창 (100창당 1.39건) | 73~73 | 3~3창 | 74.558 |
| 240044 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |
| 240069 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |
| 240091 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-TC-Z3-0076 · 06/15 02:20~02:22**

**다른 LOT:** 같은 강종 LOT 240044, 240069, 240091에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -72.2%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +9.04℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 02:26 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 초기 승온로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/75유효창 (100창당 1.33건) | 12.89~12.89 | 2~2창 | 846.06 |
| 240044 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |
| 240069 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |
| 240091 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z3-0111 · 06/15 02:55~02:57**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 29건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·57.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 29건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.24℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -55.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:00. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z3-0128 · 06/15 03:12~03:14**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 29건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·72.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 29건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.07℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -78.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:23 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-TC-Z3-0129 · 06/15 03:13~03:16**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 15건(연속 1~5개 1분창, 절대 편차 2.45~11.1℃)이 있다. 이 구간의 3창·10.6℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -72.5%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 +11.3℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:23 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z4-0133 · 06/15 03:17~03:19**

**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 6.15~6.15%p)이 있다. 이 구간의 2창·7.9%p는 비교 관측의 상한을 넘는다(절대 편차 7.9>6.15%p, 길이 2>1창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.7℃(10/2유효창). 같은 전후 기준의 OP-Z4 변화는 -8.41%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:22. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/127유효창 (100창당 0.79건) | 7.899~7.899 | 2~2창 | 54.132 |
| 240044 | 0건/53유효창 (100창당 0.00건) | — | — | 41.819 |
| 240069 | 1건/737유효창 (100창당 0.14건) | 6.15~6.15 | 1~1창 | 42.121 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 44.37 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-TC-Z4-0134 · 06/15 03:18~03:19**

**다른 LOT:** 같은 강종 LOT 240044, 240069, 240091의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -7.79%p(10/1유효창). 같은 전후 기준의 TC-Z4 변화는 +1.61℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:22. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/127유효창 (100창당 0.79건) | 1.68~1.68 | 1~1창 | 859.8 |
| 240044 | 0건/52유효창 (100창당 0.00건) | — | — | 859.65 |
| 240069 | 0건/737유효창 (100창당 0.00건) | — | — | 859.66 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 859.69 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z6-0140 · 06/15 03:24~03:26**

**다른 LOT:** 동종 LOT 240069, 240091에도 같은 방향 후보 3건(연속 2~3개 1분창, 절대 편차 6.42~9.17%p)이 있다. 이 구간의 2창·9.68%p는 비교 관측의 상한을 넘는다(절대 편차 9.68>9.17%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.44℃(10/2유효창). 같은 전후 기준의 OP-Z6 변화는 +7.73%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:32 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/127유효창 (100창당 0.79건) | 9.685~9.685 | 2~2창 | 66.614 |
| 240044 | 0건/53유효창 (100창당 0.00건) | — | — | 67.168 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 8.909~9.171 | 2~3창 | 68.085 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 6.416~6.416 | 2~2창 | 67.844 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z3-0194 · 06/15 04:18~04:20**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 29건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 2창·42.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 29건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.19℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -39.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:23. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-TC-Z3-0195 · 06/15 04:19~04:20**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 15건(연속 1~5개 1분창, 절대 편차 2.45~11.1℃)이 있다. 이 구간의 1창·2.91℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -44.2%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.64℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:23. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-TC-Z3-0202 · 06/15 04:26~04:28**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 15건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 2창·3.2℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +24.6%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -3.84℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:32 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240061-Z-OP-Z3-0224 · 06/15 04:48~04:49**

**다른 LOT:** 동종 LOT 240044, 240069, 240091에도 같은 방향 후보 29건(연속 1~4개 1분창, 절대 편차 11~77.2%p)이 있다. 이 구간의 1창·69.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 29건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.63℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -87.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240061 (현재 LOT) | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 226 | 202 | 8 | 24 | 4 | 20 | 0 |
| OP-Z3 | 226 | 199 | 10 | 27 | 7 | 20 | 0 |
| TC-Z4 | 226 | 202 | 1 | 24 | 4 | 20 | 0 |
| OP-Z4 | 226 | 202 | 2 | 24 | 4 | 20 | 0 |
| TC-Z5 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| OP-Z5 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| TC-Z6 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| OP-Z6 | 226 | 202 | 2 | 24 | 4 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z5: 유효 202창, 최대 절대 Z=1.88 / OP-Z5: 유효 202창, 최대 절대 Z=2.24 / TC-Z6: 유효 202창, 최대 절대 Z=2.71. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 06/15 02:19:00–06/15 02:22:00 (끝 제외); 240061-Z-OP-Z3-0075 | 최대 절대 Z=4.18 (06/15 02:20:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 72.995%, 차이 -72.995%p; SD=17.481; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 02:25:00 / 시작 고정 기준 3창 복귀 06/15 02:26:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +497.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/15 02:20:00–06/15 02:22:00 (끝 제외); 240061-Z-TC-Z3-0076 | 최대 절대 Z=3.52 (06/15 02:21:30, 창 중심); 1분 중앙값 863.300℃, 과거 중심 850.408℃, 차이 +12.892℃; SD=3.658; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 02:25:00 / 시작 고정 기준 3창 복귀 06/15 02:26:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +498.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/15 02:55:00–06/15 02:57:00 (끝 제외); 240061-Z-OP-Z3-0111 | 최대 절대 Z=5.15 (06/15 02:55:30, 창 중심); 1분 중앙값 15.330%, 과거 중심 73.007%, 차이 -57.677%p; SD=11.193; 실측 54/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:00:00 / 시작 고정 기준 3창 복귀 06/15 03:00:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +532.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/15 03:12:00–06/15 03:14:00 (끝 제외); 240061-Z-OP-Z3-0128 | 최대 절대 Z=3.67 (06/15 03:12:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 72.677%, 차이 -72.677%p; SD=19.808; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:17:00 / 시작 고정 기준 3창 복귀 06/15 03:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +549.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/15 03:13:00–06/15 03:16:00 (끝 제외); 240061-Z-TC-Z3-0129 | 최대 절대 Z=3.70 (06/15 03:14:30, 창 중심); 1분 중앙값 869.878℃, 과거 중심 859.317℃, 차이 +10.561℃; SD=2.852; 실측 56/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:19:00 / 시작 고정 기준 3창 복귀 06/15 03:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +551.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 06/15 03:17:00–06/15 03:19:00 (끝 제외); 240061-Z-OP-Z4-0133 | 최대 절대 Z=3.95 (06/15 03:17:30, 창 중심); 1분 중앙값 46.465%, 과거 중심 54.364%, 차이 -7.899%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:22:00 / 시작 고정 기준 3창 복귀 06/15 03:22:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +554.50분 (근접 시각이며 인과 미확정) |
| TC-Z4 · 06/15 03:18:00–06/15 03:19:00 (끝 제외); 240061-Z-TC-Z4-0134 | 최대 절대 Z=3.35 (06/15 03:18:30, 창 중심); 1분 중앙값 861.444℃, 과거 중심 859.764℃, 차이 +1.680℃; SD=0.502; 실측 56/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:22:00 / 시작 고정 기준 3창 복귀 06/15 03:22:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +555.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 06/15 03:24:00–06/15 03:26:00 (끝 제외); 240061-Z-OP-Z6-0140 | 최대 절대 Z=3.66 (06/15 03:25:30, 창 중심); 1분 중앙값 76.393%, 과거 중심 66.709%, 차이 +9.685%p; SD=2.649; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 03:29:00 / 시작 고정 기준 3창 복귀 06/15 03:32:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +562.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/15 04:18:00–06/15 04:20:00 (끝 제외); 240061-Z-OP-Z3-0194 | 최대 절대 Z=4.23 (06/15 04:19:30, 창 중심); 1분 중앙값 31.559%, 과거 중심 74.485%, 차이 -42.925%p; SD=10.155; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 04:23:00 / 시작 고정 기준 3창 복귀 06/15 04:23:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +616.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/15 04:19:00–06/15 04:20:00 (끝 제외); 240061-Z-TC-Z3-0195 | 최대 절대 Z=3.59 (06/15 04:19:30, 창 중심); 1분 중앙값 862.158℃, 과거 중심 859.246℃, 차이 +2.912℃; SD=0.810; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 04:23:00 / 시작 고정 기준 3창 복귀 06/15 04:23:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +616.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 06/15 04:26:00–06/15 04:28:00 (끝 제외); 240061-Z-TC-Z3-0202 | 최대 절대 Z=3.53 (06/15 04:26:30, 창 중심); 1분 중앙값 856.286℃, 과거 중심 859.485℃, 차이 -3.199℃; SD=0.906; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 06/15 04:31:00 / 시작 고정 기준 3창 복귀 06/15 04:32:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +623.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 06/15 04:48:00–06/15 04:49:00 (끝 제외); 240061-Z-OP-Z3-0224 | 최대 절대 Z=3.42 (06/15 04:48:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 69.053%, 차이 -69.053%p; SD=20.189; 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 짧은 국소 변화지만 고정 범위 복귀는 미확인. 원본의 폭·전후 값을 보존하며 1초 스파이크로 단정하지 않음; 정비 대조: WO-2024-00972 완료 대비 Z 최대창 중심 +645.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240061-Z-OP-Z3-0075:** TC-Z3 사건창 중앙값−직전10분 중앙값 +8.237 (10/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240061-Z-TC-Z3-0129:** OP-Z3 사건창 중앙값−직전10분 중앙값 -72.45 (10/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 06/15 01:04:00–06/15 01:27:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 0개 <54 / 과거 유효 최소 0창 <20 |
| 06/15 01:47:00–06/15 01:48:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 06/15 02:40:00–06/15 02:41:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/15 04:34:00–06/15 04:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 06/15 04:49:00–06/15 04:50:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 24개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240061, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| OP-Z7 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| TC-Z8 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |
| OP-Z8 | 226 | 202 | 0 | 24 | 4 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 202창, 최대 절대 Z=1.91 / OP-Z7: 유효 202창, 최대 절대 Z=2.24 / TC-Z8: 유효 202창, 최대 절대 Z=1.80 / OP-Z8: 유효 202창, 최대 절대 Z=1.70. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 06/15 01:04:00–06/15 01:24:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 06/15 01:47:00–06/15 01:48:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 06/15 02:40:00–06/15 02:41:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/15 04:34:00–06/15 04:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 06/15 04:49:00–06/15 04:50:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 24개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240061)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 13299 | 4 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.020911 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 13299 | 4 | 원본 중앙값 1.13649e-10, 최저 3.73038e-11, 최고 1.1556e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 13299 | 4 | 원본 중앙값 325.486, 최저 318.745, 최고 342.128; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 13299 | 4 | 원본 중앙값 282.588, 최저 257.319, 최고 298.179; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 13299 | 4 | 원본 중앙값 278.769, 최저 265.557, 최고 292.375; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-00972 · TBM / 메모: 1호기 재발, SD작업 수행**

**실제 시각:** 감지 06/14 14:09:00 / 작업 06/14 15:35:00~06/14 18:03:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-00972 완료 9시간 43분 30초 후에 Z 최대창 중심.

**실측으로 비교할 수 있는 범위:** 완료 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 작업 또는 제어 조건 변경과 동반됐을 가능성을 검토한다. 같은 시각이라는 이유만으로 정비가 원인이라고 판단하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 4건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 4건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00940 · TBM | 06/12 06:09:00 | 06/12 09:51:00 | 06/12 15:23:00 | 수리기간 중 점검 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-00952 · TBM | 06/12 23:51:00 | 06/13 02:51:00 | 06/13 05:16:00 | 정기수리 부속작업 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-00960 · TBM | 06/13 14:22:00 | 06/13 18:19:00 | 06/13 22:29:00 | 1호기 재발, SD작업 수행 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-00972 · TBM | 06/14 14:09:00 | 06/14 15:35:00 | 06/14 18:03:00 | 1호기 재발, SD작업 수행 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-00940 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 17건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.020911 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.55 / OP-Z1 2건·최대 절대 Z 3.63 / OP-Z2 2건·최대 절대 Z 4.66 / OP-Z3 5건·최대 절대 Z 5.15 / OP-Z4 1건·최대 절대 Z 3.95 / OP-Z6 1건·최대 절대 Z 3.66 / TC-Z3 4건·최대 절대 Z 3.70 / TC-Z4 1건·최대 절대 Z 3.35. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/12 06:09:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/12 09:51:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/12 15:23:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-00952 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 17건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.020911 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.55 / OP-Z1 2건·최대 절대 Z 3.63 / OP-Z2 2건·최대 절대 Z 4.66 / OP-Z3 5건·최대 절대 Z 5.15 / OP-Z4 1건·최대 절대 Z 3.95 / OP-Z6 1건·최대 절대 Z 3.66 / TC-Z3 4건·최대 절대 Z 3.70 / TC-Z4 1건·최대 절대 Z 3.35. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/12 23:51:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/13 02:51:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/13 05:16:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-00960 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 17건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.020911 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.55 / OP-Z1 2건·최대 절대 Z 3.63 / OP-Z2 2건·최대 절대 Z 4.66 / OP-Z3 5건·최대 절대 Z 5.15 / OP-Z4 1건·최대 절대 Z 3.95 / OP-Z6 1건·최대 절대 Z 3.66 / TC-Z3 4건·최대 절대 Z 3.70 / TC-Z4 1건·최대 절대 Z 3.35. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/13 14:22:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/13 18:19:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/13 22:29:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-00972 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 17건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.020911 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.55 / OP-Z1 2건·최대 절대 Z 3.63 / OP-Z2 2건·최대 절대 Z 4.66 / OP-Z3 5건·최대 절대 Z 5.15 / OP-Z4 1건·최대 절대 Z 3.95 / OP-Z6 1건·최대 절대 Z 3.66 / TC-Z3 4건·최대 절대 Z 3.70 / TC-Z4 1건·최대 절대 Z 3.35. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 06/14 14:09:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 06/14 15:35:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 06/14 18:03:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240061)

**240061 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240061)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240061)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240061(SPHC)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- OP-Z4: 이 LOT 54.132 - 대조 42.121 = **+12.011%p**, 다른 3 LOT과 대조.
- OP-Z8: 이 LOT 62.137 - 대조 55.684 = **+6.454%p**, 다른 3 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 +0.145℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- TC-Z8: 차이 -0.583℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**이 사례의 핵심:** OP-Z4의 차이가 커서 TC-Z4를 함께 본 것이다. TC-Z4 자체를 이상 온도로 선별한 것이 아니다. OP-Z8도 표시 경계를 넘으므로 TC-Z8·OP-Z8을 함께 표시했다. 비슷한 온도 수준에서 출력 수준이 다르게 관측되는 이유를 설정·속도·두께·부하·보정 기록으로 확인해야 한다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240044: 전환 제외 없음.
- 240061: 06/15 01:04–02:40.
- 240069: 전환 제외 없음.
- 240091: 전환 제외 없음.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240061)
lot_conclusion(240061)
lot_execution.append({'LOT':240061,'확대 구간':len(LOT_EVENTS[240061]),'강종 대조':240061 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240069"></a>
### 8.8 LOT 240069 · SPHC · 07/07

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240069)

#### LOT 240069 · 먼저 볼 변화

**지속·반복 후보 12건 / CP 원본 후보 7건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 07/08 02:31:29**

**관측:** 원본 편차 **+0.4590무차원**, 515점·첫/끝 간격 529초. 1초 단발 여부: 아님.  
**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 529초 간격·0.459무차원는 비교 관측의 상한을 넘는다(절대 편차 0.459>0.0524무차원, 길이 529>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/08 02:41:06. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 6시간 17분 31초 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — OP-Z1 · 07/08 00:46~00:51**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.12**, 과거 중심 대비 **-11.6%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·11.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.52℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11.7%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 00:57 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z2 · 07/08 06:11~06:16**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.24**, 과거 중심 대비 **-8.47%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 5창·8.47%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.3℃(9/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.69%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:23 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240069

- **관측 범위:** 07/07 16:57:00–07/08 06:29:34.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 672개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 12개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 76개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 7건 중 확인된 1초 단발 0건, 복귀 펄스 후보 6건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240069, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 5건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 07/08 00:46~00:51**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.12**, 과거 중심 대비 **-11.6%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·11.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.52℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11.7%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 00:57 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 07/08 06:11~06:16**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.24**, 과거 중심 대비 **-8.47%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 5창·8.47%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.3℃(9/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.69%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:23 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z2 · 07/08 00:49~00:53**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.30**, 과거 중심 대비 **-8.85%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 4창·8.85%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.12℃(10/4유효창). 같은 전후 기준의 OP-Z2 변화는 -8.61%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 01:00 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 6건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240069-Z-OP-Z1-0469 · 07/08 00:46~00:51**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·11.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.52℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11.7%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 00:57 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 10.54~11.55 | 5~5창 | 72.861 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.55~10.55 | 5~5창 | 74.266 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.656~12.58 | 2~3창 | 72.702 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 9.204~9.204 | 5~5창 | 73.431 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z1-0471 · 07/08 00:48~00:51**

**다른 LOT:** 동종 LOT 240044에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 1.91~1.91℃)이 있다. 이 구간의 3창·1.72℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -12.7%p(10/3유효창). 같은 전후 기준의 TC-Z1 변화는 +1.67℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 00:55 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 1.717~1.799 | 2~3창 | 99.998 |
| 240044 | 1건/52유효창 (100창당 1.92건) | 1.907~1.907 | 3~3창 | 99.946 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 99.97 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 100 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z2-0472 · 07/08 00:49~00:53**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 4창·8.85%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.12℃(10/4유효창). 같은 전후 기준의 OP-Z2 변화는 -8.61%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 01:00 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 8.473~8.846 | 4~5창 | 21.399 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.68~10.68 | 6~6창 | 25.212 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.682~9.561 | 1~3창 | 19.189 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 8.744~8.744 | 6~6창 | 22.73 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z1-0792 · 07/08 06:09~06:14**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·10.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.64℃(9/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 06:20. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 10.54~11.55 | 5~5창 | 72.861 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.55~10.55 | 5~5창 | 74.266 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.656~12.58 | 2~3창 | 72.702 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 9.204~9.204 | 5~5창 | 73.431 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z2-0794 · 07/08 06:11~06:16**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 4건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 5창·8.47%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 4건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.3℃(9/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.69%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 06:23 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 8.473~8.846 | 4~5창 | 21.399 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.68~10.68 | 6~6창 | 25.212 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.682~9.561 | 1~3창 | 19.189 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 8.744~8.744 | 6~6창 | 22.73 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z1-0794 · 07/08 06:11~06:13**

**다른 LOT:** 동종 LOT 240044에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 1.91~1.91℃)이 있다. 이 구간의 2창·1.8℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -13.6%p(9/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.73℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 06:20 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 1.717~1.799 | 2~3창 | 99.998 |
| 240044 | 1건/52유효창 (100창당 1.92건) | 1.907~1.907 | 3~3창 | 99.946 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 99.97 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 100 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| TC-Z1 | 813 | 737 | 5 | 76 | 56 | 20 | 0 |
| OP-Z1 | 813 | 737 | 10 | 76 | 56 | 20 | 0 |
| TC-Z2 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| OP-Z2 | 813 | 737 | 9 | 76 | 56 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 737창, 최대 절대 Z=1.82 / TC-Z2: 유효 737창, 최대 절대 Z=2.71. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 07/08 00:46:00–07/08 00:51:00 (끝 제외); 240069-Z-OP-Z1-0469 | 최대 절대 Z=4.12 (07/08 00:48:30, 창 중심); 1분 중앙값 62.119%, 과거 중심 73.670%, 차이 -11.551%p; SD=2.806; 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 00:54:00 / 시작 고정 기준 3창 복귀 07/08 00:57:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -480.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 07/08 00:48:00–07/08 00:51:00 (끝 제외); 240069-Z-TC-Z1-0471 | 최대 절대 Z=3.43 (07/08 00:49:30, 창 중심); 1분 중앙값 101.670℃, 과거 중심 99.953℃, 차이 +1.717℃; SD=0.500 (하한 적용); 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 00:54:00 / 시작 고정 기준 3창 복귀 07/08 00:55:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -479.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 07/08 00:49:00–07/08 00:53:00 (끝 제외); 240069-Z-OP-Z2-0472 | 최대 절대 Z=4.30 (07/08 00:50:30, 창 중심); 1분 중앙값 12.594%, 과거 중심 21.440%, 차이 -8.846%p; SD=2.058; 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 00:56:00 / 시작 고정 기준 3창 복귀 07/08 01:00:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -478.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 07/08 06:09:00–07/08 06:14:00 (끝 제외); 240069-Z-OP-Z1-0792 | 최대 절대 Z=5.05 (07/08 06:10:30, 창 중심); 1분 중앙값 61.236%, 과거 중심 71.778%, 차이 -10.542%p; SD=2.086; 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 06:20:00 / 시작 고정 기준 3창 복귀 07/08 06:20:00 | 같은 방향의 초과가 최장 5창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -158.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 07/08 06:11:00–07/08 06:16:00 (끝 제외); 240069-Z-OP-Z2-0794 | 최대 절대 Z=4.24 (07/08 06:12:30, 창 중심); 1분 중앙값 11.348%, 과거 중심 19.820%, 차이 -8.473%p; SD=2.000 (하한 적용); 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 06:20:00 / 시작 고정 기준 3창 복귀 07/08 06:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -156.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 07/08 06:11:00–07/08 06:13:00 (끝 제외); 240069-Z-TC-Z1-0794 | 최대 절대 Z=3.49 (07/08 06:12:30, 창 중심); 1분 중앙값 101.908℃, 과거 중심 100.109℃, 차이 +1.799℃; SD=0.515; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 06:16:00 / 시작 고정 기준 3창 복귀 07/08 06:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -156.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240069-Z-OP-Z1-0792:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.644 (9/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240069-Z-OP-Z2-0794:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.299 (9/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 07/07 16:57:00–07/07 17:20:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 / 과거 유효 최소 0창 <20 |
| 07/07 17:35:00–07/07 17:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 18:07:00–07/07 18:08:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/07 18:34:00–07/07 18:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 18:38:00–07/07 18:39:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 18:41:00–07/07 18:42:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/07 18:52:00–07/07 18:53:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/07 19:02:00–07/07 19:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 19:07:00–07/07 19:08:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/07 19:17:00–07/07 19:18:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 19:46:00–07/07 19:47:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/07 19:55:00–07/07 19:56:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/07 20:08:00–07/07 20:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 20:13:00–07/07 20:14:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 20:49:00–07/07 20:50:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/07 20:59:00–07/07 21:00:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/07 21:08:00–07/07 21:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/07 21:45:00–07/07 21:46:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/07 23:15:00–07/07 23:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/07 23:29:00–07/07 23:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/07 23:39:00–07/07 23:40:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/07 23:48:00–07/07 23:49:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/07 23:56:00–07/07 23:57:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 00:01:00–07/08 00:02:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/08 00:15:00–07/08 00:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/08 00:20:00–07/08 00:21:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 46개 <54 |
| 07/08 00:30:00–07/08 00:31:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 07/08 00:34:00–07/08 00:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 01:06:00–07/08 01:07:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/08 01:16:00–07/08 01:17:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 01:20:00–07/08 01:21:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/08 01:35:00–07/08 01:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/08 01:49:00–07/08 01:50:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 02:18:00–07/08 02:19:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 02:22:00–07/08 02:23:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 02:51:00–07/08 02:52:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 03:18:00–07/08 03:19:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 07/08 03:20:00–07/08 03:21:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 47개 <54 |
| 07/08 03:30:00–07/08 03:31:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/08 03:34:00–07/08 03:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 03:39:00–07/08 03:40:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/08 04:34:00–07/08 04:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 04:39:00–07/08 04:40:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 04:44:00–07/08 04:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 04:49:00–07/08 04:50:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 05:02:00–07/08 05:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 07/08 05:21:00–07/08 05:22:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 05:26:00–07/08 05:27:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 05:35:00–07/08 05:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 07/08 05:44:00–07/08 05:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 07/08 05:54:00–07/08 05:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 48개 <54 |
| 07/08 06:03:00–07/08 06:04:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 06:16:00–07/08 06:17:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 07/08 06:29:00–07/08 06:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 35개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240069, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 7건 · 짧은 노랑 후보 29건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 07/08 02:31:29**

**관측:** 원본 편차 **+0.4590무차원**, 515점·첫/끝 간격 529초. 1초 단발 여부: 아님.  
**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 529초 간격·0.459무차원는 비교 관측의 상한을 넘는다(절대 편차 0.459>0.0524무차원, 길이 529>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/08 02:41:06. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 6시간 17분 31초 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 07/07 23:30~23:34**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 2회. 최대 |Z| **4.62**, 과거 중심 대비 **+9.65℃**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 4창·9.65℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -68.9%p(9/4유효창). 같은 전후 기준의 TC-Z3 변화는 +9.56℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 23:48 (이동 Z 복귀보다 11분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z4 · 07/08 04:35~04:38**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.47**, 과거 중심 대비 **+6.95%p**.  
**다른 LOT:** 동종 LOT 240044에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 7.68~7.68%p)이 있다. 이 구간의 3창·6.95%p는 비교 관측의 상한을 넘는다(길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 -0.715℃(9/3유효창). 같은 전후 기준의 OP-Z4 변화는 +3.63%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 04:49 (이동 Z 복귀보다 6분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 43건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240069-Z-TC-Z3-0074 · 07/07 18:11~18:12**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 10건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 1창·3.32℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +13.7%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.58℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 18:15. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-01 · 07/07 19:00:51**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 184초 간격·0.165무차원는 비교 관측의 상한을 넘는다(절대 편차 0.165>0.0524무차원, 길이 184>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/07 19:04:25. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 13시간 48분 9초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-02 · 07/07 19:08:33**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 90초 간격·0.0851무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0851>0.0524무차원, 길이 90>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/07 19:10:37. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 13시간 40분 27초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-03 · 07/07 19:11:29**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 71초 간격·0.0704무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0704>0.0524무차원, 길이 71>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/07 19:13:47. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 13시간 37분 31초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-04 · 07/07 19:21:29**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 75초 간격·0.0753무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0753>0.0524무차원, 길이 75>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/07 19:24:03. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 13시간 27분 31초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0156 · 07/07 19:33~19:34**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·11.3%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.399℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -9.36%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:38 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-05 · 07/07 19:38:35**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 94초 간격·0.12무차원는 비교 관측의 상한을 넘는다(절대 편차 0.12>0.0524무차원, 길이 94>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/07 19:40:48. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 13시간 10분 25초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-06 · 07/07 19:41:40**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 67초 간격·0.0741무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0741>0.0524무차원, 길이 67>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/07 19:43:48. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 13시간 7분 20초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0167 · 07/07 19:44~19:45**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·23.4%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.09℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -20.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:50. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0187 · 07/07 20:04~20:05**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·59.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.15℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -56.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 20:17 (이동 Z 복귀보다 9분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0187 · 07/07 20:04~20:06**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 2창·3.12℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -44.4%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 +3.31℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 20:21 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0192 · 07/07 20:09~20:10**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 10건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 1창·4.4℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +8.65%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -4.25℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 20:13. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0264 · 07/07 21:21~21:22**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·12.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.569℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -8.66%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 21:25. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0276 · 07/07 21:33~21:34**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·17.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.06℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -20.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 21:37. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0356 · 07/07 22:53~22:54**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 12~12%p)이 있다. 이 구간의 1창·17.4%p는 비교 관측의 상한을 넘는다(절대 편차 17.4>12%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -1.47℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 +20.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 22:57. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 1건/737유효창 (100창당 0.14건) | 17.43~17.43 | 1~1창 | 74.152 |
| 240044 | 0건/51유효창 (100창당 0.00건) | — | — | 76.119 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 72.806 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 12.04~12.04 | 2~2창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0360 · 07/07 22:57~22:59**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 2창·61.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.42℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -48.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:06 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0360 · 07/07 22:57~23:00**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 3창·3.61℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -37%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 +2.68℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:13 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0364 · 07/07 23:01~23:04**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 10건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 3창·4.79℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +21.1%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 -4.77℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:09 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0393 · 07/07 23:30~23:34**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 4창·73%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +9.56℃(9/4유효창). 같은 전후 기준의 OP-Z3 변화는 -68.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:39 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0393 · 07/07 23:30~23:34**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 4창·9.65℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -68.9%p(9/4유효창). 같은 전후 기준의 TC-Z3 변화는 +9.56℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:48 (이동 Z 복귀보다 11분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z4-0397 · 07/07 23:34~23:35**

**다른 LOT:** 동종 LOT 240061에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 7.9~7.9%p)이 있다. 이 구간의 1창·6.15%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.03℃(9/1유효창). 같은 전후 기준의 OP-Z4 변화는 -5.87%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:38. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 1건/737유효창 (100창당 0.14건) | 6.15~6.15 | 1~1창 | 42.121 |
| 240044 | 0건/53유효창 (100창당 0.00건) | — | — | 41.819 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 7.899~7.899 | 2~2창 | 54.132 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 44.37 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z6-0405 · 07/07 23:42~23:44**

**다른 LOT:** 동종 LOT 240061, 240091에도 같은 방향 후보 2건(연속 2~2개 1분창, 절대 편차 6.42~9.68%p)이 있다. 이 구간의 2창·8.91%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.3℃(9/2유효창). 같은 전후 기준의 OP-Z6 변화는 +7.72%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:47. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 8.909~9.171 | 2~3창 | 68.085 |
| 240044 | 0건/53유효창 (100창당 0.00건) | — | — | 67.168 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 9.685~9.685 | 2~2창 | 66.614 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 6.416~6.416 | 2~2창 | 67.844 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z6-0406 · 07/07 23:43~23:44**

**다른 LOT:** 같은 강종 LOT 240044, 240061, 240091의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 +8.37%p(9/1유효창). 같은 전후 기준의 TC-Z6 변화는 -1.47℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:47. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 1.519~1.586 | 1~2창 | 859.72 |
| 240044 | 0건/52유효창 (100창당 0.00건) | — | — | 859.65 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 859.73 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 859.71 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0454 · 07/08 00:31~00:32**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·45.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.95℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -48.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:38. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0476 · 07/08 00:53~00:55**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 2창·62.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.15℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -64.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:58. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0477 · 07/08 00:54~00:55**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 1창·4.46℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -66.3%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +4.19℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 01:04 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0481 · 07/08 00:58~00:59**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 10건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 1창·5.86℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +31.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -6.61℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 01:03 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-CP-07 · 07/08 02:31:29**

**다른 LOT:** 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 529초 간격·0.459무차원는 비교 관측의 상한을 넘는다(절대 편차 0.459>0.0524무차원, 길이 529>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 07/08 02:41:06. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01094 감지 6시간 17분 31초 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240091 | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0595 · 07/08 02:52~02:53**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·72.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.65℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -91.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:56. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0626 · 07/08 03:23~03:25**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 2창·77.2%p는 비교 관측의 상한을 넘는다(절대 편차 77.2>75%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.97℃(8/2유효창). 같은 전후 기준의 OP-Z3 변화는 -53.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:29 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0627 · 07/08 03:24~03:26**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 2창·6.17℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -55.6%p(8/2유효창). 같은 전후 기준의 TC-Z3 변화는 +5.22℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:29. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0681 · 07/08 04:18~04:20**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 2창·50.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.66℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -57%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:23. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0682 · 07/08 04:19~04:20**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 1창·3.6℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -61.1%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.6℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:43 (이동 Z 복귀보다 16분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0685 · 07/08 04:22~04:24**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 10건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 2창·3.88℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +23.8%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -3.83℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:28 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0689 · 07/08 04:26~04:28**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 2창·73.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.14℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -62%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:31. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z4-0698 · 07/08 04:35~04:38**

**다른 LOT:** 동종 LOT 240044에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 7.68~7.68%p)이 있다. 이 구간의 3창·6.95%p는 비교 관측의 상한을 넘는다(길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 -0.715℃(9/3유효창). 같은 전후 기준의 OP-Z4 변화는 +3.63%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:49 (이동 Z 복귀보다 6분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 1건/737유효창 (100창당 0.14건) | 6.946~6.946 | 3~3창 | 42.121 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 7.678~7.678 | 2~2창 | 41.819 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 54.132 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 44.37 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0713 · 07/08 04:50~04:51**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 1창·77.2%p는 비교 관측의 상한을 넘는다(절대 편차 77.2>75%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.24℃(8/1유효창). 같은 전후 기준의 OP-Z3 변화는 -78.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:00 (이동 Z 복귀보다 6분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0714 · 07/08 04:51~04:54**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 3창·11.1℃는 비교 관측의 상한을 넘는다(절대 편차 11.1>10.6℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -80.9%p(8/3유효창). 같은 전후 기준의 TC-Z3 변화는 +11℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:01 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z6-0726 · 07/08 05:03~05:06**

**다른 LOT:** 동종 LOT 240061, 240091에도 같은 방향 후보 2건(연속 2~2개 1분창, 절대 편차 6.42~9.68%p)이 있다. 이 구간의 3창·9.17%p는 비교 관측의 상한을 넘는다(길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.52℃(9/3유효창). 같은 전후 기준의 OP-Z6 변화는 +7.69%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:10 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 8.909~9.171 | 2~3창 | 68.085 |
| 240044 | 0건/53유효창 (100창당 0.00건) | — | — | 67.168 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 9.685~9.685 | 2~2창 | 66.614 |
| 240091 | 1건/597유효창 (100창당 0.17건) | 6.416~6.416 | 2~2창 | 67.844 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z6-0727 · 07/08 05:04~05:06**

**다른 LOT:** 같은 강종 LOT 240044, 240061, 240091의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 +7.63%p(9/2유효창). 같은 전후 기준의 TC-Z6 변화는 -1.51℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:09. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 2건/737유효창 (100창당 0.27건) | 1.519~1.586 | 1~2창 | 859.72 |
| 240044 | 0건/52유효창 (100창당 0.00건) | — | — | 859.65 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 859.73 |
| 240091 | 0건/597유효창 (100창당 0.00건) | — | — | 859.71 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-OP-Z3-0802 · 07/08 06:19~06:21**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 18건(연속 1~4개 1분창, 절대 편차 11~75%p)이 있다. 이 구간의 2창·72.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 18건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.62℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -56%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:25 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240091 | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0803 · 07/08 06:20~06:22**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 9건(연속 1~5개 1분창, 절대 편차 2.45~10.6℃)이 있다. 이 구간의 2창·3.76℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 9건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -50.6%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 +3.57℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240091 | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240069-Z-TC-Z3-0807 · 07/08 06:24~06:26**

**다른 LOT:** 동종 LOT 240044, 240061, 240091에도 같은 방향 후보 10건(연속 1~3개 1분창, 절대 편차 2.28~9.5℃)이 있다. 이 구간의 2창·5.49℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 10건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +29.7%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 -6.23℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240069 (현재 LOT) | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240091 | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 813 | 737 | 28 | 76 | 56 | 20 | 0 |
| OP-Z3 | 813 | 737 | 25 | 76 | 56 | 20 | 0 |
| TC-Z4 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| OP-Z4 | 813 | 737 | 4 | 76 | 56 | 20 | 0 |
| TC-Z5 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| OP-Z5 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| TC-Z6 | 813 | 737 | 3 | 76 | 56 | 20 | 0 |
| OP-Z6 | 813 | 737 | 5 | 76 | 56 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 7건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 737창, 최대 절대 Z=2.50 / TC-Z5: 유효 737창, 최대 절대 Z=2.54 / OP-Z5: 유효 737창, 최대 절대 Z=2.55. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| TC-Z3 · 07/07 18:11:00–07/07 18:12:00 (끝 제외); 240069-Z-TC-Z3-0074 | 최대 절대 Z=3.12 (07/07 18:11:30, 창 중심); 1분 중앙값 856.132℃, 과거 중심 859.457℃, 차이 -3.325℃; SD=1.064; 실측 56/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 18:15:00 / 시작 고정 기준 3창 복귀 07/07 18:15:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -877.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 19:33:00–07/07 19:34:00 (끝 제외); 240069-Z-OP-Z3-0156 | 최대 절대 Z=3.85 (07/07 19:33:30, 창 중심); 1분 중앙값 64.076%, 과거 중심 75.420%, 차이 -11.344%p; SD=2.943; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 19:37:00 / 시작 고정 기준 3창 복귀 07/07 19:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -795.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 19:44:00–07/07 19:45:00 (끝 제외); 240069-Z-OP-Z3-0167 | 최대 절대 Z=5.19 (07/07 19:44:30, 창 중심); 1분 중앙값 51.007%, 과거 중심 74.384%, 차이 -23.377%p; SD=4.506; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 19:50:00 / 시작 고정 기준 3창 복귀 07/07 19:50:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -784.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 20:04:00–07/07 20:05:00 (끝 제외); 240069-Z-OP-Z3-0187 | 최대 절대 Z=5.64 (07/07 20:04:30, 창 중심); 1분 중앙값 11.443%, 과거 중심 71.209%, 차이 -59.766%p; SD=10.602; 실측 55/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 20:08:00 / 시작 고정 기준 3창 복귀 07/07 20:17:00 | 고정 범위 복귀가 이동 Z 복귀보다 9분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -764.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/07 20:04:00–07/07 20:06:00 (끝 제외); 240069-Z-TC-Z3-0187 | 최대 절대 Z=4.94 (07/07 20:04:30, 창 중심); 1분 중앙값 862.800℃, 과거 중심 859.679℃, 차이 +3.121℃; SD=0.632; 실측 55/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 20:13:00 / 시작 고정 기준 3창 복귀 07/07 20:21:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -764.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/07 20:09:00–07/07 20:10:00 (끝 제외); 240069-Z-TC-Z3-0192 | 최대 절대 Z=3.69 (07/07 20:09:30, 창 중심); 1분 중앙값 855.401℃, 과거 중심 859.796℃, 차이 -4.395℃; SD=1.190; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 20:13:00 / 시작 고정 기준 3창 복귀 07/07 20:13:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -759.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 21:21:00–07/07 21:22:00 (끝 제외); 240069-Z-OP-Z3-0264 | 최대 절대 Z=3.03 (07/07 21:21:30, 창 중심); 1분 중앙값 61.723%, 과거 중심 73.868%, 차이 -12.145%p; SD=4.011; 실측 58/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 21:25:00 / 시작 고정 기준 3창 복귀 07/07 21:25:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -687.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 21:33:00–07/07 21:34:00 (끝 제외); 240069-Z-OP-Z3-0276 | 최대 절대 Z=3.03 (07/07 21:33:30, 창 중심); 1분 중앙값 56.816%, 과거 중심 74.029%, 차이 -17.213%p; SD=5.672; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 21:37:00 / 시작 고정 기준 3창 복귀 07/07 21:37:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -675.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 22:53:00–07/07 22:54:00 (끝 제외); 240069-Z-OP-Z3-0356 | 최대 절대 Z=3.54 (07/07 22:53:30, 창 중심); 1분 중앙값 91.456%, 과거 중심 74.024%, 차이 +17.432%p; SD=4.929; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 22:57:00 / 시작 고정 기준 3창 복귀 07/07 22:57:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -595.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 22:57:00–07/07 22:59:00 (끝 제외); 240069-Z-OP-Z3-0360 | 최대 절대 Z=6.41 (07/07 22:58:30, 창 중심); 1분 중앙값 12.576%, 과거 중심 74.467%, 차이 -61.891%p; SD=9.651; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:02:00 / 시작 고정 기준 3창 복귀 07/07 23:06:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -590.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/07 22:57:00–07/07 23:00:00 (끝 제외); 240069-Z-TC-Z3-0360 | 최대 절대 Z=6.74 (07/07 22:58:30, 창 중심); 1분 중앙값 863.311℃, 과거 중심 859.704℃, 차이 +3.607℃; SD=0.535; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:07:00 / 시작 고정 기준 3창 복귀 07/07 23:13:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -590.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/07 23:01:00–07/07 23:04:00 (끝 제외); 240069-Z-TC-Z3-0364 | 최대 절대 Z=4.22 (07/07 23:02:30, 창 중심); 1분 중앙값 854.946℃, 과거 중심 859.736℃, 차이 -4.790℃; SD=1.135; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:07:00 / 시작 고정 기준 3창 복귀 07/07 23:09:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -586.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/07 23:30:00–07/07 23:34:00 (끝 제외); 240069-Z-OP-Z3-0393 | 최대 절대 Z=8.58 (07/07 23:30:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 73.044%, 차이 -73.044%p; SD=8.514; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:37:00 / 시작 고정 기준 3창 복귀 07/07 23:39:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -558.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/07 23:30:00–07/07 23:34:00 (끝 제외); 240069-Z-TC-Z3-0393 | 최대 절대 Z=4.62 (07/07 23:31:30, 창 중심); 1분 중앙값 868.824℃, 과거 중심 859.175℃, 차이 +9.650℃; SD=2.088; 실측 58/60개·과거 28/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:37:00 / 시작 고정 기준 3창 복귀 07/07 23:48:00 | 고정 범위 복귀가 이동 Z 복귀보다 11분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -557.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 07/07 23:34:00–07/07 23:35:00 (끝 제외); 240069-Z-OP-Z4-0397 | 최대 절대 Z=3.07 (07/07 23:34:30, 창 중심); 1분 중앙값 36.414%, 과거 중심 42.563%, 차이 -6.150%p; SD=2.000 (하한 적용); 실측 58/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:38:00 / 시작 고정 기준 3창 복귀 07/07 23:38:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -554.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 07/07 23:42:00–07/07 23:44:00 (끝 제외); 240069-Z-OP-Z6-0405 | 최대 절대 Z=4.26 (07/07 23:43:30, 창 중심); 1분 중앙값 74.667%, 과거 중심 65.758%, 차이 +8.909%p; SD=2.089; 실측 54/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:47:00 / 시작 고정 기준 3창 복귀 07/07 23:47:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -545.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 07/07 23:43:00–07/07 23:44:00 (끝 제외); 240069-Z-TC-Z6-0406 | 최대 절대 Z=3.04 (07/07 23:43:30, 창 중심); 1분 중앙값 858.258℃, 과거 중심 859.777℃, 차이 -1.519℃; SD=0.500 (하한 적용); 실측 54/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/07 23:47:00 / 시작 고정 기준 3창 복귀 07/07 23:47:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -545.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 00:31:00–07/08 00:32:00 (끝 제외); 240069-Z-OP-Z3-0454 | 최대 절대 Z=4.86 (07/08 00:31:30, 창 중심); 1분 중앙값 30.304%, 과거 중심 76.029%, 차이 -45.725%p; SD=9.407; 실측 60/60개·과거 26/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 00:38:00 / 시작 고정 기준 3창 복귀 07/08 00:38:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -497.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 00:53:00–07/08 00:55:00 (끝 제외); 240069-Z-OP-Z3-0476 | 최대 절대 Z=4.52 (07/08 00:53:30, 창 중심); 1분 중앙값 9.514%, 과거 중심 71.663%, 차이 -62.149%p; SD=13.762; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 00:58:00 / 시작 고정 기준 3창 복귀 07/08 00:58:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -475.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 00:54:00–07/08 00:55:00 (끝 제외); 240069-Z-TC-Z3-0477 | 최대 절대 Z=3.39 (07/08 00:54:30, 창 중심); 1분 중앙값 863.831℃, 과거 중심 859.370℃, 차이 +4.461℃; SD=1.315; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 00:58:00 / 시작 고정 기준 3창 복귀 07/08 01:04:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -474.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 00:58:00–07/08 00:59:00 (끝 제외); 240069-Z-TC-Z3-0481 | 최대 절대 Z=3.16 (07/08 00:58:30, 창 중심); 1분 중앙값 853.636℃, 과거 중심 859.500℃, 차이 -5.864℃; SD=1.857; 실측 58/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 01:02:00 / 시작 고정 기준 3창 복귀 07/08 01:03:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -470.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 02:52:00–07/08 02:53:00 (끝 제외); 240069-Z-OP-Z3-0595 | 최대 절대 Z=3.32 (07/08 02:52:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 72.516%, 차이 -72.516%p; SD=21.842; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 02:56:00 / 시작 고정 기준 3창 복귀 07/08 02:56:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -356.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 03:23:00–07/08 03:25:00 (끝 제외); 240069-Z-OP-Z3-0626 | 최대 절대 Z=3.74 (07/08 03:24:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 77.179%, 차이 -77.179%p; SD=20.614; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 03:28:00 / 시작 고정 기준 3창 복귀 07/08 03:29:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -324.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 03:24:00–07/08 03:26:00 (끝 제외); 240069-Z-TC-Z3-0627 | 최대 절대 Z=3.05 (07/08 03:24:30, 창 중심); 1분 중앙값 864.342℃, 과거 중심 858.168℃, 차이 +6.174℃; SD=2.022; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 03:29:00 / 시작 고정 기준 3창 복귀 07/08 03:29:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -324.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 04:18:00–07/08 04:20:00 (끝 제외); 240069-Z-OP-Z3-0681 | 최대 절대 Z=5.43 (07/08 04:18:30, 창 중심); 1분 중앙값 23.801%, 과거 중심 74.588%, 차이 -50.786%p; SD=9.353; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:23:00 / 시작 고정 기준 3창 복귀 07/08 04:23:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -270.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 04:19:00–07/08 04:20:00 (끝 제외); 240069-Z-TC-Z3-0682 | 최대 절대 Z=4.90 (07/08 04:19:30, 창 중심); 1분 중앙값 863.310℃, 과거 중심 859.711℃, 차이 +3.599℃; SD=0.734; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:27:00 / 시작 고정 기준 3창 복귀 07/08 04:43:00 | 고정 범위 복귀가 이동 Z 복귀보다 16분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -269.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 04:22:00–07/08 04:24:00 (끝 제외); 240069-Z-TC-Z3-0685 | 최대 절대 Z=3.17 (07/08 04:23:30, 창 중심); 1분 중앙값 855.827℃, 과거 중심 859.711℃, 차이 -3.884℃; SD=1.224; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:27:00 / 시작 고정 기준 3창 복귀 07/08 04:28:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -265.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 04:26:00–07/08 04:28:00 (끝 제외); 240069-Z-OP-Z3-0689 | 최대 절대 Z=3.72 (07/08 04:27:30, 창 중심); 1분 중앙값 0.796%, 과거 중심 74.588%, 차이 -73.792%p; SD=19.849; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:31:00 / 시작 고정 기준 3창 복귀 07/08 04:31:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -261.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 07/08 04:35:00–07/08 04:38:00 (끝 제외); 240069-Z-OP-Z4-0698 | 최대 절대 Z=3.47 (07/08 04:35:30, 창 중심); 1분 중앙값 48.292%, 과거 중심 41.346%, 차이 +6.946%p; SD=2.000 (하한 적용); 실측 55/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:43:00 / 시작 고정 기준 3창 복귀 07/08 04:49:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -253.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 04:50:00–07/08 04:51:00 (끝 제외); 240069-Z-OP-Z3-0713 | 최대 절대 Z=3.17 (07/08 04:50:30, 창 중심); 1분 중앙값 3.506%, 과거 중심 80.741%, 차이 -77.235%p; SD=24.327; 실측 58/60개·과거 26/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:54:00 / 시작 고정 기준 3창 복귀 07/08 05:00:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -238.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 04:51:00–07/08 04:54:00 (끝 제외); 240069-Z-TC-Z3-0714 | 최대 절대 Z=3.88 (07/08 04:52:30, 창 중심); 1분 중앙값 869.432℃, 과거 중심 858.372℃, 차이 +11.060℃; SD=2.853; 실측 59/60개·과거 26/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 04:57:00 / 시작 고정 기준 3창 복귀 07/08 05:01:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -236.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 07/08 05:03:00–07/08 05:06:00 (끝 제외); 240069-Z-OP-Z6-0726 | 최대 절대 Z=4.56 (07/08 05:04:30, 창 중심); 1분 중앙값 78.347%, 과거 중심 69.176%, 차이 +9.171%p; SD=2.010; 실측 59/60개·과거 25/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 05:09:00 / 시작 고정 기준 3창 복귀 07/08 05:10:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -224.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 07/08 05:04:00–07/08 05:06:00 (끝 제외); 240069-Z-TC-Z6-0727 | 최대 절대 Z=3.17 (07/08 05:04:30, 창 중심); 1분 중앙값 858.139℃, 과거 중심 859.725℃, 차이 -1.586℃; SD=0.500 (하한 적용); 실측 59/60개·과거 25/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 05:09:00 / 시작 고정 기준 3창 복귀 07/08 05:09:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -224.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 07/08 06:19:00–07/08 06:21:00 (끝 제외); 240069-Z-OP-Z3-0802 | 최대 절대 Z=5.49 (07/08 06:20:30, 창 중심); 1분 중앙값 3.538%, 과거 중심 76.060%, 차이 -72.522%p; SD=13.221; 실측 55/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 06:24:00 / 시작 고정 기준 3창 복귀 07/08 06:25:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -148.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 06:20:00–07/08 06:22:00 (끝 제외); 240069-Z-TC-Z3-0803 | 최대 절대 Z=4.57 (07/08 06:20:30, 창 중심); 1분 중앙값 863.686℃, 과거 중심 859.924℃, 차이 +3.762℃; SD=0.823; 실측 55/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 06:29:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -148.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 07/08 06:24:00–07/08 06:26:00 (끝 제외); 240069-Z-TC-Z3-0807 | 최대 절대 Z=3.78 (07/08 06:24:30, 창 중심); 1분 중앙값 854.438℃, 과거 중심 859.924℃, 차이 -5.486℃; SD=1.452; 실측 56/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 07/08 06:29:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01094 감지 대비 Z 최대창 중심 -144.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240069-Z-TC-Z3-0393:** OP-Z3 사건창 중앙값−직전10분 중앙값 -68.85 (9/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240069-Z-OP-Z3-0393:** TC-Z3 사건창 중앙값−직전10분 중앙값 +9.562 (9/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 07/07 16:57:00–07/07 17:20:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 / 과거 유효 최소 0창 <20 |
| 07/07 17:35:00–07/07 17:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 18:07:00–07/07 18:08:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/07 18:34:00–07/07 18:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 18:38:00–07/07 18:39:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 18:41:00–07/07 18:42:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/07 18:52:00–07/07 18:53:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/07 19:02:00–07/07 19:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 19:07:00–07/07 19:08:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/07 19:17:00–07/07 19:18:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 19:46:00–07/07 19:47:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/07 19:55:00–07/07 19:56:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/07 20:08:00–07/07 20:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 20:13:00–07/07 20:14:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 20:49:00–07/07 20:50:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/07 20:59:00–07/07 21:00:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/07 21:08:00–07/07 21:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/07 21:45:00–07/07 21:46:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/07 23:15:00–07/07 23:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/07 23:29:00–07/07 23:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/07 23:39:00–07/07 23:40:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/07 23:48:00–07/07 23:49:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/07 23:56:00–07/07 23:57:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 00:01:00–07/08 00:02:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/08 00:15:00–07/08 00:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/08 00:20:00–07/08 00:21:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 46개 <54 |
| 07/08 00:30:00–07/08 00:31:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 07/08 00:34:00–07/08 00:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 01:06:00–07/08 01:07:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/08 01:16:00–07/08 01:17:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 01:20:00–07/08 01:21:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/08 01:35:00–07/08 01:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/08 01:49:00–07/08 01:50:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 02:18:00–07/08 02:19:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 02:22:00–07/08 02:23:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 02:51:00–07/08 02:52:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 03:18:00–07/08 03:19:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 07/08 03:20:00–07/08 03:21:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 47개 <54 |
| 07/08 03:30:00–07/08 03:31:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/08 03:34:00–07/08 03:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 03:39:00–07/08 03:40:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/08 04:34:00–07/08 04:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 04:39:00–07/08 04:40:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 04:44:00–07/08 04:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 04:49:00–07/08 04:50:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 05:02:00–07/08 05:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 07/08 05:21:00–07/08 05:22:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 05:26:00–07/08 05:27:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 05:35:00–07/08 05:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 07/08 05:44:00–07/08 05:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 07/08 05:54:00–07/08 05:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 48개 <54 |
| 07/08 06:03:00–07/08 06:04:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 06:16:00–07/08 06:17:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 07/08 06:29:00–07/08 06:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 35개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240069-CP-01 · 07/07 19:00:51 | +0.1653무차원 · 175점/184초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 184초 간격·0.165무차원는 비교 관측의 상한을 넘는다(절대 편차 0.165>0.0524무차원, 길이 184>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 19:04:25; WO-2024-01094 감지 13시간 48분 9초 전에 CP 원본 피크 |
| 240069-CP-02 · 07/07 19:08:33 | +0.0851무차원 · 88점/90초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 90초 간격·0.0851무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0851>0.0524무차원, 길이 90>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 19:10:37; WO-2024-01094 감지 13시간 40분 27초 전에 CP 원본 피크 |
| 240069-CP-03 · 07/07 19:11:29 | +0.0704무차원 · 69점/71초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 71초 간격·0.0704무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0704>0.0524무차원, 길이 71>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 19:13:47; WO-2024-01094 감지 13시간 37분 31초 전에 CP 원본 피크 |
| 240069-CP-04 · 07/07 19:21:29 | +0.0753무차원 · 74점/75초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 75초 간격·0.0753무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0753>0.0524무차원, 길이 75>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 19:24:03; WO-2024-01094 감지 13시간 27분 31초 전에 CP 원본 피크 |
| 240069-CP-05 · 07/07 19:38:35 | +0.1195무차원 · 94점/94초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 94초 간격·0.12무차원는 비교 관측의 상한을 넘는다(절대 편차 0.12>0.0524무차원, 길이 94>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 19:40:48; WO-2024-01094 감지 13시간 10분 25초 전에 CP 원본 피크 |
| 240069-CP-06 · 07/07 19:41:40 | +0.0741무차원 · 65점/67초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 67초 간격·0.0741무차원는 비교 관측의 상한을 넘는다(절대 편차 0.0741>0.0524무차원, 길이 67>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 19:43:48; WO-2024-01094 감지 13시간 7분 20초 전에 CP 원본 피크 |
| 240069-CP-07 · 07/08 02:31:29 | +0.4590무차원 · 515점/529초 간격 | 동종 LOT 240091에도 같은 방향 후보 1건(첫/끝 실측 간격 55~55초, 절대 편차 0.0524~0.0524무차원)이 있다. 이 구간의 529초 간격·0.459무차원는 비교 관측의 상한을 넘는다(절대 편차 0.459>0.0524무차원, 길이 529>55초 간격). SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 02:41:06; WO-2024-01094 감지 6시간 17분 31초 전에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240069-CP-01 · 피크 07/07 19:00:51 · 복귀 펄스 후보**

- **관측:** 원값 0.615396, LOT 중앙값 0.450109, 편차 +0.165288. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 175점, 첫·끝 실측 간격 184초, 기대 1초 격자 내 유효 비율 94.6%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 07/07 18:59:45=0.469127 / 직후 07/07 19:02:51=0.469689; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:04:25.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 152082–152258에 연결했다(0부터 시작).

**240069-CP-02 · 피크 07/07 19:08:33 · 복귀 미확인 원본 이탈**

- **관측:** 원값 0.535258, LOT 중앙값 0.450109, 편차 +0.0851495. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 88점, 첫·끝 실측 간격 90초, 기대 1초 격자 내 유효 비율 96.7%. 최근 30분 내 피크 2회(해당 사건 포함).
- **전후·복귀:** 직전 07/07 19:07:58=nan / 직후 07/07 19:09:30=0.469624; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:10:37.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 152552–152640에 연결했다(0부터 시작).

**240069-CP-03 · 피크 07/07 19:11:29 · 복귀 펄스 후보**

- **관측:** 원값 0.520463, LOT 중앙값 0.450109, 편차 +0.0703545. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 69점, 첫·끝 실측 간격 71초, 기대 1초 격자 내 유효 비율 95.8%. 최근 30분 내 피크 3회(해당 사건 포함).
- **전후·복귀:** 직전 07/07 19:11:10=0.466377 / 직후 07/07 19:12:23=0.469792; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:13:47.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 152738–152808에 연결했다(0부터 시작).

**240069-CP-04 · 피크 07/07 19:21:29 · 복귀 펄스 후보**

- **관측:** 원값 0.525375, LOT 중앙값 0.450109, 편차 +0.0752665. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 74점, 첫·끝 실측 간격 75초, 기대 1초 격자 내 유효 비율 97.4%. 최근 30분 내 피크 4회(해당 사건 포함).
- **전후·복귀:** 직전 07/07 19:21:03=0.468585 / 직후 07/07 19:22:20=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:24:03.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 153307–153381에 연결했다(0부터 시작).

**240069-CP-05 · 피크 07/07 19:38:35 · 복귀 펄스 후보**

- **관측:** 원값 0.569656, LOT 중앙값 0.450109, 편차 +0.119548. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 94점, 첫·끝 실측 간격 94초, 기대 1초 격자 내 유효 비율 98.9%. 최근 30분 내 피크 3회(해당 사건 포함).
- **전후·복귀:** 직전 07/07 19:38:08=0.464642 / 직후 07/07 19:39:44=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:40:48.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 154302–154396에 연결했다(0부터 시작).

**240069-CP-06 · 피크 07/07 19:41:40 · 복귀 펄스 후보**

- **관측:** 원값 0.524196, LOT 중앙값 0.450109, 편차 +0.0740875. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 65점, 첫·끝 실측 간격 67초, 기대 1초 격자 내 유효 비율 95.6%. 최근 30분 내 피크 3회(해당 사건 포함).
- **전후·복귀:** 직전 07/07 19:41:21=0.469995 / 직후 07/07 19:42:30=0.469539; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:43:48.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 154489–154555에 연결했다(0부터 시작).

**240069-CP-07 · 피크 07/08 02:31:29 · 복귀 펄스 후보**

- **관측:** 원값 0.909111, LOT 중앙값 0.450109, 편차 +0.459002. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 515점, 첫·끝 실측 간격 529초, 기대 1초 격자 내 유효 비율 97.2%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 07/08 02:30:44=0.467995 / 직후 07/08 02:39:35=0.4696; 이후 60개 연속 실측 ±0.02 복귀 확인 07/08 02:41:06.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 178112–178628에 연결했다(0부터 시작).

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240069-CP-01 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -828.15분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +1.573 (285/61점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -6.222 (285/61점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +5.02e-13 (285/61점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240069-CP-02 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -820.45분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +0.131 (289/59점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -1.719 (289/59점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +1.2e-14 (289/59점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240069-CP-03 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -817.52분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +0.158 (290/60점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -1.632 (290/60점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 -9.205e-13 (290/60점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240069-CP-04 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -807.52분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -0.023 (291/60점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +0.4517 (291/60점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 -5.83e-13 (291/60점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240069-CP-05 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -790.42분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -0.738 (295/60점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +4.421 (295/60점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +4.03e-13 (295/60점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240069-CP-06 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -787.33분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -0.23 (293/57점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +2.895 (293/57점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +4.15e-13 (293/57점); 같은 시각 동반 관측이며 원인 확정 아님 |
| 240069-CP-07 | 정비 대조: WO-2024-01094 감지 대비 CP 원본 피크 -377.52분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +0.304 (286/60점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 -1.145 (286/60점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +3.485e-13 (286/60점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240069, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| OP-Z7 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| TC-Z8 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |
| OP-Z8 | 813 | 737 | 0 | 76 | 56 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 737창, 최대 절대 Z=0.88 / OP-Z7: 유효 737창, 최대 절대 Z=2.81 / TC-Z8: 유효 737창, 최대 절대 Z=0.47 / OP-Z8: 유효 737창, 최대 절대 Z=1.07. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 07/07 16:57:00–07/07 17:20:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 / 과거 유효 최소 0창 <20 |
| 07/07 17:35:00–07/07 17:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 18:07:00–07/07 18:08:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/07 18:34:00–07/07 18:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 18:38:00–07/07 18:39:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 18:41:00–07/07 18:42:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/07 18:52:00–07/07 18:53:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/07 19:02:00–07/07 19:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 19:07:00–07/07 19:08:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/07 19:17:00–07/07 19:18:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 19:46:00–07/07 19:47:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/07 19:55:00–07/07 19:56:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/07 20:08:00–07/07 20:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 20:13:00–07/07 20:14:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 20:49:00–07/07 20:50:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/07 20:59:00–07/07 21:00:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/07 21:08:00–07/07 21:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/07 21:45:00–07/07 21:46:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/07 23:15:00–07/07 23:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/07 23:29:00–07/07 23:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/07 23:39:00–07/07 23:40:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/07 23:48:00–07/07 23:49:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/07 23:56:00–07/07 23:57:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 00:01:00–07/08 00:02:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/08 00:15:00–07/08 00:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/08 00:20:00–07/08 00:21:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 46개 <54 |
| 07/08 00:30:00–07/08 00:31:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 07/08 00:34:00–07/08 00:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 01:06:00–07/08 01:07:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/08 01:16:00–07/08 01:17:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 01:20:00–07/08 01:21:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/08 01:35:00–07/08 01:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/08 01:49:00–07/08 01:50:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 02:18:00–07/08 02:19:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 02:22:00–07/08 02:23:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 02:51:00–07/08 02:52:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 03:18:00–07/08 03:19:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 07/08 03:20:00–07/08 03:21:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 47개 <54 |
| 07/08 03:30:00–07/08 03:31:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/08 03:34:00–07/08 03:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 03:39:00–07/08 03:40:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/08 04:34:00–07/08 04:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 04:39:00–07/08 04:40:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 04:44:00–07/08 04:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 04:49:00–07/08 04:50:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 05:02:00–07/08 05:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 07/08 05:21:00–07/08 05:22:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 05:26:00–07/08 05:27:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 05:35:00–07/08 05:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 07/08 05:44:00–07/08 05:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 07/08 05:54:00–07/08 05:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 48개 <54 |
| 07/08 06:03:00–07/08 06:04:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 06:16:00–07/08 06:17:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 07/08 06:29:00–07/08 06:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 35개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240069)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **7건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 46836 | 56 | 원본 ±0.05 사건 7건 · 최대 절대 편차 0.459 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 46836 | 56 | 원본 중앙값 1.14042e-10, 최저 1.11702e-10, 최고 1.15502e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 46830 | 56 | 원본 중앙값 329.491, 최저 319.007, 최고 342.195; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 46836 | 56 | 원본 중앙값 285.476, 최저 270.484, 최고 298.217; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 46836 | 56 | 원본 중앙값 281.01, 최저 270.211, 최고 291.487; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-01094 · BM / 메모: 1호기 재발, 절연 저하**

**실제 시각:** 감지 07/08 08:49:00 / 작업 07/08 11:11:00~07/08 19:04:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-01094 감지 6시간 17분 31초 전에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 2건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 2건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01094 · BM | 07/08 08:49:00 | 07/08 11:11:00 | 07/08 19:04:00 | 1호기 재발, 절연 저하 | SEL-009 O링 세트 ×3 (교체 사유: 예방) | 작업은 LOT 관측 밖 |
| WO-2024-01111 · TBM | 07/10 22:24:00 | 07/10 23:15:00 | 07/11 00:05:00 | 내용 미기재 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-01094 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 7건, 센서별 Z 후보 42건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240069-CP-01 · 07/07 19:00:51 · CP 편차 +0.16529 · 175점/184초 · 완료 대비 피크 -1443.15분 · 감지는 피크 대비 +828.15분. 복귀 펄스 후보; 직전 07/07 18:59:45=0.469127 / 직후 07/07 19:02:51=0.469689; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:04:25.
  - 240069-CP-02 · 07/07 19:08:33 · CP 편차 +0.085149 · 88점/90초 · 완료 대비 피크 -1435.45분 · 감지는 피크 대비 +820.45분. 복귀 미확인 원본 이탈; 직전 07/07 19:07:58=nan / 직후 07/07 19:09:30=0.469624; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:10:37.
  - 240069-CP-03 · 07/07 19:11:29 · CP 편차 +0.070354 · 69점/71초 · 완료 대비 피크 -1432.52분 · 감지는 피크 대비 +817.52분. 복귀 펄스 후보; 직전 07/07 19:11:10=0.466377 / 직후 07/07 19:12:23=0.469792; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:13:47.
  - 240069-CP-04 · 07/07 19:21:29 · CP 편차 +0.075267 · 74점/75초 · 완료 대비 피크 -1422.52분 · 감지는 피크 대비 +807.52분. 복귀 펄스 후보; 직전 07/07 19:21:03=0.468585 / 직후 07/07 19:22:20=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:24:03.
  - 240069-CP-05 · 07/07 19:38:35 · CP 편차 +0.11955 · 94점/94초 · 완료 대비 피크 -1405.42분 · 감지는 피크 대비 +790.42분. 복귀 펄스 후보; 직전 07/07 19:38:08=0.464642 / 직후 07/07 19:39:44=nan; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:40:48.
  - 240069-CP-06 · 07/07 19:41:40 · CP 편차 +0.074087 · 65점/67초 · 완료 대비 피크 -1402.33분 · 감지는 피크 대비 +787.33분. 복귀 펄스 후보; 직전 07/07 19:41:21=0.469995 / 직후 07/07 19:42:30=0.469539; 이후 60개 연속 실측 ±0.02 복귀 확인 07/07 19:43:48.
  - 240069-CP-07 · 07/08 02:31:29 · CP 편차 +0.459 · 515점/529초 · 완료 대비 피크 -992.52분 · 감지는 피크 대비 +377.52분. 복귀 펄스 후보; 직전 07/08 02:30:44=0.467995 / 직후 07/08 02:39:35=0.4696; 이후 60개 연속 실측 ±0.02 복귀 확인 07/08 02:41:06.
- **동반 양상:** OP-Z1 2건·최대 절대 Z 5.05 / OP-Z2 2건·최대 절대 Z 4.30 / OP-Z3 16건·최대 절대 Z 8.58 / OP-Z4 2건·최대 절대 Z 3.47 / OP-Z6 2건·최대 절대 Z 4.56 / TC-Z1 2건·최대 절대 Z 3.49 / TC-Z3 14건·최대 절대 Z 6.74 / TC-Z6 2건·최대 절대 Z 3.17. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 07/08 08:49:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 07/08 11:11:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 07/08 19:04:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-01111 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 34건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240069-CP-07 · 07/08 02:31:29 · CP 편차 +0.459 · 515점/529초 · 완료 대비 피크 -4173.52분 · 감지는 피크 대비 +4072.52분. 복귀 펄스 후보; 직전 07/08 02:30:44=0.467995 / 직후 07/08 02:39:35=0.4696; 이후 60개 연속 실측 ±0.02 복귀 확인 07/08 02:41:06.
- **동반 양상:** OP-Z1 2건·최대 절대 Z 5.05 / OP-Z2 2건·최대 절대 Z 4.30 / OP-Z3 11건·최대 절대 Z 8.58 / OP-Z4 2건·최대 절대 Z 3.47 / OP-Z6 2건·최대 절대 Z 4.56 / TC-Z1 2건·최대 절대 Z 3.49 / TC-Z3 11건·최대 절대 Z 6.74 / TC-Z6 2건·최대 절대 Z 3.17. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 07/10 22:24:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 07/10 23:15:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 07/11 00:05:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240069)

**240069 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240069)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240069)
lot_conclusion(240069)
lot_execution.append({'LOT':240069,'확대 구간':len(LOT_EVENTS[240069]),'강종 대조':240069 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240091"></a>
### 8.9 LOT 240091 · SPHC · 09/08

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240091)

#### LOT 240091 · 먼저 볼 변화

**지속·반복 후보 9건 / CP 원본 후보 1건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 09/08 23:33:47**

**관측:** 원본 편차 **+0.0524무차원**, 56점·첫/끝 간격 55초. 1초 단발 여부: 아님.  
**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 7건(첫/끝 실측 간격 67~529초, 절대 편차 0.0704~0.459무차원)이 있다. 이 구간의 55초 간격·0.0524무차원는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 09/08 23:35:24. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01419 감지 2일 13시간 21분 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — OP-Z2 · 09/09 08:27~08:33**

**관측:** |Z|≥3 연속 **6개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.37**, 과거 중심 대비 **-8.74%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 5건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 6창·8.74%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.11℃(10/6유효창). 같은 전후 기준의 OP-Z2 변화는 -9.45%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:43 (이동 Z 복귀보다 7분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — TC-Z3 · 09/09 07:07~07:12**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **7.03**, 과거 중심 대비 **+8.06℃**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 5창·8.06℃는 비교 관측의 상한을 넘는다(길이 5>4창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71.8%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.24℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 07:20 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240091

- **관측 범위:** 09/08 22:03:00–09/09 08:44:12.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 545개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 9개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 45개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 1건 중 확인된 1초 단발 0건, 복귀 펄스 후보 1건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240091, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 2건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z2 · 09/09 08:27~08:33**

**관측:** |Z|≥3 연속 **6개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.37**, 과거 중심 대비 **-8.74%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 5건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 6창·8.74%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.11℃(10/6유효창). 같은 전후 기준의 OP-Z2 변화는 -9.45%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:43 (이동 Z 복귀보다 7분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z1 · 09/09 08:25~08:30**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.86**, 과거 중심 대비 **-9.2%p**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 5건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·9.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.29℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:39 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 09/09 08:37~08:38**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.21**, 과거 중심 대비 **-1.6℃**.  
**다른 LOT:** 동종 LOT 240061에도 같은 방향 후보 1건(연속 4~4개 1분창, 절대 편차 1.85~1.85℃)이 있다. 이 구간의 1창·1.6℃는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:44 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 3건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240091-Z-OP-Z1-0622 · 09/09 08:25~08:30**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 5건(연속 2~5개 1분창, 절대 편차 6.66~12.6%p)이 있다. 이 구간의 5창·9.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.29℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -11%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 08:39 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/597유효창 (100창당 0.17건) | 9.204~9.204 | 5~5창 | 73.431 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.55~10.55 | 5~5창 | 74.266 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.656~12.58 | 2~3창 | 72.702 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 10.54~11.55 | 5~5창 | 72.861 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z2-0624 · 09/09 08:27~08:33**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 5건(연속 1~6개 1분창, 절대 편차 6.68~10.7%p)이 있다. 이 구간의 6창·8.74%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.11℃(10/6유효창). 같은 전후 기준의 OP-Z2 변화는 -9.45%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 08:43 (이동 Z 복귀보다 7분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/597유효창 (100창당 0.17건) | 8.744~8.744 | 6~6창 | 22.73 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 10.68~10.68 | 6~6창 | 25.212 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 6.682~9.561 | 1~3창 | 19.189 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 8.473~8.846 | 4~5창 | 21.399 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-CLN-IN-0634 · 09/09 08:37~08:38**

**다른 LOT:** 동종 LOT 240061에도 같은 방향 후보 1건(연속 4~4개 1분창, 절대 편차 1.85~1.85℃)이 있다. 이 구간의 1창·1.6℃는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 08:44 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/597유효창 (100창당 0.17건) | 1.605~1.605 | 1~1창 | 66.662 |
| 240044 | 0건/52유효창 (100창당 0.00건) | — | — | 64.529 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 1.848~1.848 | 4~4창 | 69.526 |
| 240069 | 0건/737유효창 (100창당 0.00건) | — | — | 66.613 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 642 | 597 | 1 | 45 | 25 | 20 | 0 |
| TC-Z1 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| OP-Z1 | 642 | 597 | 5 | 45 | 25 | 20 | 0 |
| TC-Z2 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| OP-Z2 | 642 | 597 | 6 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z1: 유효 597창, 최대 절대 Z=2.93 / TC-Z2: 유효 597창, 최대 절대 Z=2.56. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 09/09 08:25:00–09/09 08:30:00 (끝 제외); 240091-Z-OP-Z1-0622 | 최대 절대 Z=3.86 (09/09 08:26:30, 창 중심); 1분 중앙값 64.062%, 과거 중심 73.266%, 차이 -9.204%p; SD=2.386; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:33:00 / 시작 고정 기준 3창 복귀 09/09 08:39:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3148.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 09/09 08:27:00–09/09 08:33:00 (끝 제외); 240091-Z-OP-Z2-0624 | 최대 절대 Z=4.37 (09/09 08:28:30, 창 중심); 1분 중앙값 15.542%, 과거 중심 24.286%, 차이 -8.744%p; SD=2.000 (하한 적용); 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 6창, 같은 방향 최장 6창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:36:00 / 시작 고정 기준 3창 복귀 09/09 08:43:00 | 고정 범위 복귀가 이동 Z 복귀보다 7분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3146.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 09/09 08:37:00–09/09 08:38:00 (끝 제외); 240091-Z-CLN-IN-0634 | 최대 절대 Z=3.21 (09/09 08:37:30, 창 중심); 1분 중앙값 65.568℃, 과거 중심 67.172℃, 차이 -1.605℃; SD=0.501; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:41:00 / 시작 고정 기준 3창 복귀 09/09 08:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3137.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240091-Z-OP-Z2-0624:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.108 (10/6유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240091-Z-OP-Z1-0622:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.293 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 09/08 22:03:00–09/08 22:23:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 09/08 22:46:00–09/08 22:48:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 09/09 00:12:00–09/09 00:13:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 01:13:00–09/09 01:14:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 01:16:00–09/09 01:17:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 01:18:00–09/09 01:19:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 01:59:00–09/09 02:00:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 09/09 02:15:00–09/09 02:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 02:24:00–09/09 02:25:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 03:09:00–09/09 03:10:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 03:36:00–09/09 03:37:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 03:57:00–09/09 03:58:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 04:01:00–09/09 04:02:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 04:48:00–09/09 04:49:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 09/09 05:53:00–09/09 05:54:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 05:55:00–09/09 05:56:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 06:15:00–09/09 06:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 06:19:00–09/09 06:20:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 06:28:00–09/09 06:29:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 06:33:00–09/09 06:34:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 06:48:00–09/09 06:49:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 09/09 07:41:00–09/09 07:42:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 07:54:00–09/09 07:55:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 09/09 08:12:00–09/09 08:13:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 09/09 08:44:00–09/09 08:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 12개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240091, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 7건 · 짧은 노랑 후보 23건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 09/08 23:33:47**

**관측:** 원본 편차 **+0.0524무차원**, 56점·첫/끝 간격 55초. 1초 단발 여부: 아님.  
**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 7건(첫/끝 실측 간격 67~529초, 절대 편차 0.0704~0.459무차원)이 있다. 이 구간의 55초 간격·0.0524무차원는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 09/08 23:35:24. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01419 감지 2일 13시간 21분 전에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 09/09 07:07~07:12**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **7.03**, 과거 중심 대비 **+8.06℃**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 5창·8.06℃는 비교 관측의 상한을 넘는다(길이 5>4창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71.8%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.24℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 07:20 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z3 · 09/09 00:38~00:40**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **4.30**, 과거 중심 대비 **+5.69℃**.  
**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 2창·5.69℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -68%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.77℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 00:51 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 31건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240091-Z-OP-Z3-0045 · 09/08 22:48~22:49**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·11%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.811℃(8/1유효창). 같은 전후 기준의 OP-Z3 변화는 -11.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 22:55. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0047 · 09/08 22:50~22:52**

**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 17.4~17.4%p)이 있다. 이 구간의 2창·12%p는 비교 관측의 상한을 넘는다(길이 2>1창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.621℃(8/2유효창). 같은 전후 기준의 OP-Z3 변화는 +12.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 22:55. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/597유효창 (100창당 0.17건) | 12.04~12.04 | 2~2창 | 76.641 |
| 240044 | 0건/51유효창 (100창당 0.00건) | — | — | 76.119 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 72.806 |
| 240069 | 1건/737유효창 (100창당 0.14건) | 17.43~17.43 | 1~1창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0079 · 09/08 23:22~23:23**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·26.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.938℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -19.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:26. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0082 · 09/08 23:25~23:26**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 1창·2.28℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +26.6%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.51℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 23:30 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-CP-01 · 09/08 23:33:47**

**다른 LOT:** 동종 LOT 240069에도 같은 방향 후보 7건(첫/끝 실측 간격 67~529초, 절대 편차 0.0704~0.459무차원)이 있다. 이 구간의 55초 간격·0.0524무차원는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 09/08 23:35:24. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01419 감지 2일 13시간 21분 전에 CP 원본 피크.

강종 SPHC·채널 CP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. CP의 빈도 분모는 같은 형태 구간의 실제 원본 관측점 수이다. 1만 실측점당 건수로 제시하며 경과 분·초와 혼동하지 않는다. CP 사건에 Z의 54점/20창 조건을 새로 적용하지 않았다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/37633실측점 (1만점당 0.27건) | 0.05242~0.05242 | 55~55초 간격 | 0.45005 |
| 240044 | 0건/4334실측점 (1만점당 0.00건) | — | — | 0.45012 |
| 240061 | 0건/7630실측점 (1만점당 0.00건) | — | — | 0.45004 |
| 240069 | 7건/46836실측점 (1만점당 1.49건) | 0.07035~0.459 | 67~529초 간격 | 0.45013 |

CP는 ±0.02 묶음·최대 ±0.05 선택 규칙을 적용한 원본 후보만 대조했다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0120 · 09/09 00:03~00:05**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 2창·47.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.62℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -40.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:08. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0121 · 09/09 00:04~00:05**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 1창·2.7℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -38.8%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.38℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:16 (이동 Z 복귀보다 5분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0124 · 09/09 00:07~00:08**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 1창·2.67℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +24.1%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.36℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:12 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0154 · 09/09 00:37~00:40**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 3창·73.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.25℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -66.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:43. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0155 · 09/09 00:38~00:40**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 2창·5.69℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -68%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.77℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:51 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0160 · 09/09 00:43~00:44**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 1창·5.86℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +28%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -6.64℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 00:49 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0227 · 09/09 01:50~01:53**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 3창·48.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.97℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -48.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:04 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0228 · 09/09 01:51~01:53**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 2창·3.38℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -50.9%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +2.66℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:03. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0232 · 09/09 01:55~01:58**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 3창·3.65℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +22.3%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 -4.3℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 02:03. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0305 · 09/09 03:08~03:09**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·12.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.69℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -10.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:13. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0318 · 09/09 03:21~03:23**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 2창·33.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.83℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -36.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:30 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0319 · 09/09 03:22~03:23**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 1창·2.45℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -38.2%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.37℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:31 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0322 · 09/09 03:25~03:27**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 2창·2.52℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +23.4%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -3.03℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:31 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0341 · 09/09 03:44~03:45**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·44.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.74℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -44%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 03:48. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0365 · 09/09 04:08~04:09**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·43.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.77℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -42.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:12. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0400 · 09/09 04:43~04:44**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·43.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.51℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -43.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 04:47. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0465 · 09/09 05:48~05:49**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 1창·18.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -15.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:59 (이동 Z 복귀보다 7분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0490 · 09/09 06:13~06:15**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 2창·2.42℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +26.7%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.84℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:19. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0543 · 09/09 07:06~07:10**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 4창·75%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.63℃(10/4유효창). 같은 전후 기준의 OP-Z3 변화는 -73.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:19 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0544 · 09/09 07:07~07:12**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 5창·8.06℃는 비교 관측의 상한을 넘는다(길이 5>4창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71.8%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.24℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:20 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z6-0557 · 09/09 07:20~07:22**

**다른 LOT:** 동종 LOT 240061, 240069에도 같은 방향 후보 3건(연속 2~3개 1분창, 절대 편차 8.91~9.68%p)이 있다. 이 구간의 2창·6.42%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.973℃(10/2유효창). 같은 전후 기준의 OP-Z6 변화는 +5.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:25. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/597유효창 (100창당 0.17건) | 6.416~6.416 | 2~2창 | 67.844 |
| 240044 | 0건/53유효창 (100창당 0.00건) | — | — | 67.168 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 9.685~9.685 | 2~2창 | 66.614 |
| 240069 | 2건/737유효창 (100창당 0.27건) | 8.909~9.171 | 2~3창 | 68.085 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z6-0568 · 09/09 07:31~07:32**

**다른 LOT:** 같은 강종 LOT 240044, 240061, 240069의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 -7.71%p(10/1유효창). 같은 전후 기준의 TC-Z6 변화는 +1.4℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 07:41 (이동 Z 복귀보다 6분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 1건/597유효창 (100창당 0.17건) | 1.619~1.619 | 1~1창 | 859.71 |
| 240044 | 0건/52유효창 (100창당 0.00건) | — | — | 859.65 |
| 240061 | 0건/127유효창 (100창당 0.00건) | — | — | 859.73 |
| 240069 | 0건/737유효창 (100창당 0.00건) | — | — | 859.72 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-OP-Z3-0603 · 09/09 08:06~08:08**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 20건(연속 1~4개 1분창, 절대 편차 11.3~77.2%p)이 있다. 이 구간의 2창·58.9%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 20건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.69℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -53.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:12 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 13건/597유효창 (100창당 2.18건) | 10.96~74.98 | 1~4창 | 76.641 |
| 240044 | 1건/51유효창 (100창당 1.96건) | 52.6~52.6 | 2~2창 | 76.119 |
| 240061 | 4건/127유효창 (100창당 3.15건) | 42.93~72.68 | 1~2창 | 72.806 |
| 240069 | 15건/737유효창 (100창당 2.04건) | 11.34~77.23 | 1~4창 | 74.152 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0603 · 09/09 08:06~08:09**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 11건(연속 1~4개 1분창, 절대 편차 2.91~11.1℃)이 있다. 이 구간의 3창·3.6℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 11건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -45.1%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 +3.43℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:20 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 6건/597유효창 (100창당 1.01건) | 2.446~8.056 | 1~5창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 3.167~3.167 | 1~1창 | 860.57 |
| 240061 | 2건/127유효창 (100창당 1.57건) | 2.912~10.56 | 1~3창 | 859.22 |
| 240069 | 8건/737유효창 (100창당 1.09건) | 3.121~11.06 | 1~4창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0608 · 09/09 08:11~08:12**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 1창·5.49℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +23.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -5.59℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:19 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240091-Z-TC-Z3-0610 · 09/09 08:13~08:14**

**다른 LOT:** 동종 LOT 240044, 240061, 240069에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 3.2~9.5℃)이 있다. 이 구간의 1창·5.72℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +26.1%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -5.85℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:18 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPHC·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240091 (현재 LOT) | 8건/597유효창 (100창당 1.34건) | 2.283~5.856 | 1~3창 | 860.31 |
| 240044 | 1건/53유효창 (100창당 1.89건) | 9.501~9.501 | 3~3창 | 860.57 |
| 240061 | 1건/127유효창 (100창당 0.79건) | 3.199~3.199 | 2~2창 | 859.22 |
| 240069 | 6건/737유효창 (100창당 0.81건) | 3.325~5.864 | 1~3창 | 859.32 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 642 | 597 | 26 | 45 | 25 | 20 | 0 |
| OP-Z3 | 642 | 597 | 25 | 45 | 25 | 20 | 0 |
| TC-Z4 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| OP-Z4 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| TC-Z5 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| OP-Z5 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| TC-Z6 | 642 | 597 | 1 | 45 | 25 | 20 | 0 |
| OP-Z6 | 642 | 597 | 2 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 1건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 597창, 최대 절대 Z=2.30 / OP-Z4: 유효 597창, 최대 절대 Z=2.95 / TC-Z5: 유효 597창, 최대 절대 Z=2.10 / OP-Z5: 유효 597창, 최대 절대 Z=2.04. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 09/08 22:48:00–09/08 22:49:00 (끝 제외); 240091-Z-OP-Z3-0045 | 최대 절대 Z=3.38 (09/08 22:48:30, 창 중심); 1분 중앙값 66.117%, 과거 중심 77.079%, 차이 -10.962%p; SD=3.239; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/08 22:55:00 / 시작 고정 기준 3창 복귀 09/08 22:55:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3726.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/08 22:50:00–09/08 22:52:00 (끝 제외); 240091-Z-OP-Z3-0047 | 최대 절대 Z=3.12 (09/08 22:50:30, 창 중심); 1분 중앙값 89.014%, 과거 중심 76.971%, 차이 +12.043%p; SD=3.866; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/08 22:55:00 / 시작 고정 기준 3창 복귀 09/08 22:55:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3724.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/08 23:22:00–09/08 23:23:00 (끝 제외); 240091-Z-OP-Z3-0079 | 최대 절대 Z=3.90 (09/08 23:22:30, 창 중심); 1분 중앙값 50.627%, 과거 중심 77.356%, 차이 -26.729%p; SD=6.850; 실측 56/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/08 23:26:00 / 시작 고정 기준 3창 복귀 09/08 23:26:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3692.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/08 23:25:00–09/08 23:26:00 (끝 제외); 240091-Z-TC-Z3-0082 | 최대 절대 Z=4.51 (09/08 23:25:30, 창 중심); 1분 중앙값 858.303℃, 과거 중심 860.586℃, 차이 -2.283℃; SD=0.506; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/08 23:29:00 / 시작 고정 기준 3창 복귀 09/08 23:30:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3689.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 00:03:00–09/09 00:05:00 (끝 제외); 240091-Z-OP-Z3-0120 | 최대 절대 Z=5.46 (09/09 00:03:30, 창 중심); 1분 중앙값 27.015%, 과거 중심 74.162%, 차이 -47.147%p; SD=8.640; 실측 54/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 00:08:00 / 시작 고정 기준 3창 복귀 09/09 00:08:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3651.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 00:04:00–09/09 00:05:00 (끝 제외); 240091-Z-TC-Z3-0121 | 최대 절대 Z=4.01 (09/09 00:04:30, 창 중심); 1분 중앙값 863.289℃, 과거 중심 860.586℃, 차이 +2.703℃; SD=0.675; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 00:11:00 / 시작 고정 기준 3창 복귀 09/09 00:16:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3650.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 00:07:00–09/09 00:08:00 (끝 제외); 240091-Z-TC-Z3-0124 | 최대 절대 Z=3.03 (09/09 00:07:30, 창 중심); 1분 중앙값 857.995℃, 과거 중심 860.663℃, 차이 -2.668℃; SD=0.881; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 00:11:00 / 시작 고정 기준 3창 복귀 09/09 00:12:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3647.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 00:37:00–09/09 00:40:00 (끝 제외); 240091-Z-OP-Z3-0154 | 최대 절대 Z=5.13 (09/09 00:38:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 73.145%, 차이 -73.145%p; SD=14.259; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 00:43:00 / 시작 고정 기준 3창 복귀 09/09 00:43:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3616.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 00:38:00–09/09 00:40:00 (끝 제외); 240091-Z-TC-Z3-0155 | 최대 절대 Z=4.30 (09/09 00:39:30, 창 중심); 1분 중앙값 865.947℃, 과거 중심 860.261℃, 차이 +5.686℃; SD=1.323; 실측 56/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 00:43:00 / 시작 고정 기준 3창 복귀 09/09 00:51:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3615.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 00:43:00–09/09 00:44:00 (끝 제외); 240091-Z-TC-Z3-0160 | 최대 절대 Z=3.08 (09/09 00:43:30, 창 중심); 1분 중앙값 854.589℃, 과거 중심 860.446℃, 차이 -5.856℃; SD=1.902; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 00:47:00 / 시작 고정 기준 3창 복귀 09/09 00:49:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3611.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 01:50:00–09/09 01:53:00 (끝 제외); 240091-Z-OP-Z3-0227 | 최대 절대 Z=17.84 (09/09 01:51:30, 창 중심); 1분 중앙값 28.116%, 과거 중심 76.655%, 차이 -48.539%p; SD=2.721; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 01:56:00 / 시작 고정 기준 3창 복귀 09/09 02:04:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3543.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 01:51:00–09/09 01:53:00 (끝 제외); 240091-Z-TC-Z3-0228 | 최대 절대 Z=6.75 (09/09 01:52:30, 창 중심); 1분 중앙값 863.801℃, 과거 중심 860.426℃, 차이 +3.375℃; SD=0.500 (하한 적용); 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 02:03:00 / 시작 고정 기준 3창 복귀 09/09 02:03:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3542.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 01:55:00–09/09 01:58:00 (끝 제외); 240091-Z-TC-Z3-0232 | 최대 절대 Z=4.50 (09/09 01:55:30, 창 중심); 1분 중앙값 856.794℃, 과거 중심 860.442℃, 차이 -3.648℃; SD=0.810; 실측 58/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 02:03:00 / 시작 고정 기준 3창 복귀 09/09 02:03:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3539.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 03:08:00–09/09 03:09:00 (끝 제외); 240091-Z-OP-Z3-0305 | 최대 절대 Z=3.13 (09/09 03:08:30, 창 중심); 1분 중앙값 64.879%, 과거 중심 76.946%, 차이 -12.067%p; SD=3.861; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 03:13:00 / 시작 고정 기준 3창 복귀 09/09 03:13:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3466.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 03:21:00–09/09 03:23:00 (끝 제외); 240091-Z-OP-Z3-0318 | 최대 절대 Z=6.60 (09/09 03:21:30, 창 중심); 1분 중앙값 43.262%, 과거 중심 77.203%, 차이 -33.941%p; SD=5.146; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 03:26:00 / 시작 고정 기준 3창 복귀 09/09 03:30:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3453.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 03:22:00–09/09 03:23:00 (끝 제외); 240091-Z-TC-Z3-0319 | 최대 절대 Z=4.89 (09/09 03:22:30, 창 중심); 1분 중앙값 862.878℃, 과거 중심 860.432℃, 차이 +2.446℃; SD=0.500 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 03:30:00 / 시작 고정 기준 3창 복귀 09/09 03:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3452.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 03:25:00–09/09 03:27:00 (끝 제외); 240091-Z-TC-Z3-0322 | 최대 절대 Z=3.81 (09/09 03:25:30, 창 중심); 1분 중앙값 857.971℃, 과거 중심 860.495℃, 차이 -2.524℃; SD=0.662; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 03:30:00 / 시작 고정 기준 3창 복귀 09/09 03:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3449.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 03:44:00–09/09 03:45:00 (끝 제외); 240091-Z-OP-Z3-0341 | 최대 절대 Z=3.22 (09/09 03:44:30, 창 중심); 1분 중앙값 30.436%, 과거 중심 75.347%, 차이 -44.911%p; SD=13.963; 실측 56/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 03:48:00 / 시작 고정 기준 3창 복귀 09/09 03:48:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3430.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 04:08:00–09/09 04:09:00 (끝 제외); 240091-Z-OP-Z3-0365 | 최대 절대 Z=3.08 (09/09 04:08:30, 창 중심); 1분 중앙값 31.093%, 과거 중심 74.702%, 차이 -43.609%p; SD=14.148; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 04:12:00 / 시작 고정 기준 3창 복귀 09/09 04:12:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3406.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 04:43:00–09/09 04:44:00 (끝 제외); 240091-Z-OP-Z3-0400 | 최대 절대 Z=3.34 (09/09 04:43:30, 창 중심); 1분 중앙값 33.193%, 과거 중심 77.123%, 차이 -43.930%p; SD=13.142; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 04:47:00 / 시작 고정 기준 3창 복귀 09/09 04:47:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3371.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 05:48:00–09/09 05:49:00 (끝 제외); 240091-Z-OP-Z3-0465 | 최대 절대 Z=3.77 (09/09 05:48:30, 창 중심); 1분 중앙값 59.617%, 과거 중심 78.164%, 차이 -18.547%p; SD=4.918; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 05:52:00 / 시작 고정 기준 3창 복귀 09/09 05:59:00 | 고정 범위 복귀가 이동 Z 복귀보다 7분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3306.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 06:13:00–09/09 06:15:00 (끝 제외); 240091-Z-TC-Z3-0490 | 최대 절대 Z=3.23 (09/09 06:13:30, 창 중심); 1분 중앙값 858.258℃, 과거 중심 860.682℃, 차이 -2.424℃; SD=0.750; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 06:19:00 / 시작 고정 기준 3창 복귀 09/09 06:19:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3281.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 07:06:00–09/09 07:10:00 (끝 제외); 240091-Z-OP-Z3-0543 | 최대 절대 Z=6.23 (09/09 07:07:30, 창 중심); 1분 중앙값 0.000%, 과거 중심 74.981%, 차이 -74.981%p; SD=12.028; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 07:13:00 / 시작 고정 기준 3창 복귀 09/09 07:19:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3227.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 07:07:00–09/09 07:12:00 (끝 제외); 240091-Z-TC-Z3-0544 | 최대 절대 Z=7.03 (09/09 07:08:30, 창 중심); 1분 중앙값 868.110℃, 과거 중심 860.054℃, 차이 +8.056℃; SD=1.147; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 07:15:00 / 시작 고정 기준 3창 복귀 09/09 07:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3226.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 09/09 07:20:00–09/09 07:22:00 (끝 제외); 240091-Z-OP-Z6-0557 | 최대 절대 Z=3.21 (09/09 07:20:30, 창 중심); 1분 중앙값 75.793%, 과거 중심 69.377%, 차이 +6.416%p; SD=2.000 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 07:25:00 / 시작 고정 기준 3창 복귀 09/09 07:25:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3214.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 09/09 07:31:00–09/09 07:32:00 (끝 제외); 240091-Z-TC-Z6-0568 | 최대 절대 Z=3.21 (09/09 07:31:30, 창 중심); 1분 중앙값 861.162℃, 과거 중심 859.543℃, 차이 +1.619℃; SD=0.505; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 07:35:00 / 시작 고정 기준 3창 복귀 09/09 07:41:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3203.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 09/09 08:06:00–09/09 08:08:00 (끝 제외); 240091-Z-OP-Z3-0603 | 최대 절대 Z=4.51 (09/09 08:07:30, 창 중심); 1분 중앙값 15.665%, 과거 중심 74.571%, 차이 -58.906%p; SD=13.047; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:11:00 / 시작 고정 기준 3창 복귀 09/09 08:12:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3167.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 08:06:00–09/09 08:09:00 (끝 제외); 240091-Z-TC-Z3-0603 | 최대 절대 Z=4.65 (09/09 08:07:30, 창 중심); 1분 중앙값 864.099℃, 과거 중심 860.504℃, 차이 +3.596℃; SD=0.773; 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:17:00 / 시작 고정 기준 3창 복귀 09/09 08:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3167.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 08:11:00–09/09 08:12:00 (끝 제외); 240091-Z-TC-Z3-0608 | 최대 절대 Z=3.94 (09/09 08:11:30, 창 중심); 1분 중앙값 855.128℃, 과거 중심 860.616℃, 차이 -5.489℃; SD=1.393; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:17:00 / 시작 고정 기준 3창 복귀 09/09 08:19:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3163.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 09/09 08:13:00–09/09 08:14:00 (끝 제외); 240091-Z-TC-Z3-0610 | 최대 절대 Z=3.33 (09/09 08:13:30, 창 중심); 1분 중앙값 854.897℃, 과거 중심 860.616℃, 차이 -5.719℃; SD=1.716; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 09/09 08:17:00 / 시작 고정 기준 3창 복귀 09/09 08:18:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01419 감지 대비 Z 최대창 중심 -3161.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240091-Z-TC-Z3-0544:** OP-Z3 사건창 중앙값−직전10분 중앙값 -71.82 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240091-Z-OP-Z3-0543:** TC-Z3 사건창 중앙값−직전10분 중앙값 +5.628 (10/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 09/08 22:03:00–09/08 22:23:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 과거 유효 최소 0창 <20 |
| 09/08 22:46:00–09/08 22:48:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 09/09 00:12:00–09/09 00:13:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 01:13:00–09/09 01:14:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 01:16:00–09/09 01:17:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 01:18:00–09/09 01:19:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 01:59:00–09/09 02:00:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 09/09 02:15:00–09/09 02:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 02:24:00–09/09 02:25:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 03:09:00–09/09 03:10:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 03:36:00–09/09 03:37:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 03:57:00–09/09 03:58:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 04:01:00–09/09 04:02:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 04:48:00–09/09 04:49:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 09/09 05:53:00–09/09 05:54:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 05:55:00–09/09 05:56:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 06:15:00–09/09 06:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 06:19:00–09/09 06:20:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 06:28:00–09/09 06:29:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 06:33:00–09/09 06:34:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 06:48:00–09/09 06:49:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 09/09 07:41:00–09/09 07:42:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 07:54:00–09/09 07:55:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 09/09 08:12:00–09/09 08:13:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 09/09 08:44:00–09/09 08:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 12개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240091-CP-01 · 09/08 23:33:47 | +0.0524무차원 · 56점/55초 간격 | 동종 LOT 240069에도 같은 방향 후보 7건(첫/끝 실측 간격 67~529초, 절대 편차 0.0704~0.459무차원)이 있다. 이 구간의 55초 간격·0.0524무차원는 비교한 변화 폭·길이의 상한 이내다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다. | 60개 연속 실제 1초 값 복귀 23:35:24; WO-2024-01419 감지 2일 13시간 21분 전에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240091-CP-01 · 피크 09/08 23:33:47 · 복귀 펄스 후보**

- **관측:** 원값 0.502486, LOT 중앙값 0.450068, 편차 +0.052418. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 56점, 첫·끝 실측 간격 55초, 기대 1초 격자 내 유효 비율 100.0%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 09/08 23:33:28=0.467155 / 직후 09/08 23:34:25=0.468748; 이후 60개 연속 실측 ±0.02 복귀 확인 09/08 23:35:24.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 197154–197211에 연결했다(0부터 시작).

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240091-CP-01 | 정비 대조: WO-2024-01419 감지 대비 CP 원본 피크 -3681.22분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 +0.079 (295/61점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +3.891 (295/61점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 -6.09e-13 (295/61점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240091, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| OP-Z7 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| TC-Z8 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |
| OP-Z8 | 642 | 597 | 0 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 597창, 최대 절대 Z=0.56 / OP-Z7: 유효 597창, 최대 절대 Z=2.56 / TC-Z8: 유효 597창, 최대 절대 Z=0.76 / OP-Z8: 유효 597창, 최대 절대 Z=1.74. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 09/08 22:03:00–09/08 22:23:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 09/08 22:46:00–09/08 22:48:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 09/09 00:12:00–09/09 00:13:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 01:13:00–09/09 01:14:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 01:16:00–09/09 01:17:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 01:18:00–09/09 01:19:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 01:59:00–09/09 02:00:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 09/09 02:15:00–09/09 02:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 02:24:00–09/09 02:25:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 03:09:00–09/09 03:10:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 03:36:00–09/09 03:37:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 03:57:00–09/09 03:58:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 04:01:00–09/09 04:02:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 04:48:00–09/09 04:49:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 09/09 05:53:00–09/09 05:54:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 05:55:00–09/09 05:56:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 06:15:00–09/09 06:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 06:19:00–09/09 06:20:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 06:28:00–09/09 06:29:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 06:33:00–09/09 06:34:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 06:48:00–09/09 06:49:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 09/09 07:41:00–09/09 07:42:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 07:54:00–09/09 07:55:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 09/09 08:12:00–09/09 08:13:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 09/09 08:44:00–09/09 08:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 12개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240091)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **1건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 37633 | 25 | 원본 ±0.05 사건 1건 · 최대 절대 편차 0.052418 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 37633 | 25 | 원본 중앙값 1.1395e-10, 최저 1.12244e-10, 최고 1.15067e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 37632 | 25 | 원본 중앙값 329.304, 최저 319.021, 최고 342.116; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 37632 | 25 | 원본 중앙값 285.081, 최저 270.736, 최고 298.21; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 37632 | 25 | 원본 중앙값 281.683, 최저 270.755, 최고 292.218; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-01419 · BM / 메모: 절연 저하**

**실제 시각:** 감지 09/11 12:55:00 / 작업 09/11 17:01:00~09/11 20:05:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-01419 감지 2일 13시간 21분 전에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 2건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 2건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01419 · BM | 09/11 12:55:00 | 09/11 17:01:00 | 09/11 20:05:00 | 절연 저하 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-01421 · TBM | 09/11 23:31:00 | 09/12 01:37:00 | 09/12 06:12:00 | 정기점검 - 이상없음 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-01419 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 33건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240091-CP-01 · 09/08 23:33:47 · CP 편차 +0.052418 · 56점/55초 · 완료 대비 피크 -4111.22분 · 감지는 피크 대비 +3681.22분. 복귀 펄스 후보; 직전 09/08 23:33:28=0.467155 / 직후 09/08 23:34:25=0.468748; 이후 60개 연속 실측 ±0.02 복귀 확인 09/08 23:35:24.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.21 / OP-Z1 1건·최대 절대 Z 3.86 / OP-Z2 1건·최대 절대 Z 4.37 / OP-Z3 14건·최대 절대 Z 17.84 / OP-Z6 1건·최대 절대 Z 3.21 / TC-Z3 14건·최대 절대 Z 7.03 / TC-Z6 1건·최대 절대 Z 3.21. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 09/11 12:55:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 09/11 17:01:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 09/11 20:05:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-01421 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 29건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240091-CP-01 · 09/08 23:33:47 · CP 편차 +0.052418 · 56점/55초 · 완료 대비 피크 -4718.22분 · 감지는 피크 대비 +4317.22분. 복귀 펄스 후보; 직전 09/08 23:33:28=0.467155 / 직후 09/08 23:34:25=0.468748; 이후 60개 연속 실측 ±0.02 복귀 확인 09/08 23:35:24.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.21 / OP-Z1 1건·최대 절대 Z 3.86 / OP-Z2 1건·최대 절대 Z 4.37 / OP-Z3 11건·최대 절대 Z 17.84 / OP-Z6 1건·최대 절대 Z 3.21 / TC-Z3 13건·최대 절대 Z 7.03 / TC-Z6 1건·최대 절대 Z 3.21. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 09/11 23:31:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 09/12 01:37:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 09/12 06:12:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **시간 순서가 허용하는 가설:** CP 변화가 감지보다 앞섰다. 공정·제어 변동 또는 계측 변동이 선행했을 가능성은 검토할 수 있으나, 관측 공백과 센서–부품 연결 미확인 때문에 고장 전조로 확정하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240091)

**240091 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240091)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240091)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240091(SPHC)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 332.964 - 대조 329.807 = **+3.157℃**, 다른 3 LOT과 대조.
- TC-Z8: 이 LOT 333.295 - 대조 330.183 = **+3.112℃**, 다른 3 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- OP-Z7: 차이 -1.554%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 -0.764%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240044: 전환 제외 없음.
- 240061: 06/15 01:04–02:40.
- 240069: 전환 제외 없음.
- 240091: 전환 제외 없음.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240091)
lot_conclusion(240091)
lot_execution.append({'LOT':240091,'확대 구간':len(LOT_EVENTS[240091]),'강종 대조':240091 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240108"></a>
### 8.10 LOT 240108 · SPFH590 · 10/18

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240108)

#### LOT 240108 · 먼저 볼 변화

**지속·반복 후보 13건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — CLN-IN · 10/18 07:09~07:16**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.00**, 과거 중심 대비 **-2℃**.  
**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:06 (이동 Z 복귀보다 47분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z3 · 10/18 05:45~05:49**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.44**, 과거 중심 대비 **-73.8%p**.  
**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 4창·73.8%p는 비교 관측의 상한을 넘는다(길이 4>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.49℃(9/4유효창). 같은 전후 기준의 OP-Z3 변화는 -75.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 05:56 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 10/18 11:28~11:29**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.08**, 과거 중심 대비 **-1.57℃**.  
**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240108

- **관측 범위:** 10/18 04:05:00–10/18 11:50:53.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 361개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 13개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 46개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240108, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 7건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — CLN-IN · 10/18 07:09~07:16**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.00**, 과거 중심 대비 **-2℃**.  
**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 08:06 (이동 Z 복귀보다 47분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 10/18 11:35~11:38**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 2회. 최대 |Z| **4.29**, 과거 중심 대비 **-11%p**.  
**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 9.16~9.16%p)이 있다. 이 구간의 3창·11%p는 비교 관측의 상한을 넘는다(절대 편차 11>9.16%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.18℃(9/3유효창). 같은 전후 기준의 OP-Z2 변화는 -9.79%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 11:43. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 10/18 11:28~11:29**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.08**, 과거 중심 대비 **-1.57℃**.  
**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 11건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240108-Z-OP-Z1-0179 · 10/18 07:04~07:06**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 10.4~10.4%p)이 있다. 이 구간의 2창·7.57%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.794℃(10/2유효창). 같은 전후 기준의 OP-Z1 변화는 -7.13%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 07:17 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 7.573~14.02 | 1~3창 | 71.137 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 10.41~10.41 | 5~5창 | 72.476 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z1-0183 · 10/18 07:08~07:10**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 10.4~10.4%p)이 있다. 이 구간의 2창·11.4%p는 비교 관측의 상한을 넘는다(절대 편차 11.4>10.4%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.546℃(8/2유효창). 같은 전후 기준의 OP-Z1 변화는 -7.32%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 07:15 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 7.573~14.02 | 1~3창 | 71.137 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 10.41~10.41 | 5~5창 | 72.476 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z2-0183 · 10/18 07:08~07:11**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 9.16~9.16%p)이 있다. 이 구간의 3창·9.42%p는 비교 관측의 상한을 넘는다(절대 편차 9.42>9.16%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.797℃(8/3유효창). 같은 전후 기준의 OP-Z2 변화는 -7.94%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 07:24 (이동 Z 복귀보다 10분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 3건/421유효창 (100창당 0.71건) | 8.589~11.01 | 2~3창 | 21.025 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 9.162~9.162 | 5~5창 | 22.248 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-CLN-IN-0184 · 10/18 07:09~07:16**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 08:06 (이동 Z 복귀보다 47분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 3건/421유효창 (100창당 0.71건) | 1.565~2.002 | 1~7창 | 69.564 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 69.663 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-CLN-IN-0443 · 10/18 11:28~11:29**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 3건/421유효창 (100창당 0.71건) | 1.565~2.002 | 1~7창 | 69.564 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 69.663 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-CLN-IN-0445 · 10/18 11:30~11:31**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 3건/421유효창 (100창당 0.71건) | 1.565~2.002 | 1~7창 | 69.564 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 69.663 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z1-0446 · 10/18 11:31~11:34**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 10.4~10.4%p)이 있다. 이 구간의 3창·12.2%p는 비교 관측의 상한을 넘는다(절대 편차 12.2>10.4%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.7℃(10/3유효창). 같은 전후 기준의 OP-Z1 변화는 -12.8%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 11:43 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 7.573~14.02 | 1~3창 | 71.137 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 10.41~10.41 | 5~5창 | 72.476 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z2-0447 · 10/18 11:32~11:34**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 9.16~9.16%p)이 있다. 이 구간의 2창·8.59%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.95℃(10/2유효창). 같은 전후 기준의 OP-Z2 변화는 -7.43%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 11:44 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 3건/421유효창 (100창당 0.71건) | 8.589~11.01 | 2~3창 | 21.025 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 9.162~9.162 | 5~5창 | 22.248 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z1-0447 · 10/18 11:32~11:33**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 1.64~1.64℃)이 있다. 이 구간의 1창·1.6℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -12.8%p(10/1유효창). 같은 전후 기준의 TC-Z1 변화는 +1.7℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 11:38. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 1건/421유효창 (100창당 0.24건) | 1.601~1.601 | 1~1창 | 99.999 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 1.638~1.638 | 2~2창 | 100.02 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z1-0450 · 10/18 11:35~11:36**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 10.4~10.4%p)이 있다. 이 구간의 1창·14%p는 비교 관측의 상한을 넘는다(절대 편차 14>10.4%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.17℃(9/1유효창). 같은 전후 기준의 OP-Z1 변화는 -9.76%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 11:39. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 7.573~14.02 | 1~3창 | 71.137 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 10.41~10.41 | 5~5창 | 72.476 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z2-0450 · 10/18 11:35~11:38**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 9.16~9.16%p)이 있다. 이 구간의 3창·11%p는 비교 관측의 상한을 넘는다(절대 편차 11>9.16%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.18℃(9/3유효창). 같은 전후 기준의 OP-Z2 변화는 -9.79%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 11:43. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 3건/421유효창 (100창당 0.71건) | 8.589~11.01 | 2~3창 | 21.025 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 9.162~9.162 | 5~5창 | 22.248 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 466 | 421 | 9 | 45 | 25 | 20 | 0 |
| TC-Z1 | 466 | 421 | 1 | 45 | 25 | 20 | 0 |
| OP-Z1 | 466 | 421 | 8 | 45 | 25 | 20 | 0 |
| TC-Z2 | 466 | 420 | 0 | 46 | 26 | 20 | 0 |
| OP-Z2 | 466 | 421 | 8 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z2: 유효 420창, 최대 절대 Z=2.72. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 10/18 07:04:00–10/18 07:06:00 (끝 제외); 240108-Z-OP-Z1-0179 | 최대 절대 Z=3.79 (10/18 07:04:30, 창 중심); 1분 중앙값 64.320%, 과거 중심 71.893%, 차이 -7.573%p; SD=2.000 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 07:13:00 / 시작 고정 기준 3창 복귀 10/18 07:17:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -525.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 10/18 07:08:00–10/18 07:10:00 (끝 제외); 240108-Z-OP-Z1-0183 | 최대 절대 Z=3.99 (10/18 07:08:30, 창 중심); 1분 중앙값 60.523%, 과거 중심 71.893%, 차이 -11.370%p; SD=2.852; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 07:13:00 / 시작 고정 기준 3창 복귀 10/18 07:15:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -521.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 10/18 07:08:00–10/18 07:11:00 (끝 제외); 240108-Z-OP-Z2-0183 | 최대 절대 Z=4.36 (10/18 07:09:30, 창 중심); 1분 중앙값 12.534%, 과거 중심 21.954%, 차이 -9.420%p; SD=2.160; 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 07:14:00 / 시작 고정 기준 3창 복귀 10/18 07:24:00 | 고정 범위 복귀가 이동 Z 복귀보다 10분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -520.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 10/18 07:09:00–10/18 07:16:00 (끝 제외); 240108-Z-CLN-IN-0184 | 최대 절대 Z=4.00 (10/18 07:10:30, 창 중심); 1분 중앙값 68.876℃, 과거 중심 70.878℃, 차이 -2.002℃; SD=0.500 (하한 적용); 실측 58/60개·과거 27/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 07:19:00 / 시작 고정 기준 3창 복귀 10/18 08:06:00 | 고정 범위 복귀가 이동 Z 복귀보다 47분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -519.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 10/18 11:28:00–10/18 11:29:00 (끝 제외); 240108-Z-CLN-IN-0443 | 최대 절대 Z=3.08 (10/18 11:28:30, 창 중심); 1분 중앙값 69.291℃, 과거 중심 70.856℃, 차이 -1.565℃; SD=0.508; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:34:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -261.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 10/18 11:30:00–10/18 11:31:00 (끝 제외); 240108-Z-CLN-IN-0445 | 최대 절대 Z=3.04 (10/18 11:30:30, 창 중심); 1분 중앙값 68.922℃, 과거 중심 70.813℃, 차이 -1.891℃; SD=0.621; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:34:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -259.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 10/18 11:31:00–10/18 11:34:00 (끝 제외); 240108-Z-OP-Z1-0446 | 최대 절대 Z=3.75 (10/18 11:32:30, 창 중심); 1분 중앙값 58.342%, 과거 중심 70.590%, 차이 -12.247%p; SD=3.267; 실측 59/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:39:00 / 시작 고정 기준 3창 복귀 10/18 11:43:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -257.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 10/18 11:32:00–10/18 11:34:00 (끝 제외); 240108-Z-OP-Z2-0447 | 최대 절대 Z=4.08 (10/18 11:33:30, 창 중심); 1분 중앙값 13.282%, 과거 중심 21.871%, 차이 -8.589%p; SD=2.103; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:43:00 / 시작 고정 기준 3창 복귀 10/18 11:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -256.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 10/18 11:32:00–10/18 11:33:00 (끝 제외); 240108-Z-TC-Z1-0447 | 최대 절대 Z=3.01 (10/18 11:32:30, 창 중심); 1분 중앙값 101.749℃, 과거 중심 100.148℃, 차이 +1.601℃; SD=0.532; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:38:00 / 시작 고정 기준 3창 복귀 10/18 11:38:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -257.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 10/18 11:35:00–10/18 11:36:00 (끝 제외); 240108-Z-OP-Z1-0450 | 최대 절대 Z=3.04 (10/18 11:35:30, 창 중심); 1분 중앙값 56.745%, 과거 중심 70.768%, 차이 -14.023%p; SD=4.611; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:39:00 / 시작 고정 기준 3창 복귀 10/18 11:39:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -254.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 10/18 11:35:00–10/18 11:38:00 (끝 제외); 240108-Z-OP-Z2-0450 | 최대 절대 Z=4.29 (10/18 11:35:30, 창 중심); 1분 중앙값 10.937%, 과거 중심 21.951%, 차이 -11.014%p; SD=2.570; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:43:00 / 시작 고정 기준 3창 복귀 10/18 11:43:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -254.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240108-Z-CLN-IN-0184:** 대응 OP 채널 정의 없음; 다른 신호의 동시 반응은 전체 공정 그림에서 확인. 동시 관측의 근거이며 원인 확정은 아니다.
- **240108-Z-OP-Z2-0450:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.176 (9/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 10/18 04:05:00–10/18 04:27:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 / 과거 유효 최소 0창 <20 |
| 10/18 04:34:00–10/18 04:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 04:48:00–10/18 04:49:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 05:41:00–10/18 05:42:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 10/18 06:26:00–10/18 06:27:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 06:31:00–10/18 06:32:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 06:44:00–10/18 06:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 07:06:00–10/18 07:08:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 07:26:00–10/18 07:27:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 10/18 07:43:00–10/18 07:44:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 10/18 07:48:00–10/18 07:49:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 07:57:00–10/18 07:58:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 10/18 08:02:00–10/18 08:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 10/18 08:38:00–10/18 08:40:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 08:48:00–10/18 08:49:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 10:02:00–10/18 10:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 10:07:00–10/18 10:08:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 10:12:00–10/18 10:13:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 10:27:00–10/18 10:28:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 10:31:00–10/18 10:32:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 10/18 10:56:00–10/18 10:57:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 11:34:00–10/18 11:35:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 10/18 11:39:00–10/18 11:40:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240108, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 9건 · 짧은 노랑 후보 20건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 주황 · 지속·반복 확인 — OP-Z3 · 10/18 05:45~05:49**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.44**, 과거 중심 대비 **-73.8%p**.  
**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 4창·73.8%p는 비교 관측의 상한을 넘는다(길이 4>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.49℃(9/4유효창). 같은 전후 기준의 OP-Z3 변화는 -75.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 05:56 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z4 · 10/18 05:51~05:55**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.90**, 과거 중심 대비 **-7.81%p**.  
**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.42℃(9/4유효창). 같은 전후 기준의 OP-Z4 변화는 -7.52%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 06:08 (이동 Z 복귀보다 10분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z3 · 10/18 09:44~09:46**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 3회. 최대 |Z| **4.69**, 과거 중심 대비 **+5.43℃**.  
**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 2창·5.43℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -67.4%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.17℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 09:57 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 29건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240108-Z-OP-Z3-0059 · 10/18 05:04~05:05**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 1창·15.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.761℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -15.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:08. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0070 · 10/18 05:15~05:16**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 1창·26.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.44℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -27.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:19. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0073 · 10/18 05:18~05:20**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 1.53~3.38℃)이 있다. 이 구간의 2창·2.34℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +14.6%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.69℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:24 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |
| 240024 | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0100 · 10/18 05:45~05:49**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 4창·73.8%p는 비교 관측의 상한을 넘는다(길이 4>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +6.49℃(9/4유효창). 같은 전후 기준의 OP-Z3 변화는 -75.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:56 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0101 · 10/18 05:46~05:50**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 4창·8.22℃는 비교 관측의 상한을 넘는다(절대 편차 8.22>6.21℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -74.8%p(9/4유효창). 같은 전후 기준의 TC-Z3 변화는 +8.39℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:57 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |
| 240024 | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z4-0106 · 10/18 05:51~05:55**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.42℃(9/4유효창). 같은 전후 기준의 OP-Z4 변화는 -7.52%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:08 (이동 Z 복귀보다 10분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 2건/421유효창 (100창당 0.48건) | 6.544~7.809 | 1~4창 | 55.334 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 54.026 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z4-0107 · 10/18 05:52~05:54**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -7.32%p(10/2유효창). 같은 전후 기준의 TC-Z4 변화는 +1.41℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 05:57. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 1건/421유효창 (100창당 0.24건) | 1.559~1.559 | 2~2창 | 860.36 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 860.41 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z6-0114 · 10/18 05:59~06:00**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 7.6~7.6%p)이 있다. 이 구간의 1창·6.75%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.784℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +5.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:03. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 6.755~8.832 | 1~2창 | 69.353 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 7.596~7.596 | 3~3창 | 73.831 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z6-0143 · 10/18 06:28~06:29**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 7.6~7.6%p)이 있다. 이 구간의 1창·7.15%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -2.13℃(9/1유효창). 같은 전후 기준의 OP-Z6 변화는 +9.68%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:35. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 6.755~8.832 | 1~2창 | 69.353 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 7.596~7.596 | 3~3창 | 73.831 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z6-0143 · 10/18 06:28~06:29**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 +9.68%p(9/1유효창). 같은 전후 기준의 TC-Z6 변화는 -2.13℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 06:35. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 1건/421유효창 (100창당 0.24건) | 1.671~1.671 | 1~1창 | 860.47 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 860.52 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0233 · 10/18 07:58~08:00**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 1.53~3.38℃)이 있다. 이 구간의 2창·2.42℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +18.5%p(8/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.36℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:06. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |
| 240024 | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0238 · 10/18 08:03~08:04**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 1창·26.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.93℃(8/1유효창). 같은 전후 기준의 OP-Z3 변화는 -22.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:07. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0259 · 10/18 08:24~08:25**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 1창·30%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.4℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -31.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:28. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0266 · 10/18 08:31~08:33**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 2창·70.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.61℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -53.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:37 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0267 · 10/18 08:32~08:34**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 2창·3.75℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -51.2%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +3.75℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:44 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |
| 240024 | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0271 · 10/18 08:36~08:38**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 1.53~3.38℃)이 있다. 이 구간의 2창·5.13℃는 비교 관측의 상한을 넘는다(절대 편차 5.13>3.38℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +10.5%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -5.63℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:43. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |
| 240024 | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0312 · 10/18 09:17~09:18**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 1창·29.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.37℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -29.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:22 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0313 · 10/18 09:18~09:19**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 1창·1.85℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -23.6%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +1.77℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:25. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |
| 240024 | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0316 · 10/18 09:21~09:22**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 1.53~3.38℃)이 있다. 이 구간의 1창·2.11℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +13.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.35℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:25. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |
| 240024 | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0339 · 10/18 09:44~09:46**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 2창·71.1%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.17℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -67.4%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:49. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0339 · 10/18 09:44~09:46**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 2창·5.43℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -67.4%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.17℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:57 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |
| 240024 | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0344 · 10/18 09:49~09:50**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 5건(연속 1~2개 1분창, 절대 편차 1.53~3.38℃)이 있다. 이 구간의 1창·5.55℃는 비교 관측의 상한을 넘는다(절대 편차 5.55>3.38℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 5건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +24.2%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -6.3℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:53. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 5건/421유효창 (100창당 1.19건) | 2.111~5.552 | 1~2창 | 861.04 |
| 240024 | 5건/500유효창 (100창당 1.00건) | 1.532~3.385 | 1~2창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0368 · 10/18 10:13~10:14**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 1창·76.7%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.59℃(8/1유효창). 같은 전후 기준의 OP-Z3 변화는 -74.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:22 (이동 Z 복귀보다 5분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0369 · 10/18 10:14~10:16**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 2창·9.08℃는 비교 관측의 상한을 넘는다(절대 편차 9.08>6.21℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -73.7%p(8/2유효창). 같은 전후 기준의 TC-Z3 변화는 +8.28℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:21 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |
| 240024 | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z6-0379 · 10/18 10:24~10:26**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 7.6~7.6%p)이 있다. 이 구간의 2창·8.83%p는 비교 관측의 상한을 넘는다(절대 편차 8.83>7.6%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.25℃(10/2유효창). 같은 전후 기준의 OP-Z6 변화는 +7.97%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:31. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 6.755~8.832 | 1~2창 | 69.353 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 7.596~7.596 | 3~3창 | 73.831 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z4-0389 · 10/18 10:34~10:35**

**다른 LOT:** 같은 강종 LOT 240024의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.24℃(8/1유효창). 같은 전후 기준의 OP-Z4 변화는 -6.58%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:38. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 2건/421유효창 (100창당 0.48건) | 6.544~7.809 | 1~4창 | 55.334 |
| 240024 | 0건/500유효창 (100창당 0.00건) | — | — | 54.026 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z3-0415 · 10/18 11:00~11:03**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 23.5~78.8%p)이 있다. 이 구간의 3창·82.1%p는 비교 관측의 상한을 넘는다(절대 편차 82.1>78.8%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +8.61℃(9/3유효창). 같은 전후 기준의 OP-Z3 변화는 -87%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:10 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 10건/421유효창 (100창당 2.38건) | 15.69~82.05 | 1~4창 | 85.451 |
| 240024 | 8건/500유효창 (100창당 1.60건) | 23.46~78.76 | 1~3창 | 86.906 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-TC-Z3-0416 · 10/18 11:01~11:04**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 1.93~6.21℃)이 있다. 이 구간의 3창·9.85℃는 비교 관측의 상한을 넘는다(절대 편차 9.85>6.21℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -87%p(9/3유효창). 같은 전후 기준의 TC-Z3 변화는 +10.6℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:10 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 6건/421유효창 (100창당 1.43건) | 1.847~9.852 | 1~4창 | 861.04 |
| 240024 | 3건/500유효창 (100창당 0.60건) | 1.931~6.213 | 1~4창 | 860.62 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240108-Z-OP-Z6-0428 · 10/18 11:13~11:14**

**다른 LOT:** 동종 LOT 240024에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 7.6~7.6%p)이 있다. 이 구간의 1창·7.09%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.01℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +6.06%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:18 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPFH590·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240108 (현재 LOT) | 4건/421유효창 (100창당 0.95건) | 6.755~8.832 | 1~2창 | 69.353 |
| 240024 | 1건/500유효창 (100창당 0.20건) | 7.596~7.596 | 3~3창 | 73.831 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 466 | 421 | 22 | 45 | 25 | 20 | 0 |
| OP-Z3 | 466 | 421 | 17 | 45 | 25 | 20 | 0 |
| TC-Z4 | 466 | 421 | 2 | 45 | 25 | 20 | 0 |
| OP-Z4 | 466 | 421 | 5 | 45 | 25 | 20 | 0 |
| TC-Z5 | 466 | 421 | 0 | 45 | 25 | 20 | 0 |
| OP-Z5 | 466 | 421 | 0 | 45 | 25 | 20 | 0 |
| TC-Z6 | 466 | 421 | 1 | 45 | 25 | 20 | 0 |
| OP-Z6 | 466 | 421 | 5 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z5: 유효 421창, 최대 절대 Z=2.42 / OP-Z5: 유효 421창, 최대 절대 Z=2.71. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 10/18 05:04:00–10/18 05:05:00 (끝 제외); 240108-Z-OP-Z3-0059 | 최대 절대 Z=3.06 (10/18 05:04:30, 창 중심); 1분 중앙값 69.260%, 과거 중심 84.951%, 차이 -15.691%p; SD=5.129; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:08:00 / 시작 고정 기준 3창 복귀 10/18 05:08:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -645.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 05:15:00–10/18 05:16:00 (끝 제외); 240108-Z-OP-Z3-0070 | 최대 절대 Z=3.67 (10/18 05:15:30, 창 중심); 1분 중앙값 58.432%, 과거 중심 84.576%, 차이 -26.144%p; SD=7.118; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:19:00 / 시작 고정 기준 3창 복귀 10/18 05:19:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -634.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 05:18:00–10/18 05:20:00 (끝 제외); 240108-Z-TC-Z3-0073 | 최대 절대 Z=3.86 (10/18 05:18:30, 창 중심); 1분 중앙값 859.074℃, 과거 중심 861.416℃, 차이 -2.342℃; SD=0.606; 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:23:00 / 시작 고정 기준 3창 복귀 10/18 05:24:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -631.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 05:45:00–10/18 05:49:00 (끝 제외); 240108-Z-OP-Z3-0100 | 최대 절대 Z=6.44 (10/18 05:46:30, 창 중심); 1분 중앙값 14.712%, 과거 중심 88.481%, 차이 -73.769%p; SD=11.462; 실측 56/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:52:00 / 시작 고정 기준 3창 복귀 10/18 05:56:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -603.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 05:46:00–10/18 05:50:00 (끝 제외); 240108-Z-TC-Z3-0101 | 최대 절대 Z=6.43 (10/18 05:47:30, 창 중심); 1분 중앙값 869.258℃, 과거 중심 861.038℃, 차이 +8.220℃; SD=1.278; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:53:00 / 시작 고정 기준 3창 복귀 10/18 05:57:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -602.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 10/18 05:51:00–10/18 05:55:00 (끝 제외); 240108-Z-OP-Z4-0106 | 최대 절대 Z=3.90 (10/18 05:52:30, 창 중심); 1분 중앙값 48.743%, 과거 중심 56.552%, 차이 -7.809%p; SD=2.000 (하한 적용); 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:58:00 / 시작 고정 기준 3창 복귀 10/18 06:08:00 | 고정 범위 복귀가 이동 Z 복귀보다 10분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -597.50분 (근접 시각이며 인과 미확정) |
| TC-Z4 · 10/18 05:52:00–10/18 05:54:00 (끝 제외); 240108-Z-TC-Z4-0107 | 최대 절대 Z=3.12 (10/18 05:52:30, 창 중심); 1분 중앙값 861.838℃, 과거 중심 860.279℃, 차이 +1.559℃; SD=0.500 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 05:57:00 / 시작 고정 기준 3창 복귀 10/18 05:57:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -597.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 10/18 05:59:00–10/18 06:00:00 (끝 제외); 240108-Z-OP-Z6-0114 | 최대 절대 Z=3.38 (10/18 05:59:30, 창 중심); 1분 중앙값 75.803%, 과거 중심 69.049%, 차이 +6.755%p; SD=2.000 (하한 적용); 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 06:03:00 / 시작 고정 기준 3창 복귀 10/18 06:03:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -590.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 10/18 06:28:00–10/18 06:29:00 (끝 제외); 240108-Z-OP-Z6-0143 | 최대 절대 Z=3.05 (10/18 06:28:30, 창 중심); 1분 중앙값 78.261%, 과거 중심 71.112%, 차이 +7.149%p; SD=2.341; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 06:35:00 / 시작 고정 기준 3창 복귀 10/18 06:35:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -561.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 10/18 06:28:00–10/18 06:29:00 (끝 제외); 240108-Z-TC-Z6-0143 | 최대 절대 Z=3.23 (10/18 06:28:30, 창 중심); 1분 중앙값 858.848℃, 과거 중심 860.519℃, 차이 -1.671℃; SD=0.517; 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 06:35:00 / 시작 고정 기준 3창 복귀 10/18 06:35:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -561.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 07:58:00–10/18 08:00:00 (끝 제외); 240108-Z-TC-Z3-0233 | 최대 절대 Z=4.34 (10/18 07:59:30, 창 중심); 1분 중앙값 858.912℃, 과거 중심 861.331℃, 차이 -2.419℃; SD=0.557; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 08:06:00 / 시작 고정 기준 3창 복귀 10/18 08:06:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -470.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 08:03:00–10/18 08:04:00 (끝 제외); 240108-Z-OP-Z3-0238 | 최대 절대 Z=3.05 (10/18 08:03:30, 창 중심); 1분 중앙값 59.910%, 과거 중심 86.079%, 차이 -26.169%p; SD=8.575; 실측 59/60개·과거 26/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 08:07:00 / 시작 고정 기준 3창 복귀 10/18 08:07:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -466.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 08:24:00–10/18 08:25:00 (끝 제외); 240108-Z-OP-Z3-0259 | 최대 절대 Z=3.02 (10/18 08:24:30, 창 중심); 1분 중앙값 56.017%, 과거 중심 86.019%, 차이 -30.002%p; SD=9.928; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 08:28:00 / 시작 고정 기준 3창 복귀 10/18 08:28:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -445.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 08:31:00–10/18 08:33:00 (끝 제외); 240108-Z-OP-Z3-0266 | 최대 절대 Z=5.24 (10/18 08:32:30, 창 중심); 1분 중앙값 14.932%, 과거 중심 85.128%, 차이 -70.196%p; SD=13.408; 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 08:36:00 / 시작 고정 기준 3창 복귀 10/18 08:37:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -437.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 08:32:00–10/18 08:34:00 (끝 제외); 240108-Z-TC-Z3-0267 | 최대 절대 Z=4.14 (10/18 08:32:30, 창 중심); 1분 중앙값 865.042℃, 과거 중심 861.287℃, 차이 +3.755℃; SD=0.907; 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 08:43:00 / 시작 고정 기준 3창 복귀 10/18 08:44:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -437.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 08:36:00–10/18 08:38:00 (끝 제외); 240108-Z-TC-Z3-0271 | 최대 절대 Z=3.58 (10/18 08:36:30, 창 중심); 1분 중앙값 856.153℃, 과거 중심 861.287℃, 차이 -5.134℃; SD=1.436; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 08:43:00 / 시작 고정 기준 3창 복귀 10/18 08:43:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -433.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 09:17:00–10/18 09:18:00 (끝 제외); 240108-Z-OP-Z3-0312 | 최대 절대 Z=4.10 (10/18 09:17:30, 창 중심); 1분 중앙값 56.750%, 과거 중심 86.498%, 차이 -29.748%p; SD=7.249; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 09:21:00 / 시작 고정 기준 3창 복귀 10/18 09:22:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -392.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 09:18:00–10/18 09:19:00 (끝 제외); 240108-Z-TC-Z3-0313 | 최대 절대 Z=3.40 (10/18 09:18:30, 창 중심); 1분 중앙값 863.132℃, 과거 중심 861.285℃, 차이 +1.847℃; SD=0.543; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 09:25:00 / 시작 고정 기준 3창 복귀 10/18 09:25:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -391.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 09:21:00–10/18 09:22:00 (끝 제외); 240108-Z-TC-Z3-0316 | 최대 절대 Z=3.15 (10/18 09:21:30, 창 중심); 1분 중앙값 859.181℃, 과거 중심 861.292℃, 차이 -2.111℃; SD=0.671; 실측 54/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 09:25:00 / 시작 고정 기준 3창 복귀 10/18 09:25:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -388.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 09:44:00–10/18 09:46:00 (끝 제외); 240108-Z-OP-Z3-0339 | 최대 절대 Z=5.51 (10/18 09:44:30, 창 중심); 1분 중앙값 12.773%, 과거 중심 83.872%, 차이 -71.099%p; SD=12.910; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 09:49:00 / 시작 고정 기준 3창 복귀 10/18 09:49:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -365.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 09:44:00–10/18 09:46:00 (끝 제외); 240108-Z-TC-Z3-0339 | 최대 절대 Z=4.69 (10/18 09:45:30, 창 중심); 1분 중앙값 866.690℃, 과거 중심 861.256℃, 차이 +5.434℃; SD=1.159; 실측 55/60개·과거 30/30창 | 지속·반복 후보: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 09:49:00 / 시작 고정 기준 3창 복귀 10/18 09:57:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -364.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 09:49:00–10/18 09:50:00 (끝 제외); 240108-Z-TC-Z3-0344 | 최대 절대 Z=3.03 (10/18 09:49:30, 창 중심); 1분 중앙값 855.640℃, 과거 중심 861.192℃, 차이 -5.552℃; SD=1.830; 실측 56/60개·과거 30/30창 | 지속·반복 후보: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 09:53:00 / 시작 고정 기준 3창 복귀 10/18 09:53:00 | 짧아도 최근 30분에 3회 시작했다. 재발 간격과 동일 운전 조건을 추적; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -360.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 10:13:00–10/18 10:14:00 (끝 제외); 240108-Z-OP-Z3-0368 | 최대 절대 Z=3.31 (10/18 10:13:30, 창 중심); 1분 중앙값 11.351%, 과거 중심 88.096%, 차이 -76.745%p; SD=23.156; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 10:17:00 / 시작 고정 기준 3창 복귀 10/18 10:22:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -336.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 10:14:00–10/18 10:16:00 (끝 제외); 240108-Z-TC-Z3-0369 | 최대 절대 Z=3.12 (10/18 10:14:30, 창 중심); 1분 중앙값 869.422℃, 과거 중심 860.343℃, 차이 +9.079℃; SD=2.915; 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 10:19:00 / 시작 고정 기준 3창 복귀 10/18 10:21:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -335.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 10/18 10:24:00–10/18 10:26:00 (끝 제외); 240108-Z-OP-Z6-0379 | 최대 절대 Z=3.35 (10/18 10:25:30, 창 중심); 1분 중앙값 78.875%, 과거 중심 70.044%, 차이 +8.832%p; SD=2.635; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 10:31:00 / 시작 고정 기준 3창 복귀 10/18 10:31:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -324.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 10/18 10:34:00–10/18 10:35:00 (끝 제외); 240108-Z-OP-Z4-0389 | 최대 절대 Z=3.15 (10/18 10:34:30, 창 중심); 1분 중앙값 49.854%, 과거 중심 56.399%, 차이 -6.544%p; SD=2.081; 실측 59/60개·과거 26/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 10:38:00 / 시작 고정 기준 3창 복귀 10/18 10:38:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -315.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 10/18 11:00:00–10/18 11:03:00 (끝 제외); 240108-Z-OP-Z3-0415 | 최대 절대 Z=3.74 (10/18 11:00:30, 창 중심); 1분 중앙값 12.960%, 과거 중심 95.012%, 차이 -82.052%p; SD=21.939; 실측 56/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:06:00 / 시작 고정 기준 3창 복귀 10/18 11:10:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -289.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 10/18 11:01:00–10/18 11:04:00 (끝 제외); 240108-Z-TC-Z3-0416 | 최대 절대 Z=3.58 (10/18 11:02:30, 창 중심); 1분 중앙값 869.734℃, 과거 중심 859.882℃, 차이 +9.852℃; SD=2.748; 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:07:00 / 시작 고정 기준 3창 복귀 10/18 11:10:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -287.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 10/18 11:13:00–10/18 11:14:00 (끝 제외); 240108-Z-OP-Z6-0428 | 최대 절대 Z=3.43 (10/18 11:13:30, 창 중심); 1분 중앙값 76.259%, 과거 중심 69.174%, 차이 +7.085%p; SD=2.065; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 10/18 11:17:00 / 시작 고정 기준 3창 복귀 10/18 11:18:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01678 감지 대비 Z 최대창 중심 -276.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240108-Z-OP-Z3-0100:** TC-Z3 사건창 중앙값−직전10분 중앙값 +6.489 (9/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240108-Z-TC-Z3-0101:** OP-Z3 사건창 중앙값−직전10분 중앙값 -74.85 (9/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 10/18 04:05:00–10/18 04:26:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 10/18 04:34:00–10/18 04:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 04:48:00–10/18 04:49:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 05:41:00–10/18 05:42:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 10/18 06:26:00–10/18 06:27:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 06:31:00–10/18 06:32:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 06:44:00–10/18 06:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 07:06:00–10/18 07:08:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 07:26:00–10/18 07:27:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 10/18 07:43:00–10/18 07:44:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 10/18 07:48:00–10/18 07:49:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 07:57:00–10/18 07:58:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 10/18 08:02:00–10/18 08:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 10/18 08:38:00–10/18 08:40:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 08:48:00–10/18 08:49:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 10:02:00–10/18 10:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 10:07:00–10/18 10:08:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 10:12:00–10/18 10:13:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 10:27:00–10/18 10:28:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 10:31:00–10/18 10:32:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 10/18 10:56:00–10/18 10:57:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 11:34:00–10/18 11:35:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 10/18 11:39:00–10/18 11:40:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240108, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 466 | 421 | 0 | 45 | 25 | 20 | 0 |
| OP-Z7 | 466 | 421 | 0 | 45 | 25 | 20 | 0 |
| TC-Z8 | 466 | 421 | 0 | 45 | 25 | 20 | 0 |
| OP-Z8 | 466 | 421 | 0 | 45 | 25 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 421창, 최대 절대 Z=0.65 / OP-Z7: 유효 421창, 최대 절대 Z=2.91 / TC-Z8: 유효 421창, 최대 절대 Z=0.55 / OP-Z8: 유효 421창, 최대 절대 Z=1.26. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 10/18 04:05:00–10/18 04:26:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 10/18 04:34:00–10/18 04:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 04:48:00–10/18 04:49:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 05:41:00–10/18 05:42:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 10/18 06:26:00–10/18 06:27:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 06:31:00–10/18 06:32:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 06:44:00–10/18 06:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 07:06:00–10/18 07:08:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 07:26:00–10/18 07:27:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 10/18 07:43:00–10/18 07:44:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 10/18 07:48:00–10/18 07:49:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 07:57:00–10/18 07:58:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 10/18 08:02:00–10/18 08:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 10/18 08:38:00–10/18 08:40:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 08:48:00–10/18 08:49:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 10:02:00–10/18 10:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 10:07:00–10/18 10:08:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 10:12:00–10/18 10:13:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 10:27:00–10/18 10:28:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 10:31:00–10/18 10:32:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 10/18 10:56:00–10/18 10:57:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 11:34:00–10/18 11:35:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 10/18 11:39:00–10/18 11:40:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240108)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 26982 | 25 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.045414 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 26981 | 25 | 원본 중앙값 1.1521e-10, 최저 1.13751e-10, 최고 1.17034e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 26982 | 25 | 원본 중앙값 327.996, 최저 317, 최고 340.124; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 26981 | 25 | 원본 중앙값 279.446, 최저 264.31, 최고 292.598; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 26982 | 25 | 원본 중앙값 275.178, 최저 264.319, 최고 285.728; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-01678 · CBM / 메모: 원인 미상**

**실제 시각:** 감지 10/18 15:50:00 / 작업 10/26 23:53:00~10/27 02:42:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-01678 감지 8시간 39분 30초 전에 Z 최대창 중심.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01678 · CBM | 10/18 15:50:00 | 10/26 23:53:00 | 10/27 02:42:00 | 원인 미상 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-01678 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 40건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.045414 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 3건·최대 절대 Z 4.00 / OP-Z1 4건·최대 절대 Z 3.99 / OP-Z2 3건·최대 절대 Z 4.36 / OP-Z3 10건·최대 절대 Z 6.44 / OP-Z4 2건·최대 절대 Z 3.90 / OP-Z6 4건·최대 절대 Z 3.43 / TC-Z1 1건·최대 절대 Z 3.01 / TC-Z3 11건·최대 절대 Z 6.43 / TC-Z4 1건·최대 절대 Z 3.12 / TC-Z6 1건·최대 절대 Z 3.23. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 10/18 15:50:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 10/26 23:53:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 10/27 02:42:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240108)

**240108 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240108)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240108)
lot_conclusion(240108)
lot_execution.append({'LOT':240108,'확대 구간':len(LOT_EVENTS[240108]),'강종 대조':240108 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240114"></a>
### 8.11 LOT 240114 · SPCEN · 11/04

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240114)

#### LOT 240114 · 먼저 볼 변화

**지속·반복 후보 21건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — TC-Z4 · 11/04 20:09~20:21**

**관측:** |Z|≥3 연속 **12개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.28**, 과거 중심 대비 **-2.24℃**.  
**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -1.34%p(8/12유효창). 같은 전후 기준의 TC-Z4 변화는 -4.68℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — TC-Z5 · 11/04 20:09~20:21**

**관측:** |Z|≥3 연속 **12개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.45**, 과거 중심 대비 **-2.59℃**.  
**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** OP-Z5의 사건창 중앙값−직전 10분 중앙값은 -0.922%p(8/12유효창). 같은 전후 기준의 TC-Z5 변화는 -4.95℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 11/04 19:54~19:58**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.44**, 과거 중심 대비 **-1.78℃**.  
**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 1.61~1.61℃)이 있다. 이 구간의 4창·1.78℃는 비교 관측의 상한을 넘는다(절대 편차 1.78>1.61℃, 길이 4>2창). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240114

- **관측 범위:** 11/04 15:40:00–11/04 20:59:02.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 221개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 21개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 31개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240114, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 6건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 11/04 20:49~20:58**

**관측:** |Z|≥3 연속 **9개 1분창**, 최근 30분 시작 3회. 최대 |Z| **4.27**, 과거 중심 대비 **-30.1%p**.  
**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.468℃(10/9유효창). 같은 전후 기준의 OP-Z1 변화는 -45.1%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — TC-Z2 · 11/04 20:44~20:51**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.95**, 과거 중심 대비 **-1.97℃**.  
**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**왜 이렇게 보는가:** OP-Z2의 사건창 중앙값−직전 10분 중앙값은 +5.49%p(9/7유효창). 같은 전후 기준의 TC-Z2 변화는 -2.7℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 11/04 19:54~19:58**

**관측:** |Z|≥3 연속 **4개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.44**, 과거 중심 대비 **-1.78℃**.  
**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 1.61~1.61℃)이 있다. 이 구간의 4창·1.78℃는 비교 관측의 상한을 넘는다(절대 편차 1.78>1.61℃, 길이 4>2창). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 7건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240114-Z-CLN-IN-0254 · 11/04 19:54~19:58**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 1.61~1.61℃)이 있다. 이 구간의 4창·1.78℃는 비교 관측의 상한을 넘는다(절대 편차 1.78>1.61℃, 길이 4>2창). SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 1.784~1.784 | 4~4창 | 66.486 |
| 240038 | 1건/103유효창 (100창당 0.97건) | 1.612~1.612 | 2~2창 | 69.389 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z1-0293 · 11/04 20:33~20:35**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.244℃(10/2유효창). 같은 전후 기준의 OP-Z1 변화는 -6.09%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 20:44 (이동 Z 복귀보다 2분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z1·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 3건/35유효창 (100창당 8.57건) | 8.309~30.06 | 2~9창 | 67.351 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z1-0296 · 11/04 20:36~20:39**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.317℃(9/3유효창). 같은 전후 기준의 OP-Z1 변화는 -9.15%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 20:43 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z1·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 3건/35유효창 (100창당 8.57건) | 8.309~30.06 | 2~9창 | 67.351 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z2-0296 · 11/04 20:36~20:41**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.285℃(9/5유효창). 같은 전후 기준의 OP-Z2 변화는 -9.1%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 20:47 (이동 Z 복귀보다 3분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z2·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 10.17~10.17 | 5~5창 | 16.807 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z1-0301 · 11/04 20:41~20:47**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 +10%p(9/6유효창). 같은 전후 기준의 TC-Z1 변화는 -2.89℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z1·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 2.158~2.158 | 6~6창 | 97.524 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z2-0304 · 11/04 20:44~20:51**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** OP-Z2의 사건창 중앙값−직전 10분 중앙값은 +5.49%p(9/7유효창). 같은 전후 기준의 TC-Z2 변화는 -2.7℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z2·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 1.974~1.974 | 7~7창 | 98.485 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z1-0309 · 11/04 20:49~20:58**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +0.468℃(10/9유효창). 같은 전후 기준의 OP-Z1 변화는 -45.1%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z1·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 3건/35유효창 (100창당 8.57건) | 8.309~30.06 | 2~9창 | 67.351 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 320 | 289 | 4 | 31 | 11 | 20 | 0 |
| TC-Z1 | 320 | 289 | 6 | 31 | 11 | 20 | 0 |
| OP-Z1 | 320 | 289 | 14 | 31 | 11 | 20 | 0 |
| TC-Z2 | 320 | 289 | 7 | 31 | 11 | 20 | 0 |
| OP-Z2 | 320 | 289 | 5 | 31 | 11 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| CLN-IN · 11/04 19:54:00–11/04 19:58:00 (끝 제외); 240114-Z-CLN-IN-0254 | 최대 절대 Z=3.44 (11/04 19:55:30, 창 중심); 1분 중앙값 65.529℃, 과거 중심 67.313℃, 차이 -1.784℃; SD=0.519; 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:01:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2153.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 11/04 20:33:00–11/04 20:35:00 (끝 제외); 240114-Z-OP-Z1-0293 | 최대 절대 Z=4.15 (11/04 20:34:30, 창 중심); 1분 중앙값 65.384%, 과거 중심 73.693%, 차이 -8.309%p; SD=2.000 (하한 적용); 실측 60/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:42:00 / 시작 고정 기준 3창 복귀 11/04 20:44:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2114.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 11/04 20:36:00–11/04 20:39:00 (끝 제외); 240114-Z-OP-Z1-0296 | 최대 절대 Z=4.43 (11/04 20:36:30, 창 중심); 1분 중앙값 62.728%, 과거 중심 73.693%, 차이 -10.965%p; SD=2.474; 실측 60/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:42:00 / 시작 고정 기준 3창 복귀 11/04 20:43:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2112.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 11/04 20:36:00–11/04 20:41:00 (끝 제외); 240114-Z-OP-Z2-0296 | 최대 절대 Z=4.33 (11/04 20:38:30, 창 중심); 1분 중앙값 11.563%, 과거 중심 21.735%, 차이 -10.172%p; SD=2.347; 실측 60/60개·과거 27/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:44:00 / 시작 고정 기준 3창 복귀 11/04 20:47:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2110.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 11/04 20:41:00–11/04 20:47:00 (끝 제외); 240114-Z-TC-Z1-0301 | 최대 절대 Z=4.32 (11/04 20:42:30, 창 중심); 1분 중앙값 96.911℃, 과거 중심 99.069℃, 차이 -2.158℃; SD=0.500 (하한 적용); 실측 57/60개·과거 28/30창 | 지속·반복 후보: 연속 6창, 같은 방향 최장 6창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:50:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2106.50분 (근접 시각이며 인과 미확정) |
| TC-Z2 · 11/04 20:44:00–11/04 20:51:00 (끝 제외); 240114-Z-TC-Z2-0304 | 최대 절대 Z=3.95 (11/04 20:45:30, 창 중심); 1분 중앙값 96.808℃, 과거 중심 98.782℃, 차이 -1.974℃; SD=0.500 (하한 적용); 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:54:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2103.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 11/04 20:49:00–11/04 20:58:00 (끝 제외); 240114-Z-OP-Z1-0309 | 최대 절대 Z=4.27 (11/04 20:51:30, 창 중심); 1분 중앙값 41.076%, 과거 중심 71.134%, 차이 -30.058%p; SD=7.033; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 9창, 같은 방향 최장 9창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2097.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240114-Z-OP-Z1-0309:** TC-Z1 사건창 중앙값−직전10분 중앙값 +0.4685 (10/9유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240114-Z-TC-Z2-0304:** OP-Z2 사건창 중앙값−직전10분 중앙값 +5.491 (9/7유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/04 15:40:00–11/04 16:00:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 11/04 16:15:00–11/04 16:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 49개 <54 |
| 11/04 16:20:00–11/04 16:21:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/04 17:25:00–11/04 17:26:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/04 18:49:00–11/04 18:50:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/04 19:07:00–11/04 19:08:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/04 19:25:00–11/04 19:26:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/04 20:04:00–11/04 20:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/04 20:08:00–11/04 20:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/04 20:22:00–11/04 20:23:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/04 20:35:00–11/04 20:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/04 20:59:00–11/04 21:00:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 3개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240114, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 11건 · 짧은 노랑 후보 10건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 주황 · 지속·반복 확인 — TC-Z4 · 11/04 20:09~20:21**

**관측:** |Z|≥3 연속 **12개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.28**, 과거 중심 대비 **-2.24℃**.  
**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -1.34%p(8/12유효창). 같은 전후 기준의 TC-Z4 변화는 -4.68℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — TC-Z5 · 11/04 20:09~20:21**

**관측:** |Z|≥3 연속 **12개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.45**, 과거 중심 대비 **-2.59℃**.  
**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** OP-Z5의 사건창 중앙값−직전 10분 중앙값은 -0.922%p(8/12유효창). 같은 전후 기준의 TC-Z5 변화는 -4.95℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z6 · 11/04 20:09~20:20**

**관측:** |Z|≥3 연속 **11개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.62**, 과거 중심 대비 **-3.06℃**.  
**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.515%p(8/11유효창). 같은 전후 기준의 TC-Z6 변화는 -4.66℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 21건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240114-Z-OP-Z3-0025 · 11/04 16:05~16:06**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 36.1~77.7%p)이 있다. 이 구간의 1창·44.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.24℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -40.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:09. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |
| 240038 | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0071 · 11/04 16:51~16:52**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.904℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 +15.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:55. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 2건/254유효창 (100창당 0.79건) | 7.343~14.58 | 1~1창 | 86.778 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0077 · 11/04 16:57~16:59**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 36.1~77.7%p)이 있다. 이 구간의 2창·40.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.2℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -38%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:02. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |
| 240038 | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z3-0077 · 11/04 16:57~16:59**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 7.81~7.81℃)이 있다. 이 구간의 2창·2.7℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -38%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +2.2℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:09 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 2건/254유효창 (100창당 0.79건) | 2.698~6.252 | 2~5창 | 860.32 |
| 240038 | 1건/103유효창 (100창당 0.97건) | 7.806~7.806 | 5~5창 | 861.59 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z3-0080 · 11/04 17:00~17:04**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 1~1개 1분창, 절대 편차 2.51~2.51℃)이 있다. 이 구간의 4창·4.18℃는 비교 관측의 상한을 넘는다(절대 편차 4.18>2.51℃, 길이 4>1창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +19.7%p(10/4유효창). 같은 전후 기준의 TC-Z3 변화는 -4.88℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:09 (이동 Z 복귀보다 2분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 4.179~4.179 | 4~4창 | 860.32 |
| 240038 | 1건/103유효창 (100창당 0.97건) | 2.508~2.508 | 1~1창 | 861.59 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0134 · 11/04 17:54~17:55**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.369℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 +7.11%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:58. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 2건/254유효창 (100창당 0.79건) | 7.343~14.58 | 1~1창 | 86.778 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0146 · 11/04 18:06~18:07**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 36.1~77.7%p)이 있다. 이 구간의 1창·15.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.683℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -14%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 18:11 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |
| 240038 | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0184 · 11/04 18:44~18:46**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 36.1~77.7%p)이 있다. 이 구간의 2창·12.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.588℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -11.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 18:49. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |
| 240038 | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0195 · 11/04 18:55~18:56**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 36.1~77.7%p)이 있다. 이 구간의 1창·22.5%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.837℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -20.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:03 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |
| 240038 | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0214 · 11/04 19:14~19:18**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 36.1~77.7%p)이 있다. 이 구간의 4창·56.4%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +8.03℃(9/4유효창). 같은 전후 기준의 OP-Z3 변화는 -82.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:29 (이동 Z 복귀보다 8분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 6건/254유효창 (100창당 2.36건) | 12.17~56.35 | 1~4창 | 86.778 |
| 240038 | 3건/103유효창 (100창당 2.91건) | 36.15~77.66 | 1~4창 | 90.406 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z3-0214 · 11/04 19:14~19:19**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 5~5개 1분창, 절대 편차 7.81~7.81℃)이 있다. 이 구간의 5창·6.25℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -82.3%p(9/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.68℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:37 (이동 Z 복귀보다 15분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 2건/254유효창 (100창당 0.79건) | 2.698~6.252 | 2~5창 | 860.32 |
| 240038 | 1건/103유효창 (100창당 0.97건) | 7.806~7.806 | 5~5창 | 861.59 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z6-0226 · 11/04 19:26~19:29**

**다른 LOT:** 동종 LOT 240038에도 같은 방향 후보 1건(연속 2~2개 1분창, 절대 편차 7.57~7.57%p)이 있다. 이 구간의 3창·8.89%p는 비교 관측의 상한을 넘는다(절대 편차 8.89>7.57%p, 길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.19℃(9/3유효창). 같은 전후 기준의 OP-Z6 변화는 +8.04%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 19:35 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 8.888~8.888 | 3~3창 | 73.502 |
| 240038 | 1건/103유효창 (100창당 0.97건) | 7.567~7.567 | 2~2창 | 73.205 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z4-0269 · 11/04 20:09~20:21**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 -1.34%p(8/12유효창). 같은 전후 기준의 TC-Z4 변화는 -4.68℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 2.244~2.244 | 12~12창 | 860.41 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 860.35 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z5-0269 · 11/04 20:09~20:21**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z5의 사건창 중앙값−직전 10분 중앙값은 -0.922%p(8/12유효창). 같은 전후 기준의 TC-Z5 변화는 -4.95℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z5·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 2.592~2.592 | 12~12창 | 860.44 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 860.29 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z6-0269 · 11/04 20:09~20:20**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.515%p(8/11유효창). 같은 전후 기준의 TC-Z6 변화는 -4.66℃. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 3.057~3.057 | 11~11창 | 860.24 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 860.48 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z4-0283 · 11/04 20:23~20:24**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** OP-Z4의 사건창 중앙값−직전 10분 중앙값은 +1.49%p(9/1유효창). 같은 전후 기준의 TC-Z4 변화는 -4.33℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z4·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 9.379~9.379 | 1~1창 | 834.23 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z3-0284 · 11/04 20:24~20:26**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 -0.976℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -66%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 20:29. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z3·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 64.25~64.25 | 2~2창 | 74.951 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z3-0287 · 11/04 20:27~20:30**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +17.9%p(9/3유효창). 같은 전후 기준의 TC-Z3 변화는 -9.22℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z3·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 12.55~12.55 | 3~3창 | 835.61 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z4-0308 · 11/04 20:48~20:59**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 -12.4℃(10/11유효창). 같은 전후 기준의 OP-Z4 변화는 -29.3%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z4·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 16.52~16.52 | 11~11창 | 50.391 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z5-0308 · 11/04 20:48~20:59**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z5의 사건창 중앙값−직전 10분 중앙값은 -12℃(10/11유효창). 같은 전후 기준의 OP-Z5 변화는 -32.8%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z5·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 16.64~16.64 | 11~11창 | 54.869 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z6-0308 · 11/04 20:48~20:59**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -11.5℃(10/11유효창). 같은 전후 기준의 OP-Z6 변화는 -45.8%p. 가열 존의 온도·출력이 같은 방향으로 변했다. SV 변경·부하/라인 속도 변화·수동 조작·응답 지연을 먼저 구분하고 제어 불량으로 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z6·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 18.55~18.55 | 11~11창 | 73.392 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 320 | 289 | 14 | 31 | 11 | 20 | 0 |
| OP-Z3 | 320 | 289 | 15 | 31 | 11 | 20 | 0 |
| TC-Z4 | 320 | 289 | 13 | 31 | 11 | 20 | 0 |
| OP-Z4 | 320 | 289 | 11 | 31 | 11 | 20 | 0 |
| TC-Z5 | 320 | 289 | 12 | 31 | 11 | 20 | 0 |
| OP-Z5 | 320 | 289 | 11 | 31 | 11 | 20 | 0 |
| TC-Z6 | 320 | 289 | 11 | 31 | 11 | 20 | 0 |
| OP-Z6 | 320 | 289 | 14 | 31 | 11 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 11/04 16:05:00–11/04 16:06:00 (끝 제외); 240114-Z-OP-Z3-0025 | 최대 절대 Z=4.77 (11/04 16:05:30, 창 중심); 1분 중앙값 45.137%, 과거 중심 89.616%, 차이 -44.479%p; SD=9.332; 실측 60/60개·과거 25/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 16:09:00 / 시작 고정 기준 3창 복귀 11/04 16:09:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2383.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 16:51:00–11/04 16:52:00 (끝 제외); 240114-Z-OP-Z3-0071 | 최대 절대 Z=3.09 (11/04 16:51:30, 창 중심); 1분 중앙값 100.000%, 과거 중심 85.420%, 차이 +14.580%p; SD=4.714; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 16:55:00 / 시작 고정 기준 3창 복귀 11/04 16:55:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2337.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 16:57:00–11/04 16:59:00 (끝 제외); 240114-Z-OP-Z3-0077 | 최대 절대 Z=6.60 (11/04 16:57:30, 창 중심); 1분 중앙값 44.359%, 과거 중심 84.846%, 차이 -40.487%p; SD=6.131; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 17:02:00 / 시작 고정 기준 3창 복귀 11/04 17:02:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2331.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/04 16:57:00–11/04 16:59:00 (끝 제외); 240114-Z-TC-Z3-0077 | 최대 절대 Z=4.94 (11/04 16:58:30, 창 중심); 1분 중앙값 863.189℃, 과거 중심 860.491℃, 차이 +2.698℃; SD=0.546; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 17:07:00 / 시작 고정 기준 3창 복귀 11/04 17:09:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2330.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/04 17:00:00–11/04 17:04:00 (끝 제외); 240114-Z-TC-Z3-0080 | 최대 절대 Z=4.89 (11/04 17:01:30, 창 중심); 1분 중앙값 856.312℃, 과거 중심 860.491℃, 차이 -4.179℃; SD=0.855; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 17:07:00 / 시작 고정 기준 3창 복귀 11/04 17:09:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2327.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 17:54:00–11/04 17:55:00 (끝 제외); 240114-Z-OP-Z3-0134 | 최대 절대 Z=3.03 (11/04 17:54:30, 창 중심); 1분 중앙값 94.614%, 과거 중심 87.271%, 차이 +7.343%p; SD=2.422; 실측 54/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 17:58:00 / 시작 고정 기준 3창 복귀 11/04 17:58:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2274.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 18:06:00–11/04 18:07:00 (끝 제외); 240114-Z-OP-Z3-0146 | 최대 절대 Z=4.53 (11/04 18:06:30, 창 중심); 1분 중앙값 71.167%, 과거 중심 86.724%, 차이 -15.557%p; SD=3.434; 실측 56/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 18:10:00 / 시작 고정 기준 3창 복귀 11/04 18:11:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2262.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 18:44:00–11/04 18:46:00 (끝 제외); 240114-Z-OP-Z3-0184 | 최대 절대 Z=3.34 (11/04 18:45:30, 창 중심); 1분 중앙값 73.924%, 과거 중심 86.089%, 차이 -12.165%p; SD=3.640; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 18:49:00 / 시작 고정 기준 3창 복귀 11/04 18:49:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2223.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 18:55:00–11/04 18:56:00 (끝 제외); 240114-Z-OP-Z3-0195 | 최대 절대 Z=5.68 (11/04 18:55:30, 창 중심); 1분 중앙값 63.553%, 과거 중심 86.046%, 차이 -22.493%p; SD=3.963; 실측 59/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 18:59:00 / 시작 고정 기준 3창 복귀 11/04 19:03:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2213.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 19:14:00–11/04 19:18:00 (끝 제외); 240114-Z-OP-Z3-0214 | 최대 절대 Z=6.45 (11/04 19:14:30, 창 중심); 1분 중앙값 28.951%, 과거 중심 85.303%, 차이 -56.351%p; SD=8.734; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 19:21:00 / 시작 고정 기준 3창 복귀 11/04 19:29:00 | 고정 범위 복귀가 이동 Z 복귀보다 8분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2194.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/04 19:14:00–11/04 19:19:00 (끝 제외); 240114-Z-TC-Z3-0214 | 최대 절대 Z=8.42 (11/04 19:15:30, 창 중심); 1분 중앙값 866.690℃, 과거 중심 860.438℃, 차이 +6.252℃; SD=0.742; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 19:22:00 / 시작 고정 기준 3창 복귀 11/04 19:37:00 | 고정 범위 복귀가 이동 Z 복귀보다 15분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2193.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 11/04 19:26:00–11/04 19:29:00 (끝 제외); 240114-Z-OP-Z6-0226 | 최대 절대 Z=4.28 (11/04 19:27:30, 창 중심); 1분 중앙값 83.359%, 과거 중심 74.470%, 차이 +8.888%p; SD=2.077; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 19:32:00 / 시작 고정 기준 3창 복귀 11/04 19:35:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2181.50분 (근접 시각이며 인과 미확정) |
| TC-Z4 · 11/04 20:09:00–11/04 20:21:00 (끝 제외); 240114-Z-TC-Z4-0269 | 최대 절대 Z=4.28 (11/04 20:09:30, 창 중심); 1분 중앙값 857.814℃, 과거 중심 860.058℃, 차이 -2.244℃; SD=0.524; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 12창, 같은 방향 최장 12창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:27:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2139.50분 (근접 시각이며 인과 미확정) |
| TC-Z5 · 11/04 20:09:00–11/04 20:21:00 (끝 제외); 240114-Z-TC-Z5-0269 | 최대 절대 Z=4.45 (11/04 20:10:30, 창 중심); 1분 중앙값 857.356℃, 과거 중심 859.948℃, 차이 -2.592℃; SD=0.583; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 12창, 같은 방향 최장 12창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:26:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2138.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 11/04 20:09:00–11/04 20:20:00 (끝 제외); 240114-Z-TC-Z6-0269 | 최대 절대 Z=3.62 (11/04 20:10:30, 창 중심); 1분 중앙값 857.091℃, 과거 중심 860.148℃, 차이 -3.057℃; SD=0.843; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 11창, 같은 방향 최장 11창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:26:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2138.50분 (근접 시각이며 인과 미확정) |
| TC-Z4 · 11/04 20:23:00–11/04 20:24:00 (끝 제외); 240114-Z-TC-Z4-0283 | 최대 절대 Z=3.01 (11/04 20:23:30, 창 중심); 1분 중앙값 849.231℃, 과거 중심 858.610℃, 차이 -9.379℃; SD=3.115; 실측 60/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:27:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2125.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/04 20:24:00–11/04 20:26:00 (끝 제외); 240114-Z-OP-Z3-0284 | 최대 절대 Z=5.24 (11/04 20:24:30, 창 중심); 1분 중앙값 23.622%, 과거 중심 87.868%, 차이 -64.245%p; SD=12.263; 실측 60/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:29:00 / 시작 고정 기준 3창 복귀 11/04 20:29:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2124.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/04 20:27:00–11/04 20:30:00 (끝 제외); 240114-Z-TC-Z3-0287 | 최대 절대 Z=3.35 (11/04 20:28:30, 창 중심); 1분 중앙값 842.455℃, 과거 중심 855.005℃, 차이 -12.551℃; SD=3.750; 실측 60/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:33:00 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2120.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 11/04 20:48:00–11/04 20:59:00 (끝 제외); 240114-Z-OP-Z4-0308 | 최대 절대 Z=6.10 (11/04 20:50:30, 창 중심); 1분 중앙값 34.263%, 과거 중심 50.786%, 차이 -16.523%p; SD=2.708; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 11창, 같은 방향 최장 11창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2098.50분 (근접 시각이며 인과 미확정) |
| OP-Z5 · 11/04 20:48:00–11/04 20:59:00 (끝 제외); 240114-Z-OP-Z5-0308 | 최대 절대 Z=7.43 (11/04 20:49:30, 창 중심); 1분 중앙값 38.775%, 과거 중심 55.412%, 차이 -16.637%p; SD=2.240; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 11창, 같은 방향 최장 11창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2099.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 11/04 20:48:00–11/04 20:59:00 (끝 제외); 240114-Z-OP-Z6-0308 | 최대 절대 Z=6.25 (11/04 20:49:30, 창 중심); 1분 중앙값 56.063%, 과거 중심 74.615%, 차이 -18.552%p; SD=2.969; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 11창, 같은 방향 최장 11창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2099.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240114-Z-TC-Z5-0269:** OP-Z5 사건창 중앙값−직전10분 중앙값 -0.9218 (8/12유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240114-Z-TC-Z4-0269:** OP-Z4 사건창 중앙값−직전10분 중앙값 -1.343 (8/12유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/04 15:40:00–11/04 16:00:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 과거 유효 최소 0창 <20 |
| 11/04 16:15:00–11/04 16:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 49개 <54 |
| 11/04 16:20:00–11/04 16:21:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/04 17:25:00–11/04 17:26:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/04 18:49:00–11/04 18:50:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/04 19:07:00–11/04 19:08:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/04 19:25:00–11/04 19:26:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/04 20:04:00–11/04 20:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/04 20:08:00–11/04 20:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/04 20:22:00–11/04 20:23:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/04 20:35:00–11/04 20:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/04 20:59:00–11/04 21:00:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 3개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240114, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z7 · 11/04 20:48~20:59**

**관측:** |Z|≥3 연속 **11개 1분창**, 최근 30분 시작 1회. 최대 |Z| **6.43**, 과거 중심 대비 **-12.9%p**.  
**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**왜 이렇게 보는가:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -4.22℃(10/11유효창). 같은 전후 기준의 OP-Z7 변화는 -40.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z8 · 11/04 20:48~20:59**

**관측:** |Z|≥3 연속 **11개 1분창**, 최근 30분 시작 1회. 최대 |Z| **7.39**, 과거 중심 대비 **-16.2%p**.  
**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**왜 이렇게 보는가:** TC-Z8의 사건창 중앙값−직전 10분 중앙값은 -4.32℃(10/11유효창). 같은 전후 기준의 OP-Z8 변화는 -35.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z8 · 11/04 20:12~20:19**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **3.51**, 과거 중심 대비 **-1.75℃**.  
**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** OP-Z8의 사건창 중앙값−직전 10분 중앙값은 -0.169%p(8/7유효창). 같은 전후 기준의 TC-Z8 변화는 -1.44℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 4건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240114-Z-TC-Z8-0272 · 11/04 20:12~20:19**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z8의 사건창 중앙값−직전 10분 중앙값은 -0.169%p(8/7유효창). 같은 전후 기준의 TC-Z8 변화는 -1.44℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z8·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 1.753~1.753 | 7~7창 | 333.6 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 330.2 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-TC-Z7-0273 · 11/04 20:13~20:20**

**다른 LOT:** 같은 강종 LOT 240038의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** OP-Z7의 사건창 중앙값−직전 10분 중앙값은 +2.74%p(8/7유효창). 같은 전후 기준의 TC-Z7 변화는 -1.61℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 TC-Z7·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/254유효창 (100창당 0.39건) | 1.908~1.908 | 7~7창 | 333.23 |
| 240038 | 0건/103유효창 (100창당 0.00건) | — | — | 330.19 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z7-0308 · 11/04 20:48~20:59**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 -4.22℃(10/11유효창). 같은 전후 기준의 OP-Z7 변화는 -40.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z7·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 12.86~12.86 | 11~11창 | 67.169 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240114-Z-OP-Z8-0308 · 11/04 20:48~20:59**

**다른 LOT:** 같은 강종 LOT 240038에는 같은 관측 형태·유효 자료가 없어 직접 대조를 보류한다.  
**해석 가설:** TC-Z8의 사건창 중앙값−직전 10분 중앙값은 -4.32℃(10/11유효창). 같은 전후 기준의 OP-Z8 변화는 -35.3%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCEN·채널 OP-Z8·동일 편차 방향·사후 관측 형태 종료 전환로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240114 (현재 LOT) | 1건/35유효창 (100창당 2.86건) | 16.24~16.24 | 11~11창 | 58.457 |
| 240038 | 비교 보류 | 해당 형태 유효 관측 없음 | — | — |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 320 | 289 | 7 | 31 | 11 | 20 | 0 |
| OP-Z7 | 320 | 289 | 11 | 31 | 11 | 20 | 0 |
| TC-Z8 | 320 | 289 | 7 | 31 | 11 | 20 | 0 |
| OP-Z8 | 320 | 289 | 11 | 31 | 11 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| TC-Z8 · 11/04 20:12:00–11/04 20:19:00 (끝 제외); 240114-Z-TC-Z8-0272 | 최대 절대 Z=3.51 (11/04 20:13:30, 창 중심); 1분 중앙값 331.663℃, 과거 중심 333.417℃, 차이 -1.753℃; SD=0.500 (하한 적용); 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:22:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2135.50분 (근접 시각이며 인과 미확정) |
| TC-Z7 · 11/04 20:13:00–11/04 20:20:00 (끝 제외); 240114-Z-TC-Z7-0273 | 최대 절대 Z=3.80 (11/04 20:14:30, 창 중심); 1분 중앙값 331.187℃, 과거 중심 333.094℃, 차이 -1.908℃; SD=0.502; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/04 20:26:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2134.50분 (근접 시각이며 인과 미확정) |
| OP-Z7 · 11/04 20:48:00–11/04 20:59:00 (끝 제외); 240114-Z-OP-Z7-0308 | 최대 절대 Z=6.43 (11/04 20:48:30, 창 중심); 1분 중앙값 55.343%, 과거 중심 68.201%, 차이 -12.858%p; SD=2.000 (하한 적용); 실측 59/60개·과거 28/30창 | 지속·반복 후보: 연속 11창, 같은 방향 최장 11창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2100.50분 (근접 시각이며 인과 미확정) |
| OP-Z8 · 11/04 20:48:00–11/04 20:59:00 (끝 제외); 240114-Z-OP-Z8-0308 | 최대 절대 Z=7.39 (11/04 20:49:30, 창 중심); 1분 중앙값 42.355%, 과거 중심 58.593%, 차이 -16.237%p; SD=2.197; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 11창, 같은 방향 최장 11창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01775 감지 대비 Z 최대창 중심 -2099.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240114-Z-OP-Z8-0308:** TC-Z8 사건창 중앙값−직전10분 중앙값 -4.324 (10/11유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240114-Z-OP-Z7-0308:** TC-Z7 사건창 중앙값−직전10분 중앙값 -4.221 (10/11유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/04 15:40:00–11/04 16:00:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 11/04 16:15:00–11/04 16:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 49개 <54 |
| 11/04 16:20:00–11/04 16:21:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/04 17:25:00–11/04 17:26:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/04 18:49:00–11/04 18:50:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/04 19:07:00–11/04 19:08:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/04 19:25:00–11/04 19:26:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/04 20:04:00–11/04 20:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/04 20:08:00–11/04 20:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/04 20:22:00–11/04 20:23:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/04 20:35:00–11/04 20:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/04 20:59:00–11/04 21:00:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 3개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240114)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 18763 | 11 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.015488 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 18758 | 11 | 원본 중앙값 1.15243e-10, 최저 3.76705e-11, 최고 1.1706e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 18758 | 11 | 원본 중앙값 330.735, 최저 313.579, 최고 342.605; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 18763 | 11 | 원본 중앙값 285.047, 최저 264.255, 최고 298.262; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 18760 | 11 | 원본 중앙값 280.588, 최저 265.287, 최고 291.281; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-01775 · TBM / 메모: 정기수리 실시**

**실제 시각:** 감지 11/06 07:49:00 / 작업 11/06 10:40:00~11/06 12:23:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-01775 감지 1일 11시간 39분 전에 Z 최대창 중심.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 2건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 2건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01775 · TBM | 11/06 07:49:00 | 11/06 10:40:00 | 11/06 12:23:00 | 정기수리 실시 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-01794 · TBM | 11/07 09:20:00 | 11/07 12:53:00 | 11/07 14:14:00 | 정기수리 부속작업 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-01775 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 32건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.015488 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.44 / OP-Z1 3건·최대 절대 Z 4.43 / OP-Z2 1건·최대 절대 Z 4.33 / OP-Z3 9건·최대 절대 Z 6.60 / OP-Z4 1건·최대 절대 Z 6.10 / OP-Z5 1건·최대 절대 Z 7.43 / OP-Z6 2건·최대 절대 Z 6.25 / OP-Z7 1건·최대 절대 Z 6.43 / OP-Z8 1건·최대 절대 Z 7.39 / TC-Z1 1건·최대 절대 Z 4.32 / TC-Z2 1건·최대 절대 Z 3.95 / TC-Z3 4건·최대 절대 Z 8.42 / TC-Z4 2건·최대 절대 Z 4.28 / TC-Z5 1건·최대 절대 Z 4.45 / TC-Z6 1건·최대 절대 Z 3.62 / TC-Z7 1건·최대 절대 Z 3.80 / TC-Z8 1건·최대 절대 Z 3.51. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 11/06 07:49:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 11/06 10:40:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 11/06 12:23:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-01794 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 32건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.015488 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.44 / OP-Z1 3건·최대 절대 Z 4.43 / OP-Z2 1건·최대 절대 Z 4.33 / OP-Z3 9건·최대 절대 Z 6.60 / OP-Z4 1건·최대 절대 Z 6.10 / OP-Z5 1건·최대 절대 Z 7.43 / OP-Z6 2건·최대 절대 Z 6.25 / OP-Z7 1건·최대 절대 Z 6.43 / OP-Z8 1건·최대 절대 Z 7.39 / TC-Z1 1건·최대 절대 Z 4.32 / TC-Z2 1건·최대 절대 Z 3.95 / TC-Z3 4건·최대 절대 Z 8.42 / TC-Z4 2건·최대 절대 Z 4.28 / TC-Z5 1건·최대 절대 Z 4.45 / TC-Z6 1건·최대 절대 Z 3.62 / TC-Z7 1건·최대 절대 Z 3.80 / TC-Z8 1건·최대 절대 Z 3.51. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 11/07 09:20:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 11/07 12:53:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 11/07 14:14:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240114)

**240114 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240114)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240114)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240114(SPCEN)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 333.233 - 대조 330.187 = **+3.046℃**, 다른 1 LOT과 대조.
- TC-Z8: 이 LOT 333.607 - 대조 330.200 = **+3.407℃**, 다른 1 LOT과 대조.
- OP-Z4: 이 LOT 53.331 - 대조 70.460 = **-17.129%p**, 다른 1 LOT과 대조.
- OP-Z5: 이 LOT 54.505 - 대조 58.724 = **-4.219%p**, 다른 1 LOT과 대조.
- OP-Z7: 이 LOT 66.910 - 대조 71.180 = **-4.270%p**, 다른 1 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 +0.058℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- TC-Z5: 차이 +0.148℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 -3.022%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240038: 전환 제외 없음.
- 240114: 11/04 20:08–20:59.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.
같은 강종이 두 LOT뿐이므로 차이의 방향은 대칭적이다. 어느 LOT이 정상인지 정할 수 없다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240114)
lot_conclusion(240114)
lot_execution.append({'LOT':240114,'확대 구간':len(LOT_EVENTS[240114]),'강종 대조':240114 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240117"></a>
### 8.12 LOT 240117 · SPCD · 11/13

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

정비 대조의 전·후는 선택한 **CBM 01854** 기준이다. 착수 전 패널에도 **TBM 01859가 진행 중**이므로 이 구간을 정비가 없는 정상 기준으로 해석하지 않는다.

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240117)

#### LOT 240117 · 먼저 볼 변화

**지속·반복 후보 6건 / CP 원본 후보 1건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 11/13 13:04:50**

**관측:** 원본 편차 **-0.0829무차원**, 283점·첫/끝 간격 290초. 1초 단발 여부: 아님.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 11/13 13:09:27. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01854 완료 20분 50초 후에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 11/13 12:55~13:00**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.84**, 과거 중심 대비 **+7.41℃**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -74.3%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.33℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:07 (이동 Z 복귀보다 4분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z1 · 11/13 14:14~14:19**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.14**, 과거 중심 대비 **-11.9%p**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.66℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 14:32 (이동 Z 복귀보다 10분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240117

- **관측 범위:** 11/13 07:08:00–11/13 14:32:19.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 373개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 6개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 42개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 1건 중 확인된 1초 단발 0건, 복귀 펄스 후보 1건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240117, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 3건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 11/13 14:14~14:19**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.14**, 과거 중심 대비 **-11.9%p**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.66℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 14:32 (이동 Z 복귀보다 10분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 11/13 14:15~14:20**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.00**, 과거 중심 대비 **-10.1%p**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.04℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -10.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 11/13 14:11~14:12**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.14**, 과거 중심 대비 **-1.59℃**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 4건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240117-Z-CLN-IN-0423 · 11/13 14:11~14:12**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z1-0426 · 11/13 14:14~14:19**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.66℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.5%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 14:32 (이동 Z 복귀보다 10분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z2-0427 · 11/13 14:15~14:20**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.04℃(10/5유효창). 같은 전후 기준의 OP-Z2 변화는 -10.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-TC-Z1-0427 · 11/13 14:15~14:19**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -14%p(10/4유효창). 같은 전후 기준의 TC-Z1 변화는 +1.66℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 14:23 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 445 | 403 | 1 | 42 | 22 | 20 | 0 |
| TC-Z1 | 445 | 403 | 4 | 42 | 22 | 20 | 0 |
| OP-Z1 | 445 | 403 | 5 | 42 | 22 | 20 | 0 |
| TC-Z2 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| OP-Z2 | 445 | 403 | 5 | 42 | 22 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z2: 유효 403창, 최대 절대 Z=2.73. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| CLN-IN · 11/13 14:11:00–11/13 14:12:00 (끝 제외); 240117-Z-CLN-IN-0423 | 최대 절대 Z=3.14 (11/13 14:11:30, 창 중심); 1분 중앙값 69.106℃, 과거 중심 70.697℃, 차이 -1.591℃; SD=0.507; 실측 56/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 14:15:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +87.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 11/13 14:14:00–11/13 14:19:00 (끝 제외); 240117-Z-OP-Z1-0426 | 최대 절대 Z=5.14 (11/13 14:15:30, 창 중심); 1분 중앙값 62.695%, 과거 중심 74.587%, 차이 -11.892%p; SD=2.316; 실측 56/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 14:22:00 / 시작 고정 기준 3창 복귀 11/13 14:32:00 | 고정 범위 복귀가 이동 Z 복귀보다 10분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +91.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 11/13 14:15:00–11/13 14:20:00 (끝 제외); 240117-Z-OP-Z2-0427 | 최대 절대 Z=4.00 (11/13 14:16:30, 창 중심); 1분 중앙값 16.124%, 과거 중심 26.191%, 차이 -10.067%p; SD=2.515; 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 14:23:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +92.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 11/13 14:15:00–11/13 14:19:00 (끝 제외); 240117-Z-TC-Z1-0427 | 최대 절대 Z=3.73 (11/13 14:16:30, 창 중심); 1분 중앙값 101.804℃, 과거 중심 99.939℃, 차이 +1.865℃; SD=0.500 (하한 적용); 실측 58/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 14:22:00 / 시작 고정 기준 3창 복귀 11/13 14:23:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +92.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240117-Z-OP-Z1-0426:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.657 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240117-Z-OP-Z2-0427:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.04 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/13 07:08:00–11/13 07:29:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 11/13 07:32:00–11/13 07:34:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/13 08:10:00–11/13 08:11:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/13 08:19:00–11/13 08:20:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 08:24:00–11/13 08:25:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 08:29:00–11/13 08:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 08:43:00–11/13 08:44:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 09:02:00–11/13 09:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 09:17:00–11/13 09:18:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/13 09:26:00–11/13 09:27:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/13 09:33:00–11/13 09:34:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 09:35:00–11/13 09:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 11/13 09:40:00–11/13 09:41:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 10:51:00–11/13 10:52:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 12:04:00–11/13 12:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/13 12:40:00–11/13 12:41:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/13 13:15:00–11/13 13:16:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/13 13:39:00–11/13 13:40:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/13 13:47:00–11/13 13:48:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/13 14:28:00–11/13 14:29:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/13 14:32:00–11/13 14:33:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 20개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240117, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 3건 · 짧은 노랑 후보 12건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 분홍 · CP 원본 추적 — CP-Z4 · 11/13 13:04:50**

**관측:** 원본 편차 **-0.0829무차원**, 283점·첫/끝 간격 290초. 1초 단발 여부: 아님.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**어떻게 볼 것인가:** 60개 연속 실제 1초 값의 ±0.02 복귀 11/13 13:09:27. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01854 완료 20분 50초 후에 CP 원본 피크.

**2. 주황 · 지속·반복 확인 — TC-Z3 · 11/13 12:55~13:00**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 2회. 최대 |Z| **6.84**, 과거 중심 대비 **+7.41℃**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -74.3%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.33℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:07 (이동 Z 복귀보다 4분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 주황 · 다른 존의 변화 — OP-Z6 · 11/13 13:07~13:10**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.09**, 과거 중심 대비 **+9.52%p**.  
**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.4℃(10/3유효창). 같은 전후 기준의 OP-Z6 변화는 +9.18%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 13:19 (이동 Z 복귀보다 6분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 16건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240117-Z-OP-Z3-0036 · 11/13 07:44~07:46**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.02℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -40.1%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 07:49. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0075 · 11/13 08:23~08:24**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.58℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -40.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다. 승온·하강 형태와 겹쳐 운전 전환 여부를 먼저 확인한다.  
**추적:** 시작 고정 대역 3창 복귀 08:35 (이동 Z 복귀보다 7분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0098 · 11/13 08:46~08:48**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.28℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -68.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:51. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0142 · 11/13 09:30~09:31**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.66℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -60.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:39. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0183 · 11/13 10:11~10:12**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.35℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -68.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:15. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-TC-Z3-0183 · 11/13 10:11~10:12**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -68.3%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +4.35℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:21 (이동 Z 복귀보다 6분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0201 · 11/13 10:29~10:31**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.17℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -73.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:34. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0261 · 11/13 11:29~11:30**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.14℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -75.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:33. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-TC-Z3-0261 · 11/13 11:29~11:30**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -75.3%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +5.14℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:34 (이동 Z 복귀보다 1분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0314 · 11/13 12:22~12:24**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.947℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -22.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:38 (이동 Z 복귀보다 11분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-TC-Z3-0318 · 11/13 12:26~12:28**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +20.2%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -2.19℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:31. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z3-0347 · 11/13 12:55~12:58**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +7.18℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -74.3%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:07 (이동 Z 복귀보다 6분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-TC-Z3-0347 · 11/13 12:55~13:00**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -74.3%p(10/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.33℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:07 (이동 Z 복귀보다 4분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-CP-01 · 11/13 13:04:50**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** 동일 1분에 Z3~Z6의 TC·OP 기준 초과는 없었다. CP는 무차원이나 물리량·CP-Z4M과의 계산 관계가 미확인이다. 공정 동반 변화와 CP 계측/계산 경로의 변동을 경쟁 가설로 둔다.  
**추적:** 60개 연속 실제 1초 값의 ±0.02 복귀 11/13 13:09:27. 원값·전후 값·행 번호를 보존하고 재발 간격, CP-Z4M 원본, 수집·교정 로그를 대조한다. WO-2024-01854 완료 20분 50초 후에 CP 원본 피크.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-OP-Z6-0359 · 11/13 13:07~13:10**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.4℃(10/3유효창). 같은 전후 기준의 OP-Z6 변화는 +9.18%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:19 (이동 Z 복귀보다 6분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

**240117-Z-TC-Z3-0439 · 11/13 14:27~14:28**

**다른 LOT:** SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +24.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.88℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT가 없어 비교표를 만들 수 없다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 445 | 403 | 10 | 42 | 22 | 20 | 0 |
| OP-Z3 | 445 | 403 | 15 | 42 | 22 | 20 | 0 |
| TC-Z4 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| OP-Z4 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| TC-Z5 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| OP-Z5 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| TC-Z6 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| OP-Z6 | 445 | 403 | 3 | 42 | 22 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 1건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 403창, 최대 절대 Z=2.51 / OP-Z4: 유효 403창, 최대 절대 Z=2.92 / TC-Z5: 유효 403창, 최대 절대 Z=1.89 / OP-Z5: 유효 403창, 최대 절대 Z=2.06 / TC-Z6: 유효 403창, 최대 절대 Z=2.85. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 11/13 07:44:00–11/13 07:46:00 (끝 제외); 240117-Z-OP-Z3-0036 | 최대 절대 Z=3.60 (11/13 07:44:30, 창 중심); 1분 중앙값 38.607%, 과거 중심 78.734%, 차이 -40.127%p; SD=11.136; 실측 56/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 07:49:00 / 시작 고정 기준 3창 복귀 11/13 07:49:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01859 착수 대비 Z 최대창 중심 +4.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 08:23:00–11/13 08:24:00 (끝 제외); 240117-Z-OP-Z3-0075 | 최대 절대 Z=6.60 (11/13 08:23:30, 창 중심); 1분 중앙값 36.388%, 과거 중심 76.992%, 차이 -40.604%p; SD=6.150; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 08:28:00 / 시작 고정 기준 3창 복귀 11/13 08:35:00 | 승온·종료 전환 형태와 겹친다. 실제 모드·SV로 의도된 전환인지 확인; 정비 대조: WO-2024-01854 착수 대비 Z 최대창 중심 -30.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 08:46:00–11/13 08:48:00 (끝 제외); 240117-Z-OP-Z3-0098 | 최대 절대 Z=4.54 (11/13 08:46:30, 창 중심); 1분 중앙값 2.332%, 과거 중심 77.446%, 차이 -75.114%p; SD=16.547; 실측 59/60개·과거 26/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 08:51:00 / 시작 고정 기준 3창 복귀 11/13 08:51:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01854 착수 대비 Z 최대창 중심 -7.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 09:30:00–11/13 09:31:00 (끝 제외); 240117-Z-OP-Z3-0142 | 최대 절대 Z=3.52 (11/13 09:30:30, 창 중심); 1분 중앙값 12.540%, 과거 중심 79.535%, 차이 -66.995%p; SD=19.029; 실측 60/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 09:39:00 / 시작 고정 기준 3창 복귀 11/13 09:39:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01859 완료 대비 Z 최대창 중심 -4.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 10:11:00–11/13 10:12:00 (끝 제외); 240117-Z-OP-Z3-0183 | 최대 절대 Z=3.53 (11/13 10:11:30, 창 중심); 1분 중앙값 4.039%, 과거 중심 76.172%, 차이 -72.132%p; SD=20.447; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 10:15:00 / 시작 고정 기준 3창 복귀 11/13 10:15:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01859 완료 대비 Z 최대창 중심 +36.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/13 10:11:00–11/13 10:12:00 (끝 제외); 240117-Z-TC-Z3-0183 | 최대 절대 Z=3.22 (11/13 10:11:30, 창 중심); 1분 중앙값 864.183℃, 과거 중심 859.295℃, 차이 +4.888℃; SD=1.516; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 10:15:00 / 시작 고정 기준 3창 복귀 11/13 10:21:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01859 완료 대비 Z 최대창 중심 +36.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 10:29:00–11/13 10:31:00 (끝 제외); 240117-Z-OP-Z3-0201 | 최대 절대 Z=3.63 (11/13 10:29:30, 창 중심); 1분 중앙값 0.621%, 과거 중심 74.884%, 차이 -74.264%p; SD=20.477; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 10:34:00 / 시작 고정 기준 3창 복귀 11/13 10:34:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01859 완료 대비 Z 최대창 중심 +54.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 11:29:00–11/13 11:30:00 (끝 제외); 240117-Z-OP-Z3-0261 | 최대 절대 Z=3.10 (11/13 11:29:30, 창 중심); 1분 중앙값 0.850%, 과거 중심 77.585%, 차이 -76.735%p; SD=24.774; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 11:33:00 / 시작 고정 기준 3창 복귀 11/13 11:33:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 -74.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/13 11:29:00–11/13 11:30:00 (끝 제외); 240117-Z-TC-Z3-0261 | 최대 절대 Z=3.32 (11/13 11:29:30, 창 중심); 1분 중앙값 864.776℃, 과거 중심 858.483℃, 차이 +6.293℃; SD=1.896; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 11:33:00 / 시작 고정 기준 3창 복귀 11/13 11:34:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 -74.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 12:22:00–11/13 12:24:00 (끝 제외); 240117-Z-OP-Z3-0314 | 최대 절대 Z=5.44 (11/13 12:22:30, 창 중심); 1분 중앙값 53.379%, 과거 중심 78.266%, 차이 -24.887%p; SD=4.575; 실측 55/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 12:27:00 / 시작 고정 기준 3창 복귀 11/13 12:38:00 | 고정 범위 복귀가 이동 Z 복귀보다 11분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 -21.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/13 12:26:00–11/13 12:28:00 (끝 제외); 240117-Z-TC-Z3-0318 | 최대 절대 Z=3.78 (11/13 12:26:30, 창 중심); 1분 중앙값 858.170℃, 과거 중심 860.061℃, 차이 -1.891℃; SD=0.500 (하한 적용); 실측 58/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 12:31:00 / 시작 고정 기준 3창 복귀 11/13 12:31:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 -17.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/13 12:55:00–11/13 12:58:00 (끝 제외); 240117-Z-OP-Z3-0347 | 최대 절대 Z=7.03 (11/13 12:55:30, 창 중심); 1분 중앙값 0.651%, 과거 중심 80.152%, 차이 -79.501%p; SD=11.314; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 13:01:00 / 시작 고정 기준 3창 복귀 11/13 13:07:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +11.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/13 12:55:00–11/13 13:00:00 (끝 제외); 240117-Z-TC-Z3-0347 | 최대 절대 Z=6.84 (11/13 12:56:30, 창 중심); 1분 중앙값 867.397℃, 과거 중심 859.984℃, 차이 +7.413℃; SD=1.083; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 13:03:00 / 시작 고정 기준 3창 복귀 11/13 13:07:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +12.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 11/13 13:07:00–11/13 13:10:00 (끝 제외); 240117-Z-OP-Z6-0359 | 최대 절대 Z=4.09 (11/13 13:08:30, 창 중심); 1분 중앙값 80.001%, 과거 중심 70.485%, 차이 +9.517%p; SD=2.328; 실측 59/60개·과거 29/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 13:13:00 / 시작 고정 기준 3창 복귀 11/13 13:19:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +24.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/13 14:27:00–11/13 14:28:00 (끝 제외); 240117-Z-TC-Z3-0439 | 최대 절대 Z=3.38 (11/13 14:27:30, 창 중심); 1분 중앙값 857.552℃, 과거 중심 860.433℃, 차이 -2.881℃; SD=0.853; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/13 14:32:00 / 시작 고정 기준 3창 복귀 미확인 | 이동 Z는 복귀했지만 시작 고정 범위의 3창 복귀는 미확인. 기준선 적응과 지속 수준 변화를 구분해 추적; 정비 대조: WO-2024-01854 완료 대비 Z 최대창 중심 +103.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240117-Z-TC-Z3-0347:** OP-Z3 사건창 중앙값−직전10분 중앙값 -74.31 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240117-Z-OP-Z3-0347:** TC-Z3 사건창 중앙값−직전10분 중앙값 +7.18 (10/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/13 07:08:00–11/13 07:29:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 11/13 07:32:00–11/13 07:34:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/13 08:10:00–11/13 08:11:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/13 08:19:00–11/13 08:20:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 08:24:00–11/13 08:25:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 08:29:00–11/13 08:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 08:43:00–11/13 08:44:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 09:02:00–11/13 09:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 09:17:00–11/13 09:18:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/13 09:26:00–11/13 09:27:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/13 09:33:00–11/13 09:34:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 09:35:00–11/13 09:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 11/13 09:40:00–11/13 09:41:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 10:51:00–11/13 10:52:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 12:04:00–11/13 12:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/13 12:40:00–11/13 12:41:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/13 13:15:00–11/13 13:16:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/13 13:39:00–11/13 13:40:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/13 13:47:00–11/13 13:48:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/13 14:28:00–11/13 14:29:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/13 14:32:00–11/13 14:33:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 20개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

#### CP 원본 · 표시한 피크마다 남기는 근거

**±0.05는 기존 탐색 파라미터이며 공정 허용차·정상 한계의 검증 근거가 없다.** 원값을 삭제하거나 복귀만으로 정상 처리하지 않는다.

| 피크·ID | 원본 관측 | 동종 LOT 대조 | 복귀·정비 시간 관계 |
| --- | --- | --- | --- |
| 240117-CP-01 · 11/13 13:04:50 | -0.0829무차원 · 283점/290초 간격 | SPCD의 다른 LOT가 없어 동종 반복 여부를 판단할 수 없다. | 60개 연속 실제 1초 값 복귀 13:09:27; WO-2024-01854 완료 20분 50초 후에 CP 원본 피크 |

<details>
<summary><strong>CP 원값·전후 값·동반 신호·정비 연결 상세 펼치기</strong></summary>

#### CP 원본 이탈 · 단발 여부와 추적 근거

CP는 무차원이다. 원자료를 삭제·대체하지 않고, 피크와 전후 실측·복귀를 함께 남겼다. 1분 중앙값의 결과와 원본 순간 이탈은 별도로 읽는다.

**240117-CP-01 · 피크 11/13 13:04:50 · 복귀 펄스 후보**

- **관측:** 원값 0.367097, LOT 중앙값 0.450003, 편차 -0.082906. 탐색 경계 ±0.05 이상이다.
- **폭·반복:** 실측 283점, 첫·끝 실측 간격 290초, 기대 1초 격자 내 유효 비율 97.3%. 최근 30분 내 피크 1회(해당 사건 포함).
- **전후·복귀:** 직전 11/13 13:02:39=0.430515 / 직후 11/13 13:07:31=0.430221; 이후 60개 연속 실측 ±0.02 복귀 확인 11/13 13:09:27.
- **같은 1분의 결과:** 17신호 모두 절대 Z<3으로, 1분 집계는 정상 범위 참고였다. 그래도 원본 이탈은 별도 기록한다.
- **판단·추적:** CP 물리량 정의·채널 연결 미확인 (단위는 무차원) / 관측 공백 있음 / 다점 펄스: 단발 폭 기준 미확정. 사건 ID·원본 행 번호·전후 값을 보존; 같은 채널 재발 간격과 TC/OP 동시 반응, 센서·배선·수집·교정 로그 대조. 원본 DataFrame 행 번호 295755–296039에 연결했다(0부터 시작).

**시간 순서로 배제되는 설명:** CBM 완료 11/13 12:44:00 뒤 20.83분에 CP 피크가 발생했다. 이 CP 사건을 해당 CBM의 사전 경보로 볼 수 없다. 정비와 원인이 같거나 수리 후 정상이라는 뜻은 아니다.

**설비 판정:** 원본 복귀는 관측 사실이다. 정상 제어·계측 문제·설비 변화 중 원인은 현재 자료만으로 확정하지 않는다.

**CP 피크와 정비 시각 대조**

| 원본 사건 | 정비와의 시간 관계 | 동반 변화: TC ℃·OP %p·CP 무차원 |
| --- | --- | --- |
| 240117-CP-01 | 정비 대조: WO-2024-01854 완료 대비 CP 원본 피크 +20.83분 (근접 시각이며 인과 미확정) | TC-Z3 피크±30초 중앙값−전10~5분 중앙값 -7.543 (292/59점) / OP-Z3 피크±30초 중앙값−전10~5분 중앙값 +57.28 (292/59점) / CP-Z4M 피크±30초 중앙값−전10~5분 중앙값 +1.592e-12 (292/59점); 같은 시각 동반 관측이며 원인 확정 아님 |

</details>

In [ ]:
review_zscore_group_plot(240117, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| OP-Z7 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| TC-Z8 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |
| OP-Z8 | 445 | 403 | 0 | 42 | 22 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 403창, 최대 절대 Z=1.90 / OP-Z7: 유효 403창, 최대 절대 Z=2.70 / TC-Z8: 유효 403창, 최대 절대 Z=1.94 / OP-Z8: 유효 403창, 최대 절대 Z=1.60. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/13 07:08:00–11/13 07:29:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 / 과거 유효 최소 0창 <20 |
| 11/13 07:32:00–11/13 07:34:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/13 08:10:00–11/13 08:11:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/13 08:19:00–11/13 08:20:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 08:24:00–11/13 08:25:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 08:29:00–11/13 08:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 08:43:00–11/13 08:44:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 09:02:00–11/13 09:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 09:17:00–11/13 09:18:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/13 09:26:00–11/13 09:27:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/13 09:33:00–11/13 09:34:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 09:35:00–11/13 09:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 11/13 09:40:00–11/13 09:41:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 10:51:00–11/13 10:52:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 12:04:00–11/13 12:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/13 12:40:00–11/13 12:41:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/13 13:15:00–11/13 13:16:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/13 13:39:00–11/13 13:40:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/13 13:47:00–11/13 13:48:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/13 14:28:00–11/13 14:29:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/13 14:32:00–11/13 14:33:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 20개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240117)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **1건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 25708 | 22 | 원본 ±0.05 사건 1건 · 최대 절대 편차 0.082906 · 1초 단발 0건 | 원본 이탈 추적·설비 판정 보류 |
| CP-Z4M | 무차원 | 25707 | 22 | 원본 중앙값 1.14405e-10, 최저 3.68592e-11, 최고 1.15996e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 25707 | 22 | 원본 중앙값 328.113, 최저 318.7, 최고 341.759; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 25707 | 22 | 원본 중앙값 281.832, 최저 255.768, 최고 295.591; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 25708 | 22 | 원본 중앙값 278.104, 최저 256.993, 최고 289.67; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-01854 · CBM / 메모: 마모량 초과**

**실제 시각:** 감지 11/12 15:00:00 / 작업 11/13 08:54:00~11/13 12:44:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-01854 완료 20분 50초 후에 CP 원본 피크.

**실측으로 비교할 수 있는 범위:** 완료 경계 전후 5분의 중앙값 차이(후−전): **CP-Z4 -0.0002915무차원 / TC-Z3 -0.256℃ / OP-Z3 +6.932%p**. 피크 자체와 구분해서 읽는다.

**검토할 가설:** 정비 완료 뒤의 CP 사건이므로 해당 작업의 사전 경보로 볼 수 없다. 재가동·제어 복원 응답, 계측 변동, 남아 있는 설비 변화를 구분해 검토한다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 3건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 3건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01834 · TBM | 11/10 07:00:00 | 11/10 10:40:00 | 11/10 12:52:00 | 정기수리 부속작업 | 교체 상세 없음 | 작업은 LOT 관측 밖 |
| WO-2024-01859 · TBM | 11/13 07:13:00 | 11/13 07:40:00 | 11/13 09:35:00 | 계획 점검 수행 | 교체 상세 없음 | 작업과 LOT 관측 겹침 |
| WO-2024-01854 · CBM | 11/12 15:00:00 | 11/13 08:54:00 | 11/13 12:44:00 | 마모량 초과 | 교체 상세 없음 | 작업과 LOT 관측 겹침 |

**WO-2024-01834 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 11건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.082906 (무차원), 원본 ±0.05 사건 1건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** OP-Z3 8건·최대 절대 Z 6.60 / TC-Z3 3건·최대 절대 Z 3.78. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 11/10 07:00:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 11/10 10:40:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 11/10 12:52:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-01859 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 19건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240117-CP-01 · 11/13 13:04:50 · CP 편차 -0.082906 · 283점/290초 · 완료 대비 피크 +209.83분 · 감지는 피크 대비 -351.83분. 복귀 펄스 후보; 직전 11/13 13:02:39=0.430515 / 직후 11/13 13:07:31=0.430221; 이후 60개 연속 실측 ±0.02 복귀 확인 11/13 13:09:27.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.14 / OP-Z1 1건·최대 절대 Z 5.14 / OP-Z2 1건·최대 절대 Z 4.00 / OP-Z3 9건·최대 절대 Z 7.03 / OP-Z6 1건·최대 절대 Z 4.09 / TC-Z1 1건·최대 절대 Z 3.73 / TC-Z3 5건·최대 절대 Z 6.84. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 11/13 07:13:00: CP-Z4 후−전 중앙값 +0.00104 무차원 (290/290점) / CP-Z4M 후−전 중앙값 +9.218e-12 무차원 (289/290점) / TC-Z3 후−전 중앙값 +2.661 ℃ (290/290점) / OP-Z3 후−전 중앙값 +11.78 %p (290/290점)
  - 착수 11/13 07:40:00: CP-Z4 후−전 중앙값 +0.0008415 무차원 (292/292점) / CP-Z4M 후−전 중앙값 +4.231e-12 무차원 (292/292점) / TC-Z3 후−전 중앙값 +2.523 ℃ (292/292점) / OP-Z3 후−전 중앙값 -7.56 %p (292/292점)
  - 완료 11/13 09:35:00: CP-Z4 후−전 중앙값 -0.001224 무차원 (282/286점) / CP-Z4M 후−전 중앙값 -1.6e-13 무차원 (282/286점) / TC-Z3 후−전 중앙값 -2.797 ℃ (282/286점) / OP-Z3 후−전 중앙값 -24.11 %p (282/286점)
- **시간 순서로 제한되는 해석:** 완료 뒤의 CP 사건은 해당 작업의 사전 경보가 아니다. 재가동·설정 복원 응답, 계측 경로 변동, 남아 있는 설비 변화를 경쟁 가설로 검토한다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

**WO-2024-01854 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 1건, 센서별 Z 후보 19건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
  - 240117-CP-01 · 11/13 13:04:50 · CP 편차 -0.082906 · 283점/290초 · 완료 대비 피크 +20.83분 · 감지는 피크 대비 -1324.83분. 복귀 펄스 후보; 직전 11/13 13:02:39=0.430515 / 직후 11/13 13:07:31=0.430221; 이후 60개 연속 실측 ±0.02 복귀 확인 11/13 13:09:27.
- **동반 양상:** CLN-IN 1건·최대 절대 Z 3.14 / OP-Z1 1건·최대 절대 Z 5.14 / OP-Z2 1건·최대 절대 Z 4.00 / OP-Z3 9건·최대 절대 Z 7.03 / OP-Z6 1건·최대 절대 Z 4.09 / TC-Z1 1건·최대 절대 Z 3.73 / TC-Z3 5건·최대 절대 Z 6.84. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 11/12 15:00:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 11/13 08:54:00: CP-Z4 후−전 중앙값 +0.000436 무차원 (291/291점) / CP-Z4M 후−전 중앙값 +2.43e-13 무차원 (291/291점) / TC-Z3 후−전 중앙값 +1.267 ℃ (291/291점) / OP-Z3 후−전 중앙값 -12.76 %p (291/291점)
  - 완료 11/13 12:44:00: CP-Z4 후−전 중앙값 -0.0002915 무차원 (288/292점) / CP-Z4M 후−전 중앙값 +3.87e-13 무차원 (288/292점) / TC-Z3 후−전 중앙값 -0.256 ℃ (288/292점) / OP-Z3 후−전 중앙값 +6.932 %p (288/292점)
- **시간 순서로 제한되는 해석:** 완료 뒤의 CP 사건은 해당 작업의 사전 경보가 아니다. 재가동·설정 복원 응답, 계측 경로 변동, 남아 있는 설비 변화를 경쟁 가설로 검토한다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240117)

**240117 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240117)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240117)
lot_conclusion(240117)
lot_execution.append({'LOT':240117,'확대 구간':len(LOT_EVENTS[240117]),'강종 대조':240117 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240121"></a>
### 8.13 LOT 240121 · SPCE · 11/24

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240121)

#### LOT 240121 · 먼저 볼 변화

**지속·반복 후보 14건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — OP-Z2 · 11/24 17:58~18:05**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.51**, 과거 중심 대비 **-9.36%p**.  
**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 3건(연속 2~5개 1분창, 절대 편차 9.38~10.7%p)이 있다. 이 구간의 7창·9.36%p는 비교 관측의 상한을 넘는다(길이 7>5창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.22℃(10/7유효창). 같은 전후 기준의 OP-Z2 변화는 -10.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 18:13 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — TC-Z3 · 11/24 16:38~16:43**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **8.52**, 과거 중심 대비 **+6.9℃**.  
**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 5창·6.9℃는 비교 관측의 상한을 넘는다(절대 편차 6.9>6.61℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71%p(9/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.87℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 17:03 (이동 Z 복귀보다 17분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 11/24 11:01~11:02**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.32**, 과거 중심 대비 **-1.66℃**.  
**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 11:42 (이동 Z 복귀보다 33분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240121

- **관측 범위:** 11/24 07:31:00–11/24 18:22:12.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 545개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 14개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 41개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240121, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 4건 · 짧은 노랑 후보 2건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z2 · 11/24 17:58~18:05**

**관측:** |Z|≥3 연속 **7개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.51**, 과거 중심 대비 **-9.36%p**.  
**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 3건(연속 2~5개 1분창, 절대 편차 9.38~10.7%p)이 있다. 이 구간의 7창·9.36%p는 비교 관측의 상한을 넘는다(길이 7>5창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.22℃(10/7유효창). 같은 전후 기준의 OP-Z2 변화는 -10.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 18:13 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z1 · 11/24 17:57~18:02**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.04**, 과거 중심 대비 **-11.2%p**.  
**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~10.2%p)이 있다. 이 구간의 5창·11.2%p는 비교 관측의 상한을 넘는다(절대 편차 11.2>10.2%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.67℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -12.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 18:09 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — CLN-IN · 11/24 11:01~11:02**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.32**, 과거 중심 대비 **-1.66℃**.  
**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 11:42 (이동 Z 복귀보다 33분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 6건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240121-Z-CLN-IN-0210 · 11/24 11:01~11:02**

**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 11:42 (이동 Z 복귀보다 33분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 3건/611유효창 (100창당 0.49건) | 1.66~1.973 | 1~4창 | 66.003 |
| 240037 | 0건/568유효창 (100창당 0.00건) | — | — | 65.781 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 69.253 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-CLN-IN-0212 · 11/24 11:03~11:06**

**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 11:40 (이동 Z 복귀보다 31분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 3건/611유효창 (100창당 0.49건) | 1.66~1.973 | 1~4창 | 66.003 |
| 240037 | 0건/568유효창 (100창당 0.00건) | — | — | 65.781 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 69.253 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-CLN-IN-0326 · 11/24 12:57~13:01**

**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** 대응 채널의 유효 전후 관측이 부족하거나 정의가 없다. 국소 계측 변동과 공정 변화를 구분할 근거가 부족하다.  
**추적:** 시작 고정 대역 3창 복귀 13:28 (이동 Z 복귀보다 24분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 CLN-IN·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 3건/611유효창 (100창당 0.49건) | 1.66~1.973 | 1~4창 | 66.003 |
| 240037 | 0건/568유효창 (100창당 0.00건) | — | — | 65.781 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 69.253 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z1-0626 · 11/24 17:57~18:02**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~10.2%p)이 있다. 이 구간의 5창·11.2%p는 비교 관측의 상한을 넘는다(절대 편차 11.2>10.2%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.67℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -12.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 18:09 (이동 Z 복귀보다 4분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 1건/611유효창 (100창당 0.16건) | 11.22~11.22 | 5~5창 | 74.374 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 10.22~10.22 | 5~5창 | 73.753 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 10.16~10.16 | 5~5창 | 73.551 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z2-0627 · 11/24 17:58~18:05**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 3건(연속 2~5개 1분창, 절대 편차 9.38~10.7%p)이 있다. 이 구간의 7창·9.36%p는 비교 관측의 상한을 넘는다(길이 7>5창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.22℃(10/7유효창). 같은 전후 기준의 OP-Z2 변화는 -10.2%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 18:13 (이동 Z 복귀보다 5분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 1건/611유효창 (100창당 0.16건) | 9.364~9.364 | 7~7창 | 23.41 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 9.645~9.645 | 5~5창 | 22.003 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 9.377~10.7 | 2~3창 | 23.012 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z1-0628 · 11/24 17:59~18:01**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 2건(연속 2~3개 1분창, 절대 편차 1.74~1.92℃)이 있다. 이 구간의 2창·1.74℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -13%p(10/2유효창). 같은 전후 기준의 TC-Z1 변화는 +1.71℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 18:05 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 1건/611유효창 (100창당 0.16건) | 1.743~1.743 | 2~2창 | 100.01 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 1.743~1.743 | 2~2창 | 100 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 1.918~1.918 | 3~3창 | 99.941 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 652 | 611 | 8 | 41 | 21 | 20 | 0 |
| TC-Z1 | 652 | 611 | 2 | 41 | 21 | 20 | 0 |
| OP-Z1 | 652 | 611 | 5 | 41 | 21 | 20 | 0 |
| TC-Z2 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |
| OP-Z2 | 652 | 611 | 7 | 41 | 21 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z2: 유효 611창, 최대 절대 Z=2.56. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| CLN-IN · 11/24 11:01:00–11/24 11:02:00 (끝 제외); 240121-Z-CLN-IN-0210 | 최대 절대 Z=3.32 (11/24 11:01:30, 창 중심); 1분 중앙값 65.522℃, 과거 중심 67.182℃, 차이 -1.660℃; SD=0.500 (하한 적용); 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:09:00 / 시작 고정 기준 3창 복귀 11/24 11:42:00 | 고정 범위 복귀가 이동 Z 복귀보다 33분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1637.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 11/24 11:03:00–11/24 11:06:00 (끝 제외); 240121-Z-CLN-IN-0212 | 최대 절대 Z=3.80 (11/24 11:03:30, 창 중심); 1분 중앙값 65.202℃, 과거 중심 67.176℃, 차이 -1.973℃; SD=0.520; 실측 60/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:09:00 / 시작 고정 기준 3창 복귀 11/24 11:40:00 | 고정 범위 복귀가 이동 Z 복귀보다 31분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1635.50분 (근접 시각이며 인과 미확정) |
| CLN-IN · 11/24 12:57:00–11/24 13:01:00 (끝 제외); 240121-Z-CLN-IN-0326 | 최대 절대 Z=3.68 (11/24 12:58:30, 창 중심); 1분 중앙값 64.663℃, 과거 중심 66.505℃, 차이 -1.842℃; SD=0.500 (하한 적용); 실측 57/60개·과거 30/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:04:00 / 시작 고정 기준 3창 복귀 11/24 13:28:00 | 고정 범위 복귀가 이동 Z 복귀보다 24분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1520.50분 (근접 시각이며 인과 미확정) |
| OP-Z1 · 11/24 17:57:00–11/24 18:02:00 (끝 제외); 240121-Z-OP-Z1-0626 | 최대 절대 Z=5.04 (11/24 17:58:30, 창 중심); 1분 중앙값 61.863%, 과거 중심 73.083%, 차이 -11.219%p; SD=2.225; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:05:00 / 시작 고정 기준 3창 복귀 11/24 18:09:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1220.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 11/24 17:58:00–11/24 18:05:00 (끝 제외); 240121-Z-OP-Z2-0627 | 최대 절대 Z=4.51 (11/24 18:00:30, 창 중심); 1분 중앙값 13.140%, 과거 중심 22.504%, 차이 -9.364%p; SD=2.075; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 7창, 같은 방향 최장 7창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:08:00 / 시작 고정 기준 3창 복귀 11/24 18:13:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1218.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 11/24 17:59:00–11/24 18:01:00 (끝 제외); 240121-Z-TC-Z1-0628 | 최대 절대 Z=3.49 (11/24 18:00:30, 창 중심); 1분 중앙값 101.779℃, 과거 중심 100.036℃, 차이 +1.743℃; SD=0.500 (하한 적용); 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:04:00 / 시작 고정 기준 3창 복귀 11/24 18:05:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1218.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240121-Z-OP-Z2-0627:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.22 (10/7유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240121-Z-OP-Z1-0626:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.668 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/24 07:31:00–11/24 07:51:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 과거 유효 최소 0창 <20 |
| 11/24 09:56:00–11/24 09:57:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 10:14:00–11/24 10:15:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 10:27:00–11/24 10:28:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/24 10:35:00–11/24 10:36:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 10:44:00–11/24 10:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 11:02:00–11/24 11:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/24 13:04:00–11/24 13:05:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/24 13:12:00–11/24 13:13:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/24 13:20:00–11/24 13:21:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 13:24:00–11/24 13:25:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/24 13:28:00–11/24 13:29:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 |
| 11/24 13:36:00–11/24 13:37:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 14:02:00–11/24 14:03:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 14:10:00–11/24 14:12:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 14:19:00–11/24 14:20:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 14:55:00–11/24 14:56:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 15:08:00–11/24 15:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 11/24 16:29:00–11/24 16:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 11/24 16:50:00–11/24 16:51:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 11/24 18:22:00–11/24 18:23:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 13개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240121, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 10건 · 짧은 노랑 후보 27건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 주황 · 지속·반복 확인 — TC-Z3 · 11/24 16:38~16:43**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **8.52**, 과거 중심 대비 **+6.9℃**.  
**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 5창·6.9℃는 비교 관측의 상한을 넘는다(절대 편차 6.9>6.61℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71%p(9/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.87℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 17:03 (이동 Z 복귀보다 17분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z6 · 11/24 16:51~16:54**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 2회. 최대 |Z| **5.16**, 과거 중심 대비 **+10.6%p**.  
**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 2건(연속 1~4개 1분창, 절대 편차 8.59~9.38%p)이 있다. 이 구간의 3창·10.6%p는 비교 관측의 상한을 넘는다(절대 편차 10.6>9.38%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.61℃(9/3유효창). 같은 전후 기준의 OP-Z6 변화는 +9.65%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 17:00 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — TC-Z3 · 11/24 11:46~11:48**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 1회. 최대 |Z| **5.53**, 과거 중심 대비 **+3.77℃**.  
**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 2창·3.77℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -58.1%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +3.28℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 12:02 (이동 Z 복귀보다 11분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 37건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240121-Z-OP-Z3-0046 · 11/24 08:17~08:19**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·47.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.81℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -37.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:22. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0047 · 11/24 08:18~08:19**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 1창·2.66℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -47.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.38℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 08:28 (이동 Z 복귀보다 6분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0104 · 11/24 09:15~09:17**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·47%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.68℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -41.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:20. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0104 · 11/24 09:15~09:17**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 2창·3.21℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -41.7%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +2.68℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:27 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0108 · 11/24 09:19~09:21**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 2창·3.59℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.71%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -3.87℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 09:26 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0171 · 11/24 10:22~10:24**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·73.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.62℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -52.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:31 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0172 · 11/24 10:23~10:25**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 2창·3.93℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -59.3%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:35 (이동 Z 복귀보다 3분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0177 · 11/24 10:28~10:29**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 1창·5.09℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +15.6%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -5.58℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 10:33 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0204 · 11/24 10:55~10:58**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 3창·83.4%p는 비교 관측의 상한을 넘는다(절대 편차 83.4>76.6%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +4.54℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -70.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:01. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0205 · 11/24 10:56~10:58**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 2창·6.72℃는 비교 관측의 상한을 넘는다(절대 편차 6.72>6.61℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71.2%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +4.99℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:01. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0254 · 11/24 11:45~11:48**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 3창·61%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.82℃(10/3유효창). 같은 전후 기준의 OP-Z3 변화는 -54.9%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:51. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0255 · 11/24 11:46~11:48**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 2창·3.77℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -58.1%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +3.28℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 12:02 (이동 Z 복귀보다 11분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0260 · 11/24 11:51~11:53**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 2창·3.98℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +12.8%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 -4.13℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 11:56. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0324 · 11/24 12:55~12:57**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·17.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.03℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -16.8%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:00. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0328 · 11/24 12:59~13:00**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 1창·1.7℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +11.1%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -1.77℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:04 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0340 · 11/24 13:11~13:12**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 1창·42.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.3℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -40.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:16. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0340 · 11/24 13:11~13:12**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 1창·2.16℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -40.7%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 +2.3℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:20 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0344 · 11/24 13:15~13:16**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 1창·2.6℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +9.36%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.52℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 13:20 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0384 · 11/24 13:55~13:56**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 1창·1.51℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +13.7%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -1.38℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:00 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0407 · 11/24 14:18~14:19**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 1창·74.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.27℃(8/1유효창). 같은 전후 기준의 OP-Z3 변화는 -73.6%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:23. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0407 · 11/24 14:18~14:19**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 1창·3.69℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -73.6%p(8/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.27℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:32 (이동 Z 복귀보다 5분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0409 · 11/24 14:20~14:21**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 1창·3.88℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -8.86%p(7/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.52℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:31 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0411 · 11/24 14:22~14:24**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 2창·5.51℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +20.9%p(9/2유효창). 같은 전후 기준의 TC-Z3 변화는 -5.33℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 14:29 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z4-0458 · 11/24 15:09~15:11**

**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z4의 사건창 중앙값−직전 10분 중앙값은 +0.441℃(9/2유효창). 같은 전후 기준의 OP-Z4 변화는 -4.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:21 (이동 Z 복귀보다 7분 늦음). 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z4·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 1건/611유효창 (100창당 0.16건) | 7.415~7.415 | 2~2창 | 56.849 |
| 240037 | 0건/568유효창 (100창당 0.00건) | — | — | 53.811 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 51.74 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0464 · 11/24 15:15~15:17**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·51.6%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +2.13℃(9/2유효창). 같은 전후 기준의 OP-Z3 변화는 -41.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:20. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0465 · 11/24 15:16~15:17**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 1창·2.81℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -51.8%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 +3.1℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:27 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0468 · 11/24 15:19~15:22**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 3창·3.79℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +16.6%p(10/3유효창). 같은 전후 기준의 TC-Z3 변화는 -3.63℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:26 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0488 · 11/24 15:39~15:41**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·77.6%p는 비교 관측의 상한을 넘는다(절대 편차 77.6>76.6%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +3.39℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -65.2%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:45 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0489 · 11/24 15:40~15:42**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 2창·5.23℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -66.2%p(10/2유효창). 같은 전후 기준의 TC-Z3 변화는 +5.43℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:45. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0546 · 11/24 16:37~16:41**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 4창·69.5%p는 비교 관측의 상한을 넘는다(길이 4>3창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +5.05℃(9/4유효창). 같은 전후 기준의 OP-Z3 변화는 -75%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:54 (이동 Z 복귀보다 10분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0547 · 11/24 16:38~16:43**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 7건(연속 1~5개 1분창, 절대 편차 2.58~6.61℃)이 있다. 이 구간의 5창·6.9℃는 비교 관측의 상한을 넘는다(절대 편차 6.9>6.61℃). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 7건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 -71%p(9/5유효창). 같은 전후 기준의 TC-Z3 변화는 +9.87℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:03 (이동 Z 복귀보다 17분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 11건/611유효창 (100창당 1.80건) | 2.165~6.901 | 1~5창 | 860.99 |
| 240037 | 7건/568유효창 (100창당 1.23건) | 2.577~6.609 | 1~5창 | 860.87 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z6-0558 · 11/24 16:49~16:50**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 2건(연속 1~4개 1분창, 절대 편차 8.59~9.38%p)이 있다. 이 구간의 1창·6.86%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -0.934℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +5.83%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:01 (이동 Z 복귀보다 4분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 2건/611유효창 (100창당 0.33건) | 6.858~10.64 | 1~3창 | 72.826 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 9.377~9.377 | 4~4창 | 74.298 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 8.591~8.591 | 1~1창 | 71.427 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z6-0560 · 11/24 16:51~16:54**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 2건(연속 1~4개 1분창, 절대 편차 8.59~9.38%p)이 있다. 이 구간의 3창·10.6%p는 비교 관측의 상한을 넘는다(절대 편차 10.6>9.38%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.61℃(9/3유효창). 같은 전후 기준의 OP-Z6 변화는 +9.65%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 17:00 (이동 Z 복귀보다 3분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 2건/611유효창 (100창당 0.33건) | 6.858~10.64 | 1~3창 | 72.826 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 9.377~9.377 | 4~4창 | 74.298 |
| 240133 | 1건/82유효창 (100창당 1.22건) | 8.591~8.591 | 1~1창 | 71.427 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z6-0561 · 11/24 16:52~16:54**

**다른 LOT:** 동종 LOT 240037에도 같은 방향 후보 1건(연속 3~3개 1분창, 절대 편차 1.8~1.8℃)이 있다. 이 구간의 2창·1.77℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 1건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z6의 사건창 중앙값−직전 10분 중앙값은 +9.15%p(9/2유효창). 같은 전후 기준의 TC-Z6 변화는 -1.5℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:57. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 1건/611유효창 (100창당 0.16건) | 1.769~1.769 | 2~2창 | 860.41 |
| 240037 | 1건/567유효창 (100창당 0.18건) | 1.795~1.795 | 3~3창 | 860.44 |
| 240133 | 0건/81유효창 (100창당 0.00건) | — | — | 860.27 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0628 · 11/24 17:59~18:01**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 2창·18.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.666℃(10/2유효창). 같은 전후 기준의 OP-Z3 변화는 -13.5%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 18:09 (이동 Z 복귀보다 5분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-OP-Z3-0643 · 11/24 18:14~18:15**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 13건(연속 1~3개 1분창, 절대 편차 19.8~76.6%p)이 있다. 이 구간의 1창·28.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 13건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.39℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -27.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 18:18. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240121-Z-TC-Z3-0647 · 11/24 18:18~18:19**

**다른 LOT:** 동종 LOT 240037, 240133에도 같은 방향 후보 8건(연속 1~3개 1분창, 절대 편차 1.8~6.8℃)이 있다. 이 구간의 1창·2.34℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 8건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +18.9%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.67℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 18:22. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240133 | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 652 | 611 | 34 | 41 | 21 | 20 | 0 |
| OP-Z3 | 652 | 611 | 27 | 41 | 21 | 20 | 0 |
| TC-Z4 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |
| OP-Z4 | 652 | 611 | 2 | 41 | 21 | 20 | 0 |
| TC-Z5 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |
| OP-Z5 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |
| TC-Z6 | 652 | 611 | 2 | 41 | 21 | 20 | 0 |
| OP-Z6 | 652 | 611 | 4 | 41 | 21 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 611창, 최대 절대 Z=2.73 / TC-Z5: 유효 611창, 최대 절대 Z=1.87 / OP-Z5: 유효 611창, 최대 절대 Z=2.37. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z3 · 11/24 08:17:00–11/24 08:19:00 (끝 제외); 240121-Z-OP-Z3-0046 | 최대 절대 Z=5.31 (11/24 08:18:30, 창 중심); 1분 중앙값 41.425%, 과거 중심 89.217%, 차이 -47.792%p; SD=8.997; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 08:22:00 / 시작 고정 기준 3창 복귀 11/24 08:22:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1800.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 08:18:00–11/24 08:19:00 (끝 제외); 240121-Z-TC-Z3-0047 | 최대 절대 Z=4.04 (11/24 08:18:30, 창 중심); 1분 중앙값 863.631℃, 과거 중심 860.972℃, 차이 +2.659℃; SD=0.657; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 08:22:00 / 시작 고정 기준 3창 복귀 11/24 08:28:00 | 고정 범위 복귀가 이동 Z 복귀보다 6분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1800.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 09:15:00–11/24 09:17:00 (끝 제외); 240121-Z-OP-Z3-0104 | 최대 절대 Z=5.40 (11/24 09:15:30, 창 중심); 1분 중앙값 43.406%, 과거 중심 90.435%, 차이 -47.029%p; SD=8.709; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 09:20:00 / 시작 고정 기준 3창 복귀 11/24 09:20:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1743.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 09:15:00–11/24 09:17:00 (끝 제외); 240121-Z-TC-Z3-0104 | 최대 절대 Z=4.21 (11/24 09:16:30, 창 중심); 1분 중앙값 864.216℃, 과거 중심 861.007℃, 차이 +3.209℃; SD=0.763; 실측 56/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 09:24:00 / 시작 고정 기준 3창 복귀 11/24 09:27:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1742.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 09:19:00–11/24 09:21:00 (끝 제외); 240121-Z-TC-Z3-0108 | 최대 절대 Z=3.83 (11/24 09:19:30, 창 중심); 1분 중앙값 857.480℃, 과거 중심 861.066℃, 차이 -3.586℃; SD=0.937; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 09:24:00 / 시작 고정 기준 3창 복귀 11/24 09:26:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1739.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 10:22:00–11/24 10:24:00 (끝 제외); 240121-Z-OP-Z3-0171 | 최대 절대 Z=5.66 (11/24 10:23:30, 창 중심); 1분 중앙값 14.615%, 과거 중심 87.786%, 차이 -73.171%p; SD=12.924; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 10:27:00 / 시작 고정 기준 3창 복귀 11/24 10:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1675.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 10:23:00–11/24 10:25:00 (끝 제외); 240121-Z-TC-Z3-0172 | 최대 절대 Z=3.82 (11/24 10:23:30, 창 중심); 1분 중앙값 864.897℃, 과거 중심 860.963℃, 차이 +3.934℃; SD=1.029; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 10:32:00 / 시작 고정 기준 3창 복귀 11/24 10:35:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1675.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 10:28:00–11/24 10:29:00 (끝 제외); 240121-Z-TC-Z3-0177 | 최대 절대 Z=3.23 (11/24 10:28:30, 창 중심); 1분 중앙값 855.987℃, 과거 중심 861.081℃, 차이 -5.094℃; SD=1.579; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 10:32:00 / 시작 고정 기준 3창 복귀 11/24 10:33:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1670.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 10:55:00–11/24 10:58:00 (끝 제외); 240121-Z-OP-Z3-0204 | 최대 절대 Z=5.67 (11/24 10:56:30, 창 중심); 1분 중앙값 8.272%, 과거 중심 91.715%, 차이 -83.444%p; SD=14.715; 실측 60/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:01:00 / 시작 고정 기준 3창 복귀 11/24 11:01:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1642.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 10:56:00–11/24 10:58:00 (끝 제외); 240121-Z-TC-Z3-0205 | 최대 절대 Z=3.18 (11/24 10:57:30, 창 중심); 1분 중앙값 866.802℃, 과거 중심 860.085℃, 차이 +6.717℃; SD=2.114; 실측 58/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:01:00 / 시작 고정 기준 3창 복귀 11/24 11:01:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1641.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 11:45:00–11/24 11:48:00 (끝 제외); 240121-Z-OP-Z3-0254 | 최대 절대 Z=8.99 (11/24 11:46:30, 창 중심); 1분 중앙값 27.429%, 과거 중심 88.426%, 차이 -60.997%p; SD=6.786; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:51:00 / 시작 고정 기준 3창 복귀 11/24 11:51:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1592.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 11:46:00–11/24 11:48:00 (끝 제외); 240121-Z-TC-Z3-0255 | 최대 절대 Z=5.53 (11/24 11:47:30, 창 중심); 1분 중앙값 864.974℃, 과거 중심 861.201℃, 차이 +3.773℃; SD=0.682; 실측 56/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:51:00 / 시작 고정 기준 3창 복귀 11/24 12:02:00 | 고정 범위 복귀가 이동 Z 복귀보다 11분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1591.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 11:51:00–11/24 11:53:00 (끝 제외); 240121-Z-TC-Z3-0260 | 최대 절대 Z=3.50 (11/24 11:51:30, 창 중심); 1분 중앙값 857.193℃, 과거 중심 861.169℃, 차이 -3.976℃; SD=1.137; 실측 59/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 11:56:00 / 시작 고정 기준 3창 복귀 11/24 11:56:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1587.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 12:55:00–11/24 12:57:00 (끝 제외); 240121-Z-OP-Z3-0324 | 최대 절대 Z=3.28 (11/24 12:55:30, 창 중심); 1분 중앙값 70.866%, 과거 중심 88.039%, 차이 -17.172%p; SD=5.240; 실측 55/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:00:00 / 시작 고정 기준 3창 복귀 11/24 13:00:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1523.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 12:59:00–11/24 13:00:00 (끝 제외); 240121-Z-TC-Z3-0328 | 최대 절대 Z=3.40 (11/24 12:59:30, 창 중심); 1분 중앙값 859.494℃, 과거 중심 861.195℃, 차이 -1.701℃; SD=0.500 (하한 적용); 실측 55/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:03:00 / 시작 고정 기준 3창 복귀 11/24 13:04:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1519.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 13:11:00–11/24 13:12:00 (끝 제외); 240121-Z-OP-Z3-0340 | 최대 절대 Z=5.18 (11/24 13:11:30, 창 중심); 1분 중앙값 47.767%, 과거 중심 89.949%, 차이 -42.181%p; SD=8.147; 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:16:00 / 시작 고정 기준 3창 복귀 11/24 13:16:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1507.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 13:11:00–11/24 13:12:00 (끝 제외); 240121-Z-TC-Z3-0340 | 최대 절대 Z=3.05 (11/24 13:11:30, 창 중심); 1분 중앙값 863.073℃, 과거 중심 860.908℃, 차이 +2.165℃; SD=0.709; 실측 57/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:19:00 / 시작 고정 기준 3창 복귀 11/24 13:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1507.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 13:15:00–11/24 13:16:00 (끝 제외); 240121-Z-TC-Z3-0344 | 최대 절대 Z=3.06 (11/24 13:15:30, 창 중심); 1분 중앙값 858.412℃, 과거 중심 861.017℃, 차이 -2.605℃; SD=0.850; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:19:00 / 시작 고정 기준 3창 복귀 11/24 13:20:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1503.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 13:55:00–11/24 13:56:00 (끝 제외); 240121-Z-TC-Z3-0384 | 최대 절대 Z=3.03 (11/24 13:55:30, 창 중심); 1분 중앙값 859.732℃, 과거 중심 861.245℃, 차이 -1.513℃; SD=0.500 (하한 적용); 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 13:59:00 / 시작 고정 기준 3창 복귀 11/24 14:00:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1463.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 14:18:00–11/24 14:19:00 (끝 제외); 240121-Z-OP-Z3-0407 | 최대 절대 Z=7.18 (11/24 14:18:30, 창 중심); 1분 중앙값 8.825%, 과거 중심 83.607%, 차이 -74.782%p; SD=10.410; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 14:23:00 / 시작 고정 기준 3창 복귀 11/24 14:23:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1440.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 14:18:00–11/24 14:19:00 (끝 제외); 240121-Z-TC-Z3-0407 | 최대 절대 Z=4.82 (11/24 14:18:30, 창 중심); 1분 중앙값 864.938℃, 과거 중심 861.246℃, 차이 +3.692℃; SD=0.765; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 14:27:00 / 시작 고정 기준 3창 복귀 11/24 14:32:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1440.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 14:20:00–11/24 14:21:00 (끝 제외); 240121-Z-TC-Z3-0409 | 최대 절대 Z=3.65 (11/24 14:20:30, 창 중심); 1분 중앙값 865.335℃, 과거 중심 861.455℃, 차이 +3.880℃; SD=1.062; 실측 59/60개·과거 26/30창 | 지속·반복 후보: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 14:27:00 / 시작 고정 기준 3창 복귀 11/24 14:31:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1438.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 14:22:00–11/24 14:24:00 (끝 제외); 240121-Z-TC-Z3-0411 | 최대 절대 Z=3.60 (11/24 14:23:30, 창 중심); 1분 중앙값 855.630℃, 과거 중심 861.136℃, 차이 -5.506℃; SD=1.531; 실측 59/60개·과거 26/30창 | 지속·반복 후보: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 4회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 14:27:00 / 시작 고정 기준 3창 복귀 11/24 14:29:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1435.50분 (근접 시각이며 인과 미확정) |
| OP-Z4 · 11/24 15:09:00–11/24 15:11:00 (끝 제외); 240121-Z-OP-Z4-0458 | 최대 절대 Z=3.67 (11/24 15:09:30, 창 중심); 1분 중앙값 50.408%, 과거 중심 57.823%, 차이 -7.415%p; SD=2.022; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 15:14:00 / 시작 고정 기준 3창 복귀 11/24 15:21:00 | 고정 범위 복귀가 이동 Z 복귀보다 7분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1389.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 15:15:00–11/24 15:17:00 (끝 제외); 240121-Z-OP-Z3-0464 | 최대 절대 Z=5.79 (11/24 15:16:30, 창 중심); 1분 중앙값 33.598%, 과거 중심 85.237%, 차이 -51.639%p; SD=8.921; 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 15:20:00 / 시작 고정 기준 3창 복귀 11/24 15:20:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1382.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 15:16:00–11/24 15:17:00 (끝 제외); 240121-Z-TC-Z3-0465 | 최대 절대 Z=5.62 (11/24 15:16:30, 창 중심); 1분 중앙값 864.128℃, 과거 중심 861.317℃, 차이 +2.810℃; SD=0.500 (하한 적용); 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 15:25:00 / 시작 고정 기준 3창 복귀 11/24 15:27:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1382.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 15:19:00–11/24 15:22:00 (끝 제외); 240121-Z-TC-Z3-0468 | 최대 절대 Z=3.90 (11/24 15:20:30, 창 중심); 1분 중앙값 857.529℃, 과거 중심 861.317℃, 차이 -3.788℃; SD=0.972; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 15:25:00 / 시작 고정 기준 3창 복귀 11/24 15:26:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1378.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 15:39:00–11/24 15:41:00 (끝 제외); 240121-Z-OP-Z3-0488 | 최대 절대 Z=4.61 (11/24 15:40:30, 창 중심); 1분 중앙값 8.422%, 과거 중심 86.066%, 차이 -77.645%p; SD=16.855; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 15:44:00 / 시작 고정 기준 3창 복귀 11/24 15:45:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1358.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 15:40:00–11/24 15:42:00 (끝 제외); 240121-Z-TC-Z3-0489 | 최대 절대 Z=3.61 (11/24 15:40:30, 창 중심); 1분 중앙값 866.051℃, 과거 중심 860.816℃, 차이 +5.235℃; SD=1.452; 실측 60/60개·과거 30/30창 | 지속·반복 후보: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 3회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 15:45:00 / 시작 고정 기준 3창 복귀 11/24 15:45:00 | 짧아도 최근 30분에 3회 시작했다. 재발 간격과 동일 운전 조건을 추적; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1358.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 16:37:00–11/24 16:41:00 (끝 제외); 240121-Z-OP-Z3-0546 | 최대 절대 Z=7.76 (11/24 16:38:30, 창 중심); 1분 중앙값 17.767%, 과거 중심 87.263%, 차이 -69.495%p; SD=8.950; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 4창, 같은 방향 최장 4창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 16:44:00 / 시작 고정 기준 3창 복귀 11/24 16:54:00 | 고정 범위 복귀가 이동 Z 복귀보다 10분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1300.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 16:38:00–11/24 16:43:00 (끝 제외); 240121-Z-TC-Z3-0547 | 최대 절대 Z=8.52 (11/24 16:39:30, 창 중심); 1분 중앙값 868.117℃, 과거 중심 861.216℃, 차이 +6.901℃; SD=0.810; 실측 60/60개·과거 29/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 16:46:00 / 시작 고정 기준 3창 복귀 11/24 17:03:00 | 고정 범위 복귀가 이동 Z 복귀보다 17분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1299.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 11/24 16:49:00–11/24 16:50:00 (끝 제외); 240121-Z-OP-Z6-0558 | 최대 절대 Z=3.43 (11/24 16:49:30, 창 중심); 1분 중앙값 78.489%, 과거 중심 71.631%, 차이 +6.858%p; SD=2.000 (하한 적용); 실측 60/60개·과거 29/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 16:57:00 / 시작 고정 기준 3창 복귀 11/24 17:01:00 | 고정 범위 복귀가 이동 Z 복귀보다 4분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1289.50분 (근접 시각이며 인과 미확정) |
| OP-Z6 · 11/24 16:51:00–11/24 16:54:00 (끝 제외); 240121-Z-OP-Z6-0560 | 최대 절대 Z=5.16 (11/24 16:51:30, 창 중심); 1분 중앙값 82.422%, 과거 중심 71.783%, 차이 +10.639%p; SD=2.061; 실측 60/60개·과거 28/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 16:57:00 / 시작 고정 기준 3창 복귀 11/24 17:00:00 | 고정 범위 복귀가 이동 Z 복귀보다 3분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1287.50분 (근접 시각이며 인과 미확정) |
| TC-Z6 · 11/24 16:52:00–11/24 16:54:00 (끝 제외); 240121-Z-TC-Z6-0561 | 최대 절대 Z=3.54 (11/24 16:53:30, 창 중심); 1분 중앙값 858.466℃, 과거 중심 860.235℃, 차이 -1.769℃; SD=0.500 (하한 적용); 실측 60/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 16:57:00 / 시작 고정 기준 3창 복귀 11/24 16:57:00 | 2개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1285.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 17:59:00–11/24 18:01:00 (끝 제외); 240121-Z-OP-Z3-0628 | 최대 절대 Z=7.03 (11/24 18:00:30, 창 중심); 1분 중앙값 67.766%, 과거 중심 86.529%, 차이 -18.763%p; SD=2.667; 실측 60/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:04:00 / 시작 고정 기준 3창 복귀 11/24 18:09:00 | 고정 범위 복귀가 이동 Z 복귀보다 5분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1218.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 11/24 18:14:00–11/24 18:15:00 (끝 제외); 240121-Z-OP-Z3-0643 | 최대 절대 Z=3.99 (11/24 18:14:30, 창 중심); 1분 중앙값 58.478%, 과거 중심 86.718%, 차이 -28.240%p; SD=7.082; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:18:00 / 시작 고정 기준 3창 복귀 11/24 18:18:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1204.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 11/24 18:18:00–11/24 18:19:00 (끝 제외); 240121-Z-TC-Z3-0647 | 최대 절대 Z=3.14 (11/24 18:18:30, 창 중심); 1분 중앙값 858.803℃, 과거 중심 861.141℃, 차이 -2.338℃; SD=0.744; 실측 58/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:22:00 / 시작 고정 기준 3창 복귀 11/24 18:22:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1200.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240121-Z-TC-Z3-0547:** OP-Z3 사건창 중앙값−직전10분 중앙값 -70.95 (9/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240121-Z-OP-Z3-0546:** TC-Z3 사건창 중앙값−직전10분 중앙값 +5.048 (9/4유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/24 07:31:00–11/24 07:51:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 과거 유효 최소 0창 <20 |
| 11/24 09:56:00–11/24 09:57:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 10:14:00–11/24 10:15:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 10:27:00–11/24 10:28:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/24 10:35:00–11/24 10:36:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 10:44:00–11/24 10:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 11:02:00–11/24 11:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/24 13:04:00–11/24 13:05:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/24 13:12:00–11/24 13:13:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/24 13:20:00–11/24 13:21:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 13:24:00–11/24 13:25:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/24 13:28:00–11/24 13:29:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 |
| 11/24 13:36:00–11/24 13:37:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 14:02:00–11/24 14:03:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 14:10:00–11/24 14:12:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 14:19:00–11/24 14:20:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 14:55:00–11/24 14:56:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 15:08:00–11/24 15:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 11/24 16:29:00–11/24 16:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 11/24 16:50:00–11/24 16:51:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 11/24 18:22:00–11/24 18:23:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 13개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240121, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 노랑 · 짧은 국소 변화 — OP-Z7 · 11/24 18:10~18:11**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.01**, 과거 중심 대비 **-6.02%p**.  
**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**왜 이렇게 보는가:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 +0.225℃(10/1유효창). 같은 전후 기준의 OP-Z7 변화는 -4.09%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 18:14. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 1건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240121-Z-OP-Z7-0639 · 11/24 18:10~18:11**

**다른 LOT:** 같은 강종 LOT 240037, 240133의 같은 관측 형태에서는 같은 채널·방향의 후보가 없었다. 조건·관측 길이가 달라 이 LOT만의 이상으로 확정할 수 없다.  
**해석 가설:** TC-Z7의 사건창 중앙값−직전 10분 중앙값은 +0.225℃(10/1유효창). 같은 전후 기준의 OP-Z7 변화는 -4.09%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 18:14. 단독 변화의 원인을 열어 두고, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z7·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240121 (현재 LOT) | 1건/611유효창 (100창당 0.16건) | 6.024~6.024 | 1~1창 | 66.953 |
| 240037 | 0건/567유효창 (100창당 0.00건) | — | — | 66.183 |
| 240133 | 0건/82유효창 (100창당 0.00건) | — | — | 70.311 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |
| OP-Z7 | 652 | 611 | 1 | 41 | 21 | 20 | 0 |
| TC-Z8 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |
| OP-Z8 | 652 | 611 | 0 | 41 | 21 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 611창, 최대 절대 Z=0.66 / TC-Z8: 유효 611창, 최대 절대 Z=0.75 / OP-Z8: 유효 611창, 최대 절대 Z=1.71. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z7 · 11/24 18:10:00–11/24 18:11:00 (끝 제외); 240121-Z-OP-Z7-0639 | 최대 절대 Z=3.01 (11/24 18:10:30, 창 중심); 1분 중앙값 60.962%, 과거 중심 66.986%, 차이 -6.024%p; SD=2.000 (하한 적용); 실측 54/60개·과거 30/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 11/24 18:14:00 / 시작 고정 기준 3창 복귀 11/24 18:14:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-01922 감지 대비 Z 최대창 중심 -1208.50분 (근접 시각이며 인과 미확정) |

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 11/24 07:31:00–11/24 07:51:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 과거 유효 최소 0창 <20 |
| 11/24 09:56:00–11/24 09:57:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 10:14:00–11/24 10:15:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 10:27:00–11/24 10:28:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/24 10:35:00–11/24 10:36:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 10:44:00–11/24 10:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 11:02:00–11/24 11:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/24 13:04:00–11/24 13:05:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/24 13:12:00–11/24 13:13:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/24 13:20:00–11/24 13:21:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 13:24:00–11/24 13:25:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/24 13:28:00–11/24 13:29:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 |
| 11/24 13:36:00–11/24 13:37:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 14:02:00–11/24 14:03:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 14:10:00–11/24 14:12:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 14:19:00–11/24 14:20:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 14:55:00–11/24 14:56:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 15:08:00–11/24 15:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 11/24 16:29:00–11/24 16:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 11/24 16:50:00–11/24 16:51:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 11/24 18:22:00–11/24 18:23:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 13개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240121)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 38237 | 21 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.027302 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 38237 | 21 | 원본 중앙값 1.15255e-10, 최저 1.13765e-10, 최고 1.1685e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 38236 | 21 | 원본 중앙값 329.403, 최저 319.282, 최고 342.386; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 38237 | 21 | 원본 중앙값 285.922, 최저 271.25, 최고 298.759; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 38236 | 21 | 원본 중앙값 281.432, 최저 270.952, 최고 292.441; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-01922 · CBM / 메모: 1호기 재발, 설비 트립**

**실제 시각:** 감지 11/25 14:19:00 / 작업 11/30 04:19:00~11/30 07:12:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-01922 감지 20시간 18분 30초 전에 Z 최대창 중심.

**실측으로 비교할 수 있는 범위:** 감지 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 변화가 감지보다 앞섰다. 선행 신호라는 가설은 검토할 수 있지만, 반복 제어 응답·계측 변동도 남아 있다. 관측 공백만큼은 연결해 추정하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01922 · CBM | 11/25 14:19:00 | 11/30 04:19:00 | 11/30 07:12:00 | 1호기 재발, 설비 트립 | 교체 상세 없음 | 작업은 LOT 관측 밖 |

**WO-2024-01922 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 44건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.027302 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** CLN-IN 3건·최대 절대 Z 3.80 / OP-Z1 1건·최대 절대 Z 5.04 / OP-Z2 1건·최대 절대 Z 4.51 / OP-Z3 13건·최대 절대 Z 8.99 / OP-Z4 1건·최대 절대 Z 3.67 / OP-Z6 2건·최대 절대 Z 5.16 / OP-Z7 1건·최대 절대 Z 3.01 / TC-Z1 1건·최대 절대 Z 3.49 / TC-Z3 20건·최대 절대 Z 8.52 / TC-Z6 1건·최대 절대 Z 3.54. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 11/25 14:19:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 착수 11/30 04:19:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 11/30 07:12:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240121)

**240121 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240121)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240121)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240121(SPCE)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 333.308 - 대조 330.184 = **+3.124℃**, 다른 2 LOT과 대조.
- TC-Z8: 이 LOT 333.532 - 대조 330.528 = **+3.004℃**, 다른 2 LOT과 대조.
- OP-Z4: 이 LOT 56.849 - 대조 52.775 = **+4.073%p**, 다른 2 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 -0.024℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z7: 차이 -1.294%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 -0.032%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240037: 전환 제외 없음.
- 240121: 전환 제외 없음.
- 240133: 전환 제외 없음.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240121)
lot_conclusion(240121)
lot_execution.append({'LOT':240121,'확대 구간':len(LOT_EVENTS[240121]),'강종 대조':240121 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240133"></a>
### 8.14 LOT 240133 · SPCE · 12/28

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240133)

#### LOT 240133 · 먼저 볼 변화

**지속·반복 후보 3건 / CP 원본 후보 0건**. 관측 결과와 설비 이상 판정은 구분한다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 12/28 16:24~16:29**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.08**, 과거 중심 대비 **-10.2%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~11.2%p)이 있다. 이 구간의 5창·10.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.83℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 16:35 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 12/28 16:26~16:29**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.32**, 과거 중심 대비 **-9.38%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~7개 1분창, 절대 편차 9.36~9.64%p)이 있다. 이 구간의 3창·9.38%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.16℃(10/3유효창). 같은 전후 기준의 OP-Z2 변화는 -8.39%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 16:37 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z6 · 12/28 15:19~15:20**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.79**, 과거 중심 대비 **+8.59%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 6.86~10.6%p)이 있다. 이 구간의 1창·8.59%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.38℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +8.35%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 15:25 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

같은 강종의 다른 LOT와 대조했으며 SV·속도·소재 규격·실제 운전 모드 일치는 미확인이다. 비슷한 응답은 반복 공정/제어 가설의 근거로 남기고, 더 큰 폭·긴 지속·복귀 지연·새 존 동반은 우선 추적한다.

<details>
<summary><strong>LOT 전체 집계·그림 읽는 법 펼치기</strong></summary>

#### Z-score로 확인한 구간 · LOT 240133

- **관측 범위:** 12/28 14:57:00–12/28 16:44:24.
- **정상 범위 참고:** 17신호가 모두 평가 가능하고 모두 절대 Z<3인 창 71개. 설비 정상 판정은 아니다.
- **지속·반복 후보:** 3개 센서별 사건. 연속 3창 또는 최근 30분 내 시작 3회(해당 사건 포함) 기준이다.
- **관측·기준 부족:** 17신호 중 Z가 없는 창 27개. 회색 구간은 정상으로 채우지 않았다.
- **원본 스파이크 확인:** CP 원본 이탈 0건 중 확인된 1초 단발 0건, 복귀 펄스 후보 0건. 여러 점으로 이어진 펄스는 1초 스파이크로 부르지 않는다. CP는 무차원이다.

**아래 그림의 읽는 법:** 왼쪽은 1분 내 원본 최저~최고와 유효 중앙값, 오른쪽은 같은 센서의 Z다. 주황은 지속·반복 후보, 노랑은 짧은 국소 후보, 회색은 계산 보류다. 세정·건조 → 가열·균열 → 과시효의 17개 Z 채널과, 가열·균열 그림에서 함께 보는 CP 두 채널과, 별도의 CP·Z9·냉각·정비 시간축을 함께 확인한다.

</details>

In [ ]:
review_zscore_group_plot(240133, '01_세정건조')


#### 세정·건조 · 표시한 변화의 해석

**우선 볼 주황 후보 3건 · 짧은 노랑 후보 1건**. 센서별 관측 건수이며 고장 건수가 아니다.

**1. 주황 · 지속·반복 확인 — OP-Z1 · 12/28 16:24~16:29**

**관측:** |Z|≥3 연속 **5개 1분창**, 최근 30분 시작 1회. 최대 |Z| **5.08**, 과거 중심 대비 **-10.2%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~11.2%p)이 있다. 이 구간의 5창·10.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.83℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 16:35 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**2. 주황 · 다른 존의 변화 — OP-Z2 · 12/28 16:26~16:29**

**관측:** |Z|≥3 연속 **3개 1분창**, 최근 30분 시작 1회. 최대 |Z| **4.32**, 과거 중심 대비 **-9.38%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~7개 1분창, 절대 편차 9.36~9.64%p)이 있다. 이 구간의 3창·9.38%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.16℃(10/3유효창). 같은 전후 기준의 OP-Z2 변화는 -8.39%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 16:37 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

**3. 복귀 추가 확인 — OP-Z2 · 12/28 16:30~16:32**

**관측:** |Z|≥3 연속 2창, 최근 30분 시작 2회. 최대 |Z| **3.95**, 과거 중심 대비 **-10.7%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~7개 1분창, 절대 편차 9.36~9.64%p)이 있다. 이 구간의 2창·10.7%p는 비교 관측의 상한을 넘는다(절대 편차 10.7>9.64%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.898℃(9/2유효창). 같은 전후 기준의 OP-Z2 변화는 -8.57%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 16:36 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 4건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240133-Z-OP-Z1-0087 · 12/28 16:24~16:29**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~5개 1분창, 절대 편차 10.2~11.2%p)이 있다. 이 구간의 5창·10.2%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z1의 사건창 중앙값−직전 10분 중앙값은 +1.83℃(10/5유효창). 같은 전후 기준의 OP-Z1 변화는 -13.9%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 16:35 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 1건/82유효창 (100창당 1.22건) | 10.16~10.16 | 5~5창 | 73.551 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 10.22~10.22 | 5~5창 | 73.753 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 11.22~11.22 | 5~5창 | 74.374 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-OP-Z2-0089 · 12/28 16:26~16:29**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~7개 1분창, 절대 편차 9.36~9.64%p)이 있다. 이 구간의 3창·9.38%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +1.16℃(10/3유효창). 같은 전후 기준의 OP-Z2 변화는 -8.39%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 16:37 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 2건/82유효창 (100창당 2.44건) | 9.377~10.7 | 2~3창 | 23.012 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 9.645~9.645 | 5~5창 | 22.003 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 9.364~9.364 | 7~7창 | 23.41 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-TC-Z1-0089 · 12/28 16:26~16:29**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 2~2개 1분창, 절대 편차 1.74~1.74℃)이 있다. 이 구간의 3창·1.92℃는 비교 관측의 상한을 넘는다(절대 편차 1.92>1.74℃, 길이 3>2창). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z1의 사건창 중앙값−직전 10분 중앙값은 -14.7%p(10/3유효창). 같은 전후 기준의 TC-Z1 변화는 +1.98℃. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 16:33. 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z1·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 1건/82유효창 (100창당 1.22건) | 1.918~1.918 | 3~3창 | 99.941 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 1.743~1.743 | 2~2창 | 100 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 1.743~1.743 | 2~2창 | 100.01 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-OP-Z2-0093 · 12/28 16:30~16:32**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 2건(연속 5~7개 1분창, 절대 편차 9.36~9.64%p)이 있다. 이 구간의 2창·10.7%p는 비교 관측의 상한을 넘는다(절대 편차 10.7>9.64%p). 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 2건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z2의 사건창 중앙값−직전 10분 중앙값은 +0.898℃(9/2유효창). 같은 전후 기준의 OP-Z2 변화는 -8.57%p. 제어 방향은 공정·구동부에 따라 다르다. 대응 TC·OP의 움직임과 SV·모드를 대조해 공정 응답과 계측 변동을 구분한다.  
**추적:** 시작 고정 대역 3창 복귀 16:36 (이동 Z 복귀보다 1분 늦음). 비교 LOT의 폭/길이 상한을 넘어 우선 추적한다. 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z2·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 2건/82유효창 (100창당 2.44건) | 9.377~10.7 | 2~3창 | 23.012 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 9.645~9.645 | 5~5창 | 22.003 |
| 240121 | 1건/611유효창 (100창당 0.16건) | 9.364~9.364 | 7~7창 | 23.41 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| CLN-IN | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| TC-Z1 | 108 | 82 | 3 | 26 | 6 | 20 | 0 |
| OP-Z1 | 108 | 82 | 5 | 26 | 6 | 20 | 0 |
| TC-Z2 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| OP-Z2 | 108 | 82 | 5 | 26 | 6 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 세정·건조 · 구간별 판단 근거

**정상 범위 참고의 근거:** CLN-IN: 유효 82창, 최대 절대 Z=2.78 / TC-Z2: 유효 82창, 최대 절대 Z=2.32. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z1 · 12/28 16:24:00–12/28 16:29:00 (끝 제외); 240133-Z-OP-Z1-0087 | 최대 절대 Z=5.08 (12/28 16:25:30, 창 중심); 1분 중앙값 62.936%, 과거 중심 73.094%, 차이 -10.158%p; SD=2.000 (하한 적용); 실측 56/60개·과거 27/30창 | 지속·반복 후보: 연속 5창, 같은 방향 최장 5창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:33:00 / 시작 고정 기준 3창 복귀 12/28 16:35:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -28.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 12/28 16:26:00–12/28 16:29:00 (끝 제외); 240133-Z-OP-Z2-0089 | 최대 절대 Z=4.32 (12/28 16:28:30, 창 중심); 1분 중앙값 11.399%, 과거 중심 20.775%, 차이 -9.377%p; SD=2.169; 실측 59/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:35:00 / 시작 고정 기준 3창 복귀 12/28 16:37:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -25.50분 (근접 시각이며 인과 미확정) |
| TC-Z1 · 12/28 16:26:00–12/28 16:29:00 (끝 제외); 240133-Z-TC-Z1-0089 | 최대 절대 Z=3.84 (12/28 16:27:30, 창 중심); 1분 중앙값 101.962℃, 과거 중심 100.044℃, 차이 +1.918℃; SD=0.500 (하한 적용); 실측 60/60개·과거 27/30창 | 지속·반복 후보: 연속 3창, 같은 방향 최장 3창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:33:00 / 시작 고정 기준 3창 복귀 12/28 16:33:00 | 같은 방향의 초과가 최장 3창 이어졌다. 목표값 변경인지 지속 수준 변화인지 SV와 대조; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -26.50분 (근접 시각이며 인과 미확정) |
| OP-Z2 · 12/28 16:30:00–12/28 16:32:00 (끝 제외); 240133-Z-OP-Z2-0093 | 최대 절대 Z=3.95 (12/28 16:30:30, 창 중심); 1분 중앙값 9.930%, 과거 중심 20.628%, 차이 -10.698%p; SD=2.706; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 2창, 같은 방향 최장 2창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:35:00 / 시작 고정 기준 3창 복귀 12/28 16:36:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -23.50분 (근접 시각이며 인과 미확정) |

**동반 신호로 다시 볼 근거**

- **240133-Z-OP-Z1-0087:** TC-Z1 사건창 중앙값−직전10분 중앙값 +1.826 (10/5유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.
- **240133-Z-OP-Z2-0089:** TC-Z2 사건창 중앙값−직전10분 중앙값 +1.164 (10/3유효창); 대응 존의 동시 관측이며 인과 미확정. 동시 관측의 근거이며 원인 확정은 아니다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 12/28 14:57:00–12/28 15:18:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 50개 <54 / 과거 유효 최소 0창 <20 |
| 12/28 15:58:00–12/28 15:59:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 51개 <54 |
| 12/28 16:08:00–12/28 16:09:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 12/28 16:12:00–12/28 16:13:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 52개 <54 |
| 12/28 16:29:00–12/28 16:30:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 53개 <54 |
| 12/28 16:44:00–12/28 16:45:00 (끝 제외) | CLN-IN, TC-Z1, OP-Z1, TC-Z2, OP-Z2 | 현재창 실측 최소 25개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240133, '02_가열균열')


#### 가열·균열 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 5건**. 센서별 관측 건수이며 고장 건수가 아니다.

CP 위 2행과 Z3~Z6 아래 8행은 같은 시간축이다. 분홍 CP 피크 시각에서 TC·OP 반응을 대조한다.

**1. 복귀 추가 확인 — OP-Z6 · 12/28 15:19~15:20**

**관측:** |Z|≥3 연속 1창, 최근 30분 시작 1회. 최대 |Z| **3.79**, 과거 중심 대비 **+8.59%p**.  
**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 6.86~10.6%p)이 있다. 이 구간의 1창·8.59%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**왜 이렇게 보는가:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.38℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +8.35%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**어떻게 볼 것인가:** 시작 고정 대역 3창 복귀 15:25 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.


<details>
<summary><strong>표시한 모든 후보 5건의 LOT 대조·해석·추적 근거 펼치기</strong></summary>

**240133-Z-OP-Z6-0022 · 12/28 15:19~15:20**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 3건(연속 1~4개 1분창, 절대 편차 6.86~10.6%p)이 있다. 이 구간의 1창·8.59%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 3건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z6의 사건창 중앙값−직전 10분 중앙값은 -1.38℃(10/1유효창). 같은 전후 기준의 OP-Z6 변화는 +8.35%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 15:25 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z6·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 1건/82유효창 (100창당 1.22건) | 8.591~8.591 | 1~1창 | 71.427 |
| 240037 | 1건/568유효창 (100창당 0.18건) | 9.377~9.377 | 4~4창 | 74.298 |
| 240121 | 2건/611유효창 (100창당 0.33건) | 6.858~10.64 | 1~3창 | 72.826 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-OP-Z3-0072 · 12/28 16:09~16:10**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 24건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 1창·19.8%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 24건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +0.888℃(9/1유효창). 같은 전후 기준의 OP-Z3 변화는 -17.7%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:18 (이동 Z 복귀보다 2분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-OP-Z3-0089 · 12/28 16:26~16:27**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 24건(연속 1~4개 1분창, 절대 편차 17.2~83.4%p)이 있다. 이 구간의 1창·29%p는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 24건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** TC-Z3의 사건창 중앙값−직전 10분 중앙값은 +1.5℃(10/1유효창). 같은 전후 기준의 OP-Z3 변화는 -30%p. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:33. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 OP-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 2건/82유효창 (100창당 2.44건) | 19.83~29.05 | 1~1창 | 87.377 |
| 240037 | 11건/568유효창 (100창당 1.94건) | 22.53~76.59 | 1~3창 | 87.394 |
| 240121 | 13건/611유효창 (100창당 2.13건) | 17.17~83.44 | 1~4창 | 87.668 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-TC-Z3-0093 · 12/28 16:30~16:31**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 15건(연속 1~3개 1분창, 절대 편차 1.51~6.8℃)이 있다. 이 구간의 1창·2.11℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +17.5%p(9/1유효창). 같은 전후 기준의 TC-Z3 변화는 -2.06℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 16:35 (이동 Z 복귀보다 1분 늦음). 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

**240133-Z-TC-Z3-0104 · 12/28 16:41~16:42**

**다른 LOT:** 동종 LOT 240037, 240121에도 같은 방향 후보 15건(연속 1~3개 1분창, 절대 편차 1.51~6.8℃)이 있다. 이 구간의 1창·3.51℃는 비교한 변화 폭·길이의 상한 이내다. 이 중 같은 전후 기준의 TC·OP 변화 방향까지 일치한 후보는 15건이다. SV·속도·규격·모드가 없어 조건 일치는 미확인이다.  
**해석 가설:** OP-Z3의 사건창 중앙값−직전 10분 중앙값은 +22.2%p(10/1유효창). 같은 전후 기준의 TC-Z3 변화는 -3.79℃. OP가 해당 존의 가열 출력이고 자동 역동작 제어라면, 온도와 출력의 반대 변화는 편차를 줄이려는 보상 응답과 양립한다. 시간창 동반 관측만으로 선후·원인은 확정하지 않는다.  
**추적:** 시작 고정 대역 3창 복귀 미확인. 다른 LOT의 반복 응답 가설을 남기되, 조건 일치 후 폭·길이·빈도·복귀 지연이 더 커지는지 추적한다. 원자료 보존·설비 판정 보류.

강종 SPCE·채널 TC-Z3·동일 편차 방향·사후 관측 형태 안정 참고로 제한했다. 행은 같은 운전 조건이 확인된 정상 표본이 아니다. Z의 빈도 분모는 계산 가능한 1분창이다. 초기·현재/과거 관측 부족을 제외한다.

| LOT | 같은 방향 후보/비교 노출 | 절대 편차 범위 | 후보 길이 | 참고 중심 수준 |
| --- | --- | --- | --- | --- |
| 240133 (현재 LOT) | 2건/82유효창 (100창당 2.44건) | 2.106~3.515 | 1~1창 | 860.24 |
| 240037 | 6건/568유효창 (100창당 1.06건) | 1.803~6.796 | 1~3창 | 860.87 |
| 240121 | 9건/611유효창 (100창당 1.47건) | 1.513~5.506 | 1~3창 | 860.99 |

기존 |Z|≥3 후보만 대조했다. 기준을 넘지 않은 원본 변화까지 없었다는 뜻은 아니다. 다른 LOT에 흔해도 공통 제어 문제일 수 있다. 드물어도 정상 조업 전환일 수 있다. 과거·미래 LOT를 포함한 사후 EDA이며 온라인 판정에 사용하지 않는다.

</details>

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z3 | 108 | 82 | 2 | 26 | 6 | 20 | 0 |
| OP-Z3 | 108 | 82 | 2 | 26 | 6 | 20 | 0 |
| TC-Z4 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| OP-Z4 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| TC-Z5 | 108 | 81 | 0 | 27 | 7 | 20 | 0 |
| OP-Z5 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| TC-Z6 | 108 | 81 | 0 | 27 | 7 | 20 | 0 |
| OP-Z6 | 108 | 82 | 1 | 26 | 6 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 가열·균열 · 구간별 판단 근거

**같은 그림의 CP 대조:** 위 2행은 CP-Z4·CP-Z4M 1초 원본과 LOT 중앙값 편차, 이어지는 8행은 Z3~Z6의 TC·OP다. CP-Z4 원본 이탈 0건의 피크 시각을 분홍 세로선으로 모든 행에 맞췄다. 같은 시각의 반응을 확인하기 위한 표시이며 인과·이상 판정이 아니다. CP는 무차원이며 새 Z를 적용하지 않았다.

**정상 범위 참고의 근거:** TC-Z4: 유효 82창, 최대 절대 Z=0.80 / OP-Z4: 유효 82창, 최대 절대 Z=1.55 / TC-Z5: 유효 81창, 최대 절대 Z=1.57 / OP-Z5: 유효 82창, 최대 절대 Z=1.56 / TC-Z6: 유효 81창, 최대 절대 Z=2.69. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

| 구간·센서 | 관측 근거 | 분류한 이유 | 복귀 확인 | 해석·다음 확인 |
| --- | --- | --- | --- | --- |
| OP-Z6 · 12/28 15:19:00–12/28 15:20:00 (끝 제외); 240133-Z-OP-Z6-0022 | 최대 절대 Z=3.79 (12/28 15:19:30, 창 중심); 1분 중앙값 81.700%, 과거 중심 73.109%, 차이 +8.591%p; SD=2.269; 실측 58/60개·과거 21/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 15:23:00 / 시작 고정 기준 3창 복귀 12/28 15:25:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-02105 감지 대비 Z 최대창 중심 -34.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 12/28 16:09:00–12/28 16:10:00 (끝 제외); 240133-Z-OP-Z3-0072 | 최대 절대 Z=3.67 (12/28 16:09:30, 창 중심); 1분 중앙값 67.547%, 과거 중심 87.377%, 차이 -19.829%p; SD=5.401; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:16:00 / 시작 고정 기준 3창 복귀 12/28 16:18:00 | 고정 범위 복귀가 이동 Z 복귀보다 2분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-02105 감지 대비 Z 최대창 중심 +15.50분 (근접 시각이며 인과 미확정) |
| OP-Z3 · 12/28 16:26:00–12/28 16:27:00 (끝 제외); 240133-Z-OP-Z3-0089 | 최대 절대 Z=3.07 (12/28 16:26:30, 창 중심); 1분 중앙값 55.631%, 과거 중심 84.677%, 차이 -29.047%p; SD=9.467; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:33:00 / 시작 고정 기준 3창 복귀 12/28 16:33:00 | 1개 창의 짧은 초과 후 고정 범위 3창 복귀 확인. 지속·반복 기준 미충족; 원본 폭과 전후 값을 확인; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -27.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 12/28 16:30:00–12/28 16:31:00 (끝 제외); 240133-Z-TC-Z3-0093 | 최대 절대 Z=3.21 (12/28 16:30:30, 창 중심); 1분 중앙값 858.465℃, 과거 중심 860.571℃, 차이 -2.106℃; SD=0.655; 실측 59/60개·과거 27/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 1회(해당 사건 포함) | 이동 Z 3창 복귀 12/28 16:34:00 / 시작 고정 기준 3창 복귀 12/28 16:35:00 | 고정 범위 복귀가 이동 Z 복귀보다 1분 늦게 확인됐다. Z 감소만으로 복귀 시각을 앞당기지 않음; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -23.50분 (근접 시각이며 인과 미확정) |
| TC-Z3 · 12/28 16:41:00–12/28 16:42:00 (끝 제외); 240133-Z-TC-Z3-0104 | 최대 절대 Z=3.23 (12/28 16:41:30, 창 중심); 1분 중앙값 857.022℃, 과거 중심 860.537℃, 차이 -3.515℃; SD=1.087; 실측 59/60개·과거 28/30창 | 짧은 국소 후보·보류: 연속 1창, 같은 방향 최장 1창, 최근 30분 내 시작 2회(해당 사건 포함) | 이동 Z 3창 복귀 미확인 / 시작 고정 기준 3창 복귀 미확인 | 짧은 국소 변화지만 고정 범위 복귀는 미확인. 원본의 폭·전후 값을 보존하며 1초 스파이크로 단정하지 않음; 정비 대조: WO-2024-02105 착수 대비 Z 최대창 중심 -12.50분 (근접 시각이며 인과 미확정) |

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 12/28 14:57:00–12/28 15:19:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 50개 <54 / 과거 유효 최소 0창 <20 |
| 12/28 15:58:00–12/28 15:59:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 51개 <54 |
| 12/28 16:08:00–12/28 16:09:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 12/28 16:12:00–12/28 16:13:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 52개 <54 |
| 12/28 16:29:00–12/28 16:30:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 53개 <54 |
| 12/28 16:44:00–12/28 16:45:00 (끝 제외) | TC-Z3, OP-Z3, TC-Z4, OP-Z4, TC-Z5, OP-Z5, TC-Z6, OP-Z6 | 현재창 실측 최소 25개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_zscore_group_plot(240133, '03_과시효')


#### 과시효 · 표시한 변화의 해석

**우선 볼 주황 후보 0건 · 짧은 노랑 후보 0건**. 센서별 관측 건수이며 고장 건수가 아니다.

계산 가능한 창에 |Z|≥3 후보는 없다. 실측·과거 부족 구간과 설비 판정은 별도로 남긴다.

<details>
<summary><strong>창 수의 의미·계산 보류 이유 펼치기</strong></summary>

각 창은 1분이다. 유효는 현재 실측 ≥54개와 이전 30분의 유효창 ≥20개를 모두 충족한 창이다. 초과는 유효창의 부분집합이므로 초과+유효+보류를 더하지 않는다. 보류 사유는 현재 실측 부족을 먼저 세고, 이를 통과한 창에서 과거 부족을 따로 세어 중복을 없앴다. 54/60(90%)·20/30·SD 하한은 자료 충실도/과도한 Z 방지용 분석 설정이며 설비 규격이 아니다.

| 센서 | 전체 1분창 | 계산 가능 | 그중 \|Z\|≥3 | 보류 합계 | 현재 <54점 | 현재 충족·과거 <20창 | 그 외 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| OP-Z7 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| TC-Z8 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |
| OP-Z8 | 108 | 82 | 0 | 26 | 6 | 20 | 0 |

</details>

<details>
<summary><strong>원값·전체 통계 근거·회색 구간 상세 펼치기</strong></summary>

#### 과시효 · 구간별 판단 근거

**정상 범위 참고의 근거:** TC-Z7: 유효 82창, 최대 절대 Z=0.43 / OP-Z7: 유효 82창, 최대 절대 Z=1.95 / TC-Z8: 유효 82창, 최대 절대 Z=0.55 / OP-Z8: 유효 82창, 최대 절대 Z=1.26. 계산 가능한 창에서는 기준을 넘지 않았다. 원자료의 모든 순간값이나 설비 정상까지 확인한 결과는 아니다.

이 공정의 계산 가능한 창에서는 절대 Z≥3 사건이 검출되지 않았다. 회색 구간의 미검출은 판단 근거로 쓰지 않는다.

**판정 보류의 근거 — 그림의 회색 구간**

| 구간 | 해당 채널 | 점수를 계산하지 않은 이유 |
| --- | --- | --- |
| 12/28 14:57:00–12/28 15:18:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 50개 <54 / 과거 유효 최소 0창 <20 |
| 12/28 15:58:00–12/28 15:59:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 51개 <54 |
| 12/28 16:08:00–12/28 16:09:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 12/28 16:12:00–12/28 16:13:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 52개 <54 |
| 12/28 16:29:00–12/28 16:30:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 53개 <54 |
| 12/28 16:44:00–12/28 16:45:00 (끝 제외) | TC-Z7, OP-Z7, TC-Z8, OP-Z8 | 현재창 실측 최소 25개 <54 |

**현재 결론:** 복귀 시각은 연속 3개 유효창이 확인된 종료 시각이다. 시작 고정 기준은 해당 사건 시작 직전의 중심±3SD를 유지한 비교다. 위 표는 통계적으로 확인한 변화와 검토 이유다. 실제 설비 이상 여부는 SV·실제 운전 모드·관련 점검 결과로 별도 판정한다. 이동 Z 감소만으로 정상 복귀를 판단하지 않는다. 연속 1분창 수는 원자료가 그 시간 내내 이탈했다는 뜻이 아니다.

</details>

In [ ]:
review_cp_cooling_plot(240133)


#### CP·냉각 보조 대조 · 확인 범위

CP 원본 이탈 **0건**을 보존했고, CP-Z4M·Z9·냉각에는 새 Z나 정상 경계를 추가하지 않았다. CP와 Z3~Z6의 동반 반응은 앞의 가열·균열 그림에서, 실제 날짜의 정비 관계는 이 그림 위 시간축에서 확인한다.

<details>
<summary><strong>5개 채널의 원본 범위·실측 수·단위와 해석 범위 펼치기</strong></summary>

#### CP·Z9·냉각 · 생략하지 않은 채널의 근거

| 채널 | 단위 | 실측 수 | 54개 미만 1분창 | 확인한 사실 | 해석 범위 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 무차원 | 6218 | 6 | 원본 ±0.05 사건 0건 · 최대 절대 편차 0.013593 · 1초 단발 0건 | 원본 탐색 기준 내 참고·설비 판정 보류 |
| CP-Z4M | 무차원 | 6216 | 7 | 원본 중앙값 1.15249e-10, 최저 1.14019e-10, 최고 1.17002e-10; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-Z9 | ℃ | 6218 | 6 | 원본 중앙값 325.011, 최저 319.281, 최고 339.038; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT1 | ℃ | 6218 | 6 | 원본 중앙값 287.807, 최저 268.878, 최고 298.63; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |
| TC-OUT2 | ℃ | 6218 | 6 | 원본 중앙값 278.992, 최저 272.818, 최고 290.199; 별도 Z·이탈 경계 미적용 | 관측 참고·판정 보류 |

CP 두 채널은 원본 1초 격자로 그렸다. 위 정비 시간축에서 파란 구간은 LOT 관측창, 보라 역삼각형은 감지, 보라 굵은 구간은 착수~완료다. CP-Z4M은 읽기 위해 원값×10¹⁰으로 표시했고 단위는 무차원이다. CP-Z4는 LOT 중앙값 대비 ±0.02를 묶음·복귀 참고, ±0.05를 원본 이탈 탐색 경계로 사용했다. CP-Z4M·Z9·냉각에는 새 Z나 임의의 정상 경계를 만들지 않았다. 원본 최저~최고 띠와 복귀는 설비 정상의 증거가 아니다.

</details>

#### 정비 대조 · 사실과 가설을 나눠 보기

**먼저 볼 기록: WO-2024-02105 · CBM / 메모: 마모량 초과**

**실제 시각:** 감지 12/28 15:54:00 / 작업 12/28 16:54:00~12/28 23:33:00. 감지와 수리 시점을 구분한다.

**시간 관계:** WO-2024-02105 착수 28분 30초 전에 Z 최대창 중심.

**실측으로 비교할 수 있는 범위:** 착수 전후 5분 CP 실측 **0/0점**. 양쪽 190점 기준을 못 채워 **정비 효과 비교 보류**.

**검토할 가설:** 작업 또는 제어 조건 변경과 동반됐을 가능성을 검토한다. 같은 시각이라는 이유만으로 정비가 원인이라고 판단하지 않는다.

**다음 확인:** 당시 SV·실제 운전 모드, 센서/수집/교정 로그, WO 대상 부품과 해당 채널의 연결을 대조한다.

같은 설비의 관련 WO는 총 1건이다. 위 기록은 작업과 관측의 겹침, 대표 사건과의 시간 거리를 기준으로 먼저 제시했다.

<details>
<summary><strong>정비 1건의 전체 시각·메모·부품·전후 관측·가설 펼치기</strong></summary>

#### 정비이력 대조 · 관측 사실과 검토할 가설

같은 설비 P1-CR1-CAL01만 대조했다. LOT 관측창 전후 72시간 안에 감지·착수·완료 중 하나가 있거나 작업이 겹치는 기록을 제시한다. 72시간은 탐색 범위이며 고장 전조 임계값이 아니다. 그림의 보라 점선=감지, 파선=착수, 일점쇄선=완료, 옅은 보라 음영=작업 교집합이다.

| 정비 기록 | 감지 | 착수 | 완료 | REMARK | 부품 기록 | 관측 조건 |
| --- | --- | --- | --- | --- | --- | --- |
| WO-2024-02105 · CBM | 12/28 15:54:00 | 12/28 16:54:00 | 12/28 23:33:00 | 마모량 초과 | INS-006 가스 분석계 셀 ×3 (교체 사유: 예방) | 작업은 LOT 관측 밖 |

**WO-2024-02105 · 사실 → 시간 관계 → 가설 → 확인**

- **관측 사실:** 해당 LOT에서 이 정비의 감지·착수·완료 중 하나와 72시간 이내인 CP 원본 이탈 0건, 센서별 Z 후보 9건. 중첩 센서·시간창을 합친 고장 건수가 아니다.
- **CP 대조:** LOT 전체의 최대 절대 편차 0.013593 (무차원), 원본 ±0.05 사건 0건. Z 변화와 CP 이탈을 같은 결과로 취급하지 않는다.
- **동반 양상:** OP-Z1 1건·최대 절대 Z 5.08 / OP-Z2 2건·최대 절대 Z 4.32 / OP-Z3 2건·최대 절대 Z 3.67 / OP-Z6 1건·최대 절대 Z 3.79 / TC-Z1 1건·최대 절대 Z 3.84 / TC-Z3 2건·최대 절대 Z 3.23. 상세 시각·연속·복귀는 위 구간 근거표에서 확인한다.
- **정비 경계의 실제 전후 관측:**
  - 감지 12/28 15:54:00: CP-Z4 후−전 중앙값 -0.000342 무차원 (291/283점) / CP-Z4M 후−전 중앙값 -1.19e-13 무차원 (291/283점) / TC-Z3 후−전 중앙값 +0.482 ℃ (291/283점) / OP-Z3 후−전 중앙값 -5.361 %p (291/283점)
  - 착수 12/28 16:54:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
  - 완료 12/28 23:33:00: 전/후 5분 CP 실측 0/0점; 양쪽 190점 기준 미충족으로 효과 비교 보류
- **검토할 가설:** 운전·제어 조건 변화, 계측·수집 변동, 정비 대상의 변화와 관련된 반응을 구분한다. 가까운 시각이나 작업 중 동반 관측만으로 한 가설을 채택하지 않는다.
- **가설을 가를 자료:** 당시 SV·실제 모드·재가동/설정 변경 로그, 센서·배선·수집·교정 로그, WO 대상 부품과 채널의 연결, 같은 조건의 후속 관측을 대조한다. 전후 실측 부족 또는 부품 연결 미확인 항목이 남으면 원인·정비 효과 판정은 보류한다.

</details>

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240133)

**240133 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240133)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240133)
lot_conclusion(240133)
lot_execution.append({'LOT':240133,'확대 구간':len(LOT_EVENTS[240133]),'강종 대조':240133 in LEVEL_TARGETS,'WO 전중후':'완료'})

### 8.15 월별 관측창·44 WO 전체 기록

In [ ]:
from matplotlib.patches import Rectangle

months = pd.Index(range(1,13),name="MONTH")
monthly = pd.DataFrame(index=months)
monthly["온도 LOT[개]"] = lots.groupby(lots.START.dt.month).size().reindex(months,fill_value=0)
monthly["온도 관측창 합계[시간]"] = lots.groupby(lots.START.dt.month).MINUTES.sum().reindex(months,fill_value=0)/60
monthly["CAL01 작업[WO]"] = cal_work.groupby(cal_work.STRT_DT.dt.month).size().reindex(months,fill_value=0)
monthly["CR1 강종변경[건]"] = grade_evt.groupby(grade_evt.EVT_DT.dt.month).size().reindex(months,fill_value=0)
for kind in ["BM","CBM","TBM"]:
    work = cal_work.loc[cal_work.MNT_TYPE.eq(kind)]
    monthly[f"{kind} 작업[WO]"] = work.groupby(work.STRT_DT.dt.month).size().reindex(months,fill_value=0)

# 작업 구간을 LOT 관측창으로 자르고, 여러 WO가 동시에 진행된 시간은 한 번만 센다.
def observed_work_minutes(start,end,work):
    intervals = sorted((max(start,t.STRT_DT),min(end,t.END_DT)) for t in work.itertuples())
    total = 0.0
    current_end = start
    for left,right in intervals:
        left = max(left,current_end)
        if right > left:
            total += (right-left).total_seconds()/60
            current_end = right
    return total

links = []
for r in lots.itertuples():
    work = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    links.append({"LOT":r.LOT_NO,"MONTH":r.START.month,
                  "온도·작업 겹친 LOT[개]":int(len(work)>0),"온도·작업 겹친 WO[개]":len(work),
                  "온도·작업 겹침[분]":observed_work_minutes(r.START,r.END,work)})
monthly_links = pd.DataFrame(links)
for column in monthly_links.columns[2:]:
    monthly[column] = monthly_links.groupby("MONTH")[column].sum().reindex(months,fill_value=0)

# 월별 건수 대신 실제 관측창과 정비 REMARK를 보여준다.
# START–END는 제공된 기록의 범위이며, 실제 가동시간을 뜻하지 않는다.
GRADE_COLOR_MONTHLY = report_viz.GRADE_COLORS.copy()


def monthly_clock_range(start, end, seconds=False):
    clock = "%H:%M:%S" if seconds else "%H:%M"
    left = start.strftime("%m/%d " + clock)
    right = end.strftime(clock if start.date() == end.date() else "%m/%d " + clock)
    return f"{left}–{right}"


def monthly_part_records(wo_no):
    records = cal_parts.loc[cal_parts.WO_NO.eq(wo_no)]
    return [f"{r.PART_NM} [{r.PART_CD}] ×{r.QTY:g} ({r.CHG_RSN})"
            for r in records.itertuples()]


# 같은 월이라는 이유로 연결하지 않는다. 작업 구간과 LOT 관측창을 직접 대조한다.
work_observation_links = []
detection_observation_links = []
for task in cal_work.itertuples():
    for lot in lots.itertuples():
        left, right = max(task.STRT_DT, lot.START), min(task.END_DT, lot.END)
        if right > left:
            work_observation_links.append({
                "WO_NO":task.WO_NO, "LOT":int(lot.LOT_NO), "TYPE":task.MNT_TYPE,
                "START":left, "END":right, "MINUTES":(right-left).total_seconds()/60})
        if lot.START <= task.DETC_DT <= lot.END:
            detection_observation_links.append({
                "WO_NO":task.WO_NO, "LOT":int(lot.LOT_NO), "DETC_DT":task.DETC_DT,
                "WORK_START":task.STRT_DT})
work_observation_links = pd.DataFrame(work_observation_links)
detection_observation_links = pd.DataFrame(detection_observation_links)

# 각 월의 내용 길이에 맞춰 높이를 잡는다. 숫자 집계와 해석 문장은 그림에서 뺀다.
month_cards = []
for month in months:
    records = []
    month_lots = lots.loc[lots.START.dt.month.eq(month)]
    month_work = cal_work.loc[cal_work.STRT_DT.dt.month.eq(month)]
    month_plans = plans.loc[plans.STRT_DT.dt.month.eq(month)]
    for plan in month_plans.itertuples():
        records.append(("plan", f"계획수리 {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}", "#087f9c"))
    records.append(("section", "온도 · 제공된 LOT 관측창", "#17344d"))
    for lot in month_lots.itertuples():
        records.append(("lot", f"{lot.LOT_NO} {lot.GRADE} · {monthly_clock_range(lot.START,lot.END)}",
                        GRADE_COLOR_MONTHLY[lot.GRADE]))
    if month_lots.empty:
        records.append(("empty", "제공된 온도 기록 없음", "#6c7c89"))
    records.append(("space", "", ""))
    records.append(("section", "정비 · CAL01 작업 기록", "#17344d"))
    for task in month_work.itertuples():
        remark = str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재"
        short_wo = task.WO_NO.rsplit("-",1)[-1]
        records.append(("work", f"{task.STRT_DT:%m/%d} {task.MNT_TYPE} {short_wo} · {remark}",
                        COL[task.MNT_TYPE]))
        for part_record in monthly_part_records(task.WO_NO):
            records.append(("part", "교체: " + part_record, COL["PART"]))
        observed = work_observation_links.loc[work_observation_links.WO_NO.eq(task.WO_NO)]
        for link in observed.itertuples():
            records.append(("focus", f"LOT {link.LOT} 관측 중 · {link.START:%H:%M}–{link.END:%H:%M} · {link.MINUTES:g}분",
                            "#7540a8"))
        detected = detection_observation_links.loc[detection_observation_links.WO_NO.eq(task.WO_NO)]
        if observed.empty:
            for link in detected.itertuples():
                records.append(("focus", f"LOT {link.LOT} 감지 시각 관측 · {link.DETC_DT:%H:%M}", "#34495e"))
    if month_work.empty:
        records.append(("empty", "제공된 정비 작업 기록 없음", "#6c7c89"))
    card_height = 48 + sum(10 if kind=="space" else 23 for kind,_,_ in records)
    month_cards.append({"month":month,"records":records,"height":card_height})

row_heights = [max(card["height"] for card in month_cards[row*3:(row+1)*3]) for row in range(4)]
header_height, row_gap, margin_bottom = 90, 16, 16
figure_height = (header_height + sum(row_heights) + 3*row_gap + margin_bottom)/72
fig = plt.figure(figsize=(24,figure_height),facecolor="white")
fig.suptitle("월별 관측창·정비 작업 내용",x=.027,y=1-18/(72*figure_height),
             ha="left",va="top",fontsize=23,fontweight="bold",color="#17344d")
fig.text(.027,1-52/(72*figure_height),
         "온도: LOT별 관측 날짜·시각  |  정비: 작업일·유형·WO·REMARK  |  교체 부품은 기록이 있는 작업에만 표시",
         ha="left",va="top",fontsize=12.8,color="#526d80")
fig.text(.973,1-25/(72*figure_height),"2024년 · 실제 기록 시각",ha="right",va="top",
         fontsize=12.5,color="#526d80")
row_top = header_height
left_margin, right_margin, column_gap = .027, .027, .019
card_width = (1-left_margin-right_margin-2*column_gap)/3
for row in range(4):
    for column in range(3):
        card = month_cards[row*3+column]
        height = row_heights[row]
        ax = fig.add_axes([left_margin+column*(card_width+column_gap),
                           1-(row_top+height)/(72*figure_height),
                           card_width,height/(72*figure_height)])
        ax.set(xlim=(0,1),ylim=(height,0),xticks=[],yticks=[])
        ax.spines[:].set_visible(False)
        focused = card["month"]==11
        ax.add_patch(Rectangle((0,0),1,height,facecolor="#fafcfd",
                               edgecolor="#7540a8" if focused else "#dce5eb",lw=1.4))
        if card["month"] in [6,11]:
            ax.add_patch(Rectangle((0,0),1,35,facecolor=COL["PLAN"],alpha=.35,edgecolor="none"))
        ax.text(.025,12,f"{card['month']:02d}월",ha="left",va="top",fontsize=17,
                color="#17344d",fontweight="bold")
        cursor = 45
        for kind,text,color in card["records"]:
            if kind=="space":
                cursor += 10
                continue
            fontsize = 11.6 if kind=="part" else 12.4 if kind=="focus" else 12.8
            x = .045 if kind in ["part","focus"] else .025
            artist = ax.text(x,cursor,text,ha="left",va="top",fontsize=fontsize,color=color,
                             fontweight="bold" if kind in ["section","focus"] else "normal")
            artist._monthly_card = card["month"]
            cursor += 23
    row_top += height + row_gap

# 줄을 접거나 글씨를 줄이지 않고, 각 월의 글자가 칸 안에 들어가는지 검사한다.
fig.set_dpi(160)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for ax in fig.axes:
    bounds = ax.get_window_extent(renderer)
    for text in ax.texts:
        box = text.get_window_extent(renderer)
        assert bounds.contains(box.x0,box.y0) and bounds.contains(box.x1,box.y1), (
            f"월별 칸의 글자 잘림: {text.get_text()}")
fig._folded='월별·전체 WO 원기록과 시각 대조 펼치기'
finish(fig,"01_개요/03_월별_기록비교.png", save=True)   # 현재 노트북에서 새로 그려 저장한다.

# 정확한 초 단위의 관측창과 WO별 작업 시간을 표로 보존한다.
observation_rows = []
for lot in lots.itertuples():
    work_links = work_observation_links.loc[work_observation_links.LOT.eq(lot.LOT_NO)]
    detection_links = detection_observation_links.loc[detection_observation_links.LOT.eq(lot.LOT_NO)]
    observation_rows.append({
        "월":f"{lot.START.month:02d}월","LOT":int(lot.LOT_NO),"강종":lot.GRADE,
        "첫 측정":lot.START,"마지막 측정":lot.END,"제공된 측정 기록[행]":lot.N_ROWS,
        "관측 중 작업":", ".join(work_links.WO_NO),
        "관측 중 감지 기록":", ".join(detection_links.WO_NO)})
observation_windows = pd.DataFrame(observation_rows)
maintenance_rows = []
for task in cal_work.itertuples():
    observed = work_observation_links.loc[work_observation_links.WO_NO.eq(task.WO_NO)]
    maintenance_rows.append({
        "월":f"{task.STRT_DT.month:02d}월","유형":task.MNT_TYPE,"WO_NO":task.WO_NO,
        "감지 기록 시각":task.DETC_DT,"작업 시작":task.STRT_DT,"작업 완료":task.END_DT,
        "기록된 작업 내용":str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재",
        "기록된 교체 부품":"; ".join(monthly_part_records(task.WO_NO)),
        "온도 관측 중 작업":"; ".join(
            f"LOT {r.LOT} · {monthly_clock_range(r.START,r.END,seconds=True)} · {r.MINUTES:g}분"
            for r in observed.itertuples())})
monthly_maintenance_details = pd.DataFrame(maintenance_rows)
table(observation_windows)
table(monthly_maintenance_details)

**관찰:** 6월·11월은 계획수리 전후에 관측창이 놓여 있다. 11월 13일에는 작업 중 온도가 기록되지만, 12월 28일에는 감지 시각에만 온도가 기록된다.

**가설 검토:** ‘같은 달의 온도와 부품 교체는 직접 연결된다’는 가설은 시각 교집합으로 제한한다. 12월 셀 교체 뒤 온도·CP는 제공 자료에 없어 교체 효과를 검증할 수 없다.

**판단 범위:** 작업 내용은 REMARK 원문, 교체는 동일 WO의 기록을 사용했다. 작업 중 온도가 있다는 사실이 설비의 계속 가동이나 해당 부품의 실제 교체 순간을 뜻하지 않는다.

### 8.16 CAL01 44 WO의 감지 → 착수 → 완료

**읽을 질문:** 변화는 감지 전인가, 대기 중인가, 작업 중인가, 완료 뒤인가?

In [ ]:
# CAL01 작업 44건을 '그림이 들어간 표'로 보인다: 한 줄 = 작업 1건, 월별로 묶고 칸을 맞춘다.
# 시간 막대는 모양만(6시간 넘는 대기는 //로 끊음) · 정확한 시간은 숫자 칸에서 읽는다.
PAD_H = 72                                              # 가까운 LOT으로 볼 시간
TYPE_C = {"BM": report_viz.MAINTENANCE_COLORS["BM"], "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"]}
w = cal_work.sort_values("DETC_DT").reset_index(drop=True)
w["감지→착수[h]"] = (w.STRT_DT - w.DETC_DT).dt.total_seconds() / 3600
w["작업 시간[h]"] = (w.END_DT - w.STRT_DT).dt.total_seconds() / 3600
parts_of = cal_parts.groupby("WO_NO").apply(lambda d: "\n".join(f"{r.PART_NM} ×{int(r.QTY)} ({r.CHG_RSN})" for r in d.itertuples()), include_groups=False)
w["교체 부품"] = w.WO_NO.map(parts_of).fillna("")
def near_lot(t):
    """작업 기간과 LOT 관측의 시간 관계(겹침 · 관측 끝 뒤 · 관측 시작 전, 72시간 안)."""
    out = []
    for k, r in lot_meta.iterrows():
        if t.DETC_DT <= r.END and t.END_DT >= r.START: out.append(f"{k} 관측 중")
        elif pd.Timedelta(0) <= t.DETC_DT - r.END <= pd.Timedelta(hours=PAD_H): out.append(f"{k} 관측 끝 {(t.DETC_DT - r.END).total_seconds() / 3600:.0f}h 뒤")
        elif pd.Timedelta(0) <= r.START - t.END_DT <= pd.Timedelta(hours=PAD_H): out.append(f"{k} 관측 시작 {(r.START - t.END_DT).total_seconds() / 3600:.0f}h 전")
    return "\n".join(out)
def temp_record(t):
    """그 작업을 온도로 확인할 수 있는가: 작업 중 기록 / 감지 때만 / 앞뒤 LOT이 72시간 안."""
    if any(t.STRT_DT < r.END and t.END_DT > r.START for _, r in lot_meta.iterrows()): return "작업 중 기록"
    if any(r.START <= t.DETC_DT <= r.END for _, r in lot_meta.iterrows()): return "감지 때만"
    before = lot_meta.END.le(t.DETC_DT) & (t.DETC_DT - lot_meta.END).le(pd.Timedelta(hours=PAD_H))
    after = lot_meta.START.ge(t.END_DT) & (lot_meta.START - t.END_DT).le(pd.Timedelta(hours=PAD_H))
    return "앞뒤 LOT" if before.any() and after.any() else ""
w["가까운 LOT"] = [near_lot(t) for t in w.itertuples()]
w["온도 기록"] = [temp_record(t) for t in w.itertuples()]
plan_of = lambda t: next((p for p in plans.itertuples() if p.STRT_DT <= t < p.END_EXCL), None)
hours = lambda h: f"{h / 24:.1f}일" if h >= 24 else f"{h:.1f}h"

# 줄 목록: 월 머리 줄 + 작업 줄
rows = []
for m, d in w.groupby(w.DETC_DT.dt.month):
    ps = [p for p in plans.itertuples() if p.STRT_DT.month == m]
    rows.append(("month", f"{m}월 · {len(d)}건" + "".join(f"   ·   CR1 정기수리 {p.STRT_DT:%m/%d}~{p.END_DT:%m/%d} ({p.SCOPE})" for p in ps)))
    rows += [("wo", t) for _, t in d.iterrows()]
COLS = [(.0, "감지 시각"), (.072, "종류"), (.112, "WO"), (.165, "◆ 감지 · 점선 = 착수까지 대기 · 막대 = 작업"), (.35, "감지→착수"), (.405, "작업 시간"), (.452, "설비 정지"),
        (.5, "REMARK · 고장 유형"), (.645, "교체 부품"), (.79, "온도 기록"), (.86, "가까운 LOT (±72시간)")]
G0, GW, HMAX = .165, .165, 16                            # 시간 막대 칸 시작·폭, 칸 안에서 보이는 최대 시간
gx = lambda h: G0 + GW * min(h, HMAX) / HMAX
H = .5 * len(rows) + 4.0
fig = plt.figure(figsize=(30, H)); ax = fig.add_axes([.02, 1.0 / H, .97, 1 - 3.5 / H]); ax.axis("off")   # 위 2.5인치는 제목·범례, 아래 1인치는 계산식
ax.set_xlim(0, 1); ax.set_ylim(len(rows) - .5, -1.2)
for x, name in COLS:
    ax.text(x, -.95, name, fontsize=13, fontweight="bold", color="#526d80", va="center")
ax.plot([0, 1], [-.55, -.55], color="#9fb3c3", lw=1.2)
for i, (kind, t) in enumerate(rows):
    if kind == "month":
        ax.add_patch(Rectangle((0, i - .42), 1, .84, color="#e9eff4", lw=0, zorder=0))
        ax.text(.004, i, t, fontsize=13.5, fontweight="bold", color="#17344d", va="center")
        continue
    c = TYPE_C.get(t.MNT_TYPE, "#7c8995")
    if plan_of(t.STRT_DT) is not None:
        ax.add_patch(Rectangle((0, i - .45), 1, .9, color="#f3f7fa", lw=0, zorder=0))
    ax.add_patch(Rectangle((0, i - .4), .003, .8, color=c, lw=0))
    ax.text(.008, i, f"{t.DETC_DT:%m/%d %H:%M}", fontsize=12.5, va="center", color="#17344d")
    ax.text(.072, i, t.MNT_TYPE, fontsize=12, va="center", color="white", fontweight="bold", bbox=dict(boxstyle="round,pad=.25", fc=c, ec="none"))
    ax.text(.112, i, t.WO_NO.replace("WO-2024-", ""), fontsize=12, va="center", color="#526d80")
    wait, work = t["감지→착수[h]"], t["작업 시간[h]"]
    wait_s = wait if wait <= 6 else 7                           # 6시간 넘는 대기는 끊어 그림(//)
    ax.plot([gx(0), gx(wait_s)], [i, i], color=c, ls=":", lw=2)
    if wait > 6:
        ax.text(gx(6.5), i, "//", fontsize=13, color=c, ha="center", va="center", fontweight="bold", bbox=dict(boxstyle="square,pad=.05", fc="white", ec="none"))
    ax.add_patch(Rectangle((gx(wait_s), i - .17), gx(wait_s + min(work, HMAX - wait_s)) - gx(wait_s), .34, color=c, lw=0))
    ax.plot(gx(0), i, "D", ms=7, color=c)
    ax.text(.35, i, hours(wait), fontsize=12.5, va="center", color=c if wait >= 24 else "#17344d", fontweight="bold" if wait >= 24 else "normal")
    ax.text(.405, i, hours(work), fontsize=12.5, va="center", color="#17344d")
    ax.text(.452, i, f"{t.DOWN_MIN:.0f}분", fontsize=12.5, va="center", color="#17344d" if t.DOWN_MIN else "#9aa6ae")
    rem = (str(t.REMARK).strip() if pd.notna(t.REMARK) else "내용 미기재") + (f" · {t.FAIL_MODE}" if pd.notna(t.FAIL_MODE) else "")
    ax.text(.5, i, rem, fontsize=12.5, va="center", color="#17344d" if pd.notna(t.REMARK) else "#9aa6ae")
    if t["교체 부품"]:
        ax.text(.645, i, t["교체 부품"], fontsize=11.5, va="center", color=SELECT_COLOR, fontweight="bold", linespacing=1.15)
    if t["온도 기록"]:
        ax.text(.79, i, t["온도 기록"], fontsize=12, va="center", color="#17344d" if t["온도 기록"] == "작업 중 기록" else "#526d80",
                fontweight="bold" if t["온도 기록"] == "작업 중 기록" else "normal")
    if t["가까운 LOT"]:
        ax.text(.86, i, t["가까운 LOT"], fontsize=11.5, va="center", color="#17344d", linespacing=1.15)
n_part = int(w["교체 부품"].ne("").sum()); n_long = int(w["감지→착수[h]"].ge(24).sum()); n_rec = int(w["온도 기록"].eq("작업 중 기록").sum())
header(fig, f"CAL01 정비 이력 {len(w)}건 · 감지 → 착수 → 완료와 작업 내용",
       f"한 줄 = 작업 1건(감지 순, 월별 묶음) · 감지→착수 중앙값 {w['감지→착수[h]'].median():.1f}h, 1일 넘게 기다린 작업 {n_long}건 · 부품 교체 {n_part}건(빨강) · 작업 중 온도 기록 {n_rec}건 · 연한 바탕 = CR1 정기수리 기간",
       [Patch(facecolor=TYPE_C[k], label=f"{k} {int(w.MNT_TYPE.eq(k).sum())}건") for k in TYPE_C], ncol=3)
compact_header(fig, .975 - .6 / H, .975 - 1.1 / H, title_size=25, sub_size=14, legend_size=13)
fig.text(.02, .25 / H, "감지→착수 = 작업 시작(STRT_DT) - 감지(DETC_DT) · 작업 시간 = 완료(END_DT) - 작업 시작 · 설비 정지 = 기록된 정지 시간(DOWN_MIN) · "
         "온도 기록 = 작업하는 동안 LOT 온도 기록이 있음(작업 중 기록) / 감지 시각에만 있음(감지 때만) / 앞뒤 LOT이 모두 72시간 안(앞뒤 LOT)",
         ha="left", va="bottom", fontsize=12.5, color="#526d80")
axis_tag(fig, "작업 1건 = 1줄 · 감지 순서")
fig._folded='월별·전체 WO 원기록과 시각 대조 펼치기'
finish(fig, "01_개요/04_WO44_작업흐름.png", save=True)   # 현재 노트북에서 새로 그려 저장한다.
table(w[["WO_NO", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "감지→착수[h]", "작업 시간[h]", "DOWN_MIN", "FAIL_MODE", "REMARK", "교체 부품", "온도 기록", "가까운 LOT"]]
      .rename(columns={"DOWN_MIN": "설비 정지[분]"}).round({"감지→착수[h]": 1, "작업 시간[h]": 1}))

**관찰:** 감지(DETC_DT), 착수(STRT_DT), 완료(END_DT)는 서로 다르다. 6시간을 넘는 대기 막대는 `//`로 축약했으므로 숫자 칸에서 정확한 시간을 읽는다. DOWN_MIN은 기록된 정지 시간으로 작업 시간과 별도다.

**가설 검토:** ‘관측 중 DETC_DT가 있으므로 작업 전후 효과를 볼 수 있다’는 가설은 240133에서 성립하지 않는다. 작업 자체가 관측 이후다. 240117은 실제 작업 중과 완료 뒤를 구분해 볼 수 있다.

**판단 범위:** DETC_DT는 정비 자료의 감지 기록 시각이다. 온도 센서 기반 자동 경보 시각이나 최초 고장 발생 시각으로 단정하지 않는다.

<details><summary><b>전체 판정 근거표 펼치기</b></summary>

### 8.17 추가 판정 근거

| 검토 항목 | 관측으로 확인한 결과 | 현재 판정 |
|---|---|---|
| CP 순간 이탈 | 240037·240069에서 반복, 240117에서 작업 완료 뒤 하강 | 원본 확대·공정 반응 확인 대상. 원인은 미확정 |
| 같은 강종의 출력 수준 | 240038(SPCEN)·240061(SPHC)의 OP-Z4·OP-Z8 등의 차이 | 설정·부하·계측 차이 가설을 남김 |
| 같은 강종의 온도 수준 | 240091·240114·240121의 Z7·Z8이 대조 LOT보다 높음 | 과시효 설정·센서 보정 여부 확인 대상 |
| 수리 전후 큰 변화 | 6월·11월 전후의 하강·승온 형태 | 일정과 부합하는 대안 설명이 있음. 고장 증거로 바로 쓰지 않음 |
| 정비와의 직접 비교 | 실제 작업 교집합은 240117, 감지 시각만 관측된 240133은 작업이 관측 종료 뒤 | 감지·착수·완료를 구분해야 함 |

위 8 LOT은 **선택한 기준으로 더 볼 대상**이다. 나머지를 정상으로 확정한 결과가 아니다. 단독 강종·짧은 관측·관측 공백 때문에 비교할 수 없는 범위가 있다.

</details>

# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def zoom_flow(k):
    # 모든 구간 열에 같은 공정 행을 적용한다. 비교 중 공정 위치가 바뀌지 않는다.
    evs=LOT_EVENTS[k];chosen={2,3,4,5}
    if any(e['sensor']=='CLN-IN' for e in evs):chosen.add(0)
    if any(degree_zone(e['sensor']) in [1,2] for e in evs):chosen.add(1)
    out=[]
    for i,(name,left,right) in enumerate(FLOW_ROWS):
        if i not in chosen:continue
        out.append((name+' · '+('CP' if i==4 else '온도'),left))
        if right:out.append((name+' · '+('CP-Z4M' if i==4 else '출력'),right))
    return out

def lot_result_markdown(k):
    return '**구간별 검토 결과**\n\n'+ '\n\n'.join(event_result_markdown(k,e) for e in LOT_EVENTS[k])+ '\n\n**읽은 뒤 판단:** (a)(b)(c)는 서로 다른 시간·근거다. 최대값 하나로 나머지를 정상 처리하지 않는다. 각 구간의 TC–OP 동반 반응·CP 원본·정비 시각을 위의 동일 크기 상세에서 각각 확인한다.'

def plan_result_markdown(idx,pre,post):
    d=plan_case_data(idx,pre,post);pr,po,ws=d['pre'],d['post'],d['ws']
    tp=expected.loc[expected.LOT.eq(pre)].iloc[0];ta=expected.loc[expected.LOT.eq(post)].iloc[0]
    g=raw[post].set_index('MEAS_DT');a,b=d['zoom_a'],d['zoom_b'];q=g.loc[a:b]
    base=g['CP-Z4'].median();cpdev=(q['CP-Z4']-base).abs().max();t=d['point']
    before=g.loc[t-pd.Timedelta(minutes=10):t-pd.Timedelta(minutes=5),'TC-Z3'].median()
    after=g.loc[t+pd.Timedelta(minutes=5):t+pd.Timedelta(minutes=10),'TC-Z3'].median()
    lines=['**이 그림에서 확인한 사실**','',
      f'1. **수리 기간과 관측을 분리했다.** 계획수리 안에는 온도 0행이다. 위 파란 날짜 범위에서는 작업 WO와 강종 이벤트를 확인하며, 온도는 수리 전(A) {pre}({pr.GRADE})·수리 후(B) {post}({po.GRADE})에서만 관측된다. 전체 범위 실제 작업은 {len(ws)} WO다. 기간 안의 온도·정지 상태를 추정해서 그리지 않았다. ◆는 교체 WO 완료 시각을 대용하며 실제 교체 순간은 아니다.',
      f'2. **일정과 맞는 전환은 먼저 설명한다.** 전 LOT 하강 {tp.시작:%m/%d %H:%M}–{tp.끝:%H:%M}, 후 LOT 승온 {ta.시작:%m/%d %H:%M}–{ta.끝:%H:%M}를 회색으로 표시했다. 여러 TC·OP의 동시 변화와 수리 전후라는 위치는 운전 전환 가설을 뒷받침한다. 사후 형태 분류여서 실제 모드·SV·정지 명령 확인은 남는다.',
      f'3. **수리 후(B)의 남는 응답을 (B-1)로 확대했다.** (B-1) {a:%m/%d %H:%M}–{b:%H:%M}의 TC·OP·CP는 같은 시간축이다. 확인 시각 전 10~5분과 후 5~10분 TC-Z3 중앙값은 {before:.2f}℃ → {after:.2f}℃다. 원본의 복귀·지속과 함께 읽는다. 이 창의 CP 최대 절대 편차는 {cpdev:.4f}다. 국소 응답 근거이며 정상 범위의 검증 자료는 아니다.','',
      '**어떻게 이상이 아닐 가능성을 검토하는가?** 일정과 다신호 동시 하강·승온 → 기준 수준으로 접근·복귀 → 같은 TC–OP의 반복 제어 응답 → 별도 CP 이탈 잔존 여부 순서로 본다. 근거가 있는 전환은 **고장 후보와 분리한 설명 가능 구간**으로 둔다. 구간 전체의 정상 보증으로 바꾸지 않는다. 전환 뒤의 단발 응답·CP 사건도 별도 검토한다.','',
      f'**수리 효과로 결론 내릴 수 없는 이유:** 강종 {pr.GRADE}→{po.GRADE}, 관측 시점과 운전 조건이 다르고 수리 중 온도도 없다. 출력·온도 높낮이를 수리의 개선·악화 효과로 해석하지 않는다.']
    reason=('승온 참고 구간 이후 TC-Z3의 절대 Z가 가장 큰 창을 코드로 선택했다. 모든 온도 또는 전체 수리 효과가 이상이라는 뜻은 아니다.' if post==240061 else 'CBM 완료 뒤 CP 원본 이탈이 남는 창을 선택했다. 온도 최대점만 보고 정한 창이 아니다.')
    lines+=['','**왜 (B-1)을 골랐는가?** '+reason,'',
      '**수리를 했는데 왜 다시 변화가 있는가? — 현재 자료로 나눠 볼 가설**','',
      '- **재가동 전환:** 공통 승온은 재가동 형태로 설명할 수 있어 수준 비교에서 제외했다. (B-1)은 그 이후의 반응을 따로 확인한다.',
      '- **국소 제어 응답:** TC-Z3의 단발 상승·하강과 OP-Z3의 저하·복귀를 같은 시간에 대조한다. 이 동반 반응은 제어 응답 가설과 부합할 수 있지만 설정값·제어 로그가 없어 원인을 확정하지 않는다.',
      '- **정비 대상·원인 불일치:** 캘린더의 정기수리와 WO REMARK는 작업 기록이다. 그 작업이 Z3 제어 또는 CP 변화의 원인을 제거했다는 검증 기록은 없다. 수리 후 관측됐다는 이유만으로 수리 실패·재고장으로 부르지 않는다.',
      '- **추가 검증:** 같은 강종·설정·속도·두께에서 재현되는지, SV 대비 편차와 OP 응답, 정비 대상 부품·계측 보정·완료 점검 결과를 확인해야 원인을 좁힐 수 있다.']
    if post==240117:lines+=['','**11월 확대의 핵심:** CP 하강은 CBM 12:44 완료 후에 발생한다. 승온이라는 설명만으로 이 CP 사건까지 정상 처리하지 않으며, 해당 CBM 전 경보라는 해석도 시간 순서와 맞지 않는다.']
    else:lines+=['','**6월 확대의 핵심:** 승온 이후 Z3의 단발 응답이 있어도 Z4–Z6·CP가 같은 폭으로 지속 이탈하는지 따로 본다. 온도·출력이 돌아오는 모습은 일시 제어 반응의 근거이며, 정상 제어임을 확정하려면 SV·조업 조건이 필요하다.']
    return '\n'.join(lines)

def level_result_markdown(k):
    grade=lot_meta.loc[k,'GRADE'];d=level_check.loc[level_check.LOT.eq(k)]
    hit=d.loc[d.DISPLAY_HIT];zones=sorted({degree_zone(s) for s in hit.SENSOR})
    pairs=[f'{prefix}-Z{z}' for z in zones for prefix in ['TC','OP']]
    companions=d.loc[d.SENSOR.isin(pairs)&~d.DISPLAY_HIT]
    lines=['**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**','',
      '같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.','',
      f'이 그림은 기존 우선 검토 LOT {k}({grade})의 상세다. 4.3절에서 16신호 분포를 확인한 뒤, **절대 차이 {LEVEL_TC_LIMIT:g}℃ 또는 {LEVEL_OP_LIMIT:g}%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.','',
      '**표시 경계를 넘은 신호:**','']
    for r in hit.itertuples():
        u='℃' if r.SENSOR in LEVEL_TC else '%p'
        lines.append(f'- {r.SENSOR}: 이 LOT {r.LOT_MEDIAN:.3f} - 대조 {r.PEER_MEDIAN:.3f} = **{r.GAP:+.3f}{u}**, 다른 {r.N_PEERS} LOT과 대조.')
    if len(companions):
        lines+=['','**대응 신호로 함께 보여 준 것:**','']
        for r in companions.itertuples():
            u='℃' if r.SENSOR in LEVEL_TC else '%p'
            lines.append(f'- {r.SENSOR}: 차이 {r.GAP:+.3f}{u}. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.')
    if k==240061:
        lines+=['','**이 사례의 핵심:** OP-Z4의 차이가 커서 TC-Z4를 함께 본 것이다. TC-Z4 자체를 이상 온도로 선별한 것이 아니다. OP-Z8도 표시 경계를 넘으므로 TC-Z8·OP-Z8을 함께 표시했다. 비슷한 온도 수준에서 출력 수준이 다르게 관측되는 이유를 설정·속도·두께·부하·보정 기록으로 확인해야 한다.']
    lines+=['','**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.','']
    for kk in lots.loc[lots.GRADE.eq(grade),'LOT_NO'].astype(int):
        ranges=runs(comparison_states[kk].ne('안정 참고'))
        spans=' / '.join(f'{at(kk,a):%m/%d %H:%M}–{min(at(kk,b),lot_meta.loc[kk,"END"]):%H:%M}' for a,b in ranges) or '전환 제외 없음'
        lines.append(f'- {kk}: {spans}.')
    lines+=['','**판정:** 위는 같은 강종에서 관측된 수준 차이다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.']
    if int(d.N_PEERS.min())==1:lines+=['같은 강종이 두 LOT뿐이므로 차이의 방향은 대칭적이다. 어느 LOT이 정상인지 정할 수 없다.']
    return '\n'.join(lines)

<details><summary><b>계산 정의·재현 조건 펼치기</b></summary>

- **온도·출력 추세:** 1분에 54개 이상 실제 기록이 있는 창의 중앙값. 평균과 중앙값은 순간 최고·최저가 아니다.
- **시간순 Z:** 이전 30분 중앙값과 이전 표준편차, 최소 20개 과거 유효창. SD 하한 TC 0.5℃·OP 2%p. `|Z|≥3`은 탐색 표시선이다.
- **IQR:** LOT·신호별 전환 제외 1분 중앙값. 상자는 Q1~Q3, 수염은 1.5 IQR 안 실제 끝값. IQR 0은 판정 보류한다.
- **CP:** LOT 원본 중앙값 대비 절대편차 0.02 초과 묶음 중 최대편차 0.05 이상을 검사한다. 짧은 CP가 1분 요약에서 작아질 수 있어 원본을 유지한다.
- **판단 범위:** CP·Z·IQR은 다른 질문을 위한 탐색 기준이다. 정상 확률·고장 라벨·정비 효과로 합치지 않는다.

</details>

### 실행·산출물 검사

14 LOT 각각의 전체 그림, 표시한 모든 확대, 해당 강종 대조, 두 정기수리 일정 그림과 실제 대조 가능한 WO 그림이 만들어졌는지 확인한다. 새 그림은 저장 파일을 다시 읽지 않고 Figure를 직접 출력한다. 제목·범례·시간 눈금의 실제 경계를 검사한다.

In [ ]:
# focus_event: 바로 위 첫 사용 절의 생성 함수를 재사용한다.
assert len(temp)==345390 and len(lots)==14 and len(grade_evt)==134
assert cal_work.MNT_TYPE.value_counts().to_dict()=={'TBM':22,'BM':12,'CBM':10}
assert set(work_observation_links.LOT)=={240117} and len(work_observation_links)==2
assert np.isclose(sum(b-a for a,b in work_ranges[240117]),304)
assert cp_count.loc[240037]==3 and cp_count.loc[240069]==7
assert set(pd.DataFrame(lot_execution).LOT)==set(lots.LOT_NO.astype(int))
assert plan_compare['수리 중 온도 행'].eq(0).all()
checks=pd.DataFrame(figure_checks)
assert not checks.FILE.duplicated().any()
expected_full={f'03_상세/01_LOT전체/{k}_{lot_meta.loc[k,"GRADE"]}.png' for k in LOT_EVENTS}
expected_zoom={f'03_상세/02_구간확대/{k}_전체구간_공정순서.png' for k in LOT_EVENTS}
expected_wo={f'03_상세/04_정비대조/{k}_WO전중후.png' for k in [240069,240117,240133]}
expected_plans={f'03_상세/04_정비대조/정기수리_{month}_일정_전후_확대.png' for month in ['6월','11월']}
assert expected_plans<=set(checks.FILE)
assert expected_full|expected_zoom|expected_wo<=set(checks.FILE)
assert all((OUT_DIR/name).is_file() for name in checks.FILE)
show_table(pd.DataFrame(lot_execution))
show_table(checks[['FILE','TEXT_OVERLAP','LEGEND_BOUNDS']])
note(f'검증 통과 · 전체 LOT {len(expected_full)}개 / LOT별 통합 확대 {len(expected_zoom)}개 / 강종 수준 대조 {len(LEVEL_TARGETS)}개 / 실제 WO 대조 {len(expected_wo)}개 / 정기수리 일정 2개 / 전체 그림 {len(checks)}개')
assert quality_effect['3이상 후보'].sum()==zscreen.MAX_ABS_Z.ge(Z_LIMIT).sum()
assert len(cp_quality_evidence)==len(cp_real) and peak_counts.ge(54).all()

assert len(expected_zoom)==14 and len(expected_plans)==2
assert len(checks)==61
assert all(len(events)<=3 for events in LOT_EVENTS.values())
for k,events in LOT_EVENTS.items():
    expected_peaks=set(cp_real.loc[cp_real.LOT.eq(k),'PEAK'])
    shown_peaks={p.PEAK for e in events if e['kind']=='CP' for p in e['pulses'].itertuples()}
    assert expected_peaks==shown_peaks, f'CP 사건 누락: {k}'
assert len(screening_result)==14
assert set(screening_result['LOT·강종'].str[:6].astype(int))==set(lots.LOT_NO.astype(int))
assert len(cp_quality_evidence)==14
assert len(level_check)==len(lots)*16
assert set(LEVEL_TARGETS[240061])=={'OP-Z4','OP-Z8'}
assert all((level_check.loc[level_check.LOT.eq(k)&level_check.SENSOR.isin(tags),'DISPLAY_HIT']).all() for k,tags in LEVEL_TARGETS.items())
assert len(zoom_presentations)==sum(len(e) for e in LOT_EVENTS.values())
assert {r['LOT'] for r in zoom_presentations}==set(LOT_EVENTS)
for k,events in LOT_EVENTS.items():
    focus=focus_event(k);cp=[e for e in events if e['kind']=='CP']
    if cp:assert abs(focus['deviation'])==max(abs(e['deviation']) for e in cp)
assert focus_event(240069)['tag']=='(c)' and focus_event(240117)['tag']=='(b)'
assert all(len(lot_cards(k))==3 for k in LOT_EVENTS)

assert {(r['LOT'],r['TAG']) for r in zoom_presentations}=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events}
assert len(iqr_summary)==224
assert iqr_summary.OUT.sum()==sum(mask.sum() for mask in iqr_masks.values())

assert set(locator_checks)=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events}
assert {(k,tag) for k,tag,sensor in overview_trigger_checks}=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events if e['kind'] in ['CP','Z']}

assert set(report_case_checks)=={240069,240037,240117,240061} and len(report_case_checks)==4
assert cp_time_features.SPAN_SEC.ge(0).all()
assert cp_time_features.POINTS.ge(1).all()
assert len(cp_time_features)==14 and len(interval_time_features)==32
assert {f'01_개요/사례_{k}_보고서.png' for k in report_case_checks}<=set(checks.FILE)

assert len(cp69_peak_checks)==7 and set(cp69_peak_checks)==set(cp69_pulses.ID)
assert len(cp69_signalstats)==147 and cp69_signalstats.N_BASE.ge(190).all()
assert cp69_b_compare.CP_DEV.lt(.05).all()
assert cp69_b_compare.iloc[1].X>cp69_b_compare.iloc[0].X and cp69_b_compare.iloc[1].Z<cp69_b_compare.iloc[0].Z

assert len(cp69_zone_metrics)==42 and cp69_zone_metrics.N_PEAK.ge(24).all()
assert len(cp69_zone_peak_checks)==36 and set(cp69_zone_peak_checks)=={(p.ID,s) for p in cp69_pulses.iloc[:6].itertuples() for s in CP69_ZONE_SENSORS}

b_evidence=cp69_b_observations()
assert b_evidence["CP_HITS"]==0 and b_evidence["CP_MAX_DEV"]<.05
assert b_evidence["CP_VALID"]+b_evidence["DATA"]["CP-Z4"].isna().sum()==b_evidence["SECONDS"]
assert b_evidence["POINTS"][1]["TC_AT"]==cp69_later_point

# 관계 분석: 14사건×9신호, 모든 LOT·시차·짝의 실제 기록을 검증한다.
assert len(relation_events)==14 and relation_events.LOT.nunique()==6
assert len(relation_event_delta)==14*9
assert relation_event_delta.loc[relation_event_delta.VALID,'N_BASE'].ge(190).all()
assert relation_event_delta.loc[relation_event_delta.VALID,'N_PEAK'].ge(24).all()
assert relation_lags.loc[relation_lags.R.notna()&relation_lags.PAIR.eq('CP'),'N_PAIRS'].ge(40).all()
assert relation_lags.loc[relation_lags.R.notna()&relation_lags.PAIR.eq('OP_TC'),'N_PAIRS'].ge(200).all()
assert relation_lags.R.dropna().between(-1,1).all()
assert set(relation_lags.loc[relation_lags.PAIR.eq('OP_TC'),'LOT'])==set(raw)
assert len(relation_match_coverage)==14
assert len(relation_matches)==8*relation_match_coverage.STATUS.eq('대조 가능').sum()
assert relation_matches.CONTROL_CP_MAX.le(.02).all() and relation_matches.BASE_TC456_MAX_GAP.le(2.5).all()
assert not relation_matches.duplicated(['LOT','SENSOR','CONTROL_START']).any()
assert {f'05_관계검토/{name}' for name in ['01_CP사건별_Z3-Z6_변화량.png','02_CP하강_주변존_동시변화.png','03_CP이탈_유무_실제파형.png','04_CP두채널_관측관계.png']}<=set(checks.FILE)

assert relation_events.ID.is_unique and set(relation_events.EVENT_ID)==set(cp_time_features.EVENT_ID)

# 확대창: 같은 실제 x축·눈금, 유형색, 실제 Z 1분창을 검증한다.
assert len(zoom_alignment_checks)==sum(len(events) for events in LOT_EVENTS.values())
assert {(x['LOT'],x['TAG']) for x in zoom_alignment_checks}=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events}
assert all(x['LINE']==sensor_color(x['SENSOR']) for x in locator_visual_checks)

# 계산용 ID·HTML 앵커는 보존하고 표시만 (a-1) 형식으로 통일한다.
assert event_text('240069-a1')=='240069 (a-1)'
assert event_text('240069-a1 · CP-Z4',include_lot=False)=='(a-1) · CP-Z4'
assert event_text('b-1 · TC-Z3',include_lot=False)=='(b-1) · TC-Z3'
assert event_text('(a-1) CP-Z4')=='(a-1) CP-Z4'
assert event_text('#event-240069-a')=='#event-240069-a'
assert len(display_label_checks)>0 and all(event_text(label,include_lot=False)==label for _,label in display_label_checks)

assert event_text('B-1 수리 후 확인 구간')=='(B-1) 수리 후 확인 구간'

# CP 방향별 실제 파형: 하강 2·상승 12사건, 모든 존, 같은 x축을 확인한다.
assert len(relation_wave_checks)==14 and len(relation_wave_metrics)==14*8
assert {r['ID'] for r in relation_wave_checks}==set(relation_events.ID)
assert sum(r['DIRECTION']=='하강' for r in relation_wave_checks)==2
assert all(r['N_PANELS']==5 and r['SENSORS']==set(REL_SENSORS+['CP-Z4']) for r in relation_wave_checks)
assert not (OUT_DIR/'05_관계검토/02_사건별_시차와_OP_TC.png').exists()

# CP 이탈이 있던 때/없던 다른 때의 실제 파형과 원래 4분 최대폭 일치를 확인한다.
assert len(relation_pair_wave_checks)==12
assert len(relation_pair_amplitude_checks)==12*8*2
assert {v['ID'] for v in relation_pair_wave_checks}==set(relation_matches.ID)
assert all(v['N_PANELS']==9 for v in relation_pair_wave_checks)
assert not (OUT_DIR/'05_관계검토/03_CP사건과_비사건_짝대조.png').exists()

assert '05_관계검토/05_TC_OP_동시파형.png' in set(checks.FILE)
assert len(tcop_zero)==56 and tcop_zero.N_PAIRS.ge(200).all()

# TC–OP 고출력·추가 하강·고정 기준 복귀 계산을 확인한다.
assert "05_관계검토/06_TC_OP_고출력후_하강과복귀.png" in set(checks.FILE)
assert len(response_returns)==6 and len(output_fall_coverage)==56
assert set(output_fall_coverage.LOT)==set(raw)
assert set(output_fall_coverage.ZONE)=={3,4,5,6}
assert response_returns.loc[response_returns.TAG.eq("(b-1)")&response_returns.BAND_C.eq(1.),"CONFIRMED"].notna().all()
assert response_returns.loc[response_returns.TAG.eq("(b-2)")&response_returns.BAND_C.eq(1.),"CONFIRMED"].isna().all()


In [ ]:

# Z 17신호 + 원본 별도 경로 5신호: 모든 LOT의 22신호와 후보 누락을 확인한다.
assert len(review_z_coverage)==len(lots)*len(REVIEW_Z_GROUPS)==42
assert len(review_process_coverage)==len(lots)*4==56
for k in lots.LOT_NO.astype(int):
    coverage=[v for (lot,_),v in review_z_coverage.items() if lot==k]
    assert set().union(*(v['SENSORS'] for v in coverage))==set(REVIEW_SIGNALS)
    assert set().union(*(v['EVENT_IDS'] for v in coverage))==set(review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('Z'),'ID'])
    actual=[v for (lot,_),v in review_process_coverage.items() if lot==k]
    assert set().union(*actual)==set(SIGNALS)
assert len(review_exports)==len(review_layout_checks)==56
assert all(Path(v['FILE']).is_file() for v in review_exports.values())
assert review_events.loc[review_events.KIND.eq('CP')].shape[0]==len(cp_events)
assert all(np.array_equal(review_raw_context.loc[review_raw_context.LOT.eq(k),'VALUE'].to_numpy(),
    temp.loc[review_raw_context.loc[review_raw_context.LOT.eq(k),'SOURCE_ROW'],'CP-Z4'].to_numpy(),equal_nan=True)
    for k in lots.LOT_NO.astype(int))
assert all(v['GLYPHS']=='통과' and v['TEXT_OVERLAP']=='통과' and v['LEGEND_OVERLAP']=='통과' and v['BOUNDS']=='통과' for v in render_audit_checks.values())
display(Markdown(f"검증 통과 · {len(lots)} LOT × 전체 {len(SIGNALS)}신호 · LOT별 그림 {len(review_exports)}개 · "
                 f"Z 후보 {review_events.KIND.eq('Z').sum()}개·CP 원본 이탈 {review_events.KIND.eq('CP').sum()}개와 정비 대조 근거. "
                 "새 Z·예측 모델을 추가하지 않았으며 원값·기존 Z 계산을 유지했다."))
display(Markdown(f"저장 PNG {len(list(OUT_DIR.rglob('*.png')))}개 · 새 그림 전부 글리프·글자/범례 겹침·잘림 검사 통과. "
                 "같은 Figure로 노트북 표시와 figures 저장을 생성했다."))

assert len(review_heating_checks)==len(lots)==14
for k,v in review_heating_checks.items():
    assert v['ROWS']==10
    assert v['SENSORS']==set(['CP-Z4','CP-Z4M']+[s for z in range(3,7) for s in (f'TC-Z{z}',f'OP-Z{z}')])
    assert v['CP_IDS']==set(review_events.loc[review_events.LOT.eq(k)&review_events.KIND.eq('CP'),'ID'])
